# CausalSight EXP-005: object-color binding

Upload this notebook at https://colab.research.google.com/ and choose a **free GPU**
under Runtime → Change runtime type. No paid API, subscription or API token is used.
If no free GPU is available, retry later. This notebook has not yet been GPU-executed.

The exact versioned dataset, configuration and Python modules are embedded and hashed.
Development: 16 families / 64 visual requests. Evaluation: 128 balanced families /
512 visual requests. Each split also makes two actual text-only requests and explicitly
projects their answers onto the repeated questions; these are not independent trials.

Each request has one image/question and no conversation history. FP16 Instruct weights
are distinct from the previous local quantized thinking smoke model.
Inspect development results before evaluation. An operational failure prevents freezing.
Original accuracy is not used to exclude families or decide whether to run evaluation.

Raw rows flush immediately. On an interrupted runtime, download available artifacts
with the final download cell. The command-line runner supports --resume on unchanged
artifacts/model/runtime; failed requests are preserved and never selectively retried.
Colab storage is temporary: download the ZIP before disconnecting.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No free GPU allocated. Retry later; do not purchase compute.")
print(torch.cuda.get_device_name(0))
if torch.cuda.get_device_properties(0).total_memory < 6*2**30:
    raise RuntimeError("Less than 6 GiB GPU memory: stop before downloading weights.")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "transformers==4.57.1", "accelerate>=1.0,<2", "Pillow>=10,<12"], check=True)

In [ ]:
import base64, hashlib, json, uuid, sys
from pathlib import Path
WORK = Path("/content/causalsight-binding")
WORK.mkdir(parents=True, exist_ok=True)

ASSETS = {'configs/experiments/EXP-005-binding-v1.json': {'base64': 'ew0KICAiZXhwZXJpbWVudCI6ICJFWFAtMDA1LWJpbmRpbmctdjEiLA0KICAiZGF0YXNldCI6ICJkYXRhL2JpbmRpbmctdjEvZXhhbXBsZXMuanNvbmwiLA0KICAiZGF0YXNldF9zaGEyNTYiOiAiZjRhMjBhYjRhYWUyYzA4ODBmNDJmM2Q1OWY2M2Q1ZjViMmNiNGY0NDgwZWY2ZGVmYjM0OTBmMGIxYjFlZjU5ZCIsDQogICJwcm92aWRlciI6ICJ0cmFuc2Zvcm1lcnMiLA0KICAibW9kZWwiOiAiUXdlbi9Rd2VuMy1WTC0yQi1JbnN0cnVjdCIsDQogICJtYXhfdG9rZW5zIjogMjU2LA0KICAiZXhwZWN0ZWRfcm93cyI6IHsNCiAgICAiZGV2ZWxvcG1lbnQiOiA2NCwNCiAgICAiZXZhbHVhdGlvbiI6IDUxMg0KICB9LA0KICAicGFyc2VyIjogImJpbmRpbmctMS4wLjAiLA0KICAic3BsaXRfcG9saWN5IjogIjE2IGRldmVsb3BtZW50IGZhbWlsaWVzOyAxMjggYmFsYW5jZWQgZXZhbHVhdGlvbiBmYW1pbGllczsgbm8gdHJhaW5pbmciLA0KICAic2VlZCI6IDExMDE3LA0KICAiY29tcHV0ZV9wb2xpY3kiOiAiZnJlZSBHUFUgb25seTsgbm8gcGFpZCBBUEkgb3IgYXV0b21hdGljIG1vZGVsIHN1YnN0aXR1dGlvbiINCn0NCg==', 'sha256': '4a52d39af8e6522999385ddc8a6ee570f3564eedb20abdb8b89423b55decf63b'}, 'data/binding-v1/audit.json': {'base64': 'ew0KICAiZmFtaWxpZXMiOiAxNDQsDQogICJyb3dzIjogNTc2LA0KICAiaW1hZ2VzIjogNDMyLA0KICAibWFuaWZlc3Rfc2hhMjU2IjogImY0YTIwYWI0YWFlMmMwODgwZjQyZjNkNTlmNjNkNWY1YjJjYjRmNDQ4MGVmNmRlZmIzNDkwZjBiMWIxZWY1OWQiLA0KICAiZmFtaWxpZXNfYnlfc3BsaXQiOiB7DQogICAgImRldmVsb3BtZW50IjogMTYsDQogICAgImV2YWx1YXRpb24iOiAxMjgNCiAgfSwNCiAgImV2YWx1YXRpb25fY2VsbHMiOiA2NCwNCiAgImZhbWlsaWVzX3Blcl9ldmFsdWF0aW9uX2NlbGwiOiAyLA0KICAidHJhbnNpdGlvbl9jb3VudHMiOiB7DQogICAgImRldmVsb3BtZW50X2NhdXNhbCI6IHsNCiAgICAgICJibHVlLT5ncmVlbiI6IDEsDQogICAgICAiYmx1ZS0+cmVkIjogMSwNCiAgICAgICJibHVlLT55ZWxsb3ciOiAyLA0KICAgICAgImdyZWVuLT5ibHVlIjogMSwNCiAgICAgICJncmVlbi0+cmVkIjogMiwNCiAgICAgICJncmVlbi0+eWVsbG93IjogMSwNCiAgICAgICJyZWQtPmJsdWUiOiAyLA0KICAgICAgInJlZC0+eWVsbG93IjogMiwNCiAgICAgICJ5ZWxsb3ctPmJsdWUiOiAyLA0KICAgICAgInllbGxvdy0+Z3JlZW4iOiAxLA0KICAgICAgInllbGxvdy0+cmVkIjogMQ0KICAgIH0sDQogICAgImRldmVsb3BtZW50X251aXNhbmNlIjogew0KICAgICAgImJsdWUtPnllbGxvdyI6IDQsDQogICAgICAiZ3JlZW4tPmJsdWUiOiAzLA0KICAgICAgImdyZWVuLT5yZWQiOiAxLA0KICAgICAgInJlZC0+Ymx1ZSI6IDMsDQogICAgICAicmVkLT5ncmVlbiI6IDEsDQogICAgICAieWVsbG93LT5ncmVlbiI6IDIsDQogICAgICAieWVsbG93LT5yZWQiOiAyDQogICAgfSwNCiAgICAiZXZhbHVhdGlvbl9jYXVzYWwiOiB7DQogICAgICAiYmx1ZS0+Z3JlZW4iOiAxMCwNCiAgICAgICJibHVlLT5yZWQiOiAxMSwNCiAgICAgICJibHVlLT55ZWxsb3ciOiAxMSwNCiAgICAgICJncmVlbi0+Ymx1ZSI6IDExLA0KICAgICAgImdyZWVuLT5yZWQiOiAxMSwNCiAgICAgICJncmVlbi0+eWVsbG93IjogMTAsDQogICAgICAicmVkLT5ibHVlIjogMTAsDQogICAgICAicmVkLT5ncmVlbiI6IDExLA0KICAgICAgInJlZC0+eWVsbG93IjogMTEsDQogICAgICAieWVsbG93LT5ibHVlIjogMTEsDQogICAgICAieWVsbG93LT5ncmVlbiI6IDExLA0KICAgICAgInllbGxvdy0+cmVkIjogMTANCiAgICB9LA0KICAgICJldmFsdWF0aW9uX251aXNhbmNlIjogew0KICAgICAgImJsdWUtPmdyZWVuIjogMTAsDQogICAgICAiYmx1ZS0+cmVkIjogMTEsDQogICAgICAiYmx1ZS0+eWVsbG93IjogMTEsDQogICAgICAiZ3JlZW4tPmJsdWUiOiAxMSwNCiAgICAgICJncmVlbi0+cmVkIjogMTEsDQogICAgICAiZ3JlZW4tPnllbGxvdyI6IDEwLA0KICAgICAgInJlZC0+Ymx1ZSI6IDEwLA0KICAgICAgInJlZC0+Z3JlZW4iOiAxMSwNCiAgICAgICJyZWQtPnllbGxvdyI6IDExLA0KICAgICAgInllbGxvdy0+Ymx1ZSI6IDExLA0KICAgICAgInllbGxvdy0+Z3JlZW4iOiAxMSwNCiAgICAgICJ5ZWxsb3ctPnJlZCI6IDEwDQogICAgfQ0KICB9LA0KICAiY2hlY2tzIjogIkV4YWN0IHJlbmRlcmVkIHBpeGVscywgc2luZ2xlIGNvbG9yIGNoYW5nZXMsIHF1ZXN0aW9uIHN3aXRjaGVzLCBsYWJlbHMsIHVuaXF1ZSBvcmlnaW5hbCBpbnB1dHMsIHNwbGl0IHNlcGFyYXRpb24gYW5kIGZhY3RvcmlhbCBiYWxhbmNlIHBhc3NlZC4iDQp9DQo=', 'sha256': 'e50ed26ad15b9c766dbede1fe020559bbbc64095c5d7f81a9ff8d0e69f28afd9'}, 'data/binding-v1/examples.jsonl': {'base64': '...', 'sha256': 'f4a20ab4aae2c0880f42f3d59f63d5f5b2cb4f4480ef6defb3490f0b1b1ef59d'}, 'data/binding-v1/images/binding-development-0000-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE+UlEQVR4nO3bMW4iSRiAUWY13IfAki8x8RzS8V5iJAccZgJLJBsgYQSL7WEwX7nrvciyoKuS/lT9t/3t5eX3CqDwT7IqgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZ7tzTcxmbzsFqo7fZ5tWhOQEBGgICMRzCW49/1erUIP3a71RycgICMAAEZj2D3eAuz+HcZcB0Buser3+MPixEcCNC9/+pk/3UZAgG6TXfWj09vf3f36+elqykRMxOgK+vzbnQuffgkRpvNgwYxLQH63PSc23/9OEMeypiW1/B/UJ/149Nf1ueNSy34H5rgEgG66LgIN0zPsZPLahCzEaAP1edT19IgpiVAcX3OV9Eg5iFApw73/yc9dl1yvJwGMQkBulifZAMaxFQE6NVo9/xo+4GbE6Bs7nOJeRDzEKBTbX3G2QPcgQANMfo5ZxjEDAToC9zh4+8QriNAr8Y5/oy5H7g5AQIyswdowOnPMZMglm32AAGhqQM0+PFnzyGIBZs6QEBLgICMAAGZeQP0JQZAe8ZALNW8AQJyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZOYN0Hb7vP9h9+vnamyHHR72DMswb4CAnAABGQECMlMH6EuMgQyAWLCpAwS0Zg/Q4Icgxx+WbfYAASEBejXaIWi0/cDNCdAXeLoZf4dwHQEadBJk+sMMBOjUCA0aYQ9wBwK0Oj9otPf/8eqOPyybAK2GvdtH2w/cnACNNQwy+mEqAvRWg+6ZoePl1IdJCNAQ8yBzH+YkQH2D1IdpCdBHG/QZGTq5rPowGwF6y0kRbpih80upDxP6Xm9gdPsubDYPh9/sw7F+fLrugucJkx6mJUAfst0+HzfopCPvxuiNc5P6MDMB+qhDKU5KdN2UWndAgG7zUHbF1wEBut5xR96NkejA/xKgG9AXuI7X8EBGgICMRzCW48duV2+BP+MEBGQECMh8e3n53a0OTM0JCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBV5T+aIwe8xeuGSQAAAABJRU5ErkJggg==', 'sha256': '64e291e3b6f6351aa5dd983a04424e533c8a262cb8d4ea0513cf69fce615553f'}, 'data/binding-v1/images/binding-development-0000-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE+UlEQVR4nO3bMW4iSRiAUWY13IfA0l5i4jnkxr7ESA44zASWSDZAwggW7GExX7nrvciyoKuS/lT9t/3t9fX3CqDwV7IqgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZ7tzTcx2bztFqo7fZltWhOQEBGgICMRzCWY/33P6tF2P36uZqDExCQESAg4xHsEW9hFv8uA24jQI949Xv8YTGCAwF69F+d7L8uQyBA9+nO83p9/bs/drtLV1MiZiZAN9bn3ehc+vBJjDabJw1iWgL0uek5t//6cYY8lDEtr+H/oD7P6/X/rM+VSy34H5rgEgG66LgId0zPsZPLahCzEaAP1edT19IgpiVAcX3OV9Eg5iFApw73/yc9dl1yvJwGMQkBulifZAMaxFQE6M1o9/xo+4G7E6Bs7nOJeRDzEKBTbX3G2QM8gAANMfo5ZxjEDAToC9zh4+8QbiNAb8Y5/oy5H7g7AQIyswdowOnPMZMglm32AAGhqQM0+PFnzyGIBZs6QEBLgICMAAGZeQP0JQZAe8ZALNW8AQJyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZOYN0Hb7sv/hx263Gtthh4c9wzLMGyAgJ0BARoCAzNQB+hJjIAMgFmzqAAGt2QM0+CHI8Ydlmz1AQEiA3ox2CBptP3B3AvQFnm7G3yHcRoAGnQSZ/jADATo1QoNG2AM8gACtzg8a7f1/vLrjD8smQKth7/bR9gN3J0BjDYOMfpiKAF1r0CMzdLyc+jAJARpiHmTuw5wEqG+Q+jAtAfpogz4jQyeXVR9mI0DXnBThjhk6v5T6MKHv9QZGt+/CZvN0+M0+HM/r9W0XPE+Y9DAtAfqQ7fbluEEnHXk3RlfOTerDzAToow6lOCnRbVNq3QEBus9D2Q1fBwTodscdeTdGogP/SYDuQF/gNl7DAxkBAjIewViO3a+f9Rb4M05AQEaAgMy319ff3erA1JyAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECFhV/gVggAe8rtHvRgAAAABJRU5ErkJggg==', 'sha256': '3aa27aeef8ab87457ef6b07cefa2aaccf801722ebbdb7d7eb9ee9b58c347a68e'}, 'data/binding-v1/images/binding-development-0000-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE70lEQVR4nO3cPU4bQRiAYSeK7+OCY+S4HIPCh0mBRJPCklnZ4SfW4nfYeZ4KIbw7zb6a/Qb48fz8ZwdQ+JncFUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzq7s1rONweNht1PH4tNs0OyAgI0BAxisY2/G43+824ffLy24OdkBARoCAjFewe5zCbP4sA24jQPc4+l3+sBjBmQDd+7dOTh+XIRCgdbrz4eHL9aHG+WpKxMwE6Mb6/NeJ7/KHL2J0ODxoENMSoK9Nz7XTx5cZ8lLGtBzD/0d9Hvf7tX7V7fpSG/6DJniLAL1pWYQV07N0cVkNYjYC9Kn6fOm9NIhpCVBcn+u7aBDzEKBL5+f/i1673rK8nQYxCQF6sz7JAjSIqQjQq9Ge+dHWA6sToGzu8xbzIOYhQCP+U6sR1gB3IEBDjH6uGQYxAwH6Bk/4+CuE2wjQq3G2P2OuB1YnQEBm9gANOP1ZMgli22YPEBCaOkCDb39ObILYsKkDBLQECMgIEJCZN0DfYgB0YgzEVs0bICAnQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAZt4AHY9Ppy9+v7zsxnZe4XnNsA3zBgjICRCQESAgM3WAvsUYyACIDZs6QEBr9gANvgmy/WHbZg8QEBKgV6NtgkZbD6xOgL7B2834K4TbCNCgkyDTH2YgQJdGaNAIa4A7EKDd9Uajff6Xd7f9YdsEaDfs0z7aemB1AjTWMMjoh6kI0HsNumeGlrdTHyYhQEPMg8x9mJMA9Q1SH6YlQJ9t0Fdk6OKy6sNsBOg9F0VYMUPXl1IfJvSrXsDoTl04HB7O3zmF43G/v+2C1wmTHqYlQJ9yPD4tG3TRkQ9j9M6+SX2YmQB91rkUFyW6bUqtOyBA67yU3fBxQIBut+zIhzESHfgnAVqBvsBtHMMDGQECMl7B2A7/yO3bsQMCMgIEZH48P//p7g5MzQ4IyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgF3lL7gbB+QdT+B+AAAAAElFTkSuQmCC', 'sha256': 'f3432a8744aea077af6defeb0401e1e379872fa23f8ab2c3c4638e51184fc5a4'}, 'data/binding-v1/images/binding-development-0001-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFCklEQVR4nO3dMW4bRxiAUTqw7+MigKtcw4fwgXKIXCNVABc8jAsBblIQIATRJEVK9Dcz+14tcgcS98O/o5X2w9PTjx1A4Y/kqAACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMh+7Q8PNPn/+c7eo/f77bntMQEBGgICMSzCm9OnLP7sl/Pzv627DTEBARoCAjAABGQECMgIEZAQIyPg1/EB38W7zXli2TIAG+tOB518sRmyBAA36J0vHd1AiFiZAvy89//798/Jr//r26dwbyhBLEqDHpudqdM598YsYyRBLEqBH1eem9Jw6vPw0QxrESgTo/evzxvRczpAGsRIBGjQ9FzLkcoxluBFx9Pqce/+F/zcg22ECutPz8//R6bkwCpmDmJoJaJr6/PKI5iCmJkCT1ef0uBrEvAToflV9Rjg6vAsBus1x3Bjh/D+uwRDEpAToBiOf5yOvDc4RoJm2fk7ZDGJqAnSzceoz5nrg9QToVWYZLmZZJxwI0ArjxpirgqsEaLWxYq7VsnECtMigMfLa4BwBAjICNNOdh5e5L5HpCBCQESAgI0BARoAW2QA6sA3EXAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARoEuOT14/Po59cMd1emY8UxAgICNAQEaAgIwArbMNZAOI6QgQkBGgG4w8BI28NjhHgK6b64pmrtWycQK0wqAx5qrgKgFaaqyYZZ1wIEDTjxujrQdeT4DuGS7GOeefr8T4w3QE6AYjn+Ejrw3OEaCJ70t05yGzE6D7tQ0aoYDwRgI05WaQrR/WIEDzNUh9WMbHegGz2u+/H5+/fijCb3h+/IvYqQ+zMwHd78X5/+hRSH1YjwnoTQ4VePQoJD2sSoDe+XLsfTN0OlWpDysRoIc06O0Z+uUFnfqwGAF61OXYaUeuxujCLpL0sKQPT08/6jUs6EWG3kJ6HvSNHc1+kz9oE9BjP0x3nzDb/DiyNQL0WM87cjVGosPWuAQDMm5EBDICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKv8D1cLCnHRdDFuAAAAAElFTkSuQmCC', 'sha256': '83c132bb378fd1cae000d4fe8dda0a2320a6aa4386231660945192bb1933ff85'}, 'data/binding-v1/images/binding-development-0001-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFGUlEQVR4nO3cMW7bSBiAUWUR38dFAFe5Rg6xB9pD5BpbBUjhw6Qw4GYLAYIgryRLlvxxhu9VLixyYJsf/qEof3l5+bMBKPyVnBVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZrd2q42OPjt82knp9/b9bHBARkBAjI2IIxpH//ed1M4fvfD5sVMwEBGQECMgIEZAQIyAgQkBEgIONt+AU9xbvOZ2FZMwFa0EcH9r9ZjFgDAVroR5Z2R1AiJiZAn5eeh6efp1/7+uvHsQPKEFMSoPum52x0jn3zQYxkiCl5F+xe9Xl4+nlRfd7z8on/GQXrJEA3cNCFD6bn9KE0iJnYgt04Pfc4y/awu02Z7RjTMAEtvT7Hjm8UYgICdKX96/+Ge67TDk6kQYxOgG5Qn08+uwYxDQEarD5vz6tBjEuArlfVZwlnh5sQoMvsxo0lXP+7NRiCGJQAXWDJ1/mS1wbHCNBIt37ecjOIoQnQxZZTn2WuB95PgN5llOFilHXClgDNMG4sc1VwlgDNNlaMtVpWToAmGTSWvDY4RoCAjACN9OThaZ5LZDgCBGQECMgIEJARoEluAG25DcRYBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBGgU56ff2+/eP31YzOC3Tp3K4clEyAgI0BARoCAjADNcxvIDSCGI0BARoAusOQhaMlrg2ME6LyxdjRjrZaVE6AZBo1lrgrOEqCpxopR1glbAjT8uLG09cD7CdA1w8Vyrvn9lRh/GI4AXWDJV/iS1wbHCNDAzyV68pDRCdD12gYtoYDwQQI05M0gt36YgwCN1yD1YRoCdJsGfU6GDk6kPoxOgK53cP3fu0EHx1cfJvC1XsDYthV4fPy234iHp5+3PYv0MCsT0F1GoVtNQ28PpT7MRIBu420XPpih/325+jAZW7B7bce29iNydmt2IljSw5S+vLz8qdcwoYMMfYT03OkHuzTPq/xFm4Du+8d09QWzzj9H1kaA7mu/I2djJDqsjS0YkPEuGJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgE3lP910CmX/de6tAAAAAElFTkSuQmCC', 'sha256': 'e8daf7271a51c778faf5a76a7f446a4124d4a47cc861697cd2d17b69774b3843'}, 'data/binding-v1/images/binding-development-0001-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEklEQVR4nO3cTU4bSxhAUecp7IdBJHaSRbKTSBmwmAyQMnkDS5ZlfhyDza2qPmfEALpLQF991W772/Pznx1A4b/krAACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMt+7U8PF7u9/7Bb19PR7tz0mICAjQEDGFowp3T087pbw99fP3YaZgICMAAEZAQIyAgRkBAjICBCQ8TL8QE/xbvNZWLZMgAZ668DxN4sRWyBAg75l6XAEJWJhAvR16Tn78O6rD8XuDyhDLEmAbpuei94xcPzNJzGSIZbkVbBb1efu4fEz71d69ccX/jAKtkmAruCkC59Mz/uH0iBWYgt25fTc4iz7wx42ZbZjLMMENHp93jq+UYgFCNAHHV//V9xzve/kRBrE7AToCvX54rNrEMsQoMnq8/K8GsS8BGjWTwVd5jNJ2TIBusxh3Bjh+j+swRDEpAToAiNf5yOvDd4iQDPd+nnJzSCmJkAXG6c+Y64H/p0A/ZNZhotZ1gl7ArTCuDHmquAsAVptrJhrtWycAC0yaIy8NniLAAEZAZrpycP3eS6R6QgQkBEgICNAQEaAFrkBtOc2EHMRICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoDe8/T0e//F318/dzM4rPOwchiZAAEZAQIyAgRkBGid20BuADEdAQIyAnSBkYegkdcGbxGg8+ba0cy1WjZOgFYYNMZcFZwlQEuNFbOsE/YEaPpxY7T1wL8ToI8MF+Nc88crMf4wHQG6wMhX+Mhrg7cI0MTPJXrykNkJ0Me1DRqhgPBJAjTlzSC3fliDAM3XIPVhGQJ0nQZ9TYZOTqQ+zE6APu7k+r91g06Orz4s4Hu9gLntK3B//+O4EXcPj9c9i/SwKhPQTUaha01DLw+lPqxEgK7jZRc+maFXf1x9WIwt2K22Y3vHETm7NXsnWNLDkr49P/+p17Cgkwx9hvTc6Bc7mqdN/qFNQLf9Z/rwBbPNf0e2RoBu67gjZ2MkOmyNLRiQ8SoYkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAXeV/vgcKZT1K6qoAAAAASUVORK5CYII=', 'sha256': 'b40ea59855d28e780c908e551e8648d2362bba9c3e12e33d6b78b07155c202b8'}, 'data/binding-v1/images/binding-development-0002-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHElEQVR4nO3dQU4bWRRAUWiF5bTEIBI76Y1kHdkIO2mpB0jZCgNLmfTAEjEYDDbG91f9c6YJrjegrl59jLnebB6vAAp/JVcFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMy37tJrcHv7/fB/eHj471KzwPJcbzaP9Qxri85bxAheEKAv784+JYItAfpUem7u7g9/7e9//3nrn2QIBOjo9LwbnaNiJEPMTICOqM/J6TmcIQ1iWgL0ofqcJT0HMqRBzEmALp2eXTLE5LwRMavP/uuf8QdtsAg2oNfv/69Oz4FVyB7EPGxAfX1eXNEexDwEqK/P/nU1iEkI0DNVfUa4OlyeM6A/68Yg9//TeZDDoNOscn98WOk3w+wb0MjfrCPPBmcxdYBGOPrZ5zCIefg8oLHqs3Vzd3/gt1j5oN8//r5auJufv65Wbd4NaCnLxVLmhBPMG6Bh15+Rp4LzmjRAy1orljUtfNykAVrEojHybHAWswcICM0YoNHeeXjA04SewlilGQMEDEKAgIwAAZnpArSgA6Atx0Cs2HQBAsYhQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAZroAPf19paV86rs/E8aKTRcgYBwCBGQECMjMGKAFHQM5AGLdZgwQMIjZAzTyEjTybHAWkwZoWU80y5oWPm7SAI2/aIw5FZzXvAFaylqxlDnhBPMGaOR1Y7R54ItMHaDd5WKce353EusP6zZ1gAa/w0eeDc5i9gCN9r5E7zxkKgL0TNugEQoIlyRAoxwGOfphQgI0RIPUhzkJ0JsNukyGXlxIfZiKAD3z4v7/6ga9eH31YTbf6gGGs63A7e333Ubc3N2f9yrSAzagI1ahc21D+y+lPkxLgN6034VPZujVL1cfZuYR7IjHsa3diLz7aHYgWNIDAnRihrZOW4ikB7YE6OhqvFqio14B2BKgT3Xk3RiJDhwgQJ+iL/AZfgoGZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkfCIiq3Xz81c9Au+wAQEZAQIy15vNY3d1YGo2ICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEXFX+B0h4KwRfMtXJAAAAAElFTkSuQmCC', 'sha256': '4955afa4c5592196e8223c6866a24372f4d7eec3e1f9c57228c519a1d7bbc994'}, 'data/binding-v1/images/binding-development-0002-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3dPW4bVxSAUSmwl2PARYDsxBvxOrKR7MSACwLZigoCalIQUGRRpvj/vZl3TmtZcwvxw52nEfm43T49ABT+SK4KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZlP3aXX4OvXPw9/wWbz816zwPI8brdP9Qxri87viBG8IUA3784+JYIdAbooPc/fvxz+v5///vd3/yRDIEAnp+fD6JwUIxliZgJ0Qn3OTs/hDGkQ0xKgo+pzlfQcyJAGMScBund6XpMhJudBxKw++9//ir9og0WwAb3/+r91eg6sQvYg5mED6uvz5or2IOYhQH199q+rQUxCgH5R1WeEq8P9OQP6f90Y5PX/ch7kMOg8q9wfNyv9YZh9Axr5h3Xk2eAqpg7QCEc/+xwGMQ/vBzRWfXaev3858FesHOnzX/88LNzzj28PqzbvBrSU5WIpc8IZ5g3QsOvPyFPBdU0aoGWtFcuaFo43aYAWsWiMPBtcxewBAkIzBmi0Jw8PeJnQXRirNGOAgEEIEJARICAzXYAWdAC04xiIFZsuQMA4BAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyEwXoJfPV1rKu777mDBWbLoAAeMQICAjQEBmxgAt6BjIARDrNmOAgEHMHqCRl6CRZ4OrmDRAy7qjWda0cLxJAzT+ojHmVHBd8wZoKWvFUuaEM8wboJHXjdHmgRuZOkCvl4txXvOvJ7H+sG5TB2jwV/jIs8FVzB6g0Z5L9OQhUxGgX7QNGqGAcE8CNMphkKMfJiRAQzRIfZjTp3qAgWw2P18+f31XhDt8fvyb2KkPU7EBPRx4/d96FVIfJmcDemtXgVuvQtIDAnTU7dh1M7S/VakP0xKgYxt0eYbevaFTH2YmQCfcju135MMYHThFkh4QoDMzdMkptfTAjgCdXI13S3TSdwB2BOiijnwYI9GBAwToIvoCl/AgIpARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPGOiKzW849v9Qh8wAYEZAQIyDxut0/d1YGp2YCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEPBQ+Q+VEB003w1UQAAAAABJRU5ErkJggg==', 'sha256': 'ec1103988a54be5628ff86c66876a107a463d8ec6adca13200d8a133d1372e08'}, 'data/binding-v1/images/binding-development-0002-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFC0lEQVR4nO3dS27bSBRAUbuRLCdABr2zrKN3loGA3ooHAjLpgQC3Yzm0/rfIOmfqDwuweP1YpsXn/f7lCaDwV3JUAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh86Q69Bd+//738Cbvdz0etBdbneb9/qdewtej8iRjBOwJ09+4cUyI4EKCr0vPrx7flr/36z79/+pAMgQCdnZ5Po3NWjGSImQnQGfW5OD3LGdIgpiVAJ9XnJulZyJAGMScBenR63pIhJudGxKw+x9//hn9og1UwAX18/t87PQujkDmIeZiA+vq8O6I5iHkIUF+f4+NqEJMQoN9U9Rnh6PB49oD+HzcGOf9f94NsBl1mk/PjbqMvhtknoJFfrCOvDW5i6gCNsPVzzGYQ8/B+QGPV5+DXj28L/8XKSn+sF/i69ZfBvBPQWoaLtawTLjBvgAb/PTnmquC2Jg3QusaKda0WTjdpgFYxaIy8NriJ2QMEhGYM0Gh3Hi54XaGrMDZpxgABgxAgICNAQGa6AK1oA+jANhAbNl2AgHEIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQmS5Ar89XWsvbfXtMGBs2XYCAcQgQkBEgIDNjgFa0DWQDiG2bMUDAIGYP0MhD0Mhrg5uYNEDruqJZ12rhdJMGaPxBY8xVwW3NG6C1jBVrWSdcYN4AjTxujLYeuJOpA/R2uBjnnH+7EuMP2zZ1gAY/w0deG9zE7AEa7b5Edx4yFQH6TdugEQoIjyRAo2wG2fphQgI0RIPUhzl9qRcwkN3u5+vz1w9FeMDz49/FTn2YignoaeH8v/copD5MzgT03qEC9x6FpAcE6KTLsdtm6HiqUh+mJUCnNuj6DH14Qac+zEyAzrgcO+7IpzFa2EWSHhCgCzN0zS619MCBAJ1djQ9LdNZ3AA4E6KqOfBoj0YEFAnQVfYFruBERyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIeEdENuveD7bleiYgICNAQOZ5v3/pjg5MzQQEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICnyn97LBx/E4z3PAAAAABJRU5ErkJggg==', 'sha256': 'c2e4f78e6d08dd55d9c521577815818725cf6d6fd11412c09fedb7226dca2efd'}, 'data/binding-v1/images/binding-development-0003-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3aQW4aSRiAUWcU7uNFtrlBTpIDzUl8g9lm4cNkYcmbWSAhhG0MpuHrKt5bZQFdJYv+8nfBt5eXvw8AhX+SVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg871b+o48Pv44/oLn5z+32su9/6nX7/mePgwCtIo7Yf/Fd/X5484J0Or+B95dQYmYngDdIj1Pm83x9/56ff3ogjL0Nf/9+86fdM1+/v7kQzIlAbpWej6NzkcvPoiRDDEx34ItX5+nzeas+pzy9gnOVuEtAbrIQRcuTM/xS2kQ8/EItlh6rrHK9rK7hzKPY0zGBLTe+nx0faMQ0xCgs+3f/ws+cx13sJAGMQcBuqg+N15dg5iMAA1Tn7frahCjE6CvqOqzhtVhQQJ0qt24sYb7f7cHQxBDE6CTrPk+X/Pe4DgBGuPo5y2HQUxAgM6wnvqscz9wLgH6xCjDxSj7hH0CNPa4sc5dwYkEaJ6xYqzdggDNMGiseW9wnAABGQEa45eHx/ldIoMSICAjQEBGgICMAA1/ALTlGIgRCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNA73t+/rP9x6/X14cR7Pa52zmsnwABGQECMgIEZARohmMgB0AMSoCAjACdZM1D0Jr3BscJ0DFjPdGMtVsQoOEHjXXuCk4kQJOMFaPsE/YJ0MDjxtr2A+cSoPOGi/Xc8/s7Mf4wKAE6yZrv8DXvDY4ToCF/l+iXh8xBgL6ibdAaCgiLEKDBDoMc/TATARqpQerDZATo0gbdJkMHC6kPcxCgrzi4/6/doIPrqw/T+F5vYFTbCjw+/thvxNNms+wq0sPcTEALj0JLTUNvL6U+zEeALvW2Cxdm6N23qw9T8gi2/OPY1n5EPn00OxIs6WFiAnTdDG19bSCSHqYnQAvbVePdEp11BZieAF3Lfkc+jZHoLO7n74W/keQaBOgW9AXe5VswIPPt5eVvtzpw10xAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMBD5X8Y8AxI6Ad3RQAAAABJRU5ErkJggg==', 'sha256': '3a951f518848829615cfe18ffb11be391a8b2af051e927d23afd1fb83c714911'}, 'data/binding-v1/images/binding-development-0003-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFCUlEQVR4nO3dMW4bRxiAUTuw7+MibW7gk+RAOUlukNYFD+NCgJoUAghCpJakROqbGb5XU7sDgvvp38EC+/Xp6fcXgMIfyVkBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPOtO/UD+fHjz+0P7Ha/Pmstj/5Vj2/3SD8GARriSjj88EP9/nhwAjTcf+D9EZSI5QnQZ6Tnv3+et//2r7+/v3VAGXqff7+f+EpH9vP5zI9kSQJ0r/Scjc5bH34VIxliYQJ0+/pclZ5jL39+nCENYj0CdMv6fDA92xnSINYjQMOlZyNDbsdYjAcRx63PW8df4FEXeGECutrh9X/v9GyMQuYgFmACmqA+J89oDmIBAjRNfY7Pq0HMToDeo6rPCGeHGxKgS+3HjRGu//0aDEFMTYAuMvJ1PvLaYJsAzbH1c8xmEAsQoCuMU58x1wPXEqAzZhkuZlknHBKguceNMVcFFxKgdcaKuVYLArTCoDHy2mCbAAEZAZrjycNtnktkUgIEZAQIyAgQkBGg6TeAXtgGYkYCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICNBp+zev71/HPrj9Or0znokIEJARICAjQEBGgFbYBrIBxKQECMgI0EVGHoJGXhtsE6Atc93RzLVaEKDpB40xVwUXEqBFxopZ1gmHBGjicWO09cC1BOi64WKca/5wJcYfJiVAFxn5Ch95bbBNgKZ8LtGTh6xBgN6jbdAIBYSbEKDJNoNs/bASAZqpQerDYr7VC5jPbvdr//71lyJ8wvvjX8VOfViDCeg9Xl3/9x6F1IdVmYDe6aUC9x6FpIe1CdDNbsdum6HjqUp9WI8A3bhBH8/QyRs69WFJAnT727HjjpyN0cYukvSwMAG6b4Y+skstPSxPgG5sX42TJbrqCLA8AbqXw46cjZHo3NzP57s/nMXHCdBn0Bc4yYOIQObr09Pv7uzAQzMBARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAFfKv8DIpoMVMstOPYAAAAASUVORK5CYII=', 'sha256': 'c25cdd883df49441c941252f240ec4524b7d1a520e690cfa6cb1a83ac2142a8d'}, 'data/binding-v1/images/binding-development-0003-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFAElEQVR4nO3dQW4aSRiA0WSU3CeL2eZCc6C5ULZZcJgsLM1mFkgIGYzBbviqivfWmC5Z9Mff5bb668vLny8Ahb+SowIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5lt36Cfy48ffl1+w2/1+1Fqe/Vc9vt0zfRgEaIgz4fjFT/X548kJ0HDfwId3UCKWJ0CPSM+vf/+7/LM///n+1hvK0Me8+zsfzc9zn4HlCdC90nPTCXD84lcfRBliYQK0fX0++d27//HTDGkQ6xGgLeuz4dh/miENYj0CNFx6LmTI5RiLcSPiuPV56/0XuNUF9kxANzs+/x/2p5bTUcgcxAJMQBPU5+wRzUEsQICmqc/pcTWI2QnQfDe5TXeLHbxFgK51GDdGOP8PazAEMTUBusrI5/nIa4PLBGiOrZ9TNoNYgADdYJz6jLkeuJUAvWOW4WKWdcIxAZp73BhzVXAlAVpnrJhrtSBAKwwaI68NLhMgICNAc9x5eJn7EpmUAAEZAQIyAgRkBGj6DaA920DMSICAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBOu/w5PXD49gHd1inZ8YzEQECMgIEZAQIyAjQCttANoCYlAABGQG6yshD0Mhrg8sE6JK5rmjmWi0I0PSDxpirgisJ0CJjxSzrhGMCNPG4Mdp64FYCdNtwMc45f7wS4w+TEqCrjHyGj7w2uEyAprwv0Z2HrEGAPqJt0AgFhE0I0GSbQbZ+WIkAzdQg9WEx3+oFzGe3+314/vq+CA94fvyr2KkPazABfcSr8//eo5D6sCoT0AftK3DvUUh6WJsAbXY5tm2GTqcq9WE9ArRxgz6fobMXdOrDkgRo+8ux0468G6MLu0jSw8IE6L4Z+swutfSwPAHa2KEaZ0t00zvA8gToXo478m6MRGdz/mFlCgL0CPoCZ7kREch8fXn50x0deGomICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgS+V/AmAKvumGugsAAAAASUVORK5CYII=', 'sha256': 'f5469e2201f5c468f5fa4de4d3c371269789a8f83323de24131d47918ca4eeb3'}, 'data/binding-v1/images/binding-development-0004-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUElEQVR4nO3dMU5bWRSAYRgl+3GBNNVsI4uYBbGI2QZVpBQshgKJZgokywJhjI39v/P8fW1IfFDkP+fdG+Tb5+enG4DCX8mrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjI/OheGpZis7nb/wWPj38uNct1uX1+fqpngMUVZz89+i4CxHU5MT27ZOh0AsRV2N+dh/uX/b/9n39/7vlVJTqaAHGl6fk0Ol+NkQwdQYC4rvoc3Z1DSqRBXyVArNNZ07NLhk4hQKy/PmdKz54MadCBBIg11+cC6fkoQxp0CP8TmlUJ6/PmFb/xvn/FVrgBreAv3j+eE+uzZQ86nB/FYG31CdOzO8BrhjabOw3awyMYa7Cc+mxtJ1nBSn4+a96Afv79380oL79/1SOMtMD6vHq4f7EH7WcDYrYp+8WUOS9MgBhsIafOe7gX20+AWINl1mf5s+UEiKkWe/TzngPpjwgQI819J8+d/BwEiNmWv/7MmvPCBIh5pi8R0+f/RgLEYLPWilnTXoYAARkBYphBl1/vuQ57Q4CAjAAxyZoWh82KvpejCRAjTXz+mj75OQgQkBEgICNAQEaAGGP0Bfwul/FbAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQY2w/Xv31A9fn2s7/ePUfGC9AQEaAgIwAARkBYqS5x0BzJz8HAWKSNZ3aPq7oezmaAAEZAWKY0ZfxLuDfECAgI0AMNmsJmjXtZQgQ80x/fpk+/zcSIGabslZMmfPCBIiR5i4Rcyc/BwFiqkHXYS6/PiJArMGSG7Tk2XICxGC7C8Uy3+e7U1l/3hMgZpvyrp4y54UJEOMt9jDI0c+nBIg1WGCD1OcQPw76Kli8x8c/m83d9p3/cP9STeLc53A2INZjCWfS6vMlAsSqtA1Sn6/yCMZqn8Uu+Tj2JnbqcyABYuUNOneG3u9Z6nM4AWKdXitw1gxJz+kEiCtahXarcXSJPjpaUp8jCBBXtwq978inMdp/ni09RxMgrsK2Ee9LdPR9me6cToC4Lh8tREf8IZxOgLhGbwryaY8U50wECPQls+YAvfz+VY8A7ONHMYCMAAGZ2+fnp+7VgatmAwIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBNxU/gfaOz0jjx8BVwAAAABJRU5ErkJggg==', 'sha256': '1700b7a9c8c26cf2eee1906f6a2711c204a168fe405a2059ba22ca3ea0d7407d'}, 'data/binding-v1/images/binding-development-0004-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFWElEQVR4nO3dMU4jWRRAURi19+OAtHcwK+kFzUrYwaQdsBgCpE4mQEKWPRhjsG+98jlhN1APobr8+t/I9y8vz3cAhb+SqwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyPzoLg1Lsd0+HP+Ap6ff15rltty/vDzXM8DiinOcHn0XAeK2fDE9u2To6wSIm3C8O4+bzfFP//vPnyP/q0RnEyBuND0fRuezMZKhMwgQt1Wfs7tzSok06LMEiHW6aHp2ydBXCBDrr8+F0nMkQxp0IgFizfW5Qnrey5AGncIroVmVsD57V/zG8/4VW+EKaAU/eL88J9bnjXXQ6ayAWFt9HjebsD57A6zg1+FFCRBrq8/dMmjQrf8x6r//HHv16gL9/LWUm2eWBdbn1eNm8/o4tt0+eBb7X1ZAzDZlfTFlzisTIAZbyK7zEc7FjhMg1mCZ9Vn+bDkBYqrFbv0csiH9HgFipLl38tzJL0GAmG35y59Zc16ZADHP9EXE9Pm/kQAx2Kxlxaxpr0OAgIwAMcygw69DjsP2CBCQESAmWdPCYbui7+VsAsRIE5+/pk9+CQIEZAQIyAgQkBEgxhh9AL/LYfwbAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZASIMd7eXv31Ddfnepv/6ebfMF6AgIwAARkBAjICxEhzt4HmTn4JAsQka9q1fVrR93I2AQIyAsQwow/jHcDvESAgI0AMNmsRNGva6xAg5pn+/DJ9/m8kQMw2ZVkxZc4rEyBGmruImDv5JQgQUw06DnP49R4BYg2W3KAlz5YTIAbbXVAs8z7fncry55AAMduUu3rKnFcmQIy32M0gWz8fEiDWYIENUp9TCBArbFCbod0B1Oc4AWI9lrAnbdf5UwSIVWkbpD6fJUCsvEHXydDehdTnRALECu3d/xfN0OEXV5/T/fjEx8IcrxXYbh/e/uU1E4+bzXdd4jBq0vNZAsSaPT393m3QbjXOLtF7iyn1OYMAcXNLocOOfBij409w0nM2AeImvDXisERnn5fpztcJELflvQXRGV+ErxMgbtFeQT7skeJciACBvmTWHKCfv77twBW4BC9EBDICBGTuX16eu6sDN80KCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQcFf5D5byUFdXE6j/AAAAAElFTkSuQmCC', 'sha256': '8268926a1448f82d685f02f417e9514247d973db163a01f446c6dddada71c0fa'}, 'data/binding-v1/images/binding-development-0004-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVElEQVR4nO3dQU4bWRRAUWjF+/EgUm+ix1lkj9lEpAxYDAMkJj1AQpbdGGOwb73yOcOEUA9FvryqD/L98/PTHUDhr+SqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyP7pLw1Jstz+Pf8Dj459rzXJb7p+fn+oZYHHFOU6PvosAcVu+mJ5dMvR1AsRNON6dh83m+D//5+XlyN8q0dkEiBtNz4fR+WyMZOgMAsRt1efs7pxSIg36LAFinS6anl0y9BUCxPrrc6H0HMmQBp1IgFhzfa6QnvcypEGn8JPQrEpYn70rfuN5/4qtcANawX+8b54T6/PGHnQ6GxBrq8/DZhPWZ2+AFXw7vCgBYm31uVsGDbr1X0bd/P3v3Sgvv3/VI4y0wPq8ethsXm/Httuf7sX+lw2I2absF1PmvDIBYrCFPHU+wrnYcQLEGiyzPsufLSdATLXYRz+HPJB+jwAx0txX8tzJL0GAmG3568+sOa9MgJhn+hIxff5vJEAMNmutmDXtdQgQkBEghhl0+HXIcdgeAQIyAsQka1octiv6Ws4mQIw08f5r+uSXIEBARoCAjAABGQFijNEH8Lscxr8RICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoAY4+3t1V/fcH2ut/kfb/4N4wUIyAgQkBEgICNAjDT3MdDcyS9BgJhkTU9tH1f0tZxNgICMADHM6MN4B/B7BAjICBCDzVqCZk17HQLEPNPvX6bP/40EiNmmrBVT5rwyAWKkuUvE3MkvQYCYatBxmMOv9wgQa7DkBi15tpwAMdjuQrHM1/nuVNafQwLEbFNe1VPmvDIBYrzFPgzy6OdDAsQaLLBB6nMKAWKFDWoztDuA+hwnQKzHEp5Je+r8KQLEqrQNUp/PEiBW3qDrZGjvQupzIgFihfZe/xfN0OEnV5/T/fjEx8IcrxXYbn++/clrJh42m++6xGHUpOezBIg1e3z8s9ug3WqcXaL3lin1OYMAcXOr0GFHPozR8Ts46TmbAHET3hpxWKKzz8t05+sEiNvy3kJ0xifh6wSIW7RXkA97pDgXIkCgL5k1B+jl9696BOAYP4gIZAQIyNw/Pz91Vwdumg0IyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBBwV/kPyhRRxf+001IAAAAASUVORK5CYII=', 'sha256': '5383a1399fb7255c391247b5f62ea9e510f6c2644d08bac6fe70025b339af506'}, 'data/binding-v1/images/binding-development-0005-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE/klEQVR4nO3dQW4aSRiAUTKKjzMSi9xszjE3ywJprpKFpWyiUWtQDxCwsc1X7npvaRl3bfpT1U8bvjw//9gBFP5IrgogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDm625i+/233YYcDt/rJcDr2AEBGQECMlMfwY5+/vXn7tN6+vufeglwJzsgICNAQMYRDN76bqn3H+8mQPDWRzTWvyxGryJA8J5Phy0vl6EXEiC40Z2bb5KevxF5/GtKdJ0AwYX6vOrJjPUvn8Rov/+mQVcIEFN7Y3rOLS9fZ8ih7AoBYl4n9XnH51EvZkiDzgkQk1rX54MehT/JkAad8yAiM3pAfS7+/Y19AMPbCRDTeWR9zq+iQWuOYMzleP8/+D+Q18cxZ7EjOyAmUtXn6Hhd+6CFADGL0e75/WDrSQgQU3j83Od3zIPWBIi5jPDhcyOsYRACxPblo59zhkELAWLjxr/D98Ov8OMIELMYZ/sz5noSAgRkBIgtG3D6s/Zz+kmQAAEZAWKzBt/+LCbfBAkQkBEgICNAQEaA2KZPMQBazDwGEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaA2KbjV48uX0Y6sqf/Vjjh16UKEJARICAjQEBGgNisTzEGepp4ACRAQEmA2LLBN0FPc29/BAgoCRCzGG0T9DTYehICxMaNf7o5DL/CjyNAbN+AkyDTn4UAMZcRGjTCGgYhQExhvdFo7//11Q9zb38EiImMdrcfBltPQoCYSD4MMvo58fX0B7Bph8P35QtIlxY87HtTnbwusgNiOo+fB6nP7wgQM3pkg9TnCkcwZj+Lfdxx7CRt6nNOgJjXukHvm6HzXZX6XCRATG3pwjtmSHpeRYDgdCt00pGbMboyRVKf6wQI/leKkxLdN6XWnRcSILhxKLvj5byQAP3LPwdypSM3YyQ6dxMguEFfPo4HEYHMl+fnH93VganZAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwK7yCzG6BjQx5r0mAAAAAElFTkSuQmCC', 'sha256': '39875380b6d9030f7df79ddac2a3c5ef7fb564760fd3107d9ae94b9739b5daa7'}, 'data/binding-v1/images/binding-development-0005-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFBklEQVR4nO3cQW4aSRiAUWcU34eFJd8kh/RNImXBYbKwxGYUoUE9YIODjb9y13srC0F3bfpT9d/gb8/Pv+8ACv8kZwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDz/W5im83D3Ypst7/qJcDfsQMCMgIEZKa+BTu4f3y6+7J2P3/US4Ar2QEBGQECMm7B4L1PSz1/vJoAwXu/orF8sxj9FQGCj/x22P7jMvRGAgQXunPxIenpg8jD0ZToPAGCF+rzV9/MWL75KEabzYMGnSFATO2d6Tm1//gyQ27KzvAYnnkd1ef+8emjvpJ6eqiV/fDwowgQk1oW4QPTs3R0WA06JUDM6Kg+Nz2XBp0hQEznM+tzehYNWhIg5nK4/m902/Wa5ek06ECAmLQ+yQI06IgAMYvRrvnNYOtJCBBT+Py5z2vMg5YEiLmM8M/nRljDIASI9ctHP6cMg/YEiJUb/wrfDL/C2xEgZjHO9mfM9SQECMgIEGs24PRn6X76SZAAARkBYrUG3/7sTb4JEiAgI0BARoCAjACxTl9iALQ38xhIgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQFinbbbX/s/dj9/3I1t998KD2uehwABGQECMgIEZASI1foSY6DdxAMgAQJKAsSaDb4J2s29/REgoCRAzGK0TdBusPUkBIiVG//uZjv8Cm9HgFi/ASdBpj97AsRcRmjQCGsYhAAxheVGo73+l2ffzr39ESAmMtrVvh1sPQkBYiL5MMjo54gAMW+DPjNDy9Opz4EAMZ3PnweZ+7xGgJjRZzZIfc4QICZ11KBbZOjosOpzSoCY11ERPjBDp4dSnxd9f/llmMO+C5vNw+GVfTjuH5+uO+BpwqTnDAGCP41YNuioIxdjdGbfpD7nCRD8rxRHJbpuSq07byRAcOGm7IqP80YC9IcfB3KmIxdjJDpXEyC4QF9ux2N4IPPt+fl3d3ZganZAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBBwV/kXl7sHks7VBwIAAAAASUVORK5CYII=', 'sha256': '37cf13a366cb77eb54825d82b3d7f471b6b2813557f0f61307410e445f40439a'}, 'data/binding-v1/images/binding-development-0005-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFD0lEQVR4nO3bQWrbWhiAUfdRL+eBB4HspBt56+hGspNCB4ZupYOAJ49ialQ7sVMnzqfonjMKxpLuRB9Xv5xPj48/VwCFf5KrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPm8Gthmc7dakO32e70E+Dt2QEBGgIDM0I9gB7v//l19WOuvP+olwJXsgICMAAEZj2Dw2rel3j9eTYDgtT/RmH5ZjP6KAMFb/jpsf7gMvZAAwYXurO8fzh+7+/blubMp0XkCBE/U52J0nvvyUYw2mzsNOkOAGNor03Nqf/g0Qx7KzvAannEd1Wd9//DK+pw51cL+8fCtCBCDmhbhDdMzdXRaDTolQIzoqD43vZYGnSFADOc963N6FQ2aEiDGcrj/b/TY9Zzp5TToQIAYtD7JAjToiAAxirnd85uZrSchQAzh/ec+zzEPmhIgxtLWZz5rmAkBYvny0c8pw6A9AWLh5n+Hb2a/wtsRIEYxn+3PPNeTECAgI0As2QynP1Pr4SdBAgRkBIjFmvn2Z2/wTZAAARkBAjICBGQEiGX6EAOgvZHHQAIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEMu03X7f/7H79mU1b7vfKzyseRwCBGQECMgIEJARIBbrQ4yBdgMPgAQIKAkQSzbzTdBu7O2PAAElAWIUc9sE7Wa2noQAsXDzf7rZzn6FtyNALN8MJ0GmP3sCxFjm0KA5rGEmBIghTDca7f0/vfp27O2PADGQud3t25mtJyFADCQfBhn9HBEgxm3Qe2Zoejn1ORAghvP+8yBzn+cIECN6zwapzxkCxKCOGnSLDB2dVn1OCRDjOirCG2bo9FTq86TPT38MY9h3YbO5O3yyD8f6/uG6E54mTHrOECD41Yhpg446cjFGZ/ZN6nOeAMEfpTgq0XVTat15IQGCCw9lVxzOCwnQL+uvP+olMC/TjlyMkehcTYDgAn25Ha/hgcynx8ef3dWBodkBARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DAqvI//RYG3fsL54oAAAAASUVORK5CYII=', 'sha256': '19618fe2b3720df14cab9162e329d452cf40e04c3b5fcf11fefb43a78d0c9aea'}, 'data/binding-v1/images/binding-development-0006-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFSUlEQVR4nO3aO27kRhRAUY4x2k8Hk3oHXIkX5JVoB04VcDETCFDioDFCT0M/W59LVp0TS2QRIC9eFfrb/f3PBaDwR3JXAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg830Zwun0Yxnatt3VS4CPZwICMoNMQLDn0dsAO0uA/vn7YRnIn3/d1EvgA/b7l/8iRiMHCHZ+zvh4nU2JBAjemZ7bm1em1PXh4YXLbnNnSIDgv6Xn1eK8/PdXPTrNnSEBgs9Kz5POF5GhMwGCV+rzId258njN9aJEp9OP2RokQPCl6Xl5IDpNNgr5ISJk9XnuXsP/sv+RAMH1N397c/OV9XnyppM0SIDguj7hSm4na5AAMbv91GfCBgkQU9tbfWZrkAAxr33WZ6oGCRCT2nN95mmQADGj/ddnkgYJEFPbc32OssL3ECCm8zhKHOXbvv21zvGGIAFiLkf/hk8HX/8VAWJSRxl/jrjatxMgJnK4zdfwGzEBAjICxCwOPf6MOgQJEJARIKYwzMgw2BMJEHM57v5rjPVfESAgI0CMb4Dj51GPogUIyAgQkBEgICNADG6Ag5KBn06AmMUYJ9CDPYsAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAsQs1oeHZRTrKM8iQAxu2+6WcW0HfzoBAjICBGQECMgIEON7PCgZ4+x2/fUURz8AEiCgJEDM5ehD0Hrw9V8RIKYwwG5lyCcSICAjQMxigKPodaDj5zMBAjICxEQOPQStw40/AsS8jtWg9VCrfTsBYi5HHx+2g6//igAxncNtxNYRN19nAsTU9t+gdfcrfA8BYkaXo8Sev/D1Ym3jjT8CxLz236B19PoIEFPbc4PWCeojQMxunw1a56iPAMHuGrROUx8Bgn01aJ2pPgIETzfo6zO0/n7TGeojQLA8981/ZYPW3+81SX2WZfleLwB25Pzln04/Lrtwe3PzeXdcZ03PmQDBtW27e2zQZSM+sERPjlfbZPVZluXb/f3P5fguX5chTfhq7va9emeGpOeSCQjeuiN7siCv9ujls6Rt1vScCRD8nwy9/6x6mzs9Q23B4Cj7fd25JEDw6TESnecIEJDxQ0QgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgKXyL5xvO1XEQmggAAAAAElFTkSuQmCC', 'sha256': '8342f027ffaba4b514a552b4b0b9d38bdd3b8c6edf4e63f9561651bb7b052e0a'}, 'data/binding-v1/images/binding-development-0006-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRUlEQVR4nO3aQW7bRhiAUaYIjxNACx+jt8k5cpsewwsCuUoWBrzpQoihqrac1o4/cua9tU3NAOKHfwb69PDwYwEo/JF8KoAAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPm8DOF0uluGtm339RLg/ZmAgMwgExDsefQ2wM4SoL/WdRnIn4+P9RJ4h/P+5b+I0cgBgp3fMz49Z1MiAYI3pufx65fb/75++37jsdvcGRIg+G/pebU4t//+qkenuTMkQPC70vOs80Nk6EyA4JX6vEt3rjw9c70o0el0N1uDBAg+ND23B6LTZKOQHyJCVp+XPmv4X/Y/MQHB9Tv/kem5MQrNMAeZgKCvz7OfPsMcJEDMbj/1mbBBAsTU9laf2RokQMxrn/WZqkECxKT2XJ95GiRAzGj/9ZmkQQLE1PZcn6Os8C0EiOk8jRJHebcff65zvCFIgJjL0d/h08HXf0WAmNRRxp8jrvbXCRATOdzha/iDmAABGQFiFocef0YdggQIyAgQUxhmZBhsRwLEXI57/hpj/VcECMgIEOMb4Pp51KtoAQIyAgRkBAjICBCDG+CiZODdCRCzGOMGerC9CBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAzGL99n0ZxTrKXgSIwW3b/TKu7eC7EyAgI0BARoCAjAAxvqeLkjHubtefuzj6BZAAASUBYi5HH4LWg6//igAxhQFOK0PuSICAjAAxiwGuoteBrp/PBAjICBATOfQQtA43/ggQ8zpWg9ZDrfbXCRBzOfr4sB18/VcEiOkc7iC2jnj4OhMgprb/Bq27X+FbCBAzuhwl9vyGrxdrG2/8ESDmtf8GraPXR4CY2p4btE5QHwFidvts0DpHfQQIdtegdZr6CBDsq0HrTPVZluVzvQDYhW27P53uLivw+PXLRy5g/Wf4ZqiPCQhefOc/chRap6yPCQiWf7/5HzkKrbOm50yA4NZx7LIR71iiZ8erbbL6LMvy6eHhx3J8l1+XIU341dzt9+qNGZKeSyYg+NUT2bMFebVHt++StlnTcyZA8H8y9Pa76m3u9Ax1BIOjnPd155IAwW+Pkei8RICAjB8iAhkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgQslb8BQpM7WXILqs0AAAAASUVORK5CYII=', 'sha256': '014d9b9c1e8603663f583a5774192386df37519c52af9e7350fd2579de7e7f51'}, 'data/binding-v1/images/binding-development-0006-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFR0lEQVR4nO3aMW7cRhiAUTowj2NgC7W5UFqfw20ulFYFAV/FhQA1KQQLm420UiJZHznzXi1xZ4Dlh38G++nu7scCUPgt+VQAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzeRnC6XSzDG3bbuslwPszAQGZQSYg2PPobYCdJUB//Xm/DOT3P9Z6CbzDef/8X8Ro5ADBzu8ZH5+zKZEAwRvTc//1y/V/X799v/LYbe4MCRD8t/S8WJzrf3/Ro9PcGRIg+FXpedLDQ2TogQDBC/V5l+5ceHzmelai0+lmtgYJEHxoeq4PRKfJRiE/RISsPs991vC/7H9kAoLLd/4j03NlFJphDjIBQV+fJz99hjlIgJjdfuozYYMEiKntrT6zNUiAmNc+6zNVgwSISe25PvM0SICY0f7rM0mDBIip7bk+R1nhWwgQ03kcJY7ybt//XOd4Q5AAMZejv8Ong6//ggAxqaOMP0dc7esJEBM53OFr+IOYAAEZAWIWhx5/Rh2CBAjICBBTGGZkGGxHAsRcjnv+GmP9FwQIyAgQ4xvg+nnUq2gBAjICBGQECMgIEIMb4KJk4N0JELMY4wZ6sL0IEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DMYv32fRnFOspeBIjBbdvtMq7t4LsTICAjQEBGgICMADG+x4uSMe5u15+7OPoFkAABJQFiLkcfgtaDr/+CADGFAU4rQ+5IgICMADGLAa6i14Gunx8IEJARICZy6CFoHW78ESDmdawGrYda7esJEHM5+viwHXz9FwSI6RzuILaOePh6IEBMbf8NWne/wrcQIGZ0Pkrs+Q1fz9Y23vgjQMxr/w1aR6+PADG1PTdonaA+AsTs9tmgdY76CBDsrkHrNPURINhXg9aZ6rMsy+d6AbAL23Z7Ot2cV+D+65ePXMD6z/DNUB8TEDz7zn/kKLROWR8TECz/fvM/chRaZ03PAwGCa8ex80a8Y4meHK+2yeqzLMunu7sfy/Gdf12GNOFXc7ffqzdmSHrOmYDgtSeyJwvyYo+u3yVts6bngQDB/8nQ2++qt7nTM9QRDI5y3tedcwIEvzxGovMcAQIyfogIZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELBU/gaK7DtZn0ZpEgAAAABJRU5ErkJggg==', 'sha256': 'bc2dc49b6ac9fb9c6c1559c6d35c0a72581ca9f67c0bcee339edf60f22042231'}, 'data/binding-v1/images/binding-development-0007-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFE0lEQVR4nO3aPU5jSRhAUXrU7IegJTYxcS+SeDbRUgcspgMkkgmQLMsG4582t6p8TkRgv/fJxpeqMt9eXv7cART+Se4KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyHzvbj2ih4cfd9N6fv5dj7Ds++61vRIB4kad9Mdm+8Fi9BcJELfl8kXu5gpKdDkBet9/9/d3k/j39bUeYfr03D8+HX7u66+fH11Qhi4hQNxoej6NzkcP3omRDF3Ct2DcXH3uH59Oqs8xT5/664uQFRDL2onCJdHZt7naZkFkKXQGKyDWdNX6HLiypdBJBIgFbVfgwg3XMXZuoUHHEyBWs1OfL7uvBp1BgFhKVZ/9O2rQMQSIdbT12b+vBn1KgFjECPXZv7sGHSZArGCc+uzPoEEHCBBLGaE+o00yMgFiepslxmif+c08FkEfESDmNstne5Y5v5gAsYjRlj8jTzUOAWJiw26+ttmIHSBAQEaAmNWMC4oZZ74qAWJ6I++/ZpmwIkBARoCY0hTHz9scRb9LgICMAAEZAQIyAsR8pjsAeuMYaJ8AARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAsR8np9/v/3w+uvn3Tw2027mR4CAjAABGQECMgLElKY7BnIA9C4BAjICxPTGXwSNP2FFgJjVjHuZGWe+KgECMgLExKY4inb8fIAAsYgxGzTmVOMQIOY2y7Jiljm/mAAxvWE3YjZfnxIgljJOg8aZZGQCxAq2lxgjfPK3Z7D8OUCAWMQ4DVKf4wkQ6xihQepzEgFiKW2D1OdUAsRqqgapzxkEiPUbdO0M7dxCfY4nQKxppwLXa9DOldXnJN9PezjM460FDw8/dkpx//h0+cX3iyY9Z/j28vLnnOctavPLOiMfgJPe1rMz9O5iyot/Hisgbm4ptN+RT2N0YAcnPZcQIG46Q5ecEEnP5WzBuFFnb7d15y8SIPg8RqJzJQIEZPwfEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQEC7ir/A8wTDpN8dBgTAAAAAElFTkSuQmCC', 'sha256': '413ff79d37d71b3f4b8091f7510a0392d118229cc581689aabea0db2ef01de0e'}, 'data/binding-v1/images/binding-development-0007-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFG0lEQVR4nO3aMW5aSxiAUfIUL+dJLtJmQ2mzjrTZUFoXSG8rKSy5eYUlhACDAeNv5nJO5cL33hGYz/+M+PL8/HcFUPgneSqAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg87V79IgeH7+tprVeP9VLWOz77rW9EQHiTp31z2b7l8XoAwkQ9+X6IXdzByW6ngAd9vLz39UkHn79Vy9h+vT8+f1y/NrvPx7euqEMXUOAuNP0nIzOW7+8EyMZuoYAcXf1OSs9+14v38+QBl1AgLiX9FzZnR2bu21KZBS6gO8BsUw3rc+RO0/9TY7PZwJigbYrcLv07DxiexQyB72TCYil+eT6HHyWOeidBIhFqeqz/0QNeg8BYjna+uw/V4NOEiAWYoT67D9dg44TIJZgnPrsr0GDjhAgFmWE+oy2kpEJENPbjBijfeY36zEEvUWAmNssn+1Z1vnJBIiFGG38GXlV4xAgJjbs5mubjdgRAgRkBIhZzThQzLjmmxIgpjfy/muWFVYECMgIEFOa4vh5m6PogwQIyAgQkBEgICNAzGe6A6BXjoH2CRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAzGe9fnr94fuPh9U8NqvdrB8BAjICBGQECMgIEFOa7hjIAdBBAgRkBIjpjT8Ejb/CigAxqxn3MjOu+aYECMgIEBOb4ija8fMRAsRCjNmgMVc1DgFibrOMFbOs85MJENMbdiNm83WSALEo4zRonJWMTIBYgu0RY4RP/vYajD9HCBALMU6D1Of9BIjlGKFB6nMWAWJR2gapz7kEiKWpGqQ+F/h6yUUwtvX66fHx23YX/vx+ud3jdjKnPu9nAmKZdipwu1FIfa5hAmKxXluwMwp91DS0XzTpucCX5+e/l1y3UJs/1hn5AJz1tl6coYPDlBf/MiYg7m4U2u/IyRgd2cFJzzUEiLvO0DUnRNJzPVsw7tTF223d+UACBKdjJDo3IkBAxveAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCwqvwP/30RU3tDZ+oAAAAASUVORK5CYII=', 'sha256': 'b03b5d5781bdab06bcc5ec5e6f87ab41b341d34752684ae3b15b2d6f8bd52853'}, 'data/binding-v1/images/binding-development-0007-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFGElEQVR4nO3avW1bSRhAUXph9aPAqTvYSlyQK9kOnCpgMQ4EKNlAAEGQFH9F3ZnHcyIF4nsDUu/qmwG/vb7+XQEU/knuCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh87249oufnH6tprdcv9RIW+7l7b+9EgHhQF/2z2f5lMfpEAsRjuX3I3VxBiW4nQIf99/S0msS/b2/1EqZPz5/fJ97Dn7+ePrqgDN1CgHjQ9JyMzke/vBMjGbqFAPFw9bkoPfveX76fIQ26ggDxKOm5sTs7NlfblMgodAXfA2KZ7lqfI1ee+pscX88ExAJtV+B+6dm5xfYoZA46kwmIpfni+hy8lznoTALEolT12b+jBp1DgFiOtj7799WgkwSIhRihPvt316DjBIglGKc++2vQoCMEiEUZoT6jrWRkAsT0NiPGaM/8Zj2GoI8IEHOb5dmeZZ1fTIBYiNHGn5FXNQ4BYmLDbr622YgdIUBARoCY1YwDxYxrvisBYnoj779mWWFFgICMADGlKY6ftzmKPkiAgIwAARkBAjICxHymOwB65xhonwABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICxHzW65f3H37+elrNY7PazfoRICAjQEBGgICMADGl6Y6BHAAdJEBARoCY3vhD0PgrrAgQs5pxLzPjmu9KgICMADGxKY6iHT8fIUAsxJgNGnNV4xAg5jbLWDHLOr+YADG9YTdiNl8nCRCLMk6DxlnJyASIJdgeMUZ48rfXYPw5QoBYiHEapD7nEyCWY4QGqc9FBIhFaRukPpcSIJamapD6XOH7NS+Csa3XL8/PP7a78Of32/1ut5M59TmfCYhl2qnA/UYh9bmFCYjFem/Bzij0WdPQftGk5wrfXl//XvO6hdr8sc7IA3DRx3p1hg4OU97865iAeLhRaL8jJ2N0ZAcnPbcQIB46Q7ecEEnP7WzBeFBXb7d15xMJEJyOkejciQABGd8DAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DAqvI/a94RU8KpTisAAAAASUVORK5CYII=', 'sha256': '069996796bd4545e3ae8cf859e423bf7bca8d141f36b3044364639e823ff8b34'}, 'data/binding-v1/images/binding-development-0008-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJ0lEQVR4nO3dTWobWRSAUbtJltPgQaB3ko30OnojvZNABobeigcCT3pgEEKyZev3e6/qnGki1yWhvlw9KtTjZvPyAFD4I7kqgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5lt3aSg9Pf04/huen3/fa5b1etxsXuoZYJTofESMbkSAWL6zu3NIia5LgFhpel7//vP4Z7//899HvyRD1yJA1/9HcjTrvFve/Qv9NDonxWidf7DX5RCaVdTn7PTsfnwvQ09PPzToQgLEwutzYXqOZ0iDLiRA+77/9e/DIrz++vmwMrdLz5EMvV1Uhs7jQUQW4j71+ejnL/gM8aZsQCzB7v1/6/QcWYXsQaeyATG9pD7vXtEedCoBYm5tfQ6vq0EnESAWoqrPCFeflwAxse26McL9v53BEvR1AsSsRr7PR55tKALElEY4+jnkMOhUAsTcxqnPmPMMToCYzyzLxSxzhgSIiY25bow51ZgEiMnMtVbMNe39CRCzGnnRGHm2oQgQkBEgZjLUk4fHeS7xKwQIyAgQkBEgICNATGOiA6A3joE+JUBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAMY3tm9e3r2Mf3HZO74z/iAABGQECMgIEZASImUx0DOQA6CsECMgIELMaeQkaebahCBCTmesbzVzT3p8AMbExF40xpxqTADGfWdaKWeYMCRBzG23dGG2ewQkQU9pdLsa553cnsf58hQAxq5Hv8JFnG4oAMbGhnkv05OEZBIiFaBs0QgFnJEDMbYTDIEc/ZxMgptc2SH0u8e2iT8MYnp9/b9+//laEO7w/fi926nMGGxALsXf/33oVUp+rsAGxHG8VuPUqJD1XJEAs+evYdTN0uFWpz4UEiOU36PIMvfuFTn0uJ0Cs4uvYYUc+jdGRUyTpuRYBYnUZuuSUWnquS4BYvm013i3RST+B6xIgVmS3I5/GSHTuQIBYKX0ZgQcRgYwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZ/xVj3+uvn/UIsBY2ICAjQEDmcbN56a4OrJoNCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAh4qPwPO4wUgE+4bq8AAAAASUVORK5CYII=', 'sha256': '2074f953e270ac70039a08ff03cb0c2b394db7a56c55512a703452659c862b30'}, 'data/binding-v1/images/binding-development-0008-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJUlEQVR4nO3dQU7bWhiA0fLU7IdBl9HlsowOWAyDSEzeIFIUBQiQmHz32ueMkEp8r1L88dtO1Yf9/uUXQOG/ZFUAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM725pKD0+/rn8Dc/P/+61l+162O9f6j3AKNH5iBj9EAFi/a7uzltKtCwBYqPpedrtLr/27+vrR38kQ0sRoOV/SY5mm2fLu3+hn0bnWzHa5hu7LE/B2ER9nna7q+vz0ctX/EvrbgSItTnrwo3puXwoDbqRx/DnlvphzV24hbGd9PzEKofDHt/ew6Iux65jAmIl7lOfj45vFLqOALEGp+f/gtdcl50tpEFXECDWVp87r65BtxAg5tbW5+26GvQtAsRKtE8PVvPs4s4EiIkdx40Rzv/jHgxBXydAzGrk83zkvQ1FgJjSCLd+3nIz6LsEiLmNU58x9zM4AWI+swwXs+wzJEBMbMxxY8xdjUmAmMxcY8Vcu70/AWJWIw8aI+9tKAIEZASImQz1ycPLfC7xKwQIyAgQkBEgICNATGOiG0AHbgN9SoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBYhrPz/8OX/x9ff01g+M+jzvnjAABGQECMgIEZASImUx0G8gNoK8QICAjQMxq5CFo5L0NRYCYzFxXNHPt9v4EiImNOWiMuasxCRDzmWWsmGWfIQFibqONG6PtZ3ACxJROh4txzvnTnRh/vkKAmNXIZ/jIexuKADGxoT6X6JOHVxAgVqJt0AgFnJEAMbcRbga59XM1AWJ6bYPU5xYCxAobdJ8MnS2kPlcQIFbi7Pz/6QadHV99rvP7ytfBeA4VeHz8c9qIp91u2VWkZ0EmINY/Ci01Db09lPrcSIBYobdduDFD775cfW7nEoxNXI4dnEbk00uzC8GSnqUIEJvL0MF1A5H0LEuAWL9jNd4t0beOwLIEiA057cinMRKdOxAgNkpfRuApGJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg459inPP/q8DdmICAjAABmYf9/qVbHdg0ExCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQ8KvyP0v9FlZpuYlTAAAAAElFTkSuQmCC', 'sha256': '53d4ff17cfc32c315e5feacd06bc0f1e7f5652d71ba767c3f93e72f2b76b70cd'}, 'data/binding-v1/images/binding-development-0008-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJElEQVR4nO3dTWobWRSAUbuJl9PgQZaR3fQ6eje9jAwMvRUPBJpkYBBCtmXr93uv6pxpItclob5cPSrU42bz+gBQ+Cu5KoAAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZHd2koPT//PP4bXl5+32uW9XrcbF7rGWCU6HxGjG5EgFi+s7vznhJdlwCx0vRs//n7+Gef/v3/s1+SoWsRoOv/Izmadd4tH/6Ffhmdk2K0zj/Y63IIzSrqc3Z69j9+kKHn558adCEBYuH1uTA9xzOkQRcSoEP/PT09LMKv7fZhZW6XniMZeruoDJ3Hg4gsxH3q89nPX/AZ4k3ZgFiC/fv/1uk5sgrZg05lA2J6SX0+vKI96FQCxNza+ry/rgadRIBYiKo+I1x9XgLExHbrxgj3/24GS9D3CRCzGvk+H3m2oQgQUxrh6Oc9h0GnEiDmNk59xpxncALEfGZZLmaZMyRATGzMdWPMqcYkQExmrrVirmnvT4CY1ciLxsizDUWAgIwAMZOhnjw8znOJ3yFAQEaAgIwAARkBYhoTHQC9cQz0JQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQEiGns3ry+ex374HZzemf8ZwQIyAgQkBEgICNAzGSiYyAHQN8hQEBGgJjVyEvQyLMNRYCYzFzfaOaa9v4EiImNuWiMOdWYBIj5zLJWzDJnSICY22jrxmjzDE6AmNL+cjHOPb8/ifXnOwSIWY18h48821AEiIkN9VyiJw/PIEAsRNugEQo4IwFibiMcBjn6OZsAMb22QepziR8XfRrG8PLye/f+9bci3OH98QexU58z2IBYiIP7/9arkPpchQ2I5XirwK1XIem5IgFiyV/Hrpuh91uV+lxIgFh+gy7P0Idf6NTncgLEKr6Ove/IlzE6cookPdciQKwuQ5ecUkvPdQkQy7erxoclOukncF0CxIrsd+TLGInOHQgQK6UvI/AgIpARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg479iHPq13dYjwFrYgICMAAGZx83mtbs6sGo2ICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgofIHxJUUqFREWu0AAAAASUVORK5CYII=', 'sha256': '802e05bf6c07bb57c432f8ece3897a065385339182c69ff0e5a4d912c6b8c981'}, 'data/binding-v1/images/binding-development-0009-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFUlEQVR4nO3dO05jSxRA0abVPR8HpG9Cb0BMiLQDBkNgiaQDJMuyGxt/YFfdu1bKpyrA26dKxn7Ybl9/ABR+JqsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZXtzT0NpvH09/w8vLnu/ayRg/b7Wu9BxgrOh8Ro7sTINbi6u4cU6J7ESBWnZ7np7fTP/vf/78/+pIM3U6AWF16zkbnohjJ0C0EiBXV5+r0nM6QBl1NgFh+eu7SnbMlkqEreB0QS/M99Tn+zXe85F4PExCLsl+Br0vPiVHIHHQRExDLkdTnYC1z0EUEiIWo6nO8ogZ9ngCxBG19jtfVoE8SIKY3Qn2OV9egzxAg5jZOfY73oEFnCRALMUJ9RtvJ+ASIie1GjNEe87v9GIJOEyBmNctje5Z9Jlb9hmSL/8tYyYviRht/3j0/vZ34T3remYCY0rCHr30OYmcJEJBZ9RFsZ+Rn0SssfvKfcaDYbB5XciK+iAmIiY3/zDH+DlsCBGQEiMlMcf28z1X0CQIEZAQIyAgQkBEgZjLdBdA710AfESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAmMnuTQXnetfH3W69KeIBAQIyAgRkBAjICBCTme4ayAXQCQIEZASIiY0/BI2/w5YAMZ8ZzzIz7vkbCBCQESCmNMVVtOvnswSI6Y3ZoDF3NRoBYlazjBWz7DMhQExs2IOYw9cnCRALMU6DxtnJ+ASIue2PGCM88vf3YPw5S4CY3jgNUp9LCRBLMEKD1OcKAsRCtA1Sn+sIEMtRNUh9rvar3gDc08vLn83mcb8Lz09vX7fcQebU51ImIJbmoAJfNwqpz+1MQCzQewsORqF7TUPHRZOeqwkQqziO3eVQ9s9hSn1uIUCsaBQ67sjZGJ04wUnP7QSIlWbolhsi6bkXAWItdtX4Z4ku+g3ciwCxOvsdORsj0flSAsSq6UvL64CAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg43PBrv98XuBGJiAgI0BA5mG7fe1WB1bNBARkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAH5W/vGQao3YzvtgAAAAASUVORK5CYII=', 'sha256': '1b1eb0043a77a4bc85ca09199c7eb688aefe21e29c5c3ad882c10757f1bd44ac'}, 'data/binding-v1/images/binding-development-0009-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFI0lEQVR4nO3dT0qjSRyAYR26jyO4EOYmfRHPMReZmzT0QuiruAi4GQYhBNMm5o++Vd/3PFs1VQvz8qsift5uNs83AIW/klUBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPOtWxp69/cPh7/h6enXV+1ljW43m+d6DzBWdN4jRlcnQKzF2d3Zp0TXIkCsOj0vj3eHf/b7P7/f+5IMXU6AWF16jkbnpBjJ0CUEiBXV5+z0HM6QBp1NgFh+eq7SnaMlkqEz+BwQS/M19dl/5Stecq+HCYhF2a3A56XnwChkDjqJCYjlSOrzZi1z0EkEiIWo6rO/ogZ9nACxBG199tfVoA8SIKY3Qn32V9egjxAg5jZOffb3oEFHCRALMUJ9RtvJ+ASIiW1HjNHe89v9GIIOEyBmNct7e5Z9Jlb9QLLF/2as5ENxo40/r14e7w78JT2vTEBMadjD1y4HsaMECMis+gi29f3vf28W5OXnj5tFm3GguL9/WMmJ+CQmICY28vlrlh22BAjICBCTmeL6eZer6AMECMgIEJARICAjQMxkugugV66B3iNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMADGT7UMF53re+3a3Hor4hgABGQECMgIEZASIyUx3DeQC6AABAjICxMTGH4LG32FLgJjPjGeZGff8BQQIyAgQU5riKtr181ECxPTGbNCYuxqNADGrWcaKWfaZECAmNuxBzOHrgwSIhRinQePsZHwCxNx2R4wR3vm7ezD+HCVATG+cBqnPqQSIJRihQepzBgFiIdoGqc95BIjlqBqkPmf7Vm8Arunp6df9/cNuF14e7z5vuTeZU59TmYBYmjcV+LxRSH0uZwJigV5b8GYUutY0tF806TmbALGK49hVDmV/HKbU5xICxIpGof2OHI3RgROc9FxOgFhphi65IZKeaxEg1mJbjT+W6KRX4FoEiNXZ7cjRGInOpxIgVk1fWj4HBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARn/F+x/Lz9/1FuANTIBARkBAjK3m81ztzqwaiYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBNxU/gNqGCBnaMFNPQAAAABJRU5ErkJggg==', 'sha256': '8d2a4ff4ba8aa16776491ca1d60894c29827bc642bcfc9ee8b7c0ec8bc2cb225'}, 'data/binding-v1/images/binding-development-0009-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFI0lEQVR4nO3dPU5jSRhAURh174cAqaPZRi9iFsQiZhtELXXAYggskYxGSJaFGxv/wK1675wUcFWAr74qmcftZvN8A1D4K1kVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjLfuqWhd3d3f/gbnp5+f9Ve1uh2s3mu9wBjRec9YnR1AsRanN2dfUp0LQLEqtPz+PBy+Gf//uf7e1+SocsJEKtLz9HonBQjGbqEALGi+pydnsMZ0qCzCRDLT89VunO0RDJ0Bp8DYmm+pj77r3zFS+71MAGxKLsV+Lz0HBiFzEEnMQGxHEl93qxlDjqJALEQVX32V9SgjxMglqCtz/66GvRBAsT0RqjP/uoa9BECxNzGqc/+HjToKAFiIUaoz2g7GZ8AMbHtiDHae367H0PQYQLErGZ5b8+yz8SqH0i2+N+MlXwobrTx59Xjw8uBv6TnlQmIKQ17+NrlIHaUAAGZVR/Btr7/+PdmQV5+/bxZtBkHiru7+5WciE9iAmJiI5+/ZtlhS4CAjAAxmSmun3e5ij5AgICMAAEZAQIyAsRMprsAeuUa6D0CBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBAz2T5UcK7nvW9366GIbwgQkBEgICNAQEaAmMx010AugA4QICAjQExs/CFo/B22BIj5zHiWmXHPX0CAgIwAMaUprqJdPx8lQExvzAaNuavRCBCzmmWsmGWfCQFiYsMexBy+PkiAWIhxGjTOTsYnQMxtd8QY4Z2/uwfjz1ECxPTGaZD6nEqAWIIRGqQ+ZxAgFqJtkPqcR4BYjqpB6nO2b/UG4Jqenn7f3d3vduHx4eXzlnuTOfU5lQmIpXlTgc8bhdTnciYgFui1BW9GoWtNQ/tFk56zCRCrOI5d5VD2x2FKfS4hQKxoFNrvyNEYHTjBSc/lBIiVZuiSGyLpuRYBYi221fhjiU56Ba5FgFid3Y4cjZHofCoBYtX0peVzQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJDxf8H+9/LrZ70FWCMTEJARICBzu9k8d6sDq2YCAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMBN5T8+6xwRg15utQAAAABJRU5ErkJggg==', 'sha256': 'e0b45f7d867042a6b919f8209ee5c6d4d8dc406d931bc7ad1c271ab61c1d0e1a'}, 'data/binding-v1/images/binding-development-0010-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFAElEQVR4nO3cMW4iSxRAUc/XsB8CL2OW62U4YDETWHLyAyTUAg8YY3OL6nOikQXdlfRV1Wt7fr29/X0CKPyX3BVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD53d16RNvt89MUdrvXeglwmR0QkBEgIOMI9rGXzebpAf15f6+XMPkZ3Nn2ewkQa3fV4G/5YTG6nQCxXje+c9h/XYZuIUCszpnuXDx6nx5yD1dToi8QINZen6vmfcsPH8Vou33WoGsJEGtxY3pO7b++zJBD2bW8hmeN9XnZbL7rRefppab5ddY7ECDmtyzCN6Zn6eiyGvRJAsS66vOj99KgawkQM7tnfU7vokEXCRDTOjz/P3Ts+pfl7TToPAFi/vokC9CgzxAgJjTaMz/aesYhQMzm/nOffzEPukiAmNYI/6XBCGsYmQAxlXz0c8ow6AwBYh7jP+Hjr/DOBIgJjbP9GXM94xAgICNATGLA6c+SSdCHBAjICBAzGHz7s2cTdEqAgIwAARkBAjICxMN7iAHQnjHQEQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQEiIe3273u//Hn/f1pbIcVHta8cgIEZAQIyAgQkBEgZvAQYyADoFMCBGQEiEkMvgmy/fmQAAEZAWJCo22CRlvPOASIeYx/uhl/hXcmQExlwEmQ6c8ZAsS0RmjQCGsYmQAxm+VGo33+l3e3/fmQADGh0Z720dYzDgFiTvkwyOjnMwSIVTTonhla3k59zhMgZnb/eZC5z1UEiMnds0Hqcy0BYnUN+okMHV1WfT5JgFiFoyJ8Y4ZOL6U+n/f7is/CI9t3Ybt9PvxkH46XzeZrFzxNmPRcS4BYl93uddmgo45cjNGZfZP6fIEAsTqHUhyV6GtTat25hQCxXqeHsi98nVsIEGu37MjFGInO9xKgj/kj5nXSlzvzGh7ICBCQ+fX29re7O7BqdkBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQKeKv8DP1wH4ITjJfUAAAAASUVORK5CYII=', 'sha256': '9bc3ee06ac8746c72807630a86a6cff7c6dcaee810d2f891c974945cab6faa7a'}, 'data/binding-v1/images/binding-development-0010-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFC0lEQVR4nO3aQUojTRiAYWcYjzOQheBN5iL/OeYi3mTARWCuMgvBzb8ISEg0MUbzVqqfZxUk3VWbfqn+4renp383AIXvyaoAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMj27pEa1WdzdTWK8f6y3AcU5AQEaAgIxXsNc9//fz5grd/v5bb2Hyd3Dvtp9LgFi6kwZ/218Wo/MJEMt15m8Om8tl6BwCxOIc6M7t/cPha5///Hrrbkr0AQLE0utzNDpvfXknRqvVnQadSoBYijPTs29z+XaGvJSdys/wLLE+t/cPZ9bnwK2m+XfWCxAg5rddhE9Mz7ad22rQOwkQy6rPl66lQacSIGZ2yfrsr6JBRwkQ03p5/r/otest28tp0GECxPz1STagQe8hQExotGd+tP2MQ4CYzeXnPm8xDzpKgJhWW59x9jAyAWIq+ehnn2HQAQLEPMZ/wsff4YUJEBMa5/gz5n7GIUBARoCYxIDTn20mQa8SICAjQMxg8OPPhkPQPgECMgIEZAQIyAgQV+8qBkAbxkA7BAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgrt56/bj58Pzn183YXnb4sueFEyAgI0BARoCAjAAxg6sYAxkA7RMgICNATGLwQ5Djz6sECMgIEBMa7RA02n7GIUDMY/y3m/F3eGECxFQGnASZ/hwgQExrhAaNsIeRCRCz2T5otM//9uqOP68SICY02tM+2n7GIUDMKR8GGf28hwCxiAZdMkPby6nPYQLEzC4/DzL3OYkAMblLNkh9TiVALK5BX5GhnduqzzsJEIuwU4RPzND+rdTn/X6c8F24ZpsurFZ3L3/ZhOP2/uFjN9xPmPScSoBYlvX6cbtBOx05GqMD5yb1+QABYnFeSrFToo9NqXXnHALEcu2/lH3gcs4hQCzddkeOxkh0PpcAve729996CwT05cL8DA9kBAjIfHt6+tetDiyaExCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICbyv8TSAbbOHqiEwAAAABJRU5ErkJggg==', 'sha256': '5384b9b5885bd6c5745e75b42fd2e536f5830f900edada9f6fd6f43bb447f678'}, 'data/binding-v1/images/binding-development-0010-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFCElEQVR4nO3awW0iSRiAUc9qCGclDg5jstk4JhuH4QPSpDIHS1z2gGQhsMEYm6+ofu+ELLqrLv2p+sc/Xl7+PgAU/klWBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZH52S49ovX58mMJm81xvAc5zAgIyAgRkvIK9bfvfvw93aPX7T72Fyd/Bvdt+LQFi6S4a/O1/WYyuJ0As15W/Oewul6FrCBCLc6I7T6vV6Wt/bbfv3U2JPkGAWHp9zkbnvS8fxGi9ftSgSwkQS3Fleo7tLt/PkJeyS/kZniXW52m1urI+J241zb+z3oAAMb/9InxhevYd3FaDPkiAWFZ9vnUtDbqUADGzW9bneBUNOkuAmNbr8/9Nr13v2V9Og04TIOavT7IBDfoIAWJCoz3zo+1nHALEbG4/93mPedBZAsS02vqMs4eRCRBTyUc/xwyDThAg5jH+Ez7+Dm9MgJjQOMefMfczDgECMgLEJAac/uwzCXqTAAEZAWIGgx9/dhyCjgkQkBEgICNAQEaAuHt3MQDaMQY6IEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAcfc2m+fdh1/b7cPYXnf4uueFEyAgI0BARoCAjAAxg7sYAxkAHRMgICNATGLwQ5Djz5sECMgIEBMa7RA02n7GIUDMY/y3m/F3eGMCxFQGnASZ/pwgQExrhAaNsIeRCRCz2T9otM///uqOP28SICY02tM+2n7GIUDMKR8GGf18hACxiAbdMkP7y6nPaQLEzG4/DzL3uYgAMblbNkh9LiVALK5B35Ghg9uqzwcJEItwUIQvzNDxrdTn435e8F24Z7surNePr3/ZheNptfrcDY8TJj2XEiCWZbN53m/QQUfOxujEuUl9PkGAWJzXUhyU6HNTat25hgCxXMcvZZ+4nGsIEEu335GzMRKdryVAb1v9/lNvgYC+3Jif4YGMAAGZHy8vf7vVgUVzAgIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEPBQ+R93IQcDSaoIUAAAAABJRU5ErkJggg==', 'sha256': '8c797cba431ca38d4993d7ed5786dec0211ae0b61c25201e5808bdf7c9b6fe00'}, 'data/binding-v1/images/binding-development-0011-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMklEQVR4nO3dQU4bSRiAUTIajjMSi0i5SS6Sc8xFcpNIWVjKVbJAYjMLK5YHjG2M8VfV/d4KCeguBP3pr6YNnx4ff98BFP5KzgogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjJ/d6ceyMPD57vF2Wx+1kuAE0xAQEaAgIwt2P88ffvnbnL3//6qlwDnMgEBGQECMgIEZAQIyAgQkBEgICNAQMZzQHDWy3G8tOUjCBDr9abXAO5/sBhdiwCxOu9/7fHuCEr0TgLEWhzvzv2X78c//enH1yPHVKLLCBDrrc/J6Lz2wS9j9PDwWYMuIECsLj1v6s5BuyPsl2h7Ihl6E7+GZ0X1uf/y/f31OX7ARf5xu49jAmIt6fm4c20PvpuGjELnMwGxQLesz2tnMQqdQ4BYmv0r/+p7ruOenU6DThIgllyfZA0adD4BYjlGqM/Ls2vQEQLEQoxTn5dr0KDXCBBLMFp9tjToJAFiUcapz5jrGY0AMb3dcDHm1b5blSHoJQFibnNd1XOt9gYEiIUYc/wZf20tAWJig2++9tmIHSRAzGreK3nelV+dADG98cefudZ5SwIEZASIKU1092efO0HPCBCQESAgI0DMZ9L915Zd2D4BAjICBGQECMgIEJOZ+gbQlttAOwIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJPZbH5u33j68fVuTruVb/58LaslQEBGgICMAAEZAWI+U98GcgNonwABGQECMgLElCbdhdl/PSNAQEaAmN4sQ9As67wlAWJW8+5i5l351QkQE5voTpC7PwcJEAsxcoNGXltLgJjbXAPFXKu9AQFieoNvxGy+jhAgFmW0Bo22ntEIEEuwP1yMc83vr8T4c9Cnx8ffh9+zJov8F90r/Inf/z7m/zlefc5hAmI5xpmD1OdMAsSijNAg9TmfLRjL31PfbDv2LHnqc5IJiAV6duXfZhRSnwuYgFiy24xC0nMxAWJ1v+K8YoZezlbq8yYCxHqftLi4RAf3dNJzAQFiLY4/7XUyRsdvJKnPZQSI1bnic6e6804CxHpdXCLduRYBgrNiJDofQYCAjAcRgYwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIC7yn9DhhWBt8/GuQAAAABJRU5ErkJggg==', 'sha256': '17f6e12cb7904e50c73d8c59f181fb1956059ff7e3ec40f26ba5a452ede0b153'}, 'data/binding-v1/images/binding-development-0011-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJUlEQVR4nO3dMW7bSBiAUWWxuY+LtLlQDrQX2jaFD5PCwDZbCCtwLUWSJVnfDPleZcCGOIbND//QpPzl7e3XDqDwR3JUAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+bM79EBeXr7tVuf19We9BLjABARkBAjI2IL9z99//bOb3PcfX+slwLVMQEBGgICMAAEZAQIyAgRkBAjICBCQcR8QXPU4jkdbPoMAsV0fegZw+cVi9CgCxObc/+zx4RWU6E4CxFac787Fp3BOPuOiRHcSILZbnw89+rf84uMYvbx806AbCBCbS8/9jxwfXmFZov2BZOhDBIgN1efh73awf8F3GdKg6wkQ6/TZ6TmTIaPQ9dyIyAo9sz6/O8oq3+f34UxArM3yzH/yO8wdj0LmoPNMQKxKWJ+TxzUHnSdArMcI9Tk+ugadIUCsxDj1OV6DBv2OALEGo9VnT4MuEiBWZZz6jLme0QgQ0zsMF2Oe7YdVGYKOCRBzm+usnmu1TyBArMSY48/4a2sJEBMbfPO1ZCN2kgAxq3nP5HlX/nACxPTGH3/mWuczCRCQESCmNNHVnyVXgt4RICAjQEBGgJjPpPuvPbuwJQECMgIEZAQIyAgQk5n6AtCey0AHAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQk3l9/bn/4PuPr7s5HVb++t/3slkCBGQECMgIEJARIOYz9WUgF4CWBAjICBCQESCmNOkuzP7rHQECMgLE9GYZgmZZ5zMJELOadxcz78ofToCY2ERXglz9OUmAWImRGzTy2loCxNzmGijmWu0TCBDTG3wjZvN1hgCxKqM1aLT1jEaAWIPlcDHOOb9cifHnpC9vb79Of2ZLVvkvujf4G7/8Oeb/OV59rmECYj3GmYPU50oCxKqM0CD1uZ4tGOvfUz9tO/YueepzkQmIFXp35j9nFFKfG5iAWLPnjELSczMBYnN/4nxgho5nK/X5EAFiu3da3Fyik3s66bmBALEV5+/2uhij8xeS1Oc2AsTmPPC+U925kwCxXTeXSHceRYDgqhiJzmcQICDjRkQgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYFf5F5qzJPvqmcc4AAAAAElFTkSuQmCC', 'sha256': 'f9b1ae5e87773058a3e2f58db51196793a47a6f9499c55f9356735c91b130d04'}, 'data/binding-v1/images/binding-development-0011-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFM0lEQVR4nO3dQU4bSRiAUTIa7sMiUla5Rg6RA80hco1ZRcrCh8kCic0srFgeMLYxxl9V93srJKC7EPSnv5o2fHp8/H0HUPgrOSuAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMj83Z16IA8Pn+8WZ7P5VS8BTjABARkBAjK2YP/z7z9Pd5P7+v2+XgKcywQEZAQIyAgQkBEgICNAQEaAgIwAARnPAcFZL8fx0paPIECs15teA7j/wWJ0LQLE6rz/tce7IyjROwkQa3G8O/dffhz/9Kef344cU4kuI0Cstz4no/PaB7+M0cPDZw26gACxuvS8qTsH7Y6wX6LtiWToTfwanhXV5/7Lj/fX5/gBF/nH7T6OCYi1pOfjzrU9+G4aMgqdzwTEAt2yPq+dxSh0DgFiafav/KvvuY57djoNOkmAWHJ9kjVo0PkEiOUYoT4vz65BRwgQCzFOfV6uQYNeI0AswWj12dKgkwSIRRmnPmOuZzQCxPR2w8WYV/tuVYaglwSIuc11Vc+12hsQIBZizPFn/LW1BIiJDb752mcjdpAAMat5r+R5V351AsT0xh9/5lrnLQkQkBEgpjTR3Z997gQ9I0BARoCAjAAxn0n3X1t2YfsECMgIEJARICAjQExm6htAW24D7QgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQExms/m1fePp57e7Oe1WvvnztayWAAEZAQIyAgRkBIj5TH0byA2gfQIEZAQIyAgQU5p0F2b/9YwAARkBYnqzDEGzrPOWBIhZzbuLmXflVydATGyiO0Hu/hwkQCzEyA0aeW0tAWJucw0Uc632BgSI6Q2+EbP5OkKAWJTRGjTaekYjQCzB/nAxzjW/vxLjz0GfHh9/H37PmizyX3Sv8Cd+//uY/+d49TmHCYjlGGcOUp8zCRCLMkKD1Od8tmAsf099s+3Ys+Spz0kmIBbo2ZV/m1FIfS5gAmLJbjMKSc/FBIjV/Yrzihl6OVupz5sIEOt90uLiEh3c00nPBQSItTj+tNfJGB2/kaQ+lxEgVueKz53qzjsJEOt1cYl051oECM6Kkeh8BAECMh5EBDICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQLuKv8B0HEVgbswcKEAAAAASUVORK5CYII=', 'sha256': '1dda229cc5af1ac39bf5dd29989d89b4922a5e2f4aa8633a0150f0691f18e852'}, 'data/binding-v1/images/binding-development-0012-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEklEQVR4nO3dS04jSRRAUWhV74dBTXtDvSA2xLQGLIYBEpMaWEIWBoP5+Eb6nTPlkyFZefUisMnrx8eHK4DCP8lVAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZH51l96qm5vfx7/h/v7PudYC23b9+PhQr+ESovMWMYIjBOhHunNIieCQAJ2cnrvbp+M/+9///771JRmCfQL0ofS8G52TYiRDsCNAx9Lz6e58pEQyBAL0en2+MT1HMqRBDCdAL+vzQ+l5K0MaxGTTA3SeweeQUQimvxO6qs/htb7x7/2wIXMnoDNvu95iO/ZFm273/fhXfOgEtEh9Xlx90/cSfMLQAC1Sn3XWAImJH0Z9HjTWufPvbp92e7Gbm9/G8s9Z59V815H3yk8zbgJaf5uz/grhu8wK0DpHP4ccBjHQrAAtW5+VVwU/Z1CAFjz6OfS8NkMQEwwKELCaKQHaxPizYwhijikBAhY0IkAbGn92DEEMMSJAwJoECMhcfoA2t//asQtjgssPELAsAQIyAgRkLjxAGz0A2nEMxMW78AABKxMgICNAQEaAgIwAARkBAjICBGQECMgIEJARICBz4QF6fsroFp9F+bxmz0rlUl14gICVCRCQESAgc/kB2ugxkAMgJrj8AAHLEiAgMyJAm9uF2X8xxIgAAWuaEqANDUHGH+aYEiBgQYMCtIkhyPjDKIMCtG/NBq25Kvg5swK0P1asdrfvr8f4wxCzArSJe3v9FcJ3GRegNQ+DHP0w08QA7VuhQSusARJDA7TOYZCjHyb7dTXV/f2fm5vf+xW4u3065wJehE99GGjoBPTqPX/OUUh9YHqAdnf++bdjL7Zd6sNYc7dgyXbM4AP7BOiVBv1Ehg5nK/UBAXpZhMMMfaVEr+7ppAd2BOj9DL3oyLsxOnKQJD2wT4BOyNBXDqqlBw4J0Ieq8WqJTvoNwCEBOrkj78ZIdOCDBOhk+gLfZfobEYGQAAEZWzAuhP9qskUmICAjQEDm+vHxobs6MJoJCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DAVeUvVJooiUmH5kYAAAAASUVORK5CYII=', 'sha256': '07f7e53a8ea4cc8b7c584567515bb5cdd764b16283e79dc57e1617bb71eafed1'}, 'data/binding-v1/images/binding-development-0012-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHElEQVR4nO3dTUpcWxSAUX3E4Qg2HEZm4zjebN4w0hDeVNIQqpNGgRSWf2XU79zaa3VNcg/I/dj7RKnLh4ffFwCFf5KnAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyPzoHr1VNze3r/+B+/tf33UW2LbLh4ff9RnOITovESN4hQB9SXeOKREcE6CT07O7u3797179+/9LX5IhOCRA70rPm9E5KUYyBHsC9Fp6Ptyd95RIhkCAnq/PJ6bnlQxpEMMJ0NP6fFF6XsqQBjHZ9AB9z+BzzCgE038SuqrP8bM+8f/7YUPmTkDfvHa9xDr2lzbd7vvx3/GhE9Ai9Xny9E2/S/ABQwO0SH3WOQMkJv4y6uOgsc6bv7u73u9iNze3xvKP+e/q6mIjfu529RFWMW4CWn/NWf+E8FlmBWidq59jLoMYaFaAlq3PyqeCrzMoQAte/Rx7PJshiAkGBQhYzZQAbWL82TMEMceUAAELGhGgDY0/e4YghhgRIGBNAgRkzj9Am9u/9mxhTHD+AQKWJUBARoCAzJkHaKMXQHuugTh7Zx4gYGUCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkzjxAj58yuv/c0W15PLPPSuVcnXmAgJUJEJARICBz/gHa6DWQCyAmOP8AAcsSICAzIkCb28LsXwwxIkDAmqYEaENDkPGHOaYECFjQoABtYggy/jDKoAAdWrNBa54Kvs6sAB2OFau97YfnMf4wxKwAbeLdXv+E8FnGBWjNyyBXP8w0MUCHVmjQCmeAxNAArXMZ5OqHyX5cTHV//+vm5vawAru76+88wJPwqQ8DDZ2Ann3nv3MUUh+YHqD9m//969iTtUt9GGvuCpasYwYfOCRAzzToKzJ0PFupDwjQ0yIcZ+hvSvTsTic9sCdAb2foSUfejNErF0nSA4cE6IQM/c1FtfTAMQF6VzWeLdFJ/wJwTIBO7sibMRIdeCcBOpm+wGeZ/oOIQEiAgIwVjDPxc7erj8DJTEBARoCAzOXDw+/u6cBoJiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAReVP6L9MKC8b1+uAAAAAElFTkSuQmCC', 'sha256': '708812f2eeb67bf43661c385248ae51bcb78722d55681bef929309835018899b'}, 'data/binding-v1/images/binding-development-0012-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFH0lEQVR4nO3dS04jSRRAUWhV7YcB09pBraQXxEp6B0xrwGIYIDHpgSVkYb6F8Y30O2fKJ0MyefUiSMuXDw/3FwCFf5KrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyPzoLr1VV1fXb3/D3d2fU60Ftu3y4eG+XsM5ROc1YgRvEKBv6c4hJYJDAvTp9NzePL79s7/+/fnal2QI9gnQh9LzbnQ+FSMZgh0Beis9f92dj5RIhkCAXq7PEdPzRoY0iOEE6Hl9vik9r2VIg5hseoBOM/gcMgrB9Cehq/ocXuuI/++HDZk7AZ142/Ua27Ev2nS778a/4kMnoEXq8+zqm76X4C8MDdAi9VlnDZCY+GbUp0FjnTv/9uZxtxe7uro2lv+d/36++gD6an4/rvKHlxs3Aa2/zVl/hXAsswK0ztHPIYdBDDQrQMvWZ+VVwfcZFKAFj34OPa3NEMQEgwIErGZKgDYx/uwYgphjSoCABY0I0IbGnx1DEEOMCBCwJgECMucfoM3tv3bswpjg/AMELEuAgIwAAZkzD9BGD4B2HANx9s48QMDKBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyJx5gJ4+ZXT3uaPb8rRmn5XKuTrzAAErEyAgI0BA5vwDtNFjIAdATHD+AQKWJUBAZkSANrcLs/9iiBEBAtY0JUAbGoKMP8wxJUDAggYFaBNDkPGHUQYFaN+aDVpzVfB9ZgVof6xY7W7fX4/xhyFmBWgT9/b6K4RjGRegNQ+DHP0w08QA7VuhQSusARJDA7TOYZCjHyb7cTHV3d2fq6vr/Qrc3jyecgHPwqc+DDR0Anrxnj/lKKQ+MD1Auzv/9NuxZ9su9WGsuVuwZDtm8IF9AvRCg74jQ4ezlfqAAD0vwmGGvlKiF/d00gM7AvR+hp515N0YvXGQJD2wT4A+kaGvHFRLDxwSoA9V48USfeo3AIcE6NMdeTdGogMfJECfpi9wLNMfRARCAgRkbME4E78fT/peYo7CBARkBAjIXD483HdXB0YzAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIuKj8D40KKh/YrQFUAAAAAElFTkSuQmCC', 'sha256': '319ed71a3ae5f2689e83ba0765041798af3d8b1254a6f9f1d07992c9270ac6a8'}, 'data/binding-v1/images/binding-development-0013-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFB0lEQVR4nO3dQU4iWxiAUe00+2HgMnq5LuMNejEOSHrSAxKbB4IK6Her7jlThXsTUl/+Kovycbd7eQAo/EhWBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyP7ul12a7fbr8C79///dde4FleNztXuo9rDk654gRCNB3d+eUEjEzAbpbep43m8uv/fXnz7kfyRBzEqCb0vNudD4VIxliNgJ0ZX2uTs/lDGkQUxGgT9fnLum5kCENYh4CVKbnkAwxITciDlGf0/e/4x/aYFgmoLcdHv9fnZ4Lo5A5iHUzAY1Vn6MVzUGsmwCNVZ/TdTWIFROgs6r6jLA6fA8B+p/XcWOE4/91D4Yg1kqA/hn5OB95b3A1ARro0s8pF4NYNwE6Nk59xtwP3JEALWm4WMo+4YMEaAHjxpi7gtt5JOvCxort9snt0ev4KI/M+bGagJYxaIy8N7iaAAGZ2U/Bhrrz8LLnzWb/PVVnYe8a/9P8yIN6Z2ACAjICBGQECMhMHaAFXQDa8/VUVmbqAAEtAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMlMH6PWhFkt5JMLrPj2Og3WYOkBAS4CAjAABmdkDtKDLQC4AsT6zBwgICdA/Iw9BI+8NriZACzujWdZu4TIBWsCgMeau4HYCtKSxYin7hA8SoNHHjdH2A3ckQG8MF+Mc84c7Mf6wPgL0sIgjfOS9wdUEaNz7Et15yOoJ0Fltg0YoIHw1ARrxYpBLP0xCgIZrkPowDwH6UIO+J0NHC6kPqydAZx0d/1/doKP3Vx9m8LPewND2Fdhunw4b8bzZ3HcV6WFaJqBrRqF7TUOnb6U+TEWAPuS0Czdm6M2Xqw+zcQp25enY3mFE3j01uxAs6WFOAnSHDO1dNxBJDzMToIdbqvFmiT71DjAzAbrJYUfejZHowBEBuht9gc/yVzAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh4HhArVP1PbT7LBARkBAjIPO52L93qwNRMQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIeKj8BfDILUxYATjnAAAAAElFTkSuQmCC', 'sha256': 'bd4b18adaf0c8508e3fdec1d31629cbc8f54665b1a410e7f0c330e39fad5891b'}, 'data/binding-v1/images/binding-development-0013-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFCklEQVR4nO3dQU7jSBiAURg198kCiZvMIblJS7Pow7CI1JtZRKLTCQmQBL6y670tJFVS5E+/jWPut9uXO4DCP8mqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmR7f02mw2j+d/4dev/75rL7AM99vtS72HNUfnFDECAfru7hxTImYmQDdLz8PT8/nX/v7576kfyRBzEqCr0vNudD4VIxliNgJ0YX0uTs/5DGkQUxGgT9fnJuk5kyENYh4CVKZnnwwxITciDlGf4/e/4R/aYFgmoLftH/9fnZ4zo5A5iHUzAY1Vn4MVzUGsmwCNVZ/jdTWIFROgk6r6jLA6fA8B+svruDHC8f+6B0MQayVAf4x8nI+8N7iYAA106eeYi0GsmwAdGqc+Y+4HbkiAljRcLGWf8EECtIBxY8xdwfU8knVhY8Vm8+j26HV8lAfm/FhNQMsYNEbeG1xMgIDM7KdgQ915eN7D0/Pue6rOwt41/qf5kQf1zsAEBGQECMgIEJCZOkALugC04+uprMzUAQJaAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZKYO0OtDLZbySITXfXocB+swdYCAlgABGQECMrMHaEGXgVwAYn1mDxAQEqA/Rh6CRt4bXEyAFnZGs6zdwnkCtIBBY8xdwfUEaEljxVL2CR8kQKOPG6PtB25IgN4YLsY55vd3YvxhfQTobhFH+Mh7g4sJ0Lj3JbrzkNUToJPaBo1QQPhqAjTixSCXfpiEAA3XIPVhHgL0oQZ9T4YOFlIfVk+ATjo4/r+6QQfvrz7M4Ee9gaHtKrDZPO434uHp+barSA/TMgFdMgrdaho6fiv1YSoC9CHHXbgyQ2++XH2YjVOwC0/HdvYj8u6p2ZlgSQ9zEqAbZGjnsoFIepiZAN1dU403S/Spd4CZCdBV9jvyboxEBw4I0M3oC3yWv4IBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDG84BYoep/avNZJiAgI0BA5n67felWB6ZmAgIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwF3lf0GZKqRCx4IGAAAAAElFTkSuQmCC', 'sha256': '6aee5a3dc6c0078601ef25fc657c1c40a80ca6067e891332f067b43f4479d803'}, 'data/binding-v1/images/binding-development-0013-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFE0lEQVR4nO3dMU4jWRRAUTNq78dBS72JiXuRE7OJliboxRBYIpkAifbYYMA23F/1z0kB15NQXb0qyuZuv3/YABT+So4KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGS+dYdem93u+/lv+P3736+aBZbhbr9/qGdYc3ReI0YgQF/dnVNKxMwE6Gbpud9uz//s34+Pr31JhpiTAF2Vnjej86EYyRCzEaAL63Nxes5nSIOYigB9uD43Sc+ZDGkQ8xCgMj2HZIgJeRBxiPqcvv4N/9AGw7IBvezw/P/s9JxZhexBrJsNaKz6HB3RHsS6CdBY9Tk9rgaxYgL0qqo+IxwdvoYA/c/zujHC+f88gyWItRKgP0Y+z0eeDS4mQAPd+jnlZhDrJkDHxqnPmPPADQnQkpaLpcwJ7yRAC1g3xpwKrucjWRe2Vux23z0evY5f5ZE5f602oGUsGiPPBhcTICAz+yXYUE8enne/3T69T9VV2Ju2P/7ZLMTjr5+bidmAgIwAARkBAjJTB2hBN4CeeHsqKzN1gICWAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmakD9PyhFof/jn1kz3P6OA7WYeoAAS0BAjICBGRmD9CCbgO5AcT6zB4gICRAf4y8BI08G1xMgBZ2RbOsaeE8AVrAojHmVHA9AVrSWrGUOeGdBGj0dWO0eeCGBOiF5WKcc/5wEusP6yNAm0Wc4SPPBhcToHGfS/TkIasnQK9qGzRCAeGzCdCIN4Pc+mESAjRcg9SHeQjQuxr0NRk6OpD6sHoC9Kqj8/+zG3T0+urDDL7VAwztqQK73ffDRtxvt7c9ivQwLRvQJavQrbah05dSH6YiQO9y2oUrM/Tij6sPs3EJduHl2JPDiLx5aXYmWNLDnAToBhl6ctlCJD3MTIA211TjxRJ96BVgZgJ0lcOOvBkj0YEjAnQz+gIf5a9gQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ8XlArNDjr5/1CLyLDQjICBCQudvvH7qjA1OzAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYFP5DyTiLSQZ1pSEAAAAAElFTkSuQmCC', 'sha256': '32232a2c3df9b28a4ed52b305616ba6487f01828ac38df674ef3046e3b678ce2'}, 'data/binding-v1/images/binding-development-0014-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFBUlEQVR4nO3dQU4bSRiAURiF+7CIlFWukUPkQHOIXGNWkbLgMCwssYlGlixkGxtM219X93srEOAuhPrTX2Vb3G82z3cAhX+SqwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZL50l164x8evp7/h6enPrdYCM3W/2TzXa1hRdN4iRqyTAJXdOaRErIoAXSs9D99+nf7Zl98/3vqSDLESAjRles5G50MxkiEWT4AmSM/F3XlPiWSIBROgT9VnwvScyJAGsVQCdGF9rpSetzKkQSySAM1o8DlkFGLZBGi+9dnSoGu83OH21vyHO8FbMT627bpxfQ4vOvRNCHsEaEaHPidoEIvkvWDv0tZnt4YTr11clf/+fbkbxPefD/USZs0E9KbdoDGH+uytxBDEMgjQcfO/w+e/QjhLgGZ99HPIYRBLIkCnzK0+c14VXECABjj6OeQwiGUQICAjQOONP1uGIBZAgICMAA05/mwZghidAAEZAQIyAjTq/mvLLoyhCRCQESAgI0BARoAGPgDacgzEuAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARoP89Pf3ZfjDi/z7erXn3W8AoBAjICBCQESAgI0BjHwM5AGJoAgRkBAjICNDAuzD7L0YnQEBGgEYdgow/LIAAARkBGnIIMv6wDAJ0yjwbNM9VwQUE6IjXY8Xc7vbX6zH+MDoBOm7+9/b8VwhnCdBIh0GOflgYAXqXOTRoDmuAaQnQGIdBjn5YJAH6WINun6G9i6oPSyJA5+3d87ds0N611IeF+VIvYAzbO//x8ev205ffPx6+/br2RQ0+LJ4JaI7bMdsuVkKAPrsdmzZDhw+oPiyYLdhnt2Ov90oX78uOVkx6WDwBmixDex05G6MTo5P0sBICNH2Gti7bmkkPqyJAE9hV42iJPvQIsCoCNKXXHTkbI9EBAboWfYGzPA0PZExAjOf7z4d6CUzDBARkBAjI3G82z93VgVUzAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIOCu8hfU+TmTTNoRZwAAAABJRU5ErkJggg==', 'sha256': '96fd430ed5b1bd7fa3d450c00ebac76de686379bf13bccab4be91c1a496e73c7'}, 'data/binding-v1/images/binding-development-0014-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFCElEQVR4nO3dPWpbWRiA4XiIlzOgwm02NO2sI202lNaFIFtxIXAzDAJhLFnyj+z3nHufpzIk9v2CrZfvHGRys9s9fAMo/JU8FUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg87179MJtNnfn/8J2e/9Vs8Cgbna7h3qGFUXnJWLEOglQ2Z1jSsSqCNBnpef3r8fzn/vjn9uX/kiGWAkBumZ6LkbnTTGSIRZPgK6Qnnd35zUlkiEWTIA+VJ8rpudMhjSIpRKgd9bnk9LzUoY0iEUSoIEWn2NWIZZNgMatz54GfcbbHb7emr9xZ3gn9FjHrmP7hx4ytNnc+VFmMfwu2ND1Ofn0qRcBeMoG9CptfQ4znHnv4qo8/vv3t0nc/vxTjzA0G9CLDovGCPV5NokliGUQoNPGf4WPPyFcJEBDX/0ccxnEkgjQOaPVZ+Sp4B0EaIKrn2Mug1gGAQIyAjTf+rNnCWIBBAjICNCU68+eJYjZCRCQESAgI0Cznr/2nMKYmgABGQECMgIEZARo4gugPddAzEuAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAfrfdnu//2DG//v4MPPhXwGzECAgI0BARoCAjADNfQ3kAoipCRCQESAgI0ATn8Kcv5idAAEZAZp1CbL+sAACBGQEaMolyPrDMgjQOWM2aMyp4B0E6ISna8Vor/an81h/mJ0AnTb+a3v8CeEiAZrpMsjVDwsjQK8yQoNGmAGuS4DmuAxy9cMifa8HGN12e7/Z3D2twO9fj185wLPwqQ9LYgO67Nlr/itXIfVh2QToVbbb+68/jj07dqkPy+MINuJxzOLDSgjQ+xv0GRk63q3UhwUToDfbF+E4Qx8p0ckznfSweAJ0tQw968jFGJ25SJIeVkKArp+hj1xUSw+rIkBXcKjGyRK96SvAqgjQNT3tyMUYiQ4I0GfRF7jIGxGBjA2I+dz+/FOPwHXYgICMAAGZm93uoXs6sGo2ICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBHyr/AcUcSnwhT3AbAAAAABJRU5ErkJggg==', 'sha256': 'c870645bc1d9b47b062374599f352b6ecfc2c96cf7529ce6e01515e7143c32ec'}, 'data/binding-v1/images/binding-development-0014-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFBElEQVR4nO3dQUojWxiAUX20y3ngoKF30ht56+iN9E4aeiD0VhwEnDSPQJAkJhor+epWnTNS1NQVqY//3iR4v9k83wEU/kmuCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dJdeuEeH7+e/oanp9+3WgvM1P1m81yvYUXReYsYsU4CVHbnkBKxKgJ0rfQ8fPt5+mdffn1/60syxEoI0JTpORudD8VIhlg8AZogPRd35z0lkiEWTIA+VZ8J03MiQxrEUgnQhfW5UnreypAGsUgCNKPB55BRiGUToPnWZ0uDrvFyh9tb8x/uBG/F+Ni268b1Obzo0Dch7BGgGR36nKBBLJL3gr1LW5/dGk68dnFVXv77924QDz/+1EuYNRPQm3aDxhzqs7cSQxDLIEDHzf8On/8K4SwBmvXRzyGHQSyJAJ0yt/rMeVVwAQEa4OjnkMMglkGAgIwAjTf+bBmCWAABAjICNOT4s2UIYnQCBGQECMgI0Kj7ry27MIYmQEBGgICMAAEZARr4AGjLMRDjEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaA/vf09Hv7wYj/+3i35t1vAaMQICAjQEBGgICMAI19DOQAiKEJEJARICAjQAPvwuy/GJ0AARkBGnUIMv6wAAIEZARoyCHI+MMyCNAp82zQPFcFFxCgI16PFXO721+vx/jD6ATouPnf2/NfIZwlQCMdBjn6YWEE6F3m0KA5rAGmJUBjHAY5+mGRBOhjDbp9hvYuqj4siQCdt3fP37JBe9dSHxbmS72AMWzv/MfHr9tPX359f/j289oXNfiweCagOW7HbLtYCQH67HZs2gwdPqD6sGC2YJ/djr3eK128LztaMelh8QRosgztdeRsjE6MTtLDSgjQ9BnaumxrJj2sigBNYFeNoyX60CPAqgjQlF535GyMRAcE6Fr0Bc7yNDyQMQExnocff+olMA0TEJARICBzv9k8d1cHVs0EBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgLvKX2yGOkz0Y6X3AAAAAElFTkSuQmCC', 'sha256': '8272e9f6ea5e255a4322b040a470e26c073e4a57feccc141e1c8e47ba343b7b6'}, 'data/binding-v1/images/binding-development-0015-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUklEQVR4nO3bS2pbSRiAUadJlhPwwNA7yUayjmykdxLwQJCteGDwpAcGocix5MiP71bVOSODH/cX+H6uWyV/ur+/uwIo/JNcFUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5nN36YFdX9+c/oLd7vajZoGBfbq/v6tnmKE4p+kR/JEAvW96DskQHBGgS7rz8P3r6W//8uPXic8qETwSoJem52x0/jZGMgQCdKY+F3fnJSXSIBYnQB+RnkMyBHsC9If6vFN6TmRIg1iTAP1Wnw9Iz3MZ0iAWtHqAwvo80qBtvmdim3bT/YYs/a8YeX2Orjv9/QNH1v1XjP3dXqVn73GAx6XQ9fXNfH/l4DmLBmg79dl7+P5Vg97Kl3//u5rIw89vV5Na8RFsg/U5msezGItYLkCj3NujzAmvsVaAtrDrfJo9aZayVoA2Xp/tzwZva6EAbXbr5ymbQSxilQCNeyePOzmctUqA9ra//BlrTniNJQI0+iJi9Plh6QANuqwYa1q4wFoBAjZl/gANdPj1lOMw5jZ/gIDNmjxAMy0cZnotsESA9kZ8/hp9cjhrlQABGyRAQEaAgMzMARr6AP6Qw3hmNXOAgI0TICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgM3OAdrvbxw++/Ph1NbL9/PtXBHOYOUDAxgkQkBEgILNKgMbdBhp3clg9QDPt2s70WmCJAAFbNn+Ahj6MdwDP3OYPELBZawVorEXQWNPCBZYI0OjPL6PPD0sHaMRlxShzwmusEqBxFxHjTg5nrRKgsY7DHH6xiIUCdGjLDdrybPC21grQ4YJim/f54VSWP0xvrQANdFePMie8xnIB2vJmkK0fVrNigLbZIPVhQZ+vVrXb3V5f3+zv/IfvX6tJ7PuwrEVXQNvZk1YfVrZ0gPIGqQ+LW/cR7Omz2Ec+jh3FTn1YkwAdN+i9M/R0naU+LEuAfqvAu2ZIeuCIAJ1aCh1W4+ISPbe1pD4gQOeXQk87cjZGp/ezpQceCdCZRjwt0cXnZboDRwTowgXRBT8EOCJAlxTkbI8UB15CgC6hL/AmVn8nNBASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ8U5oJvTw81s9Ai9iBQRkBAjIfLq/v+uuDizNCgjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIuKr8D+6QQ6TPJ0dhAAAAAElFTkSuQmCC', 'sha256': 'b53925379229e2c49821edb99de5eeba8d7ebe9520e92bcdb09b5c9c88b4dbc4'}, 'data/binding-v1/images/binding-development-0015-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFXElEQVR4nO3bvW1bSRhAUXlh9aPAgCK34SJc0BahNjZawAGLcSDAyQYECJq0SS0l676ZOScS9MP3EdC7GM6QH56fv98BFP5KrgogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPOxu/TAHh4+Xf6F3e7be80CA/vw/Py9nmGG4lymR/BLAvRn03NMhuCEAN3SnfvHp8t//uPfLxd+qkSwJ0AvTc/V6PzfGMkQCNCV+tzcnZeUSINYnAC9R3qOyRAcCNAv6vOH0nMhQxrEmgTop/q8Q3p+lyENYkGrByisz54GbfM9E9u0m+4/ZOmPYuT1Obnu9PcPnFg3QIe7/f7xqarP+QAaxFIW/SzYcX3utuH+8Wn/cuzh4dN8K+139s/fP+4m8vnr/d2kVlwBbbA+e9ZBrGa5AI1yb48yJ7zGWgHawq7zZfakWcpaAdp4fbY/G7ythQK02a2fczaDWMQqARr3Th53crhqlQAdbH/5M9ac8BpLBGj0RcTo88PSARp0WTHWtHCDtQIEbMr8ARro8Ouc4zDmNn+AgM2aPEAzLRxmei6wRIAORnz9NfrkcNUqAQI2SICAjAABmZkDNPQB/DGH8cxq5gABGydAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBm5gDtdt/2X/z498vdyA7zH54RzGHmAAEbJ0BARoCAzCoBGncbaNzJYfUAzbRrO9NzgSUCBGzZ/AEa+jDeATxzmz9AwGatFaCxFkFjTQs3WCJAo79+GX1+WDpAIy4rRpkTXmOVAI27iBh3crhqlQCNdRzm8ItFLBSgY1tu0JZng7e1VoCOFxTbvM+Pp7L8YXprBWigu3qUOeE1lgvQljeDbP2wmhUDtM0GqQ8LWjRAJw1qM3Q8gPqwlHUDtJE9abvOrGzpAOUNUh8Wt3qAzhv0Phk6uZD6sCYB+sX9/0czdP7g6sOyPtYDbMW+Ag8Pnw7f2Wfi/vHprS5xHjXpYXEC9JPd7ttxg46rcXOJfreYUh8QoOtLofOOXI3R5Vdw0gN7AnSlEecluvm8THfghADduCC64UGAEwJ0S0Gu9khx4CUE6Bb6Am/C+4CAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIx3QjOhz1/v6xF4ESsgICNAQObD8/P37urA0qyAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIC7yn8xVE3/9m7PAgAAAABJRU5ErkJggg==', 'sha256': 'fa630906a8eb8aae54c63262492a1cc02d662570553945eaadb519109befbbd6'}, 'data/binding-v1/images/binding-development-0015-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUUlEQVR4nO3bPU5bSxiAYXIVloNEQZsNpWUdae+GbpvCUraSAonmFkiWYxKbmJ/3zMzzVEj8nM8S52XOjPn08PDzCqDwT3JVAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZz92lB3Z7e3f6C3a77x81Cwzs08PDz3qGGYpzmh7BbwnQ+6bnkAzBEQG6pDuP9zenv/36248Tn1UieCJAL03P2ej8bYxkCAToTH0u7s5LSqRBLE6APiI9h2QI9gToN/V5p/ScyJAGsSYB+qU+H5CeP2VIg1jQ6gEK6/NEg7b5nolt2k33G7L0v2Lk9Tm67vT3DxxZ918x9nd7lZ69pwGelkK3t3fz/ZWDP1k0QNupz97j/Y0GvZX//n28msiXr9dXk1rxEWyD9Tmax7MYi1guQKPc26PMCa+xVoC2sOt8mj1plrJWgDZen+3PBm9roQBtduvnOZtBLGKVAI17J487OZy1SoD2tr/8GWtOeI0lAjT6ImL0+WHpAA26rBhrWrjAWgECNmX+AA10+PWc4zDmNn+AgM2aPEAzLRxmei2wRID2Rnz+Gn1yOGuVAAEbJEBARoCAzMwBGvoA/pDDeGY1c4CAjRMgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAzc4B2u+9PH1x/+3E1sv38+1cEc5g5QMDGCRCQESAgs0qAxt0GGndyWD1AM+3azvRaYIkAAVs2f4CGPox3AM/c5g8QsFlrBWisRdBY08IFlgjQ6M8vo88PSwdoxGXFKHPCa6wSoHEXEeNODmetEqCxjsMcfrGIhQJ0aMsN2vJs8LbWCtDhgmKb9/nhVJY/TG+tAA10V48yJ7zGcgHa8maQrR9Ws2KAttkg9WFBn69Wtdt9v72929/5j/c31ST2fVjWoiug7exJqw8rWzpAeYPUh8Wt+wj2/FnsIx/HjmKnPqxJgI4b9N4Zer7OUh+WJUC/VOBdMyQ9cESATi2FDqtxcYn+tLWkPiBA55dCzztyNkan97OlB54I0JlGPC/RxedlugNHBOjCBdEFPwQ4IkCXFORsjxQHXkKALqEv8CZWfyc0EBIgICNAQEaAgIwAARkBAjICBGQECMgIEJDxTmgm9OXrdT0CL2IFBGQECMh8enj42V0dWJoVEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBBwVfkfsp1CNlA5UdEAAAAASUVORK5CYII=', 'sha256': 'a4f249556f490fa2b58dfcceb73ae722cfd1a10a4fe32eb6f2aed7572be1b2da'}, 'data/binding-v1/images/binding-evaluation-0000-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFXklEQVR4nO3bMW4TTRiAYYPIfVIg5RJ/zSGp/0sgUfgwFJHSUESyLNsJgTh+d2aep0qReL8V3lczu8unx8dfO4DC5+SoAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMl+7Q8FHu77/uprbf/9xNwQoIyAgQkLEFY2b/393tJvLf09NuLlZAQEaAgIwt2HWeqkzzVAJuSYCu8xz35Pf1CN5CgD7k/ZHD5ygRvEKA3pSeu4fvr//5049vr3ysDMFFAvRiev4YnZd++TxGMgQXLR2gi+n5q+5cdPiEkxLJEJxYN0An9Xl/d95Yovv7rxoES78HdIP6vPL50/9XSXij5VZAN07PyYEOSyHbMVhuBVTV56UjWgqxuIUCdHy13z18v319Lh5ag1jZ5zXrk85yOoMGsawlArS1+jzTIJg/QNuszzMNYnHzB2iz9dnyVHAbkwfosKzY8nV+mM0iiNXMHKARr+cRZ4Z/Nm2Atnzr55ybQaxp2gANVJ+x5oQrmjNAoy8iRp8flg7QoMuKsaaF95swQEM8+XqJJ2IsZcIAAaOYLUAzLRxmOhdYIkAHI+6/Rp8c/ta0AQK2b6oAzbdnme+MYNoATbOLGX1+WDpAwBAECMgIEJCZJ0BDvwB9zivRrGCeAAHDESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIDNPgPb7n88/PP34thvf4SwO5wXzmSdAwHAECMgIEJCZM0Cj3wYafX5YMUDz3a+d74xg2gABY5k2QOPuYsadHFYP0Ex7lpnOBZYIEDCQCQM09CvRXoBmKRMG6NhYDRprWni/OQM0+vJh9Plh6QCNuKwYZU64omkDdLyI2P61fTyh5Q/rmDZAg17JI84M/2zmAI3yRMyTL5Y1eYCObbNB25wKbmP+AG35ZpBbPyxu/gBttkHqA0sEaIMNUh9YKEDnDaoydHJo9WFlCwXo/Gq/fYNOjqg+LO7LbjHP1/z9/dfjItw9fP/o40oPnPv0+Phrt6RDg559aIPUp/3Hnc9+lq/Qciugg/3+5/HX9NCIK5bo4hZvmq8OvN+6ATrfjl2rRC/dWpIeOLHuFuyvFu1/jNHr97OlBy4SoI+9dyA98AoB+pAS6Q68hQBdp0eKA/9AgIDMWm9CA5siQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKv8BhJtLYmO8ic2AAAAAElFTkSuQmCC', 'sha256': '153ecda8e17e4ffd77c354b8ce926e7fdd9505e177aca725e8da824cc8a6a130'}, 'data/binding-v1/images/binding-evaluation-0000-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFXklEQVR4nO3bMW7bSBiAUSWI7uMiQC6Reg+5dS6xQAodJoUBN1sYEARJdpxY1seZea9yYYs/EfHDDMl8enz8tQMofE6OCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfOkODR/l4eHrbmqHw8/dFKyAgIwAARlbMGa2//bvbiJP//2zm4sVEJARICBjC3abpyrTPJWAexKg2zzHPft9PYK3EKAPeX/k+DlKBK8QoDel58d+//qff396euVjZQiuEqAX0/Pb6Lz0y5cxkiG4aukAXU3PH3XnquMnnJVIhuDMugE6q8/7u/PGEj08fNUgWPo9oDvU55XPn/6/SsIbLbcCunN6zg50XArZjsFyK6CqPi8d0VKIxS0UoNOr/cd+f//6XD20BrGyz2vWJ53lfAYNYllLBGhr9XmmQTB/gLZZn2caxOLmD9Bm67PlqeA+Jg/QcVmx5ev8OJtFEKuZOUAjXs8jzgx/bdoAbfnWzyU3g1jTtAEaqD5jzQk3NGeARl9EjD4/LB2gQZcVY00L7zdhgIZ48vUST8RYyoQBAkYxW4BmWjjMdC6wRICORtx/jT45/KlpAwRs31QBmm/PMt8ZwbQBmmYXM/r8sHSAgCEIEJARICAzT4CGfgH6kleiWcE8AQKGI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQGaeAB0OP59/+P70tBvf8SyO5wXzmSdAwHAECMgIEJCZM0Cj3wYafX5YMUDz3a+d74xg2gABY5k2QOPuYsadHFYP0Ex7lpnOBZYIEDCQCQM09CvRXoBmKRMG6NRYDRprWni/OQM0+vJh9Plh6QCNuKwYZU64oWkDdLqI2P61fTqh5Q/rmDZAg17JI84Mf23mAI3yRMyTL5Y1eYBObbNB25wK7mP+AG35ZpBbPyxu/gBttkHqA0sEaIMNUh9YKECXDaoydHZo9WFlCwXo8mq/f4POjqg+LO7LbjHP1/zDw9fTIvzY7z/6uNIDlz49Pv7aLenYoGcf2iD1af9x53OY5Su03Aro6HD4efo1PTbihiW6usWb5qsD77dugC63Y7cq0Uu3lqQHzqy7BfujRftvY/T6/WzpgasE6GPvHUgPvEKAPqREugNvIUC36ZHiwF8QICCz1pvQwKYIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE7Cr/A6wiLRFr2j/3AAAAAElFTkSuQmCC', 'sha256': '516c31f91b1f7b15c3d21b7b6856042b7a587285c37987258607327241092b64'}, 'data/binding-v1/images/binding-evaluation-0000-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUklEQVR4nO3bTU4bMQCA0bQq92HRY/S4PUYXHKYLJDZdRIqiSaD8BL6x/d4KIZTxSMyHPTbfHh//HgAK35OrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyP7pLw2e5v/95mNrDw5/DFMyAgIwAARlLMGb2++7uMJFfT0+HuZgBARkBAjKWYLfZVZlmVwK+kgDdZh938/N6BK8hQJ9yfuT0OUoELxCgV6Xnv5spz21PHD9WhuAqAXo2PW/awT3/4csYyRBctXSArqbn4ydHTp+wKZEMwca6AdrU5zNOrF0t0f39Tw2Cpc8BfUF9Xvj86f9VEl5puRnQF6dnc6HTVMhyDJabAVX1ee6KpkIsbqEAnT/tv+/uqn9T3Fxag1jZ9zXrk45lOwYNYllLBGhv9TnSIJg/QPusz5EGsbj5A7Tb+ux5VPA1Jg/QaVqx5+f8NDaTIFYzc4BGfJ5HHDO827QB2vOrn0teBrGmaQM0UH3GGifc0JwBGn0SMfr4YekADTqtGGu08HETBmiIna/n2BFjKRMGCBjFbAGaaeIw073AEgE6GXH9NfrI4a2mDRCwf1MFaL41y3x3BNMGaJpVzOjjh6UDBAxBgICMAAGZeQI09AHoS45Es4J5AgQMR4CAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMw8AXp4+HP84tfT02F8p7s43RfMZ54AAcMRICAjQEBmzgCN/hpo9PHDigGa733tfHcE0wYIGMu0ARp3FTPuyGH1AM20ZpnpXmCJAAEDmTBAQx+JdgCapUwYoHNjNWis0cLHzRmg0acPo48flg7QiNOKUcYJNzRtgM4nEft/ts9HaPrDOqYN0KBP8ohjhnebOUCj7IjZ+WJZkwfo3D4btM9RwdeYP0B7fhnk1Q+Lmz9Au22Q+sASAdphg9QHFgrQZYOqDG0urT6sbKEAXT7tX9+gzRXVh8X9OCzm+Mzf3/88L8Lvu7vPvq70wKXlAnT08PDn1KBjHT61QepT2cP7Pl6waICuNuj4xQ1LdPW3X33gZN0AXS7HblWi5/7qSg9sfHt8/Lv93pI2Gdr4b4xenupLD1wlQG/I0DtID7xAgD6lRLoDryFAt+mR4sA7CBCQWeskNLArAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAYfKP98XLiD/TIHbAAAAAElFTkSuQmCC', 'sha256': 'd8e64a07ded4688a64a0699835a315597b35255e8ff1342e9ecbd9dfb8fa7f78'}, 'data/binding-v1/images/binding-evaluation-0001-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFZ0lEQVR4nO3aQWobWRRAUaVJlhPwINA7yUayjmykdxLIQJCtZGDwpAcGIWRZVmxHt/7/54wCSVSvQHV5VaoP9/e/dwCFf5KjAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZD52hx7J3d2Xy/9gv/95q1lgHh/u73/XMwxZnMv0CK4hQO/ZnaeUCC4QoKvS8/Dt8+X//un7rwt/K0NwlgA9m54Xo/OnMZKh7W+s27Ff49uy9EPos1/fV3fn6SeclOjxcIt8seAa6wbopD5v786VJbq7+6JBsHSAblCfk8/XoL/q07//7abw8OPrbiXLBejG6Tk50CFDbsdguTehq/o8d8SJn6HCNRbagI6v9tun58IqZA9iWatsQBupz9kZ7EEsa4kAba0+jzQI5g/QNuvzSINY3PwB2mx9tjwV3MbkATqsFVu+zg+zWYJYzcwBGvF6HnFmeLVpA7TlRz9PeRjEmqYN0ED1GWtOeEdzBmj0JWL0+WHpAA26Vow1LbzdhAEa4pev5/hFjKVMGCBgFLMFaKbFYaZzgSUCdDDi/dfok8OfmjZAwPZNFaD57lnmOyOYNkDT3MWMPj8sHSBgCAIEZAQIyMwToKFfgH7KK9GsYJ4AAcMRICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgM0+A9vufj3/49P3XbnyHszicF8xnngABwxEgICNAQGbOAI3+GGj0+WHFAM33vHa+M4JpAwSMZdoAjXsXM+7ksHqAZrpnmelcYIkAAQOZMEBDvxLtBWiWMmGAjo3VoLGmhbebM0Cjrw+jzw9LB2jEtWKUOeEdTRug4yVi+9f28YTWH9YxbYAGvZJHnBlebeYAjfKLmF++WNbkATq2zQZtcyq4jfkDtOWHQR79sLj5A7TZBqkPLBGgDTZIfWC3233cLWO//3l39+X4+n/49vn2Y5zkT31Y2Sob0Nmr/farkPrAohvQ8TV/+1VIemD1DahahdQHzlpuAzr7SOi4Ee+4DZ3tmvrAwboBeno79l4lem6fkh44sXSALmTopCMvxujyTZz0wFkC9EKG3viQSHrgAgE634vnSvSnnwNcIEBXFeTFHikOvIIAXUVf4G8QIGbw8ONrPQKvseiLiMAWCBCQ+XB//7s7OrA0GxCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKv8D6WyPFPpx3OaAAAAAElFTkSuQmCC', 'sha256': 'dc0319181b4feec251f7e2d485de3452961657d76c6d0e0a4a65e17f719ce44b'}, 'data/binding-v1/images/binding-evaluation-0001-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFZklEQVR4nO3aQW4TSxRA0YDwcr6UActgN38d7IZlMIjEVhhEyoRBJMuyHcckwber6pwREuB6Lbmvqsv96fHx9x1A4XOyKoAAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZLt/RI7u+/Xv4HDw8/bzULzOPT4+PveoYhi3OZHsE1BOgju3NKieACAboqPT92u8v//dvT04W/lSE4S4BeTM+r0fnbGMnQ9nes2/Gwxrdl6UPos1/fN3fn9BOOSvS83CJfLLjGugE6qs/7u3Nlie7vv2oQLB2gG9Tn6PM16J96+v+/uynsvv+6W8lyAbpxeo4W2mfI4xgs9yZ0VZ+XVpz4DBWusVCADu/2H7vd7etzdmkNYmWf16xPOsvxDBrEspYI0Nbq80yDYP4AbbM+zzSIxc0foM3WZ8tTwW1MHqD9tmLL9/l+NpsgVjNzgEa8n0ecGd5s2gBt+ejnlMMg1jRtgAaqz1hzwgeaM0CjbyJGnx+WDtCg24qxpoX3mzBAQ/zy9RK/iLGUCQMEjGK2AM20cZjpWmCJAO2N+Pw1+uTwt6YNELB9UwVovmeW+a4Ipg3QNE8xo88PSwcIGIIAARkBAjLzBGjoF6BPeSWaFcwTIGA4AgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZOYJ0MPDz+c/fHt6uhvf/ir21wXzmSdAwHAECMgIEJCZM0CjHwONPj+sGKD5zmvnuyKYNkDAWKYN0LhPMeNODqsHaKZnlpmuBZYIEDCQCQM09CvRXoBmKRMG6NBYDRprWni/OQM0+vZh9Plh6QCNuK0YZU74QNMG6HATsf17+3BC2x/WMW2ABr2TR5wZ3mzmAI3yi5hfvljW5AE6tM0GbXMquI35A7TlwyBHPyxu/gBttkHqA0sEaIMNUh9YKECnDaoydLS0+rCyhQJ0erffvkFHK6oPi/tyt5jne/7+/uthEX7sdv96XemB1XdA1VZIfeCs5XZAew8PP/f7oMNGfOBu6GzX1Af21g3Q6ePYR5Xopf2U9MCRpQN0IUNHHXk1Rpcf4qQHzhKgVzL0zkMi6YELBOh8L14q0d9+DnCBAF1VkFd7pDjwBgJ0FX2Bf0GAmMHu+696BN5i0RcRgS0QICDz6fHxd7c6sDQ7ICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgrvIH9D8+Ma7kdVoAAAAASUVORK5CYII=', 'sha256': 'c235cba91d49240e0924d258eba45382700844941f1ad62805e54af5bb46ec8d'}, 'data/binding-v1/images/binding-evaluation-0001-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFZUlEQVR4nO3aMW4TTRiA4YDwfVIgcQlqDvnXXAKJIoehiJTmLyJZlu04Jgl+d2aep0ICPN9K3lez4/30+PjnDqDwOVkVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg86VbeiT3918v/4OHh9+3mgXm8enx8U89w5DFuUyP4BoC9JHdOaVEcIEAXZWen7vd5f/+/enpwt/KEJwlQC+m59Xo/G2MZGj7O9bteFjj27L0IfTZr++bu3P6CUclel5ukS8WXGPdAB3V5/3dubJE9/dfNQiWDtAN6nP0+Rr0T+2+/Xc3hadfP+5WslyAbpyeo4X2GfI4Bsu9CV3V56UVJz5DhWssFKDDu/3nbnf7+pxdWoNY2ec165POcjyDBrGsJQK0tfo80yCYP0DbrM8zDWJx8wdos/XZ8lRwG5MHaL+t2PJ9vp/NJojVzBygEe/nEWeGN5s2QFs++jnlMIg1TRuggeoz1pzwgeYM0OibiNHnh6UDNOi2Yqxp4f0mDNAQv3y9xC9iLGXCAAGjmC1AM20cZroWWCJAeyM+f40+OfytaQMEbN9UAZrvmWW+K4JpAzTNU8zo88PSAQKGIEBARoCAzDwBGvoF6FNeiWYF8wQIGI4AARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZeQL08PD7+Q/fn57uxre/iv11wXzmCRAwHAECMgIEZOYM0OjHQKPPDysGaL7z2vmuCKYNEDCWaQM07lPMuJPD6gGa6ZllpmuBJQIEDGTCAA39SrQXoFnKhAE6NFaDxpoW3m/OAI2+fRh9flg6QCNuK0aZEz7QtAE63ERs/94+nND2h3VMG6BB7+QRZ4Y3mzlAo/wi5pcvljV5gA5ts0HbnApuY/4AbfkwyNEPi5s/QJttkPrAEgHaYIPUBxYK0GmDqgwdLa0+rGyhAJ3e7bdv0NGK6sPivtwt5vmev7//eliEn7vdv15XemD1HVC1FVIfOGu5HdDew8Pv/T7osBEfuBs62zX1gb11A3T6OPZRJXppPyU9cGTpAF3I0FFHXo3R5Yc46YGzBOiVDL3zkEh64AIBOt+Ll0r0t58DXCBAVxXk1R4pDryBAF1FX+BfECBm8PTrRz0Cb7Hoi4jAFggQkPn0+PinWx1Ymh0QkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEHBX+R8r1D7mYF55TQAAAABJRU5ErkJggg==', 'sha256': 'e962223cf6c4f010b61c761557511691f8737b9c21335520b937cd157b79a248'}, 'data/binding-v1/images/binding-evaluation-0002-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE6UlEQVR4nO3dPW4bVxSAUTqwlxNAhdtsKG3WkTYbSuuCQLbiQoCbFAQIQuJPBA/5PeaeU4uaV2g+vHc10nx6ff2+Ayj8klwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZz92lYTMvL193T2u//7abyg4IyAgQkHEE43/lxx+/7p7Elz//qZfQswMCMgIEZAQIyAgQkBEgICNAQMav4Td+4nbyU63wUQK08WP+p18sRnCdAN3xL4wOH5chuESAPtadv//6cf2zv/3+5dJ3UyJ4Q4Bu1+dmdC598ZsYvbx81SA4JUCbpee9w8dPM+RQBqcE6Hx9fjI9NzOkQSBAZ+qzYXquZEiDwIOID6rP2e//1P/EDzYxPUCPrM/7q2gQw40+gh3v/8ek5+xxzFmMyebugKr6HB2vax/EWEMDtNo9v9p64DEmBujxc59LzIMYbmKAFqnPOmuAyrgA5aOf9wyDGGtWgNa/w9dfIWxoVoCO1tn+rLkeeIyhAQJWMChAC05/TpkEMdCgAAGrmRKgxbc/BzZBTDMlQMCCBAjICBCQGRGgpxgAHRgDMcqIAAFrEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIDMiQMdXjx5eRrqy4wq9LpUJRgQIWJMAARkBAjJTAvQUYyADIKaZEiBgQYMCtPgmyPaHgQYFCFjN0ACttglabT3wGLMCtP7pZv0VwoZmBWjNSZDpD2ONC9CpFRq0whqgMjFApxuN9v4/vbrtDwNNDNCCd/tq64HH+PT6+n03VfvCQqOfDT31exz3g38APu8G2++/HX5wDy14WIacvGD0ESycB6kPHI0+gp3dwN91H6Q+cEqAzg8RNs/Qm+2V+oAA3RhkbpKh9yc79YEDAbpjhqQHrhOgj/1O92aMrkyy1QfeEKC7P12iO3CJAN0xQ9ID1wnQxjESHfjvBAjITH8SGggJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwK7yL1nkCeULrZRvAAAAAElFTkSuQmCC', 'sha256': '7cf6940e6e132555c384a40753d6013ec2cad2c54ec585e95d75eeaa440cb2d0'}, 'data/binding-v1/images/binding-evaluation-0002-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE7ElEQVR4nO3dMW4bVxRAUTqI9qPCrXfglWRBWYl3kNaFFuNCAJsUBIQBJUoRMuT99DunUiFyfqG5+P9xJH15fv51ACj8kVwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZP7tLw24eH78e7tbT08/DVHZAQEaAgIwjGL+Vf/4+Hu7Et78e6iX07ICAjAABGQECMgIEZAQIyAgQkPEx/M5P3E5+qhU+S4B2fsx/+81iBO8ToCv+htHp5TIElwjQ57rz4+GDp1e/H4+X3k2J4IwAfVyfD6Nz6ZvPYvT4+FWDYEuAdkvPa6eXbzPkUAZbPoZ/uz4/Hh7+Z33eeau7/uM1sCMBOi/CjunZOntbDQIBeqM+V72WBsHW9ADdsj6vr6JBDDc6QC/3/5WOXZdsL6dBTDY3QNv6JAvQIBgaoNXu+dXWA7cxMUC3n/tcYh7EcBMDtEh91lkDVMYFKB/9vGYYxFizArT+Hb7+CmFHswL0Yp3tz5rrgdsYGiBgBYMCtOD0Z8skiIEGBQhYzZQALb79ObEJYpopAQIWJEBARoCAzIgA3cUA6MQYiFFGBAhYkwABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZkRAXp6+nn64vvxeFjbywpf1gy/sREBAtYkQEBGgIDMlADdxRjIAIhppgQIWNCgAC2+CbL9YaBBAQJWMzRAq22CVlsP3MasAK1/ull/hbCjWQFacxJk+sNY4wK0tUKDVlgDVCYGaLvRaO//7dVtfxhoYoAWvNtXWw/cxpfn51+Hqdp/WGj0s6O7/j+OT4N/AIbugF4PpG95FttebvIPH4wOUDIPMveBF6OPYG9u4K96HFMf2BKgt4cIu2fobHulPiBAHwwyd8nQ65Od+sCJAF0xQ9ID7xOgz32m+2GM3plkqw+cEaCrP12iO3CJAF0xQ9ID7xOgnWMkOvDfCRCQmf4kNBASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgEPlX4GJ9rpFWxjRAAAAAElFTkSuQmCC', 'sha256': '29fe45d6cd6c73d50d83082b4fdf64076d4c4d78ee6014061039d4e7c772bf92'}, 'data/binding-v1/images/binding-evaluation-0002-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE7ElEQVR4nO3dMW4bVxRAUTqIlhNAhZfh3WQd2Y2X4UJAtuJCgBoXBIQBJVIRMuT99DunUiFyfqG5+P9xJH15fv55ACj8kVwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZP7tLw24eH78e7tbT04/DVHZAQEaAgIwjGL+Vl7//OtyJh3/+rZfQswMCMgIEZAQIyAgQkBEgICNAQMbH8Ds/cTv5qVb4LAHa+TH/7TeLEVwmQFf8DaPjy2UIzhGgz3Xn+8PD5dd+e3k5925KBCcE6OP6fBidc998EqPHx68aBFsCtFt63jq+fJshhzLY8jH8+/X5/vDwP+tz4a3u+o/XwI4E6LQIO6Zn6+RtNQgE6J36XPVaGgRb0wN0y/q8vYoGMdzoAL3e/1c6dp2zvZwGMdncAG3rkyxAg2BogFa751dbD9zGxADdfu5zjnkQw00M0CL1WWcNUBkXoHz085ZhEGPNCtD6d/j6K4QdzQrQq3W2P2uuB25jaICAFQwK0ILTny2TIAYaFCBgNVMCtPj258gmiGmmBAhYkAABGQECMiMCdBcDoCNjIEYZESBgTQIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGRGBOjp6cfxi28vL4e1va7wdc3wGxsRIGBNAgRkBAjITAnQXYyBDICYZkqAgAUNCtDimyDbHwYaFCBgNUMDtNomaLX1wG3MCtD6p5v1Vwg7mhWgNSdBpj+MNS5AWys0aIU1QGVigLYbjfb+317d9oeBJgZowbt9tfXAbXx5fv55mKr9h4VGPzu66//j+DT4B2DoDujtQPqWZ7Ht5Sb/8MHoACXzIHMfeDX6CPbuBv6qxzH1gS0Ben+IsHuGTrZX6gMC9MEgc5cMvT3ZqQ8cCdAVMyQ9cJkAfe4z3Q9jdGGSrT5wQoCu/nSJ7sA5AnTFDEkPXCZAO8dIdOC/EyAgM/1JaCAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAYfKLwX29rpWUqcrAAAAAElFTkSuQmCC', 'sha256': 'a995812b0a1cf6e41c55fdf7b7a4c39dbc754811d6145b7f130681cfa0602e3b'}, 'data/binding-v1/images/binding-evaluation-0003-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIklEQVR4nO3cQU4bSRiAUWcU7uMFUla5BofIgeYQucasImXBYbJAYjMLSx4PECcGh6+q670VSNBdSPSnvwqbDw8PP3YAhb+SuwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnY3RquZr+/3U3r/v77blUmICAjQEDGFoxN+efvx90kPn+5qZfQMwEBGQECMgIEZAQIyAgQkBEgICNAQMbrgN7v3QArv+IeXiRA7/cWpNMvFiMQoOytj8crKBErE6Crdefm09fz3/747e7MNZWIBQnQm+rzy+j87Iufx2i/v9UgViNAr0nPRd150fEKpyU63EiGWIc/w19Wn5tPX99en/MXnPp/a8FFTEAXpOfP3etw8eM0ZBRiESagvj4/u4tRiM0ToBecPvlX33Od9+R2GsS2CdAv6pOsQYNYhAANV5/nd9cgtkqARqzP8zVoEJskQIPW50CD2DYBemqc+oy5HrgiAfrfcDHm035clSGIjRGgyZ7quVYL5wnQf8Ycf8ZfG7za6gEafPN1ykaM7Vk6QPM+yfOuHE4tHaCj8cefudYJv0mAgMy6AZro9OeUkyC2ZN0AATkBAjKLBmjS/deBXRibsWiAgBEIEJARICCzYoCmPgA6cAzENqwYIGAQAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZFYM0P3998MHj9/udnM6rvz4s8CMVgwQMAgBAjICBGQWDdDUx0AOgNiMRQMEjECAgMy6AZp0F2b/xZasGyAg97FewBAev91N8f+h5xrWEp+/3NRL4AJLT0Dz7mLmXTmcWjpAc50EOf1he2zB5tiIjd/HlihPavUJaLrf3blWC+cJ0AQbMZsvtkqAnhqtQaOtB65IgHbPh4txnvnTlRh/2B4BGrdB6sPmCdCgDVIfViBAIzZIfVjEh4eHH/UaRrTf355++m6vD3qSPPVh20xAL3vy5L/PKKQ+rMYENMQoJD2sSYAubtB1M/R8tlIf1iFAr8/QW0r04p5OeliNAL21Qb8fo/MHSerDggTo+iW6iO6wMgFqSqQ7IEDvGiPRgScECMh4ISKQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgF3lXwDiF5KuK/N5AAAAAElFTkSuQmCC', 'sha256': '582c3a5ba830d1c6c09970b01296855b4049ca7b340ce7a59daed60af1f97be9'}, 'data/binding-v1/images/binding-evaluation-0003-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEUlEQVR4nO3cQU4bSRiAUWcU7uMFx+C4OUYWHCYLJG9mYcnjsRkSg8dfVdd7K5CQuxKlP/1V6ebb29uvHUDhr+SqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkvneXhrvZ759303p9/blblQkIyAgQkLEFY1N+PD3tJvFyONRL6JmAgIwAARkBAjICBGQECMgIEJARICDjOaDHvQ2w8hP38C4BetwrSOc/LEYgQNmrj6dPUCJWJkB3685vXwJ499F7JWJlAvSl+tz05tH5D1/HaL9/1iBWI0CfSc/X33g8fcJ5iY4XkiHW4b/hb6vPj6en+75vff2BU/9uLbiJCeiG9Px/1zp++GkaMgqxCBNQX5//uopRiM0ToHec3/l333N97OJyGsS2CdBv6pOsQYNYhAANV5/rq2sQWyVAI9bneg0axCYJ0KD1OdIgtk2ALo1TnzHXA3ckQP8aLsa820+rMgSxMQI02V0912rhYwL0jzHHn/HXBp+2eoAG33ydsxFje5YO0Lx38rwrh3NLB+hk/PFnrnXCHxIgILNugCY6/TnnJIgtWTdAQE6AgMyiAZp0/3VkF8ZmLBogYAQCBGQECMisGKCpD4COHAOxDSsGCBiEAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmRUD9Pr68/jFy+Gwm9Np5ac/C8xoxQABgxAgICNAQGbRAE19DOQAiM1YNEDACAQIyKwboEl3YfZfbMm6AQJy3+sFDOHlcJji90PPNawl/BXNZekJaN5dzLwrh3NLB2iukyCnP2yPLdgcG7Hx+9gS5UmtPgFN9293rtXCxwRogo2YzRdbJUCXRmvQaOuBOxKg3fVwMc49f74S4w/bI0DjNkh92DwBGrRB6sMKBGjEBqkPi/j29varXsOI9vvn828f9nzQRfLUh20zAb3v4s5/zCikPqzGBDTEKCQ9rEmAbm7QfTN0PVupD+sQoM9n6CslendPJz2sRoC+2qA/j9HHB0nqw4IE6P4luonusDIBakqkOyBAD42R6MAFAQIyHkQEMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELCr/A0ulRcgAT934wAAAABJRU5ErkJggg==', 'sha256': '216649094640dc47978380a049bb3528c65d20e8c6a0371cc6afa3e93588777a'}, 'data/binding-v1/images/binding-evaluation-0003-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHklEQVR4nO3cQU4bSRiAUWcU7uNFtrkBJ8mB5iS5wWyz4DBZIHkzC0sejyEQg8NX1fXeCiTkLgT96a+izafHx587gMJfyVUBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh87i4NN7Pff9lN6+Hhx25VJiAgI0BAxhaMTfnn78NuEl+/3dVL6JmAgIwAARkBAjICBGQECMgIEJARICDjOaCPezfAyk/cw7ME6OPegnT+xWIEApS99fH0CkrEygToZt35fvfKk/X3h2feJaBErEyA3lWfV6Pzqy9+GqP9/osGsRoBekt6rurOs06vcF6i44VkiHX4M/x19fl+d/f++rz8glP/by24ignoivT8uWsdX/w0DRmFWIQJqK/Pr65iFGLzBOgZ53f+zfdcL7u4nAaxbQL0Sn2SNWgQixCg4erz9OoaxFYJ0Ij1eboGDWKTBGjQ+hxpENsmQJfGqc+Y64EbEqD/DRdj3u2nVRmC2BgBmuyunmu18DIB+s+Y48/4a4M3Wz1Ag2++ztmIsT1LB2jeO3nelcO5pQN0Mv74M9c64TcJEJBZN0ATnf6ccxLElqwbICAnQEBm0QBNuv86sgtjMxYNEDACAQIyAgRkVgzQ1AdAR46B2IYVAwQMQoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMyKAXp4+HH84P5w2M3ptPLT9wIzWjFAwCAECMgIEJBZNEBTHwM5AGIzFg0QMAIBAjLrBmjSXZj9F1uyboCA3Od6AUO4Pxym+P/Qcw1ria/fJvg5crL0BDTvLmbelcO5pQM010mQ0x+2xxZsjo3Y+H1sifKkVp+ApvvdnWu18DIBmmAjZvPFVgnQpdEaNNp64IYEaPd0uBjnnj9fifGH7RGgcRukPmyeAA3aIPVhBQI0YoPUh0V8enz8Wa9hRPv9l/NPP+z5oIvkqQ/bZgJ63sWd/zGjkPqwGhPQEKOQ9LAmAbq6QbfN0NPZSn1YhwC9PUPvKdGzezrpYTUC9N4G/X6MXj5IUh8WJEC3L9FVdIeVCVBTIt0BAfrQGIkOXBAgIONBRCAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABu8q/io0VikIuwVIAAAAASUVORK5CYII=', 'sha256': '155fc6e59ea0bb340cbbfb4eb3ed6f650dd123d42e7325cf8730e46277f67bf7'}, 'data/binding-v1/images/binding-evaluation-0004-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQklEQVR4nO3aQU4bWRRAUdNqlhPJA5bRu8k6spteRgZI2QoDJCY9QLJQwA7QyLd+vXNGHoD9S3ZdvV9VN4+PDweAwl/JpwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMn93H70fx+Pd5T+4v/95rbXASm4eHx/qNeywOJfpETwToOt15zUl2uxXcx33438AtmBf8Pt++v7t8r/f/vh14W39ChlLgD6Znj9G59wfv46RDDGWAH0sPR/qzptO7/BbiWToS/x7e3vYvH+enuolbIUAvas+/7877yzR8XinQczhOaCmPhfef9HrqfAJJqAyPb990GkUsh1jCBNQX59zn2gUYvdMQG+c7ddPz4VRyBzEjpmANlSfN9dgDmLHBGhz9XmmQUwwPUDbrM8zDWL3pgdos/XZ8qrgq4wO0Gms2PJ5flqbIYj9mRugFc/nFdcMFwwN0JYv/bzmYhB7NTRAC9VnrXXCh0wM0OpDxOrrh9EBWnSsWGu18B7jArTEna9z3BFjZ8YFCNiOWQHa0+Cwp2NhrFkBOllx/7X6yuG1oQECtmBQgPa3Z9nfETHNoADtZhez+vphdICAjRAgICNAQGZKgJZ+APo1j0SzD1MCBGyQAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmSkBur//+fzi9sevw/pOR3E6LljRlAABGyRAQEaAgMzEAK1+GWj19cPEAO3veu3+johpBgUI2JqhAVp3F7PuymF6gPa0Z9nTsTDWrAABmzIuQEs/Eu0BaHZmXIBeWqtBa60W3mNigFYfH1ZfP4wO0IpjxSrrhA8ZGqCXQ8T2z+2XKzT+sCdDA7TombzimuGCuQFa5Y6YO1/s2OgAvbTNBm1zVfBVbh4fHw6zHY93p9dP378dNsOln899iQu5H/+1moA2ekFafZhAgLbYIPVhCFuws2N8sh37LX/qw76ZgM6e7dcfhdSHaUxAmxiFpIeZTED9KKQ+jGUC+tid3S+cht7smvowigB98gGTT5fo3DwlPQwkQF/wnNsfY3R5Eyc9jCVA5eO20sNwAhSUSHfgmQBdo0eKA28SICDjOSAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgQcKv8BqrYwgnfb6xcAAAAASUVORK5CYII=', 'sha256': 'bba5bbb05d2d19d82ed0b8c1a30d80e3d55ad41e10d333f4331116579c7a5233'}, 'data/binding-v1/images/binding-evaluation-0004-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFR0lEQVR4nO3aTUokaRRA0bRpl9PgQHAntZFeR2/EnRTUQOitOBCc9EBIxJ8stSVvfPHOGTlQ8wWZcXkRGRcPD/cHgMIfyasCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjJ/di+9H1dX16d/4e7u17lmgZVcPDzc1zPssDin6RE8EaDzdec1JdrsW3Med+M/AC7BvuHzfXlze/rPH3/+OPFvfQoZS4C+mJ7fRue9X34dIxliLAH6XHo+1Z03Hf/DixLJ0Ld4/Puvw+Zd/vNvPcJWCNCH6vP/u/PBEl1dXWsQc3gOqKnPif+/6P1U+AIbUJmeFy90XIVcjjGEDaivz3uvaBVi9wTojbP98ub2/PV586U1iH0ToDfqk87ycgYNYscEaHP1eaJBTDA9QNuszxMNYvemB2iz9dnyVPBdRgfouFZs+Tw/zmYJYn/mBmjF83nFmeGEoQHa8q2f19wMYq+GBmih+qw1J3zKxACtvkSsPj+MDtCia8Va08JHjAvQEt98vcc3YuzMuAAB2zErQHtaHPZ0LIw1K0BHK15/rT45vDY0QMAWDArQ/q5Z9ndETDMoQLu5ill9fhgdIGAjBAjICBCQmRKgpR+Afs0j0ezDlAABGyRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBmSoDu7n49/fD488dhfcejOB4XrGhKgIANEiAgI0BAZmKAVr8NtPr8MDFA+7tfu78jYppBAQK2ZmiA1r2KWXdymB6gPV2z7OlYGGtWgIBNGRegpR+J9gA0OzMuQM+t1aC1poWPmBig1deH1eeH0QFaca1YZU74lKEBer5EbP/cfj6h9Yc9GRqgRc/kFWeGE+YGaJVvxHzzxY6NDtBz22zQNqeC73Lx8HB/mO3q6vr48+XN7WEz3Pr52pu4kLvxb6sNaKM3pNWHCQRoiw1SH4ZwCfbuGp9cjr3In/qwbzagd8/2869C6sM0NqBNrELSw0w2oH4VUh/GsgF97pvdb9yG3uya+jCKAH3xAZMvl+i9fUp6GEiAvuE5t9/G6PRFnPQwlgCVj9tKD8MJUFAi3YEnAnSOHikOvEmAgIzngICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEHCo/AdQvSjTm8gRkQAAAABJRU5ErkJggg==', 'sha256': '04c063ad133a78938eeaee1fd0c34ef764d6a19cbfc74f9e05b27301abe51c57'}, 'data/binding-v1/images/binding-evaluation-0004-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQ0lEQVR4nO3aMU4jWRRAURgN+3HQEpuYuBdJPJtoqQMWQ4BEMgGShcC4gbG49eudExEAfiW7rl6V6/rx8eEKoPBX8qoAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMzf3Uvvx+Hw4/wv3N///q5ZYCXXj48P9Qw7LM55egTPBOj7uvOWEm32rfke9+M/AC7BLvD5vrm9O//nT79+nvm3PoWMJUBfTM8fo/PeL7+NkQwxlgB9Lj2f6s5Jx//wqkQydBH/3txcbd4/T0/1CFshQB+qz//vzgdLdDj80CDm8BxQU58z/3/R+6nwBTagMj2vXui4CrkcYwgbUF+f917RKsTuCdCJs/3m9u7763PypTWIfROgE/VJZ3k9gwaxYwK0ufo80yAmmB6gbdbnmQaxe9MDtNn6bHkquJTRATquFVs+z4+zWYLYn7kBWvF8XnFmOGNogLZ86+ctN4PYq6EBWqg+a80JnzIxQKsvEavPD6MDtOhasda08BHjArTEN1/v8Y0YOzMuQMB2zArQnhaHPR0LY80K0NGK11+rTw5vDQ0QsAWDArS/a5b9HRHTDArQbq5iVp8fRgcI2AgBAjICBGSmBGjpB6Df8kg0+zAlQMAGCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkJkSoPv7388/PP36ebW+41EcjwtWNCVAwAYJEJARICAzMUCr3wZafX6YGKD93a/d3xExzaAAAVszNEDrXsWsOzlMD9Cerln2dCyMNStAwKaMC9DSj0R7AJqdGRegl9Zq0FrTwkdMDNDq68Pq88PoAK24VqwyJ3zK0AC9XCK2f26/nND6w54MDdCiZ/KKM8MZcwO0yjdivvlix0YH6KVtNmibU8GlXD8+PlzNdjj8OP58c3t3tRlu/XztTVzI/fi31Qa00RvS6sMEArTFBqkPQ7gEe3eNTy7HXuVPfdg3G9C7Z/v3r0LqwzQ2oE2sQtLDTDagfhVSH8ayAX3um90LbkMnu6Y+jCJAX3zA5Mslem+fkh4GEqALPOf2xxidv4iTHsYSoPJxW+lhOAEKSqQ78EyAvqNHigMnCRCQ8RwQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECrir/AcI9KNODZRCnAAAAAElFTkSuQmCC', 'sha256': '356a402ed9ea74426241137aeb1ce06e780fe8ab1688990b422874f943e66191'}, 'data/binding-v1/images/binding-evaluation-0005-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFM0lEQVR4nO3dMW4bVxRA0XEQ7UeFgVTeRhbhBWUR2UaqAC64GBcC3KQgQBCyQkGCzDvz3zm1RP6BMFfvf5Lgp6en7xtA4bfkWQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+b17avjlHh8/b0s4nb5tKzIBARkBAjK2YIzw8Mff2wH9+PfPbWkmICAjQEDGFuyur7as+loGvI8A3fUl3utfESMQoOytJZfHUSLGEqAPS88/f/24/etfvj7ceFgZYiABen96Xi3O7Z9/1iMZYiABukd6XnR+EBliMgF6Q30+pDvPXB7zukSPj581iAkEKEvP7YHIKMQE3oi4i/r833Mt82FueJEJ6Lnre/6e6bkxCpmDWJUJaF/1efHZzUGsSoD2WJ+f16BBLEmAdlqfMw1ibQK03/qcaRALE6Bd1+dMg1jV9ADtvz5nGsSSpgfoEPU5ygrhrUYH6DJKHOXevqzTEMQa5gbo6Pfw0dcPowN0cZTx54irhduGBuhwm69rNmIsY2iAgD2YGKBDjz9nhiDWMDFAwE6MC9B6I8N6V8Qc4wJ0cdz91xrrh9EBAnKzArTA8fM1R9Ec3awAAbsiQEBGgIDMoACtfVCy9tWxqkEBuljjBHq9a2GgiQECdkKAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMxAB9+fqwrWKla2GgQQE6nb5t61r76ljVoAABeyNAQEaAgMysAF0OStY4u71chQMgDmpWgIBdmRugow9BR18/TAzQeruV9a6IOcYFCNiPiQFa4Cja8TNrmBggYCeGBujQQ5Dxh2UMDdC1YzXoWKuF2+YG6Ojjw9HXD9u2fXp6+r4NdvlG40N8xajN19hvrD4t+hefOwEdbmuz/xXCW00P0PU/lj3f4ddrW/WfIQNN34L9PKjvcC+mPqxq+gS0/zlIfViYAO26QerD2gRovw1SH5YnQDttkPowgUPo1988cudj6WfhUx8WZgJ6wbN7/p6jkPowigloL6OQ9DCQAL3nvfwfWKIXxyv1YQgByjIkPSBAH/bJxld7dPssSXoYSID6D1hLD2MJUFYi3QEBumuMRAeuCRCQ8UZEICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DAVvkPm0E/rvSjEucAAAAASUVORK5CYII=', 'sha256': '4b9bece18d5ffabeca3a5b3917a25a6014174d448e7eb6b478166424c56bc4c4'}, 'data/binding-v1/images/binding-evaluation-0005-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNElEQVR4nO3dMW4bVxRA0XEQ7UeFAVfZhheRBWUR3kYqAy60GBcC1KQgQBCyQkGCzDvz3zm1RP6BMFfvf5Lgp8fHnxtA4Y/kWQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+bN7avjt7u8/b0t4ePixrcgEBGQECMjYgjHCv/88bQf0199329JMQEBGgICMLdhNX21Z9bUMeB8BuulLvJe/IkYgQNlbS86Po0SMJUAflp67L9+u//rT969XHlaGGEiA3p+eV4tz/eef9UiGGEiAbpGeF50eRIaYTIDeUJ8P6c4z58e8LNH9/WcNYgIBytJzfSAyCjGBNyLuoj7/91zLfJgbXiRAz13e83dfvt2yPi8+qQaxMAG6Vp9wJRrEBAK0x/r8ugYNYkkCtNP6nGgQaxOg/dbnRINYmADtuj4nGsSqpgdo//U50SCWND1Ah6jPUVYIbzU6QOdR4ij39nmdhiDWMDdAR7+Hj75+GB2gs6OMP0dcLVw3NECH23xdshFjGUMDBOzBxAAdevw5MQSxhokBAnZiXIDWGxnWuyLmGBegs+Puv9ZYP4wOEJCbFaAFjp8vOYrm6GYFCNgVAQIyAgRkBgVo7YOSta+OVQ0K0NkaJ9DrXQsDTQwQsBMCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkJgbo6fvXbRUrXQsDDQrQw8OPbV1rXx2rGhQgYG8ECMgIEJCZFaDzQckaZ7fnq3AAxEHNChCwK3MDdPQh6Ojrh4kBWm+3st4VMce4AAH7MTFACxxFO35mDRMDBOzE0AAdeggy/rCMoQG6dKwGHWu1cN3cAB19fDj6+mHbtk+Pjz+3wc7faHyIrxi1+Rr7jdUPi/7F505Ah9va7H+F8FbTA3T5j2XPd/jl2lb9Z8hA07dgvw7qO9yLqQ+rmj4B7X8OUh8WJkC7bpD6sDYB2m+D1IflCdBOG6Q+TOAQ+vU3j9z4WPpZ+NSHhZmAXvDsnr/lKKQ+jGIC2ssoJD0MJEDveS//B5boxfFKfRhCgLIMSQ8I0Id9svHVHl0/S5IeBhKg/gPW0sNYApSVSHdAgG4aI9GBSwIEZLwREcgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsFX+A0kiMBQKDM0JAAAAAElFTkSuQmCC', 'sha256': 'ad941792fd6d87032e24b8a7067da664cda537381fb442f56c967822d56ce6b7'}, 'data/binding-v1/images/binding-evaluation-0005-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKklEQVR4nO3dQW7TUBRAUYNgPx0gsRMWyU6QGHQxDCoxYRApikJJ1ark2v+dM26Tb1W+ff52lA9PT782gMLH5F0BBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnUvTX8dw8PX7YlPD7+3FZkAgIyAgRkXIIxwuev37cD+v3j27Y0ExCQESAg4xLsrndbVr2XAW8jQHe9xXv5K2IEApQ9WnJ+HSViLAF6t/S8eJ/lX3c0Ti8rQwwkQG9Pz2vv7F79/FWPZIiBBOge6XnW6UVkiMkE6BX1+R8Ps51f87JEDw9fNIgJBChLz+2ByCjEBB5E3EV9/vVey3yYG54lQNcuz/nPX7/f/zNEV2+qQSxMgG7VJ1yJBjGBAO2xPn+vQYNYkgDttD4nGsTaBGi/9TnRIBYmQLuuz4kGsarpAdp/fU40iCVND9Ah6nOUFcJrjQ7QeZQ4yrl9XqchiDXMDdDRz+Gjrx9GB+jsKOPPEVcLtw0N0OEuvi65EGMZQwME7MHEAB16/DkxBLGGiQECdmJcgNYbGdY7IuYYF6Cz415/rbF+GB0gIDcrQAtsP1+yFc3RzQoQsCsCBGQECMgMCtDaGyVrHx2rGhSgszV2oNc7FgaaGCBgJwQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMhMDNDvH9+2Vax0LAw0KECPjz+3da19dKxqUICAvREgICNAQGZWgM4bJWvs3Z6PwgYQBzUrQMCuzA3Q0Yego68fJgZovauV9Y6IOcYFCNiPiQFaYCva9jNrmBggYCeGBujQQ5Dxh2UMDdClYzXoWKuF2+YG6Ojjw9HXD9u2fXh6+rUNdv5G40N8xaiLr7HfWP246F987gR0uEub/a8QXmt6gC7/sez5DL9c26r/DBlo+iXY34P6Dq/F1IdVTZ+A9j8HqQ8LE6BdN0h9WJsA7bdB6sPyBGinDVIfJrAJ/fLDI3felr4Kn/qwMBPQM67O+XuOQurDKCagvYxC0sNAAvSWZ/nfsUTPjlfqwxAClGVIekCA3u2TjS/26PZekvQwkAD1H7CWHsYSoKxEugMCdNcYiQ5cEiAg40FEICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DAVvkDQ3EwFGsNynoAAAAASUVORK5CYII=', 'sha256': 'a6d9c2c88f7a1d3d4c018806ba9b199d4bd5dd567aa475b85cbc270a9102176f'}, 'data/binding-v1/images/binding-evaluation-0006-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPklEQVR4nO3aTU5bSRhAUWjFy2mJQZaR3fQ6shuWkYGlbIUBEpMeIDkOJnZMCPdV1TkjxN/7LPld1avy7ePjww1A4Z/kqgACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkPnWXntbd3efzv7Dff/uoWWDTbh8fH+oZ5i/OeXrEsgSoTM8xGWJBAvT+3bnf7c7/+ZenpzM/VSLWIUDvk56L0bk2RjLECgToj+rz5u78Tok0iOkJUJ+eYzLEUgTo6vr8pfScyZAGbeGUYDv2E70ffBDxinfw/W73AfU5vdCUdxEI0HX1+eCraxDT80nojdbncN3D49jd3eeZ1t4f7Om/f28Gt/v6/WY6VkAX6vNhj12/cjyAdRCTEaAL9bnZBg1iSgI0QH2eaRDzEaCfjHJvjzInnCdA29p1Ps+5GJMRoFdssz7bnw2uJUBb3/o5ZTOIaQjQ2HfyuJODAL20/eXPWHPCeQI0/CJi9PlZmQCNuqwYa1p4lQABmdUDNNDh1ynHYYxu9QABoaUDNNPCYabXwjqWDtDBiM9fo08OAgSUBAjICBCQWTdAQx/AH3MYz7jWDRCQEyAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgILNugPb7b89ffHl6uhnZYf7DK4JRrBsgICdAQEaAgIwAjb0NNO7ksHqAZtq1nem1sI6lAwS0Vg/Q0IfxDuAZ3eoBAkIC9MNYi6CxpoVXCdDwzy+jz8/KBGjIZcUoc8J5AjT2ImLcyUGAhjwOc/jFNAToFVtu0JZng2sJ0A/HC4pt3ufHU1n+MAEB+skod/Uoc8J5AjTMZpCtH+YjQGM0SH2YkgBdblCboeMB1IfJCNCm96TtOjM3Adpug9SH6QnQdQ36mAy9uJD6MCsBuuzF/f9XM3T6z9WHiX2qBxjDcwXu7j4fvvOcifvd7r0ucRo16WF6AnSF/f7bcYOOq/HmEv1qMaU+rECA/nQpdNqRizE6/wQnPaxDgN7i0IjTEr35vEx3WJAA/ZUF0Rv+CSxIgN7Bi4Jc7JHiwDMBen/6Ar/J54CAjAABGQECMvaAWMLu6/d6BF5hBQRkBAjI3D4+PnRXB5ZmBQRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICbyv+VS0YOyYCM8gAAAABJRU5ErkJggg==', 'sha256': 'f4e529313513a5eae0929baf6ea4838a5999a7e24f38f8d7480d78edc74e9115'}, 'data/binding-v1/images/binding-evaluation-0006-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPUlEQVR4nO3aTU4bSxhAUXgK+2EQiU1knEVm/DYRKQMvhgESkwyQHAcTOybA7ao6Z4T468+S+6q6ytcPD/dXAIX/kqsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnUXXpat7efT//Cbvfjo2aBTbt+eLivZ5i/OKfpEcsSoDI9h2SIBQnQ23fn5u7b6T9//P71xE+ViHUI0Nuk52x0Lo2RDLECAfqn+ry6O39TIg1iegLUp+eQDLEUAbq4Pu+UnhMZ0qAtnBJsx26i94MPIl7wDr65+/YB9Tm+0JR3EQjQZfX54KtrENPzSeiN1md/3f3j2O3t55nW3h/s/5ubq8F9eXy8mo4V0Jn6fNhj158cDmAdxGQE6Ex9rrZBg5iSAA1QnycaxHwE6Dej3NujzAmnCdC2dp1Pcy7GZAToBdusz/Zng0sJ0Na3fo7ZDGIaAjT2nTzu5CBAz21/+TPWnHCaAA2/iBh9flYmQKMuK8aaFl4kQEBm9QANdPh1zHEYo1s9QEBo6QDNtHCY6bWwjqUDtDfi89fok4MAASUBAjICBGTWDdDQB/CHHMYzrnUDBOQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIrBug3e7H0xeP379ejWw///4VwSjWDRCQEyAgI0BARoDG3gYad3JYPUAz7drO9FpYx9IBAlqrB2jow3gH8Ixu9QABIQH6ZaxF0FjTwosEaPjnl9HnZ2UCNOSyYpQ54TQBGnsRMe7kIEBDHoc5/GIaAvSCLTdoy7PBpQTol8MFxTbv88OpLH+YgAD9ZpS7epQ54TQBGmYzyNYP8xGgMRqkPkxJgM43qM3Q4QDqw2QEaNN70nadmZsAbbdB6sP0BOiyBn1Mhp5dSH2YlQCd9+z+f9cMHf9z9WFin+oBxvBUgdvbz/vvPGXi5u7bW13iOGrSw/QE6AK73Y/DBh1W49Ul+tNiSn1YgQD961LouCNnY3T6CU56WIcAvca+EcclevV5me6wIAF6lwXRK/4JLEiA3sCzgpztkeLAEwF6e/oCf8nngICMAAEZAQIy9oBYwpfHx3oEXmAFBGQECMhcPzzcd1cHlmYFBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgKvKT37SRYOojD7gAAAAAElFTkSuQmCC', 'sha256': '9bca7fb0ed628a4666f6949d860d72fd55643db5da2316ed22c2dd4b6a765d80'}, 'data/binding-v1/images/binding-evaluation-0006-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQklEQVR4nO3aTWpbSRhAUaeJl9PggSE7yUZ6HdmIdxLwQJCteGDIpAcGRbEcKXIc31dV54yM/94n0LvUq9KHx8eHK4DCP8lVAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh87C49rZub29O/sNvdv9cssGkfHh8f6hnmL85pesSyBKhMzyEZYkEC9Pbduf50d/rPv3/9fOKnSsQ6BOht0nM2OpfGSIZYgQD9UX1e3Z3fKZEGMT0B6tNzSIZYigBdXJ+/lJ4TGdKgLZwSbMduoveDDyJe8A6+/nT3DvU5vtCUdxEI0GX1eeeraxDT80nojdZnf93949jNze1Ma+939v2/f68Gd/3l29V0rIDO1OfdHrt+5XAA6yAmI0Bn6nO1DRrElARogPo80SDmI0A/GeXeHmVOOE2AtrXrfJpzMSYjQC/YZn22PxtcSoC2vvVzzGYQ0xCgse/kcScHAXpu+8ufseaE0wRo+EXE6POzMgEadVkx1rTwIgECMqsHaKDDr2OOwxjd6gECQksHaKaFw0yvhXUsHaC9EZ+/Rp8cBAgoCRCQESAgs26Ahj6AP+QwnnGtGyAgJ0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQGbdAO12909ffP/6+Wpk+/n3rwhGsW6AgJwAARkBAjICNPY20LiTw+oBmmnXdqbXwjqWDhDQWj1AQx/GO4BndKsHCAgJ0A9jLYLGmhZeJEDDP7+MPj8rE6AhlxWjzAmnCdDYi4hxJwcBGvI4zOEX0xCgF2y5QVueDS4lQD8cLii2eZ8fTmX5wwQE6Cej3NWjzAmnCdAwm0G2fpiPAI3RIPVhSgJ0vkFthg4HUB8mI0Cb3pO268zcBGi7DVIfpidAlzXofTL07ELqw6wE6Lxn9/9fzdDxP1cfJvaxHmAMTxW4ubndf+cpE9ef7t7qEsdRkx6mJ0AX2O3uDxt0WI1Xl+hXiyn1YQUC9KdLoeOOnI3R6Sc46WEdAvQa+0Ycl+jV52W6w4IE6K8siF7xT2BBAvQGnhXkbI8UB54I0NvTF/hNPgcEZAQIyAgQkLEHxBKuv3yrR+AFVkBARoCAzIfHx4fu6sDSrICAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBBwVfkfi0xEpkERRNMAAAAASUVORK5CYII=', 'sha256': '25183bdc8cb049ece1feae085c259d8674dad568e32ff8f13743738a38ffac7a'}, 'data/binding-v1/images/binding-evaluation-0007-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFAElEQVR4nO3cMU5bWxRAUecrHk4kirSZUNo/DtpM6LcpkJhKCqQ0v0CyLBsMDnb2vddrVSCB3ynw1rkP25+enn5tAAr/JFcFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM5+7ScLa7u6+byT08/KxHGIgNCMgIEJBxBGNK//34vZnKt+/beoQR2YCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARlvxejfwO3t0dwsAeo/NWL/h8WImyJAY31aze4RlIhbIEBXT8/vf7+c/t3t/eNrDyhDrE2ArpKeN6Pz2g8fxEiGWJsAXbg+Z6Xn2POvH2dIg1iSAF0mPR/szoHdo+1KZBViSV4HNFx9TjzyAh/JDvtsQGfbr8D10nNwif1VyB7EMmxAQ9fnxWvZg1iGAE1Qn+MrahBrEKA56nN8XQ1iAQI0TX2Or65BzE6AZqrP8QwaxNQE6Awj1Ge0SeAjBOgNuxVjtOf8bh5LEPMSoFNmeW7PMiccEKB3GW39GXkqeD8Bmu/wtc9BjKkJEJARoJfNuFDMODM3ToDeMPL5a5YJ4TUCBGQEaNbbz/vcimZSAgRkBAjICBCQEaDpbwA9cxuIGQkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQIceHn4+f7G9f9zMYzftbn4YnwABGQECMgIEZARohdtAbgAxKQECMgL0hvGXoPEnhNcI0MtmPMvMODM3ToCAjADNfSva7Wem9rkeYA7b+8cBP6N+5DJe27fv23oELsAGdMosa8Usc8IBAZr1IObwxQIcwaY8iI1Ww79GbRdjAzrvj36EZ/7+DJ6QTE2AJmuQ+rASAZqpQerDYgRomgapD+sRoDkapD4s6dPT0696hvnc3X3d//aq/xo7yJz6sBIb0J84qMD1ViH1YW02oEuuQpfaho6LJj0sSYAu36CPZOjFZUp9WJUAXTFD74zRiROc9LA2AfpLGTqX9HALBOgq/rhEusNNEaA+RqLDzRIgION1QEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABm8r/GDEQ6T2ZjJkAAAAASUVORK5CYII=', 'sha256': 'c98c803c0680e08dd47d6c7e48c80869459b64e7139b0e9bd4f2f6073fed91a1'}, 'data/binding-v1/images/binding-evaluation-0007-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE+UlEQVR4nO3cQU4bSxRAUfKV7IdBpOyERWYnSBmwmAyQmPwBkmXZDmDA3H7lc0YgGXch0Vev2nR/e3z8ewNQ+C85KoAAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZL53h4az3d7+vBnu4eFPvYQNMQEBGQECMrZgjPTj1++bUZ7u7+olbJEJCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQcStGfwO326O5WgLUPzVi/8VixFURoG09rWb3DkrENRCgi6fn1fu2T94n/fyGMsTaBOgi6TnrYRH7Lz6IkQyxNp+CfXJ9fvz6/ZFH1Zz88QWeQwonmYDe6SAKn/t8rN277QYioxBLMgFtrj4vvLNRiMUI0Nn2K/DBDddbHBxCg1iJAH2oPl92XA1iSQI0oD7HR9Qg1iBAM+pzfFwNYgECNKY+x0fXIKYToEn1OV6DBjGaAJ1hC/XZ2krgIwToFbsRY2vn/G49hiDmEqCXTDm3p6wTDgjQm2xt/NnyquDtBGje5mufjRijCRCQEaDTJg4UE9fMlROgV2x5/zVlhfAvAgRkBGjq5ed9LkUzlAABGQECMgIEZARo/AWgZy4DMZEAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAnTo4eHP8xdP93c3c+xWu1s/bJ8AARkBAjICBGQEaIXLQC4AMZQAARkBesX2h6DtrxD+RYBOm7iXmbhmrpwAARkBmn0p2uVnRvteL2CGp/u7DT6jfstlvLRr/t1XYgJ6yZSxYso64YAATd2I2XyxAFuwkRuxrdXwy6jtYkxA5/3Rb+HM31+DE5LRBGhYg9SHlQjQpAapD4sRoDENUh/WI0AzGqQ+LOnb4+Pfeg3z3N7+3P/2oh+NHWROfViJCeg9DipwuVFIfVibCegzR6HPmoaOiyY9LEmAPr9BH8nQyWFKfViVAF0wQ2+M0Qs7OOlhbQL0RRk6l/RwDQToIt5dIt3hqghQHyPR4WoJEJDxf0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAATeV/wFL3AihmKgplwAAAABJRU5ErkJggg==', 'sha256': '00164e9b3db8690473b0428d77f33176437c52ce495b96c795a2dce98c91d7a8'}, 'data/binding-v1/images/binding-evaluation-0007-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFAUlEQVR4nO3cQU5aURSAYdrofhw06ajbcBFdUBfRbXRk0oGL6cDESQcmhIAiCPi/e/m+kSbIOwP5c+5T+PL09G8FUPiaXBVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjI33aXhaHd331aDe3z8W4+wIDYgICNAQMYRjCH9+fW8GsqPn7f1CEtkAwIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZLwVo38Dt7dHc7UEqP/UiM0HixFXRYCW9Wk162dQIq6BAF08Pbfff+//2eeH+7eeUIaYmwBdJD3vRuetB2/FSIaYm7+Cnbk+t99/H1WfQ358gs8hhVfZgD5oKwqnRGfX+tnWC5FViCnZgBZXnz3PbBViMgJ0tM0KnHjgOsTWJTSImQjQSfX5tOtqEFMSoAHqs3tFDWIOAjRGfXavq0FMQICGqc/u1TWI0QnQSPXZnUGDGJoAHWEJ9VnaJHAKAXrHesVY2mt+PY8liHEJ0D6jvLZHmRO2CNBBlrb+LHkqOJwAjXf42uQgxtAECMgI0OtGXChGnJkrJ0DvWPL5a5QJ4S0CBGQEaNTbz5vcimZQAgRkBAjICBCQEaDhbwC9cBuIEQkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQNseH/++fPH8cL8ax3ra9fywfAIEZAQIyAgQkBGgGW4DuQHEoAQIyAjQO5a/BC1/QniLAL1uxLPMiDNz5QQIyAjQ2Lei3X5maDf1AGN4frhf4GfUL7mMl/bj5209AmdgA9pnlLVilDlhiwCNehBz+GICjmBDHsSWVsNPo7aTsQEd90u/hFf+5gxekAxNgAZrkPowEwEaqUHqw2QEaJgGqQ/zEaAxGqQ+TOnL09O/eobx3N192/z2on8a28qc+jATG9BHbFXgcquQ+jA3G9A5V6FzbUO7RZMepiRA52/QKRl6dZlSH2YlQBfM0IEx2nOCkx7mJkCflKFjSQ/XQIAu4sMl0h2uigD1MRIdrpYAARn/BwRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKr8B60pBzMAm8vXAAAAAElFTkSuQmCC', 'sha256': '3fc024ceab5492a0f093e929918c314cd7b7e11d166430d13e047c316d6f7da4'}, 'data/binding-v1/images/binding-evaluation-0008-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMklEQVR4nO3bQU4bSRiAURiF+3jBNjfISXKgnGRukC0LH4aFJTZZIHmc4GmDg/25qt9bIUHo34r7S3WVc7/bPd8BFP5JrgogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZL50l16RzeZx+Qe226drzQI35H63e65nWGNxlukRKyFAt5WeQzLE9AToGt35+eNl+Y9//f6w8F0lYlYCdKn0nIzOR2MkQ8xHgD65Pmd35z0l0iAmI0C3mJ5DMsTEBOgT6nOh9CxkSIOueRpQ2a7gb9nngP7qnX2F9BxeaJ+hzeZxDe9OpueT0APU5+gVJ/gXHqyAhqnP/rrWQR/y78PSRxxu0LeX5q2VEKAP16dKz9HHMQ1iaB7BBqvP3n4Sz2KMS4CGrM8rDWJ0AnTCKPf2KHPCIQG69V3nZc7FGJoAvctt1uf2Z4NlAjTe1s9bNoMYlAAdN+6dPO7krJAAnXD7y5+x5oRDAjThImL0+VkPAZpnWTHWtCBAQEmABj78estxGGMRICAjQL+ZaeEw02thVgJ03IjPX6NPzgoJEJARICAjQEBGgCY5gD/kMJ5RCBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNA/9lun16/+Pr94W5k+/n3rwhukwABGQECMgIEZATouHG3gcadnBUSoN/MtGs702thVgIEZARoqsN4B/CMRYCAjAAtGWsRNNa0IEDHjf78Mvr8rIcATbKsGGVOOCRAsy0ixp2cFRKgGY7DHH4xKAF6l1tu0C3PBssEaMnhguI27/PDqSx/GI4AnTDKXT3KnHBIgAbeDLL1w+gEaNQGqQ8T+FIPMIzt9mmzedzf+T9/vFST2PdhGlZAg+1Jqw8zEaCRGqQ+TMYj2PnPYtd8HPsjdurDHATobxt06Qy9XWepD9MQoDO9VuCiGZIepidAn7kUOqzG2SX6v60l9WE+AvT5S6G3HTkZo+X9bOlhVgL0OfaNeFuis8/LdIfpCdCVFkRn/BI+xbeX7COjnCRAF/FHQU72SHFYJwG6Bn2Bo+53u+fj3wG4MP8VA8gIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBNxVfgEydDo9/QOwCQAAAABJRU5ErkJggg==', 'sha256': 'f80a4a26f84a2492e954303730cdf9b3f4024d21f43f84c3efbca1ea3e5581b6'}, 'data/binding-v1/images/binding-evaluation-0008-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMklEQVR4nO3bPU5bSxiAYbgKy4nkgjYbSpt1pL0bui2FpWyFwhJNCiRfJzg2OLZfz5znqZD4OZ8F52XODNxvNs93AIV/kqsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZlP3aUXZLV6PPwB6/XTtWaBG3K/2TzXMyyxOIfpEQshQLeVnl0yxPQE6Brdefn2+fCnP3z/ceC9SsSsBOhS6TkanY/GSIaYjwCduT4nd+c9JdIgJiNAt5ieXTLExAToDPW5UHoOZEiDrnkaUFkv4Lvs74D+6if7CunZvdA2Q6vV4xJ+Opmev4QeoD57rzjBb3iwAhqmPtvrWgd9yH//vtwN5cvXh7vFEKAP16dKz97HMQ1iaB7BBqvP1nYSz2KMS4CGrM8rDWJ0AnTEKPf2KHPCLgG69V3nw5yLMTQBepfbrM/tzwaHCdB4Wz9v2QxiUAK037h38riTs0ACdMTtL3/GmhN2CdCEi4jR52c5BGieZcVY04IAASUBGvjw6y3HYYxFgICMAP1ipoXDTK+FWQnQfiM+f40+OQskQEBGgICMAAEZAZrkAH6Xw3hGIUBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwA/W+9fnp94+H7j7uRbeffviK4TQIEZAQIyAgQkBGg/cbdBhp3chZIgH4x067tTK+FWQkQkBGgqQ7jHcAzFgECMgJ0yFiLoLGmBQHab/Tnl9HnZzkEaJJlxShzwi4Bmm0RMe7kLJAAzXAc5vCLQQnQu9xyg255NjhMgA7ZXVDc5n2+O5XlD8MRoCNGuatHmRN2CdDAm0G2fhidAI3aIPVhAp/qAYaxXj+tVo/bO//l2+dqEvs+TMMKaLA9afVhJgI0UoPUh8l4BDv9Weyaj2O/xU59mIMA/W2DLp2ht+ss9WEaAnSi1wpcNEPSw/QE6JxLod1qnFyiP20tqQ/zEaDzL4XeduRojA7vZ0sPsxKg89g24m2JTj4v0x2mJ0BXWhCd8EU4iy9fH+oR+CMBuojfCnK0R4rDMgnQNegL7HW/2Tzvfw/AhflXDCAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEHBX+QnmWTz9o9n5WAAAAABJRU5ErkJggg==', 'sha256': '02a7db9ca0139687de68be18c7fd892beb3386d5f8458edd2bb176f1fecb0950'}, 'data/binding-v1/images/binding-evaluation-0008-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMElEQVR4nO3bTU5bSRhAUWiF5UTygGVkN1lHdtPLYGApW2FgyZMeILmd4Njg2L6ueueMkPh5nwXvUq8KHjeb1weAwj/JVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzJfu0guyWj0f/4D1+uVWs8AdedxsXusZllic4/SIhRCg+0rPPhliegJ0i+5sv389/ulPP34eea8SMSsBulZ6TkbnszGSIeYjQBeuz9nd+UiJNIjJCNA9pmefDDExAbpAfa6UniMZ0qBbngZU1gv4Lvs7oL/6yb5BevYvtMvQavW8hJ9OpucvoQeoz8ErTvAbHqyAhqnP7rrWQZ/y79PTw1C+bbcPiyFAn65PlZ6Dj2MaxNA8gg1Wn53dJJ7FGJcADVmfNxrE6ATohFHu7VHmhH0CdO+7zsc5F2NoAvQh91mf+58NjhOg8bZ+3rMZxKAE6LBx7+RxJ2eBBOiE+1/+jDUn7BOgCRcRo8/PcgjQPMuKsaYFAQJKAjTw4dd7jsMYiwABGQH6xUwLh5leC7MSoMNGfP4afXIWSICAjAABGQECMgI0yQH8PofxjEKAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAfrfev3y9sbTj58PI9vNv3tFcJ8ECMgIEJARICAjQIeNuw007uQskAD9YqZd25leC7MSICAjQFMdxjuAZywCBGQE6JixFkFjTQsCdNjozy+jz89yCNAky4pR5oR9AjTbImLcyVkgAZrhOMzhF4MSoA+55wbd82xwnAAds7+guM/7fH8qyx+GI0AnjHJXjzIn7BOggTeDbP0wOgEatUHqwwS+1AMMY71+Wa2ed3f+9vvXahL7PkzDCmiwPWn1YSYCNFKD1IfJeAQ7/1nslo9jv8VOfZiDAP1tg66doffrLPVhGgJ0prcKXDVD0sP0BOiSS6H9apxdoj9tLakP8xGgyy+F3nfkZIyO72dLD7MSoMvYNeJ9ic4+L9MdpidAN1oQnfFFuIhv2209An8kQFfxW0FO9khxWCYBugV9gYMeN5vXw+8BuDL/igFkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIeKv8Bq0c+kwLtPywAAAAASUVORK5CYII=', 'sha256': '815d7b87cb610c2b2f306868d1ba81edfc65b8b606ce5c660703aa67eeec26c3'}, 'data/binding-v1/images/binding-evaluation-0009-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUklEQVR4nO3dPW7cZhRAUTnw7EdFgGwitReZ2psw4GIW40KAmhQChMnImtFfdPnxnVMZsCU+QuDFI4eCv9zd/boBKPyRHBVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnaHXoPbm//vPwPjsefnzULrOfL3d2veoZdFecyPYJTAvQZ3XlKiUCA3pue74fD5S//+/7+wt/KEMMJ0KvTczU6r42RDDGWAL00PW/uzgtLJEMMJEBX6vOB3blaIg1iGu8BZfV5+v0//FE3bJwNqEnPGasQM9mA+vpYhRjLBvSfq/3z03NhFbIHsXvTA7Sp+jzQoMt2vx4eJ/3QR9+CbbA+Z5Ps/mJjuLkB2mZ9HmgQQ/ht+M3V58H3w+Hyr3Fw+Oufmx25//HtZp6hG9DjWrHN+pzNZgliryYGaMXrecWZ4apxAdryo5+nPAxi38YFaKH6rDUnvMGsAK2+RKw+P4wO0KJrxVrTwssNCtASn3w9xydi7NKgAAFbMyVAe1oc9nQuDDclQI9WvP9afXJ4zrgAAdsxIkD7u2fZ3xkx04gA7eYuZvX5YXSAgE0RICAjQEBm/wFa+gXop7wSzZ7sP0DAZgkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJDZf4COx58Pf9jHf7X+eBaP5wXr2n+AgM0SICAjQEBmVoBWfwy0+vwwMUD7e167vzNiphEBArZpXIDWvYtZd3KYHqA93bPs6VwYbkqAgA0aFKClX4n2AjS7NChAp9Zq0FrTwsvNCtDq68Pq88PoAK24VqwyJ7zBuACdLhHbv7ZPJ7T+sD/jArTolbzizHDVxACt8omYT77YvaEBOrXNBm1zKvhYcwO05YdBHv0wxNwAbbZB6sMcowO0wQapD6NMD9DTBlUZOju0+jCBAP3mav/8Bp0dUX0Y4ms9wFY8XPO3t3+eFuH74fB/H1d6mMwGVK5C6sNwNqBzx+PPxz3otBEfuA39tmvqw0ACdP127KNK9Nw+JT2MJUCvy9BZR67G6PJNnPQwnAC9MUPvfEgkPSBAr+7FcyV67fcBBOi9BbnaI8WBCwToXfQF3sN7QEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAxpvQrOr+x7d6BN7LBgRkBAjIfLm7+9UdHRjNBgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE3FT+BSGmTCmYDQd6AAAAAElFTkSuQmCC', 'sha256': '0d54912c8cd991a57d594f531b8cc9b1768c0ff4978eee208617ba8a21564d5c'}, 'data/binding-v1/images/binding-evaluation-0009-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFTklEQVR4nO3dTUocaxSAYb3E5QQcZBnZTdZxd3OXkUHD3UoGQk8yEKSjsds/fOur8zwjwZ8+hdTLqeoSr+/ufl0BFP5JXhVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnSvfQe3N5+O/8Fh8PPz5oF1nN9d/ernmFXxTlPj+CUAH1Gd55SIhCg96bn+OPr+W+/+ff/M5+VIYYToFen52J0XhsjGWIsAXppet7cnReWSIYYSIAu1OcDu3OxRBrENJ4Dyurz9Od/+K1u2DgbUJOeR6xCzGQD6utjFWIsG9AfZ/vnp+fMKmQPYvemB2hT9bmnQeftfj08TPqlj74E22B9Hk2y+5ON4eYGaJv1uadBDOGv4TdXn3vHH1/P/xkH/93cXO3I9+Pxap6hG9DDWrHN+jyazRLEXk0M0Irn84ozw0XjArTlWz9PuRnEvo0L0EL1WWtOeINZAVp9iVh9fhgdoEXXirWmhZcbFKAl3vl6jnfE2KVBAQK2ZkqA9rQ47OlYGG5KgB6seP21+uTwnHEBArZjRID2d82yvyNiphEB2s1VzOrzw+gAAZsiQEBGgIDM/gO09APQT3kkmj3Zf4CAzRIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICCz/wAdDj/vP9jHv1p/OIqH44J17T9AwGYJEJARICAzK0Cr3wZafX6YGKD93a/d3xEx04gAAds0LkDrXsWsOzlMD9Cerln2dCwMNyVAwAYNCtDSj0R7AJpdGhSgU2s1aK1p4eVmBWj19WH1+WF0gFZcK1aZE95gXIBOl4jtn9unE1p/2J9xAVr0TF5xZrhoYoBWeUfMO1/s3tAAndpmg7Y5FXysuQHa8s0gt34YYm6ANtsg9WGO0QHaYIPUh1G+1AP0Doeft7ffTs//44+vnz/Go/ypDxNM34D+erZ//iqkPsxkA/rjnP/8VUh6mMwGVK5C6sNwNqBzt4ROG/GB29Bfu6Y+DCRAly/HPqpEz+1T0sNYAvS6DD3qyMUYnb+Ikx6GE6A3ZuidN4mkBwTo1b14rkSv/TmAAL23IBd7pDhwhgC9i77Ae3gOCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIeBKaVX0/HusReC8bEJARICBzfXf3q3t1YDQbEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBBwVfkNr9RBXJKx/KMAAAAASUVORK5CYII=', 'sha256': '1d6fd07d7e7cb757ad052df80a7fb23e3eca5f2664319dd3022a4c968da333a9'}, 'data/binding-v1/images/binding-evaluation-0009-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUUlEQVR4nO3dS0okaRSAUaupWo7goKB3UhupdfRGeicFDhJ6Kw4EJz0QJEvLTF/4xR/3nJHgI28g8XEjMsQvt7c3FwCFv5JXBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkvnYvvQdXV99Pf8HhcP1Zs8B6vtze3tQz7Ko4p+kRHBOgz+jOU0oEAvTe9Nz9vDz97d/++e/EZ2WI4QTo1ek5G53XxkiGGEuAXpqeN3fnhSWSIQYSoDP1+cDunC2RBjGN54Cy+jz9+R9+qxs2zgbUpOcRqxAz2YD6+liFGMsG9NvZ/vnpObEK2YPYvekB2lR97mnQabtfDw+TfumjL8E2WJ9Hk+z+ZGO4uQHaZn3uaRBD+Gv4zdXn3t3Py9N/xsG3v/+92JG7Xz8u5hm6AT2sFdusz6PZLEHs1cQArXg+rzgznDUuQFu+9fOUm0Hs27gALVSfteaEN5gVoNWXiNXnh9EBWnStWGtaeLlBAVrina/neEeMXRoUIGBrpgRoT4vDno6F4aYE6MGK11+rTw7PGRcgYDtGBGh/1yz7OyJmGhGg3VzFrD4/jA4QsCkCBGQECMjsP0BLPwD9lEei2ZP9BwjYLAECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjL7D9DhcH3/wT7+1frDUTwcF6xr/wECNkuAgIwAAZlZAVr9NtDq88PEAO3vfu3+joiZRgQI2KZxAVr3KmbdyWF6gPZ0zbKnY2G4KQECNmhQgJZ+JNoD0OzSoAAdW6tBa00LLzcrQKuvD6vPD6MDtOJascqc8AbjAnS8RGz/3D6e0PrD/owL0KJn8oozw1kTA7TKO2Le+WL3hgbo2DYbtM2p4GPNDdCWbwa59cMQcwO02QapD3OMDtAGG6Q+jPK1HqB3OFxfXX0/Pv/vfl5+/hiP8qc+TDB9A/rj2f75q5D6MJMN6Ldz/vNXIelhMhtQuQqpD8PZgE7dEjpuxAduQ3/smvowkACdvxz7qBI9t09JD2MJ0Osy9KgjZ2N0+iJOehhOgN6YoXfeJJIeEKBX9+K5Er325wAC9N6CnO2R4sAJAvQu+gLv4TkgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDjSWhWdffrRz0C72UDAjICBGS+3N7edK8OjGYDAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIuKv8DaktBNPmXPDwAAAAASUVORK5CYII=', 'sha256': 'd701ef5e46fe20fb7ce83a80a67de350a903ae92cc2f48844b5358f8fa29cf30'}, 'data/binding-v1/images/binding-evaluation-0010-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFH0lEQVR4nO3ZS2ocSRRAUbmxltOggUE70UZ6Hb0R7UTgQUFvxQOBJj0oKAp90a9uZL1zRjaUM0Mo8/pF1I/7+z8XAIW/krsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dndetuurn69/oHd7vep1sLX//pWtjujR0uAvuupPf7wOT0x8IUE6BT/Wx6uoERwTIA+mJ7L69vX/+3D3c1LF5ShdTz88/fFRlz++9/F2RGgd6Tnzei89OFHMZIh2BOgt9Pzru4863CF4xLJEPga/rX6XF7ffr4+r19w01/HwCcJ0PMt+PL0HHt0cQ1iLFuw5wefE9z08vr2sCOzHWMmE1BTn2fvZRRimukBOtm26yW2Y0w2OkCP6hOuRIOYaXSAFqnPOmuAE5sboMOgsc6bf1iJIYghhgZo/Td8/RXC500M0DpHP085DGKUiQFatj4rrwq+w7gALXj085TDIIYYFyBgHbMCtInxZ88QxASzAgQsZVCANjT+7BmCOHuDAgSsRoCAzJQAbW7/tWcXxnmbEiBgQQIEZAQIyIwI0EYPgPYcA3HGRgQIWJMAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZEQHa7X7v//Bwd3OxNYc1H34KOBsjAgSsSYCAjAABmSkB2ugxkAMgztuUAAELEiAgMyhAm9uF2X9x9gYFCFjNrABtaAgy/jDBrAABSxkXoE0MQcYfhhgXoGNrNmjNVcF3mBig47Fitbf9eD3GH87exABt4t1ef4XweUMDtOZhkKMfppkboGMrNGiFNcCJjQ7QOodBjn6YaXSAnjbo9Bl6dFP1YZTpAXr6zp+yQY/upT5M87NewBL2b/7V1a/9Xx/ubi6vb7/7pgYf+HF//6dew0IODdr7pgwZfFb7RW/L7oweGFuwN7ZjX7sje3rBc3qY4L1swd7Yjh0PLB8eiJ6tmPSALdgHB/U3Y/TK6CQ9sCdAJz0vkB44JkCnKJHuwLME6LtiJDrwJgECMr6GBzICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECLir/A0y9F03mN8yLAAAAAElFTkSuQmCC', 'sha256': '1924e93c9b6abce148d3714cac8886290ba8ed0bf17af35dab853ef8e1cccedd'}, 'data/binding-v1/images/binding-evaluation-0010-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFH0lEQVR4nO3aQWpbSRRAUaVJlhPwwNA76Y1kHdlI76TBA0G2koHBkyYIhJAVmcSRbknvnJEHsn4Z/X/9qtCH5+fvG4DCX8lVAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyHzsLn3bHh4ez79gu3261lr48x/fyrZ3dGsJ0KXu2sMX39MdA3+QAF3jv+X+HZQIDgnQb6bn5cvn87/76eu3n72hDK3j09//bm7Ey3//bO6OAP1Cet6Mzs9efBQjGYIdAXo7Pb/UnZP273BYIhkCATpXn/en58juDY8ypEGMJUCn6/PH03MmQxrEWL6I+MPDw+PV6nPyKkcLgCEE6OLbrjOOrqVBTDN9C3b9weeI7RiTjZ6A8vqcvLo5iDlGB2iR+qyzBriyuQHaDxrrPPn7lRiCGGJogNZ/wtdfIbzfxACtc/TzmsMgRpkYoGXrs/Kq4BLGBWjBo5/XHAYxxLgAAeuYFaCbGH92DEFMMCtAwFIGBeiGxp8dQxB3b1CAgNUIEJCZEqCb23/t2IVx36YECFiQAAEZAQIyIwJ0owdAO46BuGMjAgSsSYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMyIAG23T7sfPn39trk1+zXv/wq4GyMCBKxJgICMAAGZKQG60WMgB0DctykBAhYkQEBmUIBubhdm/8XdGxQgYDWzAnRDQ5DxhwlmBQhYyrgA3cQQZPxhiHEBOrRmg9ZcFVzCxAAdjhWrPe2H6zH+cPcmBugmnu31VwjvNzRAax4GOfphmrkBOrRCg1ZYA1zZ6ACtcxjk6IeZPm5m226fHh4eDyvw8uXzNRdwFD71YZTRE9DJZ/6ao5D6MJwA/bDdPl1/O3a07VIfBvrw/Py9XsNC9tuxnQttxww+q33Qt2V7RzfM9DOgM0dClzgVej1b3dPNBL9KgDYni/A6Q+8p0ck9nfSALdhvDupvxujMQZL0wI4AXfW8QHrgkABdo0S6AycJ0KViJDrwJgECMr6ICGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEbCr/Aye3G3dDKm4IAAAAAElFTkSuQmCC', 'sha256': '25b93bc29dc2a30bf5e1ce25a5499a8722d565e22b7cfeab1fface8d9397c27b'}, 'data/binding-v1/images/binding-evaluation-0010-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFklEQVR4nO3aQU4bSxRAUfIVlhOJATvLOrIzBpaylQyQMokiS5Zl/EEE7Fvtd86IgXEXcvflVclfnp9/3QEU/kuuCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZrd+lte3h4fP0Fu93TtdbC5398K9vd0K0lQJe6a49ffEt3DHwiAbrGf8vDOygRHBOgf0zP7+/fXv/d+x8//+8NZWgdb36O67g/d0dtnQC9Iz3vulmPX3xy68gQ7AnQ2+n5+D/Jwzscl0iGQIBeq8+nz+f7NzzJkAYxlgCdr89FjwZOMqRBjOWLiH89PDxerT5nr3KyABhCgC6+7XrFybU0iGmmb8GuP/icsB1jstETUF6fs1c3BzHH6AAt9W20FdYAVzY3QIdBY50n/7ASQxBDDA3Q+k/4+iuEj5sYoHWOfl5yGMQoEwO0bH1WXhVcwrgALXj085LDIIYYFyBgHbMCtInxZ88QxASzAgQsZVCANjT+7BmCuHmDAgSsRoCAzJQAbW7/tWcXxm2bEiBgQQIEZAQIyIwI0EYPgPYcA3HDRgQIWJMAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZEQHa7Z72P9z/+Hm3NYc1H/4KuBkjAgSsSYCAjAABmSkB2ugxkAMgbtuUAAELEiAgMyhAm9uF2X9x8wYFCFjNrABtaAgy/jDBrAABSxkXoE0MQcYfhhgXoGNrNmjNVcElTAzQ8Vix2tN+vB7jDzdvYoA28Wyvv0L4uKEBWvMwyNEP08wN0LEVGrTCGuDKRgdoncMgRz/M9PVutt3u6eHh8bgCv79/u+YCTsKnPowyegI6+8xfcxRSH4YToL92u6frb8dOtl3qw0Bfnp9/1WtYyGE7tneh7ZjBZ7UPelt2N3TDTD8DeuVI6BKnQi9nq1u6meC9BOjubBFeZugjJTq7p5MesAX7x0H9zRi9cpAkPbAnQFc9L5AeOCZA1yiR7sBZAnSpGIkOvEmAgIwvIgIZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAXeVP6CeG3edJXwqAAAAAElFTkSuQmCC', 'sha256': 'ef2ccb3d9380cbe33ebb1b086b7c6a32d5a06b0f43c4d393be53a0106e9e4d86'}, 'data/binding-v1/images/binding-evaluation-0011-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFB0lEQVR4nO3dQU5bSRRAUWgl+2GQaW+oF8SGmGbgxTCwxCQDJMexjd02Jrf+r3OmBP6THK5eVSB+3G5fHwAK/yRPBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnWPXo9np5+nP8Dm83PvzULLMnjdvtaz7DO6HxEjGBHgP5Sd44pEQjQHdLz8vx2/nP//e/7Rx+SIWYmQDem52J0roqRDDEnAbq6Pjen53yGNIgJCdD/Tc9dunOxRDLEVPwcUFmf4698x0tuGJ8N6IT9Cnxdes6sQvYgJmEDGqI+B8+yBzEJARqiPsdP1CBm4Ag2Sn12nMWusuhSb6Z/fW1AY9XHHsRUBGis+hzPoEGsmN+G/8MI9Xn38vx25hc4GPm1u8gru2MD+r1ijPY3eDePJYi1mj1AS/neXsqccJXZA7Qz2voz8lRwL1MHaNjD1z4HMVZs6gABrXkDtMSFYokzwxnzBmhn5PPXUiaE2wgQkJk0QIu4ft7nKppVmjRAwAgECMgIEJCZMUCLuwB65xqI9ZkxQMAgBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyMwYoN374S7r7Zl203o/X1ZjxgABgxAgICNAQGbSAC3uGsgFEKs0aYCAEQjQApag8SeE28wboCWeZZY4M5wxb4CA3NQBWsRVtOtnVmzqAO0bs0FjTgX3MnuAlrJWLGVOuMrsARr5IObwxeoJ0B/GadA4k8DXEaDDFWOE7/z9Gaw/rJgADdcg9WEeAjRWg9SHqQjQQA1SH2YjQKM0SH2Y0Ld6gBFtNj+fnn7sd+Hl+e3rHneQOfVhHjag0w4q8HWrkPowMxvQh95bcLAK3WsbOi6a9DAhAbriOHaXQ9nJZUp9mJMAXb0KHXfkYozOnOCkh5kJ0Kcy9JkbIukBAbqxGidLdNVXAATo4fMduRgj0YGTBOgO9AVu4+eAgIwAARkBAjICBGQECMj4VzBWYoT/S5dr2YCAjAABmcft9rV7OjA1GxCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwEPlF1NfG0j5JekQAAAAAElFTkSuQmCC', 'sha256': '185487dc25d91171869dbcd79d927ee68e4ab58b3f12dc42f79655154e84f473'}, 'data/binding-v1/images/binding-evaluation-0011-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFB0lEQVR4nO3dQU5bSRRAUWgly0FikJ2xjt5ZBpayFQaWmGSA5HZsY7eNya3/65wpgf8kh6tXFYgft9vXB4DCP8lTAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZL51j16P5+cf5//AZvPzb80CS/K43b7WM6wzOh8RI9gRoL/UnWNKBAJ0h/S8vTyd/9zv//766EMyxMwE6Mb0XIzOVTGSIeYkQFfX5+b0nM+QBjEhAfq/6blLdy6WSIaYip8DKutz/JXveMkN47MBnbBfga9Lz5lVyB7EJGxAQ9Tn4Fn2ICYhQEPU5/iJGsQMHMFGqc+Os9hVFl3qzfSvrw1orPrYg5iKAI1Vn+MZNIgV89vwfxihPu/eXp7O/AIHI792F3lld2xA/60Yo/0N3s1jCWKtZg/QUr63lzInXGX2AO2Mtv6MPBXcy9QBGvbwtc9BjBWbOkBAa94ALXGhWOLMcMa8AdoZ+fy1lAnhNgIEZCYN0CKun/e5imaVJg0QMAIBAjICBGRmDNDiLoDeuQZifWYMEDAIAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMjMGaPd+uMt6e6bdtN7Pl9WYMUDAIAQIyAgQkJk0QIu7BnIBxCpNGiBgBAK0gCVo/AnhNvMGaIlnmSXODGfMGyAgN3WAFnEV7fqZFZs6QPvGbNCYU8G9zB6gpawVS5kTrjJ7gEY+iDl8sXoC9IdxGjTOJPB1BOhwxRjhO39/BusPKyZAwzVIfZiHAI3VIPVhKgI0UIPUh9kI0CgNUh8m9K0eYESbzc/n5x/7XXh7efq6xx1kTn2Yhw3otIMKfN0qpD7MzAb0ofcWHKxC99qGjosmPUxIgK44jt3lUHZymVIf5iRAV69Cxx25GKMzJzjpYWYC9KkMfeaGSHpAgG6sxskSXfUVAAF6+HxHLsZIdOAkAboDfYHb+DkgICNAQEaAgIwAARkBAjL+FYyVGOH/0uVaNiAgI0BA5nG7fe2eDkzNBgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQ8FD5DW7nIcmsH86DAAAAAElFTkSuQmCC', 'sha256': 'eb945f1cbff454e28a60cc7ce1155556135177ef9a10a8c22b22337ebd26f99c'}, 'data/binding-v1/images/binding-evaluation-0011-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFElEQVR4nO3dPU5bWRiAYRiF5SBRpM2G0rKOaWdD06awNFuhsESTAslxbDBjY/Kee8/ztPzcT0K8+s4By7fb7dMNQOGv5KkAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyX7pHr8fDw9fTn7DZ/PhTs8CS3G63T/UM64zOW8QIdgToD3XnmBKBAF0hPc+P96e/9u7v/976kAwxMwG6MD3vRuesGMkQcxKgs+tzcXpOZ0iDmJAA/d/0XKU775ZIhpiK/wMq63P8na94yQ3jswG9Yr8Cn5eeE6uQPYhJ2ICGqM/Bs+xBTEKAhqjP8RM1iBk4go1Snx1nsbMsutSb6X++NqCx6mMPYioCNFZ9jmfQIFbMq+F/M0J9Xjw/3p94AQfH/v3n+WYhvn2/q0cYhQ3o14oxTn0O5rEEsVazB2gpv9tLmRPOMnuAdkZbf0aeCq5l6gANe/ja5yDGik0dIKA1b4CWuFAscWY4Yd4A7Yx8/lrKhHAZAQIykwZoEdfP+1xFs0qTBggYgQABGQECMjMGaHEXQC9cA7E+MwYIGIQAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZGQO0ez/cZb3x1m5a7+fLaswYIGAQAgRkBAjITBqgxV0DuQBilSYNEDACAVrAEjT+hHCZeQO0xLPMEmeGE+YNEJCbOkCLuIp2/cyKTR2gfWM2aMyp4FpmD9BS1oqlzAlnmT1AIx/EHL5YPQH6zTgNGmcS+DwCdLhijPCbvz+D9YcVE6DhGqQ+zEOAxmqQ+jAVARqoQerDbARolAapDxP6Ug8wos3mx8PD1/0uPD/ef97jDjKnPszDBvS6gwp83iqkPszMBvSmlxYcrELX2oaOiyY9TEiAzjiOXeVQ9uoypT7MSYDOXoWOO/JujE6c4KSHmQnQhzL0kRsi6QEBurAar5borO8ACNDNxzvyboxEB14lQFegL3AZ/wcEZAQIyAgQkBEgICNAQMZfwViJb9/v6hE4mw0IyAgQkLndbp+6pwNTswEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBNxUfgK2FiEQ/AIHSgAAAABJRU5ErkJggg==', 'sha256': '89c5c24616db0478d1adc16c7cb266f2ba5726d0138670a18fe49cf5ecac76eb'}, 'data/binding-v1/images/binding-evaluation-0012-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFK0lEQVR4nO3dO1KbSRhAUewy+1HgZcxyvQwHLIaAKicTMKVSSVgGRtbtxzmpDd0B/9XXzUNfXl6eHwAKX5NVAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzrVt6MofD9+v/4enp5732Aov48vLyXO9h4uj8jhjBewjQLbtzSYngCgF6b3d+PD5e//B/fv268q9KBJcE6Fp9/hidD8VIg+CMAL2Rnk935z0lkiE42j1AZ/W5YXquZEiDYPcA3Sc9p2QIzmz6g4j3r8/lKjf8XhtMascJ6PTJv096roxC5iB2tt0ElNfnbF1zEDvbK0Aj1OdydQ1iWxsFaJz6XO5Bg9jTLndAo9XnyH3QnU3d+qflvkI2moAGrM+A+4F7+rrVi96YT/txV1O/OMMnrP/3gOZ6qg+H7+uN2WMa89XoE7/nPLUtJqDxv+BG3hv8PYsHaPDD1ykHMTa0coDmfZLn3Tl8yMoBOhp//Jlrn3ArWwQIGNOyAZro9ueUmyC2smyAgPEJEJBZM0CTnr9eOYWxjzUDBExBgICMAAGZBQM09QXQK9dAbGLBAAGzECAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgILNggI7vqzXvuykdd+49wljbggECZiFAQEaAgMyaAZr6GsgFEPtYM0DAFAQIyCwboElPYc5fbGXZAAHj2yJAswxBs+wTbmXlAM17ipl35/AhKwdorpsgtz9saPEAnRq5QSPvDf6e9QM010Ax127hf1o/QOMfxBy+2NYWATo1WoNG2w/c0y4BOh0uxnnmT3di/GFDuwRowAapD2wUoKEapD6wXYAGaZD6wKYBumzQPTN0tpz6sLkdA3T55N+nQWerqA98e9jV6/N/OHw/rcOPx8e/sZb0wJs2nYCujEK3nYYuP6H6wNG+E9DvRqHTgeXTA9GbFZMeOCNA/3l6+nnaoMuO/DFG10cn9YFLAvRGIy5L9OmLat2BKwTowyX60GcArhCg93bkjzESHfgoAXovfYGb2/3b8EBIgICMAAEZd0BsaoS/CYUJCMgIEJD58vLy3K0ObM0EBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIfKvyF9Q6tXaASnAAAAAElFTkSuQmCC', 'sha256': 'fdb92a5d1b6836fa0e503608f22a134f3ca946263e5980d4c99c002d9c6b3136'}, 'data/binding-v1/images/binding-evaluation-0012-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFOklEQVR4nO3dO24bSRRAUcmw96NAwESzDS9iFuRFzDYcGXCgxSgQoGQCAQRBcqiPKd6uqnNSf/oF6qtXZcG8fXp6vAEofEmeCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZr92jB3N3d3/+Nzw8/L7WLDCJ26enx3qGgaPzf8QI3kKALtmdY0oEZwjQW7vz88fz+T/+9z/fzvyqEsExATpXn1ej864YaRAcEKAT6flwd95SIhmCndUDdFCfC6bnTIY0CFYP0HXSs0+G4MCiP4h4/focP+WC/9YGg1pxA9p/86+TnjOrkD2IlS23AeX1OXiuPYiVrRWgLdTn+OkaxLIWCtB26nM8gwaxplXugLZWnx33QVc2dOsfpvsKWWgD2mB9NjgPXNOXpb7pbfNt30019Ddn+ID5/z+gsd7qu7v7+dbsbfr21783g3j+9f1mUktsQFtef7Y/G3yeyQO08cPXPgcxFjRzgMZ9k8edHN5l5gDtbH/9GWtOuJQlAgRs07QBGuj2Z5+bIJYybYCA7RMgIDNngAY9f71wCmMdcwYIGIIAARkBAjITBmjoC6AXroFYxIQBAkYhQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAZsIA7T5Xa/9Tj8eym9xnhDG3CQMEjEKAgIwAAZk5AzT0NZALINYxZ4CAIQgQkJk2QIOewpy/WMq0AQK2b4kAjbIEjTInXMrMARr3FDPu5PAuMwdorJsgtz8saPIA7dtyg7Y8G3ye+QM01kIx1rTwh+YP0PYPYg5fLGuJAO3bWoO2Ng9c0yoB2l8utvPO709i/WFBqwRogw1SH1goQJtqkPrAcgHaSIPUB158vVnPw8Pvu7v7/Rb8/PF8nUcfJE99WNxyG9DJN/86q5D6wIEVN6D99/86q5D0wEnrBuj4OPYZGTrerdQHdlYP0PEqtF+ND5fo5JlOeuCAAJ1ehY478mqMzl8kqQ8cE6ATjTgu0YcvqnUHzhCgd5foXX8DcIYAvbUjr8ZIdOC9BOit9AUubtEfRAS2QICAjAABGXdALOr51/d6BGxAQEeAgMzt09Nj93RgaTYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE3FT+AyVTMHO4VQACAAAAAElFTkSuQmCC', 'sha256': '8c77af4b5555e4eef4de2ad760d6cc1e34ae369d20e34c8314b92549f4b82311'}, 'data/binding-v1/images/binding-evaluation-0012-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFO0lEQVR4nO3du21bSRiAUcmw+1HgdDtwJVuQK9kOnDpQMQoEKNnAAEGQWlrS0vzmcU6qx52A9+M/I1K8f35+ugMofEquCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZz92lJ/Pw8PXyNzw+/rzVWmAR98/PT/UaJo7OfxEjeAsBumZ3zikRXCBAb+3Oj+8vl3/8r7+/XPiqEsE5AbpUn99G510x0iA4IUCvpOfD3XlLiWQIDnYP0El9rpieCxnSINg9QLdJzzEZghObvhDx9vU5v8oV/9YGk9pxAjq+82+TngujkDmInW03AeX1ObmuOYid7RWgEepzfnUNYlsbBWic+pyvQYPY0y5nQKPV58B50I1N3frH5R4hG01AA9ZnwPXALX3a6klvzLv9sKqpn5zhA9b/f0Bz3dUPD1/XG7PH9M+XS28eHsq3lxGfOK9iiwlo5PFn/LXBn7N4gAbffB2zEWNDKwdo3jt53pXDu6wcoIPxx5+51gnXskWAgDEtG6CJTn+OOQliK8sGCBifAAGZNQM06f7rF7sw9rFmgIApCBCQESAgs2CApj4A+sUxEJtYMEDALAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgsGKDD52odf+rxXA4r9xlhrG3BAAGzECAgI0BAZs0ATX0M5ACIfawZIGAKAgRklg3QpLsw+y+2smyAgPFtEaBZhqBZ1gnXsnKA5t3FzLtyeJeVAzTXSZDTHza0eICOjdygkdcGf876AZproJhrtfA/rR+g8TdiNl9sa4sAHRutQaOtB25plwAdDxfj3PPHKzH+sKFdAjRgg9QHNgrQUA1SH9guQIM0SH3gl893+3l8/Pnw8PW4BT++v9zm0ifJUx82t90E9Oqdf5tRSH3gxI4T0PH9f5tRSHrgVfsG6Hw79icydD5bqQ8c7B6g81HouBofLtGrezrpgRMC9PoodN6R38bo8kGS+sA5AXqlEecl+vBBte7ABQL07hK96zcAFwjQWzvy2xiJDryXAL2VvsDVbfpCRGAEAgRkBAjIOANiU99ebvQOZC4wAQEZAQIy98/PT93Vga2ZgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEHBX+ReNMzCbKijxJAAAAABJRU5ErkJggg==', 'sha256': '1b7254ddd65df2e5813cad2ab910216da329b31ec7ef9b608c66b968b93f2fe3'}, 'data/binding-v1/images/binding-evaluation-0013-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKklEQVR4nO3dPW4TQRiAYYPgPimQuAmH5CZIFDkMRSQaCkuWFScmPxu/OzPPU1FE3kHKvvlm4qw/PTz8OQAUPidXBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfOkuDVu6u/t2mNT9/e/DpExAQEaAgIwtGLP5+v3nYQp/f/04zM4EBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg449Rt3nWzMRPbIGPI0DbPODq/IvFCF5IgLZ/sN7pFZQIrhOgF3Xnv4+YefLRLUoE1wnQs/V51XOtzr/4MkZ3d980CC6tHqDL9Lz/eXqnVzgv0fFCMgTnlv41/KP6fP3+c9uneV6+4MSf3ABvsOgEdJmej7vW8cVP05BRCJaegG5Zn+euYhSCFQN0fudvvue67tHlNAg+r1yfZA0aBCsGaA/1uby6BrGyVQK0n/pcrkGDWNYSAdpbfY40CJYI0A7rs8/1wI3NH6DTcLHPu/20KkMQC5o8QGPd1WOtFt5v8gCd7HP82f/a4EPNHKCdb77O2YixpmkDNO6dPO7K4bWmDdDJ/sefsdYJG5o/QMBuzRmggU5/zjkJYjVzBggYggABmQkDNOj+68gujKVMGCBgFAIEZAQIyMwWoKEPgI4cA7GO2QIEDESAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMbAG6v/99/MffXz8OYzqt/PR/gVnNFiBgIAIEZAQIyEwYoKGPgRwAsZQJAwSMQoCAzJwBGnQXZv/FauYMEDCE+QM0yhA0yjphQ9MGaNxdzLgrh9eaNkBjnQQ5/WFNMwfo3J4btOe1wYeaPEBjDRRjrRbe79PDw5/D7Pb8aYU2X1uZ+HMc7+f93ph8Atr5Zmdv64EbWyJA5z9A9nPPn69k4h9xsPoW7HJEz/di6gOrTEB7m4PUB5YL0E4apD6w4hbsuV+X3Gw79ih56gNrTUBP3vm3GYXUBy6tOAHdeBSSHnjO0gF68t1rG2bocrZSHzi3eoCuvIn2zSV6ck8nPXBJgF70Rv7/xuj6QZL6wJME6AP/pEh34DoB2r5EugMvJEDbxEh04A0ECMis+EZEYCcECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgQcKv8AJm0VWAPuUA8AAAAASUVORK5CYII=', 'sha256': 'c90d9fc718c4e9c1d0a6a1eabb3bc94f7b76b5426123dc3e7f00278845fe0d2d'}, 'data/binding-v1/images/binding-evaluation-0013-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMklEQVR4nO3dMW4bRxiAUTqIjhNAhVtfyG3OkTYXSutCgK/iQoCbFAQIQqRoySL57cy8V7mQuWPA++mf0Yr89Pz8YwdQ+CO5KoAAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5s/u0nBNj4+fd5N6evq2m5QJCMgIEJCxBWM2P//+azeFh3++72ZnAgIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPHLqNd5r5mJ37EFbkeArvMGV8dfLEbwRgJ0/TfWO7yCEsFlAvSm7vz378/Lf/3L14cLr6lEcJYAvVqfX0bntS8+jdHj42cNglOrB+g0Pe/qzlmHVzgu0f5CMgTHlg7Qi/p8PD0v7F/wRYY0CFYP0K3TcyFDRiFY+kHEe9bntatM/CFW8HbLTUDHd/590nNhFDIHsbi1JqCwPmevaw5icQsFaAv1Ob26BrGyVQK0nfqcrkGDWNYSAdpaffY0CJYI0Abrs831wJ3NH6DDcLHNu/2wKkMQC5o8QGPd1WOtFj5u8gAdbHP82f7a4KZmDtDGN1/HbMRY07QBGvdOHnfl8F7TBuhg++PPWOuEK5o/QMBmzRmggU5/jjkJYjVzBggYggABmQkDNOj+a88ujKVMGCBgFAIEZAQIyMwWoKEPgPYcA7GO2QIEDESAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMbAF6evq2/8OXrw+7MR1Wfvi3wKxmCxAwEAECMgIEZCYM0NDHQA6AWMqEAQJGIUBAZs4ADboLs/9iNXMGCBjC/AEaZQgaZZ1wRdMGaNxdzLgrh/eaNkBjnQQ5/WFNMwfo2JYbtOW1wU1NHqCxBoqxVgsf9+n5+cdudlv+tEKbr2uZ+HMcn+b9vzH5BLTxzc7W1gN3tkSAjr+BbOeeP17JxN/iYPUt2OmInu/F1AdWmYC2NgepDywXoI00SH1gxS3Yaz8uudt27EXy1AfWmoDO3vn3GYXUB06tOAHdeRSSHnjN0gE6+/TaFTN0OlupDxxbPUAXHqL97RKd3dNJD5wSoDc9yP/LGF0+SFIfOEuAbvgrRboDlwnQ9UukO/BGAnSdGIkO/AYBAjIrPogIbIQAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBd5X+CTieunDQmkgAAAABJRU5ErkJggg==', 'sha256': 'b64c16e0a5941059afbbab95ce940e6be63fcc5ae8a038794b837e5428ab2672'}, 'data/binding-v1/images/binding-evaluation-0013-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNElEQVR4nO3dMW4bRxiAUTqI7qPCgCtfI4fwgXKIXCNVABc6jAsBblIQIAiRoiWL5Lcz817lQuaOAe+nf0Yr8tPz848dQOGP5KoAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZk/u0vDNT0+ft5N6unp+25SJiAgI0BAxhaM2Tx8+Wc3hZ///bWbnQkIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDGL6Ne571mJn7HFrgdAbrOG1wdf7EYwRsJ0PXfWO/wCkoElwnQm7rz798/L//1r98eLrymEsFZAvRqfX4Znde++DRGj4+fNQhOrR6g0/S8qztnHV7huET7C8kQHFs6QC/q8/H0vLB/wRcZ0iBYPUC3Ts+FDBmFYOkHEe9Zn9euMvGHWMHbLTcBHd/590nPhVHIHMTi1pqAwvqcva45iMUtFKAt1Of06hrEylYJ0Hbqc7oGDWJZSwRoa/XZ0yBYIkAbrM821wN3Nn+ADsPFNu/2w6oMQSxo8gCNdVePtVr4uMkDdLDN8Wf7a4ObmjlAG998HbMRY03TBmjcO3nclcN7TRugg+2PP2OtE65o/gABmzVngAY6/TnmJIjVzBkgYAgCBGQmDNCg+689uzCWMmGAgFEIEJARICAzW4CGPgDacwzEOmYLEDAQAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMrMF6Onp+/4PX7897MZ0WPnh3wKzmi1AwEAECMgIEJCZMEBDHwM5AGIpEwYIGIUAAZk5AzToLsz+i9XMGSBgCPMHaJQhaJR1whVNG6BxdzHjrhzea9oAjXUS5PSHNc0coGNbbtCW1wY3NXmAxhooxlotfNyn5+cfu9lt+dMKbb6uZeLPcXya9//G5BPQxjc7W1sP3NkSATr+BrKde/54JRN/i4PVt2CnI3q+F1MfWGUC2tocpD6wXIA20iD1gRW3YK/9uORu27EXyVMfWGsCOnvn32cUUh84teIEdOdRSHrgNUsH6OzTa1fM0OlspT5wbPUAXXiI9rdLdHZPJz1wSoDe9CD/L2N0+SBJfeAsAbrhrxTpDlwmQNcvke7AGwnQdWIkOvAbBAjIrPggIrARAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECdpX/AfQiJ65vf+7nAAAAAElFTkSuQmCC', 'sha256': 'cb0238fa73738458d1f5fe797d00ffa8bfe56158ccc2cf5af95d213346a08943'}, 'data/binding-v1/images/binding-evaluation-0014-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQklEQVR4nO3cQU4bSRiAURiF40Riwc1yjtyMhaVchYWlbGaBZDk4Y4PH8FV1vbdCAty/Bf2pugpxv9+/3AEU/kmuCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnWXXprHh+fzn/Bbvf8VbPAHO73+5d6hs0W5zw9AgEK0nNMhliZAN2sO79/fD//7Q8/f535rBKxIAH6X+m5GJ2PxkiGPnW9ObLdkj96m9DX3ABXd+f0Fd6U6PHxac1fRNYkQF+dnr++4HGGXi8qQ6xAgN5bn5un52KGNOiNT/0RVB7O7gxungBdrs+X/d6/yZAGsXn+EnqU+vz1iovsv7IsARqrPqfX1SA2zCPYHw53e77dcPw45lmMrbICGrE+B4dJrIPYJAEatz6vNIgNE6CZ7u1Z5oR3EqAhdp3PsyfNVgnQ3eD1GX82uNrqARp26+eUzSC2Z+kAzXsnzzs5HFs6QAfjL3/mmhPead0Azb6ImH1+WDpAky4r5poWzhMgILNogCY6/DrlOIzNWDRAwAhWDNCWFg5bei8saMUAHcz4/DX75HBs6QABLQECMgIEZJYL0NQH8MccxrMBywUIGIcAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZ5QK02z2/fvDw89fdzA7zH94RTGe5AAHjECAgI0BAZukAzbsNNO/ksHqAtrRru6X3woJWDBAwiEUDNPVhvAN4NmPRAAEjEKDJFkFzTQvnrRug2Z9fZp8flg7QjMuKWeaEd1o6QPMuIuadHI4tHaC5jsMcfrE9qwfo2MgNGnk2uJoA/bGgGPM+P57K8octEaCZ7upZ5oR3EqDRN4Ns/bBhAjR0g9SHbftWDzCW3e758fHpcOf//vG9msS+DyuwAhpxT1p9WIQADdcg9WEdHsEuPIt95ePYm9ipD5snQO9q0Gdn6HSdpT6sQIDOea3Ap2ZIeliZAH14KXRcjatL9F9bS+rDUgToyqXQaUcuxuj8frb0sCAB+oBDI05LdPV5me6wMgG65YLoiheBlQnQzQpysUeKA28I0M3oC3yUv4QGMgIEZAQIyAgQkBEgICNAQMYxPNMY51/lcitWQEBGgIDM/X7/0l0dWJoVEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAF3lX8B6NQ3zzCLmrgAAAAASUVORK5CYII=', 'sha256': 'f3720aeedebf3c99ff54144020052b53ee516d7ec0ce92b16918095532b58cce'}, 'data/binding-v1/images/binding-evaluation-0014-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQklEQVR4nO3cQU4bSRiAURgl9/GCLRfKgXKhbFn4MCyQspkFkuXgjA0ew1fV9d4KCXD/FvSn6irE/cvL8x1A4Z/kqgACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZlv3aW3Zrd7OP8F+/3TV80Cc7h/eXmuZ9hscc7TIxCgID3HZIiVCdDNuvPr5+/z3/744/uZzyoRCxKg/5Wei9H5aIxk6FPXmyPbL/mjtwl9zQ1wdXdOX+FNiXa7hzV/EVmTAH11ev76gscZer2oDLECAXpvfW6enosZ0qA3PvVHUHk8uzO4eQJ0uT5f9nv/JkMaxOb5S+hR6vPXKy6y/8qyBGis+pxeV4PYMI9gfzjc7fl2w/HjmGcxtsoKaMT6HBwmsQ5ikwRo3Pq80iA2TIBmurdnmRPeSYCG2HU+z540WyVAd4PXZ/zZ4GqrB2jYrZ9TNoPYnqUDNO+dPO/kcGzpAB2Mv/yZa054p3UDNPsiYvb5YekATbqsmGtaOE+AgMyiAZro8OuU4zA2Y9EAASNYMUBbWjhs6b2woBUDdDDj89fsk8OxpQMEtAQIyAgQkFkuQFMfwB9zGM8GLBcgYBwCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRklgvQfv/0+sHjj+93MzvMf3hHMJ3lAgSMQ4CAjAABmaUDNO820LyTw+oB2tKu7ZbeCwtaMUDAIBYN0NSH8Q7g2YxFAwSMQIAmWwTNNS2ct26AZn9+mX1+WDpAMy4rZpkT3mnpAM27iJh3cji2dIDmOg5z+MX2rB6gYyM3aOTZ4GoC9MeCYsz7/Hgqyx+2RIBmuqtnmRPeSYBG3wyy9cOGCdDQDVIftu1bPcBY9vun3e7hcOf/+vm7msS+DyuwAhpxT1p9WIQADdcg9WEdHsEuPIt95ePYm9ipD5snQO9q0Gdn6HSdpT6sQIDOea3Ap2ZIeliZAH14KXRcjatL9F9bS+rDUgToyqXQaUcuxuj8frb0sCAB+oBDI05LdPV5me6wMgG65YLoiheBlQnQzQpysUeKA28I0M3oC3yUv4QGMgIEZAQIyAgQkBEgICNAQMYxPNMY51/lcitWQEBGgIDM/cvLc3d1YGlWQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgTcVf4FFCE1pBgjuzMAAAAASUVORK5CYII=', 'sha256': 'f018728bffb6cd747cc5d0ce4bdf4273081b48ad2c263c87858e4ddd861c8bb8'}, 'data/binding-v1/images/binding-evaluation-0014-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFSklEQVR4nO3aMW4bVxRAUSqwlhOAhVpvyG3W4dYbcuuCgLeigoCaFAIIWlRIipF0588/pzJgm/MIey7e/NHdfv+4ASj8lVwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg86W79Npstw/n/8Bu9+uzZoEx3O33j/UMqy3OeXoEAhSk55gMMTMBerfu/PzxdP6vf/12f+Z3lYgJCdD/Ss/F6Lw1RjL0ofvmku2m/Kd3CH3LDXBzd04/4UWJttuHOf8jMicB+uz0vPqBxxl6vqgMMQMBurY+756eixnSoBee/vl7szr3339vJiZAl+vzoek5kyENYvX8JPRS6vPqFSc5f2VaArSs+pxeV4NYMY9gfzjc7VV6Xn0c8yzGWtmAllifg8Mk9iBWSYCWW59nGsSKCdBI9/Yoc8KVBGgRp87nOZNmrQRos/D6LH82uNnsAVrs0c8ph0Gsz9QBGvdOHndyODZ1gA6Wv/6MNSdcad4Ajb5EjD4/TB2gQdeKsaaF8wQIyEwaoIFefp3yOozVmDRAwBLMGKA1LQ5r+i5MaMYAHYz4/DX65HBs6gABLQECMgIEZKYL0NAv4I95Gc8KTBcgYDkECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjITBeg3e7X8y++frvfjOww/+EbwXCmCxCwHAIEZAQIyEwdoHGPgcadHGYP0JpObdf0XZjQjAECFmLSAA39Mt4LeFZj0gABSyBAgy1BY00L580boNGfX0afH6YO0IhrxShzwpWmDtC4S8S4k8OxqQM01uswL79Yn9kDdGzJDVrybHAzAfpjoVjmfX48lfWHNRGgke7qUeaEKwnQ0g+DHP2wYgK06AapD+v2pR5gWXa7X9vtw+HO//njqZrEuQ8zsAEt8UxafZiEAC2uQerDPDyCXXgW+8zHsRexUx9WT4CuatBHZ+h0z1IfZiBA5zxX4EMzJD3MTIDevAodV+PmEv3X0ZL6MBUBunEVOu3IxRidP8+WHiYkQG9waMRpiW5+X6Y7zEyA3nMhuuFDYGYC9G4FudgjxYEXBOjd6Au8lZ+EBjICBGQECMgIEJARICAjQEDGa3iGcf/9dz0C78wGBGQECMjc7feP3dWBqdmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECNhU/gU7xDZdkLlDoQAAAABJRU5ErkJggg==', 'sha256': 'b67151f86f7354adef32bda2bbd6e775bf7d13ce43350a22e8df6dbdb41659af'}, 'data/binding-v1/images/binding-evaluation-0015-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNUlEQVR4nO3aMU4sRxRAUbDMfghIvYO/Ei/IK2EHTn/AYgiQSBwgjUaYPxiEudX1zokIYKZa0315XdPXT0+PVwCF35J3BRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZH7v3nort7d3/+XXHh5+/v9rgcO4fnp6rNeweXR+RYyO+9mt4GGL88cElJ27p9fZ40yCTxCgL0vP/c3N5T//8fx84WVliIEE6PPpebc4l3//VY9kKPH3X2//V1jWH39+7KxbnAB9R3re9PIiMsRkAvSB+nxJd145veZ5iW5v7zSICQQoS8/lgcgoxAQeRFyiPr96rw2+LYYLBOi182v+/ubmO+vz5ptqEBsToEv1CVeiQUwgQCvW599r0CC2JECL1ueFBrE3AVq3Pi80iI0J0NL1eaFB7Gp6gNavzwsNYkvTA3SI+hxlhfBRowN0GiWOcm2f1mkIYg9zA3T0a/jo64fRATo5yvhzxNXCZUMDdLibr3NuxNjG0AABK5gYoEOPPy8MQexhYoCARYwL0H4jw35HxBzjAnRy3PuvPdYPowME5GYFaIPt53O2ojm6WQECliJAQEaAgMygAO29UbL30bGrQQE62WMHer9jYaCJAQIWIUBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQGZigH48P1/tYqdjYaBBAXp4+Hm1r72Pjl0NChCwGgECMgIEZGYF6LRRssfe7ekobABxULMCBCxlboCOPgQdff0wMUD73a3sd0TMMS5AwDomBmiDrWjbz+xhYoCARQwN0KGHIOMP2xgaoHPHatCxVguXzQ3Q0ceHo68fRgfoiDdibr7YzOgAnVu/QeuvED5qeoDOR4mVr/DztRl/2Mb0AB2iQerDrgRo9QapDxu7fnp6rNewitvbu9PP9zc3VwtQn2/4rA/qYYvzwQS07hykPmxPgBZtkPowgVuw9+fzb74dexU+9WFjJqA3vLrmv3MUUh9GMQGtMgpJDwMJ0Ge+LvnCEr05XqkPQwhQliHpAQH6sodH3u3R5b0k6WEgAeqfYZMexhKgrES6AwL0rTESHTgnQEDGg4hARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAq4q/wDhKzD2hctS6gAAAABJRU5ErkJggg==', 'sha256': 'd0b4187403e6a5ec26e6a4edfe7ea6f1e80ec487d5f076dcd7f0a0328e11210f'}, 'data/binding-v1/images/binding-evaluation-0015-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNUlEQVR4nO3aPU4jWRhAURgN+3FA2juYlfSCeiWzg047YDEElkgmsGRZDG0GxPi+n3MiArBfyVWXr57r/nh8vgMo/JG8K4AAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPNn99ZLORwe/8uvPT39+v/XAtO4Px6f6zUsHp3fEaN5P7sRPC1x/piAsnP3/DprnEnwCQL0Zen5+ePl+p9/+/5w5WVliA0J0OfT825xrv/+qx7JUOLvh7f/Kwzrr5ePnXWDE6BbpOdNpxeRIXYmQB+oz5d055Xza16W6HB41CB2IEBZeq4PREYhduBBxCHq87v3WuDbYrjCBPTa5TV/y/RcGYXMQazKBDRWfd58d3MQqxKgEevz7zVoEEsSoEHrc6JBrE2Axq3PiQaxMAEauj4nGsSqdg/Q+PU50SCWtHuApqjPLCuEj9o6QOdRYpZr+7xOQxBr2DdAs1/Ds68ftg7Q2Szjz4yrhes2DdB0N1+X3IixjE0DBIxgxwBNPf6cGIJYw44BAgaxXYDWGxnWOyL2sV2Azua9/1pj/bB1gIDcXgFaYPv5kq1oZrdXgIChCBCQESAgs1GA1t4oWfvoWNVGATpbYwd6vWNhQzsGCBiEAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmR0D9O37w90qVjoWNrRRgJ6eft2ta+2jY1UbBQgYjQABGQECMnsF6LxRssbe7fkobAAxqb0CBAxl3wDNPgTNvn7YMUDr3a2sd0TsY7sAAePYMUALbEXbfmYNOwYIGMSmAZp6CDL+sIxNA3RprgbNtVq4bt8AzT4+zL5+2DpAM96IufliMVsH6NL4DRp/hfBRuwfocpQY+Qq/XJvxh2XsHqApGqQ+rEqARm+Q+rCw++PxuV7DKA6Hx/PPP3+83A1AfW7wWU/qaYnzwQQ07hykPixPgAZtkPqwA7dg78/nN74dexU+9WFhJqA3vLrmbzkKqQ9bMQGNMgpJDxsSoM98XfKFJXpzvFIfNiFAWYakBwToyx4eebdH1/eSpIcNCVD/DJv0sC0BykqkOyBAN42R6MAlAQIyHkQEMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEHBX+QeMkD9Et8MIMAAAAABJRU5ErkJggg==', 'sha256': 'fbfa1cf1dfeccbbe49cf4118be34916dca354a789d256deca8b369f512b77b33'}, 'data/binding-v1/images/binding-evaluation-0015-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFLUlEQVR4nO3aPU4jWRhAURhN78cBKRvqBfWGOu3AiyGw1MkEliyLBjMgmvt+zokIjP1Kdl1/9Vz3p9PTHUDhn+RVAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmX+7l17K4fDwfx52PP76+2uBadyfTk/1GhaPzmvEaN73bgTHJT4/JqDss3t5njU+SfABAvRp6fn54/ftf3/8/u3G08oQGxKgj6fnzeLcfvyzHslQ4r1vYu7xla+xSQnQV6TnRecnkSF2JkDvqM/f+La8POd1iQ6HBw1iBwKUpef2QGQUYgduRByiPq+91gK/FsMNJqDnrs/5ZIfyz1HIHMSqTEBj1efFVzcHsSoBGrE+f65Bg1iSAA1anzMNYm0CNG59zjSIhQnQ0PU50yBWtXuAxq/PmQaxpN0DNEV9ZlkhvNfWAbqMErOc25d1GoJYw74Bmv0cnn39sHWALmYZf2ZcLdy2aYCmu/i65kKMZWwaIGAEOwZo6vHnzBDEGnYMEDCI7QK03siw3hGxj+0CdDHv9dca64etAwTk9grQAtvP12xFM7u9AgQMRYCAjAABmY0CtPZGydpHx6o2CtDFGjvQ6x0LG9oxQMAgBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyOwYoMfv3+5WsdKxsKGNAnQ8/rpb19pHx6o2ChAwGgECMgIEZPYK0GWjZI2928tR2ABiUnsFCBjKvgGafQiaff2wY4DWu1pZ74jYx3YBAsaxY4AW2Iq2/cwadgwQMIhNAzT1EGT8YRmbBujaXA2aa7Vw274Bmn18mH39sHWAZrwQc/HFYrYO0LXxGzT+CuG9dg/Q9Sgx8hl+vTbjD8vYPUBTNEh9WJUAjd4g9WFh96fTU72GURwOD5e/f/74fTcA9fmC93pSxyU+Dyagcecg9WF5AjRog9SHHbgEe3s+/+LLsWfhUx8WZgJ6wbNz/itHIfVhKyagUUYh6WFDAvSRn0s+sUQvjlfqwyYEKMuQ9IAAfdrNI2/26PZekvSwIQHq72GTHrYlQFmJdAcE6EtjJDpwTYCAjBsRgYwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgTcVf4DpVU/RF9E8jwAAAAASUVORK5CYII=', 'sha256': 'ac613dcaac2582c9b4f267e9b2875ad37fcab38d454b30e7a6e5c37a05f9a984'}, 'data/binding-v1/images/binding-evaluation-0016-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFN0lEQVR4nO3dTW4TWRSA0aJFLQepBlkGu2Ed7KaXwcASW2EQyZMeWFju4Bjnz1+V3zkjkILrRqg+3VdynE+Pj78mgMI/yVUBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZnP3aVXZFkepo3b7X7UI8CL2YCAjAABGUew//l3nqdN+brf1yPA69mAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjI+Gl4eOePr/PhcNcTIHjnz8w8/WIxukyA4AM/q/f4Ckp0lgDBy7qz//bl8j+fv/+88JpKdEqA4O/1+Wt0nvviP2O0LA8adCRA8Gx6XtSds46vcFqiw4V2MiRAcLY+b0/PE4cXfJKh3fANEiBG99HpuZChZfhVyBsRGdot6/PcVZbt/17MV7MBMa7TO/826bmwCu2G3INsQAwqrM/Z6y5D7kECxIjWUJ+DwRskQAxnPfU5GLlBAsRY1lafwRskQAxqPfVZ5zy3IUAM5LhcrPNu3/+eapwlSIAYxbbu6mVT076aADGcda4/65/tIwgQQ1j54WvYg5gAcf+2eycvm538SgLEQNa//mxrzrcTICAjQNy5DT39GfBJkAABGQECMgLEPdvo+WucU5gAARkBAjICBGQEiLu16QdAgzwGEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAuFu73Y/DH+bvP6dtmn9Pfvxe7owAARkBAjICBGQEiHu26cdA870/AJqm6XM9wLp83e/rEWAgNiAgI0DcuY2ewuYBzl+OYNMI/8ewWjYgBrKVJWjeyJxvJ0Dcv+1uuLvNTn4lAWIIG3oSNI/x9OdAgBjOmhs0r3i2jyBAjGJbC8VuU9O+mgAxkJUfxOaRDl8HAsSg1tageWXz3IYAMZbT5WI99/x8Msk4648AMaK1NWgetT4CxKDW06B54PoIEONaQ4PmseszTdOnx8df9QyQWZaH07/uv325zXWfJG83ZH1sQIzuyZ1/m1VIfY5sQHC7VUh6nhAgON+g983Qn7vVbvj6CBD8PUNvKdHZM530HAkQXNWg62N0+UGS+pwSIHhliV5Ed84SIPjAEunOZQIE7xwj0bmeAAEZb0QEMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAqbKf0mnIF0fXTr7AAAAAElFTkSuQmCC', 'sha256': '1e1711365a4f8ec073de8b0b57f5182ba0031f274fa5dadf7049ccde0913688e'}, 'data/binding-v1/images/binding-evaluation-0016-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPklEQVR4nO3dTW4TWRSA0aKFl9OSB1lG7abXwW6yDAaW2AqDSJkwsLDcieM4f/6q/M4ZgRSqLkL16b4KmG8PD78ngMI/yV0BBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZnv3a0XZLu9m1Zut/tZjwBvZgMCMgIEZBzB/ufxv3+nVdn8+FWPAO9nAwIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIONfw8Mnf3ydD4e7nADBJ39m5vEXi9F5AgRf+Fm9hyso0UkCBG/rzv1mc/6Xz4+PZ66pRMcECF6vz6vReemLn8dou73ToAMBghfT86bunHS4wnGJ9jfayZBvw8PJ+txvNh+vz/kLbtf/n0F9nA2I0T1Pz9fda3/xwza0HX4VsgExtGvW56W7bAdehQSIcR0/+Z9+5jrvye22ozZIgBjUk/okM9wP3yABYkRLqM/e4A0SIIaznPrsjdwgAWIsS6vP4A0SIAa1nPosc57rECAGclgulvm03/+dapwlSIAYxbqe6u2qpn03AWI4y1x/lj/bVxAghrDww9ewBzEB4vat90nernbyCwkQA1n++rOuOT9OgICMAHHjVvT2Z8A3QQIEZAQIyAgQt2yl569xTmECBGQECMgIEJARIG7Wql8ADfIaSICAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkB4mbtdj/3P5gfH6d1mv9Ofvi93BgBAjICBGQECMgIELds1a+B5lt/ATRN0/d6gGXZ/PhVjwADsQEBGQHixq30FDYPcP5yBJtG+DOGxbIBMZC1LEHzSub8OAHi9q13w92tdvILCRBDWNGboHmMtz97AsRwltygecGzfQUBYhTrWih2q5r23QSIgSz8IDaPdPjaEyAGtbQGzQub5zoEiLEcLxfLeebno0nGWX8EiBEtrUHzqPURIAa1nAbNA9dHgBjXEho0j12faZq+PTz8rmeAzHZ7d/zT+83mOvd9krzdkPWxATG6J0/+dVYh9TmwAcH1ViHpeUKA4HSDPjdDz3er3fD1ESB4PUMfKdHJM530HAgQXNSgy2N0/kWS+hwTIHhnid5Ed04SIPjCEunOeQIEnxwj0bmcAAEZfxERyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECJgqfwCdXBTHNLlUnQAAAABJRU5ErkJggg==', 'sha256': 'c26b05ae77919bad4975d9b75e4f5b3b31bea5aa5b64ef2bb4eaed292afefad9'}, 'data/binding-v1/images/binding-evaluation-0016-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMklEQVR4nO3dQW4TSxRA0eYL76cHWUYvN8tg4MUwiJTJH1hYJjGJkzi+3a5zRiBF7kLQV68qUPx4evo9ART+S54KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMj87B69IvP8MG3cfv+rXgJ8mAkIyAgQkLEF+8vjbjdtyvL8XC8BPs8EBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAxr+GhytfX+dyuMsJEFz5zszTLxajtwkQfONdvcdPUKKzBAg+1p13b607e0ucEp0lQPB+fT50VebpF7+O0Tw/aNCRAME/0/P1K3qPn3BaosOD9jLk2/Bwtj6Pu911Lwh//YHz9v8zqK8zATG61+n5vmcdPvw4Dc3Dj0ImIIZ2y/r86ynzwKOQADGu0zf/6nuut7143DxqgwSIQb2oT7KGx+EbJECMaA31ORi8QQLEcNZTn4ORGyRAjGVt9Rm8QQLEoNZTn3Wu5zYEiIEch4t1vu2Pf1Y1zhAkQIxiW2/1vKnVfpoAMZx1jj/rX9t3ECCGsPLN17AbMQHi/m33TZ43u/ILCRADWf/4s611fp0AARkB4s5t6PRnwJMgAQIyAgRkBIh7ttH91zi7MAECMgIEZAQIyAgQd2vTB0CDHAMJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0Dcrf3+1+EHy/PztE3Ln5Uffy13RoCAjAABGQECMgLEPdv0MdBy7wdA0zT9rBewLlv8YwrbZQICMgLEndvoLmwZYP9lCzaN8HsMq2UCYiBbGYKWjazz6wSI+7fdCXe/2ZVfSIAYwoZOgpYxTn8OBIjhrLlBy4rX9h0EiFFsa6DYb2q1nyZADGTlG7FlpM3XgQAxqLU1aFnZem5DgBjL6XCxnnd+OVnJOOOPADGitTVoGbU+AsSg1tOgZeD6CBDjWkODlrHrM03Tj6en3/UaIDPPD6c/fdztbvPcF8nbD1kfExCje/Hm32YUUp8jExDcbhSSnhcECM436LoZej1b7YevjwDB+xn6SonO7umk50iA4KIGXR6jtw+S1OeUAMEnS/QhunOWAME3lkh33iZAcOUYic7lBAjI+IuIQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMBU+R9WdRWkGlM39gAAAABJRU5ErkJggg==', 'sha256': '4e17b899007a3bc33cd7857e1707f4ce95c604c6910ea77e1d12eb70982900a3'}, 'data/binding-v1/images/binding-evaluation-0017-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQUlEQVR4nO3cQW7bRhiAUaao76NFgK54jR4iB8oheg2vCmTBw2RhwJuiMCo4qSLbUOKPnHlvbUtDQPzwz1j0h4eHrwtA4bfkXQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZH5fxnI6fVwmsG1f6iXAT2ACAjICBGRG24Kd3f3x1zKcx7//rJcAP5MJCMgIEJARICAjQEBGgICMAAEZAQIyAgRkhv0iIuzq8UOP710kQPAezzw//xUxOhMgeO//tXB+nW36EgkQ3Jqe+8+P1399/XR35WW3iTMkQPDm9LxYnOs//12PThNnSIDgtel5a3d+5Pw667MSzZkhAYL3S8/Fl10nzpAAwbX6/KL0vJihbY4GCRBcrs87pOdKhiZpkG9Cw79Op49hfS6+73dLGpIAQbDtuuL+23cfu0G2YMxuD4PPtNsxExBT22F9frQdW0YkQMxrz/WZpEECBDutz/7XdjsBYlLngWL/d/j9fyscbwgSIGZ03Dv5dNiVXyRATGf/Rz/zHAYJEPM6Sn2OuNpXEiDmcqCjnxkOgwQIyAgQEzn0+DPkECRAQEaAmMUA4894Q5AAARkBYgrDjD+DDUECBGQECMgIEOMbbP810i5MgICMAAEZAQIyAsTgDn1EMvw1ChCzGOkEepgrEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAmMX66W4Zy3r8KxIgBrdtX5bRbYe9RgECMgIEZAQIyAgQyzxHJAOc2p6dr+W4B0ACBJQECMgIEFMYbBe2DrH/EiCgJEDMYpghaB1l/BEgoCRATGSAIWgdaPwRIKAkQMzl0EPQOtb4I0BM7VgNWg+12lcSIKbzfHw4yl29PlvnMOOPADGp497D22FXftGHh4evy0BOp4/LBAb7FOaflvvPj8uOrcMd/ZyZgGDXG7F1x2u7nQAxr/0fBq2DHv0MuwWDW7btu9qLraPXR4DgwtFhnqH123Fs1PrYgsGFO7zdjq3T1McEBPvajq0TbLueEyDYxXZsnWnwORMgePnbZL80Q+v/dnyT1EeAoMzQOnF6nggQvPm79TeW6OIh9zZZep4IENz6iM+LPbr+Z7VtyvQ8ESDInjTcJk7PEwGC9y6R7pwJELxHjETnIgECMh7FADICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELBU/gHw1ElWwppy3wAAAABJRU5ErkJggg==', 'sha256': '90177097b6c6bda617625d40e17d16def65d23eae71d964adf1aa93c60ef1d78'}, 'data/binding-v1/images/binding-evaluation-0017-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQUlEQVR4nO3cMW4bVxRA0XEQ7oeF2+xgVpIFZSXcQVoXXIwLAWpSCCEYhZIs0Oad+f+cWiaHgObivW+Ovjw9fV8ACr8l7wogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPP7Mpbj8esygfP5W30J8BOYgICMAAGZ0Vawi7//el6G88efh/oS4GcyAQEZAQIyAgRkBAjICBCQESAgI0BARoCAzLBfRIRNPX7o8b2bBAge8czz9T8RowsBgkf/rYXL65ynL5EAwb3pOR0+eEZvfX5+52XPE2dIgODT6fmwOO///KseHSfOkADBj6bns915y+V11qsSzZkhAYLHpefmy64TZ8j3gOC9+pwOh19Un3feYpK/LCxAsLx15z8gPddevd0kDbKCwe3BJ7mM0+Fw2chmWMdMQLCV+ryYah0TIGYXrl1vmWcdEyCm9qo+y5acJmiQADGvLddnkgYJEGy0Ptu/tvsJEJO6DBTbv8NP/17heEOQADGj/d7Jx91e+U0CxHS2f/Qzz2GQADGvvdRnj1f7gwSIuezo6GeGwyABAjICxER2Pf4MOQQJEJARIGYxwPgz3hAkQEBGgJjCMOPPYEOQAAEZAQIyAsT4Btu/RtrCBAjICBCQESAgI0AMbtdHJMN/RgFiFiOdQA/ziQQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIGaxPj8vY1n3/4kEiMGdz9+W0Z13+xkFCMgIEJARICAjQCzzHJEMcGp7cfks+z0AEiCgJEBARoCYwmBb2DrE/iVAQEmAmMUwQ9A6yvgjQEBJgJjIAEPQOtD4I0BASYCYy66HoHWs8UeAmNq+GrTu6mp/kAAxnevxYS939Xp1ncOMPwLEpPZ7D593e+U3fXl6+r4M5Hj8ukxgsN/C/LfldDgsG7YOd/RzYQKCTS9i64av7X4CxLy2fxi0Dnr0M+wKBves7ZvaxdbR6yNAcOPoMM/Q+t9xbNT6WMHgxh3ermPrNPUxAcG21rF1grXrmgDBJtaxdabB50KA4ONvk/3SDK3/2/gmqY8AQZmhdeL0vBAg+PR36+8s0c1D7vNk6XkhQHDvIz4f9uj9/1Y7T5meFwIE2ZOG54nT80KA4NEl0p0LAYJHxEh0bhIgIONRDCAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAFL5R+76jA6m0omsgAAAABJRU5ErkJggg==', 'sha256': '771ad2f10184411d417f31e6cbb175cf75c7851a73fec16d15a3aa3ae7dd1da4'}, 'data/binding-v1/images/binding-evaluation-0017-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQElEQVR4nO3cMW4bVxRA0XEQ7oeFgWxi6iwyNTcRwAUX40KAmhRCCEahJAu0eWf+P6e2yCGguXjvm6MvT0/fF4DCb8m7AggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjI/L6M5Xj8ukzgfP5WXwL8BCYgICNAQGa0Fezi8Mdfy3Ce//6zvgT4mUxAQEaAgIwAARkBAjICBGQECMgIEJARICAz7BcRYVOPH3p87yYBgkc883z9I2J0IUDw6L+1cHmd8/QlEiC4Nz2nw+H9H1+fn9952fPEGRIg+HR6PizO+//+VY+OE2dIgOBH0/PZ7rzl8jrrVYnmzJAAwePSc/Nl14kz5HtA8F59TofDL6rPO28xyV8WFiBY3rrzH5Cea6/ebpIGWcHg9uCTXMbpcLhsZDOsYyYg2Ep9Xky1jgkQswvXrrfMs44JEFN7VZ9lS04TNEiAmNeW6zNJgwQINlqf7V/b/QSISV0Giu3f4ad/r3C8IUiAmNF+7+Tjbq/8JgFiOts/+pnnMEiAmNde6rPHq/1BAsRcdnT0M8NhkAABGQFiIrsef4YcggQIyAgQsxhg/BlvCBIgICNATGGY8WewIUiAgIwAARkBYnyD7V8jbWECBGQECMgIEJARIAa36yOS4T+jADGLkU6gh/lEAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQs1ifn5exrPv/RALE4M7nb8vozrv9jAIEZAQIyAgQkBEglnmOSAY4tb24fJb9HgAJEFASICAjQExhsC1sHWL/EiCgJEDMYpghaB1l/BEgoCRATGSAIWgdaPwRIKAkQMxl10PQOtb4I0BMbV8NWnd1tT9IgJjO9fiwl7t6vbrOYcYfAWJS+72Hz7u98pu+PD19XwZyPH5dJjDYb2H+23I6HJYNW4c7+rkwAcGmF7F1w9d2PwFiXts/DFoHPfoZdgWDe9b2Te1i6+j1ESC4cXSYZ2j97zg2an2sYHDjDm/XsXWa+piAYFvr2DrB2nVNgGAT69g60+BzIUDw8bfJfmmG1v9tfJPUR4CgzNA6cXpeCBB8+rv1d5bo5iH3ebL0vBAguPcRnw979P5/q52nTM8LAYLsScPzxOl5IUDw6BLpzoUAwSNiJDo3CRCQ8SgGkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgKXyD4tVMDrxgRx+AAAAAElFTkSuQmCC', 'sha256': '54673656f62b0389ffd13ae43e49b7ce07ac50d72f0a51e1f7aba99480ffa214'}, 'data/binding-v1/images/binding-evaluation-0018-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFV0lEQVR4nO3aTUoraRSAYW2uy2lwILiTu5FeR2/EnVxwIPRWHAhOeiBIMBpjYvLWKZ9nJP5Unaj18tVXuXx6erwAKPyVnBVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMr+7UELu+vtn9DQ8P9+ea5Ye6fHp6rGeApRRnNz36dgLEj3BkejbJ0DcSoJP8j7ZcIXv+Ta9u73b/+POf3zu+6vd8PHtA/Kz0fBqdj755O0Yvp5ChYwgQP6I+X+rOu16P8KZE19c3GnQwAXrH8z9/Xwx09e9/9QjrTM+7B9zMkKXQwbwPiNXW5+r27tvrs+Pgq9lDPCcBYiU2r/+TpmfTmxNp0FcJECusz5nPrkEHEyDGa+uzfV4N2p8AMdvr1X62266PbA6gQXsSIFZSn4tl0KAvESCmWmB9XmjQ/gSIkaZc21PmrAgQ8yxh13k3e9J7EiAGW2Z9lj/bcggQwyx262ebzaBPCRCTzL2S505+UgLESMtf/syasyJAjDF9ETF9/lMQIOaZtayYNe2ZCRCQESBmGPTwa5vHYR8RICAjQAywpoXDml7L8QSISSbef02f/KQECMgIEJARICAjQCzd6AfwmzyM3yZAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMALF0Dw/3Lx88//l9Mdnr/K+vCAECMgIEZAQIyAgQk8zdBpo7+UkJEAOsadd2Ta/leAIEZASIGUY/jPcA/iMCBGQEiHlmLYJmTXtmAsQY0+9fps9/CgLESFOWFVPmrAgQk8xdRMyd/KQEiGEGPQ7z8OtTAsRgS27QkmdbDgFins0FxTKv882pLH92ECBGmnJVT5mzIkBMtdjNIFs/+xMgBltgg9TnSwSI9TSozdDmAOqzJwFivCXsSdt1PowAsQZtg9TnYALEOht0ngy9OZH6fJUAsR5vrv+TZmj74OpzgF+H/BAs1UsFrq9vXj/zkomr27vvOsV21KTnYALECj083G82aLMaB5foo8WU+hzj8unp8agDrMibf9m5XBJ7/k0/jdHuOzi/5+NZAbFmr414t0SH7RDpzjcSIH7o3tDBB+EbuQXj5/q0R4pzagIEZLwPCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAi8r/b+c1wbYdCEQAAAAASUVORK5CYII=', 'sha256': 'cca094a7d678237348bfdcec70f66d28046e786a5911694cf810c2424fef96da'}, 'data/binding-v1/images/binding-evaluation-0018-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFS0lEQVR4nO3aO04jWxRAUXhqz4egh9HDZRgEHgwBEskLkJCFwRj/dp1irQjxqToG1+b6lu9fXp7vAAr/JWcFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg86c7NcQeHv4e/obt9ulWs/xS9y8vz/UMsJTiHKZHFydA/ApnpmeXDF2QAF3lOdpyhRz5N33cbA7/+L/X1wNf9Xs+nz0gfld6vo3OV9+8H6O3U8jQOQSIX1GfH3XnU+9H+FCih4e/GnQyAfrE+U/WxOHXC7/ENdLz6QF3f9uWQifzPiBWW5/HzeZ6/0v2D76aPcRbEiBWYvf6v2p6dn04kQb9lACxwvrc+OwadDIBYry2Pvvn1aDjCRCzvV/tN3vZ9ZXdATToSALESupztwwa9CMCxFQLrM8bDTqeADHSlGt7ypwVAWKeJew6H2ZP+kgCxGDLrM/yZ1sOAWKYxW797LMZ9C0BYpK5V/Lcya9KgBhp+cufWXNWBIgxpi8ips9/DQLEPLOWFbOmvTEBAjICxAyDbn7tczvsKwIEZASIAda0cFjTYzmfADHJxNdf0ye/KgECMgIEZAQIyAgQSzf6BvwuN+P3CRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNALN12+/T2wb/X17vJ3ud/f0QIEJARICAjQEBGgJhk7jbQ3MmvSoAYYE27tmt6LOcTICAjQMww+ma8G/BfESAgI0DMM2sRNGvaGxMgxpj++mX6/NcgQIw0ZVkxZc6KADHJ3EXE3MmvSoAYZtDtMDe/viVADLbkBi15tuUQIObZXVAs8zrfncry5wABYqQpV/WUOSsCxFSL3Qyy9XM8AWKwBTZIfX5EgFhPg9oM7Q6gPkcSIMZbwp60XefTCBBr0DZIfU4mQKyzQbfJ0IcTqc9PCRDr8eH6v2qG9g+uPif4c8oPwVK9VeDh4e/7Z94y8bjZXOoU+1GTnpMJECu03T7tNmi3GieX6KvFlPqcQ4A+sZw3lXDBpdD+H/fbGB1+JkjP+QSINXtvxH6JTv5PozsXJED86gXRCQfhgu5fXp4veTyY49seKc61CRCQ8T4gICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQLuKv8Dvb020OT5tnUAAAAASUVORK5CYII=', 'sha256': '79a65d63c90ef69bf3bab79e7b3ec7e999b344751eb9ef817c569fa55f5afb45'}, 'data/binding-v1/images/binding-evaluation-0018-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVElEQVR4nO3aS07rWBRAUSi9DKckGgzjzabGUbNhGK+BVFOhgUSnGkgoIhBCftvHrNVCfOwTwFvX17l9fn66ASj8lZwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzK/u1BC7u7vf/w2Pj3+uNcsPdfv8/FTPAEspzn56dHYCxI9wYnq2ydAZCdBF/kdbrpAD/6YPm83+H//98rLnq37Pp7MHxM9Kz5fR+eybd2P0egoZOoUA8SPq863ufOjtCO9KdHd3r0FHE6APvPzz981Am3//q0dYZ3o+POB2hiyFjuZ9QKy2Pg+bzdnrs+fgq9lDvCYBYiW2r/+LpmfbuxNp0HcJECusz5XPrkFHEyDGa+uze14NOpwAMdvb1X61267PbA+gQQcSIFZSn5tl0KBvESCmWmB9XmnQ4QSIkaZc21PmrAgQ8yxh13k/e9IHEiAGW2Z9lj/bcggQwyx262eXzaAvCRCTzL2S505+UQLESMtf/syasyJAjDF9ETF9/ksQIOaZtayYNe2VCRCQESBmGPTwa5fHYZ8RICAjQAywpoXDml7L6QSISSbef02f/KIECMgIEJARICAjQCzd6Afw2zyM3yVAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMALF0j49/Xj/4/fJyM9nb/G+vCAECMgIEZAQIyAgQk8zdBpo7+UUJEAOsadd2Ta/ldAIEZASIGUY/jPcA/jMCBGQEiHlmLYJmTXtlAsQY0+9fps9/CQLESFOWFVPmrAgQk8xdRMyd/KIEiGEGPQ7z8OtLAsRgS27QkmdbDgFinu0FxTKv8+2pLH/2ECBGmnJVT5mzIkBMtdjNIFs/hxMgBltgg9TnWwSI9TSozdD2AOpzIAFivCXsSdt1Po4AsQZtg9TnaALEOht0nQy9O5H6fJcAsR7vrv+LZmj34OpzhF/H/BAs1WsF7u7u3z7zmomHzeZcp9iNmvQcTYBYocfHP9sN2q7G0SX6bDGlPqe4fX5+OukAK/LuX3Yul8SBf9MvY7T/Ds7v+XRWQKzZWyM+LNFxO0S6c0YCxA/dGzr6IJyRWzB+ri97pDiXJkBAxvuAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAi4qfwPHYM1wS5KrcwAAAAASUVORK5CYII=', 'sha256': '595610d2332d048aceb29859daf9b2f76b00665ddbabac70aac457a3ae935c7a'}, 'data/binding-v1/images/binding-evaluation-0019-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFbklEQVR4nO3aPW4TWxiA4XDFLAfJBS0bomUdtHdDt6UYia1QREpDYV0r5McGJ+GdM+d5KiSC/Vl4Xn1zJu9ub3/cABT+Sd4VQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5n331jCww+Hj+R9Y129/a5aBvbu9/VHPAHsoznl69CQBgrdNz30ytMMAveL3Y2t8Xzf7vbr78uH8P1++fj/zt/5nj5wBwe+m52J0nvvhxzE6vsU6fYYECC7U54+686TTKzwo0eHwcfIG7SpA//17d7MLnz4v9Qgzeov0PPmC9zN0mHsV2lWA4LXq8+rpuZihdcoGCRD8Up83Tc+ZDB2mbJDfhGZ2SX2efMfDfp/nPkeAmFpbn6OZG+QWjHmdrvYqPU/ejk11L2YDYlLbqc/JaZJ59iABYkYbrM+cDRIgpjPKtX0YZM6XECDmsoVT5/PuZjqTFiAmtc36bH+21yVATGSzRz/THgYJELMY90o+DDv5RQLEdLa//ow150sIEFMYfYk4DD7/cwSIuYy1VtwNNe0VBAjICBD7N9DDr9kehwkQkBEgdm5Pi8NhR5/lSICYxYj3X6NPfpEAARkBAjICBGQEiD0b+gH8DA/jBQjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEg9mxdvx3/sHz9fjOy5f/5T59oHwQIyAgQkBEgICNAzGLcY6Bl2MkvEiB2bk+ntuuOPsuRAAEZAWL/hn4Yv+z0AfyRAAEZAWIuYy1By1DTXkGAmMLo9y/r4PM/R4CYzihrxTLInC8hQMxi3CViHXbyiwSIiQz0OGzZ9cOvEwFiUltu0LLh2V6XADGX+wvFNq/z5d5U+15/BIgZjXJVr4PM+RICxIw2exi0zHH0cyJATGqDDVomq8/Nzc37egDIrOu3w+Hj6cq/+/KhmmSZ6dznPhsQU9vCmfQya30ECOIGLRPXxy0Y/HIv9jdvx5ZfYzdhfQQInmjQW2fo8Z61TlkfAYKHFXjTDEnPAwIE51ah+9W4ukTPHS2tc9dHgOC3VqHHHbkYo/Pn2dJzJEBwoRGPS3T18zLdeUCA4MqF6IoX4QEBgmsKcrFHivM7BAiuoS+vwm9CA5ldbUCfPi/1CMAfsAEBGQECMu9ub3907w5MzQYEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBNxUfgLiQzZkgqi+6AAAAABJRU5ErkJggg==', 'sha256': '6fbdffce7d00986fee582eaecea27edffaadf8cc1f8b4d45bb4b5258e88f7b63'}, 'data/binding-v1/images/binding-evaluation-0019-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFfElEQVR4nO3aO24bSRRAUXpg7YeBgdlEx17kxNzEAA64GAcClExAjEDrQ9rU53Z1nRMZ+nQ/wuyLYpW+3N//3AEU/kruCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPO1uzUMbL//dvkHjscfnzXLwL7c3/+sZ4AtFOcyPXqRAMHHpuecDG0wQO/4/lgb79fVvq8Od3eXf315eLjwXf+zJ/aA4HfTczU6r/3w8xidbnGcPkMCBFfq80fdedHjFZ6UaL//NnmDNhWgu7//2W3Cw7/f6xFm9BHpefGC5xnaz70U8ndA8EJ9Dnd3716fCxffb3cf8zIBgl+e/w9Nz7nDrzeas0ECxOye1OeT736Yu0ECxNTa+pzM3CABYl6PT/unfex6zeFsgKkaJEBM6rw+u3U4zNcgAWJGK6zPnA0SIKYzyrO9H2TOtxAg5rKGXefLDjPtSQsQk1pnfdY/2/sSICay2q2faTeDBIhZjPsk74ed/CoBYjrrX/6MNedbCBBTGH0RsR98/tcIEHMZa1lxGGraGwgQkBEgtm+gw6/ZjsMECMgIEBu3pYXDfkOv5USAmMWIn79Gn/wqAQIyAgRkBAjICBBbNvQB/AyH8QIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEFt2PP44/WN5eNiNbPl//sdXtA0CBGQECMgIEJARIGYx7jbQMuzkVwkQG7elXdvjhl7LiQABGQFi+4Y+jF82egB/IkBARoCYy1iLoGWoaW8gQExh9M8vx8Hnf40AMZ1RlhXLIHO+hQAxi3EXEcdhJ79KgJjIQMdhy6YPvx4JEJNac4OWFc/2vgSIuZwvKNb5nC9nU217+SNAzGiUp/o4yJxvIUDMaLWbQcscWz+PBIhJrbBBy2T1ESCmdt6gNkPL2QDz1EeAmN0a9qSXmXadnxAgZtc2aJm4PgIELzToczK0/HqjCesjQPDy8/+hGXp+8eOU9dntdl/rAWAtThXY7789fuWUicPd3Xvd4nnUjrOm50SA4BfH44/zBp1X4+YSvbaYOs5dHwGC31oKPe/I1Rhd/gQnPScCBFca8bxEN5+X6c4TAgQ3LohuuAhPCBDcUpCrPVKc3yFAcAt9eRf+DgjIbGoF9PDv93oE4A9YAQEZAQIyX+7vf3Z3B6ZmBQRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE7Cr/ATk8OKMh06G9AAAAAElFTkSuQmCC', 'sha256': 'ced7b213a17400476f15b765c634d538da9ace42787a39e8898016fe9552e379'}, 'data/binding-v1/images/binding-evaluation-0019-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFfklEQVR4nO3avW0bSRiAYflg9cPAqTvYSlzQVcIOLnXAYhwIUHIBcQL1R/ooWu/OzvNEhiXvfoS5L4Yz/PLw8OsOoPBXclcAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZr92tYWC73bfzv3A4/PysWQb25eHhVz0DbKE45+nRmwQI/mx6TsnQBgN0w/fH2ni/rvZ9tb+/P//Pl8fHMz/1P3tkDwh+Nz0Xo/PeL7+O0fEWh+kzJEBwoT7/qztverrCixLtdt8mb9CmAvTP3+cWvQP5/uOj73hWkp43L3iaod3cSyHfA4I36rO/v795fc5cfLfdfczzBAiePf9/ND2n9s9vNGeDBIjZvajPJ999P3eDBIiptfU5mrlBAsS8np72T/vY9Z79yQBTNUiAmNRpfe7WYT9fgwSIGa2wPnM2SICYzijP9m6QOT9CgJjLGnadz9vPtCctQExqnfVZ/2y3JUBMZLVbP9NuBgkQsxj3Sd4NO/lFAsR01r/8GWvOjxAgpjD6ImI3+PzvESDmMtayYj/UtFcQICAjQGzfQIdfsx2HCRCQESA2bksLh92GXsuRADGLET9/jT75RQIEZAQIyAgQkBEgtmzoA/gZDuMFCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESC27HD4efzD8vh4N7Llv/mfXtE2CBCQESAgI0BARoCYxbjbQMuwk18kQGzclnZtDxt6LUcCBGQEiO0b+jB+2egB/JEAARkBYi5jLYKWoaa9ggAxhdE/vxwGn/89AsR0RllWLIPM+RECxCzGXUQchp38IgFiIgMdhy2bPvx6IkBMas0NWlY8220JEHM5XVCs8zlfTqba9vJHgJjRKE/1YZA5P0KAmNFqN4OWObZ+nggQk1phg5bJ6iNATO20QW2GlpMB5qmPADG7NexJLzPtOr8gQMyubdAycX0ECN5o0OdkaHl+ownrI0Dw9vP/RzP0+uKHKetzd3f3tR4A1uJYgd3u29PfHDOxv7+/1S1eR+0wa3qOBAieORx+njbotBpXl+i9xdRh7voIEPzWUuh1Ry7G6PwnOOk5EiC40IjXJbr6vEx3XhAguHJBdMVFeEGA4JqCXOyR4vwOAYJr6MtN+B4QkNnUCuj7j5t9XwP4BFZAQEaAgMyXh4df3d2BqVkBARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAF3lX8BCjw3NTkPc68AAAAASUVORK5CYII=', 'sha256': '893f0c5a1f595f3c8d16d571513dab832b7a13dd3ad916574b6e159f8f269608'}, 'data/binding-v1/images/binding-evaluation-0020-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJklEQVR4nO3cMVIbWRRAUXnK7EeB096BV+IFeSWzA1IHvRgHVJFMoEIjCwSWQNxu/XMiAtR6iW6936j58vDwewNQ+Cd5VwABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5utm5bbbb5sBzPOvegT4eDYgICNAQGb1R7C9f+/uNjfn++NjPQJckQ0IyAgQkBEgICNAQEaAgIwAARkBAjK38z0gWOzzQJ6kOUWA4OoPIR7+shgdEiD41Ief91eYlUiA4P3puf/5xhMz04+7Uxecx86QAMEl6XkzOqd++ShG27EzJEBwXn3OSs9zu5c/z9A8ZIMECP4qPe/szpH91aanEo25CvkeEHx2fV658naM//C5ZwOCY4cVuF56Th3KhjqO2YCgrM+L7zXOHiRA0Ndn2AYJECyiPmM2SIBgKfUZsEECBAuqz2gNEiDYLKo+S5vkqgSI0e1XjKV95u+f5rnhJUiAGNpaPtvblcx5LgGCJa4/S57qAwkQ41rs4Wucg5gAARkBYlBrXCi2K5z5dQLE6JZ8/lrLhBcTICAjQIxoFbefR7gVLUBARoCAjAABGQFiOKu7AXTDt4EECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESCGM8+/dj9MP+426zE9Tbuf/wYIEJARICAjQEBGgBjR6m4DTbd4A0iAgJIAMbrlL0HT4ie8mAAxqDWeZeYVzvw6AQIyAsS4VnErerrR2887AgTLbdC0yKk+kAAxtLWsFfNK5jyXADG6xR7Epps+fO0IEPxvOQ2aFjPJVQkQ/LFiLOGTPx3McMPrjwDB4ho0DVMfAYJlNWgaqT4CBAtq0DRYfQQIltKgabz6bDabr/UAsDjz/Gu7/XbYhfufj9d7u+nPzI1THxsQvOyoAtdbhaaB62MDgpN2LThahT5qG3petHmw9OwIEPztcexDDmUvLlPzkPURIDh7FXrekTdj9MoJbh41PTsCBJdn6D13iOax03NrAfr+eMW/U8BRNV4s0VlX4KYCBJ/psCNvxkh0ThEgeC99udiXh4ffl78a4B18ERHICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELCp/AegsBJU4hxBawAAAABJRU5ErkJggg==', 'sha256': 'c0d1e056681ff06ac5b033c4344a9b2cc59346a51771afb39d913be109496f62'}, 'data/binding-v1/images/binding-evaluation-0020-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFLUlEQVR4nO3cPW4aaxSAYXIV9kMRyVW2kUVkQXcR2catIqVgMSksTZMCmYv/cADjd4bveSoXnuE0vDrfYPzp/v73CqDwT/KqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM59XCbTZfVgPYbn/VI8D7swEBGQECMos/gu399++0ujlfv6/rEeCKbEBARoCAjAABGQECMgIEZAQIyAgQkLmdvwOC2X4fyDdpXiNAcPUvIR7+shgdEiD40C8/7++wVSIBgsvTs777cfza6ee31264HTtDAgTnpOfN6Lz2y09itBk7Qz4Fg9Pqs777cVJ9/ubyzRj/1uo5GxC87EkULonOc/u7TQ8L0ZirkA0IPro+R+68GWwVEiB46rACFx64/sb68UsM1SABgmP1+bDXXQ/ZIAGCvj7DNkiAYBb1GbNBAgRzqc+ADRIgmFF9RmuQAMFqVvWZ2yRXJUCMbr9izO09v36Y54aXIAFiaEt5b28WMuepBAjmuP7Meap3JECMa7aHr3EOYgIEZASIQS1xodgscObjBIjRzfn8tZQJzyZAQEaAGNEiHj+P8ChagICMAAEZAQIyAsRwFvcA6IYfAwkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQAxnu/21+2H6+W21HNPDtPv5b4AAARkBAjICBGQEiBEt7jHQdIsPgAQIKAkQo5v/EjTNfsKzCRCDWuJZZrvAmY8TICAjQIxrEY+ipxt9/LwjQDDfBk2znOodCRBDW8pasV3InKcSIEY324PYdNOHrx0Bgv/Np0HTbCa5KgGCRyvGHN7508EMN7z+CBDMrkHTMPURIJhXg6aR6iNAMKMGTYPVR4BgLg2axquPAMHbDbp2hqbHLzFOfQQIXvakAtdr0PT4zkPVZ7Vafa4HgJnatWCz+fKkFOu7H5ff/HnRtoOlZ8cGBMc878KFh7IXL98OWR8bEJy8Cu0cRuTNnehIsLajpmdHgOD8DO2ctxBtx07PrQXo6/d1PQK3b1+NF0t00h24qQDBRzrsyJsxEp3XCBBcSl/O9un+/vf5VwNcwMfwQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAVeUPmlUSJPWIMP4AAAAASUVORK5CYII=', 'sha256': '61fdaaa28c689e7044b09275378260006dd623183b4cdcfee921af1996979f85'}, 'data/binding-v1/images/binding-evaluation-0020-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKElEQVR4nO3cPW7bWBSAUWUQ7UdFAG8idRaZejYRIIUWk8IAmykEa/wrR5Llj+Q7p3JhUrfRh/soy1/u7/9sAAr/JK8KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh83SzcbvdtM4D9/nc9Anw8GxCQESAgs/gj2NG/2+1mdb5PUz0C3JANCMgIEJARICAjQEBGgICMAAEZAQIy6/k7IJjt94F8k+YtAgQ3/xLi418Wo8cECD71y8/HO+yVSIDg+vRs736evnb69eOtG+7HzpAAwSXpeTc6b/3ysxjtxs6QT8HgvPps736eVZ+/uXw3xr+1eskGBK97FoVrovPS8W7Tw0I05ipkA4LPrs+JO+8GW4UECJ57XIErD1x/Y/v0JYZqkADBqfp82utuh2yQAEFfn2EbJEAwi/qM2SABgrnUZ8AGCRDMqD6jNUiAYDOr+sxtkpsSIEZ3XDHm9p7fPsyz4iVIgBjaUt7bu4XMeS4BgjmuP3Oe6gMJEOOa7eFrnIOYAAEZAWJQS1wodguc+TQBYnRzPn8tZcKLCRCQESBGtIjHzyM8ihYgICNAQEaAgIwAMZzFPQBa8WMgAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZASI4ez3vw8/TL9+bJZjepj2OP8KCBCQESAgI0BARoAY0eIeA01rfAAkQEBJgBjd/JegafYTXkyAGNQSzzL7Bc58mgABGQFiXIt4FD2t9PHzgQDBfBs0zXKqDyRADG0pa8V+IXOeS4AY3WwPYtOqD18HAgT/m0+DptlMclMCBE9WjDm886dHM6x4/REgmF2DpmHqI0AwrwZNI9VHgGBGDZoGq48AwVwaNI1XHwGC9xt06wxNT19inPoIELzuWQVu16Dp6Z2Hqs9ms/laDwAzdWjBbvftWSm2dz+vv/nLou0HS8+BDQhOedmFKw9lr16+H7I+NiA4exU6eByRd3eiE8Haj5qeAwGCyzN0cNlCtB87PWsL0Pdpqkdg/Y7VeLVEZ92BVQUIPtPjjrwbI9F5iwDBtfTlYl/u7/9cfjXAFXwMD2QECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQI2FT+A5v1E7p/gSfyAAAAAElFTkSuQmCC', 'sha256': '6ed74f13853f62a75bb3e2aeb43be50f17b8e7cab3047d0e5d42134b0a85da22'}, 'data/binding-v1/images/binding-evaluation-0021-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFgElEQVR4nO3dMW4bVxRAUTmw9sPCQDbB2otMrU0YSMHFuBCgJoUQgqFMipIZXv7/z2klax5hzNWb4Qj88vz88wGg8EdyVAABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dodGgaz2Xw7/w273d+3mmUSX56ff9YzwKjFOU+P3iVAcOXuvKVEpwgQXJqep8fH8/98+/Jy5qsy9JYAwbn0vBudj8ZIhg4JEKv7ZXo+3Z0LSyRDrwSIpR3V54rdebdEOw3yHBAru2V93v78zbVvdY/IBsSKbpyeI1ahPRsQy2nrYxU6ZANiLYdn++3Tc2YV2i25B9mAWMhd1efhvzOsuQcJEKu4t/q8WrxBAsQS7rM+r1Zu0Ip/DT/l//GadxA+4d7q8+rp8fH8n3HMygbEQr9y7rM+R7NN+QvyFAFiciOez5sBZ/6cFS/B9h7//OthcC8/vtcj3LV7vvVz/kJss/m2wmW1DYgl3H99xprzWgSIaY1+IbMZfP5LCBDzG2uteBpq2t8kQMxpiHe+TlnnHTEBAjICxIRmWhw2E72WtwSImY14/TX65B8iQEBGgJjNfNcsm+le0Z4AMa3Rr2KeBp//EgIEZAQIyAgQkBEgpjL0A9ALPhItQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAAxlf2H+c3xUevbf1/FrB9SKEBARoCAjAABGQFiWqPfBtoOPv8lBIjZzHe/djfdK9oTICAjQMxs3KuY7bCTf4gAMaGZrll2E72WtwQIyAgQcxr6kejt7A9A7wkQ8xurQduhpv1NAsS0Rl8fdoPPfwkBYgmjrBXbQea8FgFiZodLxP2f29uDCVdYfwSI+Y14Ju8GnPlzBIj5DfGO2HaZd74OCRBruc8Gbe9yqhsQIJZwzzeDtuvd+tkTIFZxnw3aLlwfAWIt99ag7dr1ESBWb1CVoaND75asjwCxoqOz/fYNOjribtX6PDw8fK0HgMDrOb/ZfDsswtPj4/99XOk5YgNiXTdehdTnLRsQS9vt/t7vQYeNuOI29Muuqc8rAWJ1R5dj1yrRqX1Keg4JEJzM0FFH3o3R+Ys46XlLgOD9DP3mTSLpOUWA4GQvTpXooz+HUwQILi3Iuz1SnI8SILiUvlyd54CAjAABGQECMgIEZAQIyAgQkBEgICNAQGbpBxFffnyvR4Cl2YCAjAABmS/Pzz+7owNLswEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECHio/APe3UqaSe+2PQAAAABJRU5ErkJggg==', 'sha256': 'f09ca5efa50838a651d396406a4a4e76f3cd8d7beea45cda729223e68845617d'}, 'data/binding-v1/images/binding-evaluation-0021-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFf0lEQVR4nO3dMW4bVxRAUTmw9sPCgDfhOot0nU0EcMHFuBAwTQohBEOZFCUzuvz/n9NK1jzCmKs3wxH46enp5wNA4Y/kqAACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg87k7NAxmt/ty+Rv2+x8fNcskPj09/axngFGLc5kevUqA4MbdeUmJzhEguDY9j1+/X/7n299/XviqDL0kQHApPa9G560xkqFjAsTqfpmed3fnyhLJ0DMBYmkn9blhd14t0V6DPAfEyj6yPi9//u7Wt7pHZANiRR+cnhNWoQMbEMtp62MVOmYDYi3HZ/vHp+fCKrRfcg+yAbGQu6rPw39nWHMPEiBWcW/1ebZ4gwSIJdxnfZ6t3KAV/xp+yv/jNe8gvMO91efZ49fvl/+MY1Y2IBb6lXOf9TmZbcpfkOcIEJMb8XzeDTjz+6x4CXbw1+Pjw+C+bVs9wl2751s/ly/EdrsvK1xW24BYwv3XZ6w5b0WAmNboFzK7wee/hgAxv7HWisehpv1NAsSchnjn65x13hETICAjQExopsVhN9FreUmAmNmI11+jT/4mAgRkBIjZzHfNspvuFR0IENMa/SrmcfD5ryFAQEaAgIwAARkBYipDPwC94CPRAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQUzl8mN8cH7W+/fsqZv2QQgECMgIEZAQIyAgQ0xr9NtA2+PzXECBmM9/92v10r+hAgICMADGzca9itmEnfxMBYkIzXbPsJ3otLwkQkBEg5jT0I9Hb7A9AHwgQ8xurQdtQ0/4mAWJao68P+8Hnv4YAsYRR1optkDlvRYCY2fEScf/n9nY04QrrjwAxvxHP5P2AM7+PADG/Id4R25Z55+uYALGW+2zQdpdTfQABYgn3fDNoW+/Wz4EAsYr7bNC2cH0EiLXcW4O2tesjQKzeoCpDJ4feL1kfAWJFJ2f7xzfo5Ij7Vevz8PDwuR4AAs/n/G735bgIj1+//9/HlZ4TNiDW9cGrkPq8ZANiafv9j8MedNyIG25Dv+ya+jwTIFZ3cjl2qxKd26ek55gAwdkMnXTk1RhdvoiTnpcECF7P0G/eJJKecwQIzvbiXIne+nM4R4Dg2oK82iPFeSsBgmvpy815DgjICBCQESAgI0BARoCAjAABGQECMgIEZJZ+EPHbttUjwNJsQEBGgIDMp6enn93RgaXZgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEPFT+Af9iSQqLcgMwAAAAAElFTkSuQmCC', 'sha256': '47a5b485d40bb8a75fcea138e2111e242ba629a7439555f6cfdb153e7a95aeeb'}, 'data/binding-v1/images/binding-evaluation-0021-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFd0lEQVR4nO3dMW4bRxiAUTmw7sPCgG+SQ/omAVLwMC4EsEkhhGBIk6JkRt/OzHutZO0sjP34z3Ilfnl5+fkEUPgjOSqAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyHztDg2D2e2+3f6G/f7vz1rLJL68vPys1wCjFuc2PXqTAMGDu3NJia4RILg3Pc/ff9z+54e//rzxVRm6JEBwKz1vRue9MZKhUwLE6n6Zng93584SydArAWJpZ/V5YHfeLNFegzwHxMo+sz6XP3/36FvdIzIBsaJPTs8Zo9CRCYjltPUxCp0yAbGW06v989NzYxTaLzkHmYBYyKbq8/TfNaw5BwkQq9hafV4t3iABYgnbrM+rlRu04m/DT/l/vOYdhA/YWn1ePX//cfvXOGZlAmKhl5xt1udsbVO+QF4jQExuxOt5N+CaP2bFLdjRll8P77Tm3D7HrZ/bG7Hd7tsK22oTEEvYfn3GWuejCBDTGn0jsxt8/fcQIOY31ljxPNRqf5MAMach3vm6Zp13xAQIyAgQE5ppcNhNdC6XBIiZjbj/Gn3l7yJAQEaAmM18e5bddGd0JEBMa/RdzPPg67+HAAEZAQIyAgRkBIipDP0A9IKPRAsQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEzl+GF+c3xk4+Hfs5j1QwoFCMgIEJARICAjQExr9NtAh8HXfw8BYjbz3a/dT3dGRwIEZASImY27izkMu/J3ESAmNNOeZT/RuVwSICAjQMxp6EeiD7M/AH0kQMxvrAYdhlrtbxIgpjX6+LAffP33ECCWMMpYcRhknY8iQMzsdIjY/rV9OFnhCuOPADG/Ea/k/YBr/hgBYn5DvCN2WOadr1MCxFq22aDDJlf1CQSIJWz5ZtBhvVs/RwLEKrbZoMPC9REg1rK1Bh3Wro8AsXqDqgydHXq/ZH0EiBWdXe2f36CzI+5Xrc/T09PXegEQeL3md7tvp0V4/v7j/z6u9JwxAbGuTx6F1OeSCYil7fd/H+eg00Y8cBr6ZdfU55UAsbqz7dijSnRtnpKeUwIEVzN01pE3Y3R7Eyc9lwQI3s7Qb94kkp5rBAiu9uJaid77c7hGgODegrzZI8V5LwGCe+nLw3kOCMgIEJARICAjQEBGgICMAAEZAQIyAgRkln4QcQt/FRhWZgICMgIEZL68vPzsjg4szQQEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIOCp8g9Vi0jiUF0oigAAAABJRU5ErkJggg==', 'sha256': 'd06fd48732c3a0a2a89e48f4b2d1d5e17395714f5a18b78cdef58f5628517a31'}, 'data/binding-v1/images/binding-evaluation-0022-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFP0lEQVR4nO3bQU7cShRAUSeKlxOpB+ws68jOGLSUrWSAxOQPUCxCoAn50NeuOmcM7rKEr15Vm093dz8XgMLn5FMBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh8WQZ1Ot0swzmfb+slwHsyAQEZAQIyw27BNvffvi4Ht37/US+BN+/u7Zf/xvgBguRI8fGviNFLBAg+/KuM7TpK9IQAwZvT8+q+/qVd88NlZWgjQPB6et56kvjk55/0SIY2AgTvmZ5nPVxEhv4kQPBMfT7iy9Ptmo9LdDrdzNwgAWJ2V0jP5YHoNPEo5EVEpnb9+rz0WacR/3noVSYg5vX4mU9eWL3/YxSabQ4yATGpvD7Pfvpsc5AAMaP91GfyBgkQ09lbfWZukAAxl33WZ9oGCRAT2XN95myQADGL/ddnwgYJENPZc32OssL3IkBMYRsljvJs3/9a59hDkAAxvqM/w6eDr/8CAWIiRxl/jrjafyNADO5wm6+pNmICBGQEiJEdevyZYQgSICAjQAxrvJHhNNwdCRDjO+7+a4z1XyBAQEaAGNMAx88zHEULEJARICAjQEBGgBjQYAclA9+dADGyMU6gx7uXjQABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICxMjW7z+WUawD3ctGgBjQ+Xy7jOs80N0JEJARICAjQEBGgBj8oGSMs9v1112MdAAkQEBJgBjf0Yeg9eDrv0CAGNZgu5VlxDsSICAjQIxsgKPoddDj5wcCBGQEiMEdeghahx5/BIi5HKtB66FW+28EiPEdfXw4H3z9FwgQUzjcRmwdffP1QICYzv4btO5+he9FgJjF41Fiz0/4+mhtY48/AsRc9t+gdab6CBDT2XOD1snqI0DMaJ8NWuerjwAxqb01aJ2yPsuyfLq7+7mM6HS6WYYz1Z/mlf9I7r99rZaxzlqfZVm+1AuAzPl8uzXooQJXztD6+/A1W31swZjdk2f+mtuxdfr6jLwFg/+zZ//QUUh6NgIEl84N37FEz45X54nrI0BwjQxJz0sECN78LeqrPbp8liQ9GwGC673MIT1PCBB8eIl05yUCBB8SI9H5GwIEZLyICGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCApfIfdmA42sphhnAAAAAASUVORK5CYII=', 'sha256': 'cb2901a8d184e6042965ba749d40bb313b26ff9187160d552fb2805f7bb39b11'}, 'data/binding-v1/images/binding-evaluation-0022-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPElEQVR4nO3bQU4bSRiA0c5ofB8vInGTHJKbRMrCh8kCqTezQLHAAyYk4K+76r01dFdL9Ke/yubLw8PPBaDwT3JXAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIy/y6DOh6/LsM5nX7US4CPZAICMgIEZIbdgp0d7u6XnVu/f6uXwLt39/bLv2P8AEFypPj0V8ToNQIEn/5Rxvk6SnRBgODd6XlzX//arvnxsjJ0JkDwdnree5J48fMXPZKhMwGCj0zPix4vIkP/J0DwQn0+48PT8zWfluh4/DpzgwSI2d0gPdcHouPEo5AvIjK129fntXsdR/znoTcJEPN6+s4f7u5v/53Vw/ObTtggAWJSF/UJV3KYuEECxIy2U5/JGyRATGdr9Zm5QQLEXLZZn2kbJEBMZMv1mbNBAsQstl+fCRskQExny/XZywo/igAxhfMosZd3+/BrnWMPQQLE+Pb+Dh93vv4rBIiJ7GX82eNq/4wAMbjdbb6m2ogJEJARIEa26/FnhiFIgICMADGs8UaG43BPJECMb7/7rzHWf4UAARkBYkwDHD/PcBQtQEBGgICMAAEZAWJAgx2UDPx0AsTIxjiBHu9ZzgQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIEa2fv+2jGId6FnOBIgBnU4/lnGdBno6AQIyAgRkBAjICBCDH5SMcXa7/nqKkQ6ABAgoCRDj2/sQtO58/VcIEMMabLeyjPhEAgRkBIiRDXAUvQ56/PxIgICMADG4XQ9B69DjjwAxl301aN3Vav+MADG+vY8Pp52v/woBYgq724ito2++HgkQ09l+g9bNr/CjCBCzeDpKbPkNX5+sbezxR4CYy/YbtM5UHwFiOltu0DpZfQSIGW2zQet89REgJrW1Bq1T1mdZli8PDz+XER2PX5fhTPWneeM/ksPdfbWMddb6mICY2sUcdPtRaH1+09nqI0DM7uKdv2WD1uf3mrA+I2/B4G/27J+6I5OeMwGCa+eGH1iiF8er08T1ESC4RYak5zUCBO/+FPXNHl0/S5KeMwGC232ZQ3ouCBB8eol05zUCBJ8SI9H5HQIEZHwREcgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABS+U/FtA0mNCminUAAAAASUVORK5CYII=', 'sha256': 'cac3cb341c4aa2b44040c3b76e01a0b0b5c40ed3bd390c3399f0542ac0860195'}, 'data/binding-v1/images/binding-evaluation-0022-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRUlEQVR4nO3bQWobSRiA0c4wPs6AFgbfJBeZc8xFfJNAFoJcJQtDb2ZhIhyNLccZW1931XtrS10N6o+/SvKnh4fvC0Dhj+SqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRk/lwGdTjcLsM5Hr/WS4D3ZAICMgIEZIbdgp2sf/+17NzNP9/qJfDm3b398q8YP0CQHCk+fYkYvUSA4MO/yji9jxKdESB4c3pu7u4vv3z98vnC28rQiQDB6+l5tTiX//6sRzJ0IkDwnul51uObyNB/CRA8U5936c6Z03s+LdHhcDtzgwSI2V0hPZcHosPEo5AfIjK169fnpWsdRvznoVcJEPN6+szf3N1fsz7PXvQwX4MEiEmd1Sdcyc3EDRIgZrSd+kzeIAFiOlurz8wNEiDmss36TNsgAWIiW67PnA0SIGax/fpM2CABYjpbrs9eVvheBIgpnEaJvTzbNz/WOfYQJECMb+/P8GHn679AgJjIXsafPa729wgQg9vd5muqjZgAARkBYmS7Hn9mGIIECMgIEMMab2Q4DHdHAsT49rv/GmP9FwgQkBEgxjTA8fMMR9ECBGQECMgIEJARIAY02EHJwHcnQIxsjBPo8e7lRICAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBYmTrl8/LKNaB7uVEgBjQ8fh1GddxoLsTICAjQEBGgICMADH4QckYZ7frj7sY6QBIgICSADG+vQ9B687Xf4EAMazBdivLiHckQEBGgBjZAEfR66DHz48ECMgIEIPb9RC0Dj3+CBBz2VeD1l2t9vcIEOPb+/hw3Pn6LxAgprC7jdg6+ubrkQAxne03aN38Ct+LADGLp6PElp/w9cnaxh5/BIi5bL9B60z1ESCms+UGrZPVR4CY0TYbtM5XHwFiUltr0DplfZZl+fTw8H0Z0eFwuwxnqo/mlT8kN3f31TLWWetjAmJqZ3PQ9Ueh9eeLzlYfAWJ2Z8/8NRu0/nytCesz8hYM/s+e/UN3ZNJzIkBw6dzwHUv07Hh1nLg+AgTXyJD0vESA4M3for7ao8tnSdJzIkBwvR9zSM8ZAYIPL5HuvESA4ENiJDq/QoCAjB8iAhkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESBgqfwLoK80mGH2kb4AAAAASUVORK5CYII=', 'sha256': '113d61f9f7c20950c6f074707765e1dc03e894bb66d2379b199311642cb58429'}, 'data/binding-v1/images/binding-evaluation-0023-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKElEQVR4nO3dMW7bSBiAUWWR3EdFWl4oB8qF3LrgYVIYSLPAChG0tmxFlq2P5LxXuTDECRJ++YeWxl+enn7tAAr/JFcFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh83Y1qv/++24R5fqyXAO9kAgIyAgRkxt2CHT38/L1boenHt3oJcCsTEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg46MY8OknKzix4DUCBJ9+nMvpN4vRKQGCux4jdXyFWYkECG7vzsUDFc4eXbBXIgGC99XnqlNcTr/5ZYz2++/DNkiA4Ir03H561PEVppMSHS40YIYECP6qPh9+cN3Dfy/4LEOjNUiA4N7peSND+8FGIW9EhKw+r11lM7+y5SITEJy/8+98WPjDi1FohDnIBAR9fc5ed4Q5SIBgKfUZsEECBAuqz2gNEiBGt7T6DNUgAYLF1WeZ6/kMAsTQjsPFMu/2hz+r2uoQJECMa1139X5Vq/1LAgQLHX+Wv7bbCRCDWvjma5CNmAAxovXeyfvVrvwsAWJoyx9/1rXOawkQkBEghrOipz+bfxIkQEBGgICMADGWle6/troLEyAgI0BARoCAjAAxkFU/ANrkYyABAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBIiBzPPj4Yvpx7fdOk1/Vn78s6yaAAEZAQIyAgRkBIixrPox0LStB0ACBJQECMgIEMNZ6S5s2tz+S4CAkgAxtLUMQdNK1nmtr/UCelv9q+UN8/y40kPd5w3tv0xAjGtFT4KmLT79ORAgWHSDpgWv7XbjbsG2958J296IzVv8F2sCYmgL34hN2918HQgQ7JbZoGlh6/kMAsToToeL5dzz08lKtjr+CBAssUHTGPURIFhcg6Zh6iNAsKwGTSPVZ7fbfXl6+lWvARbk2Q/mH37+vs91p/8nb4T6mIDguWd3/n1GoWnI+piAIB6FplHTcyBA8KqX75P+wAxNL2ar0eojQHDZ2Y9rvLtEZ/d083jpORAguOztj4xdjNHbD5LmUesjQHCdD/zw6jxwd44ECO5aIt05JUDw6TESndcIEJDxRkQgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgF3lX9jjI4ziG0LiAAAAAElFTkSuQmCC', 'sha256': 'a6fa8cee3f071a0e7f2938e3cf17bc24d664004a237195be578e6da2f7d0f266'}, 'data/binding-v1/images/binding-evaluation-0023-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFOElEQVR4nO3cS2rcWBiA0UqTWk5DDQLeSTbS68hGvJNABoJsJQODJoEuUlT8fpU/SfecUQaOdA3Rl/9eu/Tp5ubXDqDwT3JXAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM592oDocvu02Yph/1EuCVTEBARoCAzLhbsJP5v393K7T/9rNeAryVCQjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPFRDLj4mxW8seAhAgQXf53L+ReL0TkBgg99jdTpCpMSCRC8vTv7q+vH//r8/esj15wGLpEAwWvq82R0HvriuzE6HL4M2yABghek50XdudfpCvNZiY43GjBDfgwPz6rP/ur67fV5/IKHrbyn/PlMQPB0ei53r/3/Fz9NQ6ONQiYgyOpzMuwoJEBw/5P/7nuux+3/vt0gDRIguL8+yRr2gzVIgGAp9RmwQQIEC6rPaA0SIEa3tPoM1SABgsXVZ5nruQQBYmin4WKZT/v+z6q2OgQJEONa11N9WNVqn0mAYKHjz/LX9nYCxKAWvvkaZCMmQIxovU/yYbUrv5cAMbTljz/rWudLCRCQESCGs6LTn82fBAkQkBEgICNAjGWl+6+t7sIECMgIEJARICAjQAxk1QdAmzwGEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAGMg0/Tj+Yf7+dbdO85+Vn76XVRMgICNAQEaAgIwAMZZVHwPN2zoAEiCgJEBARoAYzkp3YfPm9l8CBJQEiKGtZQiaV7LOl/pcL6C3//azXgIfbZp+rPSl7tOG9l8mIMa1opOgeYunP0cCBItu0Lzgtb3duFuw7f1nwrY3YtMW/8WagBjawjdi83Y3X0cCBLtlNmhe2HouQYAY3flwsZxnfj5byVbHHwGCJTZoHqM+AgSLa9A8TH0ECJbVoHmk+ux2u083N7/qNcCC3PrB/P7q+mPuO/+dvBHqYwKC2249+R8zCs1D1scEBPEoNI+aniMBggfd/T3pd8zQfGe2Gq0+AgRPu/fjGq8u0b17umm89BwJEDzt8Y+MPRmjxw+SplHrI0DwMu/44dVp4O6cCBB8aIl055wAwcVjJDoPESAg4xcRgYwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQJ2ld8XmBensWV7+wAAAABJRU5ErkJggg==', 'sha256': '8c375049939cf49771c1329654a0ab332bf503aa63310a609f9751438b1a2be4'}, 'data/binding-v1/images/binding-evaluation-0023-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFOUlEQVR4nO3cPW7bWBhAUWUQ7UdFAFfZRhaRBc0ivI2pArjgYlIYYBNghAiKY8v/viTfOVUKh3wGwpvvPVv8dHv7cwdQ+Ce5K4AAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5vNuVIfDl90mTNNNvQR4IRMQkBEgIDPuFuzkv3/n3Qp9/b6vlwCvZQICMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGR8FAPe/c0K3ljwEAGCd3+dy/kXi9E5AYIPfY3U6QqTEgkQvL47+6vry399/vHtwjWngUskQPCS+jwanYe++O8YHQ5fhm2QAMEz0vOs7tzrdIX5rETHGw2YIT+GhyfVZ391/fr6XL7gYSvvKX86ExA8np73u9f+/4ufpqHRRiETEGT1ORl2FBIguP/Jf/M912X7P283SIMECO6vT7KG/WANEiBYSn0GbJAAwYLqM1qDBIjRLa0+QzVIgGBx9Vnmet6DADG003CxzKd9/3tVWx2CBIhxreupPqxqtU8kQLDQ8Wf5a3s9AWJQC998DbIREyBGtN4n+bDald9LgBja8sefda3zuQQIyAgQw1nR6c/mT4IECMgIEJARIMay0v3XVndhAgRkBAjICBCQESAGsuoDoE0eAwkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQAxkmm6Of5h/fNut0/x75afvZdUECMgIEJARICAjQIxl1cdA87YOgAQIKAkQkBEghrPSXdi8uf2XAAElAWJoaxmC5pWs87k+1wvoff2+r5fAR5umm5W+1H3a0P7LBMS4VnQSNG/x9OdIgGDRDZoXvLbXG3cLtr3/TNj2Rmza4r9YExBDW/hGbN7u5utIgGC3zAbNC1vPexAgRnc+XCznmZ/PVrLV8UeAYIkNmseojwDB4ho0D1MfAYJlNWgeqT673e7T7e3Peg2wIHd+ML+/uv6Y+85/Jm+E+piA4K47T/7HjELzkPUxAUE8Cs2jpudIgOBBf/+e9BtmaP5rthqtPgIEj7v34xovLtG9e7ppvPQcCRA87vJHxh6N0eWDpGnU+ggQPM8bfnh1Grg7JwIEH1oi3TknQPDuMRKdhwgQkPGLiEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABu8ovB0AW7uvHX6sAAAAASUVORK5CYII=', 'sha256': '6ba0ea355aeea35b675aad6b08d64f47a56519c32ab4846fd6e721066b49005d'}, 'data/binding-v1/images/binding-evaluation-0024-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVklEQVR4nO3bS07jShiA0XDV7CeDXoaXyzIYZDEMkJjcQdQoCukQmsdnV50z5eGyEn/5XZi75+enHUDhv+SoAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMj86g4Nm7Hf/77l2w6Hx+9fy1Dunp+f6jXAhqPzN2J0CwGCr+zOW0p0hQDBTel5uL+//uPLy8uVr8rQRQIEl9PzbnH+oUcydEaAmNp3pOeUDF0nQMzrrD5f2J13S6RBRwLEjH4yPadk6IwHEZlOVZ+3x9p/9V/cNscENOZbwUfrLS/0T6bnyih0mPjFMgExkZXU5+zo+7E+/D5EgJjFeurzdg37WRvkf8HOreGt+c+uPws3s7XV5+jh/v71Jdvvf094L2YCYnzrrM/R5HOQADG4NdfnaOYGCRAjW399Jm+QADGFNddnKyv8DgLEsF5Hia1c2w9/1jnPECRAjGnr1/B+4+u/kQAxuK2MP1tc7ecJEAPa3M3XtDdiAgRkBIjRbHr8mW0IEiAgI0AMZbyRYT/cGZ0SIMa03fuvMdZ/IwECMgLEOAbYfp5tK1qAgIwAARkBAjICxCAG3ijZjXt2AsRoxtiBHu9cLhIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgBjN8vKyG8Uy0LlcJEAM4nB43I3rMOjZCRCQESAgI0BARoAYcKNkjL3b5c9ZjLoBJEBASYAY09aHoGXj67+RADGU8e5WDsOd0SkBAjICxGgG2IpeJth+PhIgICNADGjTQ9AyzfgjQIxvWw1aNrXazxMgxrT18eGw8fXfSIAY1uZuxJaZbr6OBIgprL9By+pX+B0EiJGdjhJrvsKXk7XNM/4IEONbf4OWWesjQExhzQ1aJq6PADGLdTZombs+AsRE1tagZfr6CBBzWU+D1OdIgJi6QT+fobODHiaujwAxo7Nr/icbdHasw9z12e12v+oFQOB45e/3v0+78HB//31HlJ6LBIh5HQ6Prw06bcQXlujieKU+rwSIqZ2NQl84EEnPLQQIrmXo1bs9ur6XJD0XCRBcy9Dn96ql5woBgsu9+FuJPvp7uEKA4P2C3Bgj0fkoATq3hof0WRtl+SYeRAQyAgRk7p6fn7qjA1MzAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELCr/A+j9VR8inxEEgAAAABJRU5ErkJggg==', 'sha256': 'eebe9948e4814ab7d1264892cea3e60216bbeb688fd9120c9f1a7261da3893c8'}, 'data/binding-v1/images/binding-evaluation-0024-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFY0lEQVR4nO3bTU4bSRiAYTJKHwfJi0hzk1yEc+Qic5NILCzlKlkgsZmFJ8gyjjET4O2uep4NC366WtAvXxXm08PDzxuAwl/JVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dxdGjZjt/tyzYft9/fvv5ahfHp4+FmvATYcnd8Ro2sIELxld55TogsECK5Kz+Pd7eVPX779uPBeGTpLgOB8el4szv/okQydECCm9h7pOSZDlwkQ8zqpzxt258USadCBADGjj0zPMRk64YWITKeqz/Nr7d76L26bYwIa80fBr9ZrvtEfmZ4Lo9B+4m+WCYiJrKQ+J1ffjfXL71UEiFmspz7P17CbtUH+F+zU8vc/N5v1+P1rvYSVWlt9Dh7vbp/2Yrvdlwn3YiYgxrfO+hxMPgcJEINbc30OZm6QADGy9ddn8gYJEFNYc322ssL3IEAM62mU2Mqz/fhrnfMMQQLEmLb+DO82vv4rCRCD28r4s8XV/jkBYkCb23xNuxETICAjQIxm0+PPbEOQAAEZAWIo440Mu+Hu6JgAMabt7r/GWP+VBAjICBDjGOD4ebajaAECMgIEZAQIyAgQgxj4oORm3LsTIEYzxgn0ePdylgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICxGiWbz9uRrEMdC9nCRCD2O/vb8a1H/TuBAjICBCQESAgI0AMeFAyxtnt8usuRj0AEiCgJECMaetD0LLx9V9JgBjKeLuV/XB3dEyAgIwAMZoBjqKXCY6fDwQIyAgQA9r0ELRMM/4IEOPbVoOWTa32zwkQY9r6+LDf+PqvJEAMa3MbsWWmzdeBADGF9TdoWf0K34MAMbLjUWLNT/hytLZ5xh8BYnzrb9Aya30EiCmsuUHLxPURIGaxzgYtc9dHgJjI2hq0TF8fAWIu62mQ+hx8/u8tzGG/v9/tvhxX4PHu9iMXcBK+/cT1MQExo5Nn/iNHIfU5YQJiRocn/yNHIek5S4CY1/F27LgRb1iis+OV+jwRIKZ2Mgq94UAkPdcQILiUoScv9ujyWZL0nCVAcClDf35WLT0XCBCc78XvSvTar8MFAgQvF+TKGInOawnQqcfvX+slsDrK8k68EBHICBCQ+fTw8LO7OjA1ExCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAE3lX8BY7dLGe2NaxEAAAAASUVORK5CYII=', 'sha256': '353bd02633a0f59b88a07593e796d0a22f62e9c4c1aa6cbb41319dccf98f0be6'}, 'data/binding-v1/images/binding-evaluation-0024-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFX0lEQVR4nO3bTU4bSxhAUfKUXg6SB1lGdpN1ZDdvGRlYylYYIHnyBn5BlnGMSYDbXXXOhAE/XS3oy1eF+fT4+HAHUPgnuSqAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMp+7S8Nm7HZfbvmw/f7H+69lKJ8eHx/qNcCGo/M7YnQLAYK37M5zSnSFAMFN6Tl8u7/+6cv3n1feK0MXCRBcTs+LxfmDHsnQGQFiau+RnlMydJ0AMa+z+rxhd14skQYdCRAz+sj0nJKhM16IyHSq+jy/1u6t/+K2OSagMX8U/Gq95Rv9kem5MgrtJ/5mmYCYyErqc3b13Vi//F5FgJjFeurzfA27WRvkf8HO/bssd5v19XCol7BSa6vP0eHb/dNebLf7MuFezATE+NZZn6PJ5yABYnBrrs/RzA0SIEa2/vpM3iABYgprrs9WVvgeBIhhPY0SW3m2D7/WOc8QJECMaevP8G7j67+RADG4rYw/W1zt3xMgBrS5zde0GzEBAjICxGg2Pf7MNgQJEJARIIYy3siwG+6OTgkQY9ru/muM9d9IgICMADGOAY6fZzuKFiAgI0BARoCAjAAxiIEPSu7GvTsBYjRjnECPdy8XCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAjGb5/vNuFMtA93KRADGI/f7H3bj2g96dAAEZAQIyAgRkBIgBD0rGOLtdft3FqAdAAgSUBIgxbX0IWja+/hsJEEMZb7eyH+6OTgkQkBEgRjPAUfQywfHzkQABGQFiQJsegpZpxh8BYnzbatCyqdX+PQFiTFsfH/YbX/+NBIhhbW4jtsy0+ToSIKaw/gYtq1/hexAgRnY6Sqz5CV9O1jbP+CNAjG/9DVpmrY8AMYU1N2iZuD4CxCzW2aBl7voIEBNZW4OW6esjQMxlPQ1Sn6PP/7+FOez3P3a7L6cVOHy7/8gFnIVvP3F9TEDM6OyZ/8hRSH3OmICY0fHJ/8hRSHouEiDmdbodO23EG5bo4nilPk8EiKmdjUJvOBBJzy0ECK5l6MmLPbp+liQ9FwkQXMvQ359VS88VAgSXe/G7Er3263CFAMHLBbkxRqLzWgJ07uvhUC+B1VGWd+KFiEBGgIDMp8fHh+7qwNRMQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBNxV/gNhB0tBnrOlIgAAAABJRU5ErkJggg==', 'sha256': 'd81810eabe3ecdeea4515eac664f3c0300181d1645beb874e8395bdf40c25b02'}, 'data/binding-v1/images/binding-evaluation-0025-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFH0lEQVR4nO3dO04jWxRAUfPUzMcBEjPpQTKTll7gwRBYqqQDqxFgMB8Bu+yzVob41JVQbc69huJqv7/fABT+S64KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZlf3aXhXG23N6c/YLf7/6fWct6u9vv7eg1wCdF5jRidIEDwLd05pkTHBAg+nJ7r27vTn7v8+f3au2ToMQGCd6Xnzeh8KEYydCBAcCo9n+7Oe0q0G58hAYKX6/OF6TmRod3sBgkQPK/PN6XntQztBjdIgJjuZwafY4tRyG9CM1xVn+Nrbb/u9f4zYgJirh/edr1mGbwdMwEx1Erqs3l69WlzkAAxXVuf9awhIUBM9DBorOfOv/63klFDkAAxzvrv8O3qV/hVBIhZ1nP0c2zgYZAAMdTa6rPmVX0fAWKQFR79DD8MGvpExLP+1k77VREumAmIKc5i/Jk2BAkQkBm6BXuw/h+G73nIHpc0/hxc394dvuPb7c0Fb7pNQEBGgICMAHH5zm7/NecoWoCAjAABGQECMgLEhTvTA6Ahx0ACBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBAX7uFxguf4SMnl35ov9aGIAgRkBAjICBCQESAu35keAy2XfgAkQEBJgICMADHC2e3ClgH7LwECSgLEFGc0BC0zxh8BAkoCxCBnMQQtY8YfAWKudTZoWeWqvo8AMcvjsWJtd/vyaD0Txh8BYqL139u71a/wqwgQE63wMGiZdPTzQICYbg0NWlawhoQAMdR6DoOWeUc/DwSIuZ416OcztDy96LT6CBDTPbvnf7JBy9NrDazPZrP5VS8AYoc7f7u9Oby5/Pl9fXv33RcdPvg8MAHBj27HbLseEyB4dTv2tRk6/oK72fWxBYNT27HHe6VP78terJj0HAgQvJ2hZx15M0YnRifpeUyA4AMZOvjc1kx6jgkQvKsaL5boQ1+BYwIEH+7ImzESnXcSIPgwffkqXoYHMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGSm/ynG2H+HAmtgAgIyAgRkrvb7++7qwGgmICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYFP5C8qOO6Pja/xoAAAAAElFTkSuQmCC', 'sha256': '512fc2548403844c97cb3fd0a1b8cd20454b33dbf5b61d8b4614c4175e97a8cd'}, 'data/binding-v1/images/binding-evaluation-0025-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFGElEQVR4nO3dS04jSRRAUWhVLgfJA3bGOnpnNbDUW2FgiUmrhQq5MBjcBb5pv3OmfDIkKy8vApPc7naPNwCFv5KrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmR3dpuFSbzf3xT9huf55rLZftdrd7rNcA1xCd94jREQIE39KdQ0p0SIDg5PQ8Pdwd/9rl73/e+5AM7RMg+FR6PozOSTGSoWcCBMfS87+785kSbcdnSIDg7fp8YXqOZGg7u0ECBK/r803peS9D28ENEiCmO8/gc2gxCnknNMNV9Tm81ubrft9/QUxAzHXmbdd7lsHbMRMQQ62kPje/X33aHCRATNfWZz1rSAgQE70MGuu5859+rWTUECRAjLP+O3yz+hV+FQFilvUc/RwaeBgkQAy1tvqseVXfR4AYZIVHP8MPg4Y+EfGiX9ppbxXhipmAmOIixp9pQ5AAAZmhW7AX6/9h+JmH7HFN48+zp4e751d8s7m/4k23CQjICBCQESCu38Xtv+YcRQsQkBEgICNAQEaAuHIXegA05BhIgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQHiyr08TvASHym5/FrztT4UUYCAjAABGQECMgLE9bvQY6Dl2g+ABAgoCRCQESBGuLhd2DJg/yVAQEmAmOKChqBlxvgjQEBJgBjkIoagZcz4I0DMtc4GLatc1fcRIGbZHyvWdrcve+uZMP4IEBOt/97ern6FX0WAmGiFh0HLpKOfFwLEdGto0LKCNSQEiKHWcxi0zDv6efGjXgBkttufm839fgWeHu7OuYDl9/BNq48JiOle3fPnHIWW8fURIPjvzj//duzVtms7sj62YHDu7ZjBZ58AwRsN+o4MHc5W29n1ESB4owiHGfqTEr25p5OeZwIEH2foVUc+jNGRgyTp2SdAcEKG/uSgWnoOCRB8qhpvluik78AhAYKTO/JhjETnkwQITqYvX8UbEYGMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmel/ijH236HAGpiAgIwAAZnb3e6xuzowmgkIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAi4qfwLwUIvxXD2zx8AAAAASUVORK5CYII=', 'sha256': '918647dd1104c79d195f036868125a5eb1486638455395b02d0cb0cb5a4bed70'}, 'data/binding-v1/images/binding-evaluation-0025-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJklEQVR4nO3cTUokWRSAUW0qllOQA6F3UhtxHb2R3klBDRJ6Kw4EJ00jJZapqXZpfhF5z5n6Ew8kPu57Rsbl7e3NBUDhj+SqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD50l0atmq3uzr+Dfv9j1OtZdsub29v6jXAOUTnJWJ0hADBp3TnkBIdEiB4d3rurr8e/9nlr39e+pIMPSZA8Kb0vBqdd8VIhu4JEBxLz//uzltKtB+fIQGC5+vzgek5kqH97AYJEDytzyel56UM7Qc3SICY7jSDz6HFKORJaIar6nN4rd3H/b9/Q0xAzHXibddLlsHbMRMQQ62kPhe/Xn3aHCRATNfWZz1rSAgQEz0MGuu58+9+rmTUECRAjLP+O3y3+hV+FAFilvUc/RwaeBgkQAy1tvqseVWfR4AYZIVHP8MPg4a+EXHTf9ppj4pwxkxATLGJ8WfaECRAQGboFuzB8uffFxtx9/1bvYQN29D4c+/u+uv9RzR2u6sz3nSbgICMAAEZAeL8bW7/NecoWoCAjAABGQECMgLEmdvoAdCQYyABAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBIgz9/A6wfsXDG7L8nPN5/pSRAECMgIEZAQIyAgQ52+jx0DLuR8ACRBQEiAgI0CMsLld2DJg/yVAQEmAmGJDQ9AyY/wRIKAkQAyyiSFoGTP+CBBzrbNByypX9XkEiFkejxVru9uXR+uZMP4IEBOt/97er36FH0WAmGiFh0HLpKOfBwLEdGto0LKCNSQEiKHWcxi0zDv6efClXgBk9vsfu93V4wrcXX895QKWX8M3rT4mIKZ7cs+fchRaxtdHgOC/O//027En2679yPrYgsGpt2MGn8cECJ5p0Gdk6HC22s+ujwDBM0U4zNDvlOjZPZ303BMgeD1DTzryaoyOHCRJz2MCBO/I0O8cVEvPIQGCN1Xj2RK96zdwSIDg3R15NUai80YCBO+mLx/Fg4hARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMz0j2Lcff9WLwHmMgEBGQECMpe3tzfd1YHRTEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMBF5V/LozB6kmTNbAAAAABJRU5ErkJggg==', 'sha256': 'fdc03dad7126817f7b79cdff98ebb2e6beb084248264b268c36c72e21a3d5b6d'}, 'data/binding-v1/images/binding-evaluation-0026-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE/0lEQVR4nO3dMW4TWxSA4fBElvMkF7TeEO1bB+3bUFoKS2yFIhINhYVlnNgmYOaf5Hxf7fHcZn7de+Jo3j0+fr0DKPyT3BVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGTe3711m82Huzdnt/tcLwFuwA4IyAgQkHn7R7CDb//9e/fK3X/6Ui+B64d6B+RfNyhAsMwk8fjDYnSZAMFf/CPG/nIZOkeA4GXdefj/2+Vrtx/vz32bEp0QILhen6vROffhkxhtNh806JgAwc3S89T+8uMMOZQdEyB4vj5/mJ6rGdppkADB0/rcMD0XMrTRID9EhGXq8+z3b97i/wm9iAAx3ZL1eXqXzewGOYIx2uH5XyY9zx7HNoPPYnZAzFXV5+Dhx33H7oMEiKHW9sxvVraeZQgQEy0/9znnYfY8SIAYra3PetZQESDGyUc/Tz1MHQYJELOs/wnfrH6FNyRADLWe7c8617MMAQIyAsQgK5z+DJ8ECRCQESCmWPn2Z+YmSICAjAABGQECMgLECK9iADRwDCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMADHC4dWj+5eRrtn2xwonvC5VgICMAAEZAQIyAsQUr2IMtJ00ABIgoCRADLLyTdB22PZHgICSADHU2jZB25WtZxkCxCzrP93sVr/CGxIgxlnhJGg7b/qzJ0CMtoYGbVewhooAMdHxRqN9/rdHd5+2/REg5lrb075b2XqWIUDMlQ+DtlNHPwfv6wVAabf7vH8B6b4Fi703dfjJ68AOiOmWnwepz4EAwaINUp9jjmDw01ns7x3HTtK2G18fAYLnG3TbDD3dVanPngDBaRdumCHpuUyA4MpW6KQjV2N0YYqkPicECC6V4qREvzel1p1zBAhedij7jcs5R4DgZR25GiPR+XUCBC+jLzfkh4hARoCAjAABmUEzoPtPX+olAD+xAwIyAgRk3j0+fu3uDoxmBwRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECLirfAdXxgV7WOKfPgAAAABJRU5ErkJggg==', 'sha256': '7fd5dc57e0d4d26483a1210d2960abd31acec10d89edce4c176a34d2d143852f'}, 'data/binding-v1/images/binding-evaluation-0026-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE/0lEQVR4nO3dMW4bVxRAUTnwLCcAC7fekNusI202lNYFAW/FhQA1KQgTNCWSlkPPHeudU3M4v5mL/58ozLvHx68PAIU/krsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM+4e3brf78PDm7Pef6yXAHdgBARkBAjJv/wh29O8/Tw+/uY+flnoJ3D7UOyD/uEEBgnUmiacfFqPrBAh+4R8xDpfL0CUCBK/rztNff16/dvn7y6VvU6IzAgS363MzOpc+fBaj3e6DBp0SILhbep47XH6aIYeyUwIEL9fnf6bnZob2GiRA8Lw+d0zPlQztNMgPEWGd+rz4/bu3+H9CryJATLdmfZ7fZTe7QY5gjHZ8/tdJz4vHsd3gs5gdEHNV9Tl6+nbfsfsgAWKorT3zu42tZx0CxETrz30ueZo9DxIgRmvrs501VASIcfLRz3NPU4dBAsQs23/Cd5tf4R0JEENtZ/uzzfWsQ4CAjAAxyAanP8MnQQIEZASIKTa+/Zm5CRIgICNAQEaAgIwAMcJvMQAaOAYSICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoAY4fjq0cPLSLds+bbCCa9LFSAgI0BARoCAjAAxxW8xBlomDYAECCgJEINsfBO0DNv+CBBQEiCG2tomaNnYetYhQMyy/dPNfvMrvCMBYpwNToKWedOfAwFitC00aNnAGioCxESnG432+V9O7j5t+yNAzLW1p32/sfWsQ4CYKx8GLVNHP0fv6wVAab//fHgB6aEFq703dfjJ68gOiOnWnwepz5EAwaoNUp9TjmDw3Vns1x3HztK2H18fAYKXG3TfDD3fVanPgQDBeRfumCHpuU6A4MZW6KwjN2N0ZYqkPmcECK6V4qxEPzel1p1LBAhedyj7icu5RIDgdR25GSPR+XECBK+jL3fkh4hARoCAjAABmUEzoI+flnoJwHfsgICMAAGZd4+PX7u7A6PZAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAh4q/wHm2QV70vA47wAAAABJRU5ErkJggg==', 'sha256': '93819316bb3232a4b5a4598fef48a395249409d669f004f02227fd6357bab724'}, 'data/binding-v1/images/binding-evaluation-0026-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE+klEQVR4nO3dQW5aSRRAUacVltMSg+ys19E7ywCpt5KBJU8yQEE02CaO8L9lv3PGwK/Jv6p6YP8vj48/HgAKfyVXBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABma8Pn91+/+3h0zkcvtdLgDuwAwIyAgRkPv8R7OTpn78fPrjdv//VS+D2od4B+fcNChBsM0k8f7EYvU6A4B2/xDi+XYZeIkDwtu7cPMtfn5RPn6ZEFwQIbtfnTQPE8xdfxGi//6ZB5wQI7paea8e3n2fIoeycAMHz9bnj16bPZuigQQIE1/V5p19sXGRor0F+iAjb1OfZz99/xr8TehMBYrot63N9lf3sBjmCMdrp/t/4h/Lnx7H94LOYHRBzVfU5efp13bH7IAFiqNXu+f1i69mGADHR9nOflzzNngcJEKOt8D8SnhZYQ0WAGCcf/Vx7mjoMEiBmWf8O3y+/wjsSIIZaZ/uz5nq2IUBARoAYZMHpz/BJkAABGQFiisW3PzM3QQIEZAQIyAgQkBEgRvgQA6CBYyABAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBIgRTo8ePT6MdGW7Xyuc8LhUAQIyAgRkBAjICBBTfIgx0G7SAEiAgJIAMcjim6DdsO2PAAElAWKo1TZBu8XWsw0BYpb1TzeH5Vd4RwLEOAtOgnbzpj9HAsRoKzRot8AaKgLEROcbjfb+351dfdr2R4CYa7W7/bDYerYhQMyVD4N2U0c/J1/rBUDpcPh+fADpsQWbPTd1+MnrxA6I6bafB6nPiQDBpg1Sn3OOYPC/s9j7Hccu0nYYXx8BgucbdN8MXe+q1OdIgOCyC3fMkPS8ToDgxlbooiM3Y/TKFEl9LggQvFaKixL92ZRad14iQPC2Q9kfvJ2XCBC8rSM3YyQ6v0+A4G305Y78EBHICBCQESAgM2gGNPn/zsGa7ICAjAABmS+Pjz+6qwOj2QEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIeKj8BsykGNLoKhsYAAAAASUVORK5CYII=', 'sha256': 'ca554d367fd4cd260197ad8aecdc791f22ef8734a7915e2b4f67605f0af161b2'}, 'data/binding-v1/images/binding-evaluation-0027-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQklEQVR4nO3ZvW0bSRiAYflg9cPAqTtgJVfQVaIOLlXAYhwIUHIBYWKPlCjr992deZ7IAoidEc199c3y28PDrxuAwl/JqgACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM927pldrtftwM5HC4r7cw+OfBO/weAgTv+iO0fLEYvZYAwYcNv6crKNEfEqBn/fvP481m/fz7tt7CIN25u33hndw/PvE5UaI/JEBM7cn6vBid5158GaPd7ocGXSFATOoyPa/qzpNOV1iW6LiQDD3J1/DM6Kw+d7e376/P9QsO9u3qRzEBMXt6Pm+t48VP05BR6JIJiIl8ZX2eW8UotCRAzGJ553/4meu6s+U06ESAmLE+yR406JIAMb411Ody9Z0GCRDDW099Lvewm75BAsTI1lafIw06ESCmsJ76rHM/FQFiWKfhYp13+93vXc08BAkQY9rWXb3b1G4/kAAxuHWOP+vf29cQIAa08sPX0t3cBzEBYjTbvZN3m935mwkQw1r/+LOtfX4GAQIyAsRQNvT0Z2naJ0ECBGQECMgIEOPY6Plr5lOYAAEZAQIyAgRkBIhBbPoB0LSPgQQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIAZxONwf/7F/fLzZpv3vnZ9+l+EJEJARICAjQEBGgBjHph8D7ed7ACRAQEmAgIwAMZSNnsL2U56/BAgoCRDD2soQtN/IPj+DADGa7Z5iDpvd+ZsJEAPa0JOg/axPf44EiMGtuUH7Fe/tawgQY9rWQHHY1G4/kAAxrJUfxPZzH76OBIgprK1Ba9tPRYAY2XK4WM89v9zJYeLxR4AY39oapD5LAsT41tMg9TkjQExhDQ1Sn0sCxKQN+soMnS2nPicCxETO7vyvadDZKuqz9P1/P8Hojvf/bvdjWYe729vPWEt6XmQCYkaXo9DHTkOXF1SfJ5mAmNTZKLQcWN48ED1ZMem5QoCY2uFwv2zQZUdejNH10Ul9rvv28PDrhZdM5vLjuGlugOS/3tv+h0xAcF6NN5dId15LgOBaR16Mkei8hyMYkPE1PJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgpvIfoBIiZZRt97sAAAAASUVORK5CYII=', 'sha256': 'e2c193a0b85066f0464cff422309f9215c99ff483786771f61df4dacbf8f8066'}, 'data/binding-v1/images/binding-evaluation-0027-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPUlEQVR4nO3aTW5aSRhAUacVlhOJgZfRu8k6sptehgdI2UoGlpj0AAUhwDjxD/e9qnNGloV5Zcy7/qrEl+fnXw8AhX+SqwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyX7tLL9R2+/gwkN3uqV7C4O8Hr/B7CBC865/Q6YPF6G8JEHzY8Ht8BiX6QwL0ov82m4fV+ne/r5cwSHf237/d/vHNj583nlOJbhMgpna1Pq9G56UHX8Zou33UoBsEiEldpuevunPV8RlOS3S4kAxdJUDM6Kw+70/PmcMTnmVIgy4JEHP57PTcyJBR6JIPIjKRe9bnpasM9kGzdzIBMYvTO/8+6bkxCpmDDkxATCGsz9XrmoMOBIjxLaE+l1ffapAAMbzl1OdyDdvpGyRAjGxp9TnQoCMBYgrLqc8y11MRIIZ1HC6Webfvf69q5iFIgBjTuu7q7apW+4EEiMEtc/xZ/truQ4AY0MI3X6f2c2/EBIjRrPdO3q525W8mQAxr+ePPutb5GQQIyAgQQ1nR6c+paU+CBAjICBCQESDGsdL918y7MAECMgIEZAQIyAgQg1j1AdC0x0ACBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCD2O2eDl9sfvx8WKfN75Uff5fhCRCQESAgI0BARoAYx6qPgTbzHQAJEFASICAjQAxlpbuwzZT7LwECSgLEsNYyBG1Wss7PIECMZr27mN1qV/5mAsSAVnQStJn19OdAgBjckhu0WfDa7kOAGNO6Bordqlb7gQSIYS18I7aZe/N1IEBMYWkNWtp6KgLEyE6Hi+Xc86cr2U08/ggQ41tag9TnlAAxvuU0SH3OCBBTWEKD1OfS1yvfgxHtdk/b7eNpC/bfv93n0mfJU58jExATObvz7zMKqc8NJiDmcrj/7zMKSc+rBIjZt2OfkaHL2Up9rhIgJnU2Cp1W480lurqnk54bBIipnY1Clx15NUa3D5LU57Yvz8+/XnnIZC7fjqvmBkj+9F72P2QCgvNqvLlEuvO3BAhudeTVGInOe9iCARkfRAQyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEPFT+B/1kIm3LHa5WAAAAAElFTkSuQmCC', 'sha256': '53f0b688fc648217996319ffbcb65aaeee18ecba85dae74438fca8b0f7f07a10'}, 'data/binding-v1/images/binding-evaluation-0027-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFP0lEQVR4nO3aPW5iSRhAUfeoWU5LBE5nQ5P2OjqdDU3qAKm30oElJxOgRggwbv9x36s6J7IszCtj3vVXJb48Pv66Ayj8lVwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnaXXqhttv7u4Hsdg/1EgZ/P3iF30OA4F3/hI4fLEavJUDwYcPv4RmU6A8J0LP++/fpbrX+/mdTL2GQ7jx9/3b9xzc/fl55TiW6ToCY2sX6vBid5x58HqPt9l6DrhAgJnWenld156LDMxyXaH8hGbpIgJjRSX3en54T+yc8yZAGnRMg5vLZ6bmSIaPQOR9EZCK3rM9zVxnsg2bvZAJiFsd3/m3Sc2UUMgftmYCYQlifi9c1B+0JEONbQn3Or77VIAFieMupz/kattM3SIAY2dLqs6dBBwLEFJZTn2WupyJADOswXCzzbn/6vaqZhyABYkzruqu3q1rtBxIgBrfM8Wf5a7sNAWJAC998HXuaeyMmQIxmvXfydrUrfzMBYljLH3/Wtc7PIEBARoAYyopOf45NexIkQEBGgICMADGOle6/Zt6FCRCQESAgI0BARoAYxKoPgKY9BhIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgBjEbvew/2Lz4+fdOm1+r/zwuwxPgICMAAEZAQIyAsQ4Vn0MtJnvAEiAgJIAARkBYigr3YVtptx/CRBQEiCGtZYhaLOSdX4GAWI0693F7Fa78jcTIAa0opOgzaynP3sCxOCW3KDNgtd2GwLEmNY1UOxWtdoPJEAMa+Ebsc3cm689AWIKS2vQ0tZTESBGdjxcLOeeP17JbuLxR4AY39IapD7HBIjxLadB6nNCgJjCEhqkPue+XvgejGi3e9hu749b8PT9220ufZI89TkwATGRkzv/NqOQ+lxhAmIu+/v/NqOQ9LxIgJh9O/YZGTqfrdTnIgFiUiej0HE13lyii3s66blCgJjaySh03pEXY3T9IEl9rvvy+PjrhYdM5vztuGpugORP72X/QyYgOK3Gm0ukO68lQHCtIy/GSHTewxYMyPggIpARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgrvI/KUAibTnkkP0AAAAASUVORK5CYII=', 'sha256': '5ec60a64b8509b4de4d0af9847a15976dd64aaf745e58221abf2e23c93576db6'}, 'data/binding-v1/images/binding-evaluation-0028-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFC0lEQVR4nO3dPW4aWxiAYXIV9pPiSt5E6izS9d1EpBReTApLNCnQRQgM5i+8c2aep3KBh+NiXn3nAObL+/vvFUDhn+RZAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDm62pGvn37dzWst7df9RLg2UxAQEaAgMystmA7/63Xq0F832zqJUDGBARkBAjICBCQESAgI0BARoCAjAABmXm+Dwge/jken5X5GwSI5brqw4P7DxajRxEgFuf+Dy3vrqBEdxIgFuRMetYvr+d/d/Pzx6kLytDNBIjlpufT6Jx68EGMZOhmAsTi0nNVdz60u8J+iWToBl6GZ0H1Wb+83l+f8xcc+r/iPZ8AMVv7LXh4evYdXFyDLmcLxiIGnyc86frldbcjsx27kAmIuUnq8+FzGYU+JUDMytO2XafYjl1FgJhtfcKVaNCFBIgZausznTVMnwAxE7tBYzp3/m4lhqBTBIg5mP4dPv0VJgSI4U3n6OeYw6DzBIj5mFp9pryqiRAgxjbBo59jDoNOESAgI0AMbIjxZ8sQ9CEBAjICxKgGGn+2DEHHBAjICBCQESCGNNz+a8su7IAAARkBAjICBGQEiPEMegC05RhonwABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICxHje3n5tf9j8/LEazW7Nb///FUsmQEBGgICMAAEZAWJIgx4DOQA6IEBARoCAjAAxquF2YfZfxwQIyAgQAxtoCDL+fOjrao6+bzb1EoDPmYAY2xBDkPHnFAFiPqbZoGmuaiK+vL//rtcA99r/jq1JfVnYfn2MP8dMQMzB9O/t6a8wIUDMxAQPgxz9fEqAmKEpNGgKa5g+AWI+9geN9v539HMhh9DM+UD6+WfSB+FTn/NMQMzNwT3/zFFIfa5lAmK2nvzavG3XDQSIOXvOdszgczMBYlkNemyGjvd36nMVAWKhGbqnRB+eK0nPDQSIpWfowhidOcyWnpsJEItzJkPXkp47CRDLdXOJdOdRBAguipHo/A0CBGS8ExrICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIGBV+QMLPgpoDUXrSQAAAABJRU5ErkJggg==', 'sha256': '95cacffa9e513b18a2cf56694082ecefd0b336fd10e29373146368224cd18940'}, 'data/binding-v1/images/binding-evaluation-0028-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFDElEQVR4nO3dO24bSRRAUXpgLWcABU69IaezDqezoUkdEPBWHAhw4oAYguBP/Im3q/ucSAHVLAV98arEBj+9vf1aART+St4VQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQ+r2bk9fXLaljr9Y96CfBsJiAgI0BAZlZbsK3f//y9GsTL95/1EiBjAgIyAgRkBAjICBCQESAgI0BARoCAzDw/BwQPf47HszIfQYBYrqseHtx9sRg9igCxOPc/tLy9ghLdSYBYkDPp+e/f3+d/9+u3l1MXlKGbCRDLTc+70Tn14r0YydDNBIjFpeeq7hy1vcJuiWToBgLEgupzf3r2bC64lyENupwAsYj6PDw9ZzKkQZfzQURm6PX1y9Pqc/Rd9hbAKQLE3Hz0tuuMvffSoHfZgjErzx989tiOXcUExHzk9Tn67uagMwSIGWrrM501TJ8AMRPbQWM6d/52JYagUwSIOZj+HT79FSYEiOFN5+jnkMOg8wSI+Zhafaa8qokQIMY2waOfQw6DThEgICNADGyI8WfDEHSUAAEZAWJUA40/G4agQwIEZAQIyAgQQxpu/7VhF7ZHgICMAAEZAQIyAsR4Bj0A2nAMtEuAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAWI86/WPzQ9fv72sRrNd8/r/v2LJBAjICBCQESAgI0AMadBjIAdAewQIyAgQkBEgRjXcLsz+65AAARkBYmADDUHGn6M+r+bo5fvPegnA+0xAjG2IIcj4c4oAMR/TbNA0VzURn97eftVrgHvtfsfWpL4sbLc+xp9DJiDmYPr39vRXmBAgZmKCh0GOft4lQMzQFBo0hTVMnwAxH7uDRnv/O/q5kENo5nwg/fwz6b3wqc95JiDmZu+ef+YopD7XMgExW0/+37xt1w0EiDl7znbM4HMzAWJZDXpshg73d+pzFQFioRm6p0RHz5Wk5wYCxNIzdGGMzhxmS8/NBIjFOZOha0nPnQSI5bq5RLrzKAIEF8VIdD6CAAEZn4QGMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAhYVf4ATg8bPQAmcRAAAAAASUVORK5CYII=', 'sha256': '8a4c891c0170a3f869df393a598ac02a03f4a8d496d78830d86121f32000ea5a'}, 'data/binding-v1/images/binding-evaluation-0028-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFC0lEQVR4nO3dvW0bSRiAYfpg9aPAqTtwJS7oKrkOLnWgYhwIcHIBcQTBP4mkxHdn9nkiBQR3FOyLb0Zc8cvr6+8NQOGv5KoAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPN1M5Hn52+bYb28/KqXAI9mAgIyAgRkptqC7fzz9LQZxI8/f+olQMYEBGQECMgIEJARICAjQEBGgICMAAGZOT8HBB/+HI9nZT6DALFeVz08uP9iMfooAsTq3P/Q8u4dlOhOAsSKXEjPv3+/8UzM959P595Qhm4mQKw3PW9G59yLD2IkQzcTIFaXnqu6c9LuHfZLJEM3ECBWVJ/703Ng+4YHGdKg9xMgVlGfD0/PhQxp0Pv5ICITen7+9rD6nLzKwQI4R4CYzWdvuy44uJYGvckWjKk8fvA5YDt2FRMQ88jrc/Lq5qALBIgJtfVZzhqWT4CYxG7QWM6dv1uJIegcAWIGy7/Dl7/ChAAxvOUc/RxzGHSZADGPpdVnyataCAFibAs8+jnmMOgcAQIyAsTAhhh/tgxBJwkQkBEgRjXQ+LNlCDomQEBGgICMADGk4fZfW3ZhBwQIyAgQkBEgICNAjGfQA6Atx0D7BAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgxvPy8mv7w/efT5vR7Nb88v9vsWYCBGQECMgIEJARIIY06DGQA6ADAgRkBAjICBCjGm4XZv91TICAjAAxsIGGIOPPSV83M/rxZ7z/VQ4rZAJibEMMQcafcwSIeSyzQctc1UJ8eX39Xa8B7rX/HVuL+rKw/foYf46ZgJjB8u/t5a8wIUBMYoGHQY5+3iRATGgJDVrCGpZPgJjH/qDR3v+Oft7JITQzH0g//kz6IHzqc5kJiNkc3POPHIXU51omIKb14L/N23bdQICY2WO2YwafmwkQ62rQx2boeH+nPlcRIFaaoXtKdPJcSXpuIECsPUPvjNGFw2zpuZkAsToXMnQt6bmTALFeN5dIdz6KAMG7YiQ6n0GAgIxPQgMZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGwq/wHL4BwaisnyOQAAAABJRU5ErkJggg==', 'sha256': '7d87c8c4bb0fc3d09946a0aac97fb62517f6c1b9108b4f399f73fdd5286a4bbd'}, 'data/binding-v1/images/binding-evaluation-0029-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE90lEQVR4nO3dTWobSRiAYWdIjjPgRWBukovMOeYic5OBLARzlSwM2WQhEEL/tpR+S9XPs3arC9v98lUh6E9vbz9eAAp/JHcFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh87m49qNfXry9Pa7P5Xi8B3sEEBGRMQPDuKdik+SgCdNaXv/59eRI///tWL2FdW+/9HxajewgQa3fnqd/2chn6GAFipS505+fff16+9ss//5/7NCV6FwFijY7rczU65374IEavr1816HYCxLrcmZ5j28v3M2RTdjsBYr31uTM9VzOkQVcJEGuszwPTcyFDGnSVLyKyCgvU5+TnP/UX6xcgQMxvyfoc30WDLrAFY3K753+Z9JzcjtmLnWMCYmZVfXZ29zUHnSRATGu0Z3609YxAgJjT8uc+5zgPukCAmFxbn3HWMCYBYkL50c8xh0EnCRCzGf8JH3+FixEgpjXO+DPmekYgQEBGgJjKgKc/+5wEHRAgICNAzGPw8WfLELRPgICMAAEZAQIyAsQknuIAaMsx0I4AARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAsQkdq8e3b6MdGS7FW5W/7pUAQIyAgRkBAjICBDzeIpjIAdA+wQIyAgQUxl8CDL+HBAgICNATGu0IWi09YxAgJjN+Lub8Ve4GAFiQgOeBDn9OUmAmNwIDRphDWMSIOa0P2i0z//+3Y0/BwSIaY32tI+2nhEIEDPLD4Mc/Vz2uV4A/F6bzfftC0i3LVjsval2XrcwATG/5c+D1OdGAsQqLNkg9bmdLRir24v9vu3YQdrU5yoBYqUNemyGjqcq9bmFALEu2y48MEPScw8BYo0ORqGDjlyN0YVTJPV5l09vbz/ed8XsDv4vn4v//uov7jf/MSYg1u54U/aBy/kYAYLDjlyNkeg8ii0YkPFFRCAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAh4qfwCNd8Eb3TK81YAAAAASUVORK5CYII=', 'sha256': '790a10ce581ec65e44bae8b1583f49045632b765cc206defb344789f50b97c74'}, 'data/binding-v1/images/binding-evaluation-0029-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE80lEQVR4nO3dO24bSRRAUXng2Y8Cp7OhWZA35NQBF+NAwCQTECCIJsWPRPUtVp8Ti+yCpL54VaLQ397e/rwAFP5KrgogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPneXXpQr68/Xp7Wbve7XgLcwQQEZExAcPcUbNJ8FAF616+f/708iX/+/btewra23sdfLEafIUBs3SdP/fYvl6GPESA26kJ3rg6/pyPn4d2U6C4CxBad1ueuHffxFy9i9Pr6Q4NuJ0BsyyfTc2r/8uMM2ZTdToDYbn0e+HeGsxnSoKsEiC3W54v+xLnIkAZd5YOIbMIK9Tn7/k/9wfoVCBDzW7M+p1fRoAtswZjc4f5f+ZOlx9sxe7H3mICYWVWfg8N1zUFnCRDTGu2eH209IxAg5rT+uc97nAddIEBMboR/Kh5hDWMSICaUH/2cchh0lgAxm/Hv8PFXuBoBYlrjjD9jrmcEAgRkBIipDHj6c8xJ0IIAARkBYh6Djz97hqBjAgRkBAjICBCQESAm8RQHQHuOgQ4ECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAmcXj06P5hpCM7rHC3+celChCQESAgI0BARoCYx1McAzkAOiZAQEaAmMrgQ5DxZ0GAgIwAMa3RhqDR1jMCAWI24+9uxl/hagSICQ14EuT05ywBYnIjNGiENYxJgJjT8aDR3v/HVzf+LAgQ0xrtbh9tPSMQIGaWHwY5+rnse70A+Fq73e/9A0j3LVjtual2XrcwATG/9c+D1OdGAsQmrNkg9bmdLRib24t93XZskTb1uUqA2GiDHpuh06lKfW4hQGzLvgsPzJD0fIYAsUWLUWjRkasxunCKpD53+fb29ue+V8xu8Xv5XPz2Vz9x3/mPMQGxdaebsg+8nI8RIFh25GqMROdRbMGAjA8iAhkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMBL5X9LSQRvflK4ngAAAABJRU5ErkJggg==', 'sha256': '801305175c39ff4a692a107beec3edf2f470eab84ab16ac2d4d77584e3faeb0d'}, 'data/binding-v1/images/binding-evaluation-0029-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE+UlEQVR4nO3dPW4bRxiAYTmw76PCQKpcw4fIgXKIXCNVABc8jAsBblwQIAiS4o9I7zucfZ5ayx1I2hffDAjsp7e3Hy8AhT+SuwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZD53tx7U6+vXl6e12XyvlwA3MAEBGRMQ3DwFmzQfRYDe9eXPf1+exM//v9VLWNfWe/+HxegeAsTa3Xnqt71chj5GgFipM93575+f56/96+8v732aEt1EgFij4/pcjM57P3wQo9fXrxp0PQFiXe5Mz7Ht5fsZsim7ngCx3vrcmZ6LGdKgiwSINdbngek5kyENusgXEVmFBepz8vOf+ov1CxAg5rdkfY7vokFn2IIxud3zv0x6Tm7H7MXeYwJiZlV9dnb3NQedJEBMa7RnfrT1jECAmNPy5z7vcR50hgAxubY+46xhTALEhPKjn2MOg04SIGYz/hM+/goXI0BMa5zxZ8z1jECAgIwAMZUBT3/2OQk6IEBARoCYx+Djz5YhaJ8AARkBAjICBGQEiEk8xQHQlmOgHQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQEiEnsXj26fRnpyHYr3Kz+dakCBGQECMgIEJARIObxFMdADoD2CRCQESCmMvgQZPw5IEBARoCY1mhD0GjrGYEAMZvxdzfjr3AxAsSEBjwJcvpzkgAxuREaNMIaxiRAzGl/0Gif//27G38OCBDTGu1pH209IxAgZpYfBjn6Oe9zvQD4vTab79sXkG5bsNh7U+28rmECYn7Lnwepz5UEiFVYskHqcz1bMFa3F/t927GDtKnPRQLEShv02AwdT1Xqcw0BYl22XXhghqTnHgLEGh2MQgcduRijM6dI6nOTT29vP267YnYH/5fPxX9/9Rf3m/8YExBrd7wp+8DlfIwAwWFHLsZIdB7FFgzI+CIikBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBLxUfgFWwARvJU4RbQAAAABJRU5ErkJggg==', 'sha256': '465d4384bf007b990937e7230388331e5fbbf030f1e2c9f063504bd6dcd65e26'}, 'data/binding-v1/images/binding-evaluation-0030-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFR0lEQVR4nO3dMW7bSBiAUWURH2cBFTmGb7PnyG1yjBQGcpUUBtykEFYreG1GkWR9nOF7tU0OC374OSCoT8/PP3cAhb+SswIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyn3cbsN9/2U3t6el7vQS4hAkIyAgQkNnEI9jRyz9/7yby8PVHvQS4igkIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZbX2OA678bKaPT96WALFdF3yr9/RfxOh6AsTm3Oob4cfjKNHFBIgNWU7Pt4eH5X9/fHlZOKwMXUCA2G56fluc5b9/1SMZuoAAMbmbpOdNh4PI0DUEiA3V5ybdeeV4zNMS7fdfNOgcAsSc7pCe5YHIKHQOLyIyofvX571zTf+rvFcSIGZzes9/e3i4Z33ePKkGLRAgZq5PuBINOocAMY/11Of/a9CgNwkQk1hbfQ40aJkAMYN11udAgxYIEMNbc30ONOg9AsTY1l+fAw16kwAxiTXXZ5QV3p8AMbDjKDHKvX1cpyHoQIAY1ej38H7w9d+EADG8UcafEVf70QSIIQ338HXKg9iRAAEZAWI8Q48/B4agAwECMgLEYOYbGfbTXdH5BIhRjfv8Ncf6b0KAgIwAMZIJtp9Pfdv8VrQAARkBAjICBGQEiGHMvVGyn/rq3iNAjGeOHej5ruUCAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQ43l8ednN4nGia7mAADGMp6fvu3k9TX117xEgICNAQEaAgIwAMeRGyRx7t4//XsU2N4AECCgJEKMafQh6HHz9NyFADGa+p5Wn6a7ofAIEZASI8UywFW37+eDzbksevv6olwD8xwTEkIYegow/RwLE8MZq0Fir/Wifnp9/fvhJ4IN/TXSgn/cz/pwyATGw4R7E1OcVAWIS62/Q+ld4fwLE2E5HiTXf4adrM/4cCRDDW3+D1Oc9AsQM1twg9VkgQExinQ1Sn2UCxDzW1iD1+S0BYirraZD6nMOLiMz8gmLymuKr8KnPAhMQE3p1z99zFFKfP2ICYmb3HIWk5wICxLYadPMSvTleqc+ZBIhN+IgMSc/1BIitZ+j8Hi3vJUnPBQSIzVnO0AWk52ICxHZdWSLduZ4AwR/ESHRuS4CAjBcRgYwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAG7yi9txjGY68YzZwAAAABJRU5ErkJggg==', 'sha256': '4d69d47efe04788a2e4793fda35cacb0fb3218701453a594488ed1f613e6d3bb'}, 'data/binding-v1/images/binding-evaluation-0030-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVUlEQVR4nO3dQW4aSRiAUTKK78MiW98gJ8mBcpK5QbZZcJgsLGWTBRqEPAYTwHxd1e+t7aZacn/+uwTNp5eXXxuAwj/JqwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIynzcrsN1+2Uxtt/tZLwGuYQICMgIEZFZxC3bw79PTZiJff/+ulwA3MQEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICCzrsdxwI2PzfTwyfsSINbrimf1Hv+KGN1OgFidez0j/HAcJbqaALEi59Pz4/s7T5h8/vZ05rAydAUBYr3pebc453/+VY9k6AoCxOTukp437Q8iQ7cQIFZUn7t055XDMY9LtN1+0aBLCBBzekB6zg9ERqFLeCMiE3p8fU691vTfynsjExCzOb7mH5meM6OQOegUExBTyevz5qubg04RIOaxnPr8fw0a9CYBYhJLq8+eBp0nQMxgmfXZ06AzBIjhLbk+exp0igAxtuXXZ0+D3iRATGLJ9RllhY8nQAzsMEqMcm0f1mkI2hMgRjX6NbwdfP13IUAMb5TxZ8TVfjQBYkjD3XwdcyN2IEBARoAYz9Djz54haE+AgIwAMZj5RobtdGd0OQFiVOPef82x/rsQICAjQIxkgu3nYz9WvxUtQEBGgICMAAEZAWIYc2+UbKc+u1MEiPHMsQM937lcQYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBYjzP3542s3ie6FyuIEAMY7f7uZnXbuqzO0WAgIwAARkBAjICxJAbJXPs3T7/dxbr3AASIKAkQIxq9CHoefD134UAMZj57lZ2053R5QQIyAgQ45lgK9r2897nzZp8/b3qbyCApTEBMaShhyDjz4EAMbyxGjTWaj/aKm7B/J+Z0m73c+hvE935szQBMbThbsTcfL0iQExi+Q1a/gofT4AY2/EoseQr/Hhtxp8DAWJ4y2+Q+pwiQMxgyQ1SnzMEiEkss0Hqc54AMY+lNUh93iVATGU5DVKfS3x6efl10Q/COF69QfHH94d+BvBV+NTnDBMQE3p1zT9yFFKfv2ICYmaPHIWk5woCxOTe/LzYHUv05nilPhcSIFbhIzIkPbcTIFbk/Kfn3+3R+b0k6bmCALE6d3+Ih/RcTYBYrxtLpDu3EyD4ixiJzn0JEJDxRkQgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwKbyB8wMPI4J5OKTAAAAAElFTkSuQmCC', 'sha256': 'fbee7581b9227b97b7304da3c13e35dca72d5176facfc0009dee788322a19aa9'}, 'data/binding-v1/images/binding-evaluation-0030-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFV0lEQVR4nO3dPY4aWRhAUTxyL2ckAqe9IaezDqfekFMHSN6Kg5acOEDDoJ4GY8Dceq/Oifl5JXXd/uoJincvL983AIW/kncFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZN5vVmC7/bCZ2m73tV4CXMMEBGQECMis4hLs4Mc/f28m8vTpW70EuIkJCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmXXdjgNuvG2mm0/elwCxXlfcq/f4KWJ0OwFide51j/DD6yjR1QSIFTmfni+ff5x/+vPHpzMvK0NXECDWm55fFuf841/1SIauIEBM7i7pedP+RWToFgLEiupzl+68cnjN4xJttx806BICxJwekJ7zA5FR6BI+iMiEHl+fU+81/a/y3sgExGyOz/lHpufMKGQOOsUExFTy+rz57uagUwSIeSynPv9fgwa9SYCYxNLqs6dB5wkQM1hmffY06AwBYnhLrs+eBp0iQIxt+fXZ06A3CRCTWHJ9Rlnh4wkQAzuMEqOc24d1GoL2BIhRjX4Obwdf/10IEMMbZfwZcbV/mgAxpOEuvo65EDsQICAjQIxn6PFnzxC0J0BARoAYzHwjw3a6I7qcADGqca+/5lj/XQgQkBEgRjLB9vOxL6vfihYgICNAQEaAgIwAMYy5N0q2Ux/dKQLEeObYgZ7vWK4gQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAAxnuePT5tZPE90LFcQIIax233dzGs39dGdIkBARoCAjAABGQFiyI2SOfZun/89inVuAAkQUBIgRjX6EPQ8+PrvQoAYzHxXK7vpjuhyAgRkBIjxTLAVbft57/1mTZ4+fauXAPzHBMSQhh6CjD8HAsTwxmrQWKv901ZxCeb/zJR2u69D/5rozp+lCYihDXch5uLrFQFiEstv0PJX+HgCxNiOR4kln+HHazP+HAgQw1t+g9TnFAFiBktukPqcIUBMYpkNUp/zBIh5LK1B6vNLAsRUltMg9bnEu5eX7xc9EMbx6gOKXz7/eOS7vwqf+pxhAmJCr875R45C6vNbTEDM7JGjkPRcQYCY3JvfF7tjid4cr9TnQgLEKvyJDEnP7QSIFTn/7flf9uj8XpL0XEGAWJ2738RDeq4mQKzXjSXSndsJEPxGjETnvgQIyPggIpARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESBgU/kJ1447sc6ly5sAAAAASUVORK5CYII=', 'sha256': '7293ead85c2fe926ce04c382b153dfab7b52a65dec1ec8b4a7a9b255f0b84067'}, 'data/binding-v1/images/binding-evaluation-0031-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEUlEQVR4nO3dPW4bVxSAUTqI9sMigKtsw4vIgrIIbyNVABdcjAsB06QgrMixTFFUyG+G95xKBX9eQX6473Eofnh8/LoDKPySPCuAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyPzaPfVa7Pe/7e7C4fClXgK8jQkIyAgQkLEF+9dffy67Dfr9j4d6CXAhExCQESAgI0BARoCAjAABGQECMgIEZFwHBNf6Eo8vx7xKgOBa3xx8fmMxepEAwS2+sfz0CEr0nADBhel5+Pj59H2Xvz/97AFl6EiA4A3peTU6P7vxf2IkQ0c+BYOz6vPw8fOb6nPO3ff38r+oLmYCglNReE90fvT0aE8D0X72KGQCghvV58Qj76eOQgIEL1TgnRuuczx8/xQzGyRA8EJ9bva8D7MbJECQ1edocoMEiOna+gxvkAAx2hrqM7lBAsRc66nP2AYJEKyiPmtbyW0IEEM9jRhre88/fFvPhCFIgJhoK+/t/UbWeTEBYrS1jT9rXtU1CBDjrHbzNXAjJkBARoCYZYsDxX6Daz6TADHUmvdfW1nh+wkQkBEgBtnE8fOoo2gBAjICBGQECMgIEFNs7gBowjGQAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgLEFIfDl+Mfy9+fdtuxfFvt0/rviQABGQECMgIEZASIQTZ3DLTc9QGQAAElAWKo9Q9By+pX+H4CxCxb3MscNrjmMwkQkBEgxtnEUfRy78fPRwLEaOts0LLKVV2DADHRVsaKw0bWeTEBYqjVbsSWGZuvIwGCFTVoWc1KbkOAmOv5iLGGd/7ybA0Txp/dbvfh8fHrbra7+cHJIS/Zq74Awh9NXebVxwQEq5iDlpH1ESDoG7RMrY8tGMR7sWVwfQQITh0IXjVDy/ej1sD62ILB7kQFrrcdU58jExCc9dno/zIN/Vi0w9T0HAkQvOH6jIsz9OIwdZhdHwGCyy8TezVGJ3Zw0nMkQHDTq1Wl5zkBgluUSHdeJEBwrRiJzqsECMi4DgjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBd5R+OLQT5Hq5/vwAAAABJRU5ErkJggg==', 'sha256': 'c7b5fa0157443a52446e14a864d865916681aabfd2836deff306bc2a65290546'}, 'data/binding-v1/images/binding-evaluation-0031-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEklEQVR4nO3dMW4bVxRAUTqI9sMiQKrZRhbhBWUR2YarAC64GBcC3KQgrDCSTElUqDvDd06lQiR/QV68/zUjfrq//7YDKPySvCqAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyPzavfRa7Pe/7W7C4fC1XgK8jQkIyAgQkLEF+9fd73/tNuj733/US4ALmYCAjAABGQECMgIEZAQIyAgQkBEgIOM6ILjWTTxujnmRAMG17hw8/WUxepYAwUfcsfzwDEp0SoDgwvR8+fP7+ccun+9+9oQydCRA8Ib0vBidn/3yoxjJ0JEAwavq86b0PHV8+NMMHWY3SIDgXHre2Z1HHp7toUT72aOQ64Dgg+pz5pn3t/JvOd/KBATPVOB66Xn0EsvJKDRwDjIBQVCfZ19rP28OEiDI6nM0uUECxHRtfYY3SIAYbQ31mdwgAWKu9dRnbIMECFZRn7Wt5GMIEEM9jBhr+8x/+bGeCUOQADHRVj7b+42s82ICxGhrG3/WvKprECDGWe3ma+BGTICAjAAxyxYHiv0G1/xKAsRQa95/bWWF7ydAQEaAGGQTx8+jjqIFCMgIEJARICAjQEyxuQOgCcdAAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQUxwOX48/LJ/vdtux/Fjtw/pviQABGQECMgIEZASIQTZ3DLTc9AGQAAElAWKo9Q9By+pX+H4CxCxb3MscNrjmVxIgICNAjLOJo+jl1o+fjwSI0dbZoGWVq7oGAWKirYwVh42s82ICxFCr3YgtMzZfRwIEK2rQspqVfAwBYq7TEWMNn/zlZA0Txp/dbvfp/v7bbrab+cLJIW/Zq74Bwi9NXebVxwQEq5iDlpH1ESDoG7RMrY8tGMR7sWVwfQQIzh0IXjVDy39HrYH1sQWD3ZkKXG87pj5HJiB41d9G/5dp6GnRDlPTcyRA8IbrMy7O0LPD1GF2fQQILr9M7MUYndnBSc+RAMGHXq0qPacECD6iRLrzLAGCa8VIdF4kQEDGdUBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgTsKv8ArTkV4eBQQ/0AAAAASUVORK5CYII=', 'sha256': 'c7a91c226d951a87c16398cedc9e5dfd3716584d75f014e64f1b828b905a0bb8'}, 'data/binding-v1/images/binding-evaluation-0031-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFB0lEQVR4nO3dPXIaWRSAUTw13g/BpL0hL2g25NQBi3Ggqkkm6LIGjyn9IMPXzT0nUiDBC+Cr+5664dPT0/cDQOGP5FkBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQObP7qm34nj86/AQTqdv9RLgfUxAQEaAgIwt2H++/v3PYYeWL5/rJcCVTEBARoCAjAABGQECMgIEZAQIyAgQkHEdENzqJh43x7xKgOBWdw6e/7IYXSRAcI87lp8fQYnOCRBcmZ5X7925eJfM+oAytBIgeEd63nXD4Pkv/y9GMrQSIHhTfT54r/L6579m6DS7QQIEL6Xn935GwvOjPZfoOHsUch0Q3Kk+Lzzy8VE+lvO9TEBwoQJ3+HCorz9vymZux0xAENTn4nMd581BAgRZfVaTGyRATNfWZ3iDBIjRtlCfyQ0SIObaTn3GNkiAYBP12dpK7kOAGOp5xNjae/7rj/VMGIIEiIn28t4+7mSdVxMgRtva+LPlVd2CADHOZjdfAzdiAgRkBIhZ9jhQHHe45jcSIIba8v5rLyv8OAECMgLEILs4fh51FC1AQEaAgIwAARkBYordHQBNOAYSICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCY4nT6tv6wfPl82I/lx2qf1/9IBAjICBCQESAgI0AMsrtjoOWhD4AECCgJEENtfwhaNr/CjxMgZtnjXua0wzW/kQABGQFinF0cRS+Pfvy8EiBG22aDlk2u6hYEiIn2MlacdrLOqwkQQ212I7bM2HytBAg21KBlMyu5DwFirvMRYwvv/OVsDRPGn8Ph8Onp6fthtof5wskhL9mbvgDCL01d5tXHBASbmIOWkfURIOgbtEytjy0YxHuxZXB9BAheOhC8aYaWn0etgfWxBYPDCxW43XZMfVYmIHjT/0Z/yzT0a9FOU9OzEiB4x/UZV2fo4jB1ml0fAYLrLxN7NUYv7OCkZyVAcNerVaXnnADBPUqkOxcJENwqRqLzKgECMq4DAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEg4FD5F7LqFeHmUMRcAAAAAElFTkSuQmCC', 'sha256': 'b2abbb4a776655caf8d8e2f5f8222f9842e96bc7c1afc1dcadcc321f3ceb3e85'}, 'data/binding-v1/images/binding-evaluation-0032-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFYklEQVR4nO3dTU4bSRiAYWcUH2ckL3KMvs2cI7fhGFlYmqtkgZTNLNBYTgw2OIS3q+t5lvz1h0S/VHUh/Onx8fsOoPBXclUAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZnP3aVhOw6HL9c/4Hj89lGzjOTT4+P3egbYYHGu06MnAgQfmp5zx+kzJEDwDt152O+vf/ry48eV9x5nLZEAwZ3puRmdt8boOF+GBAjeVp+7u/OaEh0na5AAQZCec8vEGRIguF2fP5SeKxmapEECBNfq8wHpeSlDMzTIX0LDWuqz+/mK73jev1oCBGupz4QNsgWDX+/2Kj0vbcc2vBcToBdt4JfPhn9wN1+fSRpkCwYrrc/ubJ4N/Dp8lgAxu1Hu7cMgc76Jf8dx249//t4NZf/133qEYazhqfN1D/v9aSN2OHzZ2F7MCgjWW5/1z/abBIh5rfbRzzwPgwSISY17Jx+GnfySADG79S9/xprzTQSIGY2+iDgMPv+JADG1sZYVD0NN+xoCBGQEiOkMdPi1+eMwAQIyAsRctrFw2Mz3IkBMasT91+iTXxIgICNAQEaAgIwAMZGhD+A3eRgvQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAAxkdPLip5ea3RQy//zj/5CqQIEZAQIyAgQkBEgJjXuY6Bl2MkvCRBzGf2p7ca+FwECMgLEdIY+jF+2cgD/RICAjAAxtbEWQctQ076GADGj0fcvx8HnPxEgZjfKsmIZZM43ESAmNe4i4jjs5JcEiHkNdBy2bOvw60SAYO0NWlY8228SIKZ2vqBY532+nE21seWPAMEwd/VxkDnfRIBgvQ+Dlo0++jkRIFhpg5at10eA4PkGtRlazgbYcH0ECFb3THrZ9FPnXwgQrKhBy0z1ESC43aCPydDy84VmqI8AwfN+uf//aIaWiy8+SX12u93negBYqacKHA5fTm95ysTDfv9el1guojZPep4IEFxzPH47b9B5Ne4u0UuLqeNk9REguGcpdNmRmzG6voM7zpeeJwIEr3JqxGWJ7j4vO87anRMBgvdZEN3xRRAgeIeC3OyR4jxLgOAd6Mt9/B0QkBEgIGMLdtv+67/1CLBNVkBARoCAzKfHx+/d1YGpWQEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBd5T9hqlXThOd26QAAAABJRU5ErkJggg==', 'sha256': '5bf159fb2b269293e057955f81cc74f1a1f033b4ab1fe0db39c0a684250c920f'}, 'data/binding-v1/images/binding-evaluation-0032-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFWElEQVR4nO3dS24aWRiAUdJKLScSgywju8k6eje9jAyQeisZWGKSgdWINDEY4virW/ecaR78llyf7wOZD09P33cAhb+SVwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnYvTRsx37/+fpfOBy+vdcsI/nw9PS9ngE2WJzr9OiZAMG7pufcYfoMCRC8QXeOXz9d/+fL3/9e+dPDrCUSIHgwPTejc2+MDvNlSIDgvvo83J3XlOgwWYMECIL0nFsmzpAAwe36/KH0XMnQJA0SILhWn3dIz0sZmqFB3gkNa6nP7udXfMP7/tUSIFhLfSZskC0Y/P9pr9Lz0nZsw3sxAXrRBn74bPgbd/P1maRBtmCw0vrszubZwI/DXxIgZjfKs70fZM67+HUct/2zLLuhfDke6xGGsYZT5+uOXz+dNmL7/eeN7cWsgGC99Vn/bL9JgJjXao9+5jkMEiAmNe6TvB928ksCxOzWv/wZa867CBAzGn0RsR98/hMBYmpjLSuOQ037GgIEZASI6Qx0+bX56zABAjICxFy2sXDYzNciQExqxP3X6JNfEiAgI0BARoCAjAAxkaEv4Dd5GS9AQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMADGR08eKnj5rdFDLf/OP/kGpAgRkBAjICBCQESAmNe4x0DLs5JcEiLmMfmq7sa9FgICMADGdoS/jl61cwD8TICAjQExtrEXQMtS0ryFAzGj0/cth8PlPBIjZjbKsWAaZ8y4CxKTGXUQchp38kgAxr4Guw5ZtXX6dCBCsvUHLimf7TQLE1M4XFOt8zpezqTa2/BEgGOapPgwy510ECNZ7GLRs9OjnRIBgpQ1atl6f3W73sR4A1uJw+Lbffz49+cevn6pJlk2f+5yzAoJ1nUkv09RHgGBdDVpmqo8tGFzbi73ndmz5OXYz1EeA4HaD/nSGlot11iT1ESB40XMF/miGlonT80yA4I6l0Hk1Hi7RS0dLh8nqI0DwyFLosiM3Y3T9PPswX3qeCRC8yqkRlyV6+L7sMGt3TgQI3mZB9MB/ggDBGxTkZo8U55cECN6AvjzGO6GBjAABGVuw274cj/UIsE1WQEBGgIDMh6en792rA1OzAgIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAbvKD1LeRH6X7i1SAAAAAElFTkSuQmCC', 'sha256': '709d145fffe8e3aad17e2f1d096a627fe8a0f69499267c6ff8fde482293a592c'}, 'data/binding-v1/images/binding-evaluation-0032-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVUlEQVR4nO3dQVITTRiA4fiX3CcLjzHH5Rgu5jAuqHLzLyhT0UACEXmnp59nqch8lM6bnm4kX56efhwACv8lVwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnaXRr243j8dv0D1vX7Z80yki9PTz/qGWCHxblOj54JEHxqes6t02dIgOADuvP48HD9jy8/f1753XXWEgkQ3Jmem9F5b4zW+TIkQPC++tzdnbeUaJ2sQQIEQXrOLRNnSIDgdn3+UXquZGiSBgkQXKvPJ6TntQzN0CDfCQ1bqc/h9yt+4Hn/ZgkQbKU+EzbIIxj8ebdX6XntcWzHz2IC9KodvPjs+B/u7uszSYM8gsFG63M4m2cHL4cvEiBmN8q9fRxkznfx4zhu29qr4k3X/9sRW9t1vu7x4eH0F3o8ftvZs5gVEGy3Ptuf7S8JEPPa7NbPPJtBAsSkxr2Tj8NOfkmAmN32lz9jzfkuAsSMRl9EHAef/0SAmNpYy4rHoaZ9CwECMgLEdAY6/Nr9cZgAARkBYi77WDjs5msRICY14vPX6JNfEiAgI0BARoCAjAAxkaEP4Hd5GC9AQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMADGR09uKjv7mscuv+Ud/o1QBAjICBGQECMgIEJMadxtoGXbySwLEXEbftd3Z1yJAQEaAmM7Qh/HLXg7gnwkQkBEgpjbWImgZatq3ECBmNPrzyzr4/CcCxOxGWVYsg8z5LgLEpMZdRKzDTn5JgJjXQMdhy74Ov04ECLbeoGXDs/0lAWJq5wuKbd7ny9lUO1v+CBAMc1evg8z5LgIE290MWna69XMiQLDRBi17r48AwcsNajO0nA2w4/oIEGxuT3rZ9a7zHwQINtSgZab6CBDcbtDnZGj5/UIz1EeA4GV/3P//NEPLxSefpD6Hw+FrPQBs1HMFjsdvp195zsTjw8NHXWK5iNo86XkmQHDNun4/b9B5Ne4u0WuLqXWy+ggQ3LMUuuzIzRhdf4Jb50vPMwGCNzk14rJEd5+XrbN250SA4GMWRHd8EgQIPqAgN3ukOC8SIPgA+nIf3wcEZAQIyHgEu207P58BdsYKCMgIEJD58vT0o7s6MDUrICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQcKj8D5ZTVrCzgiZPAAAAAElFTkSuQmCC', 'sha256': '3aeec1ec2462f9bf7b7a4bae35cd2a412f0e6339a1800519e36c8859412910b8'}, 'data/binding-v1/images/binding-evaluation-0033-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPklEQVR4nO3aPY4aWRSAUTwy+yGw1JG34UV4QbMIb2MiSw5YjIOWSCZARgiabvcPfK9450QdNFU3qU+3Hnx6fPy9Aij8k9wVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQ+rxZrs/myulPb7a96BLgFGxCQWfAGBAPu3bbX6QL037+71V34+n1dj8B7X/aP/1mMpggQjHnIeLiCEl0iQPCK9Kwffjz/2d3Pb5cuKEPnBAheSM+L0bn0zycxkqFzvgWDi/VZP/x4VX3+5uN3/PORNxAgeKIL70zP85fSoAOvYMzuPD3XuMv+soeXMq9jezYgpnab+ly6/mb6VUiAmNfx8/+B71zPO7nRZu4GCRCTOqnPje+uQXsCxIza+pzfdzNrgwSIqVX1GeHuIxAgpnNYN0Z4/td/ZphzCRIg5jLyc74ZeLYrESAmMsLRz7n1xIdBAsSMxqnPmPPcjAAxi6UsF5uFzPkhBIjpjLlurIec6toEiCksa63YLGra9xAg5jLyorEeeLYrESAgI0Dcv6F+efi89WS/SxQgICNAQEaAgIwAcecWdAA04TGQAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgLEndtuf+3/2P38tlqC3Z85D5PfMQECMgIEZAQIyAgQ929Bx0C7mQ6ABAgoCRBzGXkJ2g0825UIEFNY1hvNdlHTvocAMZ0xF43dkFNdmwAxi6WsFduFzPkhBIgZjbZu7Aab52YEiIkcLxfjPPO7o0mmWn8EiOmM/IRvB57tSgSI6Qz1u8TdZL88PCFATK1t0G6AArYEiBmNcBi0m/jo50CAmFTbIPXZEyDmddKg22To5EbbiesjQMzu5Pm/doNOrr+duz6r1epzPQDE9hXYbL4cN2L98ONj7yI9T7IBwdOr0EdtQ+eXUp8DAYKLXXhnhp78uPoc8woGF1/H9o4j8uKr2TPBkp5zAgR/laG9ty1E0nOJAMEL1XiyRK+6ApcIELyiIy/GSHReRYDgFfTlY/kWDMgIEJARICAjQEDmHg6hv35f1yMAb2EDAjICBGQ+PT7+7u4OTM0GBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAlaV/wG47wwkzmiRVQAAAABJRU5ErkJggg==', 'sha256': 'e15db51b080726baac6e68409680fee3be09d21db9a253cddf8922ef71a26b24'}, 'data/binding-v1/images/binding-evaluation-0033-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKklEQVR4nO3dPW7bWBSAUWcQ70fFtNpBVjILmpXMDtSm0GJSGHAzhRHB0I8VQaa/R71zake8DT9cPjDgt5eXX08Ahb+SqwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzPen1dps/n56UPv9z3oE+Ao2ICCz4g0IBty7ba/TBei/5+enh/Dj9bUegXsf9t//sRhNESAY85Dx8AtKdIkAwQ3p2f17ZUvd/vN86Qdl6JQAwZX0XI3OpT8+ipEMnRIguFifm9Jz6u2fn2ZIgw4ECM7U5870fJwhDToQIGa3XHo+yJDHsTdeRGRqX1OfS7+/edy3+f+QDYh5vb//l07PB6vQfuI9yAbEpJL6nL3iZuI9SICYUVuf0+tuZm2QADG1qj4jXH0EAsR0DuvGCPf/7vcMcy5BAsRcRr7PNwPPthABYiIjHP2c2k18GCRAzGic+ow5z5cRIGaxluVis5I5P4UAMZ0x143dkFMtTYCYwrrWis2qpr2HADGXkReN3cCzLUSAgIwA8fiGevPwY7vJ3ksUICAjQEBGgICMAPHgVnQANOExkAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICxIM7fHn98Dn2wW1/zznDN+MFCMgIEJARICAjQDy+FR0DbWc6ABIgoCRAzGXkJWg78GwLESCmsK4nmv2qpr2HADGdMReN7ZBTLU2AmMVa1or9Sub8FALEjEZbN7aDzfNlBIiJvF8uxrnnt+8mmWr9ESCmM/Idvh94toUIENMZ6r3E7WRvHh4RIKbWNmg7QAFbAsSMRjgM2k589HMgQEyqbZD6vPleDwCZ/f7n4fvrb0X4gu/HH8VuP3F9bEDM7uj+X3oVUp8jNiBm91aBpVch6TlLgOD4cexzM3S6VanPgQDB+Qbdn6GzD3Tq854AwcXHsdOOXI3RB6dI0nNKgOCPMnTPKbX0XCJAcKUaZ0t00y9wiQDBDR25GiPRuYkAwQ305XN5ERHICBCQESAgI0BA5hEOoX+8Lv4/mIEl2ICAjAABmW8vL7+6qwNTswEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAp8r/0QAMWKjk7RYAAAAASUVORK5CYII=', 'sha256': 'c8f61aac7abfa9b683ec71481d2329dd6b44a138b13e8799703e0f81dc50de69'}, 'data/binding-v1/images/binding-evaluation-0033-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFOElEQVR4nO3aPW4aXRSAYfIp7IciUjaROotM/W0iUgoWk8LSNClQEAJjxz/4vcN9nsqFmTnNvDpz4dPDw+8NQOG/5K4AAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPN5s1q73ZfNndrvf9UjwEewAQGZFW9AMODebXudLkD/b7ebu/BtWeoReOvL/uk/i9EUAYIxDxmPV1CiawQIXpCe7dcfT392+fn92gVl6JIAwTPpeTY61/75LEYydMm3YHC1PtuvP15Un3/5+B3/fOQVBAge6cIb0/P0pTToyCsYs7tMzy3ucrjs8aXM69iBDYipfUx9rl1/N/0qJEDM6/T5f8d3rqed3Wg3d4MEiEmd1eeD765BBwLEjNr6XN53N2uDBIipVfUZ4e4jECCmc1w3Rnj+t39nmHMJEiDmMvJzvht4thsRICYywtHPpe3Eh0ECxIzGqc+Y83wYAWIWa1kudiuZ810IENMZc93YDjnVrQkQU1jXWrFb1bRvIUDMZeRFYzvwbDciQEBGgLh/Q/3y8GnbyX6XKEBARoCAjAABGQHizq3oAGjCYyABAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBIg7t9//Ovyx/Py+WYPl75zHye+YAAEZAQIyAgRkBIj7t6JjoGWmAyABAkoCxFxGXoKWgWe7EQFiCut6o9mvatq3ECCmM+aisQw51a0JELNYy1qxX8mc70KAmNFo68Yy2DwfRoCYyOlyMc4zv5xMMtX6I0BMZ+QnfD/wbDciQExnqN8lLpP98vCMADG1tkHLAAVsCRAzGuEwaJn46OdIgJhU2yD1ORAg5nXWoI/J0NmN9hPXR4CY3dnzf+sGnV1/P3d9NpvN53oAiB0qsNt9OW3E9uuP972L9DzKBgSPr0LvtQ1dXkp9jgQIrnbhjRl69OPqc8orGFx9HTs4jcizr2ZPBEt6LgkQ/FOGDl63EEnPNQIEz1Tj0RK96ApcI0Dwgo48GyPReREBghfQl/flWzAgI0BARoCAjAABmXs4hP62LPUIwGvYgICMAAGZTw8Pv7u7A1OzAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBN5Q+1JA267UfO/wAAAABJRU5ErkJggg==', 'sha256': '0cf3163f78779e2dda5fec14179e513e0e2f3aeb9bb545591a33d4f44836ddba'}, 'data/binding-v1/images/binding-evaluation-0034-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKElEQVR4nO3bMU6bSxSAUfIUlhOJItLbSTaSdWQjbyeRKCxlKymQaF6BZFk2GLAx38z4nIoC+7+NP90ZzJeHh783AIV/kqcCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg8/VmUXd332+Ws9nc1yPAR7IBARkBAjLLHsG2bv/972Zyj79/1CPARdiAgMz6GxCc/ycL1/8XIkBcqXf9nXT3l8XoAwkQ1+X872ds30GJzidAXIsj6Xn8+e34a29//XnpDWXoHALElabn1ei89Mt7MZKhcwgQV1efd6Xn0NPLDzOkQScQIK4lPWd2Z8/23bYlsgqdwPeAWNNF63PknZf8J8TLsQGxoN0KXC49e4/YXYXsQW9kA2I1n1yfZ59lD3ojAWIpVX0On6hBbyFArKOtz+FzNehVAsQiRqjP4dM16DgBYgXj1OdwBg06QoBYygj1GW2SkQkQ09uuGKN95rfzWIJeIkDMbZbP9ixzfjIBYhGjrT8jTzUOAWJiwx6+djmIHSFAQEaAmNWMC8WMM1+UADG9kc9fs0xYESAgI0BMaYrr512uop8lQEBGgICMAAEZAWI+010APXENdEiAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAWI+m8390w+3v/7czGM77XZ+BAjICBCQESAgI0BMabprIBdAzxIgICNATG/8JWj8CSsCxKxmPMvMOPNFCRCQESAmNsVVtOvnIwSIRYzZoDGnGocAMbdZ1opZ5vxkAsT0hj2IOXy9SoBYyjgNGmeSkQkQK9hdMUb45O/OYP05QoBYxDgNUp+3EyDWMUKD1OddBIiltA1Sn/cSIFZTNUh9TvD1lBfB2Dab+7u777tdePz57XKP28uc+rydDYg17VXgcquQ+pzDBsSynlqwtwp91DZ0WDTpOYEAcUXHsQ85lD27TKnPaQSIq1uFDjvyaoyOnOCk5xwCxFVn6JwbIuk53/oBevz9ox6BgWyr8WyJ3vUOnG/9AMGrHXk1RqJzIQIE+pL58vDwt3s6cNV8ERHICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwE3lf7VYEuVrflBvAAAAAElFTkSuQmCC', 'sha256': '922daa686771c54aa0dd9d4c24164013d043ec62ec9cfca259c03298f7066747'}, 'data/binding-v1/images/binding-evaluation-0034-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3aMW4iSxRAUc/XsB8Hs4xZrpcxgRczgSUnP0BCCBgwxvhWFedEBNBdCVevqvvH29vfJ4DCf8ldAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPn5tKjn519Py3l9/VMvAb6SCQjICBCQWXYLtvOy2TxN7vf7e70EuAsTEJBZfwKC2x9ZOP6/EwHiQV31nHT/y2L0hQSIx3L7+xm7KyjR7QSIR3EmPRefVJx8DrC9oAzdQoB40PRc9Xh0/8sHMZKhW3gKxsPV52WzueXljJM/X/LN+29gAmJZB1H42jfCdlfbDURGoU8wAbGmu9bnzJWNQlcRIBa0X4EbN1wfcXALDfo4AWLx+nzbfTXoEwSIpVT1Ob6jBn2EALGOtj7H99WgiwSIRYxQn+O7a9B5AsQKxqnP8Ro06AwBYikj1Ge0lYxMgJjebsQY7T+/W48h6F8EiLnN8t+eZZ3fTIBYxGjjz8irGocAMbFhN1/7bMTOECAgI0DMasaBYsY135UAMb2R91+zrLAiQEBGgJjSFMfP+xxFnyRAQEaAgIwAARkBYj7THQBtOQY6JkBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAMZ/X1z/bD7/f35/msVvtbv0IEJARICAjQEBGgJjSdMdADoBOEiAgI0BMb/whaPwVVgSIWc24l5lxzXclQEBGgJjYFEfRjp/PECAWMWaDxlzVOASIuc0yVsyyzm8mQExv2I2YzddFAsRSxmnQOCsZmQCxgv0RY4R//v4ajD9nCBCLGKdB6vNxAsQ6RmiQ+lxFgFhK2yD1uZYAsZqqQerzCQLE+g26d4YObqE+HydArOmgAvdr0MGV1ecqP6/7Osxj24Ln518HpXjZbG6/+HHRpOcTTEAs7rgLN27KTv5cfT7HBMTDjUJb+xG5OBOdCZb03EKAeOgMbX1uIJKe260foBFezGccu2qcLNFVV+B26wcILnbkYoxE504ECPQl8+Pt7W93d+CheQwPZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIOCp8j9YJhPikgzvIAAAAABJRU5ErkJggg==', 'sha256': '52057e837b1b0d9687a28f7655269c70e41aae866c79751ff0e601cdc117d743'}, 'data/binding-v1/images/binding-evaluation-0034-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJUlEQVR4nO3bMU6bSxSAUecpLCcSRZaR3WQd2c1bRgqkbCUFkpsUSJZlgw0Y882Mz6kosP/b+NOdwXx5fPy7ASj8lzwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABma+bRd3ff98s5+Hhdz0CfCQbEJARICCz7BFs5/+7u83kfmy39QhwFTYgILP+BgSX/8nC9f+VCBA36k1/J93/ZTH6QALEbbn8+xm7d1CiywkQt+JEerY/v51+7d2vPy+9oQxdQoC40fScjc5Lv3wQIxm6hABxc/V5U3qOPb38OEMa9A4CxK2k58LuHNi9265EVqF38D0g1nTV+px45yX/CfF6bEAsaL8C10vPwSP2VyF70CvZgFjNJ9fn2WfZg15JgFhKVZ/jJ2rQawgQ62jrc/xcDTpLgFjECPU5froGnSZArGCc+hzPoEEnCBBLGaE+o00yMgFiersVY7TP/G4eS9BLBIi5zfLZnmXOTyZALGK09WfkqcYhQExs2MPXPgexEwQIyAgQs5pxoZhx5qsSIKY38vlrlgkrAgRkBIgpTXH9vM9V9LMECMgIEJARICAjQMxnugugJ66BjgkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMzn4eH30w93v/5s5rGbdjc/AgRkBAjICBCQESCmNN01kAugZwkQkBEgpjf+EjT+hBUBYlYznmVmnPmqBAjICBATm+Iq2vXzCQLEIsZs0JhTjUOAmNssa8Usc34yAWJ6wx7EHL7OEiCWMk6DxplkZALECvZXjBE++fszWH9OECAWMU6D1Of1BIh1jNAg9XkTAWIpbYPU560EiNVUDVKfd/j6nhfB2B4eft/ff9/vwvbnt+s97iBz6vN6NiDWdFCB661C6nMJGxDLemrBwSr0UdvQcdGk5x0EiBs6jn3IoezZZUp93keAuLlV6LgjZ2N04gQnPZcQIG46Q5fcEEnP5dYP0I/tth6Bgeyq8WyJ3vQOXG79AMHZjpyNkehciQCBvmS+PD7+7Z4O3DRfRAQyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKn8A2N2Ew0xtAHuAAAAAElFTkSuQmCC', 'sha256': 'eb2392049298ca0e4cf906bca2b507c055bc64bb88eadb0038cbf9d17b843f39'}, 'data/binding-v1/images/binding-evaluation-0035-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJElEQVR4nO3aO24bSRRAUXpgLWcABkq5Iadex6TekFIFBGYrDgQocSCYQzRbpEcf3mbXOREBUezSpy9eFfnl6ennBqDwV3JVAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZL52l16i7fZ+szr7/WO9BJhnAgIyAgRkbMHmPX//e3Pj7v75t14CXGACAjImIPiAdyec9L+NAMEHvCV6/GQx+nMCBB/8UYzDKyjRRQIEl9Pz8OP5/Pfuvt299oIydIYAwXx6LkbntSdPYiRDZwgQQztNz//qzqzDKxyXSIZmCRDjmtTn/emZeHnBSYY06JgAMajj+nx4es5kSIOO+SAiw9lu769Wn9mrTBYwMgFiLJ+97Tpjcq2tBtmCMZTrDz4TtmMTJiBGkddn9urbsecgAWI4bX2Ws4YlECCGcBg0lnPnP/xeychDkACxfsu/w7eLX+EnESBWbjlHP6cehj8MEiBGsbT6LHlVVyNArNkCj35OPQx8GCRAQEaAWK2bGH8GH4IECMgIEOt0Q+PPyEOQAAEZAQIyAsQK3dz+a9hdmAABGQECMgIEZASItbnRA6Axj4EECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDWZr9/fHmw+3a3uTW732s+/BTrJkBARoCAjAABGQFihW70GGg32AGQAAElAQIyAsQ63dwubDfe/kuAgJIAsVo3NATthhx/BAgoCRBrdhND0G7U8UeAGMgyG7Rb5KquRoBYueOxYml3++5oPQOOPwLEEJZ/b+8Xv8JPIkAMYYGHQbuBj34OBIjhLKFBS1jDEggQo1jOYZCjn4Ov/z2EtdvvH7fb++MKPPx4vuYCJuHbj10fExDDmdzz1xyF1OeUADGc/f7x+tuxybZLfV7YgjGoq23HDD5nCBDjOm7QZ2TodLZSnwkBYmgvRTjN0HtKNLunk55ZX56efs5/ZUjH/4ir4V///X/9izE6c5Dk93+GCQhenYbeeVAtPRcJEMxX483jsO78OVswuOxijETnbQQIyPggIpARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DApvILYssc5pHwowYAAAAASUVORK5CYII=', 'sha256': 'd8ff23331cad88d59dc38770b1d7b5c58ec676ee302904e877e48f325da66ac5'}, 'data/binding-v1/images/binding-evaluation-0035-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJ0lEQVR4nO3bO24bSRRAUXpg7oeBAW1iYi1S8WxCgAMuxoGAThwIJgiyRXr04W12nRNJkNxd+vTFq6L17eXl1wag8E9yVwABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGS+d7deot3ux2Z19vuf9RJgngkIyAgQkLEFm/ffdru5c/9OU70EuMIEBGRMQPAJr0446X8fAYJPeEn0+JPF6O8JEHzyf8U4XEGJrhIguJ6e7cPT5X87PT++dUEZukCAYD49V6Pz1iefxEiGLhAghnaenv/VnVmHKxyXSIZmeRmecZ3UZ/vw9PH6XL7gKv/W5yMEiEEdt+DT03Ps5OIadMwWjOGcDz43uOn24emwI7MdOzABMZakPrP32hmFBIih3Gzb9RbbsRMCxKD1CVeiQQcCxHDa+ixnDUsgQAzhMGgs58nf/lnJyEOQALF+y3/Cd4tf4RcRIFZuOUc/57bDHwYJEKNYWn2WvKqbESDWbIFHP+e2Ax8GCRCQESBW6y7Gn8GHIAECMgLEOt3R+DPyECRAQEaAgIwAsUJ3t/8adhcmQEBGgICMAAEZAWJt7vQAaMxjIAECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQEiLXZ73++vjE9P27uzfRnzYevYt0ECMgIEJARICAjQKzQnR4DTYMdAAkQUBIgICNArNPd7cKm8fZfAgSUBIjVuqMhaBpy/BEgoCRArNldDEHTqOOPADGQZTZoWuSqbkaAWLnjsWJpT/t0tJ4Bxx8BYgjLf7b3i1/hFxEghrDAw6Bp4KOfAwFiOEto0BLWsAQCxCiWcxjk6OdAgBi3QbfP0MlN92PXR4AYzskzf8sGndxrP3x9NpvN93oBcGuvT/5u9+P13en5cfvw9NU3NfjMMgExqJttx2y7LhAgxnW+HfvcDJ1fUH1O2IIxtJPt2PFe6d37stmKSc+sby8vv+Y/MqTjX8TV8Kv/8Z/+1RhdGJ18/y8wAcGb09DB+7Zm0nOVAMF8Nd49DuvO37MFg+uuxkh03keAgIyX4YGMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECNpXfpYUc3nJbngUAAAAASUVORK5CYII=', 'sha256': 'aaa777c55daf065b8dcbbdc4f43a70461ba39caec4ad8fb777578922b2c68dd0'}, 'data/binding-v1/images/binding-evaluation-0035-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIUlEQVR4nO3aPW4iWRhAUXrU3g+BU3YwK+kF9UpmB6QOWEwHlpxMYDWDijL0+Idb1DsnQjKinm3q6nsPvj0//9oAFP5KrgogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM9+7SS7TdPm5W53B4qpcA80xAQEaAgIwt2Lx/Hh42d+7vl5d6CXCFCQjImIDgEz6dcNL/PgIEn/CR6OmTxejPCRB88lcxjq+gRFcJEFxPz/7nlRP93Y+Ht15Qhi4QIJhPz9XovPXkSYxk6AIBYmjn6flf3Zl1fIXTEsnQLAFiXJP6fDw9E68vOMmQBp0SIAZ1Wp9PT8+FDGnQKV9EZDjb7ePN6jN7lckCRiZAjOWrt10XTK611SBbMIZy+8FnwnZswgTEKPL6zF59O/YcJEAMp63PctawBALEEI6DxnLu/P3vlYw8BAkQ67f8O3y7+BV+EQFi5ZZz9HNuP/xhkAAxiqXVZ8mruhkBYs0WePRzbj/wYZAAARkBYrXuYvwZfAgSICAjQKzTHY0/Iw9BAgRkBAjICBArdHf7r2F3YQIEZAQIyAgQkBEg1uZOD4DGPAYSICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoBYm8Ph6fXB7sfD5t7sfq/5+FusmwABGQECMgIEZASIFbrTY6DdYAdAAgSUBAjICBDrdHe7sN14+y8BAkoCxGrd0RC0G3L8ESCgJECs2V0MQbtRxx8BYiDLbNBukau6GQFi5U7HiqXd7buT9Qw4/ggQQ1j+vX1Y/Aq/iAAxhAUeBu0GPvo5EiCGs4QGLWENSyBAjGI5h0GOfo6+//cQ1u5weNpuH08rsP/5cssFTMJ3GLs+JiCGM7nnbzkKqc85AWI4h8PT7bdjk22X+ryyBWNQN9uOGXwuECDGddqgr8jQ+WylPhMCxNBei3CeoY+UaHZPJz2zvj0//5r/yZBO34ir4a3/8f/+1RhdOEjy97/ABARvTkMfPKiWnqsECOar8e5xWHf+nC0YXHc1RqLzPgIEZHwREcgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESBgU/kXumUc5sJe2WIAAAAASUVORK5CYII=', 'sha256': '7273a1a1c1b7477cd7a6c8be270cea6cce54bad1681eb993b4760f8cfbffc2a6'}, 'data/binding-v1/images/binding-evaluation-0036-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFY0lEQVR4nO3cMW7bSBiAUWUR30dF2txAJ9kD7Ul8g21T6DApDLjZQlhDkGzZkSV/nOF7VYAY5C/E/DAcivn29PR7A1D4KzkrgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIy3zcT2W5/bFZgv/9VjwC3YQUEZAQIyEx1C/bi33+eN9P5+fdDPQLcmBUQkBEgICNAQEaAgIwAARkBAjICBGTm/B4Q3Pu1Hi/E3IQAwTUvEp78vB5dR4BYu5u8w3w4iAz9KQFipS535/HhnRdfds/PF46pRB8kQKzOW+l5Nzpv/fB5jCyIPkiAWHt9/qg7r3o5wkmJttsfGnSZALEW90jPqwc8zpCl0GW+B8Qa6/P48HDz+lw4+Er+r84rCBDzO77+75qeYycn0qBXCRDrqs8Xn12DLhMgZtbW5/y8GnRCgJjWy9X+ZbddbzkeQIOOCRDz12ezDBp0ToCY0ALrc6BBJwSI2YxybW8HmfOuBIipLGHX+TJ70scEiDktsz7Ln+2LCRDzWOzWzzmbQQcCxCTGvZK3w07+eQLEbJa//BlrzrsSIGYw+iJiO/j8VxMgpjLWsuJxqGnvQYCAjAAxvIEefp17XPfjMAECMgLE2GZaOGwn+iwfJEBMYsT7r9En/zwBAjICBGQECMgIEAMb+gH8sdU+jBcgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgBjYfv/r8Ifd8/NmZLv/53/5RCshQEBGgICMAAEZAWIS424D7Yad/PMEiLHNtGu7n+izfJAAARkBYnhDP4zfrfUB/IEAARkBYipjLYJ2Q017DwLEDEa/f9kPPv/VBIjZjLKs2A0y510JEJMYdxGxH3byzxMg5jHQ47CVP/x6IUDMackNWvJsX0yAmMrxgmKZ1/nxVPt1L38EiAmNclXvB5nzrgSICS12M8jWzwkBYk4LbJD6nBMgVtGgNkPHA6jPMQFiZkvYk7brfIEAMbm2Qepz2benp9+bWWy3PzYr4Pf4878bjw8P9z7jSez8q73KCohVOLn+77ordH5w9XnL9zf/BuZyqMDxUuiQiRuuhs6jJj0rugWDz9yqX12itxZT6vMuAWKlLu8Yvhujy3dw0vNBAsTa3fDZhe78KQGCG2RIeq4jQHBNjxTnJgQIyPgeEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCwqfwHAp0mvFDXZiQAAAAASUVORK5CYII=', 'sha256': '249c7d6101c622cdb8354ab06fe90a34bf4b75834394d482412f4fb912021cee'}, 'data/binding-v1/images/binding-evaluation-0036-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFYklEQVR4nO3cQU4bSRiAUWeU3MeLSFn5GnOIHCiHmGuwGomFD5MFUjazsMZCYAwYm6+r+r1VhIj9W9Cfqqtsvjw8/N4AFP5KnhVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZmvm4lst983K7Df39cjwHVYAQEZAQIyU92CHX378c9mOn/+/bseAa7MCgjICBCQESAgI0BARoCAjAABGQECMnO+Dwhu/bEeH4i5CgGCSz5I+OT79egyAsTaXeUzzIcHkaH3EiBW6nx37n79Of/fdz+/nXlMJXojAWJ1XkrPq9F56Zufx8iC6I0EiLXX513dOen4CE9KtN1+16DzBIi1uEV6Tj7g4wxZCp0nQKyxPldPz6sZ0qCTBIh11eem6TmTIQ06yTuhmVxSn5PPuJK/Wf4uAsTM2vo8f14NesItGNM6Xu1Vek7ejrkXe8wKiDktpz5Hx0msg44EiAktsD4HGvSEADGbUa7t7SBz3pQAMZUl7DqfZ0/6MQFiTsusz/Jn+2QCxDwWu/XznM2gAwFiEuNeydthJ/84AWI2y1/+jDXnTQkQMxh9EbEdfP6LCRBTGWtZcTfUtLcgQEBGgBjeQIdfz92t+zhMgICMADG2mRYO24leyxsJEJMY8f5r9Mk/ToCAjAABGQECMgLEwIY+gH9stYfxAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQA9vv7w//2P38thnZ7v/5j69oJQQIyAgQkBEgICNATGLcbaDdsJN/nAAxtpl2bfcTvZY3EiAgI0AMb+jD+N1aD+APBAjICBBTGWsRtBtq2lsQIGYw+v3LfvD5LyZAzGaUZcVukDlvSoCYxLiLiP2wk3+cADGPgY7DVn74dSRAzGnJDVrybJ9MgJjK4wXFMq/zx1Pt1738ESAmNMpVvR9kzpsSICa02M0gWz9PCBBzWmCD1Oe5rye+BlPY7++32+/HK//u159qEvs+L7ECYmZL2JNWnzMEiMm1DVKf8748PPzezOKw3p6e3+OP/258wu3Yk9j5qZ1kD4h17Qcd3HRX6Pk6S31eIkCsxaECN82Q9Kz6Fgw+cqt+cYle2lpSn1cJECt1fsfw1Rid38+WnjcSINbuimcXuvNeAgRXyJD0XEaA4JIeKc5VCBCQ8U5oICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIGBT+Q+q/y66OtyFxwAAAABJRU5ErkJggg==', 'sha256': 'ce2078c609d47d62edbc41cb7887035d99e5c8548bdcf20e24c6ac31f7f86087'}, 'data/binding-v1/images/binding-evaluation-0036-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFYklEQVR4nO3cMW4bRxiAUTow78PCQC7BOodMrUsESMHDuBCgJgURgSAlWqZIfTuz71UGJO/+hLQfZmdJfXt+/rkBKPyRnBVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZnvm4nsdj82K3A4/FuPAPdhBQRkBAjITHUL9mr759+b6bz881c9AtyZFRCQESAgI0BARoCAjAABGQECMgIEZOZ8HxA8+mM9PhBzFwIEt3yQ8Oz79eg2AsTa3eUzzMeDyNDvEiBW6np3nrbb6/99//Jy5ZhK9EECxOq8l55fRue9b76MkQXRBwkQa6/Pb3XnTa9HOCvRbvdDg64TINbiEel584CnGbIUus77gFhjfZ6227vX58rBV/K3Om8gQMzv9Pp/aHpOnZ1Ig94kQKyrPl98dg26ToCYWVufy/Nq0BkBYlqvV/uX3Xa953QADTolQMxfn80yaNAlAWJCC6zPkQadESBmM8q1vRtkzocSIKayhF3n6+xJnxIg5rTM+ix/ti8mQMxjsVs/l2wGHQkQkxj3St4NO/nnCRCzWf7yZ6w5H0qAmMHoi4jd4PPfTICYyljLiqehpn0EAQIyAsTwBnr4delp3Y/DBAjICBBjm2nhsJvotXyQADGJEe+/Rp/88wQIyAgQkBEgICNADGzoB/CnVvswXoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBYmCHw7/Hf+xfXjYj2/8//+srWgkBAjICBGQECMgIEJMYdxtoP+zknydAjG2mXdvDRK/lgwQIyAgQwxv6Yfx+rQ/gjwQIyAgQUxlrEbQfatpHECBmMPr9y2Hw+W8mQMxmlGXFfpA5H0qAmMS4i4jDsJN/ngAxj4Eeh6384dcrAWJOS27Qkmf7YgLEVE4XFMu8zk+nOqx7+SNATGiUq/owyJwPJUBMaLGbQbZ+zggQc1pgg9TnkgCxiga1GTodQH1OCRAzW8KetF3nKwSIybUNUp/rvj0//9zMYrf7sVkBv8ef/9142m4ffcaz2PmpvckKiFU4u/4fuit0eXD1ec/3d78CczlW4HQpdMzEHVdDl1GTnhXdgsFnbtVvLtF7iyn1+SUBYqWu7xj+MkbX7+Ck54MEiLW747ML3fldAgR3yJD03EaA4JYeKc5dCBCQ8T4gICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIGBT+Q/R+Sa8zJQydwAAAABJRU5ErkJggg==', 'sha256': '2d0c320f8e84bbc2b0cfc4055e4fb1e1d68432791055205d7d3344c09de9c686'}, 'data/binding-v1/images/binding-evaluation-0037-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNElEQVR4nO3aO24bSRRAUXpg7oeBAG1iYi9S8WzCgAMuxoGATgYwYUI/0yYt6lazzokUUN1PkPqyqqhPj4/fNwCFf5K7AggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZD53tx7Obne3Wbn9/ls9ApxBgOCKb1reEk4TILjiSvnpi8XoNQF6w3/b7WZV/l2WeoTb9/c79OMVlOhIgODy7mzvH05/+/L1y4lr7qcvkQDBefX5bXR+9eLXMdrt7iZvkADBH6XnrO686XiFpyXa/bjRtBnyf0Dwm/ps7x/+vj6nL7hb/7+AXMYKCE6l53r32v64+HE1NOdSyAoIgvr86i67yZZCAgQvn/x333Odtn1+u6kaJEDwsj7JDNspGyRAzG6E+kzbIAFiauPUZ84GCRDzGq0+EzZIgGCg+ow5z/UIEJM6Li7GfNq3P6e67UWQADGjdT3Vu1VNexYBYmpjLn/Gn+29CBDTGXzzNdVGTICYy3qf5N1qJz9BgJjU+Mufdc15GQECMgLERFZ0+jPJSZAAARkBAjICxCxWuv+67V2YAAEZAQIyAgRkBIgprPoA6IaPgQQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIKaw3387fLF8/bJZp+Xn5Mef5QYIEJARICAjQEBGgJjFqo+Blls8ABIgoCRAQEaAmMhKd2HLje6/BAgoCRCTWssiaFnJnJcRIOay3l3MfrWTnyBATGdFJ0HL7Z7+HAgQUxu5QcvAs70XAWJG61pQ7Fc17VkEiEkNvhFbbn3zdSBAMFyDlsHmuR4BYl5PFxfjPPPLk0lue/kjQMxutAYtM9VHgGCgBi2T1UeAYJQGLfPVR4Dg7QZ9ZIaW57ebpz6bzebT4+P3eoZR7HZ3m5Wb6m/3A/4GtvcP177j8rx0s/0GP9cDwEAOz/8xQ4c6XClDy9zpObAFg5detODdd2SvL7ifsj62YHD2rvziBdGbFdvPmp4DAYLLTwZ/G6PTS6f93PURIAg+o9CdIwGCDyqR7rwmQHDFGInOaQIEZHwMD2QECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECNpX/ARg5H94+rBwXAAAAAElFTkSuQmCC', 'sha256': '76f56023958a154fd4d79231cbd336f107f1389c42e9ae2dcb671235446ab4d9'}, 'data/binding-v1/images/binding-evaluation-0037-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNElEQVR4nO3aPW7bWBhAUWUQ7YeFgdkE61nk1NrEACm0mBQG1AwQIYL/IkeK5UvqnVO5kMnPsHn13pO/PD5+3wAU/kruCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPna3Xpxpulhs3L7/bd6BLiAAMEN37S8JZwnQHDDlfLTF4vRawL0hu3f/25W5fDfP/UI9+/Pd+inKyjRiQDB9d3Zbbfnv30+HM5ccz98iQQILqvPu9H51Ytfx2iaHgZvkADBb6Xnou686XSFpyWaftxo2Az5PyB4pz677fbP63P+gtP6/wXkOlZAcC49t7vX7sfFT6uhMZdCVkAQ1OdXd5kGWwoJELx88j98z3Xe7vnthmqQAMHL+iQz7IZskAAxuiXUZ9gGCRBDW059xmyQADGupdVnwAYJECyoPsuc53YEiEGdFhfLfNp3P6e670WQADGidT3V06qmvYgAMbRlLn+WP9tHESCGs/DN11AbMQFiLOt9kqfVTn6GADGo5S9/1jXndQQIyAgQA1nR6c8gJ0ECBGQECMgIEKNY6f7rvndhAgRkBAjICBCQESCGsOoDoDs+BhIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgBjCfv/t+MV8OGzWaf45+elnuQMCBGQECMgIEJARIEax6mOg+R4PgAQIKAkQkBEgBrLSXdh8p/svAQJKAsSg1rIImlcy53UEiLGsdxezX+3kZwgQw1nRSdB8v6c/RwLE0JbcoHnBs30UAWJE61pQ7Fc17UUEiEEtfCM23/vm60iAYHENmhc2z+0IEON6urhYzjM/P5nkvpc/AsToltageaT6CBAsqEHzYPURIFhKg+bx6iNA8HaDPjND8/PbjVOfzWbz5fHxez3DUkzTw2blhvrb/YS/gd12e+s7zs9LN9pv8Gs9ACzI8fk/ZehYhxtlaB47PUe2YPDSixZ8+I7s9QX3Q9bHFgwu3pVfvSB6s2L7UdNzJEBw/cnguzE6v3Taj10fAYLgMwrdOREg+KQS6c5rAgQ3jJHonCdAQMbH8EBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYFP5H1iTHz7edPxLAAAAAElFTkSuQmCC', 'sha256': 'ab547b1a2ca036f4c12f04f7bae20f53544cc6c448dec112832ad0e3ad699656'}, 'data/binding-v1/images/binding-evaluation-0037-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFK0lEQVR4nO3aTW4aSxhAUfIU9sPAkneSRXonkTJgMRlY6kmkoCD/BT+I8e2izhl5YHV/lulLVcGXx8efG4DCf8ldAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMl+7W6/Obne3Gdx+/6MeAc4gQHDFNy1vCacJEFxxpfz0l8XoNQF6w/b+YTOU5fu3eoTb9+879OMVlOhIgODy7rz7XvXme4MSHQkQnFefsxbIT3/5dYx2u7vJGyRA8L/S8+8b8+MVnpZo9/tG02bI94Dgnfps7x8+9ljw9QV3438F5DJWQHAqPde71/b3xY+roTmXQlZAENTnb3fZTbYUEiB4+eR/+J7rtO3z203VIAGCl/VJZthO2SABYnZrqM+0DRIgprae+szZIAFiXmurz4QNEiBYUX3WOc/1CBCTOi4u1vm0b/9MdduLIAFiRmM91buhpj2LADG1dS5/1j/bRxEgprPyzddUGzEBYi7jPsm7YSc/QYCY1PqXP2PNeRkBAjICxEQGOv2Z5CRIgICMAAEZAWIWg+6/bnsXJkBARoCAjAABGQFiCkMfAN3wMZAAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAsQU9vsfhx+W7982Y1r+TH78W26AAAEZAQIyAgRkBIhZDH0MtNziAZAAASUBAjICxEQG3YUtN7r/EiCgJEBMapRF0DLInJcRIOYy7i5mP+zkJwgQ0xnoJGi53dOfAwFiamtu0LLi2T6KADGjsRYU+6GmPYsAMamVb8SWW998HQgQrK5By8rmuR4BYl5PFxfreeaXJ5Pc9vJHgJjd2hq0zFQfAYIVNWiZrD4CBGtp0DJffQQI3m7QZ2ZoeX67eeqz2Wy+PD7+rGdYi93ubjO4qV67n/Aa2N4/XPuOy/PSzfYf/FoPACtyeP6PGTrU4UoZWuZOz4EtGLz0ogUfviN7fcH9lPWxBYOzd+UXL4jerNh+1vQcCBBcfjL4boxOL532c9dHgCD4jEJ3jgQIPqlEuvOaAMEVYyQ6pwkQkPExPJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQI2FR+ATPhH94gwVWjAAAAAElFTkSuQmCC', 'sha256': 'deadcc8b16a87992554d0d31906280353fc74ab3d9e48ef40b17358296795796'}, 'data/binding-v1/images/binding-evaluation-0038-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE8klEQVR4nO3dTW4aSxhAUfspWU4kD7KzrOPtLANLbyseIGXyBkgWwqENpMOtrj5n6p8uCXNd9YGa58Ph7Qmg8E9yVQABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dJdGkbx8vJ9+RteX38+ai378nw4vNVrgBGjc4kYrUiA2J2703NKhlYhQOzFQnd+/fi2/LNf//3v0peU6E8IEDutz6fRuT5GGnQ3AWJyK6bnlAytQoDYUX1WSc9ChjToVgLELuqzenouZUiDbuKNiMzpYfU5+/2rvMS2HwLEhB5Zn49X0aDrOYIxm/fn/2PSc+k45ix2DTsgptLW5/S69kHXECDmMdpzfrT1DEiAmMTj5z6XmAddT4CYTVufcdawCQLEDPLRz0eGQddwP6Cnyf5Kdvjiy/iP3cvL9x0+LtewA2Ie42x/xlzPgAQIyDiCTfJfa+GGNXMbcPpz6tePb8eHxinst+yAgIwAsWGDb3+OvBy2QICAjAABGQECMgLEVm1iAHRkDHSJAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgLEVr3f32v8m7H5uNRLBAjICBCQESAgI0Bs2CbGQAZACwQIyAgQ2zb4Jsj2Z5kAARkBYh6jbYJGW8+ABIjNG/90M/4KKwLEDAacBJn+XEOAmM0IDRphDZsgQEzidKPRPv9Pr277s0yAmMdoz/bR1jMgAWIq+TDI6OcmX277dhje6+vP4weQHlvwsM9NdfK6gx0QE3r8PEh97iNAzOmRDVKfuzmCMf9Z7O8dx87Spj63EiD20qB1M/RxV6U+dxAgJnfswooZkp4VCRB73AqddeTTGC1MkdTnTwgQe/FeirMS3Tel1p1VCBC78/FQdsePswoBYqdOO/JpjETnLxEg0JeMNyICGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIybsl67jEfJQ7YAQElAQIyz4fDW3d1YNfsgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQ8FT5H2VxFtSwn4MWAAAAAElFTkSuQmCC', 'sha256': '4c5709b7b31068cd4ec37a7b7ba8ca076d8bc9a4ef83066adcc29232afb292d3'}, 'data/binding-v1/images/binding-evaluation-0038-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE+0lEQVR4nO3dQU4qWRiAUe0898PAxJ30It1JJ2/gYhyQ9KQHJIZgg8gr+W7dOmeoQt0E+bz1lyke9/v3B4DCX8lRAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmV3doGMVu93z5B97eft9rLdvyuN+/12uAEaNzjhgtSIDYnJvTc0yGFiFAbMWF7jy9vF5+7L///H3uW0r0JwSIjdbny+hcHyMNupkAMbkF03NMhhYhQGyoPouk50KGNOi7BIhN1Gfx9JzLkAZ9i39EZE53q8/J8y9yiW07BIgJ3bM+n4+iQddzCsZsPt7/90nPudMx52LXsANiKm19jo9rH3QNAWIeo73nR1vPgASISdx/7nOOedD1BIjZtPUZZw2rIEDMIB/9fGYYdA33A3qY7Ldkgxdfxn/tdrvnDb4u17ADYh7jbH/GXM+ABAjIOAWb5K/WhRvWzG3A6c+xp5fXw0vjLOx/2QEBGQFixQbf/hy4HHaBAAEZAQIyAgRkBIi1WsUA6MAY6BwBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBIi1+ri/1/g3Y/NxqecIEJARICAjQEBGgFixVYyBDIAuECAgI0Cs2+CbINufywQIyAgQ8xhtEzTaegYkQKze+Gc346+wIkDMYMBJkOnPNQSI2YzQoBHWsAoCxCSONxrt+//46LY/lwkQ8xjt3T7aegYkQEwlHwYZ/XyLADFzg+6ZoePDqc+VBIgJ3X8eZO5zGwFiTvdskPrcTIDYSoN+IkMnT6s+3yVAzOykCAtm6PNTqc8NftULgJ916MJu9/zxlUM4nl5eb3vCzwmTnpsJEJvw9vb7uEEnHfkyRhf2TerzJwSIrfgoxUmJbptS684iBIjN+XxSdsPDWYQAsVHHHfkyRqLzQwQI9CXjMjyQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDjlqyn7vNR4oAdEFASICDzuN+/d0cHNs0OCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABD5X/AIZXJKSBz10NAAAAAElFTkSuQmCC', 'sha256': '3dc9aeb70c8943b677b151a576df230bf40aaa95866d6a717c4592816731bfa3'}, 'data/binding-v1/images/binding-evaluation-0038-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFAUlEQVR4nO3dQWobSRiAUXtIjhPwIjA3yUVyjrnI3CSQhSFX8ULgzSwERsgjWVI6+qqr39vachfI+virJNSPu93LA0Dhr+SqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzqbs0jOLp6ev5X3h+/nmvtWzL4273Uq8BRozOKWK0IAFic25OzyEZWoQAsRVnuvP6/cv5x37+59epHynR7xAgNlqfD6NzeYw06GYCxOQWTM8hGVqEALGh+iySnjMZ0qBrCRCbqM/i6TmVIQ26ig8iMqe71efo7y/yFtt2CBATumd93l9Fgy5nC8Zs3l7/90nPqe2YvdglTEBMpa3P4XXNQZcQIOYx2mt+tPUMSICYxP3PfU5xHnQ5AWI2bX3GWcMqCBAzyI9+3nMYdAnfB/Qw2X/JBt98Gf+5e3r6usHn5RImIOYxzvgz5noGJEBAxhbs2Oe//31Yodcf3x42acDTn0Ov37/sP5poF/a/TEBARoBYscHHnz1vh50hQEBGgICMAAEZAWKtVnEAtOcY6BQBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBIi1evt+r7ebkQ7L7VJPESAgI0BARoCAjACxYqs4BnIAdIYAARkBYt0GH4KMP+cJEJARIOYx2hA02noGJECs3vi7m/FXWBEgZjDgSZDTn0sIELMZoUEjrGEVBIhJHA4a7ev/8OrGn/MEiHmM9mofbT0DEiCmkh8GOfq5yqfrfh2G9/z8c38D0n0L7nbfVDuvG5iAmND9z4PU5zYCxJzu2SD1uZktGPPvxf7cduwobepzLQFiKw1aNkPvpyr1uYEAMbl9FxbMkPQsSIDY4ih01JEPY3TmFEl9focAsRVvpTgq0W2n1LqzCAFic95vym54OIsQIDbqsCMfxkh0/hABAn3J+CAikBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg4ytZj73++FYvAbbCBARkBAjIPO52L93VgU0zAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEg4KHyH8xnF4mkTJzRAAAAAElFTkSuQmCC', 'sha256': 'fe33dceb17fe08b5da3dbfe1de583d64a037103d21d156bf7a2af876c3492655'}, 'data/binding-v1/images/binding-evaluation-0039-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFN0lEQVR4nO3aQW4aSRiAUTJK7sPCWy6UA+VC3nrBYbKwlM0s0DAIsB0bm6+76r2VJSO6LNOf/ir62/Pz7w1A4Z/kqgACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM980cttuHzXD2+6d6CXATExCQmWUCgk+fmk2gt5suQI+//mxWbvfzR72EYb1rq376YjH6mOkCBF9xRHh8ByV6FwFiXq93581h+eooqkTvIkBM6mp93rVDP33xZYy22wcNepMAMZ3L9Nx+Mnh8h9MSHS4kQ68QIKauz6d/KXF4w7MMadBLBIhZfHV6XsmQUeglHkRkCvesz0tXGfJx/BuZgBjf6Z1/5wfBLkchc9ApExCDC+tz9brmoFMCxMiWUJ/Lq2vQkQAxrOXU53INGnQgQIxpafU50KAzAsTgllOfZa6nJUAM6DhcLPNuP65qO/0QJECMZl139XZVq/10AsSwljn+LH9t9yRADGXhm69TjzZiAsRI1nsnb1e78hsJEANa/vizrnV+HQECMgLEIFZ0+nPqce6TIAECMgIEZASIEax0/3Uw8y5MgICMAAEZAQIyAsTqrfoAaPJjIAECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQEiNXb758OP+x+/tis0+6/lR//lkkIEJARICAjQEBGgBjBqo+BdrMeAAkQUBIgICNADGKlu7DdxPsvAQJKAsSA1jIE7Vayzq8jQIxjvbuY/WpXfiMBYigrOgnazX36cyBADGvJDVry2u5JgBjNugaK/apW++kEiAEtfCNm83UkQAxuaQ1a2npaAsSYToeL5dzzpyvZTz/+CBAjW1qD1OeSADGy5TRIfa4SIAa3hAapz0u+v/gbGMV+/7TdPpy24PHXn/tc+ix56nPGBMQUzu78+4xC6vMmExCzONz/9xmFpOcvCRDzbse+IkOXs5X6vEKAmH0UOq3Gh0t0dU8nPW8SICZ1NgpdduTNGL1+kKQ+f+Pb8/PvzQQuP2oD8BFf4MfDP+VdTEDwfzU+XCLd+RgBgusdeTNGonO7WbZgwAJ5EBHICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEbCr/AlrvIm8UF9OaAAAAAElFTkSuQmCC', 'sha256': '3cddf3a24c880485ef72f71e4c8b325797e3d797bd35a183b3b9b0ca2c7d989f'}, 'data/binding-v1/images/binding-evaluation-0039-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQ0lEQVR4nO3aTWobSRiAYWWIjhPQIjA3mYvkHLnI3GQgC0GukoXBm1mI0QjJln8U6+2uep6VwXJ3Catfvir06eHh1wag8EdyVwABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDm82YOu93XzXD2+x/1EuAmJiAgM8sEBL99ajaB3m66AG3//Huzco///FUvYVhv2qqfvliM3me6AMFHHBEer6BEbyJAzOt6dx6/fbn+59vvP69cU4leQ4CY1JP1eTE6z734Mka73VcNepEAMZ3L9LypO086XuG0RIcbydAVAsTU9bk9PWcOFzzLkAY9R4CYxUen50qGjELP8UVEpnDP+jx3lyG/jn8jExDjO33y75OeK6OQOeiUCYjBhfV58r7moFMCxMiWUJ/Lu2vQkQAxrOXU53INGnQgQIxpafU50KAzAsTgllOfZa6nJUAM6DhcLPNpP65qN/0QJECMZl1P9W5Vq/3tBIhhLXP8Wf7a7kmAGMrCN1+nHm3EBIiRrPdJ3q125TcSIAa0/PFnXev8OAIEZASIQazo9OfU49wnQQIEZAQIyAgQI1jp/utg5l2YAAEZAQIyAgRkBIjVW/UB0OTHQAIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEKu33/84/LD9/nOzTtv/Vn58L5MQICAjQEBGgICMADGCVR8DbWc9ABIgoCRAQEaAGMRKd2HbifdfAgSUBIgBrWUI2q5knR9HgBjHencx+9Wu/EYCxFBWdBK0nfv050CAGNaSG7Tktd2TADGadQ0U+1Wt9rcTIAa08I2YzdeRADG4pTVoaetpCRBjOh0ulvPMn65kP/34I0CMbGkNUp9LAsTIltMg9XmSADG4JTRIfZ7z+dnfwCj2+x+73dfTFjx++3KfW58lT33OmICYwtmTf59RSH1eZAJiFofn/z6jkPS8kgAx73bsIzJ0OVupzxUCxOyj0Gk13l2iJ/d00vMiAWJSZ6PQZUdejNH1gyT1eY1PDw+/NhO4/KgNwEd8gR8P/5Q3MQHB/9V4d4l0530ECJ7uyIsxEp3bzbIFAxbIFxGBjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwKbyLxA8Im9x/zxcAAAAAElFTkSuQmCC', 'sha256': '6de57a2a7889028de5cd394cea787bcda1262f12ab094f3f53501e0d0a2f9b69'}, 'data/binding-v1/images/binding-evaluation-0039-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRElEQVR4nO3aPW4bRxiAYTqw78PCQCpeI4fwgXyIXENVABc8jAsBblIQYQhSon5o8d2deZ5KgKjdIcR98c2Anx4ff24ACn8kdwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkPm/msN1+3Qxnv/9RLwFuYgICMrNMQPDbp2YT6O2mC9CXP//erNyvf/6qlzCsN23VT18sRu8zXYDgI44Ij1dQojcRIOZ1vTsP339d//Pdty9XrqlEryFATOrJ+rwYnedefBmj7farBr1IgJjOZXre1J0nHa9wWqLDjWToCgFi6vrcnp4zhwueZUiDniNAzOKj03MlQ0ah5/giIlO4Z32eu8uQX8e/kQmI8Z0++fdJz5VRyBx0ygTE4ML6PHlfc9ApAWJkS6jP5d016EiAGNZy6nO5Bg06ECDGtLT6HGjQGQFicMupzzLX0xIgBnQcLpb5tB9XtZ1+CBIgRrOup3q7qtX+dgLEsJY5/ix/bfckQAxl4ZuvUw82YgLESNb7JG9Xu/IbCRADWv74s651fhwBAjICxCBWdPpz6mHukyABAjICBGQEiBGsdP91MPMuTICAjAABGQECMgLE6q36AGjyYyABAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBIjV2+9/HH7YffuyWafdfys/vpdJCBCQESAgI0BARoAYwaqPgXazHgAJEFASICAjQAxipbuw3cT7LwECSgLEgNYyBO1Wss6PI0CMY727mP1qV34jAWIoKzoJ2s19+nMgQAxryQ1a8truSYAYzboGiv2qVvvbCRADWvhGzObrSIAY3NIatLT1tASIMZ0OF8t55k9Xsp9+/BEgRra0BqnPJQFiZMtpkPo8SYAY3BIapD7P+fzsb2AU+/2P7fbraQsevv+6z63Pkqc+Z0xATOHsyb/PKKQ+LzIBMYvD83+fUUh6XkmAmHc79hEZupyt1OcKAWL2Uei0Gu8u0ZN7Oul5kQAxqbNR6LIjL8bo+kGS+rzGp8fHn5sJXH7UBuAjvsCPh3/Km5iA4P9qvLtEuvM+AgRPd+TFGInO7WbZggEL5IuIQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIGBT+RdEFiJvfL+hzQAAAABJRU5ErkJggg==', 'sha256': '7bea3a5a7f67fe824ebda8ca1d8623bbff5d490b2ba2f2bd3c9507188f476e9d'}, 'data/binding-v1/images/binding-evaluation-0040-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFP0lEQVR4nO3bQU4iWxiAUX1p9uOgl8FyXUYPXEwPSJi8QaUJEUUQ5Ktbdc7MiKlb0fr87xWfd7u/TwCF/5KrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZlf3aVhLl5efl/ysre3Pz+/lnV53u3+1muA+UbnM2J0FwLEutzYnVNKdAsBYi3Op+d1szn/5dv9/sxnZeh7BIiVpufL4nyjRzJ0LQFiXem5sTuXlEiGLidALNMD0nNMhr5HgFh+fX40PWcypEFfWnuA7v43kUfy8/3l9/Rh6fksQ75H53kjIstRDT7vvG42hwZNS5Khz/hXDBZiJvX58OpDD9o/ygQ0i5/Xq5x/Q8o65duuU9Myjkchc9ApExDDm2F9PlyPOeiUADG2OddnokFnCBALMc/6zH9tLQFiYIeBYv5P+GGFhqBjAsSoxn2Sx1353QkQQ5r/0c8ph0GnBIixjVKfEVf7AALEeAY6+jnlMOiYAAEZAWIwQ48/E0PQgQABGQFiJAsYfyaGoIkAARkBYhiLGX8mr4YgAQJCAgRkBIgxLGz/NXld/S5MgICMAAEZAQIyAsQA1nBE8rKCezwlQIxkSSfQS72jqwgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQIxku98/Lct2cXd0FQFiAG9vf56W7m0F93hKgICMAAEZAQIyAsRgRyRLOrXd/ruXdR4ACRBQEiAgI0AMY2G7sO3q918CBJQEiJEsZggy/kwECMgIEINZwBBk/DkQICAjQIxn6CHI+HNMgBjbWA0aa7UPIEAM6Xh8GOWpPl6n8WciQIxq3Gd43JXfnQAxsIEOgxz9fEiAWIg5N2jOa2sJEGOb/2GQo58zBIjhzblB6nOeALHABs0hQ++WoT4fEiAW4t0T3jbo3dXV5zO/Pv3MyszhdyY3mp7zl5ff04fb/f51s3n8Mgw+lzMBsTThdsy261oCxCq2Yz+dodNLqM8lnne7vxe9EAZ02I4d3H1Tdpo26bmcALHGDN1eog9HKum5lgCx6gxd3qPzmzjp+R4BYl3OZ+gbpOcWAsRK3Vgi3bkLAYJLYyQ6dydAQMb7gICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQKeKv8D4UFZfO0UossAAAAASUVORK5CYII=', 'sha256': '22a61bb650779479d80e786256275faecf41c1e41b2cc80c10c1c6456df928b8'}, 'data/binding-v1/images/binding-evaluation-0040-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRklEQVR4nO3bQU4jRxiAURPFx4nEYiRuMhfJOXKRuclIs0Caq7Cw5E0WTpBFgzHY+Ovqfm8LqKtl96e/yuZut3vaABT+SK4KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZP7sLg1zcX//7Zxfe3z89fVrWZe73e6pXgPMNzpvEaOrECDW5cLuTCnRJQSItTidnu3Dj9N/vv/5/cRPZehzBIiVpufd4nyiRzL0UQLEutJzYXfOKZEMnU+AWKYbpOeYDH2OALH8+nxpek5kSIPetfYAXf0zkVvy/n73Nb1Zet7KkNfoNF9EZDmqweeF7cOP5wYdliRDb/GvGCzETOrz6tWHHrS/lAnoP/u//9oMYvvP73oJs5Nvu6YOyzgehcxBUyYghjfD+ry6HnPQlAAxtjnX50CDThAgFmKe9Zn/2loCxMCeB4r5P+HPKzQEHRMgRjXukzzuyq9OgBjS/I9+phwGTQkQYxulPiOu9gYEiPEMdPQz5TDomAABGQFiMEOPPweGoGcCBGQEiJEsYPw5MAQdCBCQESCGsZjx52BrCBIgICRAQEaAGMPC9l8H29XvwgQIyAgQkBEgICNADGANRyT3K7jHKQFiJEs6gV7qHX2IAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgLESPY/v2+WZb+4O/oQAWIAj4+/Nkv3uIJ7nBIgICNAQEaAgIwAMdgRyZJObff/38s6D4AECCgJEJARIIaxsF3YfvX7LwECSgLESBYzBBl/DgQIyAgQg1nAEGT8eSZAQEaAGM/QQ5Dx55gAMbaxGjTWam9AgBjS8fgwylN9vE7jz4EAMapxn+FxV351AsTABjoMcvTzKgFiIebcoDmvrSVAjG3+h0GOfk4QIIY35wapz2kCxAIbNIcMvViG+rxKgFiIF09426AXV1eft9ztdk+bFbu//7YZlrf1u6/p9uHH7Rdg8DmfCYilCbdjtl0fJUCsYjv21RmaXkJ9zrH2LRhr22JffVM2TZv0nE+AWOlJ34UlenWkkp6PEiDW4vQHDu/26PQmTno+R4BYl6t/7ik9lxAgVurCEunOVQgQnBsj0bk6AQIyvgcEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKn8CxK9Vt0cvTbhAAAAAElFTkSuQmCC', 'sha256': 'c93f491061d3e4fe6a5975ef8b48019fdb2ca787c9c3c38654416fa64e9d363b'}, 'data/binding-v1/images/binding-evaluation-0040-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQklEQVR4nO3bQU7jSBiA0TCaHGckFn0MbtPnmNtwjF4gzVVYRGLTi0yjCENISMjnst/bAnJZiT/9VQl3u93zBqDwV3JVAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPN3d2mYi/v7H6f82tPTr+9fy7rc7XbP9RpgvtH5iBhdhQCxLhd2Z0qJLiFArMXx9Dxut8f//OHl5chPZehrBIiVpufT4nyhRzJ0LgFiXem5sDunlEiGTidALNMN0nNIhr5GgFh+fb41PUcypEGfWnuArv6ZyC15f3/6mt4sPR9lyGt0nC8ishzV4PPG43b72qD9kmToI/4Vg4WYSX3evfrQg/a3MgH97+XnP5tBbP/9r17C7OTbrqn9Mg5HIXPQlAmI4c2wPu+uxxw0JUCMbc712dOgIwSIhZhnfea/tpYAMbDXgWL+T/jrCg1BhwSIUY37JI+78qsTIIY0/6OfKYdBUwLE2Eapz4irvQEBYjwDHf1MOQw6JEBARoAYzNDjz54h6JUAARkBYiQLGH/2DEF7AgRkBIhhLGb82Xs0BAkQEBIgICNAjGFh+6+9x9XvwgQIyAgQkBEgICNADGANRyT3K7jHKQFiJEs6gV7qHZ1FgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQFiJA8vL5tleVjcHZ1FgBjA09OvzdI9reAepwQIyAgQkBEgICNADHZEsqRT24c/97LOAyABAkoCBGQEiGEsbBf2sPr9lwABJQFiJIsZgow/ewIEZASIwSxgCDL+vBIgICNAjGfoIcj4c0iAGNtYDRprtTcgQAzpcHwY5ak+XKfxZ0+AGNW4z/C4K786AWJgAx0GOfp5lwCxEHNu0JzX1hIgxjb/wyBHP0cIEMObc4PU5zgBYoENmkOG3ixDfd4lQCzEmye8bdCbq6vPR+52u+fNit3f/9gMy9v609f0cbu9/QIMPqczAbE04XbMtutcAsQqtmPfnaHpJdTnFGvfgrG2LfbVN2XTtEnP6QSIlZ70XViid0cq6TmXALEWxz9w+LRHxzdx0vM1AsS6XP1zT+m5hACxUheWSHeuQoDg1BiJztUJEJDxPSAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCATeU3dDdYbYPhe24AAAAASUVORK5CYII=', 'sha256': '3a7c5abaf38f6b74226de05d6b63c443407b983c7b56486f631c3350bb53d07d'}, 'data/binding-v1/images/binding-evaluation-0041-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFGUlEQVR4nO3bMU4bQRiAUScK96GIxE04JDdBSsFhUiDRpLBkWYYYbNb+ZtbvVSBZu0Pk/fTP2Pnx+vp3A1D4mdwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGR+bVbt/v73ZnVeXv7US4BlmICAjAABmZVvwXbuHp42k3t7fqyXAAu7lQDBGUeEjtsuTYC4LSd9LrH/YjG6BAHiJnz/89DdFZRoQQLE7abn05PBD8/dtheUoUUIELeVnpM+jth/8UGMZGgRAsRNpOf7H4PurrBfIhn6Jt8DYuX1uXt4WvZLGO8vuMov3F+HALEq+y1YPD37Di6uQeexBWO1g88Vbnr38LTbkdmOncEExBok9fnwXkahkwgQ07vatut/bMfOJkCsqj7hSjToDALESozw/41HWMNcBIiJ7QaNcZ783UoMQV8hQMxq/Cd8/BXmBIgpjXP0857DoK8TIOY2Wn1GXtWABIj5DHj0857DoK8QICAjQExmivFnyxD0KQECMgLETCYaf7YMQccJEJARICAjQExjuv3Xll3YEQIEZAQIyAgQkBEg5jDpAdCWY6D/ESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAmMPLy5/tD2/Pj5vZ7Na8+yvYEiAgI0BARoCAjAAxjUmPgRwAHSFAQEaAgIwAMZPpdmH2X8cJEJARICYz0RBk/PmUAAEZAWI+UwxBxp+vECDmNmaDxlzVgASIKe2PFaM97fvrMf4cJ0DMavxne/wV5gSIiQ14GOTo5yQCxEqM0KAR1jAXAWJu4xwGOfo5gwCxtgZdP0MHN1WfrxMg1uDgmb9mgw7upT4n+XXay2FU2yf//v739te358e7h6dL39Tg800mIFblatsx265FCBDr344tm6H3F1Sfs9mCsf7t2P5e6ex92YcVk55vEiBuKEMHHfk0RkdGJ+lZxI/X17+b9Tp4862Dt377TvDvvyATEDdhV42zS6Q7lyBA3Jb9jnwaI9G5tJVvwYCR+RgeyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKn8A5rJHGn02hr1AAAAAElFTkSuQmCC', 'sha256': '7628d738537def91c3c2505513f76f9ea61048c9975c2b7229ddd3bd0d76a288'}, 'data/binding-v1/images/binding-evaluation-0041-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3bTW4aSxhAUfKULCcSg+ws68jOPEDKVjKwlEkGSAi1/TDm71Y154w8sOmyRV99VY2/vL7+2QAU/kuuCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyXzertt3+2KzObvdSLwFuwwQEZAQIyKx8C3bw9+f3zeS+/fpdLwFu7FkCBBccETpuuzcB4rl86rnE8TeL0T0IEE/h+uehh1dQohsSIJ43PR+eDL577rZ/QRm6CQHiudLzqccRx9+8iJEM3YQA8RTpuf4x6OEVjkskQ1cSIFZen5t/AmP/gosMadBlBIjV1ueuH/5aZEiDLuOT0KzEdvvjYfV59yqLBXAOAWIN7r3tOmFxLQ36FFswpvf4wWfBduxiJiDmltfn3aubg84kQKzECP9vPMIa5iJATOwwaIxz5x9WYgg6hwAxq/Hv8PFXmBMgpjTO0c9bDoPOJ0DMbbT6jLyqAQkQ8xnw6Octh0HnECAgI0BMZorxZ88Q9CEBAjICxEwmGn/2DEGnCRCQESAgI0BMY7r9155d2AkCBGQECMgIEJARIOYw6QHQnmOg/yNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMADGH3e5l/8W3X783szms+fBbsCdAQEaAgIwAARkBYhqTHgM5ADpBgICMAAEZAWIm0+3C7L9OEyAgI0BMZqIhyPjzIQECMgLEfKYYgow/5xAg5jZmg8Zc1YAEiCkdjxWj3e3H6zH+nCZAzGr8e3v8FeYEiIkNeBjk6OdTBIiVGKFBI6xhLgLE3MY5DHL0c4Gvl/wQjGS3e9lufxxX4O/P749cwCJ86nM+ExBrsLjnHzkKqc81BIiV2O1eHr8dW2y71OezbMFYlYdtxww+NyFArLlB98jQ29lKfS4mQKzQvghvM3RNid7d00nPlQSIJ8rQoiMfxujEQZL03MSX19c/m/VavPnWwVu/fSf4+9+QCYincKjGxSXSnXsQIJ7LcUc+jJHo3NvKt2DAyHwQEcgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELCp/AO9HRxxy+20cgAAAABJRU5ErkJggg==', 'sha256': '284945e2b4b353970d5c6689ed67da71e913573390d74f9ef24ff7380f433eb8'}, 'data/binding-v1/images/binding-evaluation-0041-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIElEQVR4nO3aQUpjWRiA0VRTLqfBQYE7cSO9jt6IOxFqIPRWaiA46UEghKiJiUm+e1/OGSnE966S9/HfG3+8vv5ZART+Su4KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjI/V4t2f/9rtTgvL7/rJcB5mICAjAABmYVvwTbe/vl7Nbm7f/+rlwBndisBghOOCB23XZoAcVuO+lxi+8VidAkCxE34/uehmyso0RkJELebnruHp/0/+/b8+NkFZegsBIjbSs/B6Hz24p0YydBZCBA3kZ6juvOhzRW2SyRD3+T/gFh4fe4enr5fn/0XXOQ/3F+HALEo2y04e3q27Vxcg05jC8ZiB58r3PTu4WmzI7MdO4EJiCVI6vPhvYxCRxEgpne1bddnbMdOJkAsqj7hSjToBALEQrT1GWcNcxEgJrYZNMZ58jcrMQR9hQAxq/Gf8PFXmBMgpjTO0c97DoO+ToCY22j1GXlVAxIg5jPg0c97DoO+QoCAjAAxmSnGnzVD0EECBGQEiJlMNP6sGYL2EyAgI0BARoCYxnT7rzW7sD0ECMgIEJARICAjQMxh0gOgNcdAnxEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgJjDy8vv9Rdvz4+r2WzWvPktWBMgICNAQEaAgIwAMY1Jj4EcAO0hQEBGgICMADGT6XZh9l/7CRCQESAmM9EQZPw5SICAjAAxnymGIOPPVwgQcxuzQWOuakACxJS2x4rRnvbt9Rh/9hMgZjX+sz3+CnMCxMQGPAxy9HMUAWIhRmjQCGuYiwAxt3EOgxz9nECAWFqDrp+hnZuqz9cJEEuw88xfs0E791Kfo/w87uUwqvWTf3//a/3t2/Pj3cPTpW9q8PkmExCLcrXtmG3XWQgQy9+OnTdD7y+oPiezBWP527HtvdLJ+7IPKyY93yRA3FCGdjpyMEZ7RifpOYsfr69/Vsu18+ZbBm/99p3g739GJiBuwqYaJ5dIdy5BgLgt2x05GCPRubSFb8GAkfkYHsgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELCq/A8ktxxprUjCbAAAAABJRU5ErkJggg==', 'sha256': '677e0925b79ef4ca93e0fa7775968a44ec221f6a43e920d2182cd22b1272c35c'}, 'data/binding-v1/images/binding-evaluation-0042-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMklEQVR4nO3dMU4jSRiAUVgNx0EimHQuNCnnmHQvtOkESHsVAkskG1hiLMw2tjH9dXW/l4LpCujPfxU2vt3tnm8ACn8lVwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZL51l4ZFeHj4Pv0NT0+/51rL5tzuds/1GmBx0fk/YnRdAsSGXNydY0p0FQLE1rvz8ng//fC7X/9OfFWJPkOA2GJ9PozOWTHSoIsJEBtKz8XdOaVEMnQBAWIT9blieiYypEHnEiDWZp70HJKhi3khIqsyf32Or3LFv7WtngmI9Ti88+dJz8QoZA46hQmIlcjr8+a65qBTCBBrsIT6HF9dgz4kQAxvOfU5XoMGTRMgxra0+uxp0Ik28W74jfwGbPzUczn12Xt5vJ9+DwcmIFby1LK0+rxZ1UaeAi8gQIxqrLt6rNXOZhNbsFf//P1yszo/ft7dbNsyx589G7FpJiCGtPDN1yEbsQkCxHjGvZPHXfkXESAGtvzxZ6x1zk+AgIwAMZiBTn8OOQl6lwABGQECMgLESAbdf+3ZhR0TICAjQEBGgICMADGMoQ+A9hwDvSFAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMADHeR7+O+0E3ryvf+MfYvhIgICNAQEaAgIwAMZKhj4EcAB0TICAjQEBGgBjMoLsw+693CRCQESAGNsoQNMo65ydAjGfcXcy4K/8iAsSQBjoJcvozQYAY3pIbtOS1LYEAMaqxBoqxVjsbAWJgC9+I2Xx9SIBYiaU1aGnrWSYBYmyHw8Vy7vnDlRh/JggQw1tag9TndALEGiynQepzFgFiJZbQIPU517ezHwFL9fT0++Hh+2ELXh7v57n0m+Spz4lMQKzKmzt/nlFIfS5mAmJt9vf/PKOQ9HySALH+7dhXZOh4tlKfCwgQWxmFDqtxcYne3dNJz8UEiG2NQscd+TBG0wdJ6vMZAsT6vTbiuEQXH1TrzlUIEBsyXaKzfgJXIUBs0WFHPoyR6HwdAWLr9CXkhYhARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMtv4l64+fd/USgD9MQEBGgIDM7W733F0d2DQTEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICbyn/GdzCorj5niQAAAABJRU5ErkJggg==', 'sha256': '7fc7eb7b9f44904b259503ab8dd4247cc57a69dbc29063c433ccbd1b97983f1c'}, 'data/binding-v1/images/binding-evaluation-0042-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMklEQVR4nO3dMU4jSRiAUVgNx1mJYNK90KRzjkm5EOkESHMVAkskG1hiLcw2dmP8dXW/l4LpCujPfxU2vt3tnm8ACn8lVwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZL51l4ZFuL//Pv0NT0+/r7WWzbnd7Z7rNcDiovN/xOiyBIgNmd2dY0p0EQLE1rvz+PAy/fB/ftxNfFWJPkOA2GJ9PozOWTHSoNkEiA2lZ3Z3TimRDM0gQGyiPhdMz0SGNOhcAsTaXCc9h2RoNi9EZFWuX5/jq1zwb22rZwJiPQ7v/OukZ2IUMgedwgTESuT1eXNdc9ApBIg1WEJ9jq+uQR8SIIa3nPocr0GDpgkQY1taffY06ESbeDf8Rn4DNn7quZz67D0+vEy/hwMTECt5allafd6saiNPgTMIEKMa664ea7VXs4kt2KuXn3/frM7drz8327bM8WfPRmyaCYghLXzzdchGbIIAMZ5x7+RxV/5FBIiBLX/8GWud1ydAQEaAGMxApz+HnAS9S4CAjAABGQFiJIPuv/bswo4JEJARICAjQEBGgBjG0AdAe46B3hAgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgBjvo1/H/aCb15Vv/GNsXwkQkBEgICNAQEaAGMnQx0AOgI4JEJARICAjQAxm0F2Y/de7BAjICBADG2UIGmWd1ydAjGfcXcy4K/8iAsSQBjoJcvozQYAY3pIbtOS1LYEAMaqxBoqxVns1AsTAFr4Rs/n6kACxEktr0NLWs0wCxNgOh4vl3POHKzH+TBAghre0BqnP6QSINVhOg9TnLALESiyhQepzrm9nPwKW6unp9/3998MWPD68XOfSb5KnPicyAbEqb+7864xC6jObCYi12d//1xmFpOeTBIj1b8e+IkPHs5X6zCBAbGUUOqzG7BK9u6eTntkEiG2NQscd+TBG0wdJ6vMZAsT6vTbiuESzD6p15yIEiA2ZLtFZP4GLECC26LAjH8ZIdL6OALF1+hLyQkQgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBmW/+S9e7Xn3oJwH9MQEBGgIDM7W733F0d2DQTEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICbyr8uNiuZlLmpYwAAAABJRU5ErkJggg==', 'sha256': 'e08c73c234fffd3f7e2b2c395fc09a9cb6b04ab7d9a39595bd845dd0fd78379e'}, 'data/binding-v1/images/binding-evaluation-0042-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJUlEQVR4nO3dQU4bSRiA0TBKjoPEIjfjHHOzLJDmKiwssZkFErIwaWxj+qvqfm8LpkuC/vxXYfDd4fD8A6DwT3JVAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmZ3dpGMLDw+/lT3h6+rPWWnbn7nB4rtcAw0Xnb8TotgSIHbm6O6eU6CYEiL135+Xxfvnhv/79b+GjSvQVAsQe6/NpdC6KkQZdTYDYUXqu7s45JZKhKwgQu6jPDdOzkCENupQAsTXrpOeYDF3NCxHZlPXrc3qVG/6ubfNMQGzH8Z2/TnoWRiFz0DlMQGxEXp931zUHnUOA2IIR6nN6dQ36lAAxvXHqc7oGDVomQMxttPq80qAz7eKv4XfyE7DzU89x6vPq5fF++W84MAGxkaeW0erzblU7eQq8ggAxq7nu6rlWu5pdbMHejPk8+UXm/JG/rTZiy0xATGnwzdcxG7EFAsR85r2T5135NxEgJjb++DPXOtcnQEBGgJjMRKc/x5wEfUiAgIwAARkBYiaT7r9e2YWdEiAgI0BARoCAjAAxjakPgF45BnpHgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQFivrd+nfeNbt5WvvO3sX0jQEBGgICMAAEZAWImUx8DOQA6JUBARoCAjAAxmUl3YfZfHxIgICNATGyWIWiWda5PgJjPvLuYeVf+TQSIKU10EuT0Z4EAMb2RGzTy2kYgQMxqroFirtWuRoCY2OAbMZuvTwkQGzFag0Zbz5gEiLkdDxfj3PPHKzH+LBAgpjdag9TnfALEFozTIPW5iACxESM0SH0u9fPiR8Conp7+PDz8Pm7By+P9Opd+lzz1OZMJiE15d+evMwqpz9VMQGzN6/2/zigkPV8kQGx/O/YdGTqdrdTnCgLEXkah42pcXaIP93TSczUBYl+j0GlHPo3R8kGS+nyFALF9b404LdHVB9W6cxMCxI4sl+iir8BNCBB7dNyRT2MkOt9HgNg7fQl5ISKQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICCzr3/Jus4b9QJnMgEBGQECMneHw3N3dWDXTEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECflT+B+/KMWGEOnf7AAAAAElFTkSuQmCC', 'sha256': '085dd5f61c72fa831b8edff0e545a1e4708512a298cf3ecdb9324b75d26ba95b'}, 'data/binding-v1/images/binding-evaluation-0043-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFSUlEQVR4nO3dO24bSRRAUWpg74eB096BV+IFeSWzA6UOuBgFApRMQJimx/pQH/N2V52T6tPVIPriVYmgbu7v73YAhX+SqwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dRdGkaw33+55NsOhx9/fy3bc3N/f1evAcaMzlPE6ESA4Erd+dNh+hIJELw3PbffH57/8eXb52e+epg4QwIEr07Pi8V5Q48OU2ZIgOB66Tm3yJAAwSX1+cDuvFiiw0wNEiDI0nNuzgx5IyL09dn9ca0P/4vbOpmA4JFn/prpeWYUGn4OMgHBiuqz+/3qw89BAgQrqs9sDRIgZre2+kzVIAFiauuszzwNEiDmteb6TNIgAWJS66/PDA0SIGa35vpsZYVvJkDM6DRKbOXZvv25zsGGIJ+IuNVXdPi3qP09G33FT/b7L8O8+iYg5rWV8WeLq72QADGXzW2+xt6I2YL98u/n5z62biW+PmzvsYGnmICYyKbHnyGHIAECMgLELMYYGQa7IwFiOtvdf42x/nMCBGQEiCkMcPw85FG0AAEZAQIyAgRkBIjxbf2gZOC7EyAmMsYJ9Ej3IkBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAMZHl2wb+8clU9yJAjG+Y/yM63t0JEJARICAjQEBGgJjC6aBkjLPb5eddbPoASICAkgAxna0PQcvG139OgJjF1ncrQ96RAAEZAWIiAxxFL6McPx8JEJARIOay6SFoGWv8ESCmtq0GLZta7YUEiOlsfXw4bHz95wSIGW1uI7YMt/k6EiBmt/4GLatf4ZsJEJM6HyXW/IQvZ2sbbPwRIKa2/gYtQ9dHgJjdmhu0jF4fAYKVNmiZoD4CBGts0DJHfQQIVtegZZr67Ha7T/UCYC0Ohx/7/ZfzCtx+f7jmApbfwzd8fUxAsHvmmb/mKLTMVx8TEPzf8cm/5ii0TJmeIwGCF7Zj5434wBItj41XU9VHgODSUegDByLpOREgeGOGTl7s0fNnSYcp03MkQPDGDL3/rPowcXqOBAhe3YunSvTa34MAwbsKcmGMROdRAgTvoizvIUC/fH246tteAe+EBjICBGRu7u/vuqsDUzMBARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKv8B8tXRYq2NA0tAAAAAElFTkSuQmCC', 'sha256': 'b28fe3c7f641958b4bb1290ef634120abb5cd8d9b4175fb4bfa18da7749cbcc0'}, 'data/binding-v1/images/binding-evaluation-0043-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFSUlEQVR4nO3dy20bSRRAUWpghTMAFwqjs5k4nI3C8IKAU9GCgDZeCKapsf6/2111zlafqgbRF69LBHVxPN7sAAr/JKsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnWLQ0j2O+vXvJth8OPz9/L9lwcjzf1HmDM6DxGjE4ECL6oO387TF8iAYL3puf68vLpH19ub5/46mHiDAkQvDo9zxbnDT06TJkhAYKvS8+5RYYECF5Snw/szrMlOszUIAGCLD3n5syQNyJCX5/dX2t9+F/c1skEBA/c81+ZnidGoeHnIBMQrKg+u/urDz8HCRCsqD6zNUiAmN3a6jNVgwSIqa2zPvM0SICY15rrM0mDBIhJrb8+MzRIgJjdmuuzlR2+mQAxo9MosZV7+/r3Pgcbgnwi4lZf0eHfovZ5NvqKn+z3V8O8+iYg5rWV8WeLu30hAWIum3v4GvtBzCPYH7f//btbvcvvP+stwIcxATGRTY8/Qw5BAgRkBIhZjDEyDHZFAsR0tvv8Ncb+zwkQkBEgpjDA8fOQR9ECBGQECMgIEJARIMa39YOSga9OgJjIGCfQI12LAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgLERJbb290oliGuRYAY3zD/R3S8qxMgICNAQEaAgIwAMYXTQckYZ7fL76vY9AGQAAElAWI6Wx+Clo3v/5wAMYutP60MeUUCBGQEiIkMcBS9jHL8fEeAgIwAMZdND0HLWOOPADG1bTVo2dRuX0iAmM7Wx4fDxvd/ToCY0eYexJbhHr7uCBCzW3+DltXv8M0EiEmdjxJrvsOXs70NNv4IEFNbf4OWoesjQMxuzQ1aRq+PAMFKG7RMUB8BgjU2aJmjPgIEq2vQMk19BAgebdDXZ2i5v+jw9REguOd/9/xXNmi5v9YM9dntdt/qDcC63N35+/3VeReuLy8/b8VlyvTcESB4wOHw49Sg80Z8YImWh8arqeojQPDSUegDByLpOREgeGOGTp7t0dNnSYcp03NHgOCNGXr/WfVh4vTcESB4dS8eK9Frfw8CBO8qyAtjJDoPEiB4F2V5DwH64/L7z3oLMBfvhAYyAgRkLo7Hm251YGomICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAnaVX2LeUWGA+hqLAAAAAElFTkSuQmCC', 'sha256': 'd99eabb0dd1f5949cbd6b3dcc52aca91386f30d9e870b23ac25cd3823106b358'}, 'data/binding-v1/images/binding-evaluation-0043-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFSUlEQVR4nO3dO24bSRRAUWpgLWcABkp7Q05nHU69IaUOCHgrCgQocUCYpse2/ubtrjon1aerQfTFqxJBXd3f3+0ACv8kVwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg86G7NIxgv795zrcdDl/+/lq25+r+/q5eA4wZnT8RoxMBggt151eH6UskQPDW9Nx+fnj8x5eP14989TBxhgQIXpyeJ4vzih4dpsyQAMHl0nNukSEBgufU5x2782SJDjM1SIAgS8+5OTPkjYjQ12f3y7Xe/S9u62QCgt8885dMzyOj0PBzkAkIVlSf3c9XH34OEiBYUX1ma5AAMbu11WeqBgkQU1tnfeZpkAAxrzXXZ5IGCRCTWn99ZmiQADG7NddnKyt8NQFiRqdRYivP9u33dQ42BPlExK2+osO/Re3v2egrfrLf3wzz6puAmNdWxp8trvaZBIi5bG7zNfZGzBbsh4f//t2t3vWnr/US4N2YgJjIpsefIYcgAQIyAsQsxhgZBrsjAWI6291/jbH+cwIEZASIKQxw/DzkUbQAARkBAjICBGQEiPFt/aBk4LsTICYyxgn0SPciQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAAxkeXj9W4UyxD3IkCMb5j/Izre3QkQkBEgICNAQEaAmMLpoGSMs9vl+11s+gBIgICSADGdrQ9By8bXf06AmMXWdytD3pEAARkBYiIDHEUvoxw/HwkQkBEg5rLpIWgZa/wRIKa2rQYtm1rtMwkQ09n6+HDY+PrPCRAz2txGbBlu83UkQMxu/Q1aVr/CVxMgJnU+Sqz5CV/O1jbY+CNATG39DVqGro8AMbs1N2gZvT4CBCtt0DJBfQQI1tigZY76CBCsrkHLNPXZ7XYf6gXAWhwOX/b7m/MK3H5+uOQClp/DN3x9TECwe+SZv+QotMxXHxMQ/N/xyb/kKLRMmZ4jAYIntmPnjXjHEi2/G6+mqo8AwXNHoXcciKTnRIDglRk6ebJHj58lHaZMz5EAwSsz9Paz6sPE6TkSIHhxL/5Uopf+HgQI3lSQZ8ZIdH5LgOBNlOUtBOiH609f6yXAXLwTGsgIEJC5ur+/664OTM0EBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwK7yDQWsRK3Rz8xTAAAAAElFTkSuQmCC', 'sha256': 'ecdedb706973358490b30961d68ae9e66c2839525efe14c6ef7b83eabeb9f0c9'}, 'data/binding-v1/images/binding-evaluation-0044-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFTElEQVR4nO3dTW5iRxhAUTpq9uNBpGyCcRaZsTcRqQcspgeWmGSAYiFs4198qapzxm2oJ/Suvyqg/ePh4fcGoPBH8qwAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjI/NxM5+7uz83g9vtf9RLgO5iAgMyEExBcdXY2n36hmQO0/eufzVAO//5dL2E5H9iwn/6IGH3SzAGCax8UPj6OEn2MALGWy+m5324v//jucLjwsDL0XgLE0ul5tTiX//1Zj2TovQSIFdPz3u685PFxTkskQ28nQMzseul59mFl6L18DohV6nO/3V6pPheeYoKPxV6VADGn0zv/G9Jz6uzpNOgCWzDmH3ySZdxvt487Mtuxl5iAmMqN1OfZZzcKPSVAzCPcdr3EduwyAWLO+mxuiQa9RICYwS3X50iDniVATOU263P7a6sIEMN7HChu/w5/XKEh6EiAGNu4d/LdsCv/QgLEwG7/6Ocph0GnBIgZjFKfEVd7VQLEqAY6+nnKYdCRAAEZAWJIQ48/R/eGIAECQgLEeCYYf47ulx+CBAjICBCDmWb8OVp8CBIgICNAQEaAGMlk+6+jlXdhAgRkBAjICBCQESCGscIRyd0C13hKgBjPTCfQs17RGwkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQIxndzhs5rKb7oreSIAYxn7/azO7/QLXeEqAgIwAARkBAjICxJBHJDOd2u7+v5bVDoAECCgJEJARIAYz2S5st/D+S4CAkgAxnmmGoN3a448AASUBYkgTDEG75ccfAQJKAsSohh6CjD9HAsQMxmrQWKu9qh8PD783c5ngr2sv/lvxwy/3EH/g+LQ+++VfaBMQYxv3Ht4Pu/IvJEAMb6DDIEc/Z35upuOlXdnucLjZjdjt9/H7mYCY7bfObd7njn6eJUBM4pYbpD4LvQvGys7eA823Y2cpVJ8zJiCmcnaHt6OQ+rzKBMSc8s8H2Xa9hQAxrWo7ZvB5OwFiZk8/Fn/VDD3d8anPZQLE/L4hQ9LzMQLEKp79kuAnS/TsIbf0vJ0AsZbL31V+tUeX31aTnvcSIFb05f9lgvR8jACxtE+WSHc+SYDgfTESnS8kQEDGVzGAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEbCr/AVLWM93QmbvDAAAAAElFTkSuQmCC', 'sha256': 'a35ea0dface8e257fd63cde53c2c20f0462d19669fd199b5e2ad38b05e0baf35'}, 'data/binding-v1/images/binding-evaluation-0044-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFTElEQVR4nO3dsW5aSRiAUbIKj7MSRdq80Lb7HGn3hbZNgeRXSWHJzRZoLYTJNdjGHzP3nDqGuUL38z8DxF8eH39tAAp/JM8KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzNfNdHa7b5vB7fc/6yXAZzABAZkJJyC46exsPv1AMwfo33+eNkP5/te2XsLqvGHDfvwjYvROMwcIbn1Q+Pw4SvQ2AsS6LKfn6e8/l398++Nh4WFl6FoCxKrT82pxlv/9SY9k6FoCxBrTc213fuf5cY5LJEOXEyBmdrv0nH1YGbqWALGW+twoPa9mSIMWCBDz1+cT0rOQIQ1a4JPQzGa3+xbW5+zzniyJZwLEVD5/27Xg5Nk16CVbMOZxD4PPCduxZSYgJnGH9fnddixdy30RIGZwz/U50KCzBIip3Gd97n9tFQFieM8Dxf3f4c8rNAQdCBBjG/dO3g278g8kQAzs/o9+XnIYdEyAmMEo9RlxtTclQIxqoKOflxwGHQgQkBEghjT0+HPwZAgSICAkQIxngvHn4Gn1Q5AAARkBYjDTjD8HKx+CBAjICBCQESBGMtn+62DNuzABAjICBGQECMgIEMNYwxHJbgXXeEyAGM9MJ9CzXtGFBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgxrP98bCZy3a6K7qQADGM/f7nZnb7FVzjMQECMgIEZAQIyAgQQx6RzHRqu/3/WtZ2ACRAQEmAgIwAMZjJdmHbFe+/BAgoCRDjmWYI2q57/BEgoCRADGmCIWi7+vFHgICSADGqoYcg48+BADGDsRo01mpv6svj46/NXCb469or/6345pd7iD9wfFyf/epfaBMQYxv3Ht4Pu/IPJEAMb6DDIEc/J75upuOlXbPtj4e73Yjdfx8/nwmI2X7r3Od97ujnLAFiEvfcIPVZ0btgrNnJe6D5duwkhepzwgTEVE7u8HYUUp9XmYCYU/75INuuSwgQ06q2YwafywkQM3v5sfibZujljk99lgkQ8/uEDEnP2wgQa3H2S4LvLNHZQ27puZwAsS7L31V+tUfLb6tJz7UEiDX68P8yQXreRoBYtXeWSHfeSYDguhiJzgcSICDjqxhARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECNpX/AOk7Q3DipGKCAAAAAElFTkSuQmCC', 'sha256': 'efd62ea7e133ae6c85f0430a47acda8f8d819275cccd0674d3fa0905f039b6cd'}, 'data/binding-v1/images/binding-evaluation-0044-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFTElEQVR4nO3dTW5iRxhAUTpq9uNBptkBK8mCshLvINMesJgeWGKSAYqFsI1/8aWqzhl3Qz2hd/1VAfaPh4ffG4DCH8mzAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPNzM527uz83g9vvf9VLgO9gAgIyE05AcNXZ2Xz6hWYO0L//HDZD+evvbb2E5Xxgw376X8Tok2YOEFz7oPDxcZToYwSItVxOz/32lSF0dzhceFgZei8BYun0vFqcy//+rEcy9F4CxIrpeW93XvL4OKclkqG3EyBmdr30PPuwMvRePgfEKvW5326vVJ8LTzHBx2KvSoCY0+md/w3pOXX2dBp0gS0Y8w8+yTLut9vHHZnt2EtMQEzlRurz7LMbhZ4SIOYRbrteYjt2mQAxZ302t0SDXiJAzOCW63OkQc8SIKZym/W5/bVVBIjhPQ4Ut3+HP67QEHQkQIxt3Dv5btiVfyEBYmC3f/TzlMOgUwLEDEapz4irvSoBYlQDHf085TDoSICAjAAxpKHHn6N7Q5AAASEBYjwTjD9H98sPQQIEZASIwUwz/hwtPgQJEJARICAjQIxksv3X0cq7MAECMgIEZAQIyAgQw1jhiORugWs8JUCMZ6YT6Fmv6I0ECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDGszscNnPZTXdFbyRADGO//7WZ3X6BazwlQEBGgICMAAEZAWLII5KZTm13/1/LagdAAgSUBAjICBCDmWwXtlt4/yVAQEmAGM80Q9Bu7fFHgICSADGkCYag3fLjjwABJQFiVEMPQcafIwFiBmM1aKzVXtWPh4ffm7lM8Ne1F/+p+OGXe4g/cHxan/3yL7QJiLGNew/vh135FxIghjfQYZCjnzM/N9Px0q5sdzjc7Ebs9vv4/UxAzPZT5zbvc0c/zxIgJnHLDVKfhd4FY2Vn74Hm27GzFKrPGRMQUzm7w9tRSH1eZQJiTvnng2y73kKAmFa1HTP4vJ0AMbOnH4u/aoae7vjU5zIBYn7fkCHp+RgBYhXPfknwkyV69pBbet5OgFjL5e8qv9qjy2+rSc97CRAr+vJfmSA9HyNALO2TJdKdTxIgeF+MROcLCRCQ8VUMICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZvKf0/zM91ueVoAAAAAAElFTkSuQmCC', 'sha256': 'b5967e1158da1bf53bd3de24c2e1f49ccdf8a0984b40bf88c109673d5efc3688'}, 'data/binding-v1/images/binding-evaluation-0045-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFaUlEQVR4nO3cTWobSRiAYWWIjjOgRSA3yUXmHHOR3CSQhWCukoWhN7MQY4Qc/02kvKqq51l5YXd/wu6X6urGHx4efuwACn8kZwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMh93UzgcPu0mdTx+r0eAW7ECAjICBGQmuQV7tP31524K+7//qUeAm7MCAjICBGQECMjMtgcE7QsfXpt4FwGCa75idvH9evQyAYIbvt16OogMPUeA4B3d2X/++vKPb9++vHBMJbogQPBKel6NznPf/DRGFkQXBAh+Xp93deenHo9wUaLD4ZMGnQgQq7tFen56wPMMWQqdeA+IpV3UZ//569Xr88LBD/P+F4c3EiDWdX793zQ95y5OdFi7QQLEoi7q85vPrkEnAsSK2vo8Pe9h1QYJEMt5vNp/223Xc/ZnA6zZIAFi3frs7sN+4QYJEAu5w/os3iABYhWjXNuHQea8CgFiCfew6/yy/ZJ70gLEWu6zPvc/240IEPO7262fp1bbDBIgJjfulXwYdvK3EyBWcf/Ln7HmvAoBYmajLyIOg8//KgFiCWMtK/ZDTfsrBAjICBDTGujh17KPwwQIyAgQc5pp4XCY6LNcECAmN+L91+iTv50AARkBAjICBGQEiAkN/QB+qYfxAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQEzoev5++2L592Y1s+2/+x080GQECMgIEZAQIyAgQkxt3G2gbdvK3EyDmNNOu7XGiz3JBgICMADGtoR/Gb7M/gD8RICAjQCxhrEXQNtS0v0KAmNno9y/Hwed/lQCxilGWFdsgc16FADG5cRcRx2EnfzsBYn4DPQ7b1nj49UiAWMs9N2i749luRIBYwvmC4j6v8+1sqkWWPwLEQka5qo+DzHkVAsRC7nYzaFts6+eRALGWO2zQtmp9BIjVG9RmaDsbYMH6CBCLuoc96W3JXecLAsSi2gapz8mHh4cfu/EdDp92k1r5r/P3/+XsP3+99RkvYndc+/drBcTSLq7/m+4KPT34ce367Ha7j/UAEDtV4HwpdMrEFVdDT6MmPVPdgsHtbuT/d4meW0ypzyMBgnfsJ74ao5fv4KTnggDBzZ9s6M5zBAhumCHpeZkAwTV7pDjvIkBAxntAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAXeVfEy8nrN0IoVcAAAAASUVORK5CYII=', 'sha256': 'c7d993e8110a07eebd318f92dae1555f838c27481b69eee3a25cbb3aa901632d'}, 'data/binding-v1/images/binding-evaluation-0045-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFYElEQVR4nO3cPVJbSRhAUXnK3o8Cp29DXpA3REqgxTigyokDzahUAvMzIK66+5yIgJI+Fbxb/fo1fHl4+LUDKPyTvCuAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPm6m8J+/303qcPhvh4BrsUKCMgIEJCZ5Bbs5O7n790Uth/f6hHg6qyAgIwAARkBAjKz7QFBe+DDsYk3ESD4yCNmF9+vR88TILji6dbji8jQ3wgQvKE7L57zePL8xOk1leiCAMEL6XnT4bLzb34cIwuiCwIET9fn/YdaT69wUaL9/rsGHQkQq7tGep58wfMMWQodCRBLu6jPVf+U58kMHdZukACxrvP6fNpfEV5kaL92g5yEZlFJfZ58x/28/83qRQLEitr6PH7f/aoNcgvGck5Xe/7PW+7ObsfWvBezAmItt1Ofk9MkC66DBIiF3GB9Fm+QALGKUa7t/SBzfggBYgm3sOv8vLsl96QFiLXcZn1uf7YrESDmd7NbP4+tthkkQExu3Ct5P+zkrydArOL2lz9jzfkhBIiZjb6I2A8+/4sEiCWMtay4G2ra9xAgICNATGugh1/LPg4TICAjQMxppoXDfqLPckGAmNyI91+jT/56AgRkBAjICBCQESAmNPQD+KUexgsQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEzocLg/frH9+LYb2fbf/KdPNBkBAjICBGQECMgIEJMbdxtoG3by1xMg5jTTru1hos9yQYCAjAAxraEfxm+zP4A/EiAgI0AsYaxF0DbUtO8hQMxs9PuXw+Dzv0iAWMUoy4ptkDk/hAAxuXEXEYdhJ389AWJ+Az0O29Z4+HUiQKzllhu03fBsVyJALOF8QXGb1/l2NtUiyx8BYiGjXNWHQeb8EALEQm52M2hbbOvnRIBYyw02aFu1Prvd7ms9AHy2w+F+v/9+uvLvfv6uJtmW3Pc5ZwXEim5hT3pbvj4CxLraBqnP0ZeHh1+78R1X1FNa+bfz839zPuF27CJ2h7V/vvaAWNppP+joqrtCj9dZh7XrI0DwbwWumiHpmfwWDK53I/+/S/S3rSX1OREgeMN+4osxen4/W3ouCBBc/cmG7vyNAMEVMyQ9zxMg+MgeKc6bCBCQcRIayAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCwq/wB9+8uun1H994AAAAASUVORK5CYII=', 'sha256': '8df95e284c9955a3a4432a4ec8539d5266613c93c880ed3e0c7b778e038f5959'}, 'data/binding-v1/images/binding-evaluation-0045-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFaklEQVR4nO3cPW4bRxiAYTow78PCgKtcw4fIgXIIXyOVARc8jAsB26QgIhCU9ReTfjkzz1OpkHY/QtoXs7MLfXh4+LEDKPyRnBVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyHzcTeFw+LSb1PH4vR4BbsUKCMgIEJCZ5Bbs0T9/b7sp/PnXvh4Bbs4KCMgIEJARICAz2x4QtC98eG3iXQQIrvmK2cX369HLBAhu+Hbr6SAy9BwBgnd0Z//568s/vn378sIxleiCAMEr6Xk1Os9989MYWRBdECD4eX3e1Z2fejzCRYkOh08adCJArO4W6fnpAc8zZCl04j0glnZRn/3nr1evzwsHP8z7XxzeSIBY1/n1f9P0nLs40WHtBgkQi7qoz28+uwadCBArauvz9LyHVRskQCzn8Wr/bbddz9mfDbBmgwSIdeuzuw/7hRskQCzkDuuzeIMEiFWMcm0fBpnzKgSIJdzDrvPL9kvuSQsQa7nP+tz/bDciQMzvbrd+nlptM0iAmNy4V/Jh2MnfToBYxf0vf8aa8yoEiJmNvog4DD7/qwSIJYy1rNgPNe2vECAgI0BMa6CHX8s+DhMgICNAzGmmhcNhos9yQYCY3Ij3X6NP/nYCBGQECMgIEJARICY09AP4pR7GCxCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNATOh4/H76Yvv2ZTey7b/5Hz/RZAQIyAgQkBEgICNATG7cbaBt2MnfToCY00y7tseJPssFAQIyAsS0hn4Yv83+AP5EgICMALGEsRZB21DT/goBYmaj378cB5//VQLEKkZZVmyDzHkVAsTkxl1EHIed/O0EiPkN9DhsW+Ph1yMBYi333KDtjme7EQFiCecLivu8zrezqRZZ/ggQCxnlqj4OMudVCBALudvNoG2xrZ9HAsRa7rBB26r1ESBWb1Cboe1sgAXrI0As6h72pLcld50vCBCLahukPicfHh5+7MZ3OHzaTWrlv87f/5ez//z11me8iN1x7d+vFRBLu7j+b7or9PTgx7Xrs9vtPtYDQOxUgfOl0CkTV1wNPY2a9Ex1Cwa3u5H/3yV6bjGlPo8ECN6xn/hqjF6+g5OeCwIEN3+yoTvPESC4YYak52UCBNfskeK8iwABGe8BARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECdpV/AaRwJ6wq03gmAAAAAElFTkSuQmCC', 'sha256': 'a422a417846ed2f1d266f30a56cb0f2f315f8df634f86a360811eb9d340aecb9'}, 'data/binding-v1/images/binding-evaluation-0046-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFSElEQVR4nO3bTU4bSxhAUecpLCeSByzj7SbryG7eMjKwlK1kgMTkDZAsZMD4l1vVfc7Y2NWCvnxVLX97evq7ASj8k3wqgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIy3zdrst0+bhZtt/tdLwHOYAICMgIEZNa1Bdv77+FhsyD/Pj/XS4BLrDRArNyJp4HO1O5NgFiLCx5BvP4RMboHAWLhbvXoc/8+SnRDAsRK0/P888fxH3/49efI28rQTQgQa0nPp8U5/vqDHsnQTQgQC0/Pud35yP59XpdIhq4kQCzE/dLz7tvK0E0IEAusz53S82mGNOhcAsSi6vMF6TmSIQ06l69iMLHt9jGsz7ufe7AkjhMgZvX1264jDj5dg05kC8aURhh8DtiOXcAExHwGrM9H27F0LRMQICYzcn1eaNDpBIhZjVmf8dc2FAFiJvuBYvw7fL9CQ9ARAsQ05r2T5135vQkQcxj/6Octh0GfEiAmM0t9Zlzt1xMgJjDR0c9bDoOOECAgI0CMburx54Uh6CMCBGQEiKEtYPx5YQh6lwABGQFiXIsZf14Ygt4SICAjQEBGgBjUwvZfL+zCDggQkBEgICNAQEaAGNEajki2K7jGTwkQQ1vSCfRSr+gaAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQQ3v49WezLMu7omsIECPa7X5vlm63gmv8lAABGQECMgIEZASI0Y9IlnRqu78WB0AvBAjICBCQESDGtbBdmP3XWwIEZASIoS1mCDL+vEuAgIwAMboFDEHGn48IEJARICYw9RBk/DlCgJjMXA2aa7VfT4CYw+vxYZa7+vU6jT/vEiCmMe89PO/K702AmMlEh0GOfk4hQMxq5AaNvLahCBCTGf8wyNHP6QSI+YzcIPU5y/fzXg5j2O1+b7ePr+/5558/2iUdpFB9TmECYlYHd3g7CqnPZb49Pf3drMb+f+ZSrfPv/vWvNZmDbLsuZgvG9MLtmMHnSgLE0hr0NRl6u+NTnwusawvGCnfZN8+Q9NyQALGWw74rS/TuIbf0XEmAWOkzh097dPyxmvTchACxcDd/9Ck9NyRArMWVJdKdexAg1ujEGInOvQkQkPFVDCAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DApvI/R7BA+DUq5yEAAAAASUVORK5CYII=', 'sha256': '9b38e1680bed86b1716a05d1293f0ff9c26c118ce5015c58a56d883e89aa806e'}, 'data/binding-v1/images/binding-evaluation-0046-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFSUlEQVR4nO3bPU4rSRhAUb/ReD8Ek84O3kpmQbMSdjDpC1jMC5BIJkCyLGz8g21uVfc5MXRXC/rqqzL8eH39vQEo/JHcFUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmT83a/L09Ndm0V5eftVLgCuYgICMAAGZdW3Bdv77922zIH//s62XAF+x0gCxcheeBjpTezQBYi2+8BHE/reI0SMIEAt3r48+d9dRojsSIFaanuftmYOzn29vJy4rQ3chQKwlPWeLc/rrP/RIhu5CgFh4eq7tzmd219kvkQzdSIBYiMel5+hlZegu/CEiC6zP83b7oPqcuMXi/9HnEQSI6e2/+d+Qnn0fbqdB17IFY1GDT7KM5+12tyOzHbuKCYhZDVKfo3c3Cl1IgJhSuO36jO3YFwgQ09dnMxINuooAMZmR6/NOgy4nQMxqzPqMv7ahCBAz2Q0U47/huxUagk4QIKYx75s878ofTYCYw/hHP4ccBp0lQExmlvrMuNrvJ0BMYKKjn0MOg04QICAjQIxu6vHnnSHoMwIEZASIoS1g/HlnCDpKgICMADGuxYw/7wxBhwQIyAgQkBEgBrWw/dc7u7APBAjICBCQESAgI0CMaA1HJE8reMazBIihLekEeqlPdAsBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBIih/Xx72yzL8p7oFgLEiF5efm2W7mUFz3iWAAEZAQIyAgRkBIjRj0iWdGq7exYHQO8ECMgIEJARIMa1sF2Y/dchAQIyAsTQFjMEGX+OEiAgI0CMbgFDkPHnMwIEZASICUw9BBl/ThAgJjNXg+Za7fcTIOawPz7M8lbvr9P4c5QAMY153+F5V/5oAsRMJjoMcvRzCQFiViM3aOS1DUWAmMz4h0GOfi4nQMxn5Aapz1UEiCU0aIQMfViG+lxCgJjVhze8bdCHu6vPhX68vv7erMbT01+bRVvn7/3+j/V5u/3+BRh8vswExPTC7Zht140EiGVuxx6docNbqM8XrGsLxgp32XfflB2mTXq+TIBYy2HfjSU6OlJJz40EiJV+5nC2R6c3cdJzFwLEwt39o0/puSMBYi1uLJHuPIIAsUYXxkh0Hk2AgIy/AwIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEbCr/A4p2QCNIXz8AAAAAAElFTkSuQmCC', 'sha256': 'a7e04a3f3543212bf71d13a30fda741b46b8ef76de5de7007928a8883e0b2a36'}, 'data/binding-v1/images/binding-evaluation-0046-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFS0lEQVR4nO3bMU4jSRiAUc9qOM5IDkj3QpvOOSbdC21KYGmuMgESyQZIFjJgbGzzVXW/Fxu7WtAff1XL3x4f/2wACn8lnwogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMz3zZpst/ebRdvtHuolwBlMQEBGgIDMurZge//9+7RZkL//uauXAJ+x0gCxcieeBjpTuzUBYi0+8Qji5Y+I0S0IEAt3rUef+/dRoisSIFaanqefP47/+N2v30feVoauQoBYS3o+LM7x1x/0SIauQoBYeHrO7c579u/zskQydCEBYiFul54331aGrkKAWGB9bpSeDzOkQecSIBZVny9Iz5EMadC5fBWDiW2392F93vzcgyVxnAAxq6/fdh1x8OkadCJbMKY0wuBzwHbsE0xAzGfA+ry3HUvXMgEBYjIj1+eZBp1OgJjVmPUZf21DESBmsh8oxr/D9ys0BB0hQExj3jt53pXfmgAxh/GPfl5zGPQhAWIys9RnxtV+PQFiAhMd/bzmMOgIAQIyAsToph5/nhmC3iNAQEaAGNoCxp9nhqA3CRCQESDGtZjx55kh6DUBAjICBGQEiEEtbP/1zC7sgAABGQECMgIEZASIEa3hiGS7gmv8kAAxtCWdQC/1ii4hQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAAxtLtfvzfLsrwruoQAMaLd7mGzdLsVXOOHBAjICBCQESAgI0CMfkSypFPb/bU4AHomQEBGgICMADGuhe3C7L9eEyAgI0AMbTFDkPHnTQIEZASI0S1gCDL+vEeAgIwAMYGphyDjzxECxGTmatBcq/16AsQcXo4Ps9zVL9dp/HmTADGNee/heVd+awLETCY6DHL0cwoBYlYjN2jktQ1FgJjM+IdBjn5OJ0DMZ+QGqc9Zvp/3chjDbvew3d6/vOeffv5ol3SQQvU5hQmIWR3c4e0opD6f8+3x8c9mNfb/M5dqnX/3L3+tyRxk2/VptmBML9yOGXwuJEAsrUFfk6HXOz71+YR1bcFY4S776hmSnisSINZy2Hdhid485JaeCwkQK33m8GGPjj9Wk56rECAW7uqPPqXnigSItbiwRLpzCwLEGp0YI9G5NQECMr6KAWQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjYVP4HkAlA+CL6t1EAAAAASUVORK5CYII=', 'sha256': '2e81f5cc0895c523f695c80aff4fd681e099a77ac86856188853cff8469b2484'}, 'data/binding-v1/images/binding-evaluation-0047-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFaklEQVR4nO3dsW0bWRRAUXph9cNggY2mDRfhgraIbUORAQcsxoEAJxsIJgRSlChL5p35/5zIgUy9Cebi/eEA+vTw8GMHUPgr+a0AAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPN5N6X9/u/dBh0O3+sR4CPZgICMAAGZSY9gR3f//LdbvZ/fvtQjcNXJ3Rn5rWYPEHzgs8KTn9ejVwkQ/KnvKI6fo0SXCBBcm577f3++/N+Xr3cvfKwMnRMgeCk9r0bn0g+fx0iGzgkQs3s2PW/qzrOOn3BSIhl6SoCY2kl93t+dK0u03/+tQd4DYmo3qM8Ln7/f5uv4H8sGxIxunJ6TX3RchfbTH8dsQEynqs+l37ifeBWyATGXp3f77dPzwip0mHIPsgExkZXU59kZ9lPuQQLELNZWn0eTN0iAmMI66/No5gYJEHNZW33WPNUNCBDjO64Va77P73/NNtUSJEAMbov3836DM/8eAWJka370c27Ch0ECxBTWX59tzflRBIhhbX2J2G98/msIEOPb1lpxv6lp30mAGNMmvvm6ZJ5vxAQIyAgQAxppcdgPdC3nBIiRbfH8tfXJ30SAgIwAMZrxziz74a7oSIAY1tZPMfcbn/8aAgRkBAjICBCQESCGsukXoCd8JVqAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAWIoh8P3x38sX+9227f8uorjdQ1GgICMAAEZAQIyAsSwtv4YaNn4/NcQIEYz3vPaw3BXdCRAQEaAGNl2TzHLZid/EwFiQCOdWQ4DXcs5AQIyAsSYNv1K9DL6C9BHAsT4ttWgZVPTvpMAMaytrw+Hjc9/jc+7uf389qUegVtYvt5t4i+FLTOtPzYgBvd0iVj/vb08mXCG9UeAGN8W7+TDBmf+PZ8eHn7UM8Aft/4/2bxM883XUzYg5rLOg9iyyqluQICYwpofBi3zPfo5EiBmsc4GLRPXR4CYy9oatMxdHw+hmfqBdPhY+iR/hynrYwNiRid3++1XIfU5sgExr9uvQtJzwgbEvG68CqnPORsQszvZgz58G3q2a+rzSIDgYobeU6JL+5T0PCVA8HqGrozRy4c46TknQPC2DP0G6blEgOBPlUh3XiVA8GE9Upy3EiAg4z0gICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECdpX/Ab2RMmlI+yA5AAAAAElFTkSuQmCC', 'sha256': 'cdac2f40dba1244eeebc535e3c54dd996dad28624d60a64b5ffef849e0e895d9'}, 'data/binding-v1/images/binding-evaluation-0047-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFaklEQVR4nO3dvW0bWRSAUXph9sPAgCO34SJc0BbhNjYy4IDFOBAwyQaCCYHUD2XJ/GbeOydSIJF3An64bziAPtzd/doBFP5J3hVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZD7upnQ4fNpt0PH4sx4B3pMNCMgIEJCZ9Ah28t+/y271vnzb1yNw1cndGfm1Zg8QvOO9wrPf16MXCRD8re8oTq+jRE8RILg2PfvP35//8+XH12deVoYuCRA8l54Xo/PUL1/GSIYuCRCzezQ9r+rOo06vcFYiGXpIgJjaWX3e3p0rS3Q4fNIgzwExtRvU55nXP2zzcfz3ZQNiRjdOz9kbnVahw/THMRsQ06nq89Q7HiZehQSIuTz8tO8/f799fR5968OsDRIg5q1POsv5DHM2SICYxdrqc2/yBgkQU1hnfe7N3CABYi5rq8+ap7oBAWJ8p7VizZ/z/e/ZplqCBIjBbfHzfNjgzH9GgBjZmm/9XJrwZpAAMYX112dbc74XAWJYW18iDhuf/xoCxPi2tVbsNzXtGwkQY9rEN19PmecbMQECMgLEgEZaHA4DXcslAWJkWzx/bX3yVxEgICNAjGa8M8thuCs6ESCGtfVTzH7j819DgICMAAEZAQIyAsRQNv0A9ISPRAsQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQAzlePx5/8Py4+tu+5bfV3G6rsEIEJARICAjQEBGgBjW1m8DLRuf/xoCxGjGu197HO6KTgQIyAgQI9vuKWbZ7OSvIkAMaKQzy3Gga7kkQEBGgBjTph+JXkZ/APpEgBjfthq0bGraNxIghrX19eG48fmv8XE3ty/f9vUI3MLy4+sm/lPYMtP6YwNicA+XiPV/tpcHE86w/ggQ49viJ/m4wZn/zIe7u1/1DPDXrf9fNi/TfPP1kA2IuazzILascqobECCmsOabQct8t35OBIhZrLNBy8T1ESDmsrYGLXPXx01opr4hHd6WPsvfccr62ICY0dmn/farkPqc2ICY1+1XIek5YwNiXjdehdTnkg2I2Z3tQe++DT3aNfW5J0DwZIbeUqKn9inpeUiA4OUMXRmj5w9x0nNJgOB1GfoD0vMUAYK/VSLdeZEAwbv1SHFeS4CAjOeAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQI2FX+B5vyIs8GmQB9AAAAAElFTkSuQmCC', 'sha256': '16cc52be8e1bf7b9b5ed6ff519bdcc6f2e53ad099de3dc4c465500ebd52358d6'}, 'data/binding-v1/images/binding-evaluation-0047-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFXklEQVR4nO3dMW4bORiAUWeR3EdF2rlQDrQXcptCh0lhYJstjAiCZMd2bOsbku9VKQyJA3i+/OQM4C8PD7/uAAr/JN8KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjJf75Z0OHy/G9Dx+LNeAnwkExCQESAgs+gW7OT+3//udm/78a1eAq/audsjv9XqAYIPPCu8+Hk9epEAwWc9ozh9jhI9R4Dgtel5ccP+3Gb58WNl6JoAwZ/S86ZTwvMfvo6RDF0TIFb3ZHre/3Ti9AkXJZKhcwLE0i7q8xlPRZ8s0eHwXYO8B8TSblCfP3z+YczX8T+WCYgV3Tg9F190GoUOy2/HTEAsp6rPc994WHgUMgGxlvO7PXwP/noUOi45B5mAWMhO6vPkGg5LzkECxCr2Vp9HizdIgFjCPuvzaOUGCRBr2Vt99ryqGxAg5ncaK/Z8n9//XttSQ5AAMbkR7+fDgGv+OwLEzPZ89HNtwcMgAWIJ+6/PWOv8KALEtEYfIg6Dr/81BIj5jTVW3A+12ncSIOY0xJOv56zzREyAgIwAMaGZBofDRNdyTYCY2Yj7r9FX/iYCBGQEiNnMt2c5THdFJwLEtEbfxdwPvv7XECAgI0BARoCAjAAxlaFfgF7wlWgBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBIipHI8/H/+x/fh2N77t91WcrmsyAgRkBAjICBCQESCmNfox0Db4+l9DgJjNfOe1x+mu6ESAgIwAMbNxdzHbsCt/EwFiQjPtWY4TXcs1AQIyAsSchn4lepv9BegTAWJ+YzVoG2q17yRATGv08eE4+Ppf4+vd2pb632Zl249vQ/ylsG2xX0gTEDM7HyL2f29vZytcYfwRIOY34p18HHDNf+fLw8Oveg3w6fb/J5u3ZZ58nTMBsZZ9bsS2Xa7qBgSIJez5MGhb7+jnRIBYxT4btC1cHwFiLXtr0LZ2fRxCs/SBdHgsfZG/45L1MQGxoou7/fajkPqcmIBY1+1HIem5YAJiXTcehdTnmgmI1V3MQR8+DT3ZNfV5JEDwbIbeU6Ln5inpOSdA8HKGXhmjP2/ipOeaAMHbMvQXpOc5AgSfVSLdeZEAwYf1SHHeSoCAjPeAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIuKv8D743MPujMM5ZAAAAAElFTkSuQmCC', 'sha256': '0b9efa246fc3d890fce4df4d3d9166d8aef6592b339c22a9718a433862bf16e4'}, 'data/binding-v1/images/binding-evaluation-0048-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFElEQVR4nO3dPW7jZhRA0XEw2o8Kt7ODrGQWNCvxDtKm8GJcCJgmCARoFFumor+5H8lzWlviK8yLR4qGnna7ty8AhT+SowIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5mt36MXabp+nf+H19e/fNQsM7Wm3e6tnWEV0PiNGrJkANd35SIlYIQG6f3peNpvp1/758+dnP5IhVkWA7pOes9G5KEYyxEoI0K31uTo90xnSINZAgK6vz13SM5EhDWLxBGiI9ByTIdbDg4hj1efj+9/xgzYYjQ3ojOPz/9HpmViF7EEskgANWp89DXqEmS6Vr0v8A3AJNm593h13pqcNTBCg86r6jHB0eCj/jHraYd0Y4fx/2Wz212Lb7fMi9/DKXz8+fSR9HN++93+Bj2MDOmHki52RZ4NLCdCIt34+cjOIRRKgT41TnzHngdsJ0H/MZbmYy5wwTYDmtG6MORVcTYDmulbMa1o4SYBmtmiMPBtcSoCAjACN+OThtMOErsKYOwECMgIEZAQIyAjQzG4A7bkNxDIIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0D/OnzZ1vFXIY/sMKevCWPWBAjICBCQESAgI0Dzuw3kBhCLIUBARoBOGHkJGnk2uJQA/TKvK5p5TQsnCdCcFo0xp4KrCdAs14q5zAnTBGg268Zo88DtBGhquRjnnD+exPrDYgjQCSOf4SPPBpcSoBk8l+jJQ5ZKgM5rGzRCAeFBBGjom0Fu/bBsAjRug9SHxROgyxr0ezL07kDqw1IJ0Hnvzv9HN+jd+6sPC/a1HmAe9hXYbp+PG/Gy2dz3KNLD2tiAblqF7rUNfXwr9WENBOgyH7twY4ZOvlx9WAmXYLdeju0dR+TspdlEsKSHVRGge2Zo77qFSHpYIQG6yaEaJ0t00TvACgnQfRx35GyMRAf2BOj+9AX+J5+CARkBAjICBGQECMi4Cc16fft+5//m41I2ICAjQEDmabd7644OrJoNCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIOBL5R+ZAy+Hzq8SbgAAAABJRU5ErkJggg==', 'sha256': 'e70ebed8a74ed5efe3cc705203087411f8a4eb55e5e34961c9fcbbdf33c8c601'}, 'data/binding-v1/images/binding-evaluation-0048-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEElEQVR4nO3dPXITWRSAUUzBfhyQsgNWMgtiJeyAlECLcaAqJ9SUqjQaS26hP77X3eekxtYN3F/dfjT003b78gGg8DH5VAABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfOo+erGen79M/4HN5tffmgWG9rTdvtQzrCI67xEj1kyAmu4cUyJWSIDun56f31+nv/frP5/f+5IMsSoCdJ/0nI3ORTGSIVZCgG6tz9Xpmc6QBrEGAnR9fe6SnokMaRCLJ0BDpOeQDLEeHkQcqz7HP/+Of9EGo7EBnXF4/T86PROrkD2IRRKgQeuzo0GPMNOlcrPEXwC3YOPW583nzvSygQkCdF5VnxE+HR7KP0Y9bb9ujHD9//z+ursXe37+ssg9vPLj87uPpI/j22v/G/g4NqATRr7ZGXk2uJQAjXj0c8xhEIskQO8apz5jzgO3E6D/mctyMZc5YZoAzWndGHMquJoAzXWtmNe0cJIAzWzRGHk2uJQAARkBGvHJw2n7Cd2FMXcCBGQECMgIEJARoJkdAO04BmIZBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBGgf+1ftnX4KuSR7ef0mjBmTYCAjAABGQECMgI0v2MgB0AshgABGQE6YeQlaOTZ4FIC9J953dHMa1o4SYDmtGiMORVcTYBmuVbMZU6YJkCzWTdGmwduJ0BTy8U41/zhJNYfFkOAThj5Ch95NriUAM3guURPHrJUAnRe26ARCggPIkBDHwY5+mHZBGjcBqkPi/epHmB0m82v/fvXd0X4C++PfxM79WGpbEDnvbn+H70KqQ/rYQP6I7sKPHoVkh7WRoCuvB27b4aOtyr1YQ0E6KYG3Z6hkzd06sNKCNCtt2PHHTkbo4lTJOlhVQTonhm65ZRaelghAbrJvhonS3TRT4AVEqD7OOzI2RiJDuwI0P3pC/whDyICGQECMgIEZAQIyDiEZr2+vT78PzZgmg0IyAgQkHnabl+6TwdWzQYEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQ8KHyGzyrGUem0nTKAAAAAElFTkSuQmCC', 'sha256': 'd47e263bc261f8164ba541e20209ef1247de5545586617a440c59c3ad6d6401b'}, 'data/binding-v1/images/binding-evaluation-0048-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFBElEQVR4nO3dQU4jVxRA0SZq78cDlpHlsowMWAwDSz2JIkvEAVOOsd33/6pzpoDrSVBXr4oCPx0Obz8ACn8kRwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzM/u0Ku13z8vf8Lr61+/axYY2tPh8FbPsInofEWM2DIBarrzmRKxQQJ0//S87HbLX/vnr19ffUiG2BQBuk96LkbnqhjJEBshQLfW59vpWc6QBrEFAvT9+twlPQsZ0iBWT4CGSM8pGWI7PIg4Vn0+v/4df9EGo7EBXXB6/j86PQurkD2IVRKgQetzpEGPMOlS+brGHwCXYOPW58NxJz1tYIEAXVbVZ4Sjw0P5Y9Tz3teNEc7/l93ueC223z+vcg+vjPDNvWjhufkVsAGdMfLFzsizwbUEaMRbP5+5GcQqCdCXxqnPmPPA7QToP2ZZLmaZE5YJ0EzrxphTwbcJ0KxrxVzTwlkCNNmiMfJscC0BAjICNOKTh8veJ3QVxuwECMgIEJARICAjQJPdADpyG4h1ECAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaA/vH+ZluzvAfT+5zeJoypCRCQESAgI0BARoDmuw3kBhCrIUBARoDOGHkJGnk2uJYA/WuuK5q5poWzBGimRWPMqeDbBGjKtWKWOWGZAE2zbow2D9xOgJaWi3HO+dNJrD+shgCdMfIZPvJscC0BmuC5RE8eslYCdFnboBEKCA8iQEPfDHLrh3UToHEbpD6sngBd16Dfk6EPB1If1kqALvtw/j+6QR9eX31YsZ/1AHM4VmC/fz5txMtud9+jSA9bYwO6aRW61zb0+aXUhy0QoOt87sKNGTr75erDRrgEu/Vy7Og0IhcvzRaCJT1sigDdM0NH31uIpIcNEqCbvFfjbImuegXYIAG6j9OOXIyR6MCRAN2fvsD/5LdgQEaAgIwAARkBAjJuQrNd/tlbzgYEZAQIyDwdDm/d0YFNswEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE/Kj8DbUkMR1wgBfXAAAAAElFTkSuQmCC', 'sha256': 'bb35490235b2375f8c1071bb7dc042bebf37df8dfe1acb88a368163532fff083'}, 'data/binding-v1/images/binding-evaluation-0049-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKElEQVR4nO3aT0ocTRyA4clHvI+LgDfJIb1JIAsPk4WQTRYDQ+P493PGt6rreVYK0l1i9+uvSr89Pv45ABT+S+4KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh8724Nl3R7++MwuYeH34fFCNAlH/EFHyD4DAG65O/V7ReLEbxJgK41z5+uoERf7Obu/jCVv79+HlYlQB/ozptP9rNPkhLBSwTojfp86Nfp9ovPY3R7+0ODYEuAnk/P58f40xW2JTreSIbgyP8BPa3Pzd39ZQ8Rzi+4gz8Yw0UsPQGdp+d69zpe/DQNGYVg6QnoK+vz0l2MQixu0QBt3/yL77le9+R2GsTKVgzQk/oka9AgWDFAI9Tn/O4axJrWCtA49TlfgwaxoIUCNFp9jjSIlS0UoAHrM+Z64MusEqDTcDHm235alSGIpSwRoLne6rlWC5+xRIBOxhx/xl8bXMn+AzT45mvLRozV7DxA877J864c3m/nAToZf/yZa51wEasECBjQngM00enPlpMg1rHnAAGDEyAgs9sATbr/OrILYxG7DRAwPgECMgIEZPYZoKkPgI4cA7GCfQYImIIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZfQbo4eH38YO/v34e5nRa+el7gf3ZZ4CAKQgQkBEgILPbAE19DOQAiEXsNkDA+AQIyOw5QJPuwuy/WMeeAwQMbpUAzTIEzbJOuIidB2jeXcy8K4f323mA5joJcvrDavYfoK2RGzTy2uBKlgjQXAPFXKuFz1giQONvxGy+WNMqAdoarUGjrQe+zEIB2g4X47zz25UYf1jNQgEasEHqw+LWCtBQDVIfWC5AgzRIfWDRAJ036Csz9OR26sPKvj0+/jms6vb2x/bTm7v7a9/xSenU53o/zRk9rPc8fD8s7PjzPj24xzpcKUPSA+cW3YK90oKL78jOL6g+cLT0Fuw9M/z/HoierZj0wJYAfeAc4c0YvT46qQ88IUBXP9HUHXiJAF2rRLoDbxKgS8ZIdOBDBAjI+DM8kBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQcKj8A3XTH96kbLA4AAAAAElFTkSuQmCC', 'sha256': 'a179b401c52794fff14f3129dffcb1c879f3ac6d8e67ab44262464e171af50c1'}, 'data/binding-v1/images/binding-evaluation-0049-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKklEQVR4nO3aTU7jShhA0fRTsx8GLIPlsgwGLKYHSEzeIC3LSvhtEm7Zdc4IJGQXkn35qsKv5+c/B4DCf8ldAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZ392t4ZJub+8OG/f09HiYjABd8hGf8AGC7xCgS/5dXf+wGMGHBOha8/xyBSX6YQ83N4dNuX95OcxKgL7QnQ+f7FefJCWCtwjQB/X50p/T9Q+fx+j29k6DYE2AXk/P98f45QrrEh1vJENw5P+ATuvzcHNz2UOE8wvu4ANjuIipJ6Dz9FzvXseLL9OQUQimnoB+sj5v3cUoxOQmDdD6zb/4nut9J7fTIGY2Y4BO6pOsQYNgxgCNUJ/zu2sQc5orQOPU53wNGsSEJgrQaPU50iBmNlGABqzPmOuBHzNLgJbhYsy3fVmVIYipTBGgbb3V21otfMcUAVqMOf6Mvza4kv0HaPDN15qNGLPZeYC2+yZvd+XweTsP0GL88Wdb64SLmCVAwID2HKANnf6sOQliHnsOEDA4AQIyuw3QRvdfR3ZhTGK3AQLGJ0BARoCAzD4DtOkDoCPHQMxgnwECNkGAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMPgP09PR4/OL+5eWwTcvKl98F9mefAQI2QYCAjAABmd0GaNPHQA6AmMRuAwSMT4CAzJ4DtNFdmP0X89hzgIDBzRKgrQxBW1knXMTOA7TdXcx2Vw6ft/MAbeskyOkPs9l/gNZGbtDIa4MrmSJA2xootrVa+I4pAjT+RszmiznNEqC10Ro02nrgx0wUoPVwMc47v16J8YfZTBSgARukPkxurgAN1SD1gekCNEiD1AcmDdB5g34yQye3Ux9m9vswq6enx9vbu+Xb+5eXh5uba9/0pHTqcw0jnO7xSfMGaHn/lwwdH9wrZUh64NykW7B3WnDxHdn5BdUHjn49P//5++X01juyxT8PRK9WTHpgTYA+btDnY/T+6KQ+cEKA/qVEX6I78BYBulaJdAc+JECXjJHowJcIEJDxMTyQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBBwqPwPf+EgTT1B6EAAAAAASUVORK5CYII=', 'sha256': 'a235728606187905e2cad2b1a1c9937bf4f0debfef0006ce46bb83efab5c8e26'}, 'data/binding-v1/images/binding-evaluation-0049-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMklEQVR4nO3aQU4bSRiAUWcU7uNFJC4x6xyS9VwiUhYcJgskNllYslo2GBhsvuqu91aJRLoqwv3lryLfnp7+7AAK/ySrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIy37ul4Zr2+x+7lXt8/L2bjABd8yM+4QcIPkOArvnv6vKLxQjeJEC3muePT1CiL/bf3d1uVf59ft7NSoA+0J27+4fLf/z5188Lz1QiOCFAb9Tnzei89sXnMdrvf2gQLAnQy+n5UHdedHzCskSHhWQIDvw/oNP63N0/fL4+lx+4gR8Yw1VMPQGdp+d2ax0efpyGjEIw9QT0lfV5bRWjEJObNEDLN//qZ67LTpbTIGY2Y4BO6pPsQYNgxgCNUJ/z1TWIOc0VoHHqc74HDWJCEwVotPocaBAzmyhAA9ZnzP3Al5klQMfhYsy3/bgrQxBTmSJA63qr17Vb+IwpAnQ05vgz/t7gRrYfoMEPX0sOYsxm4wFa75u83p3D+208QEfjjz/r2idcxSwBAga05QCt6PZnyU0Q89hygIDBCRCQ2WyAVnr+OnAKYxKbDRAwPgECMgIEZLYZoFVfAB24BmIG2wwQsAoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkthmgx8ffh188//q5W6fjzo9/F9iebQYIWAUBAjICBGQ2G6BVXwO5AGISmw0QMD4BAjJbDtBKT2HOX8xjywECBjdLgNYyBK1ln3AVGw/Qek8x6905vN/GA7SumyC3P8xm+wFaGrlBI+8NbmSKAK1roFjXbuEzpgjQ+Acxhy/mNEuAlkZr0Gj7gS8zUYCWw8U47/xyJ8YfZjNRgAZskPowubkCNFSD1AemC9AgDVIfmDRA5w36ygydLKc+zOzb09Of3az2+x/L397dP9x6xZPSqc/tvptr9Djf5+H7bmKH7/fxg3uow40yJD1wbtIj2IUWXP1Edv5A9YGDqY9g75nh//dA9GLFpAeWBOgD9whvxujy6KQ+cEKAbn6jqTvwGgG6VYl0B94kQNeMkejAhwgQkPFjeCAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIGBX+QtaKx/eBqhY/wAAAABJRU5ErkJggg==', 'sha256': 'ae72a4280322da4fe9aeef339590b499e1fedbb90e76d58e3651fe02ff909756'}, 'data/binding-v1/images/binding-evaluation-0050-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKUlEQVR4nO3dTWobWRSAUaVJltPgQXbW68jOPDBkKxkYMumBQAj5LwmOv1fvnjO2pVdQ9fnWk2R9enz8cQIo/JM8K4AAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZnP3VPDO7u7+3o6oIeH+9NUAvRx5/3k8wyeJUAf98f2+lfECAQoG/Ivj6NEf8PP//49Le/Lt+/1EnoC9D7pefOMf+lsOz+sDDGTAP1hen73b+zNz9/0SIaYSYD+enqedX4QGWI4AfrV+vyNbYXLY16X6O7uqwYxhAA16Xl9IDIKMYR3Qvf1eem5Dvq2Ovh1JqDTS9d88lLu01HIHMTGTEAL1efZZzcHsTEBWq4+T9egQexKgFasz5kGsT0BWrQ+ZxrE3qYHaOX6nGkQGxsdoPXrc6ZB7Gp0gA5Rn6OsEP7A3ABdRomjXNuXdRqC2MbQAB39Gj76+mF0gC6OMv4ccbXwpokBOtzN1zU3YuxkYoCARYwL0KHHnzNDENsYFyBgHbMCtN/IsN8RMcqsAF0c9/5rj/XD6AABKxgUoA22n6/ZimYDgwIErEaAgIwAAZkpAdp7o2Tvo2NjUwJ0sccO9H7HwkzjAgSsQ4CAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMy4AH359v20i52OhZmmBOjh4f60r72Pjo1NCRCwIAECMgIEZAYF6LJRssfe7eUobABxXIMCBKxmaICOPgQdff0wMUD73a3sd0SMMitAwFLGBWiDrWjbz2xjXICAdUwM0KGHIOMPO5kYoGvHatCxVgtv+nwa6eHh/tDfJmr8eZ1SH8XcCehwN2JuvtjP3ABdW79B668Q/sCnx8cfp8Gub8SW/ar16/oYf9jJ9Ano+npec8pQHzY2PUCLN0h92JsArdsg9WF7ArRog9SHCQRoxQapD0NMfxXsqZs3KH7wS2M34VMf9mYCunVzzX/kKKQ+TGMCWmIUkh5mEqDXPPt5sXcs0bPjlfowhwA1GZIeEKDf8Pqn59/s0et7SdLDTAL0e979n3hID5MJUFMi3QEB+tAYiQ7cECAg442IQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIOBU+R/S9TblyZwP9QAAAABJRU5ErkJggg==', 'sha256': 'd454cb9649cb29341374ad2f53dba24d2b3087ef11052e97119f07e9b7e23fcd'}, 'data/binding-v1/images/binding-evaluation-0050-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFM0lEQVR4nO3dMU4bXRhAUedX2A9FpGwidRaZ+t9EJAoWkwKJJgWSZTlgCCK+M+87pwb7jeS5fPM8xp8eHn4dAAr/Jc8KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5nP31PDBbm+/HHbo/v7uMJUAXe91P/l1Bs8SoOv9sT39FTECAcqG/OPjKNG/8P/NzWHzvj0+1kvoCdDHpOfm64/Lv/748/uFh5UhZhKgd6bn1eJc/vmzHskQMwnQP0/Ps54eRIYYToDeWp8P6c6Z42Oeluj29osGMYQANem5PBAZhRjCndB9fV56rp3eVgdvJ0CHl875m68/rlmfZ59Ug1ibAL1Yn3AlGsQQArS5+vy5Bg1iVQK0xfo80SCWJ0Abrc8TDWJt0wO05fo80SAWNjpA26/PEw1iVaMDtIv67GWF8A5zA3QcJfZybh/XaQhiGUMDtPdzeO/rh9EBOtrL+LPH1cKrJgZodxdfp1yIsZKJAQI2YlyAdj3+PDEEsYxxAQK2Y1aA1hsZ1jsiRpkVoKP9Xn+tsX4YHSBgCwYFaIHt51O2olnAoAABWyNAQEaAgMyUAK29UbL20bGwKQE6WmMHer1jYaZxAQK2Q4CAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMy4AD3+/H5YxUrHwkxTAnR/f3dY19pHx8KmBAjYIAECMgIEZAYF6LhRssbe7fEobACxX4MCBGzN0ADtfQja+/phYoDWu1pZ74gYZVaAgE0ZF6AFtqJtP7OMcQECtmNigHY9BBl/WMnEAJ3aV4P2tVp41efDSPf3d7v+NlHjz2XfHh/rJfAmcyeg3V2IufhiPXMDdGr7Ddr+CuEdPj08/DoMdnohttmvWj+tj/GHlUyfgE7P521OGerDwqYHaOMNUh/WJkDbbZD6sDwB2miD1IcJBGiLDVIfhpj+Ltifzm5QvPJbY2fhUx/WZgI6d3bOX3MUUh+mMQFtYhSSHmYSoEue/bzYB5bo2fFKfZhDgJoMSQ8I0F+4/On5V3t0eS9JephJgP7Oh/8TD+lhMgFqSqQ7IEBXjZHowBkBAjJuRAQyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABh8pvyDwwE1QLLLgAAAAASUVORK5CYII=', 'sha256': 'c68097fbfdaf130cd13eda605b21874306739b868ed016ccc965969d610cbf84'}, 'data/binding-v1/images/binding-evaluation-0050-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNElEQVR4nO3dQWobWRRAUaWJl9PgQZaR3fQ6spteRgaG3koGBk96YBBCceQkOL6//jtnbEu/QHX96qtkfXh8/HYCKPyVPCuAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZj91Twxu7v/90OqCHh6+nqQTo/V73k19n8CIBer8/tpe/IkYgQNmQf34cJfoT/r27Oy3v89NTvYSeAL1Nep7++fv2r999+e/Gw8oQMwnQb6bn1eLc/vmrHskQMwnQH0/Pi54fRIYYToB+tj5v0p0r58e8LNH9/ScNYggBatJzeyAyCjGEO6H7+vzouQ56Wx38PBPQ6Ufn/Hum58YoZA5iYyagherz4rObg9iYAC1Xn+/XoEHsSoBWrM8zDWJ7ArRofZ5pEHubHqCV6/NMg9jY6ACtX59nGsSuRgfoEPU5ygrhN8wN0HmUOMq5fV6nIYhtDA3Q0c/ho68fRgfo7CjjzxFXC6+aGKDDXXxdciHGTiYGCFjEuAAdevx5ZghiG+MCBKxjVoD2Gxn2OyJGmRWgs+Nef+2xfhgdIGAFgwK0wfbzJVvRbGBQgIDVCBCQESAgMyVAe2+U7H10bGxKgM722IHe71iYaVyAgHUIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQGReguy//nXax07Ew05QAPTx8Pe1r76NjY1MCBCxIgICMAAGZQQE6b5TssXd7PgobQBzXoAABqxkaoKMPQUdfP0wM0H5XK/sdEaPMChCwlHEB2mAr2vYz2xgXIGAdEwN06CHI+MNOJgbo0rEadKzVwqs+nkZ6ePh66G8TNf7c9vnpqV4CP2XuBHS4CzEXX+xnboAurd+g9VcIv+HD4+O302CXF2LLftX6ZX2MP+xk+gR0eT6vOWWoDxubHqDFG6Q+7E2A1m2Q+rA9AVq0QerDBAK0YoPUhyGmvwv2vasbFN/5rbGr8KkPezMBXbs6599zFFIfpjEBLTEKSQ8zCdAtL35e7A1L9OJ4pT7MIUBNhqQHBOgX3P70/Ks9ur2XJD3MJEC/5s3/iYf0MJkANSXSHRCgd42R6MAVAQIybkQEMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAafK/9OEN8Jf670pAAAAAElFTkSuQmCC', 'sha256': '89ecd411ecc15fbf93ddd11f0c8f1586b4d5dd347e95bf7bd3e54cedc7abc341'}, 'data/binding-v1/images/binding-evaluation-0051-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3dMU5jSRRA0WbUsx+CTmdDsyA2RNoBiyFAIumgJcuywRgw3Pp+56SAf0nYV6/K1vfN09PjD4DCP8lVAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD52V16w25vf53+hYeH39+1Ftiwm6enx3oNVxKd14gRvEaAvqo7x5QIDgjQR9Jzf/d8+m//+//f134kQ7AjQOem583ovCtGMgQCdFZ9Ppye0xnSIBCgV9Nzke68WSIZYjKfA/rW+hw/8gUPuWFzTECHFfi69JwYhcxBzGQCaupzcC1zEDNND1BVn+MrahADjQ5QW5/j62oQ08wN0Ar1Ob66BjHK0ACtU5/jNWgQcwwN0FL1WW0l8G0mvg2/GzEWfM3v3pv3xvzFXdlo+XAVz5BxE9BWnoVbWSd8xrgA7Sw4/iy7Kvgis+6IuPLma+f+7vnvRuz29td1jNmrWfm//6YTd3rZorkTEJAbFKAtnqpscc1wvkEB2tAEvv4K4SImBghYxJQAbeL4ed9unXZhXLEpAQIWJEBARoCAzIgAbe4A6C/HQFy9EQEC1iRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBmRIB2d1be1v10fUUPV29EgIA1CRCQESAgMyVAmzsGcgDEBFMCBCxoYoDWH4LWXyFcxKAAbXEvs8U1w/kGBQhYzawAbeIo2vEzc8wK0L41G7TmquCLjAvQVsaKrawTPmNcgFbeiNl8Mc3EAO1bp0HrrAS+zdAA7Y8YK7zy99dg/GGOoQFaqkHqw1hzA7RIg9SHyUYHKG+Q+jDc9ACFDVIf+FkvYAkPD79vb3/td+H+7vnrLneQOfVhLBPQyxX4ulFIfWDHBHTYgoNR6FLT0HHRpAcE6NR27CKbsheHKfUBATprFDruyJsxOrGDkx7YEaD3ZegzJ0TSAwcE6NxqvFiidz0CcECAPtKRN2MkOnAOAfoIfYGL8DkgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIuB8Q41Rfw80xExCQESAgc/P09NhdHRjNBARkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAT8qfwBj5B20QDiL6QAAAABJRU5ErkJggg==', 'sha256': '452d82a1fa9db119cf3673c8ea7711d4d54868774f51020d007c9a6c2fa67ab1'}, 'data/binding-v1/images/binding-evaluation-0051-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIUlEQVR4nO3dS05bWRRAUSiF4SDRYBiZDeOo2dQw0kCqqaSB5E41UrKQzcchhn2fz1pdPu80YOvca2Sun55+XgEU/kqeCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMt+7RG3Z3d//2Jzw+/viqWWDDrp+eftYzXEh0XiNG8BoB+qzuHFMiOCBAH0nP7uH27a+9+fvf1z4kQ7AnQKem593o/FaMZAgE6KT6fDg9b2dIg0CAXk3PWbrzbolkiMn8HdCX1uf4O5/xkhs2xwZ0WIHPS88bq5A9iJlsQE19Dp5lD2Km6QGq6nP8RA1ioNEBautz/FwNYpq5AVqhPsdP1yBGGRqgdepzPIMGMcfQAC1Vn9UmgS8z8WX4/Yqx4O/8/rV5L8yf3YWtlo8X8RMybgPayk/hVuaEPzEuQHsLrj/LTgWfZNY7Iq58+NrbPdz+Oojd3d1fxpq9mn9ubq426/tud3VB5m5AQG5QgLZ4q7LFmeF0gwK0t/L5aysTwllMDBCwiCkB2sT183P7OZ3CuGBTAgQsSICAjAABmREB2twF0C+ugbh4IwIErEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMiADt31l5/09vNsG/6OHijQgQsCYBAjICBGSmBGhz10AugJhgSoCABU0M0PpL0PoTwlkMCtAWzzJbnBlONyhAwGpmBWgTV9Gun5ljVoCeW7NBa04Fn2RcgLayVmxlTvgT4wK08kHM4YtpJgbouXUatM4k8GWGBuj5irHCb/7zGaw/zDE0QEs1SH0Ya26AFmmQ+jDZ6ADlDVIfhpseoLBB6gPf6gGW8Pj44+7u/nkXdg+3n/e4g8ypD2PZgF6uwOetQuoDezagwxYcrELn2oaOiyY9IEBvHcfOcih7cZlSHxCgk1ah4468G6M3TnDSA3sC9HsZ+pMbIumBAwJ0ajVeLNFvfQfggAB9pCPvxkh04BQC9BH6Amfh74CAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDj/YAY5/tuV4/A/2xAQEaAgMz109PP7unAaDYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAi4qvwH2x0lEomqObgAAAAASUVORK5CYII=', 'sha256': '6ae194c2b55be03136ce03b14d95c247f7f4b6ba9af9b7e7701e0d88885c2476'}, 'data/binding-v1/images/binding-evaluation-0051-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJElEQVR4nO3dPU6jSRRA0WbUvR+CTnsHvZJZECuZHZB2wGIIkEgmaMlCNj8GbG59fuekgKsC++pVGdlXDw/33wAK/ySrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzvVt6w66vf77+C3d3f75qL7BhVw8P9/UeLiQ6LxEjeIkAnas7h5QI9gjQR9Jze/P4+t/++vfHSz+SIdgRoGPT82Z03hUjGQIBOqo+H07P6xnSIBCgF9Nzku68WSIZYjL/B/Sl9Tl85BNecsPmmID2K3C+9LwyCpmDmMkE1NRnby1zEDNND1BVn8MVNYiBRgeorc/huhrENHMDtEJ9DlfXIEYZGqB16nO4Bw1ijqEBWqo+q+0EvszEt+F3I8aCr/nde/PemD+5Cxst7y7iGTJuAtrKs3Ar+4TPGBegnQXHn2V3BWcy6xMRVz587dzePP49iF1f/7yMMXs1//148cNS1vf7cd2n7gfMnYCA3KAAbfFWZYt7huMNCtDOyuevrewQTmJigIBFTAnQJq6fn9rt0ymMCzYlQMCCBAjICBCQGRGgzV0A/eUaiIs3IkDAmgQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMiMCNDuk5V3X3qzCb6ih4s3IkDAmgQIyAgQkJkSoM1dA7kAYoIpAQIWNDFA6w9B6+8QTmJQgLZ4ltninuF4gwIErGZWgDZxFe36mTlmBeipNRu05q7gTMYFaCtjxVb2CZ8xLkArH8QcvphmYoCeWqdB6+wEvszQAD0dMVZ45T/dg/GHOYYGaKkGqQ9jzQ3QIg1SHyYbHaC8QerDcNMDFDZIfeB7vYEl3N39ub7++bQLtzeP51tuL3Pqw1gmoOcrcL5RSH1gxwS034K9UehU09Bh0aQHBOi149hJDmXPDlPqAwJ01Ch02JE3Y/TKCU56YEeA3pehz9wQSQ/sEaBjq/Fsid71CMAeAfpIR96MkejAMQToI/QFTsL/AQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQMbnATHO78czfukb72ICAjICBGSuHh7uu9WB0UxAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBDwrfI/rCkfSrFLZ9UAAAAASUVORK5CYII=', 'sha256': '91242fa6597020ce51ceecb73745e9f22ac7d98caa058a4209d531917d814c37'}, 'data/binding-v1/images/binding-evaluation-0052-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFUlEQVR4nO3cMW7bShRAUeUj2o+LLCPL9TJSeDEpDLj5hQBBkBTJiSndIeecyoBtcgrz4nFI69v7++8dQOG/5KwAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzvTs1LODl5cdu5d7efu1mZQICMgIEZNyCsRGv+/1uVX5+fNRL6JmAgIwAARm3YAs8Z5n5KQZ8hQAt8HD39IfFCD5PgBZ+qeR4BCWCuwTofnruPl65+jjjcEAZghsE6Hp6/uqZ7ukPn8VIhuCG2Z+CXdbndb//yhslV399A/8uAI8wdYDOuvDF9Nw+lAbBpUlvwS7T84izHA57vClzOwZnZpyAnlOfPx3fKATzBuj0+l/wnuu2sxNpEMwYoLP6PPnsGgTzBqitz+V5NQgmCtAgn9uwuk+NgMeZJUDHcWOE6/+4BkMQk5siQCNf5yOvDR5t+wEaYevnks0gmCJAA9ZnzPXA8208QGsZLtayTljWxgM0+Lgx5qrgabYcoHWNFetaLSxiywFaxaAx8trg0aYIEDCmzQZoqDcPb/NeItPabICA8QkQkBEgILPNAK1oA+jANhBz2maAgFUQICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgs80Avb39Onzx8+NjtwbHdR5XDjPYZoCAVRAgICNAQGazAVrRNpANIKa12QAB45siQCMPQSOvDR5tywFa1x3NulYLi9hygMYfNMZcFTzNxgO0lrFiLeuEZW08QCOPG6OtB55v+wE6HS7GueZPV2L8YVrbD9DgV/jIa4NHmyJAo72X6M1DmCtAp9oGjVBAGMREARphM8jWD0waoLxB6gNTB+iyQc/J0NmJ1AcmDdDl9f/oBp0dX33g6PtuSocKvLz8OG3E636/7Fmk55ns7q/RjBPQjVFoqT/iy0OpD1yaOkBXu/DFDF39dfWBq769v/++/p3JHG/HLt29NbsRLOmBGwTosxn6W9IDdwnQwiXSHfg8AVogRqID/0aAgMzsT8GAkAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgTsKv8Dgv8IxHFI+zEAAAAASUVORK5CYII=', 'sha256': 'd79f2a11ef75aa844f8cc2dbe3d5df96c09d7544db311f3a82ae830c87fc554a'}, 'data/binding-v1/images/binding-evaluation-0052-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFGElEQVR4nO3cQU7bWhiA0fSp7IdBJXbSRbKTSh1kMR0gddIBUhQlaUIfTr5r33NGIIF9JfCn/9qGL29vv3YAhf+SswIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMzX7tSwgOfnb7uV2+9/7mZlAgIyAgRkbMHYiKeX192q/P7xvV5CzwQEZAQIyNiCLfCcZeanGPAZArTAw93jLxYj+DgBWvilksMRlAhuEqDb6bn5eOXi44z3A8oQXCFAl9PzT890j7/4JEYyBFfM/hTsvD5PL6+feaPk4rdv4M8F4B6mDtBJFz6ZnuuH0iA4N+kW7Dw99zjL+2EPmzLbMTgx4wT0mPr87fhGIZg3QMfX/4J7rutOTqRBMGOATurz4LNrEMwboLY+5+fVIJgoQIP834bV/dcIuJ9ZAnQYN0a4/g9rMAQxuSkCNPJ1PvLa4N62H6ARbv2cczMIpgjQgPUZcz3weBsP0FqGi7WsE5a18QANPm6MuSp4mC0HaF1jxbpWC4vYcoBWMWiMvDa4tykCBIxpswEa6s3D67yXyLQ2GyBgfAIEZAQIyGwzQCu6AfTObSDmtM0AAasgQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAZpsB2u9/vn/w+8f33Roc1nlYOcxgmwECVkGAgIwAAZnNBmhFt4HcAGJamw0QML4pAjTyEDTy2uDethygde1o1rVaWMSWAzT+oDHmquBhNh6gtYwVa1knLGvjARp53BhtPfB42w/Q8XAxzjV/vBLjD9PafoAGv8JHXhvc2xQBGu29RG8ewlwBOtY2aIQCwiAmCtAIN4Pc+oFJA5Q3SH1g6gCdN+gxGTo5kfrApAE6v/7v3aCT46sPHHx5e/u1m9Xz87fjT59eXpc9vvQ8/oe4RvuJfzFmnICujEJLTUPnh5r5lwz+ZuoAXezCJzN08dvVBy6aegv2wUn+5tbsSrCkB64QoHvdUJAeuEmAFi6R7sDHCdACMRId+H8ECMjM/hQMCAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DArvIH2iIIBc0bPhgAAAAASUVORK5CYII=', 'sha256': 'e0a39bfc3f4dd906f41007ee0179afc40dde406647bcddf1fe210bc8b55c1920'}, 'data/binding-v1/images/binding-evaluation-0052-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFGElEQVR4nO3bMW7jRhiAUW0Q3cfFAnuJrXPI1L5EgC18mC0MqElhQBAkW3JiSt+Q815lAxY5gM0P/5D0t9fX3zuAwh/JWQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQObP7tSwgKen77uVe3n5tZuVCQjICBCQsQVjI/Y//t6tyuGfv+ol9ExAQEaAgIwt2ALPWWZ+igFfIUALPNw9/WExgs8ToIVfKjkeQYngJgG6nZ7n/f76Z38eDh8dUIbgCgF6Pz03o/PRD5/FSIbgitmfgl3W53m//0/1+czHN/DvAnAPUwforAtfTM/1Q2kQXJp0C3aZnnuc5e2wx02Z7RicmXECekx9Pjq+UQjmDdDp9b/gnuu6sxNpEMwYoLP6PPjsGgTzBqitz+V5NQgmClBenxHODkOZJUDHcWOE6/+4BkMQk5siQCNf5yOvDe5t+wEa4dbPJTeDYIoADVifMdcDj7fxAK1luFjLOmFZGw/Q4OPGmKuCh9lygNY1VqxrtbCILQdoFYPGyGuDe5siQMCYNhugod48vM57iUxrswECxidAQEaAgMw2A7SiG0Bv3AZiTtsMELAKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZLYZoJeXX29f/DwcdmtwXOdx5TCDbQYIWAUBAjICBGQ2G6AV3QZyA4hpbTZAwPimCNDIQ9DIa4N723KA1rWjWddqYRFbDtD4g8aYq4KH2XiA1jJWrGWdsKyNB2jkcWO09cDjbT9Ap8PFONf86UqMP0xr+wEa/AofeW1wb1MEaLT3Er15CHMF6FTboBEKCIOYKEAj3Axy6wcmDVDeIPWBqQN02aDHZOjsROoDkwbo8vq/d4POjq8+cPTt9fX3blZPT99Pv33e75c9vvQ8/pe4Ri8T/2HMOAFdGYWWmoYuDzXzHxl8ZOoAvduFL2bo3Y+rD7xr6i3YJyf5m1uzK8GSHrhCgO51Q0F64CYBWrhEugOfJ0ALxEh04P8RICAz+1MwICRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABu8q/kD8HtUN5rcIAAAAASUVORK5CYII=', 'sha256': '7e2321cb8a5772edbcf429c9ad7b15b4eef9a52564f78012b34591a8a319fe6d'}, 'data/binding-v1/images/binding-evaluation-0053-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJklEQVR4nO3cPW4bRxiAYTqIjhOAhQDdxBfJOXIR3cSACwG5igsBalIQYGjK+qf1zs48T6VC2h3D2hffDNf+cn//YwdQ+CO5K4AAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg82d3a/i99vvr3XTu7r7vJmICAjICBGRswZjfw99/7Tbu6p9/dzMyAQEZAQIyAgRkBAjICBCQESAg42P4T3rpdrIXWOEiBOiT3vQ//WYxggMBCv6F0fEKSsTiBOgy3bm6uX3+xx++fX3mmkrEmgTo/fV5MTpPffPjGO331xrEggTozel5U3d+6XiF0xIdbiRDLMXH8G+oz9XN7cfr8/wFp/wvbOApJqDXpuf33etw8eM0ZBRiHSaguD5P3cUoxAoE6Nzpk3/xPdfzzm6nQUxPgJ6rT7IGDWIdAjRWfR7fXYOYmAANV5/Ha9AgZiVAI9bnQIOYngD9ZJz6jLkeuCwB+n+4GPNpP67KEMR8Vg/Qtp7qba0WXrR6gI7GHH/GXxt8xNIBGnzzdcpGjCmtG6DtPsnbXTmcWTdAR+OPP9taJ7yeAAGZRQO0odOfU06CmMyiAQJGIEBAZsUAbXT/dWAXxkxWDBAwCAECMgIEZJYL0KYPgA4cAzGN5QIEjEOAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMcgG6u/t++OLh29fdNh1XfvyzwEYtFyBgHAIEZAQIyKwYoE0fAzkAYiYrBggYhAABmUUDtNFdmP0Xk1k0QMAIBGgzQ9BW1gmvt26AtruL2e7K4cy6AdrWSZDTH6a0dIBOjdygkdcGH7F6gLY1UGxrtfCi1QM0/kbM5ouJCdBPRmvQaOuByxKg8+FinGf+dCXGH6YkQIM2SH1YwZf7+x/1Ggay318fv766ua2WoT4X/9ucxt1cvw8moOHmIPVhHQL0QoM+M0Nnt1MfpmcL9qrp/RO2Y2elUx9WIEB9hqSHZQnQmw8yL5ihx/s79WEpAvT+z1PeXaJfnitJDwsSoMt8pvtijJ4/zFYf1iRA5dslusPiBCgoke7AgQB9UoxEBx4TICDjTWggI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELCr/Af36xm1Wa0RRgAAAABJRU5ErkJggg==', 'sha256': '4f26ef6454aa0aba20e4e2fcaa2c5a701d440b21b12093077245ee8395d20bc0'}, 'data/binding-v1/images/binding-evaluation-0053-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJ0lEQVR4nO3cPW5bRxhAUTqwlmNAhYDsJBvxOryR7CSACwLeigsBalIQYGhK1i/DO2/mnEqFxDeG9a6+GT7p0/39zx1A4Y/kqgACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzOfu0vD/ur29201nv/++m4gJCMgIEJCxBWN+N3/+vdu4h3/+2s3IBARkBAjICBCQESAgI0BARoCAjLfhr/TQ7WQPsMJFCNCVnvQ//WQxggMBCn7D6PgKSsTiBOgy3Xn4+uX5L7/59uOZ11Qi1iRA76/Pi9H53Sc/jtHt7Z0GsSABenN63tSdJx1f4bREhwvJEEsRoDfU5+PpOXN4wbMMaRDrEKAmPc9kyCjEOjyIGNfnd1eZ8q/5wRkT0LnTO/866XlmFDIHMTcT0Cj1efK65iDmJkBj1efx1TWIiQnQcPV5vAYNYlYCNGJ9DjSI6QnQL8apz5jrgcsSoP+GizHv9uOqDEHMZ/UAbeuu3tZq4UWrB+hozPFn/LXBRywdoME3X6dsxJjSugHa7p283ZXDmXUDdDT++LOtdcLrCRCQWTRAGzr9OeUkiMksGiBgBAIEZFYM0Eb3Xwd2YcxkxQABgxAgICNAQGa5AG36AOjAMRDTWC5AwDgECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjILBeg/f774YObbz9223Rc+fHfAhu1XICAcQgQkBEgILNigDZ9DOQAiJmsGCBgEAIEZBYN0EZ3YfZfTGbRAAEjEKDNDEFbWSe83roB2u4uZrsrhzPrBmhbJ0FOf5jS0gE6NXKDRl4bfMTqAdrWQLGt1cKLVg/Q+Bsxmy8mJkC/GK1Bo60HLkuAzoeLce7505UYf5iSAA3aIPVhBZ/u73/WaxjI7e3d8eOHr1+qZajPxf83p7Gf6/vBBDTcHKQ+rONzvYDh7Pffjz85Dy242ih0ljz1YXq2YK+a3q/QIPVhQQLUZ0h6WJYAvfkg84IZenzMpD4sRYDe/37Ku0v05PG29LAgAbrMe7ovxuj599TUhzUJUPl0ie6wOAEKSqQ7cCBAV4qR6MBjAgRk/CoGkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjYVf4FKCkiHcGYrl8AAAAASUVORK5CYII=', 'sha256': '3c6c5f4da59b476bb9a0861fce243c232317d965ec8d192322fb281f24561dd1'}, 'data/binding-v1/images/binding-evaluation-0053-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFG0lEQVR4nO3cO24bSRRAUXow2g8DAdqJF6mdGHCgxTgQ4GQCAhyatPWleaurzokUCOwSpL54VWzqy/Pzjx1A4Z/kqgACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzL/dpeHv2u/vd9N5evq+m4gJCMgIEJCxBWN+dw+Pu437+e3rbkYmICAjQEBGgICMAAEZAQIyAgRkvA1/o4duJ3uAFa5CgG70pP/pN4sRHAhQ8Amj4ysoEYsToOt059VnbX/7JKsSsTgB+nh93vWA/+k3X8Zov7/XIBYkQO9Oz+c/WHR8hdMSHS4kQyzF2/DvqM/dw+N1P9Z4+YJT/gsb+BMT0FvT8/eudXjx4zRkFGIdJqC4Pn+6ilGIFQjQudM7/+p7rpedXU6DmJ4AvVSfZA0axDoEaKz6XF5dg5iYAA1Xn8s1aBCzEqAR63OgQUxPgH4xTn3GXA9clwD9P1yMebcfV2UIYj6rB2hbd/W2VguvWj1AR2OOP+OvDT5j6QANvvk6ZSPGlNYN0Hbv5O2uHM6sG6Cj8cefba0T3k6AgMyiAdrQ6c8pJ0FMZtEAASMQICCzYoA2uv86sAtjJisGCBiEAAEZAQIyywVo0wdAB46BmMZyAQLGIUBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQGa5AD09fT988fPb1902HVd+/Flgo5YLEDAOAQIyAgRkVgzQpo+BHAAxkxUDBAxCgIDMogHa6C7M/ovJLBogYAQCtJkhaCvrhLdbN0Db3cVsd+VwZt0AbeskyOkPU1o6QKdGbtDIa4PPWD1A2xootrVaeNXqARp/I2bzxcQE6BejNWi09cB1CdD5cDHOPX+6EuMPUxKgQRukPqzgy/Pzj3oNA9nv749f3z08VstQn6v/NqfxNNffgwlouDlIfViHAL3SoFtm6Oxy6sP0bMHeNL3fYDt2Vjr1YQUC1GdIeliWAL37IPOKGbrc36kPSxGgj7+f8uES/fZcSXpYkABd5z3dV2P08mG2+rAmASqfLtEdFidAQYl0Bw4E6EYxEh24JEBAxpPQQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESBgV/kPbgwZtb9wF6MAAAAASUVORK5CYII=', 'sha256': 'cfea28b5bf55213a7b829c91d0f8798361d2996c5c2e48a242778c9663aadae3'}, 'data/binding-v1/images/binding-evaluation-0054-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJ0lEQVR4nO3dPW7cRhiAYSmwjmNAhVtfyK3PkTYXSutCQK6iYgE1KRQIC3HFje3dfYec52n1wynEFx9HI+r+cHi+Ayj8kVwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQ+dZfejMfHL+uf8PT041ZrgV25Pxye6zVsMjofESP4/wToMt1ZUiI4S4DOp+fl++f1r33485+PPiRDsEKATqfnbHR+KkYyBCfNHqBlfX45PesZ0iBYmjpA7+pzkfSsZEiD4J1JA3Tt9ByTIfjIjAcRb1mf5fe/4C/aYOumm4CO7/9rp2dlFDIHwXQTUFifd1c0B8FcAWrrs7yuBsFEAcrrM8LVYSizBOht3Bjh/n9bgyGIyU0RoJHv85HXBte2/wCNsPWzZDMI5nof0Dj1efXy/fPKX7FycbsJ/dOOznDsfALays/cVtYJl7XzAA07/oy8KriZPT+CbWuseHz8sqfRemR///Vyt0Ffvz3c7c4UE9DIg8bIa4NrmyJAwJh2G6ChTh6ucy6Rae02QMD4BAjICBCQ2WeANrQB9Mo2EHPaZ4CATRAgICNAQEaAgIwAARkBAjICBGQECMgIEJARICCzzwC9vVpwK299f1unlyIylX0GCNgEAQIyAgRkdhugDW0D2QBiWrsNEDC+KQI08hA08trg2vYcoG090WxrtXARew7Q+IPGmKuCm9l5gLYyVmxlnXBZOw/QyOPGaOuB29t/gI6Hi3Hu+eOVGH+Y1v4DNPgdPvLa4NqmCNBo5xKdPIS5AnSsbdAIBYRBTBSgETaDbP3ApAHKG6Q+8M6nu8k8Pf14+//rr0W4wf+Pfxc79YEZJ6CT9/+1RyH1gY9MNwEdV+Dao5D0wLpJA7R8HLtshpZTlfrA0tQBWjbo9zN08oFOfeCk2QO0fBxbduRsjFZ2kaQHVgjQWoZ+Z5daeuAsATpdjZMl+qnvAJwlQOc7cjZGogO/RoDO0xe4khkPIgKDECAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh4IRnT+frtoV4C/zEBARkBAjL3h8Nzd3VgaiYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgTcVf4FKvEZr2ZMZokAAAAASUVORK5CYII=', 'sha256': 'f1f9b7fdc97b86f7acdb453fe0765f5f8b3160cb90f70cf2ef07836f1ff770c5'}, 'data/binding-v1/images/binding-evaluation-0054-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKElEQVR4nO3dvW3cSBiAYelg96PAwEXXhotwQS7i2nBkwIGKUbCAkgtkCAtxxT3bu/sOOc+T6ocTiC8+jkbU/eHwdAdQ+Cu5KoAAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfOguvRkPD5/WP+Hx8cet1gK7cn84PNVr2GR03iNG8P8J0GW6s6REcJYAnU/Pt6/P61/7z5eP731IhmCFAJ1Oz9no/FKMZAhOmj1Ay/r8dnrWM6RBsDR1gN7U5yLpWcmQBsEbkwbo2uk5JkPwnhkPIt6yPsvvf8FftMHWTTcBHd//107PyihkDoLpJqCwPm+uaA6CuQLU1md5XQ2CiQKU12eEq8NQZgnQ67gxwv3/ugZDEJObIkAj3+cjrw2ubf8BGmHrZ8lmEMz1PqBx6vPi29fnlb9i5eJ2E/rHHZ3h2PkEtJWfua2sEy5r5wEadvwZeVVwM3t+BNvWWPHw8GlPo/XIPv79790GPX//fLc7U0xAIw8aI68Nrm2KAAFj2m2Ahjp5uM65RKa12wAB4xMgICNAQGafAdrQBtAL20DMaZ8BAjZBgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzD4D9Ppqwa289f11nV6KyFT2GSBgEwQIyAgQkNltgDa0DWQDiGntNkDA+KYI0MhD0Mhrg2vbc4C29USzrdXCRew5QOMPGmOuCm5m5wHaylixlXXCZe08QCOPG6OtB25v/wE6Hi7GueePV2L8YVr7D9Dgd/jIa4NrmyJAo51LdPIQ5grQsbZBIxQQBjFRgEbYDLL1A5MGKG+Q+sAbH+4m8/j44/X/r78U4Qb/P/5N7NQHZpyATt7/1x6F1AfeM90EdFyBa49C0gPrJg3Q8nHsshlaTlXqA0tTB2jZoD/P0MkHOvWBk2YP0PJxbNmRszFa2UWSHlghQGsZ+pNdaumBswTodDVOluiXvgNwlgCd78jZGIkO/B4BOk9f4EpmPIgIDEKAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg44VkTOf5++d6CfxkAgIyAgRk7g+Hp+7qwNRMQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIuKv8B80yFg5n5c6CAAAAAElFTkSuQmCC', 'sha256': '20609e86252798a0e847d386d909bdbd286dbffe697533f1ab1a61bb1e63f357'}, 'data/binding-v1/images/binding-evaluation-0054-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKElEQVR4nO3dTU7cSBiAYRglx4nEItLcJBfJOXKRuUmkLJDmKixaYjMLItTCjXuSdPdbdj3Plh/XAr/6XBTm/nB4ugMo/JVcFUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkPnSX3oyHh8/rn/D4+ONWa4FduT8cnuo1bDI67xEj+P8E6DLdWVIiOEuAzqfn+eun9a/9+O3f9z4kQ7BCgE6n52x0filGMgQnzR6gZX1+Oz3rGdIgWJo6QG/qc5H0rGRIg+CNSQN07fQckyF4z4wHEW9Zn+X3v+Av2mDrppuAju//a6dnZRQyB8F0E1BYnzdXNAfBXAFq67O8rgbBRAHK6zPC1WEoswToddwY4f5/XYMhiMlNEaCR7/OR1wbXtv8AjbD1s2QzCOZ6H9A49Xnx/PXTyl+xcnG7Cf3jjs5w7HwC2srP3FbWCZe18wANO/6MvCq4mT0/gm1rrHh4+Lyn0XpkH//+526Dnr9/ududKSagkQeNkdcG1zZFgIAx7TZAQ508XOdcItPabYCA8QkQkBEgILPPAG1oA+iFbSDmtM8AAZsgQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAZp8Ben214Fbe+v66Ti9FZCr7DBCwCQIEZAQIyOw2QBvaBrIBxLR2GyBgfFMEaOQhaOS1wbXtOUDbeqLZ1mrhIvYcoPEHjTFXBTez8wBtZazYyjrhsnYeoJHHjdHWA7e3/wAdDxfj3PPHKzH+MK39B2jwO3zktcG1TRGg0c4lOnkIcwXoWNugEQoIg5goQCNsBtn6gUkDlDdIfeCND3eTeXz88fr/11+KcIP/H/8mduoDM05AJ+//a49C6gPvmW4COq7AtUch6YF1kwZo+Th22Qwtpyr1gaWpA7Rs0J9n6OQDnfrASbMHaPk4tuzI2Rit7CJJD6wQoLUM/ckutfTAWQJ0uhonS/RL3wE4S4DOd+RsjEQHfo8AnacvcCUzHkQEBiFAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ8UIypvP8/Uu9BH4yAQEZAQIy94fDU3d1YGomICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE3FX+A5JWGx0q7xshAAAAAElFTkSuQmCC', 'sha256': 'cd0c70ed38af16b522b461221dad71c3f98137e130b9d3c9b6020f72e60cc7d2'}, 'data/binding-v1/images/binding-evaluation-0055-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKUlEQVR4nO3dTU4bSxhAUXgK+/Eg0+wgK8mCspK3g0wZeDEMkJi8gSXEaxsTfuxb3XXOCAlwF1L31VclW9w+Pj7cABT+Sa4KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD51l16lXa77+d/YL+/v9ZaYPVuHx8f6jWsPjqvESM4T4C+vjvHlAhOEqD3pefP76fzv/vj191r35IhWBCgt9PzZnTeFSMZWsXQOrL9hm4hh9Cv3rsf7s7xK7ws0eFCW7qH4MME6ER9Pp+ehcMLLjKkQSBA/6vPl6fnTIY06HL+vXv1JG69fj5d8OasTP1GxN3u+9Xqc/IqiwXAbOYN0KW3XWcsrqVBTGvSLdj1B58F2zGYdALK63Py6uYgJjRjgAapzzhrgMp0AXoeNMZ58p9XYghiNnMFaPwnfPwVwheaKEDjHP0ccxjEnCYK0LD1GXlVcFGzBGjAo59jDoOYzSwBAgY0RYBWMf4cGIKYyhQBAsa0/QCtaPw5MAQxj+0HCBiWAAGZjQdodfuvA7swJrHxAAEjEyAgI0BAZssBWukB0IFjIGaw5QABgxMgICNAQEaAgIwAARkBAjICBGQECMgIEJARICCz5QDt9/eHL378urtZm+c1P/8VsD1bDhAwOAECMgIEZDYeoJUeAzkAYhIbDxAwMgECMtsP0Op2YfZfzGP7AQKGNUWAVjQEGX+YyhQBAsY0S4BWMQQZf5jNLAF6acwGjbkquKiJAvRyrBjtaX+5HuMP85goQKt4tsdfIXyhuQI05mGQox+mNV2AXhqhQSOsASozBmicwyBHP0zu282U9vv73e77ywr8+f10zQUswqc+zGnGCejkM3/NUUh9YPYAHZ7862/HFtsu9WFmk27Bku2YwQcWBGjZoEtk6Hi2Uh8QoGURjjP0mRKd3NNJDzwToDcytOjImzE6c5AkPbAgQH+boc8cVEsPnCRAb1fjZIne9QrASQL0vo68GSPRgb8nQO+jL/CFpn4jItASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg46MYbNDPp6v+iwE+zAQEZAQIyNw+Pj50VwemZgICMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIOCm8h/s0CU2m2SuygAAAABJRU5ErkJggg==', 'sha256': 'ea40de223070bf1a543e9473f49dd02aae46b4a4a79c2224b55890e1aa06dd81'}, 'data/binding-v1/images/binding-evaluation-0055-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFN0lEQVR4nO3bQU4bSRiAURjF92GBNJdgnUPOmktEyoLDsEDyZhaWLI9t7MCAv+qu91aJQroKyf3lr0K5f3t7vQMo/JWsCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dEtvUgPD4+Xv+Dl5fet9gKLd//29lrvYfHReY8YwWUC9PXdOaVEcJYAfSw9z5vN5b/7tN2+90cyBEcE6Hp6rkbnQzGSoUUMrSN7WdFHyCX0u5/dT3fn9AmHJdottKbPEHyaH8Ofqc/zZvP/63P5gZP8Ww2XmYD+04Kv7c6R3cP309DDw6M56Jts/v7nbnW2v37erc7UATodfG6w6PNmc9ggxzFmNu8RLKnP2bUcx5jWpAE6Onbdsj5nF9Ug5jRjgG526XOVBjG5GQM0SH3G2QNUpgvQftAY583f78QQxGzmCtD4b/j4O4QvNFGAxrn6OeUyiDlNFKBh6zPyruBbzRKgAa9+TrkMYjazBAgY0BQBWsT4s2MIYipTBAgY0/oDtKDxZ8cQxDzWHyBgWAIEZFYeoMWdv3acwpjEygMEjEyAgIwAAZk1B2ihF0A7roGYwZoDBAxOgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzJoD9PLye/eLp+32bmn2e95/F7A+aw4QMDgBAjICBGRWHqCFXgO5AGISKw8QMDIBAjLrD9DiTmHOX8xj/QEChjVFgBY0BBl/mMoUAQLGNEuAFjEEGX+YzSwBOjRmg8bcFXyriQJ0OFaM9rYf7sf4wzwmCtAi3u3xdwhfaK4AjXkZ5OqHaU0XoEMjNGiEPUBlxgCNcxnk6ofJzRig0wbdPkNHi6oPc5o0QKfv/C0bdLSW+jCtH3cT2735Dw+Pu98+bbfPm813L2rwgb15J6DbH8ccu+CIAJ0/jn1thk4fqD4w+xHswnHs8Kz06XPZ2YpJD+wJ0JUMHXXkaowujE7SA0cE6E8ztPO5o5n0wFkCdL0aZ0v0oScAZwnQxzpyNUaiA39OgD5GX+AL+TE8kBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDjv2KwQttfP+st8EdMQEBGgIDM/dvba7c6MDUTEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABd5V/AXQlK2Z2cw+9AAAAAElFTkSuQmCC', 'sha256': '8138d0f3b836968abcf9cb88e0930fce412dac99a52b638410d0f148ca2ce268'}, 'data/binding-v1/images/binding-evaluation-0055-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKElEQVR4nO3bTU5bSRhAUdMK+2EQKaNsI4vIgrKI3kZGkRh4MQyQmPTAEqKfjQ3h59Z7dc4ICXAVkt/VVyVzdX9/twMo/JOsCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dItvUo3N1/P/8B+f/tZe4HVu7q/v6v3sProPEeM4DwBev/uHFMiOEmAXpee378ezv/u95/Xz31LhmBBgC6n52J0XhUjGVrF0Dqy/YbeQi6hn33v/nV3jl/haYkOC23pPQR/TYBO1Oft6Vk4vOAiQxoEAvS/+rx7es5kSIM+zvW3f3eb8/Dnx25zpv4g4s3N10+rz8lVFhuA2cwboI8+dp2xWEuDmNakR7DPH3wWHMdg0gkor8/J1c1BTGjGAA1Sn3H2AJXpAvQ4aIzz5D/uxBDEbOYK0PhP+Pg7hHc0UYDGufo55jKIOU0UoGHrM/Ku4EPNEqABr36OuQxiNrMECBjQFAFaxfhzYAhiKlMECBjT9gO0ovHnwBDEPLYfIGBYAgRkNh6g1Z2/DpzCmMTGAwSMTICAjAABmS0HaKUXQAeugZjBlgMEDE6AgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMlgO0398evvj+83q3No97fvwrYHu2HCBgcAIEZAQIyGw8QCu9BnIBxCQ2HiBgZAIEZLYfoNWdwpy/mMf2AwQMa4oArWgIMv4wlSkCBIxplgCtYggy/jCbWQL01JgNGnNX8KEmCtDTsWK0p/3pfow/zGOiAK3i2R5/h/CO5grQmJdBrn6Y1nQBemqEBo2wB6jMGKBxLoNc/TC5L7sp7fe3Nzdfn1bg96+Hz9zAInzqw5xmnIBOPvOfOQqpD8weoMOT//nHscWxS32Y2aRHsOQ4ZvCBBQFaNugjMnQ8W6kPCNCyCMcZekuJTp7ppAceCdCFDC06cjFGZy6SpAcWBOilGXrLRbX0wEkCdLkaJ0v0qlcAThKg13XkYoxEB15OgF5HX+AdTf1BRKAlQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAxr9isEEPf37UW+BFTEBARoCAzNX9/V23OjA1ExCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAbvKf9uiJQ7HDLO2AAAAAElFTkSuQmCC', 'sha256': '54b8af84430181516d0b20a062234488e533ae5fa989d815159193ffded02c1c'}, 'data/binding-v1/images/binding-evaluation-0056-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFK0lEQVR4nO3dPW7bWBhAUWUQ7cdFgGwidRY5tTcRwIUWk8KAmsHAA0EjWXLkv8v3eE6VwiE/Fbx4fKTsL4+PvzcAhb+SswIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZL52p4ZPdXf3bTOp3e5hMyYrICAjQEDGLRirs/3+92YK+18/N4OzAgIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMt6Ejr8AOe7XCOHtBCj+yvXJz+sRqyJAy/ptD08HkSFWQoA+tjv32+31//5jv79yTCVibgL0/ul5MTqXfvg8RhZEzE2A3q0+N3XnWYcjnJTo7u6bBjElAVpEep494HGGLIWYkveA3lSf++323etz5eAT/1Zj1kmAbnB8/X9oeo6dnEiDmIkAvbI+n3x2DWJKAjRAfc7Pq0HMQYBedrjaP+2265LjATSICQjQDfXZLIMGMQ0BGqw+TzSIOQjQRaNc26PMCecEaLm7ztfZk2YCAvSCZdZn+bPBnxCgkbZ+ztkMYmgCdGrcK3ncyVktAbpo+cufseaEcwI01SJi9PlZGwGaYVkx1rRwIEBARoCGfPh1zuMwRiRAQEaANvMtHGb6LMxNgE6NeP81+uSslgABGQECMgIEZARo+AfwxzyMZywCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICNC/druHp3/82O83IzvMf/hEsGQCBGQECMgIEJARoFPjbgONOzmrJUCb+XZtZ/oszE2AgIwATfIw3gN4RiRAQEaAnjfWImisaeFAgP5n9PuX0ednbQRo+GXFKHPCOQGaZxEx7uSslgCN/TjMwy+GJkAvWHKDljwb/AkBet7xgmKZ1/nxVJY/DOprPcBy7XYPQ/yBY/W51f7Xz3oE/mMFNORmkK0f5iBA4zVIfZjGl8fH3/UMAzi+F7vfbqsx7PswGSugYfak1Yf5CNAYDVIfpuQW7DYnz8U+4XbsJHbqw0wE6Gbnz+Y/KEPn6yz1YTICtMQMSQ8rIUCvd+k1xVeX6NLWkvowKwF6q+tvS78Yo+v72dLD3ATo3bzj9zZ0h5UQoGVlSHpYFQGKe6Q4rJkAARlvQgMZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECNpV/APrdGqVdcBynAAAAAElFTkSuQmCC', 'sha256': '0cda12cb5367c79409b037f1bf3d2dea7b3bdaa7047157d033b07470b41ada45'}, 'data/binding-v1/images/binding-evaluation-0056-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIklEQVR4nO3dS04bWRiAUaeVLCcSg+ws68jOGCBlKxkgZdJqIVlu20AIj6/+W+eMGCD7euCP+6gqPt3f/zoAFP5J3hVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPO5e2v4UDc33w6Luru7PcxkBgRkBAjIWIKxO7+/fz0s4cuPn4fhzICAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwroeMbIOfeRgivJ0DxLddnv69H7IoAbetpDw8vIkPshAC9b3eeve/x6v2Ex9dUItYmQG+fnhfdbH36y5cxMiFibQL0ZvV5/UMejq9wVqKbm28axJIEaBPpufqCpxkyFWJJAvSq+rzro62uZkiDWIkA/WV9PuypemcZ0iBW4kroTdfn6jsu/N8d2BsBGlCfy/fVINZgCfa847c9f5j56XLMWowFmAGNqc/RcSTmQUwnQMPq80CDWIMAPWrKd3vKOOGSAG131/lp9qRZgAA9Y5v12f7Y4E8I0KStn0s2gxhNgM7N/SbPHTm7JUCP2v70Z9Y44ZIALTWJmD5+9kaAVphWzBotHAkQkBGgkYdflxyHMZEAARkBOqw3cVjps7A2ATo3cf01feTslgABGQECMgIEZARo/AH8KYfxzCJAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAP3n7u724YcvP34eJjuO//iJYMsECMgIEJARICAjQOfmbgPNHTm7JUCH9XZtV/osrE2AgIwALXIY7wCeiQQIyAjQdbMmQbNGC0cC9D/T1y/Tx8/eCND4acWUccIlAVpnEjF35OyWAM0+DnP4xWgC9IwtN2jLY4M/IUDXnU4otvk9Px2V6Q9Dfa4HsF13d7cj/sGx+rzUNv+i7JMZ0MjNIFs/rEGA5jVIfVjGp/v7X/UYBjhdi/3+/rUahn0fFmMGNGZPWn1YjwDNaJD6sCRLsJc5Oxf7gOXYWezUh5UI0Itdns2/U4Yu51nqw2IEaIsZkh52QoD+3mOXKf51iR7bWlIfViVAr/X01dLPxujp/WzpYW0C9Gbe8L4N3WEnBGhbGZIedkWA4h4pDnsmQEDGldBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgEPlX6YBKLp7zBLXAAAAAElFTkSuQmCC', 'sha256': 'bccaa236ac0bf2d0298bfc7c13d0ebc5826035fda4f9558c468efd30b2d8047d'}, 'data/binding-v1/images/binding-evaluation-0056-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFLElEQVR4nO3dPW7bWBhAUWUQLWcAF1lGdjPryG68DBcCspUUBtQEAw8EjWTJkf8u3+M5VQqH/FTw4vGRsr88Pv7aABT+Ss4KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD52p0aPtXd3bfNpHa7h82YrICAjAABGbdgrM7+n783U9j++LkZnBUQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJDxJnT8Bchxv0YIbydA8VeuT35ej1gVAVrWb3t4OogMsRIC9LHdud9ur//37/v9lWMqEXMToPdPz4vRufTD5zGyIGJuAvRu9bmpO886HOGkRHd33zSIKQnQItLz7AGPM2QpxJS8B/Sm+txvt+9enysHn/i3GrNOAnSD4+v/Q9Nz7OREGsRMBOiV9fnks2sQUxKgAepzfl4NYg4C9LLD1f5pt12XHA+gQUxAgG6oz2YZNIhpCNBg9XmiQcxBgC4a5doeZU44J0DL3XW+zp40ExCgFyyzPsufDf6EAI209XPOZhBDE6BT417J407OagnQRctf/ow1J5wToKkWEaPPz9oI0AzLirGmhQMBAjICNOTDr3MehzEiAQIyArSZb+Ew02dhbgJ0asT7r9EnZ7UECMgIEJARICAjQMM/gD/mYTxjESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaA/rXbPTz94/t+vxnZYf7DJ4IlEyAgI0BARoCAjACdGncbaNzJWS0B2sy3azvTZ2FuAgRkBGiSh/EewDMiAQIyAvS8sRZBY00LBwL0P6Pfv4w+P2sjQMMvK0aZE84J0DyLiHEnZ7UEaOzHYR5+MTQBesGSG7Tk2eBPCNDzjhcUy7zOj6ey/GFQX+sBlmu3exjiDxyrz622P37WI/AfK6AhN4Ns/TAHARqvQerDNL48Pv6qZxjA8b3Y/XZbjWHfh8lYAQ2zJ60+zEeAxmiQ+jAlt2C3OXku9gm3YyexUx9mIkA3O382/0EZOl9nqQ+TEaAlZkh6WAkBer1Lrym+ukSXtpbUh1kJ0Ftdf1v6xRhd38+WHuYmQO/mHb+3oTushAAtK0PSw6oIUNwjxWHNBAjIeBMayAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELCp/AaqchnwzogtNQAAAABJRU5ErkJggg==', 'sha256': 'c8386913b849edf736f5866086f9e73be5944a0b4100982235019a82556a605c'}, 'data/binding-v1/images/binding-evaluation-0057-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFElEQVR4nO3ZQU4bSRiAUTOK78MiEqtcg0PkQHMIrjErpCx8GBaW2GRhyfIYA8YYvu7q91aJBF0Fcn/6q7jZbp9WAIV/klUBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkfnRLD+v29ufbX7DZ/PmuvcCk3Wy3T/UeFhGd14gRSyZATXdeUqL29z8dmyV9EhzBrv/RX989vP29z4/3rz1wUR8+EKDrpOfd6Lz2xUcxkiEWRYA+lZ4Pdeek/RMOSyRDH/Xfv8+rmfv1e71aHn+Gv7A+67uHz9fn7QcOecEBhwToLIctuHp6Dh09XIMYmyPYhwefb1h0ffewP5E5jjEwE9Dk6nNyLaMQQxKg/tj1GscxhidAZ9Un3IkGMTABekdbn+nsAb6CAJ2wHzSm8+bvd2IIYiQCdGz6b/j0dwhnEqCJXv285DKI8QjQaVOrz5R3BRcToElf/bzkMoiRCBCQEaA5jT87hiCGIUBARoBmNv7sGIIYgwABGQECMgI0v/PXjlMYAxAgICNAQEaAgMzSAzTTC6Ad10DM3dIDBIQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjILD1Am82f3T+eH+9Xc7Pf8/6ngHlZeoCAkAABGQECMgI012sgF0AMQICAjAABGQGa5SnM+YsxCBCQEaD5DUHGH4YhQEBGgGY2BBl/GIkAnTbNBk1zV3AxAfqfw7Fiam/74X6MP4xBgI5N/92e/g7hTAI0j8sgVz8MSYDeMYUGTWEP8BUEaOqXQa5+GJgAndug78/Q0aLqw3gE6C1H7/x3NuhoLfVhSD/qDUzd7s2/vf25++/z4/367uGrFzX4sBAmoGkdxxy7WBQBuvw4dt0MvXyg+jA8R7DLj2OHZ6WLz2UnKyY9LIQAXSFDRx15N0ZvjE7Sw6II0DUztHPZ0Ux6WCAB+pR9NU6W6ENPgAUSoOs47Mi7MRId2BGg69MXOJM/wwMZAQIyjmCM4Nfvdb0FLmECAjICBGRuttunbnVg0UxAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQJWlb/diSLABmWNpAAAAABJRU5ErkJggg==', 'sha256': '41fcc93f54d7c180693d5712b9e6bac6fad41ae18b1ddbeba05c793c4ab4dbbb'}, 'data/binding-v1/images/binding-evaluation-0057-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEUlEQVR4nO3aPW7bWBhAUXsQL2cAFW6zobSzjrTZUFoXArIVFwLcpBAgeGhZ8o+sSz6eU7mQyWeDvPjeg253u8cbgMI/yV0BBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkvnW3HtZmc3/6A9vtw7XWArN2u9s91mtYRXReI0asmQA13XlJidr//3xs1/Qk2IJd/tH//evp9O9+/3H32gVX9fCBAF0mPWej89qHJzGSIVZFgD6Vnnd156jDFZ6XSIbe6+m/f28W7u7nn5v1EaAP1ufz6ZnYX3CSIQ1ibAL07vpcPD0nMqRBjM0XEc/YbO6vVp+jd5ksAEYiQOW264TJvTSIIdmCver6g8+E7RjDMwHNtD5H724OYjACdEZbn/msAb6CAB1xGDTm8+YfVmIIYiQCNDX/N3z+K4Q3EqCZHv285DCI8QjQcXOrz5xXBR8mQLM++nnJYRAjESAgI0BLGn/2DEEMQ4CAjAAtbPzZMwQxBgECMgIEZARoefuvPbswBiBAQEaAgIwAAZm1B2ihB0B7joFYurUHCAgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQWXuAttuH/Q/ff9zdLM1hzYe/ApZl7QECQgIEZAQIyAjQUo+BHAAxAAECMgIEZARokbsw+y/GIEBARoCWNwQZfxiGAAEZAVrYEGT8YSQCdNw8GzTPVcGHCdD/PB8r5va2P1+P8YcxCNDU/N/t+a8Q3kiAlnEY5OiHIQnQGXNo0BzWAF9BgOZ+GOToh4F9qxcwX9vtw2Zz/7wCv389XXMBk/CpD+MxAZ0yeeevOQqpD2sgQGdstw/X345Ntl3qw6hswea1HTP4sCoC9JEGfUWGXs5W6sPwBOgd9kV4maHPlOjonk56WAkBukCGJh05G6MTB0nSw6oI0CUz9JmDaulhhQToUw7VOFqid10BVkiALuN5R87GSHRgT4AuT1/gjXwREcgIEJCxBWMEdz//1EvgI0xAQEaAgMztbvfY3R1YNRMQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICbyl/TGyITnEAWuAAAAABJRU5ErkJggg==', 'sha256': 'ce4afd4cc070e32dc2c8eedb9dc10e8d60cd158d2abe485ec6b8d0bf404921d6'}, 'data/binding-v1/images/binding-evaluation-0057-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFElEQVR4nO3ZQUpjSxiAUX10lvPAQYM7cSNvHb0RdyL0QOitOAg46UEg5MWYxBj97q17zqgb9FYpuR9/lbfr9csNQOGfZFUAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZH93Sw7q7+3n8C56ff3/XXmDSbtfrl3oPi4jOe8SIJROgpjtvKVH7+5+O5yV9EhzBrv/RX90/Hv/e16eH9x64qA8fCNB10nMyOu998V6MZIhFEaBPpedD3Tlo+4TdEsnQR73+9+/NzK1+/blZHn+Gv7A+q/vHz9fn+AOHvOCAXQJ0lt0WXD09u/YerkGMzRHsw4PPNyy6un/cnsgcxxiYCWhy9Tm4llGIIQlQf+x6j+MYwxOgs+oT7kSDGJgAndDWZzp7gK8gQAdsB43pvPnbnRiCGIkA7Zv+Gz79HcKZBGiiVz9vuQxiPAJ02NTqM+VdwcUEaNJXP2+5DGIkAgRkBGhO48+GIYhhCBCQEaCZjT8bhiDGIEBARoCAjADN7/y14RTGAAQIyAgQkBEgILP0AM30AmjDNRBzt/QAASEBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIySw/Q8/PvzT9enx5u5ma75+1PAfOy9AABIQECMgIEZARortdALoAYgAABGQECMgI0y1OY8xdjECAgI0DzG4KMPwxDgICMAM1sCDL+MBIBOmyaDZrmruBiAvQ/u2PF1N723f0YfxiDAO2b/rs9/R3CmQRoHpdBrn4YkgCdMIUGTWEP8BUEaOqXQa5+GJgAndug78/Q3qLqw3gE6Ji9d/47G7S3lvowpB/1BqZu8+bf3f3c/Pf16WF1//jVixp8WAgT0LSOY45dLIoAXX4cu26G3j5QfRieI9jlx7Hds9LF57KDFZMeFkKArpChvY6cjNGR0Ul6WBQBumaGNi47mkkPCyRAn7KtxsESfegJsEACdB27HTkZI9GBDQG6Pn2BM/kzPJARICDjCMYIVr/+1FvgEiYgICNAQOZ2vX7pVgcWzQQEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIOCm8hfIkyN5mpTKBAAAAABJRU5ErkJggg==', 'sha256': '76c9293421c157faa88085301a91cae066f45501b8c890daf58b7248904eeb90'}, 'data/binding-v1/images/binding-evaluation-0058-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRElEQVR4nO3aQWobWRRAUSXEywl4kGX0brKO7KaXkYEgW8nA4EkPDKJwHAW3Y92q/84ZW6VfWHX1/kcfHh5+ngAKH5N3BRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5lP31vBq9/dfTks7n7+fJjEBARkT0Dt+D0/7NoPXEqB3HP63LxGjv+vfu7vTQv55fDyNJEA3OnS4XEeJ4EKA/md6Hr9+vv7yu28/rlxWhkCAXpGePxbn+t8/65EMgQC9S3pe9HQRGYItAfptff5Kd565XHNbovv7LxrETAJ0o/RcH4iMQszkh4hBfX73Xsv/zBeemT4BbZ/5W6bnyihkDmKO0RNQXp8X390cxBxzA7Sf+vy6Bg1iiKEB2lt9nmgQ00wM0D7r80SDGGVcgPZcnycaxByzArT/+jzRIIaYFaBD1OcoK4S3GxSgyyhxlGf7sk5DEKuaEqCjP8NHXz+MDtDFUcafI64WXmtEgA63+dqyEWNhIwIE7NP6ATr0+PPEEMSq1g8QsFuLB2i9kWG9O2KyxQN0cdz91xrrh9EBAnZo5QAtcPy85Sia9awcIGDnBAjICBCQWTZAax+UrH13zLFsgC7WOIFe715gRICA3RIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICCzfoDuvv04rWKle4GVA3Q+fz+ta+27Y45lAwTsnwABGQECMh8nHJSscXZ7uQsHQCxj5QABOzclQEcfgo6+fpgYoPV2K+vdEZMtHiBgz9YP0AJH0Y6fWdX6AQJ2a0SADj0EGX9Y2IgAbR2rQcdaLbzWlAAdfXw4+vphdICOuBGz+WJ5gwK0tf8G7X+F8HazArQdJfb8hG/XZvxhYbMCdIgGqQ9zjAvQzhukPowyMUC7bZD6MM3QAO2wQerDQHMDtKsGqQ8zfTrNdj5/v7//sq3A49fPt1zAs/CpD6OMnoBefOZvOQqpD8NNn4C2T/4tRyHpAQH67XZs24i/WKIXxyv1YawPDw8/6zXszjZDF2/MkPS8379mJedhHwkT0J93ZC8W5I89un6WNO1zBi8SoNdl6O1n1dIDF7ZgNxr+dQd+JUDvGCPRgesECMj4ISKQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAk6V/wD2fTtZRMUCEAAAAABJRU5ErkJggg==', 'sha256': '0158612aa798f821bb4ac08ac5b2bd4dd633de13418a86b3237e0f47ce4a0238'}, 'data/binding-v1/images/binding-evaluation-0058-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFS0lEQVR4nO3awW0bVxRAUTowywmghctwN6kj3bgMLwikFS8EaJMFAWIiyTIUWboz/52zFjl/IM7l+x/8dH//4wRQ+CO5KoAAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMp+7S8Or3d19OS3tcvl+msQEBGRMQO/4PTzt2wxeS4DecfjfvkSMfq+Hv/48LeT89z+nkQTogw4dbu+jRHAjQP8zPd/O55df/vXh4YW3lSEQoFek55fFefnvH/VIhkCA3iU9z7q+iQzBlgD9tD6/pTuP3N5zW6K7uy8axEwC9EHpeXkgMgoxkx8iBvX52bWW/5kvPDI9QNtn/tv5/JH1efaiGsQoowP0qD7hSjSImeYGaD/1eboGDWKIoQHaW32uNIhpJgZon/W50iBGGRegPdfnSoOYY1aA9l+fKw1iiFkBOkR9jrJCeLtBAbqNEkd5tm/rNASxqikBOvozfPT1w+gA3Rxl/DniauG1RgTocJuvLRsxFjYiQMA+rR+gQ48/V4YgVrV+gIDdWjxA640M690Rky0eoJvj7r/WWD+MDhCwQysHaIHj5y1H0axn5QABOydAQEaAgMyyAVr7oGTtu2OOZQN0s8YJ9Hr3AiMCBOyWAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmfUD9PXh4bSKle4FVg7Q5fL9tK617445lg0QsH8CBGQECMj8MeGgZI2z29tdOABiGSsHCNi5KQE6+hB09PXDxACtt1tZ746YbPEAAXu2foAWOIp2/Myq1g8QsFsjAnToIcj4w8JGBGjrWA061mrhtaYE6Ojjw9HXD6MDdMSNmM0XyxsUoK39N2j/K4S3mxWg7Six5yd8uzbjDwubFaBDNEh9mGNcgHbeIPVhlIkB2m2D1IdphgZohw1SHwaaG6BdNUh9mGl0gJ426OMz9Oii6sMo0wP09Jn/yAY9upb6MM3negG7cH3y7+6+bLvw7Xx+vytKDwjQf1wu328N2jbiN5bo2fFKfRjr0/39j3oNu7PN0M0bMyQ97/evWcll2EfCBPTrHdmzBfllj14+S5r2OYNnCdDrMvT2s2rpgRtbsA8a/nUHnhKgd4yR6MDLBAjI+CEikBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQJOlX8BhLY7VYbzTVAAAAAASUVORK5CYII=', 'sha256': '2ae81256477e54ff21bbe65ec85dd38e8f664dbe893b10cb3639fa6044447fd2'}, 'data/binding-v1/images/binding-evaluation-0058-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPElEQVR4nO3aTWrcShSA0U5IlhPwwDvLOrKzDBqylQwMmbxBQyMc/9DPsT+p7jlju1XC1qdbRX96ePh9Aih8Tq4KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMl+7ScLO7u/vT0s7nn6dJTEBAxgT0ju/haW8zuJUAvePwv/0VMfq3/nz/dlrI1x+/TiMJ0AcdOlw/R4ngSoD+Z3pefQM/9067fKwMgQDdkJ5bZ/5HP/+oRzIEAvQu6XnS5UNkCLYE6Nn6vMcx5/UztyW6u7vXIGYSoA9Kz8sDkVGImXwRMajPc9da/mu+8Mj0CWj7zCdfLfl7FDIHMcfoCSivz5NXNwcxx9wA7ac+f69BgxhiaID2Vp8LDWKaiQHaZ30uNIhRxgVoz/W50CDmmBWg/dfnQoMYYlaADlGfo6wQ3m5QgK6jxFGe7es6DUGsakqAjv4MH339MDpAV0cZf464WrjViAAdbvO1ZSPGwkYECNin9QN06PHnwhDEqtYPELBbiwdovZFhvTtissUDdHXc/dca64fRAQJ2aOUALXD8vOUomvWsHCBg5wQIyAgQkFk2QGsflKx9d8yxbICu1jiBXu9eYESAgN0SICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgs36Avv74dVrFSvcCKwfofP55Wtfad8ccywYI2D8BAjICBGQ+TzgoWePs9noXDoBYxsoBAnZuSoCOPgQdff0wMUDr7VbWuyMmWzxAwJ6tH6AFjqIdP7Oq9QME7NaIAB16CDL+sLARAdo6VoOOtVq41ZQAHX18OPr6YXSAjrgRs/lieYMCtLX/Bu1/hfB2swK0HSX2/IRv12b8YWGzAnSIBqkPc4wL0M4bpD6MMjFAu22Q+jDN0ADtsEHqw0BzA7SrBqkPM305zXY+/7y7u99W4M/3bx+5gEfhUx9GGT0BPfnMf+QopD4MN30C2j75HzkKSQ8I0LPbsW0j/mGJnhyv1IexPj08/K7XsDvbDF29MUPS835/mpWch/1LmIBe35E9WZBXe/TyWdK0/zN4kgDdlqG3n1VLD1zZgn3Q8K878DcBescYiQ68TICAjC8iAhkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgVPkPqT87WeVFDKoAAAAASUVORK5CYII=', 'sha256': 'db036cb1e9af04788721e735a9330086a22a9285e988f6ff4461c1b6c7c02462'}, 'data/binding-v1/images/binding-evaluation-0059-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPUlEQVR4nO3dO24bVxiAUTqw9qPCgCttI4vQgrKIbCNVABdajAsBblIIYGTKpG29vvs4pzJgmbwDaz788+Dww/391wNA4Y/kXQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfOzeGt7P9fWnwwbu7r4cpmICAjICBGQcgrGXq89/H5bz7d8/D3MyAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIONxHM3z96Z7ch28BQFqnvh58vN6xJ4EaIgnDR9fR4nYigC9SXr++evb5X9+c3t14WVliE0I0Kul56fROffDT2MkQ2xCgF6ant/qzg8dX+GkRDLE8gTomfV5eXd+sUTX1580iFW5D2iU+lx4/U2+VI8NmYDGSs/JGx1HIYdjLMkENGJ9zr2jUYjFmIDOery3v396LoxC5iCWYQIauj4/XIM5iGUI0AT1eaBBrEeA5qjPAw1iMQJ01mj1GXlV8DwC9J3jWDHyfn5cmyGI2QnQ/2bcn2dcMxwJ0ASnfp5yMog1CNCp8esz1zrhAgFaYYiYff1sS4AmHivmWi08JUBzXPk6xxUxpiZAQGb3AK00OKy0LWxi9wAdzXj8NfvKQYCAzNYBWu+YZb0tYm1bB2iZo5jZ18+2BAjICBCQESAgs2+Apr4B+im3RDOjfQME5AQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMjsG6C7uy8Pf7i5vTrM77gVx+2C8e0bICAnQEBGgICMAK1wGmj29bOtrQO03vna9baItW0dIKAlQNMfxcy7ctg9QCsds6y0LWxi9wABIQGa+5ZoN0AzNQH6zlwNmmu18JQArTA+zL5+tiVAs44Vs6wTLvhwf//10t/v5PFX+g3+bYWP62P8+RWbfF/j3Wy/DCagif/zJl0zHAnQfFfEXPliGR/rBYzr5vZqwAOxkcs4MrEekwno0m/qaHu7Uz8sRoCmaZD6sB4BmqNB6sOSXIb/jWu3ySmhk/ypDysxAV1ysre//yikPqzNBDToKCQ97MAENOIopD5swgT00tv5X3Ea+mHX1IeFCdCrfaro2SU6N09JD8sToDf5cONPY3T5IE562IQAjfUZa+lhKwI0RIl0hz0JUNMjxQEBAkruAwIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECDpX/ANywNb1/etxuAAAAAElFTkSuQmCC', 'sha256': '907671ea524cb77c854cbf6e802ce9a34d60c6be6e8738183f7fb475e8077234'}, 'data/binding-v1/images/binding-evaluation-0059-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQUlEQVR4nO3dS2obWRiAUaWJl9PgQSA78UZ6Hb2R7MSQgSBbycDgSQ8MakeyFMevr+6954wCcaRbxPXx10OlT3d3P3cAhb+SdwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg87l7a/g419dfdgvY77/vhmICAjICBGQcgrGW+3/+3k3n6t8fuzGZgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPE4jub5e8M9uQ7egwA1T/w8+nk9Yk0CtIknDR9eR4lYigC9S3quvn67/M/vb28uvKwMsQgBerP0/DY65374NEYyxCIE6LXp+aPuPOnwCkclkiGmJ0AvrM/ru/PMEl1ff9EgZuU+oK3U58LrL/KleizIBLSt9By90WEUcjjGlExAW6zPuXc0CjEZATrr8d5+9fXbx9fnybfWIGYiQM+qT7qW4zVoENMQoAHq80CDmI8AjVGfBxrEZATorK3VZ8urgpcRoF8cxoot7+eHtRmCGJ0A/W/E/XnENcOBAA1w6ueUk0HMQYCObb8+Y60TLhCgGYaI0dfPsgRo4LFirNXCKQEa48rXOa6IMTQBAjKrB2imwWGmbWERqwfoYMTjr9FXDgIEZJYO0HzHLPNtEXNbOkDTHMWMvn6WJUBARoCAjAABmXUDNPQN0KfcEs2I1g0QkBMgICNAQEaAgIwAARkBAjICBGQECMgIEJARICCzboD2++8Pf7i/vdmN77AVh+2C7Vs3QEBOgICMAAEZAZrhNNDo62dZSwdovvO1820Rc1s6QEBLgIY/ihl35bB6gGY6ZplpW1jE6gECQgI09i3RboBmaAL0i7EaNNZq4ZQAzTA+jL5+liVAo44Vo6wTLvh0d/fz0t+v5PFX+m382wof18f48xyLfF/jfrRfBhPQwP95g64ZDgRovCtirnwxjc/1Arbr/vZmgwdiWy7jlon1NpmALv2mbm1vd+qHyQjQMA1SH+YjQGM0SH2Yksvwf3DtNjkldJQ/9WEmJqBLjvb2jx+F1Ie5mYA2OgpJDyswAW1xFFIfFmECeu3t/G84DT3ZNfVhYgL0Zp8qenGJzs1T0sP0BOhdPtz42xhdPoiTHhYhQNv6jLX0sBQB2kSJdIc1CVDTI8UBAQJK7gMCMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAnaV/wBjgyeRx14tYwAAAABJRU5ErkJggg==', 'sha256': '45214af8e4c6ac4d7ac842d43448fd5c46895530d71ca088cfa22a3d051bb26e'}, 'data/binding-v1/images/binding-evaluation-0059-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFO0lEQVR4nO3dO24bVxiAUTqwlhNAhVttSG3WkTYbSutCgLfiQoCbFAIYmTJpW6/vPs6pDFgm78CaD/88OPxwf//1AFD4I3lXAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjIfu7eG93N9/emwgbu7z4epmICAjAABGYdg7OXbX38elnP195fDnExAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIeBxH8/y96Z5cB29BgJonfp78vB6xJwEa4knDx9dRIrYiQG+Snn//+Xb5n9/cXl14WRliEwL0aun5aXTO/fDTGMkQmxCgl6bnt7rzQ8dXOCmRDLE8AXpmfV7enV8s0fX1Jw1iVe4DGqU+F15/ky/VY0MmoLHSc/JGx1HI4RhLMgGNWJ9z72gUYjEmoLMe7+3vn54Lo5A5iGWYgIauzw/XYA5iGQI0QX0eaBDrEaA56vNAg1iMAJ01Wn1GXhU8jwB95zhWjLyfH9dmCGJ2AvS/GffnGdcMRwI0wamfp5wMYg0CdGr8+sy1TrhAgFYYImZfP9sSoInHirlWC08J0BxXvs5xRYypCRCQ2T1AKw0OK20Lm9g9QEczHn/NvnIQICCzdYDWO2ZZb4tY29YBWuYoZvb1sy0BAjICBGQECMjsG6Cpb4B+yi3RzGjfAAE5AQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMvsG6O7u88Mfbm6vDvM7bsVxu2B8+wYIyAkQkBEgICNAK5wGmn39bGvrAK13vna9LWJtWwcIaAnQ9Ecx864cdg/QSscsK20Lm9g9QEBIgOa+JdoN0ExNgL4zV4PmWi08JUArjA+zr59tCdCsY8Us64QLPtzff7309zt5/JV+g39b4eP6GH9+xSbf13g32y+DCWji/7xJ1wxHAjTfFTFXvljGx3oB47q5vRrwQGzkMo5MrMdkArr0mzra3u7UD4sRoGkapD6sR4DmaJD6sCSX4X/j2m1ySugkf+rDSkxAl5zs7e8/CqkPazMBDToKSQ87MAGNOAqpD5swAb30dv5XnIZ+2DX1YWEC9GqfKnp2ic7NU9LD8gToTT7c+NMYXT6Ikx42IUBjfcZaetiKAA1RIt1hTwLU9EhxQICAkvuAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBD5T9zWDW9Iq3gmgAAAABJRU5ErkJggg==', 'sha256': 'b64e60021fcc8a387c65ed593b790f3afdbc6a145275b6fad2e37fdf6649c898'}, 'data/binding-v1/images/binding-evaluation-0060-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJklEQVR4nO3aQU5bSRRAUWjFy2mJAcvIbnod2Q3LYGApW2GAxKQHSI6DjYmNw/1V/5xRpAT+i8S/elX49vn56Qag8E/yVAABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnWPXpad3f3p//Bdvv4VbPAot0+Pz/VM6wiOu8RI9ZMgJruHFIiVkiArt+dh83m9Jd/f3k58bdKxHoI0HXq82F0zoqRBrESAvSp9FzcnT8pkQwxPQG6sD5XTM+JDGkQcxOgpaRnnwxVvwpYpu28PwA+iLi4+hw+ZfoXjNUSoFP23/yHzeZr6nP0cRrElHwS+k/rk8zwsNnsjmN3d/cTr+JX8fLfvzcT2fz4eTM7G9By63P4dHsQkxGgRdfncAYNYiYCtPT6vNIgpiRA71pOfZY5D3yeAP1mt1ws823fTWUJYg4C9MtYb/VY08JRAnTEMtef5c8G5xKgMQ5f+xzEmIYAjf0mjzs5CNBby19/xpoTThMgICNAI93+7HMTxAQECMgIEJBZe4AGPX+9cgpjdGsPEBASICAjQEBm1QEa+gLolWsghrbqAAEtAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMqsO0Hb7+PqH7y8vN2PaTb77v8BAVh0goCVAQEaAgMzaAzT0NZALIEa39gABIQECMgI06inM+YsJCBCQEaDfjLIEjTInnCZAY59ixp0cBGjImyC3P0xDgI5YcoOWPBucS4B+GWuhGGtaOEqARjqIOXwxGQF619IatLR54PME6K395WI57/z+JNYfpiFAAzRIfZiVAC29QerDxARo0Q1SH+YmQGc06Csz9OZx6sOUBOgDb978r2nQm6eoD7P6Vg8wgNf3/+7ufr8OD5vN33iW9LAqNqDLV6HrbkOH31B9mJ4N6PJVaH9huXghOlox6WElBOhs2+3jfoMOO/JhjE6vTurDegjQJXaNOCzRxRfVusMKCdBfLNFZ3wFWSICuY78jH8ZIdOCVAF2fvsAf8mt4ICNAQEaAgIwAARkBAjICBGQECMgIEJARICDjk9DMY/PjZz0C57EBARkBAjK3z89P3dOBVbMBARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMBN5X9+EjoOZEREQgAAAABJRU5ErkJggg==', 'sha256': 'b8f581c6dee95ef819f2d68e08ebaae4524becbaf55d3e8db8e10276d155cfb4'}, 'data/binding-v1/images/binding-evaluation-0060-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFGklEQVR4nO3dPU4cSxhAUfNk72cCUjbkBXlDTglmMQQjOXkBEh4zMMDwc6uqz4ksGdOFcF2+Krfkq8Ph7htA4b/kqQACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPO9e/Sydrvr8x+w399+1VpgaFeHw129hk1E5zlixJYJUNOdU0rEBgnQx3fn968/5//4zc8fZ35XidgOAfqY+rwYnTfFSIPYCAF6V3ou7s5rSiRDLE+ALqzPB6bnTIY0iLUJ0CjpOSZD1T8FjGm/7l8ALyIOV5/Tpyy/wdgsLyKec7zzvyY9jx73MArtdtcL/xhkswRoxPocP1eDXq/6Nn2Sm7Ova6zBEWzc+pw+3VmMxQjQ0PU5XYMGsRIBGr0+9zSIJQnQs8apz5jrgfcToH88DBdj7vaHVRmCWIMA/TXXrp5rtfAkAXrCmOPP+GuDtxKgOQ5fxxzEWIYAzb2T5105CNBj448/c60TzhMgICNAM93+HHMTxAIECMgIEJDZeoAmPX/dcwpjdlsPEBASICAjQEBm0wGa+gLonmsgprbpAAEtAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMpsO0H5/e/+Lm58/vs3pYeUPXwtMZNMBAloCBGQECMhsPUBTXwO5AGJ2Ww8QEBIgICNAs57CnL9YgAABGQH6xyxD0CzrhPMEaO5TzLwrBwGa8ibI7Q/LEKAnjNygkdcGbyVAf801UMy1WniSAM10EHP4YjEC9KzRGjTaeuD9BOix4+FinD1/vBLjD8sQoAkapD6sSoBGb5D6sDABGrpB6sPavtcLGNp+f7vbXR+34PevP1/z6EfJUx+WZAJ6waOd/zWjkPqwESagl93v/68ZhaSHTRGgS45jn5Gh09lKfVieAF0+Ch1X4+ISPXmmkx42QoDeOwqdduTFGJ2/SFIftkOALvHQiNMSXXxRrTtskAB9Yone9BlggwToYxx35MUYiQ7cE6CPpy/wSl5EBDICBGQECMgIEJARICAjQEBGgICMAAEZAQIy3oRmHSP8Fya8iQkIyAgQkLk6HO66pwObZgICMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgG+V/wFu0CylJX703AAAAABJRU5ErkJggg==', 'sha256': '127201f50612f6af70450521ea1c3c988cd42aec066522812d79395d867abc3e'}, 'data/binding-v1/images/binding-evaluation-0060-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFLElEQVR4nO3cQU4bWRRAUWjF+/GAaXaQlWRBvRJ20FMGXgwDS0x6YMlxsDHYGG79qnNGkZJQLxJ19f7Hyv12+3wHUPgneSqAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMj86B49W+v1w/k/sNk8fdcsMGn32+1zPcMiovMWMWLJBKjpzjElYoEE6PbdeVytzv/1Xy8vZ35XiVgOAbpNfd6NzkUx0iAWQoA+lZ6ru/OREskQsydAV9bnhuk5kyENYt4EaCrpOSRD1Y8Cpmkz328AH0ScXH2OnzL7F4zFEqBzDt/8x9Xqe+pz8nEaxCz5JPRH65PM8Lha7Y9j6/XDjFfxm/jv33OfbxjOz9/Nd913sgFNtz7HT7cHMTMCNOn6HM+gQcyJAE29PjsaxCwJ0JumU59pzgOfJ0B/2S8X03zb91NZgpgHAfpjrLd6rGnhJAE6YZrrz/Rng0sJ0BiHr0MOYsyGAI39Jo87OQjQa9Nff8aaE84TICAjQCPd/hxyE8QMCBCQESAgs/QADXr+2nEKY3RLDxAQEiAgI0BAZtEBGvoCaMc1EENbdICAlgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZlFB2izedr94tfLy92Y9pPv/y0wkEUHCGgJEJARICCz9AANfQ3kAojRLT1AQEiAgIwAjXoKc/5iBgQIyAjQX0ZZgkaZE84ToLFPMeNODgI05E2Q2x9mQ4BOmHKDpjwbXEqA/hhroRhrWjhJgEY6iDl8MTMC9KapNWhq88DnCdBrh8vFdN75w0msP8yGAA3QIPVhrgRo6g1SH2ZMgCbdIPVh3gToggZ9Z4ZePU59mCUBeserN/97GvTqKerDXP2oBxjA7v1frx8O6/C4Wn3Fs6SHRbEBXb8K3XYbOv6C6sPs2YCuX4UOF5arF6KTFZMeFkKALrbZPB026Lgj78bo/OqkPiyHAF1j34jjEl19Ua07LJAAfWGJLvoKsEACdBuHHXk3RqIDOwJ0e/oCH+TH8EBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAxiehmY+fv7/kf4nj69iAgIwAAZn77fa5ezqwaDYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECLir/A/nFzlVZZ6q5gAAAABJRU5ErkJggg==', 'sha256': '0366652b677136a4093c1bf7292cb863f175fba1b98057832b29b4406fe8ae21'}, 'data/binding-v1/images/binding-evaluation-0061-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFCklEQVR4nO3aQU5iSxiAUX1p98PAxE30uBfp+G2iEwcsxgEJkzcg4V2h0WgjX0GdMyIGblUi90vdX+83m9c7gMI/yaoAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMj86Ja+PqvV4/tvWK9fLrUXuAX3m81rvYfrjs4pYgQfEqDzp2dJhm7mV1lZ3/RXyCPYJ76sD0/P7392+/vXqavd9tcIvkaAPqjPh9E59eaDGK1WjxoEBwToPOk5tvv4MkO7JWQo9O/Dw92V+Lnd3k3An+H/UJ+Hp+e/rM87l7rqeQSclwC9KcIZ07N0cFkNgp3ZA3RQn29dS4PgwNQBumR9jlfRIJg3QPv7/5seu05ZLqdBTG7SAC3rk2xAg2DSAI12z4+2H7iY6QJ0+bnPKeZBMF2ABqnPOHuA0FwBykc/xwyDmNlEARr/Dh9/h3BeEwVob5zjz5j7gYuZMUDAIGYJ0IDTnyWTIOY0S4CAAU0RoMGPPzsOQUxoigABYxIgICNAQOb2A3QVA6AdYyBmc/sBAoYlQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5vYDtF6/7F5sf/+6G9t+h/s9w227/QABwxIgICNAQGaKAF3FGMgAiAlNESBgTLMEaPBDkOMPc5olQMCAZgzQaIeg0fYDFzNRgMZ/uhl/h3BeEwVozEmQ6Q8zmytASyM0aIQ9QGi6AC0PGu39v1zd8Yc5TRegAe/20fYDFzNjgEYYBhn9wLwBOmjQJTO0XE59mNy8AUrmQeY+sDR1gC7cIPWBA7MH6LhB35Ghg8uqD+wI0B+KcMYMHV9KfWDvx/8v57brwmr1uP/JLhwPT89fu+BxwqQHDgjQG+v1y7JBBx35MEbvnJvUB44J0MlSHJToa1Nq3YF3CNAnHsq+8HHgHQL0iY58GCPRgU8RoE/QFzgvAWIiP7fbegu84f+AgIwAAZn7zea1Wx2YmhMQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEg4K7yH6V3DFKt+1VlAAAAAElFTkSuQmCC', 'sha256': '1e1600bbd9525b28fd2d2d62293523209d8422814b237a0af7ef5e1ee9efcfc4'}, 'data/binding-v1/images/binding-evaluation-0061-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFD0lEQVR4nO3aQW4aSRiAUXsU7sPC29wgJ8mB5iS+wWy94DBeIHmTBRJDmoBlB/MV1HsrZJmuktz9qfjx43b7+gBQ+CdZFUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmW/d0rdnvX46/wubzcu19gL34HG7fa33cNvROUWM4F0CdPn0HJKhu/lTVjZ3fQv5CPaBm/V5tTr/3h9vb6eudt+3EXyOAL1Tn3ejc+qXFzFar580CBYE6DLpObZ7+2GGdkvIUOi/f5dH1GF9//lXt9+t8DX8H+rzvFr9ZX3OXOqm5xFwWQL0WxEumJ5Di8tqEOzMHqBFfb50LQ2ChakDdM36HK+iQTBvgPbP/xd97DrlcDkNYnKTBuiwPskGNAgmDdBoz/xo+4GrmS5A15/7nGIeBNMFaJD6jLMHCM0VoHz0c8wwiJlNFKDxn/DxdwiXNVGA9sY5/oy5H7iaGQMEDGKWAA04/TlkEsScZgkQMKApAjT48WfHIYgJTREgYEwCBGQECMjcf4BuYgC0YwzEbO4/QMCwBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyNx/gDabl92LH29vD2Pb73C/Z7hv9x8gYFgCBGQECMhMEaCbGAMZADGhKQIEjGmWAA1+CHL8YU6zBAgY0IwBGu0QNNp+4GomCtD4n27G3yFc1kQBGnMSZPrDzOYK0KERGjTCHiA0XYAODxrt83+4uuMPc5ouQAM+7aPtB65mxgCNMAwy+oF5A7Ro0DUzdLic+jC5eQOUzIPMfeDQ1AG6coPUBxZmD9Bxg74iQ4vLqg/sCNAfinDBDB1fSn1g79v/L+e268J6/bT/yS4cz6vV5y54nDDpgQUB+s1m83LYoEVH3o3RmXOT+sAxATpZikWJPjel1h04Q4A+8KHsE28HzhCgD3Tk3RiJDnyIAH2AvsBlCRAT+f7zk99p8kX8HxCQESAg87jdvnarA1NzAgIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgQ8VH4BPNIK5EaeRPgAAAAASUVORK5CYII=', 'sha256': 'eaff0958a8fabd8f39675437e0cb8b16af4feccae0b257d9fad3ca8d0ec627d7'}, 'data/binding-v1/images/binding-evaluation-0061-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFD0lEQVR4nO3aMU4jSRiAUVgN93EwEtFcYw4xB9pDcI2NRiLwYQgskWxgydvYYyNY46/sei+ykN1VEu5P1T/cbzYvdwCFv5JVAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dYtfX1Wq++n37BeP19qL3AL7jebl3oP1x2dY8QI3iVA50/PkgzdzK+ysr7pr5BHsA98WR8en05/9vX3z2NXu+2vEXyOAL1Tn3ejc+zNezFarb5rEOwRoPOk59D248sMbZeQodA/f7/eXYkfvx7uJuDP8H+oz8Pj0/+sz4lLXfU8As5LgN4U4YzpWdq7rAbB1uwB2qvPl66lQbBn6gBdsj6Hq2gQzBug3f3/RY9dxyyX0yAmN2mAlvVJNqBBMGmARrvnR9sPXMx0Abr83OcY8yCYLkCD1GecPUBorgDlo59DhkHMbKIAjX+Hj79DOK+JArQzzvFnzP3AxcwYIGAQswRowOnPkkkQc5olQMCApgjQ4MefLYcgJjRFgIAxCRCQESAgc/sBuooB0JYxELO5/QABwxIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICBz+wFar5+3L15//7wb226Huz3Dbbv9AAHDEiAgI0BAZooAXcUYyACICU0RIGBMswRo8EOQ4w9zmiVAwIBmDNBoh6DR9gMXM1GAxn+6GX+HcF4TBWjMSZDpDzObK0BLIzRohD1AaLoALQ8a7f2/XN3xhzlNF6AB7/bR9gMXM2OARhgGGf3AvAHaa9AlM7RcTn2Y3LwBSuZB5j6wNHWALtwg9YE9swfosEFfkaG9y6oPbAnQH4pwxgwdXkp9YOfbfy/ntu3CavV995NtOB4enz53wcOESQ/sEaA31uvnZYP2OvJujE6cm9QHDgnQ0VLslehzU2rdgRME6AMPZZ/4OHCCAH2gI+/GSHTgQwToA/QFzkuAmMiPXw/1FnjD/wEBGQECMvebzUu3OjA1JyAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DAXeVfoA8KvC+p8OUAAAAASUVORK5CYII=', 'sha256': 'ec84e2b4622208384b4827b09ee55fd129233eb177974a0191588d90036a6896'}, 'data/binding-v1/images/binding-evaluation-0062-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEklEQVR4nO3dTWobSxhAUTskywl4EHg7yUayjmwkOwlkIHhb8UDgSQYGISynZRlZt6v6nKl/usDuy1eltnW/3z/eARQ+JVcFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZzd+kNeXj4tvwJu92fW60FVuR+v3+s17DR6PyLGLEdArSW7pxSIqYnQLfoztOPr8tf/uXn/wsfVSJmJUAfVZ+z0bkoRhrElAToyul5d3feUiIZYjICdLX6XDE9CxnSIGYiQOtNzzEZYkoeRBygPqdXueJrbRAyAV3m+M6/TXoWRiFzEKMzAY1UnxfXNQcxOgEaqT6nV9cghiZAg9XndA0axLgEaLz6PNMgJiBAF1hPfda5HriUAJ1xGC7WebcfVmUIYkQCtGSsu3qs1YIAvdU6x5/1rw2WeRBx1M3Xvx5Q3ObTiRNMf7tN/uBMQLP9Qo+7cjZIgM4YYvwZaJ1wzD+lZypf/vt1N5Sn39/vNswENPzpz4GX5BmOAAEZAQIyAjTJ/uuZXRhjESAgI0BARoCAjADNcwD0zDEQAxEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgF5/Z4LDm0wMZ+Nvj8FYBAjICBCQESAgI0BTHQM5AGIsAgRkBAjICNA8uzD7L4YjQEBGgM4YZQgaZZ1wTIBeN+4uZtyVs0ECNMNJkNMfBiVAb7LmBq15bbBMgJaMNVCMtVoQoOE3YjZfDE2ALrC2Bq1tPXApATrveLhYzz1/vBLjD4MSoCEbpD7MQYDGa5D6MA0BGqxB6sNMPtcLGMxu9+fh4dtxC55+fL3NpV8kT32YgAnoYi/u/NuMQurDlExA7/F8/99mFJIeJiZA19mOfUSGTmcr9WEyAnTNUei4Gu8u0at7OulhSgJ0/VHotCNnY7R8kKQ+zEqAruPQiNMSvfugWneYngDdtEQXfQeYngB9lOOOnI2R6LBNAnQL+gKv8iAikBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDjTzGYytPv7/USuIAJCMgIEJC53+8fu6sDm2YCAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABd5W/UZwzzwPg1gcAAAAASUVORK5CYII=', 'sha256': '2fb69a3e205bf581498eac5fc66326275b46f8fa49d9eceed11f366df3660e6a'}, 'data/binding-v1/images/binding-evaluation-0062-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3dQW4TWRRA0aRF9pMBEiO2wSJYUC+CbfSoJQZeTAaWmDCIZJXiYMfB8a3/65wpJP9JpK5elR18v98/3QEU/klOBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmU3f0hjw+fj79F3a7n7eaBVbkfr9/qmfYaHT+RIzYDgFaS3eOKRHTE6BbdOfhy4/TX/7r/28n/lSJmJUAfVR9zkbnohhpEFMSoCun593deUuJZIjJCNDV6nPF9JzIkAYxEwFab3qWZIgpeSPiAPU5PuWKr7VByAZ0meWVf5v0nFiF7EGMzgY0Un1enGsPYnQCNFJ9jk/XIIYmQIPV53gGDWJcAjRefZ5pEBMQoAuspz7rnAcuJUBnHJaLdV7th6ksQYxIgE4Z66oea1oQoLda5/qz/tngNG9EHPXm609vUNzmuxMn2P52m/yHswHN9gM97uRskACdMcT6M9CcsOQ/pWcq//37624oX78/3G2YDWj4pz8HXpJnOAIEZAQIyAjQJPdfz9yFMRYBAjICBGQECMgI0DwPgJ55DMRABAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBGg1z+Z4PAhE8PZ+MdjMBYBAjICBGQECMgI0FSPgTwAYiwCBGQECMgI0Dx3Ye6/GI4AARkBOmOUJWiUOWFJgF437l3MuJOzQQI0w5MgT38YlAC9yZobtObZ4DQBOmWshWKsaUGAhr8Rc/PF0AToAmtr0NrmgUsJ0HnL5WI91/xyEusPgxKgIRukPsxBgMZrkPowDQEarEHqw0wE6K8adMsMvThOfZiAAF3sxZV/mwa9OEV9mMOneoAhPV//j4+fl3V4+PLjI86SHiZmA7rmKnTdbej4G6oPk7EBXXMVWi4s716IXq2Y9DAlAbqC3e7nskHHHTkbo9Ork/owKwG6jkMjjkv07gfVusP0BOimJbroO8D0BOijLDtyNkaiwzYJ0C3oC7zKy/BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIxfxWAqX78/1CNwARsQkBEgIHO/3z91pwObZgMCMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAF3ld+5DD15GD3gIgAAAABJRU5ErkJggg==', 'sha256': '4f5ecb7adca4f519aaa9f6802396832115cc5c77c502e19d67085b61bdcb84d0'}, 'data/binding-v1/images/binding-evaluation-0062-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEklEQVR4nO3dvU4bSxiAYTgKlxOJIm1uKG2uI21u6LQpLJ1bobBEcwoky8JkjZHxuzP7PC0/OxLsq2/GC77f75/uAAr/JFcFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZLd+kNeXz8tvwJu92fW60FVuR+v3+q17DR6PyNGLEdArSW7pxSIqYnQLfozvPPr8tf/vDrv4WPKhGzEqDPqs/Z6FwUIw1iSgJ05fR8uDvvKZEMMRkBulp9rpiehQxpEDMRoPWm55gMMSUPIg5Qn9OrXPG1NgiZgC5zfOffJj0Lo5A5iNGZgEaqz6vrmoMYnQCNVJ/Tq2sQQxOgwepzugYNYlwCNF59XmgQExCgC6ynPutcD1xKgM44DBfrvNsPqzIEMSIBWjLWXT3WakGA3mud48/61wbLPIg46ubrbw8obvPpxAmmv90mf3AmoNl+ocddORskQGcMMf4MtE445p/SM5V/fz/fDeX7j4e7DTMBDX/6c+AleYYjQEBGgICMAE2y/3phF8ZYBAjICBCQESAgI0DzHAC9cAzEQAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARoLffmeDwJhPD2fjbYzAWAQIyAgRkBAjICNBUx0AOgBiLAAEZAQIyAjTPLsz+i+EIEJARoDNGGYJGWSccE6C3jbuLGXflbJAAzXAS5PSHQQnQu6y5QWteGywToCVjDRRjrRYEaPiNmM0XQxOgC6ytQWtbD1xKgM47Hi7Wc88fr8T4w6AEaMgGqQ9zEKDxGqQ+TEOABmuQ+jCTL/UCBrPb/Xl8/HbcguefX29z6VfJUx8mYAK62Ks7/zajkPowJRPQR7zc/7cZhaSHiQnQdbZjn5Gh09lKfZiMAF1zFDquxodL9OaeTnqYkgBdfxQ67cjZGC0fJKkPsxKg6zg04rREHz6o1h2mJ0A3LdFF3wGmJ0Cf5bgjZ2MkOmyTAN2CvsCbPIgIZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMj4Uwym8v3HQ70ELmACAjICBGTu9/un7urAppmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DAXeV/zH0yYePdLsUAAAAASUVORK5CYII=', 'sha256': 'd4847ef7cc9a20a6a1f21718f9bff033cff0fc591464210a43eabb0d1852ae2c'}, 'data/binding-v1/images/binding-evaluation-0063-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHklEQVR4nO3aO24bSRRAUXpgLWcABU69Iaezjkm9IacOCHgrDgQomYADghD1l6lbrXdOxEDsLkDNy1cFfrq5+b0DKPyV3BVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmc3dreKXr6y+7D2G//7mbzQQEZExAl/oS9uUGTxKgS03+p38sRhdy+8/fuw26+vdXvYRVCNB7nDgcr6BEcEqAXpmeH99vH3/v129XD11QhuBAgF6Qniej89Af34mRDMGBAD2rPi9Kz7nD288zpEEMJ0CPpeeN3bnjeLVjiYxCDOd3QO9Un0eu/GF+VgcvZQK6pwKXS8+dW5yOQuYgBjIBBfW5917mIAYSoKw+53fUIKaZHqC2Puf31SBGGR2gFepzfncNYo65AVqnPudr0CCGmBugpeqz2krgfQwN0HHEWO0zf1yPIYgJJgZoK5/trawTXm1igI5WG39WXhVcwrgALbv5OmUjxhDjAgSsY1aAtjhQbHHN8EyzAnS08v5rKyuEtxsaIGAFgwK0iePnU46i+fAGBQhYjQABGQECMlMCtLkDoAPHQHxsUwIELEiAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMlADt9z8PL75+u9ptx3G1x/XDRzIlQMCCBAjICBCQGRSgzR0DOQDiwxsUIGA1QwO0/hC0/grh7WYFaIt7mS2uGZ5pVoCApYwL0CaOoh0/M8S4AJ1as0FrrgouYWKAtjJWbGWd8GoTA7TyRszmi1GGBujUOg1aZyXwPuYG6HTEWOGTf7oG4w9DzA3QUg1SH2YaHaBFGqQ+jDU9QHmD1IfJBKhskPow3Od6AavY739eX3857cKP77eXu92dzKkPM5mAHqzA5UYh9YEDE9DuvAV3RqE/NQ2dF016GE6AntiO/ZFN2b3DlPqAAD1rFDrvyJMxemQHJz1w8Onm5vf/L3nAnQy9hfSs9h9p7cc/DyagFzwlr37uPWdwLwF6ZUeejJHowJNswYCM3wEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQJ2lf8APpEQa7s1tfoAAAAASUVORK5CYII=', 'sha256': 'adea0e167787c2725434a2da5d6f10316ba0dd7adb6b6353b5e1bd41ef11df85'}, 'data/binding-v1/images/binding-evaluation-0063-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHklEQVR4nO3aO24bSRRAUXpgLceAAqfe0KReh9PZ0KQOCHgrDgQomYADghD1l6lbrXdOxEDsLkDNy1cFfrq5+b0DKPyV3BVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmc3dreKXr66+7D2G//7mbzQQEZExAl/oS9uUGTxKgS03+p38sRhfy7z+3uw369vdVvYRVCNB7nDgcr6BEcEqAXpme2+9fHn/v1Y9fD11QhuBAgF6Qniej89Af34mRDMGBAD2rPi9Kz7nD288zpEEMJ0CPpeeN3bnjeLVjiYxCDOd3QO9Un0eu/GF+VgcvZQK6pwKXS8+dW5yOQuYgBjIBBfW5917mIAYSoKw+53fUIKaZHqC2Puf31SBGGR2gFepzfncNYo65AVqnPudr0CCGmBugpeqz2krgfQwN0HHEWO0zf1yPIYgJJgZoK5/trawTXm1igI5WG39WXhVcwrgALbv5OmUjxhDjAgSsY1aAtjhQbHHN8EyzAnS08v5rKyuEtxsaIGAFgwK0iePnU46i+fAGBQhYjQABGQECMlMCtLkDoAPHQHxsUwIELEiAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMlADt9z8PL65+/Nptx3G1x/XDRzIlQMCCBAjICBCQGRSgzR0DOQDiwxsUIGA1QwO0/hC0/grh7WYFaIt7mS2uGZ5pVoCApYwL0CaOoh0/M8S4AJ1as0FrrgouYWKAtjJWbGWd8GoTA7TyRszmi1GGBujUOg1aZyXwPuYG6HTEWOGTf7oG4w9DzA3QUg1SH2YaHaBFGqQ+jDU9QHmD1IfJBKhskPow3Od6AavY739eX3897cLt9y+Xu92dzKkPM5mAHqzA5UYh9YEDE9DuvAV3RqE/NQ2dF016GE6AntiO/ZFN2b3DlPqAAD1rFDrvyJMxemQHJz1w8Onm5vf/L3nAnQy9hfSs9h9p7cc/DyagFzwlr37uPWdwLwF6ZUeejJHowJNswYCM3wEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQJ2lf8As5gQa3La38YAAAAASUVORK5CYII=', 'sha256': '3a923d63b12d232cd200f9872ecb06850c03ca45894d5a4644fdc188de725c44'}, 'data/binding-v1/images/binding-evaluation-0063-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFE0lEQVR4nO3aO24bSxBAUfrB3o8Cp96QF+QNOVXAxTgQ4MQBHwhClE39qNvDOidiIM00oOFldYufHh5+7QAK/yV3BRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnc3Rpe6e7u6+4m7Pf3u9lMQEDGBHStD2EfbnCRAF1r8j/9YTG6kp8/fu826Nv3L/USViFAH3HicLyCEsEpAXplei5+9j75KXe4oAzBgQC9ID0vGvhPf/hRjGQIDgToWfV541nD4dfPM6RBDCdA/0rP+55xHq92LJFRiOF8D+iD6vOPK9/M1+rgpUxAT1TgA/65+2hTZjvGTCagoD5P3sscxEAClNXn/I4axDTTA9TW5/y+GsQoowO0Qn3O765BzDE3QOvU53wNGsQQcwO0VH1WWwl8jKEBOo4Yq73nj+sxBDHBxABt5b29lXXCq00M0NFq48/Kq4JrGBegZTdfp2zEGGJcgIB1zArQFgeKLa4ZnmlWgI5W3n9tZYXwdkMDBKxgUIA2cfx8ylE0N29QgIDVCBCQESAgMyVAmzsAOnAMxG2bEiBgQQIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGSmBGi/vz+8+Pb9y247jqs9rh9uyZQAAQsSICAjQEBmUIA2dwzkAIibNyhAwGqGBmj9IWj9FcLbzQrQFvcyW1wzPNOsAAFLGRegTRxFO35miHEBOrVmg9ZcFVzDxABtZazYyjrh1SYGaOWNmM0XowwN0Kl1GrTOSuBjzA3Q6Yixwjv/dA3GH4aYG6ClGqQ+zDQ6QIs0SH0Ya3qA8gapD5MJUNkg9WG4z/UCVrHf39/dfT3tws8fv693u0eZUx9mMgH9tQLXG4XUBw5MQLvzFjwahd5rGjovmvQwnABd2I69y6bsyWFKfUCAnjUKnXfkYoz+sYOTHjj49PDw6/+X/MWjDL2F9Kz2F2ntxz8PJqAXPCWvfu49Z/AkAXplRy7GSHTgIlswION7QEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgF3lD8LQEGvxl8jXAAAAAElFTkSuQmCC', 'sha256': 'e40b06300630944b4b9f3c5120a9cac44a4eaa6e05ca59fc7af4038b8adf3098'}, 'data/binding-v1/images/binding-evaluation-0064-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFLklEQVR4nO3dTU5bSRhAUdMKy0FikGVkN6yjd9PLyACpt8LAkicZIFmOje3E2FxXvXOmCbyPwbuqHxI/rNdvK4DCP8lTAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmW/foRXh+/n76L7y+/vyqWeDuPKzXb/UMyyrOaXrEogjQvaRnlwyxEAJ02+5sXp5Of/njv/+f+FMlYm4CdP30nI3O38ZIhpiVAF2tPhd3509KpEFMSYDuKz27ZIjpCdCn6nOj9JzIkAYxEwG6sD5fkJ5jGdIgpuE3oQeoz94Tr3jfDy0BGqA+h8/VIOZgC3be9m2v0nNsO2YvxuisgAarz+4k1kGMToAGq887DWIOAnTUKO/2KHPCIQG631Pn05xJMwH/H9AZ91mfd5uXp9P/lnVKg9bWjcGHrIBGOvo55DCIoQnQvnHf5HEnZ7FswY66/+XPkjdiq9Xqv8fH1d37sdnUI9w1K6CpFhGjz8/SCNDYy58Rp4UtAQIyAjTk5dch12GMSICAjACt5ls4zPSzMDcB2jfi/mv0yVksAQIyAgRkBAjICNDwF/C7XMYzFgECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQE6LcPjRv94yW28/sYPIYgQEBGgICMAAEZAdo37jHQuJOzWAK0mu/UdqafhbkJEJARoEku413AMyIBAjIC9LGxFkFjTQtbAvSb0fcvo8/P0gjQ8MuKUeaEQwI0zyJi3MlZLAEa+zrM5RdDE6Az7rlB9zwb/AkB+tjuguI+3/PdqSx/GJQAHTXKWz3KnHBIgIY8DHL0wxwEaLwGqQ/T+FYPMIDX15/Pz9+3b/7m5amaxLkPk7ECGuZMWn2YjwCN0SD1YUq2YJfsxb5yO7YXO/VhJgJ0eYNunaHDdZb6MBkB+mvvFbhphqSHhRCg6yyFdqtxcYmOHS2pD7MSoGsuhQ47cjZGp8+zpYe5CdBnbRtxWKKL78t0h4UQoJsviC74JrAQAnRlewU52yPFYckE6Lb0BU7wm9BARoCAjC0Yo/qx2dQj8FlWQEBGgIDMw3r91j0dWDQrICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAavKL5RqPvRIDVV2AAAAAElFTkSuQmCC', 'sha256': '062f12467ee62d635e6e87c17ead97bbabbadd8431f6e929322f4a5952ea7709'}, 'data/binding-v1/images/binding-evaluation-0064-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNklEQVR4nO3dS25aWRRAUVwKwynJjQyjZlPjyGw8jDQsZSpuILmThiVEwECMwZt731rNfPyOG2/rfpzwsNm8rAAK/yRPBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZb92jF+Hx8fvpP/D8/POrZoG787DZvNQzLKs4p+kRiyJA95KeXTLEQgjQbbvztF6f/uv/vb6e+F0lYm4CdP30nI3OR2MkQ8xKgK5Wn4u78zcl0iCmJED3lZ5dMsT0BOhT9blRek5kSIOYiQBdWJ8vSM+xDGkQ0/CT0APUZ++JV7zvh5YADVCfw+dqEHOwBTtv+7ZX6Tm2HbMXY3RWQIPVZ3cS6yBGJ0CD1eeNBjEHATpqlHd7lDnhkADd76nzac6kmYD/D+iM+6zPm6f1+vS/ZZ3SoLV1Y/AuK6CRjn4OOQxiaAK0b9w3edzJWSxbsKPuf/mz5I3YarV6/f/f1d1b//hVj3DXrICmWkSMPj9LI0BjL39GnBa2BAjICNCQl1+HXIcxIgECMgK0mm/hMNP3wtwEaN+I+6/RJ2exBAjICBCQESAgI0DDX8DvchnPWAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARoD8+NG70j5fYzu9j8BiCAAEZAQIyAgRkBGjfuMdA407OYgnQar5T25m+F+YmQEBGgCa5jHcBz4gECMgI0PvGWgSNNS1sCdAfRt+/jD4/SyNAwy8rRpkTDgnQPIuIcSdnsQRo7Oswl18MTYDOuOcG3fNs8DcE6H27C4r7fM93p7L8YVACdNQob/Uoc8IhARryMMjRD3MQoPEapD5MQ4A+1qA2Q7sDqA8TEKBhzqSdOjMfARqjQerDlATo8gZ9TYb2HqQ+zESAPmbv/b9phg6/uPowmW/1AON5q8Dj4/ftr7xl4mm9vtYjDqMmPUxJgC70/Pxzt0G71bi4RMcWU+rDrATomkuhw46cjdHpHZz0MDcB+qxtIw5LdPF9me6wEAJ08wXRBV8EFkKArmyvIGd7pDgsmQDdlr7ACX4OCMgIEJCxBWNU6x+/6hH4LCsgICNAQOZhs3npng4smhUQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAVeU371BPbJ7i+QIAAAAASUVORK5CYII=', 'sha256': '8742b3464e76f3f07a448d2d9fae88262ed794c4b08db6d389dad6fda0fdf2d1'}, 'data/binding-v1/images/binding-evaluation-0064-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJ0lEQVR4nO3dQU4bSxRAUfiK98Mgy8hyWUYGLIYBUiZ/gGQ5NhhwgOtXfc4wIfRDqG+qqwy+fXp6vAEo/JdcFUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkfnSX3oS7u5/nP+Dh4fd3zQJX5/bp6bGeYVvFOU+P2BQBupb0HJIhNkKAvrY797vd+X/+68+fM3+rRKxNgD4/PW9G56MxkiFWJUCfVp+Lu/OeEmkQSxKg60rPIRlieQL0T/X5ovScyZAGsRIBurA+35Ce1zKkQSzDK6EH1Ofoip943g8tARpQn9PrahBr8Aj2tv3dXqXntccxz2JMZwU0rD6Hk1gHMZ0ADavPMw1iDQL0qin39pQ54ZQAXe+u83n2pFmA3wf0huusz7P73e78z7IuaWhtnRi8yApo0tbPKZtBjCZAx+beyXMnZ7M8gr3q+pc/W34Qm/IN2ua35v2sgJZaREyfn60RoMH/uw6dFvYECMgI0MjDr1OOw5hIgICMAN2st3BY6WthbQJ0bOLz1/TJ2SwBAjICBGQECMgI0PgD+EMO45lFgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQH6603jpr+HwX5+b4PHCAIEZAQIyAgQkBGgY3O3geZOzmYJ0M16u7YrfS2sTYCAjAAtchjvAJ6JBAjICNDLZi2CZk0LewL0l+nPL9PnZ2sEaPyyYsqccEqA1llEzJ2czRKg2cdhDr8YTYDecM0NuubZ4D0E6GWHC4rrvM8Pp7L8YSgBetWUu3rKnHBKgEZuBtn6YQ0CNK9B6sMyBOhjDWozdDiA+rAAARqzJ23XmfUI0IwGqQ9LEqDLG/Q9GTq6kPqwEgH6mKP7/0szdPrJ1YfF/KgHmOe5And3P/d/8pyJ+93usy5xGjXpYUkCdKGHh9+HDTqsxsUlem0xpT6sSoA+cyl02pE3Y3T+CU56WJsA/at9I05LdPF5me6wEQL05QuiCz4JbIQAfbKjgrzZI8VhywToa+kLnOF1QEBGgICMRzCmup5fkMLFrICAjAABmdunp8fu6sCmWQEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAi4qfwPXHJQSTnhUpYAAAAASUVORK5CYII=', 'sha256': '48a920416a96f58cb5b96899cb389718ee2f6dad6f0ecc769c7d74e8a85d4aff'}, 'data/binding-v1/images/binding-evaluation-0065-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFklEQVR4nO3cMU6bSRiAYWcV7kMRKVWusYfIgXKIvUaqlSh8mBRINFsgeR2bQIgx78w/z1NR2P5Hwn75ZvSbD/f3P3YAhb+SqwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQ+dpeGd3J7+2m3afv93W5OJiAgI0BAxhaMhdx8/me3IQ///r2bnAkIyAgQkBEgICNAQEaAgIwAARkBAjLuA3q/+/3nvV8erkSA3u9LRscPFiMQoOzLjYdXUCJWJkBvlp7v3x6ef+6Xrze/ekEZYk0CdFF6XozOrx58EiMZYk0C9If1eVV6zj0+/TxDGsRSBOh16bmwOycOr3YokVGIpbgPKKvPM6+8+f/gB49MQL90XIHrpefkEsejkDmIzTMBDVGfJ69lDmLzBGig+pxfUYPYNgEaqz7n19UgNkyAhqvP+dU1iK0SoBHrc74GDWKTBOgJI9RntJXANQjQ6Ygx2mf+sB5DENsjQDN9tmdZJ/wmAfrJaOPPyKuCywnQuJuvYzZibJIAAZnVAzTjQDHjmuFJqwfoYOT91ywrhNcSICCzdICmOH4+5iiajVk6QEBLgICMAAGZdQM03QHQI8dAbMm6AQJyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZNYN0H5/9/jDl683u3kcVntYP8xr3QABOQECMgIEZJYO0HTHQA6A2JilAwS0BGg3yxA0/grhtVYP0Ix7mRnXDE9aPUBASIDmOIp2/MwmCdBPxmzQmKuCywnQTGPFLOuE3yRAo2/EbL7YMAF6wjgNGmclcA0C9L/jEWOET/7xGow/bJIADdog9WEFAjRig9SHRQjQcA1SH9YhQGM1SH1Yysd6AePa7+9ubz8dd+H7t4frXe4kc+rDCkxAzzmpwPVGIfVhTSagFzy24GQUeqtp6Lxo0sNSBOjV27E32ZQ9OUypD6v5cH//o17DTE4ydOzFGD2zg5Oe6re2Dftp3z8moIt2ZJefEM371oHLCdDukmr88Z9W3QFbsLf0YoxEB04IEJBxHxCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYFf5D66EEOJ9YUvHAAAAAElFTkSuQmCC', 'sha256': '335725919eb2638558b5aad1aa9c4ad081c95465802168f5e840a8c285e81ce6'}, 'data/binding-v1/images/binding-evaluation-0065-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEUlEQVR4nO3cPVJbSRSAUXnK2g+BU+/AK5kFzUrYwaQOWIwDqkgmoEojS/wIZPi6X58TEcB7t8ro4/ZD+Mv9/a8dQOGv5K4AAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZr92t4ZPc3Hzbbdrd3c/dnGxAQEaAgIwjGAv595+H3YZ8/3u/m5wNCMgIEJARICAjQEBGgICMAAEZAQIy3gf0ee/3n/f98vBBBOjz/sjo+JPFCAQo++PGwxWUiJUJ0B9Lz+3+lffF/3h4eO6CMsSaBOiq9Lwanec++SRGMsSa/BbsnfW53e/fVJ9Lvnzz/20NnLABveIkCtdE59zhaoeFyCrEUmxAWX1euLJViEUI0LOOK3DlgesSJ7fQIFYgQBfV59Puq0EsRYAGqs/5HTWIbROgsepzfl8NYsMEaLj6nN9dg9gqARqxPuczaBCbJEBPGKE+o00CH0GATleM0V7zh3ksQWyPAM302p5lTriQAP1mtPVn5KngegI07uHrmIMYmyRAQGb1AM24UMw4Mzxp9QAdjHz+mmVCeCsBAjJLB2iKx8/HPIpmY5YOENASICAjQEBm3QBN9wDokcdAbMm6AQJyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZNYN0N3dz8cPfjw87OZxmPYwP8xr3QABOQECMgIEZJYO0HSPgTwAYmOWDhDQEqDdLEvQ+BPCW60eoBnPMjPODE9aPUBASIDmeBTt8TObJEC/GbNBY04F1xOgmdaKWeaECwnQ6Acxhy82TICeME6DxpkEPoIA/e94xRjhlX88g/WHTRKgQRukPqxAgEZskPqwCAEarkHqwzoEaKwGqQ9LEaBLG/TRGTq5hfqwAgF6yUkFPq5BJ1dWHxbxtR5gdI8tuLn5dlKK2/3++oufF016WIoN6CLnXbjyUPbkl6sPq/lyf/+rnmEmh1Xo3Ks70QvBkp7qX20b7qb9/nEEu+pEdux9C9G83zpwPQHaXVONd/9o1R1wBPuTXo2R6MAJAQIyfgsGZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECNhV/gNuaRDayx6sIAAAAABJRU5ErkJggg==', 'sha256': 'f2cf63cbd6a8f34a516b4a2b9ef3cb453d8b6a0bd7454feeb193abdcff380850'}, 'data/binding-v1/images/binding-evaluation-0065-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEElEQVR4nO3cO24bSRRAUXpg7keBAW/C8SxyYm1iAAdajAMBSiYQwKFJfSjR0q3qOidyIHY/wOL1qxblL/f3v3YAhb+SuwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGS+dreGT3Jz8223aXd3P3dzsgEBGQECMo5gLGT//Z/dhjz8+/ducjYgICNAQEaAgIwAARkBAjICBGQECMj4HNDnfd5/3s/LwwcRoM/7JaPjLxYjEKDslxsPV1AiViZAfyw9t/v9y6/98fDw3AVliDUJ0FXpeTU6z33xSYxkiDX5Kdg763O737+pPpe8fPP/bQ2csAG94iQK10Tn3OFqh4XIKsRSbEBZfV64slWIRQjQs44rcOWB6xInt9AgViBAF9Xn0+6rQSxFgAaqz/kdNYhtE6Cx6nN+Xw1iwwRouPqc312D2CoBGrE+5zNoEJskQE8YoT6jTQIfQYBOV4zR3vOHeSxBbI8AzfTenmVOuJAA/Wa09WfkqeB6AjTu4euYgxibJEBAZvUAzbhQzDgzPGn1AB2MfP6aZUJ4KwECMksHaIrHz8c8imZjlg4Q0BIgICNAQGbdAE33AOiRx0BsyboBAnICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRk1g3Q3d3Pxz/8eHjYzeMw7WF+mNe6AQJyAgRkBAjILB2g6R4DeQDExiwdIKAlQLtZlqDxJ4S3Wj1AM55lZpwZnrR6gICQAM3xKNrjZzZJgH4zZoPGnAquJ0AzrRWzzAkXEqDRD2IOX2yYAD1hnAaNMwl8BAH63/GKMcI7/3gG6w+bJECDNkh9WIEAjdgg9WERAjRcg9SHdQjQWA1SH5YiQJc26KMzdHIL9WEFAvSSkwp8XINOrqw+LOJrPcDoHltwc/PtpBS3+/31Fz8vmvSwFBvQRc67cOWh7MmXqw+r+XJ//6ueYSaHVejcqzvRC8GSnupvbRvupv3+cQS76kR27H0L0bzfOnA9AdpdU413/9OqO+AI9ie9GiPRgRMCBGT8FAzICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKv8B0ZoENp6OKBkAAAAAElFTkSuQmCC', 'sha256': '1eb42fe1d9581ed5709583c64c5a6ec2b38cd8dccf4c08d7f73eaca60eae9b7e'}, 'data/binding-v1/images/binding-evaluation-0066-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKUlEQVR4nO3aS05bWRRAUVIKwymJRiRmkonUODIRZhKpGkiZShpI6VTDkuXCBOPw2ff5rNVCAnxPA2+dd/Gnh4efVwCFv5JTAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzOfu6K26ufny/A/c3//7UbPAtn16ePhZz3AJ0fkdMYJnCNC7dOeYEsExATo7Pde3d8//7q/vX3/3LRmCQwL0ovScjM5ZMZIh2BGgE/X54/Q8nyENAgH6bXrepDsnSyRDS93TreN+zB+GzwF9XH2OX/ki3zzwcgL0vwpc3969X32ePEKDmGz6BxEf1efDzr2+vds/jt3cfJmzcr+fX//8fbVx199+XA0zegOq6nN8oj2ImeYGqK3P8bkaxEBDA7RCfY5P1yCmmRigdepzPIMGMcrEAC1Vn9UmgY80LkD7FWO19/x+HksQc8wK0Fbe21uZE15pVoD2Vlt/Vp4K3s+gAC378HXIgxijDAoQsJopAdriQrHFmeEsUwK0t/Lz11YmhLcyLkDAOkYEaBPXz4dcRTPEiAABaxIgICNAQObyA7S5C6Ad10BMcPkBApYlQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5vIDdH//7+6LX9+/Xm3Hftr9/HB5Lj9AwLIECMgIEJAZEaDNXQO5AGKIEQEC1jQuQOsvQetPCG9lSoC2+CyzxZnhLFMCBCxoUIA2cRXt+plRBgXo0JoNWnMqeD+zArSVtWIrc8IrzQrQyg9iHr4YaFyADq3ToHUmgY80MUCHK8YK7/zDGaw/jDIxQEs1SH2YbGiAFmmQ+jDc3ADlDVIfGB2gsEHqAwL0RIPeO0OPjlAfJhOgJyrwfg169Mrqw3Cf6wFWsWvBzc2XR6W4vr17/YsfF016wAZ0dbILr3woe/LX1Qd2bEAnVqGdw4ic3ImeCZb0wCEBOiNDO3+2EEkPHBOgF1XjyRKd9QrAMQE6uyMnYyQ68EICdDZ9gbfiv2BARoCAjAABGQECMgIEZAQIyAgQkBEgIOODiFyI628/6hE4mw0IyAgQkPn08PCzOx0YzQYEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEHBV+Q/3+CU7yLitMgAAAABJRU5ErkJggg==', 'sha256': 'f009a67d2ff0810997b0149df742a2ef7fb287689c3024e4ba505e962b661082'}, 'data/binding-v1/images/binding-evaluation-0066-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHklEQVR4nO3dQU5bWRBAUdKK98Mgy8hyWUYGLCYDJCY9sGS5MWDcYO77rnNGSAl+JQVf1X9Yyo+np793AIV/klMBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMz+7orbq///X+X3h8/PNds8C2/Xh6+lvPcAvReYsYwTsE6CrdOaVEcEqALk7Pw273/vf+fn5+649kCI4J0IfSczY6F8VIhmBPgM7U53+n5/0MaRAI0Jvp+ZLunC2RDC11T7eOxzE/GD4H9H31OX3lm3zzwMcJ0H8q8LDbXa8+rx6hQUw2/YOIL+rzbec+7HaHx7H7+19zVu7r+c5/viv5/fbvT2/V6A2oqs/pifYgZpoboLY+p+dqEAMNDdAK9Tk9XYOYZmKA1qnP6QwaxCgTA7RUfVabBL7TuAAdVozV3vOHeSxBzDErQFt5b29lTvikWQE6WG39WXkquJ5BAVr24euYBzFGGRQgYDVTArTFhWKLM8NFpgToYOXnr61MCF9lXICAdYwI0Caun4+5imaIEQEC1iRAQEaAgMztB2hzF0B7roGY4PYDBCxLgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzO0H6PHxz/6L38/Pd9txmPYwP9ye2w8QsCwBAjICBGRGBGhz10AugBhiRICANY0L0PpL0PoTwleZEqAtPstscWa4yJQAAQsaFKBNXEW7fmaUQQE6tmaD1pwKrmdWgLayVmxlTvikWQFa+UHMwxcDjQvQsXUatM4k8J0mBuh4xVjhnX88g/WHUSYGaKkGqQ+TDQ3QIg1SH4abG6C8QeoDowMUNkh9QIBeadC1M/TiCPVhMgF6pQLXa9CLV1YfhvtZD7CKfQvu73+9KMXDbvf5Fz8tmvSADejubBc++VD26rerD+zZgM6sQnvHETm7E70TLOmBYwJ0QYb2/t9CJD1wSoA+VI1XS3TRKwCnBOjijpyNkejABwnQxfQFvorfggEZAQIyAgRkBAjICBCQESAgI0BARoCAjA8iciNW+A9OuJQNCMgIEJD58fT0tzsdGM0GBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBBwV/kXyasnWPm2PmsAAAAASUVORK5CYII=', 'sha256': '466434a142afcab619bff5e0eccb3f62839de9509686e75ef68ba3d7402bfeb2'}, 'data/binding-v1/images/binding-evaluation-0066-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJUlEQVR4nO3aQU5bWRBAUdKKl9MSgywju+l1ZDcsIwOk3koGSEx6YMlyYwI4YO77rnNGSIBfDfBV/Ye/PDz8ugEo/JWcCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZL52R2/V7e23l3/g/v7nZ80C2/bl4eFXPcM1ROd3xAheIEAX6c4pJYJTAnR2eu52u5d/9/vj4+++JUNwTIDelJ5Xo3NWjGQI9gTolfr8cXpezpAGgQD9Nj0f0p1XSyRDS93TreN+zB+GzwF9Xn1OX/kq3zzwdgL0vwrc7XaXq8+zR2gQk03/IOKT+nzauXe73eFx7Pb225yV+3Ie//n7ZuN2P/69GWb0BlTV5/REexAzzQ1QW5/TczWIgYYGaIX6nJ6uQUwzMUDr1Od0Bg1ilIkBWqo+q00Cn2lcgA4rxmrv+cM8liDmmBWgrby3tzInvNOsAB2stv6sPBVczqAALfvwdcyDGKMMChCwmikB2uJCscWZ4SxTAnSw8vPXViaEjzIuQMA6RgRoE9fPx1xFM8SIAAFrEiAgI0BA5voDtLkLoD3XQExw/QECliVAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDm+gN0f/9z/8X3x8eb7ThMe5gfrs/1BwhYlgABGQECMiMCtLlrIBdADDEiQMCaxgVo/SVo/Qnho0wJ0BafZbY4M5xlSoCABQ0K0Cauol0/M8qgAB1bs0FrTgWXMytAW1krtjInvNOsAK38IObhi4HGBejYOg1aZxL4TBMDdLxirPDOP57B+sMoEwO0VIPUh8mGBmiRBqkPw80NUN4g9YHRAQobpD4gQM806NIZenKE+jCZAD1Tgcs16Mkrqw/Dfa0HWMW+Bbe3356U4m63e/+LnxZNesAGdPNqF975UPbsr6sP7NmAXlmF9o4j8upO9EKwpAeOCdAZGdr7s4VIeuCUAL2pGs+W6KxXAE4J0NkdeTVGogNvJEBn0xf4KP4LBmQECMgIEJARICAjQEBGgICMAAEZAQIyPojIldj9+LcegbPZgICMAAGZLw8Pv7rTgdFsQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABN5X/AIPDJns014kSAAAAAElFTkSuQmCC', 'sha256': 'ce183ebaea408781bb70456a635d4fd66a3e3a392060856dc20bac8bf3036d0a'}, 'data/binding-v1/images/binding-evaluation-0067-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMElEQVR4nO3aMW4bRxiAUTqwjmOAhdpcKK3P4TYXSquCgK/iQoCaFAQImqQoUxb97cy+V7mglz9l7eeZWX56fv6xASj8lbwrgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmc/fWM9tuH6+/YLd7+lOzzP/DHJFfgD0Bam6Sk9f7dWSdBGgR/z/vLyJDrI0A3aU7L1+/XP/rD9++X7mmEl30378vm8H9/c9DPcKyCNCHpefN6Lz24vMYWRCxEgL0u/W5qTsXHa5wUqLt9lGDmJsAlem5eMHjDFkKMTcBek99Pjw9b2ZIg5iSAN1Wn7um50qGNIgp+Sb0Eutz8R2n/D4eKydAy63P+ftqEJOxBXvV4W6v0nNxO2YvxkysgJZen4PDJNZBTEOAxqjPngYxGQE6Ncq9PcqccIUALe7U+Tpn0sxEgC5bZn2WPxvcRIAGOPo55zCIOQjQZvQ7edzJQYBOLX/5M9accIUAzbCIGH1+VkuABl5WjDUtnBMgICNAIz38OudxGEMTICCz9gDNtHCY6bOwEmsP0MGI+6/RJwcBAjICBGQECMisOkBDP4A/5mE8g1p1gICWAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmVUHaLd72v/h4dv3zcgO8x8+EQxh1QECWgIEZAQIyAjQZvRjoHEnh7UHaKZT25k+Cyux9gABIQEa+2G8B/AMTYCAjAD9ZKxF0FjTwjkBmmH/Mvr8rJYAjbqsGGVOuEKAhl9EjDs5CNCQj8M8/GIOAnTZkhu05NngJgL0k+MFxTLv8+OpLH8YnQCdGuWuHmVOuEKARjoMcvTDZARomAapD/P5XA+wXLvd03b7eLjzX75+qSZx7sOsrICWfiatPkxMgBbdIPVhbrZgN+zF/uR27CR26sOUBOjmBt07Q+frLPVhVgL0q/YVuGuGpIe1EaDfWgodV+PdJXrtaEl9mJ4AfcBS6Lwjb8bo+nm29LASn56ff9QzjO28RO+mO/f+CS+Hf+s9K6B7LYjecRFYGwH6GCcFebNHigO2YEDJN6GBjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESBgU/kfRakuccrw38QAAAAASUVORK5CYII=', 'sha256': 'd8138f8302d87047a8f0ab0d8bf7c7532273900a1212b6a6688d9a80e3b86de6'}, 'data/binding-v1/images/binding-evaluation-0067-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNklEQVR4nO3aQU4bSRiAUWcU38cLpLlE1jlk1lxiJBY+TBZIbGZhyXJsMJgAX1f1eyuETPdvoD+qq/n2+Ph7A1D4JzkrgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDme3fqme12d9dfsN8/fNUs838zR+QX4ECAmovk7PV+HVknAVrE3+fDQWSItRGgT+nO/XZ7/ct/PD1dOaYSPWv776/N4J7++1mPsCwC9GHpeTU6L734MkYWRKyEAP1tfW7qzrOORzgr0W53p0HMTYDK9Dx7wNMMWQoxN/8H9J763G+3H16fKwef8jk0CNCbnF7/n5qeU2cn0iCmJEC31eeLz65BzE2Allufy/NqEJMRoBcdr/Yvu+16yekAGsRMBOj1+myWQYOYjwCNUZ8DDWIyAnRulGt7lDnhCgFa3K7zdfakmYkAPW+Z9Vn+bHATARpg6+eSzSDmIECb0a/kcScHATq3/OXPWHPCFQI0wyJi9PlZLQEaeFkx1rRwSYCAjACN9PDrksdhDE2AgMzaAzTTwmGm98JKrD1ARyPef40+OQgQkBEgICNAQGbVARr6AfwpD+MZ1KoDBLQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIrDpA+/3D4YMfT0+bkR3nP74jGMKqAwS0BAjICBCQEaDN6NtA404Oaw/QTLu2M70XVmLtAQJCAjT2w3gP4BmaAAEZAfrDWIugsaaFSwI0w/3L6POzWgI06rJilDnhCgEafhEx7uQgQEM+DvPwizkI0POW3KAlzwY3EaA/nC4olnmdn05l+cPoBOjcKFf1KHPCFQI00maQrR8mI0DDNEh9mI8AvalBbYZOB1AfZiJAS9+TtuvMxARo0Q1SH+YmQDc36GsydHYi9WFKAvQmZ9f/p2bo8uDqw6y+1wMM41CB3e7u+JlDJu632486xWXUpIe5CdBt9vuH0wadVuPdJXppMaU+TE+APmApdNmRV2N0/Q5OeliJb4+Pv+sZxnZZonfTnc/+Di+Hn/WBFdBnLYjecRBYGwH6GGcFebVHigNuwYCS/wMCMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCATeV/NvkuZS6mcocAAAAASUVORK5CYII=', 'sha256': 'e082c049e678db53ba3d0322dff586663f522b35b21a74146c478bdd725373fb'}, 'data/binding-v1/images/binding-evaluation-0067-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFOklEQVR4nO3aQW4aSRiAUTIK92GR7dwgJ5kDzUl8g9l6wWGysOTNLJAQAYyNY/vrqn5vFUWk+wfTX6qr/e3p6dcGoPBXclYAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMz37tQz2+1+3H7Bfv/4VbPM/2GOyBfgQICai+Ts9b6OrJMALeL/58NBZIi1EaBP6c7Ddnv7n/98fr5xTCW66r9/r3xoY/n7n1e+GGsjQB+Wnlej89KLL2NkQcRKCNCf1ueu7lx1PMJZiXa7HxrE3ASoTM/VA55myFKIufk9oPfU52G7/fD63Dj4lM+hQYDe5PT6/9T0nDo7kQYxJQG6rz5ffHYNYm4CtNz6XJ5Xg5iMAL3oeLV/2W3XS04H0CBmIkCv12ezDBrEfARojPocaBCTEaBzo1zbo8wJNwjQ4nadb7MnzUwE6Lpl1mf5s8FdBGiArZ9LNoOYgwBtRr+Sx50cBOjc8pc/Y80JNwjQDIuI0edntQRo4GXFWNPCJQECMgI00sOvSx6HMTQBAjJrD9BMC4eZ3gsrsfYAHY14/zX65CBAQEaAgIwAAZlVB2joB/CnPIxnUKsOENASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgs+oA7fePhz/8fH7ejOw4//EdwRBWHSCgJUBARoCAjABtRt8GGndyWHuAZtq1nem9sBJrDxAQEqCxH8Z7AM/QBAjICNBvxloEjTUtXBKgGe5fRp+f1RKgUZcVo8wJNwjQ8IuIcScHARrycZiHX8xBgK5bcoOWPBvcRYB+c7qgWOZ1fjqV5Q+jE6Bzo1zVo8wJNwjQSJtBtn6YjAAN0yD1YT4C9KYGtRk6HUB9mIkALX1P2q4zExOgRTdIfZibAN3doK/J0NmJ1IcpCdCbnF3/n5qhy4OrD7P6Xg8wjEMFdrsfx785ZOJhu/2oU1xGTXqYmwDdZ79/PG3QaTXeXaKXFlPqw/QE6AOWQpcdeTVGt+/gpIeV+Pb09KueYWyXJXo33fnsT3g5/KwPrIA+a0H0joPA2gjQxzgryKs9UhxwCwaU/B4QkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEbCr/A03SLmXbjZHAAAAAAElFTkSuQmCC', 'sha256': 'defa1eb33bc5fe96a0c8025d6644c5eec2ad2b367780eff0f1260260ccd34e29'}, 'data/binding-v1/images/binding-evaluation-0068-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFB0lEQVR4nO3dMW4iSRiAUbyauQ+B07nBnGQPNCfZG0zqgMM4QHKyAZJlgQ02Br6u4r3Uxl0B/emvMqgfttvnFUDhn+SqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyP7pL3531+vH4L2w2T7daCyzCw3b7XK/hrqPzETHiHgjQsrpzSImYmADdLj1//7wcf+2vf39+9CMZYkoCdN30nIzOl2IkQ0xGgK6SnrO785kSyRDTEKAL1+eC6TmSIQ1iDgJ0sfpcKT0fZUiDmIAALXrwOWQUYiY+CT1SfQ6vdcH/98PtmYCWvu36iO0YEzABDVmfvaubgxiU74Kdqa3P6xqOfHZxelNmd3Nnw6wJ6Jx3/BLqs7eSKe9GpidAn7X8O3z5K4Q9tmCDHf0c34it14/3NsOvVqv/fg6/D/39sqw31c2YgL5mafVZ8qrgJAEa8ujnkMMgRiRAQEaAZhh/dgxBDEeAgIwATTL+7BiCGIsAARkBAjICNM/+a8cujIEIEJARICAjQEBGgKY6ANpxDMQoBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBGg970+XWvEZx+/rvkOnxHGWAQIyAgQkBEgICNAsx0DOQBiIAIEZAQIyAjQVLsw+y/GIkBARoDmGYKMPwxHgICMAE0yBBl/GJEAfc0yG7TMVcFJAvQpb8eKpd3tb9dj/GEsAvRZy7+3l79C2CNAYx8GOfphaAJ0piU0aAlrgO8QoFEPgxz9MIEf9QLGs9k8rdePbyvw98/LLRewFz71YVwmoHPs3fO3HIXUh5kI0Jk2m6fbb8f2tl3qw+hswcbYjhl8mJIAXbJB18jQ4WylPkxDgC5gV4TDDH2nRO/u6aSHyQjQFTO015GTMTpykCQ9TEmAbpGh7xxUSw8TE6CreK3GuyX60l+AiQnQdb3tyMkYiQ73RoBuR19gjw8iAhkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjI+DIqw/v9ctPHInFBJiAgI0BA5mG7fe6uDtw1ExCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYFX5H0ErKs1mkoaIAAAAAElFTkSuQmCC', 'sha256': '821d9106aff589e99776d2adb100b5a392279c7dfbd93417b99990972077679f'}, 'data/binding-v1/images/binding-evaluation-0068-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFCklEQVR4nO3dPU5bTRhAYfMp7McFUqpsI4vIgrKIbOOrIqVgMRSWaFJYQsgGGxzb5874eTrEz0zBPXpnMPLdZvO0Aij8l6wKIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzpVv65qzXD4e/4PHxz7X2Aotwt9k81Xu46ei8R4y4BQK0rO7sUyImJkDXS8/911+Hv/f59/f3PiVDTEmALpueo9H5VIxkiMkI0EXSc3J3PlIiGWIaAnTm+pwxPQcypEHMQYDOVp8Lpee9DGkQExCgRQ8++4xCzMQroUeqz/5aZ/x7P1yfCWjpx673OI4xARPQkPXZWd0cxKD8L9iJ2vq87OHAaxenN2V2H29smDUBnfIbv4T67OxkyqeR6QnQRy3/CV/+DmGHI9hgVz+HD2Lr9cOtzfCr1er/n8+rwX37cb+6SSagz1lafZa8KzhKgIa8+tnnMogRCRCQEaAZxp8tQxDDESAgI0CTjD9bhiDGIkBARoCAjADNc/7acgpjIAIEZAQIyAgQkBGgqS6AtlwDMQoBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBOhtL++uNeJ7H7/s+QbfI4yxCBCQESAgI0BARoBmuwZyAcRABAjICBCQEaCpTmHOX4xFgICMAM0zBBl/GI4AARkBmmQIMv4wIgH6nGU2aJm7gqME6ENejxVLe9pf78f4w1gE6KOW/2wvf4ewQ4DGvgxy9cPQBOhES2jQEvYA/0KARr0McvXDBAToXxt0/QztLKo+jEuATrHzzF+zQTtrqQ9D+1JvYFTbJ3+9fth++Pz7+/3XX5de1ODDZExAYxzHHLuYkgCd/zh23gzt/0D1YRqOYOc/jr0+K518LnuzYtLDZAToghna6cjRGB0YnaSHKQnQNTK0ddrRTHqYmABdxEs13izRp34CTEyALut1R47GSHS4NQJ0PfoCO/wZHsgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAxj+jMrxvP+7rLXAiExCQESAgc7fZPHWrAzfNBARkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIWFX+AhtwPE8x5oVSAAAAAElFTkSuQmCC', 'sha256': '08c510c54982fd96a91aec3418372b834865797beefa33159d0ca61a50a9d992'}, 'data/binding-v1/images/binding-evaluation-0068-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFBUlEQVR4nO3dPU5bTRhAYROF/bhAyiZSZ5Gpv01ESsFiKCy5+QpLCNlgg2P73Bk/T4f4mSm4R+8MRn7YbF5WAIVvyaoAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjLfu6Xvznr9dPwLnp//3movsAgPm81LvYe7js5HxIh7IEDL6s4hJWJiAnS79Dz++H38e7d/fn30KRliSgJ03fScjM6XYiRDTEaArpKes7vzmRLJENMQoAvX54LpOZIhDWIOAnSx+lwpPR9lSIOYgAAtevA5ZBRiJl4JPVJ9Dte64N/74fZMQEs/dn3EcYwJmICGrM/e6uYgBuV/wc7U1ud1D0deuzi9KbP7fGfDrAnonN/4JdRnbydTPo1MT4A+a/lP+PJ3CHscwQa7+jl+EFuvn+5thl+tVv89Pq4G93O7Xd0lE9DXLK0+S94VnCRAQ179HHIZxIgECMgI0Azjz44hiOEIEJARoEnGnx1DEGMRICAjQEBGgOY5f+04hTEQAQIyAgRkBAjICNBUF0A7roEYhQABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjIC9L7Xd9ca8b2PX/d8h+8RxlgECMgIEJARICAjQLNdA7kAYiACBGQECMgI0FSnMOcvxiJAQEaA5hmCjD8MR4CAjABNMgQZfxiRAH3NMhu0zF3BSQL0KW/HiqU97W/3Y/xhLAL0Wct/tpe/Q9gjQGNfBrn6YWgCdKYlNGgJe4B/IUCjXga5+mECAvSvDbp9hvYWVR/GJUDn2Hvmb9mgvbXUh6F9rzcwqt2Tv14/7T7c/vn1+OP3tRc1+DAZE9AYxzHHLqYkQJc/jl02Q4c/UH2YhiPY5Y9jb89KZ5/L3q2Y9DAZAbpihvY6cjJGR0Yn6WFKAnSLDO2cdzSTHiYmQFfxWo13S/SlnwATE6DretuRkzESHe6NAN2OvsAef4YHMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJDxz6gM7+d2W2+BM5mAgIwAAZmHzealWx24ayYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMCq8j/tRz3lrmcRcQAAAABJRU5ErkJggg==', 'sha256': 'd4c1cf4c8e413064ac5966efad846c31ba6697710d0309bae8c35c77b5a72442'}, 'data/binding-v1/images/binding-evaluation-0069-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRUlEQVR4nO3aMU4jSRiAUbMa34dgpL0E8R5yYy6x0gQ+DAGSkw0sWQw2MMwAX1fVexkSuKvB/emvwjePjw87gMJfyVUBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkvnWXntbt7fdf+bbD4cfnrwU27ebx8aFew0LReYkYfeVve8sOi70TTECbeBLOr7Pa+4/FCdCnpOd+v3/9x++Ox1deVoZYhAB9THreLM7r3/+sRzL0i/Z//7ubwvG/f3ZLEqA/Ss97u/OS8+s8LZEMMT0BitNz9WVliEX4HNC763O/339SfV65xMT/92FlAvSGp0/+F6TnqWeX0yDmYwv2jsEnWcb9fn/ekdmOMRkT0Kbrc/XqRiGmIUDb2na9xHaMKQnQG/XZbYkGMRkBGqY+JxrETAToum3WZ/trg3cRoN3lQLH9J/y8QkMQQxOg3ehP8rgrBwEa4+jnksMgJiBAPxmlPiOuFi4J0EhHP5ccBjE0AQIyqwdo6PHnxBDEuFYPEBBaOkATjD8nhiAGtXSAgNa6AZpm/DkxBDGidQME5AQIyCwaoMn2Xyd2YQxn0QABWyBAQEaAgMyKAVrhiGSFe2QCKwbobKYT6FnviLktHSCgJUBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQGbpAN0dj7u5zHdHzG3FAB0OP3azW+EemcCKAQI2QoCAjAABmUUDdD4imenU9nwvDoAYxaIBArZAgIDMugGabBdm/8WI1g0QkFs6QNMMQcYfBrV0gIDW6gGaYAgy/jCu1QMEhARo7CHI+MPQBOgnYzVorNXCJQF6Pj6M8lQ/Xafxh0EJ0PDP8LgrBwEa8jDI0Q9zEKDrttygLa8N3kWABjsMcvTDTARopAapD5MRoLcbtIUMPVuG+jAHAbru2RPeNujZ1dWHaXyrF7Bdp+f89vb76cu74/F+v//6ZRh8mJgJaLvbMdsupidAv7Md++wMXV5CfZjSzePjQ72GYZy3Y2cfvim7TJv0/MofYhqHxf7czoB+/1ToaS/+sERXR6rV3ossSIA+IEOXBXmzR69v4qSHRdiCbWsvID0sRYA2USLdYU0ClMVIdECAgIzPAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAXeV/2RZH02+PbQEAAAAASUVORK5CYII=', 'sha256': '6ac2b548595711480710bda1b7ccde3c0ba5cd6a58ad8540534e936dc1933260'}, 'data/binding-v1/images/binding-evaluation-0069-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRklEQVR4nO3aPU5jSRhAUTNq9kPQEpuYmEUSzyaQOmAxBEgkE1iyaNv8dQP3VdU5GRL41QO/q68KXzw+PuwACv8kVwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD50V16WldXP9/zbff3v75+LbBpF4+PD/UaForOS8ToO3/bW3a/2DvBBLSJJ+HwOqu9/1icAH1Jei6vb1//8ae7m1deVoZYhAB9TnreLM7r33/UIxl6p/8uL3dT+PfpabckAfqr9Hy0Oy85vM7zEskQ0xOgOD1nX1aGWITPAX24PpfXt19Un1cuMfH/fViZAL3h+ZP/Del57uhyGsR8bME+MPgky7i8vj3syGzHmIwJaNP1OXt1oxDTEKBtbbteYjvGlATojfrstkSDmIwADVOfPQ1iJgJ03jbrs/21wYcI0O50oNj+E35YoSGIoQnQbvQnedyVgwCNcfRzymEQExCg34xSnxFXC6cEaKSjn1MOgxiaAAGZ1QM09PizZwhiXKsHCAgtHaAJxp89QxCDWjpAQGvdAE0z/uwZghjRugECcgIEZBYN0GT7rz27MIazaICALRAgICNAQGbFAK1wRLLCPTKBFQN0MNMJ9Kx3xNyWDhDQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgILN0gJ7ubnZzme+OmNuKAbq//7Wb3Qr3yARWDBCwEQIEZAQIyCwaoMMRyUyntod7cQDEKBYNELAFAgRk1g3QZLsw+y9GtG6AgNzSAZpmCDL+MKilAwS0Vg/QBEOQ8YdxrR4gICRAYw9Bxh+GJkC/GatBY60WTgnQ8fgwylP9fJ3GHwYlQMM/w+OuHARoyMMgRz/MQYDO23KDtrw2+BABGuwwyNEPMxGgkRqkPkxGgN5u0BYydLQM9WEOAnTe0RPeNujo6urDNH7UC9iu/XN+dfVz/+XT3c3l9e33L8Pgw8RMQNvdjtl2MT0B+pPt2Fdn6PQS6sOULh4fH+o1DOOwHTv49E3Zadqk5z1/iGncL/bndgb056dCz3vxlyU6O1Kt9l5kQQL0CRk6LcibPXp9Eyc9LMIWbFt7AelhKQK0iRLpDmsSoCxGogMCBGR8DgjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE7Cr/A+zvR/sQWTjPAAAAAElFTkSuQmCC', 'sha256': '397b2a2210cd13e36a4bcf8400bfad33cc96ee98b072b12017c33ee06d665d6d'}, 'data/binding-v1/images/binding-evaluation-0069-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQklEQVR4nO3aTU4bSxhAUfMU9sMgEjthkewEKQMvJgMkJm9gySK2+UuA21V1zgwJ3NXgvnxVcPX4+HsHUPgvuSqAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMj+7S07q5+fmeT9vvf339WmDTrh4ff9drWCg6LxGj7/xub9l+sXeCCWgTT8LxdVZ7/7E4AfqS9Fzf3r/+5U8Pd6+8rAyxCAH6nPS8WZzXP/+kRzL0Th/9tm/W0wu/kKYnQP+Uns96AI6v8/yNKENMT4Di9Fx8WRliEf4P6MP1ub69/+rJ//wSE//dh5UJ0BueP/nfkJ7nTi6nQczHFuwDg0+yjOvb++OOzHaMyZiANl2fi1c3CjENAdrWtusltmNMSYDeqM9uSzSIyQjQMPU50CBmIkCXbbM+218bfIgA7c4Hiu0/4ccVGoIYmgDtRn+Sx105CNAYRz/nHAYxAQH6wyj1GXG1cE6ARjr6OecwiKEJEJBZPUBDjz8HhiDGtXqAgNDSAZpg/DkwBDGopQMEtNYN0DTjz4EhiBGtGyAgJ0BAZtEATbb/OrALYziLBgjYAgECMgIEZFYM0ApHJCvcIxNYMUBHM51Az3pHzG3pAAEtAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMksH6OnhbjeX+e6Iua0YoP3+1252K9wjE1gxQMBGCBCQESAgs2iAjkckM53aHu/FARCjWDRAwBYIEJBZN0CT7cLsvxjRugECcksHaJohyPjDoJYOENBaPUATDEHGH8a1eoCAkACNPQQZfxiaAP1hrAaNtVo4J0Cn48MoT/XzdRp/GJQADf8Mj7tyEKAhD4Mc/TAHAbpsyw3a8trgQwRosMMgRz/MRIBGapD6MBkBertBW8jQyTLUhzkI0GUnT3jboJOrqw/T+FEvYLsOz/nNzc/Dh08Pd9e399+/DIMPEzMBbXc7ZtvF9ATob7ZjX52h80uoD1O6enz8Xa9hGMft2NGnb8rO0yY97/lBTGO/2I/bGdDfnwo978U/lujiSLXae5EFCdAnZOi8IG/26PVNnPSwCFuwbe0FpIelCNAmSqQ7rEmAshiJDggQkPF/QEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESBgV/kfDGZH+9hxYpcAAAAASUVORK5CYII=', 'sha256': '09bcc0bff4301c4de875f33bb15a79d6b6ebad8b0584e4d3968f68bff8286dbc'}, 'data/binding-v1/images/binding-evaluation-0070-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHUlEQVR4nO3dS04jSRRAUWhVLweJQe2MdfTOaoDUW2GAxKQGliwLU+ZTwI3Md86UT4Zk5dWLILGvHx8frgAK/yRXBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzo7v0Ttze/rz8Dff3v75rLbAx14+PD/UadhidPxEjOCVA39Gdc0oEAvQJ3Xm6u7n84//+9/+FryoRkwnQR+rzanTeFSMNYiwBekd6Ptydt5RIhhhIgN5Un09Mz4UMaRDTCFCTnlMyxFgeRIzrc36VT/xbGyzOBPTynf896bkwCpmDmMAEtEp9nl3XHMQEArRKfc6vrkHsngAtVJ/zNWgQ+zY9QKvV50CDGGJ6gBasz5rrga8wOkDH4WLNu/24KkMQezU3QNu6q7e1Wngjb0i26Phz8HR3c/mf6dlrr+9nPAg2dAJafPN1ykaMHZsYoO3eydtdObxo9BZs/fHnwEZs9y/x0bQXeuIEBCxiXIA2dPpzykkQuzQuQMA6BAjIzArQRvdfB3Zh7M+sAAFLESAgI0BAZlCANn0AdOAYiJ0ZFCBgNQIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQGBej4OSfbfdvd48qHfGYLuzcoQMBqBAjICBCQmRWgTR8DOQBif2YFCFiKAAGZcQHa6C7M/otdGhcgYB2jA7SVIWgr64T3mhig7e5itrtyeNHEAG3rJMjpDzs2NECnVm7QymuDvzc3QNsaKLa1WnijuQFafyNm88XujQ7QqdUatNp64CtMD9DpcLHOPX+6EuMPOzY9QAs2SH2YQ4DWapD6MIoALdQg9WGaH/UCFnJ//+v29udpC57ubr7n0s+Spz4MYQK6unDnf88opD6MZQJ67nD/f88oJD0MJ0Cvb8e+IkPns5X6MJAAvXUUOq3Gh0v04p5OehhLgN43Cp135NUYXT5IUh8mE6DXHRtxXqIPH1TrDgjQZ5boXb8BEKCrv+/IqzESHfgTAfpb+gIf5kFEICNAQEaAgIwAARkBAjICBGQECMgIEJARICDjSWj2bIWPOeECExCQESAgc/34+NBdHRjNBARkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECLiq/AZeqDFhtsT4tQAAAABJRU5ErkJggg==', 'sha256': '5477a02f7075b59490ccab3cfed7fb6cb55c9c24b817d306fa8ae91a53b9ca90'}, 'data/binding-v1/images/binding-evaluation-0070-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIElEQVR4nO3dS0psSRRAUS2e87HxwJnUIJ1JwWs4GBsJ1amGIFmmpn933HvW6vqJALmbE2FqXh4O9xcAhb+SVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjI/OqW3onr69/nP+Hu7s9P7QU25vJwuK/3sMPovESM4JgA/UR3TikRCNAXdOfq5vb8l//7z99nPqpETCZAH6nPq9F5V4w0iLEE6B3p+XB33lIiGWIgAXpTfb4wPWcypEFMI0BNeo7JEGN5IWJcn9NVvvB3bbA4E9DzT/7PpOfMKGQOYgIT0Cr1ebKuOYgJBGiV+pyurkHsngAtVJ/TPWgQ+zY9QKvV54EGMcT0AC1YnzX3A99hdIAeh4s1n/bHXRmC2Ku5AdrWU72t3cIb+Ydki44/D65ubs//MT177fXdjBeCDZ2AFj98HXMQY8cmBmi7T/J2dw7PGn0EW3/8eeAgtvsf8aNpP+iJExCwiHEB2tDtzzE3QezSuAAB6xAgIDMrQBs9fz1wCmN/ZgUIWIoAARkBAjKDArTpC6AHroHYmUEBAlYjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAZlCAHt/nZLv/dvdx50Pes4XdGxQgYDUCBGQECMjMCtCmr4FcALE/swIELEWAgMy4AG30FOb8xS6NCxCwjtEB2soQtJV9wntNDNB2TzHb3Tk8a2KAtnUT5PaHHRsaoGMrN2jlvcHnzQ3QtgaKbe0W3mhugNY/iDl8sXujA3RstQatth/4DtMDdDxcrPPMH+/E+MOOTQ/Qgg1SH+YQoLUapD6MIkALNUh9mEaAXmzQT2boyXLqwxAC9D9PnvyfadCTVdSHOX7VG1jOw/N/ff37uA5XN7ffsZb0MJwJ6K2j0NdOQ6ffUH0YyAT01lHoeGD58ED0bMWkh7EE6BV3d3+OG3TakVdjdH50Uh8mE6DXPTbitEQfvqjWHRCgryzRu74DIEAXn+/IqzESHXiJAH2WvsCH+TU8kBEgICNAQEaAgIwAARkBAjICBGQECMgIEJDxSmj2bIW3OeEMExCQESAgc3k43HerA6OZgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAReV/wCF0T0uXvpTEQAAAABJRU5ErkJggg==', 'sha256': '4b963893738d191e08f71816d08852420c80f2ca0fd502d261e8c8212a5e3a6c'}, 'data/binding-v1/images/binding-evaluation-0070-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFLklEQVR4nO3dPU5cSRSAUTwyyxmJwJJ3MhuZdXgj7GQkB0jeCgGSEwdIqIeG5p+v3rvnpEBXJe/TreoGvtzcXJ8BFP5KVgUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg87VbeicuLr6d/oarq5+ftRfYmC83N9f1HnYYnceIERwSoM/ozjElAgF6h+6cf788/eO///vnxFeViMkE6DX1eTI6L4qRBjGWAL0gPa/uznNKJEMMJEDPqs87pudEhjSIaQSoSc8hGWIsH0SM63O8yju+1waLMwE9/OR/TnpOjELmICYwAa1Sn3vrmoOYQIBWqc/x6hrE7gnQQvU53oMGsW/TA7RafW5pEENMD9CC9VlzP/ARRgfobrhY82m/25UhiL2aG6BtPdXb2i08kz9Ituj4c+v8++XpX6Znr72+mvFBsKET0OKHr0MOYuzYxABt90ne7s7hQaOPYOuPP7ccxF7t979/n23K+Y9fZ5NMnICARYwL0IZufw65CWKXxgUIWIcAAZlZAdro+euWUxj7MytAwFIECMgIEJAZFKBNXwDdcg3EzgwKELAaAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMoMCdPd/Trb795Xvdj7kf7awe4MCBKxGgICMAAGZWQHa9DWQCyD2Z1aAgKUIEJAZF6CNnsKcv9ilcQEC1jE6QFsZgrayT3ipiQHa7ilmuzuHB00M0LZugtz+sGNDA3Ro5QatvDd4u7kB2tZAsa3dwjPNDdD6BzGHL3ZvdIAOrdag1fYDH2F6gA6Hi3We+cOdGH/YsekBWrBB6sMcArRWg9SHUQRooQapD9MI0KMN+swM3VtOfRhCgP7n3pP/OQ26t4r6MMfXegPLuX3+Ly6+Hdbh/PvlR6wlPQxnAnruKPS+09DxC6oPA5mAnjsKHQ4srx6IHqyY9DCWAD3h6urnYYOOO/JkjE6PTurDZAL0tLtGHJfo1RfVugMC9J4letErAAJ09vaOPBkj0YHHCNBb6Qu8mrfhgYwAARkBAjICBGQECMgIEJARICAjQEBGgICMT0KzZ+c/ftVb4BQTEJARICDz5ebmulsdGM0EBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIOKv8AaO7PHmv+VBoAAAAAElFTkSuQmCC', 'sha256': '8ebd721e32901f0e7640bbb3c70d321e01865c3419c5df37ab8b05a5f739c4da'}, 'data/binding-v1/images/binding-evaluation-0071-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFklEQVR4nO3dPU5jSRhAURj17MdBp7OhWRAbIu2AxRBYIukACVm2sWkafOtVnZPy80pC7+qrwn6+3++f7wAK/yRXBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMj+7Sm7Hb/bz8DU9Pv261FpjK/X7/XK9hk9F5jxjBxwnQ13TnlBLBVQJ0PT2PDy+Xf/a///9970syBBcI0Pn0XI3OH8VIhuCspQN0mp5Pd+cjJZIhOLJugI7q84XpuZAhDYJDiwbosD7flJ73MqRBsG6AbjP4nDIKweqvhK7qc3qtL/x/P2zXQhPQjbdd77Edg+UmoEHqc3R1cxCLWyVAg9RnnDXACJYI0NugMc6d/7YSQxArmz9A49/h468QvsnkARrn6OeUwyCYPEDD1mfkVcHNzBygAY9+TjkMYmWeiMiIps+xl4BNPgFtYvx5ZQhiWdMGCBjfnFuwDY0/rx4fXl7forHb/TScH9rKX/CDLjw8c00mICAjQEBmwgBtbv/1ylE0C5owQMBWCBCQESAgM1uANnoA9MoxEKuZLUDAhggQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJCZLUBvjxPc4qPn3tbsoYgsYrYAARsiQEBGgIDMhAHa6DGQAyAWNGGAgK0QICAzZ4A2twuz/2JNcwYI2IRpA7ShIcj4w7KmDRAwvpkDtIkhyPjDymYO0KExGzTmquBmJg/Q4Vgx2t1+uB7jD2uaPECbuLfHXyF8k/kDNOZhkKMfWCVAh0Zo0AhrgBGsEqBxDoMc/cCbH3fLeHr6tdv9PKzA48PLLRdwFD71gVUmoLP3/C1HIfWB1QP0eufffjt2tO1SH1huC5Zsxww+cMGiATpq0Hdk6HS2Uh84sm6A3opwmqG/KdHZPZ30wFlLB+i9DB115GqMLhwkSQ9cIECXMvQ3B9XSA1cJ0PlqnC3RH/0G4CoBut6RqzESHfgcAbpOX+CbLPdCRGAcAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDGx/IwtM99LC1bYQICMgIEZO73++fu6sDSTEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE3FV+A1fCKIsoOHnZAAAAAElFTkSuQmCC', 'sha256': 'd10ced5f3d53687efde196c59ee1121f52fd007fe9f14698fb1a56ef4d3d91fb'}, 'data/binding-v1/images/binding-evaluation-0071-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJ0lEQVR4nO3dQU4bSRiAUTMKxxmJBRI3yUXmHLlIbhJpFpFyFRZIbLKwhCxs7ASBv+qq93YIcNemP/1VNs3N09PjDqDwT3JVAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh86S69GXd39+d/4OfP/6+1FpjKzdPTY72GTUbnLWIEf06APqY7x5QILhKgy+m5ffh+/neff3x961syBGcI0On0XIzOX8VIhuCkpQN0nJ53d+dPSiRD8Mq6AXpVnw9Mz5kMaRAcWjRAh/X5pPS8lSENgnUDdJ3B55hRCFb/JHRVn+NrfeD7/bBdC01AV952vcV2DJabgAapz6urm4NY3CoBGqQ+46wBRrBEgF4GjXHu/JeVGIJY2fwBGv8OH3+F8EkmD9A4Rz/HHAbB5AEatj4jrwquZuYADXj0c8xhECvzRERGNH2OfQRs8gloE+PPniGIZU0bIGB8c27BNjT+7N0+fN//icbd3b3h/NDzf//uJnL77Ve9hLGYgICMAAGZCQO0uf3XnqNoFjRhgICtECAgI0BAZrYAbfQAaM8xEKuZLUDAhggQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJCZLUAvjxPcP2BwW17W7KGILGK2AAEbIkBARoCAzIQB2ugxkAMgFjRhgICtECAgM2eANrcLs/9iTXMGCNiEaQO0oSHI+MOypg0QML6ZA7SJIcj4w8pmDtChMRs05qrgaiYP0OFYMdrdfrge4w9rmjxAm7i3x18hfJL5AzTmYZCjH1glQIdGaNAIa4ARrBKgcQ6DHP3AcgE6btD1M/TqouoDCwXo+J6/ZoNeXUt9YLfbfdktZn/n393d7798/vH19uH7Z1/U4AMnrTUBXX87ZtsFZywaoJPbsY/N0PELqg+svgU7sx073Cu9e192smLSAyctHaC3MvSqIxdjdGZ0kh44Q4DOZWjvfVsz6YGLBOh0NU6W6K9eAbhIgC535GKMRAfeR4Au0xf4JOu+DQ/kBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMf8vD0G6//aqXwCcyAQEZAQIyN09Pj93VgaWZgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjYVX4DAc467onq9Z0AAAAASUVORK5CYII=', 'sha256': '5e5733966d28af91eef1932c9e157b5c8a36637a5f071221c063efebbbe12098'}, 'data/binding-v1/images/binding-evaluation-0071-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJ0lEQVR4nO3dMU5bSxiAUfMU9kMRiSrbyCKyoLcItvGqSCmyGAokmldYQhY2doLA39yZczoE+E5zP/0zNpebp6fHHUDhn+SqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD50l16M+7uvp7/gd+/f11rLTCVm6enx3oNm4zOW8QI/pwAfUx3jikRXCRAl9Nze/9w/neff35/61syBGcI0On0XIzOX8VIhuCkpQN0nJ53d+dPSiRD8Mq6AXpVnw9Mz5kMaRAcWjRAh/X5pPS8lSENgnUDdJ3B55hRCFb/JHRVn+NrfeD7/bBdC01AV952vcV2DJabgAapz6urm4NY3CoBGqQ+46wBRrBEgF4GjXHu/JeVGIJY2fwBGv8OH3+F8EkmD9A4Rz/HHAbB5AEatj4jrwquZuYADXj0c8xhECvzRERGNH2OfQRs8gloE+PPniGIZU0bIGB8c27BNjT+7N3eP+z/ROPu7qvh/NB//z7vJvLtx229hLGYgICMAAGZCQO0uf3XnqNoFjRhgICtECAgI0BAZrYAbfQAaM8xEKuZLUDAhggQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJCZLUAvjxPcP2BwW17W7KGILGK2AAEbIkBARoCAzIQB2ugxkAMgFjRhgICtECAgM2eANrcLs/9iTXMGCNiEaQO0oSHI+MOypg0QML6ZA7SJIcj4w8pmDtChMRs05qrgaiYP0OFYMdrdfrge4w9rmjxAm7i3x18hfJL5AzTmYZCjH1glQIdGaNAIa4ARrBKgcQ6DHP3AcgE6btD1M/TqouoDCwXo+J6/ZoNeXUt9YLfbfdktZn/n39193X/5/PP77f3DZ1/U4AMnrTUBXX87ZtsFZywaoJPbsY/N0PELqg+svgU7sx073Cu9e192smLSAyctHaC3MvSqIxdjdGZ0kh44Q4DOZWjvfVsz6YGLBOh0NU6W6K9eAbhIgC535GKMRAfeR4Au0xf4JOu+DQ/kBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMf8vD0L79uK2XwCcyAQEZAQIyN09Pj93VgaWZgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjYVf4HzRo6NVSYdGQAAAAASUVORK5CYII=', 'sha256': '904aa40685cfc11811f9de71950872669486e40508b95112a14c5d6019b39845'}, 'data/binding-v1/images/binding-evaluation-0072-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE/ElEQVR4nO3dMVIbSxRAUeMy+yHwMv5yvYwfsBgCVTlxQJUsJJAACd+e7nNSYOYluvWmAelut3v6BlD4ntwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGR+dLee08PDz/Pf8Pj4/7+aBUZ3t9s91TPMH523iBGLE6CgO6eUiDUJ0I3T8+v+/vzP/vf791tfkiFWI0A3SM/F6HwoRjLEOgToqvp8Oj3nM6RBLEKAPpOem3TnYolkiOn5O6BR6nN65RsecsOYbEAXHFbg69JzZhWyBzExG9Bw9Tm6lz2IiQnQcPU5vaMGMSsBGrE+p/fVIKYkQIPW5/TuGsR8BGjc+pzOoEFMRoDeNEJ9RpsEbkuAXtivGKO95vfzWIKYiQD9tZXX9lbmhIsE6BWjrT8jTwXXEKDRH74OeRBjMgIEZARoqwvFFmeGIwL0wsjPX1uZEN7Pp2Iwvwm2xcdJ3xTBBrSN4+dDjqKZhgABGY9gLGQrS+57PkNlDjYgILN6gDZ3APTMMRBzWD1AQEiAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM6gHav83Ktv7rbz/trG8TwyJWDxAQEiAgI0BARoC2dwzkAIhpCBCQEaAXxl+Cxp8Q3k+Atvoss8WZ4YgAARkB2tJRtONnJiNArxizQWNOBdcQoL+2slZsZU64SIC28SDm4YspCdCbxmnQOJPAbQnQscMVY4RX/uEM1h8mI0BDN0h9mJsAjdsg9WF6AjRog9SHFQjQiA1SHxYhQB9r0Fdn6OgW6sPcBOiyowp8XYOOrqw+TO9HPcA2PLfg4eHnUSl+3d9ff/HTokkPi7ABfcBpF658KHv1x9WHddiArlqFnh1G5OJOdCZY0sNqBOhmGXr2uYVIeliTAH3evhqvluhDV4A1CdANHHbkYoxEB/YE6Mb0Bd7Pb8GAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZb8nKQr76k7X5KBsQkBEgIHO32z11dweWZgMCMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBHyr/AFWTydv+ZjNeAAAAABJRU5ErkJggg==', 'sha256': 'ba96b554fda920226ddf5250e16b8d7e8359d7fda84f86e8066acba1c1243df6'}, 'data/binding-v1/images/binding-evaluation-0072-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFDElEQVR4nO3dQU4bSxRAUYjCciIxiPR3ko1kHdlIdhIpA0vZCgNLTP4gkmNswASa3Orqc6aA+0189apkmev9/u4KoPAheSqAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyHzsHj2n29vPz//CbvfzX80Co7ve7+/qGeaPzlPEiI0ToKA755SIbRKghdNz//XT83978+3XUz+SIbZGgBZIz8Xo/FWMZIjtEKA31efV6Xk+QxrERgjQa9KzSHculkiGmJ7PAY1Sn/NXXvCSG8ZkA7rguALvl55nViF7EBOzAQ1Xn5Nn2YOYmAANV5/zJ2oQsxKgEetz/lwNYkoCNGh9zp+uQcxHgMatz/kMGsRkBOhJI9RntElgWQL0wGHFGO09f5jHEsRMBOiPtby31zInXCRAjxht/Rl5KngLARr98HXMQYzJCBCQEaC1LhRrnBlOCNADI5+/1jIhvJz/isH8JtgWd5N+KYINaB3Xz8dcRTMNAQIyjmBsyM1/369W5f7Hl6up2YCAzNYDtLoLoN9cAzGHrQcICAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJDZeoAOX7Ny8+3X1Xocpp31a2LYiK0HCAgJEJARICAjQOu7BnIBxDQECMgI0APjL0HjTwgvJ0BrPcuscWY4IUBARoDWdBXt+pnJCNAjxmzQmFPBWwjQH2tZK9YyJ1wkQOs4iDl8MSUBetI4DRpnEliWAJ06XjFGeOcfz2D9YTICNHSD1Ie5CdC4DVIfpidAgzZIfdgCARqxQerDRnysBxjdbvfz9vbzcRfuv356v8edZE59mJsN6LKTCrzfKqQ+bI0N6EV+t+BkFVpqGzovmvSwEQL0yuPYIoeyR5cp9WE7BOhNq9B5Ry7G6JkTnPSwNQK0WIbeckMkPWyTAL3eoRqPluivXgG2SYAWcNyRizESHTgQoIXpC7yczwEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyvpKVDbn/8aUegQdsQEBGgIDM9X5/1z0d2DQbEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgqvI/AuUclt6OA20AAAAASUVORK5CYII=', 'sha256': '8f21f23d6289dadbb849185bcaaf9e3309bd07b52817b2088fce77d11aaae277'}, 'data/binding-v1/images/binding-evaluation-0072-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFCUlEQVR4nO3dQU4bSxRAUYjCciIxyDKym6wju8kyMkDKVhhY8uQPIjnGBkxw82919TlTwP0mvnpVssztbvd4A1D4lDwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQ+d4+e0/3919d/4eHh1/81C4zudrd7rGeYPzovESM2ToCC7pxTIrZJgBZOz/77l9f/9u7H75d+JENsjQAtkJ6L0fmnGMkQ2yFAV9Xn3el5PUMaxEYI0HvSs0h3LpZIhpiezwGNUp/zV17wkhvGZAO64LgCH5eeV1YhexATswENV5+TZ9mDmJgADVef8ydqELMSoBHrc/5cDWJKAjRofc6frkHMR4DGrc/5DBrEZAToRSPUZ7RJYFkC9MRhxRjtPX+YxxLETATor7W8t9cyJ1wkQM8Ybf0ZeSq4hgCNfvg65iDGZAQIyAjQWheKNc4MJwToiZHPX2uZEN7Of8VgfhNsiw+TfimCDWgd18/HXEUzDQECMo5gbMjPu7ubVfm2399MzQYEZLYeoNVdAP3hGog5bD1AQEiAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM1gN0+JqVux+/b9bjMO2sXxPDRmw9QEBIgICMAAEZAVrfNZALIKYhQEBGgJ4Yfwkaf0J4OwFa61lmjTPDCQECMgK0pqto189MRoCeMWaDxpwKriFAf61lrVjLnHCRAK3jIObwxZQE6EXjNGicSWBZAnTqeMUY4Z1/PIP1h8kI0NANUh/mJkDjNkh9mJ4ADdog9WELBGjEBqkPG/G5HmB0Dw+/7u+/Hndh//3Lxz3uJHPqw9xsQJedVODjViH1YWtsQG/ypwUnq9BS29B50aSHjRCgdx7HFjmUPbtMqQ/bIUBXrULnHbkYo1dOcNLD1gjQYhm65oZIetgmAXq/QzWeLdE/vQJskwAt4LgjF2MkOnAgQAvTF3g7nwMCMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkfCUrG/Jtv69H4AkbEJARICBzu9s9dk8HNs0GBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAi4qfwH+U4cvnILeLsAAAAASUVORK5CYII=', 'sha256': '664e9e94414eb78520d9781776bf641e1a207ff81e06735e82e2b9b8736e5a81'}, 'data/binding-v1/images/binding-evaluation-0073-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFG0lEQVR4nO3aQU4bSRiAUWcU7sMiEjfhkNwEKQsOkwUSm1lYcjzGmHgCfFXV7+2QiLs6uD/+Kvzt+fnXDqDwT3JVAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZL53l4b33d7+2C3n6elnvYRRCNAoj5A3JRskQKP83j7+J2LERgjQiPuFw+so0cHN3cNuci+P9/UShiNAX5Gedx+et96a+5eVIVYlQJ+Snmt/XZ98/0mPZIhVCdBHpuejtgmH1zkukQyxHgEaKz1nX1aGWJUPIv5tfW7uHj77fPT1JZb8dAwbJEDXOX7yvyA9x04up0EswBbs/w8+yTJu7h4OOzLbMWZnApqpPmevbhRiXgI09LbrLbZjrEGArqvPbiQaxOwEaNb67GkQUxOgPzJmfcZfG1wmQG86DBTjP+GHFRqCmIsAnTfvkzzvytkgAZry6Oc1h0HMSIAumaU+M64WBGjuo5/XHAYxFwECMgK0zvizZwhiIgIEZARoqfFnzxDELAQIyAjQauPPniGIKQgQkBEgICNAC+6/9uzCGJ8AARkBAjICBGQEaBNHJFu4R2YkQL+tdAK96h2xGAECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQE6LeXx/vdWta7IxYjQLunp5+71W3hHpmRAAEZAQIyAgRkBOg/RyQrndoe7sUBEMMSICAjQEBGgNbchdl/MQUBAjICtOAQZPxhFgIEZARotSHI+MNEBAjICNBSQ5Dxh7kI0CVzNWiu1YIAnXc8PszyVB+v0/jDLATovHmf4XlXzgYJ0AqHQY5+mJQA/ZGRGzTy2uAyAZr7MMjRD1MToIkbpD7MToCubtAIGTpZhvowqW/Pz7/qNczh9vbH8Zc3dw/VSk4KuHZ9Tv7b17D2j+wq36/79g3bv2kOz8PL433SIIMPK7EFm2Y7ZtvFemzBPmZf8KnT0OvMqQ9rEKChMyQ9rE2APuWU9C9LdHZnJz2sR4C+4o817/bo8lmS9LAqARr6b8bSw9oEaMQS6Q4bIUCjxEh02CABAjI+iAhkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DArvIvsrg94LzOsQcAAAAASUVORK5CYII=', 'sha256': '7fda35fd11f254207b86d468e21427105765077c019a3cc8755cee38d1a12683'}, 'data/binding-v1/images/binding-evaluation-0073-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3bwU5TQRiA0Wr0cUy64M18Dt+MRRNfxQWJGxdNai2loALfzNxz1tDOhd6v/0zhw8PDjx1A4WPyrAACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh86p4anrff3+2Wczjc10sYhQCNcgt5UbJBAjTK+/b5t4gRGyFAI+4XTo+jRCc/v37ZTe7zt+/1EoYjQO+RnmdvnqdemseHlSFWJUBvkp6/fbu++PqLHskQqxKg10zPa20TTo9zXiIZYj0CNFZ6rj6sDLEqAfrf+rzD4ejVDGkQCxCgf6/PO38uc5EhDWIB/hXjpfb7u7A+V5/3YkkwHQEadNt1w8WzaxDzsgV73giDzwXbMdZgApqvPk9tx9K1wL8QoFnrc6RBTE2AXmTM+oy/NrhNgJ50GijGv8NPKzQEMRcBum7eO3nelbNBAjTl0c9jDoOYkQDdMkt9ZlwtCNDcRz+POQxiLgIEZARonfHnyBDERAQIyAjQUuPPkSGIWQgQkBGg1cafI0MQUxAgICNAQEaAFtx/HdmFMT4BAjICBGQECMgI0CaOSLZwjcxIgH5b6QR61StiMQIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgI0G+fv33frWW9K2IxArQ7HO53q9vCNTIjAQIyAgRkBAjICNAfRyQrndqersUBEMMSICAjQEBGgNbchdl/MQUBAjICtOAQZPxhFgIEZARotSHI+MNEBAjICNBSQ5Dxh7kI0C1zNWiu1YIAXXc+PsxyV5+v0/jDLATounnv4XlXzgYJ0AqHQY5+mJQAvcjIDRp5bXCbAM19GOToh6kJ0MQNUh9m96lewAQOh/v9/u78nv/59Uu7pIsUqg+T+vDw8KNewxxODToKG7Sp+lz82New9q/sr9iCvdThcD/Cduxi2+WlzNRswabZjm1q8GEjbMFeZ1/wphl6PG2pD2sQoKEzJD2sTYDe5JT0P0t09YBJeliPAL3HhzXP9uj2kbb0sCoBGvozY+lhbQI0Yol0h40QoFFiJDpskAABGX8JDWQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMCu8gtESUTeO3gvZwAAAABJRU5ErkJggg==', 'sha256': '6c04eec708a1374f61c9d190f0b9cac54b176bfb6f2fd303e12dc799586f6ccf'}, 'data/binding-v1/images/binding-evaluation-0073-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIklEQVR4nO3bTU4bSRiAYWcUjhPJC6TcZC6Sc+Qic5ORWFjKVbJAYjMLS47HmDb5gber+nnW2K4G98tXZfjw+Ph9B1D4K3lVAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZD52Lw237ff3u+kcDg/1EtZCgNZyC3lTskECtJbf2+cPESM2QoDWuF84PY8Sndx9/mc3uKd//66XsDoC9B7pefryafnhd1+/LTytDDErAXqT9NwszvLXX/RIhpiVAP3J9Pxsd15yep7zEskQ8xGgdaXn6tPKELMSoN+tzxul52aGNIgJCNCv1+cd0rOQIQ1iAv4V47X2+/uwPldf92JJMBwBWum2a8HFq2sQ47IFu20Ng88F2zHmYAIarz4vbcfStcCvEKBR63OkQQxNgF5lnfVZ/9pgmQC96DRQrP8OP63QEMRYBOi6ce/kcVfOBgnQkEc/zzkMYkQCtGSU+oy4WhCgsY9+nnMYxFgECMgI0Dzjz5EhiIEIEJARoKnGnyNDEKMQICAjQLONP0eGIIYgQEBGgICMAE24/zqyC2P9BAjICBCQESAgI0CbOCLZwjUyIgH6YaYT6FmviMkIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0A/3H39tpvLfFfEZARodzg87Ga3hWtkRAIEZAQIyAgQkBGg/x2RzHRqe7oWB0CslgABGQECMgI05y7M/oshCBCQEaAJhyDjD6MQICAjQLMNQcYfBiJAQEaAphqCjD+MRYCWjNWgsVYLAnTd+fgwyl19vk7jD6MQoOvGvYfHXTkbJEAzHAY5+mFQAvQqa27QmtcGywRo7MMgRz8MTYAGbpD6MLqP9QIGcDg87Pf35/f805dP7ZIuUqg+DOrD4+P3eg1jODXoKGzQpupz8W2fw9w/sp9iC/Zah8PDGrZjF9sub2WGZgs2zHZsU4MPG2EL9mf2BW+aoefTlvowBwFadYakh7kJ0Juckv5mia4eMEkP8xGg9/iw5maPlo+0pYdZCdCqPzOWHuYmQGsske6wEQK0lhiJDhskQEDGX0IDGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKv8B7pbRN6b1/dBAAAAAElFTkSuQmCC', 'sha256': 'c2cb175d65a935afa6410850c0c9032d4252404094c3f3d6eaa59089baed3862'}, 'data/binding-v1/images/binding-evaluation-0074-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE0ElEQVR4nO3dMU4bQRiAUYjgOJEoaHOhtDkHbS6UNgUSV6FAoknhyLJsjC1s+Maz79U2O81++mewdq9fXp6vAArfkqsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkLnpLg1Hubu7v5rC4+PfegnDMQEBGQECMrZgXIzXX9+vLtDtw1O9hHGZgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwnIgYPUfdwclgRoODNDZsfFiOWTIDil8asvi5DLJMAnb87f36/vv/dHz9v9/01JWJRBOg89TkYnX0f3orR3d29BrEcAvSl6dm1+vpmhmzKWA4B+nh9TkzPwQxpENMToI/U54zpeSdDGsT0/BBxlPq8+fdP/BcbDE6AxqrP7lU0iInZgh2wvv+/Jj1vbsfsxZiVCWjE+qytr2sOYkoCtNdo9/xo64HTCdAo5z77OA9iYgJ0QFufcdYAn0GARjz62eUwiCkJ0Lbx7/DxVwhHEqC9xhl/xlwPnE6AgIwAjX76s8lJEJMRICAjQBcz/qwYgpiJAAEZAQIyAgRkBOiSDoBWHAMxDQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQE6L/1q0dXLyMd2XqFXpfKpRMgICNAQEaAgIwAXdgxkAMgZiJAQEaALmkIMv4wmZt6AXCs24enegmcmQlor9GGoNHWA6cToG3j727GXyEc6frl5fnYzy7JaM+od/rDlExAF7DxGWEN8BkE6G2bg0Z7/29e3fjDZARor9Hu9tHWA6cToKF/FuToh7k5hD5s8wWkX3YmbefFEpiARjwPUh8WQoCGa5D6sBy2YB/ci33GdmwrberD9ATopAadK0O7U5X6sAQCFGdIelgyATpbg46P0TunSOrDogjQJ5boeLrDMglQnCHpYckEKIiR6MCKAAEZP0QEMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRcVf4BpfQFYhL56qoAAAAASUVORK5CYII=', 'sha256': 'daaeb581d80c5e2b62e1cc87c0c82fb449436174882f624f6a188558f479b223'}, 'data/binding-v1/images/binding-evaluation-0074-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE0ElEQVR4nO3dMU4bQRiAURPBcSK5oOVCtDlH2lwobQqkXIUCiSaFI8sCjC1s+Maz79U2O81++mewdq+enh5XAIVvyVUBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyFx3l4ajrNe3qyk8PPyplzAcExCQESAgYwvGxfj963l1ge7ub+oljMsEBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZDwRMXiIuoeTw4YABW9u2P2wGLFkAhS/NGbzdRlimQTo/N15/vH9/e/e/Py7768pEYsiQOepz8Ho7Pvwixit17caxHII0Jem57XN13czZFPGcgjQx+tzYnoOZkiDmJ4AfaQ+Z0zPOxnSIKbnh4ij1OfNv3/iv9hgcAI0Vn1eX0WDmJgt2AHb+/9r0vPmdsxejFmZgEasz9b2uuYgpiRAe412z4+2HjidAI1y7rOP8yAmJkAHtPUZZw3wGQRoxKOf1xwGMSUBemn8O3z8FcKRBGivccafMdcDpxMgICNAo5/+7HISxGQECMgI0MWMPxuGIGYiQEBGgICMAAEZAbqkA6ANx0BMQ4CAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkB+m/76tHNy0hHtl2h16Vy6QQIyAgQkBEgICNAF3YM5ACImQgQkBGgSxqCjD9M5rpeABzr7v6mXgJnZgLaa7QhaLT1wOkE6KXxdzfjrxCOdPX09HjsZ5dktGfUO/1hSiagC9j4jLAG+AwC9LbdQaO9/3evbvxhMgK012h3+2jrgdMJ0NA/C3L0w9wcQh+2+wLSLzuTtvNiCUxAI54HqQ8LIUDDNUh9WA5bsA/uxT5jO/YiberD9ATopAadK0Ovpyr1YQkEKM6Q9LBkAnS2Bh0fo3dOkdSHRRGgTyzR8XSHZRKgOEPSw5IJUBAj0YENAQIyfogIZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAhYVf4BocL/i9IjAngAAAAASUVORK5CYII=', 'sha256': '1ee3c75576ac0d3528aa1033c4fb4a90a893f95382cd5b2fcb3c8d28662c2115'}, 'data/binding-v1/images/binding-evaluation-0074-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAEyUlEQVR4nO3dQW4aMQBAUVKlx6mURW7Wc/RmXUTqVbJA6qYLKoQgNDRAvsfz3hoYLzIf20Hjh+32dQNQ+JJcFUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMY3dpuMjT0/NmCi8vP+shDMcMCMgIEJCxBGMxfn//tlmgrz9+1UMYlxkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ8UTE4CHqHk4OOwIUnNxw+GIxYs0EKD40Zvd2GWKdBOj23Xn32emnTynff5oSsSoCdJv6/NeBDYcvPorR09OzBrEeAvSp6Tm1e/thhizKWA8B+nh9bnhM1ZsZ0iCmJ0Afqc+dTsg7ypAGMT0/RBylPm9+/pX/YoPBCdBY9Tm9igYxMUuwd+zv/08+mPxwOWYtxqzMgEasz97+uuZBTEmAzhrtnh9tPHA9ARpl3+cc+0FMTIDe0dZnnDHAPQjQiFs/p2wGMSUBOjb+HT7+COFCAnTWONOfMccD1xMgICNAo+/+HLITxGQECMgI0GKmPzsmQcxEgICMAAEZAQIyArSkDaAd20BMQ4CAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkB+mt/9OjuMNKR7UfouFSWToCAjAABGQECMgK0sG0gG0DMRICAjAAtaRJk+sNkHusBwKXG/FbgGmZAi/lzH208cD0BOjb+6mb8EcKFHrbb10tfuyajPaPe7g9TMgNawMJnhDHAPQjQ2w4nGu39f3h10x8mI0BnjXa3jzYeuJ4ADf2zIFs/zM0m9PsODyD9tD1pKy/WwAxoxP0g9WElBGi4BqkP62EJ9sG12D2WY0dpUx+mJ0BXNehWGTqdVakPayBAcYakhzUToJs16PIY/WMXSX1YFQG6Y4kupzuskwDFGZIe1kyAghiJDuwIEJDxQ0QgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMCm8gcGswBTH4bYTAAAAABJRU5ErkJggg==', 'sha256': 'bda041b280f52ea52258f7ed89551044e0b9f92c250067ba7f53005e831fad8c'}, 'data/binding-v1/images/binding-evaluation-0075-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQklEQVR4nO3aMU5cSRRAUTxy74fA6ezAK5kFzUrYwaQOWIwDJJIJkFotwA0zhr5dr85JjfXri66rV0V/eXj4eQNQ+CN5KoAAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZmv3aOXcXv77T0/dn//4/PXAqN8eXj4Wa9h4ej8ihjBewjQR3bnJSWCMwToXem5OxzO//fvj49n/lWG4FUC9Hp63izO/+iRDMEzWwfoM9JzSobgvH0D9Kw+H9idN0ukQbBvgC6ZnlMytPqfFC7vfvqHZLsvIlb1efmsAdsDftNeATrd83eHwyXr8+pDNYjNfd22PuFK7g6H43Hs9vbb+DE798/f574kcYX+/Kv8fF7SLhPQ9dTn5RrMQWxriwBdW32eaBDMD9B11ueJBrG54QG65vo80SB2NjlA11+fJxrEtiYHaIn6rLJC+AxjA3QcJVbZ28d1GoLYx8wArb6HV18/bB2go1XGnxVXC79vYICWO3ydchBjKwMDBKxiWoCWHn+eGILYx7QAAQsZFaB5I8O8N4KxATpa9/w1Y/2wdYCAJcwJ0IDr51OuotnBnAAByxEgICNAQGZIgGZflMx+O3Y2JEBHM26g570LbBEgYCECBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkpgXo++PjzRST3gUmB+j+/sfNXLPfjp0NCRCwIgECMgIEZP6Yd1Ey4+72+BYugBhsToCA5cwM0OpD0Orrhx0DNO+0Mu+NYGyAgLVMC9CAq2jXz+xjWoCAhQwM0NJDkPGHrQwM0Km1GrTWauH3zQzQ6uPD6uuHrQO04kHM4YsNjQ3Qqetv0PWvED7D5ACdjhLXvMNP12b8YSuTA7REg9SHnQ0P0JU3SH3Y3PwAXW2D1Ae2CNAVNkh9YKMAXVWD1Ae2C9DLBl0+Q88eqj5sbq8Avdzzl2zQs2epD3y92c/Tzr+9/XbahbvD4fOeKD3wqi8PDz9vdnVs0KkPLNGr45X6tL/ftdxP/7TsOAH9ahT6wIFIeuA9tg7Qmxk6erNH5++SpAdetfUR7AJDu/TAGQL0KSXSHXgPAfqwGIkO/FcCBGS2+yIicD0ECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgTcVP4FhoxLWceme5cAAAAASUVORK5CYII=', 'sha256': 'f7459f68424ec5954cd863e8f8595eb018a0d29b68ff7e49cf102cfd3d285a2a'}, 'data/binding-v1/images/binding-evaluation-0075-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPUlEQVR4nO3awU1cSRRA0bZlwrHEwmFMNo7D2UwYXrTkVLxAYuMFUquFMTBj6Nv16pwtoF9f9L+8Kv6Hu7ufB4DCx+SqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZD51l17G7e2X13zb8fj9/dcCo3y4u/tZr2Hh6PyJGMFrCNBbdud3SgTPEKBXpef+6+fnf/zm249nvipD8CQBejo9Lxbnf/RIhuCRrQP0Huk5J0PwvH0D9Kg+b9idF0ukQbBvgC6ZnnMytPq/FC7vOP1Dst2LiFV9fr/WgMcD/tJeLyKeP/OXTM+ji55GodvbL+P/xMEzNgpQXp/zq2vQJf17c3NYyj/394c97LIFu576/L4GezG2tUWArq0+DzQI5gfoOuvzQIPY3PAAXXN9HmgQO5scoOuvzwMNYluTA7REfVZZIbyHsQE6jRKrPNundRqC2MfMAK3+DK++ftg6QCerjD8rrhb+3sAALbf5OmcjxlYGBghYxbQALT3+PDAEsY9pAQIWMipA80aGeXcEYwN0su7+a8b6YesAAUuYE6ABx8/nHEWzgzkBApYjQEBGgIDMkADNPiiZfXfsbEiATmacQM+7F9giQMBCBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyEwL0M23H4cpJt0LTA7Q8fj9MNfsu2NnQwIErEiAgIwAAZmP8w5KZpzdnu7CARCDzQkQsJyZAVp9CFp9/bBjgObtVubdEYwNELCWaQEacBTt+Jl9TAsQsJCBAVp6CDL+sJWBATq3VoPWWi38vZkBWn18WH39sHWAVtyI2XyxobEBOnf9Dbr+FcJ7mByg81Himp/w87UZf9jK5AAt0SD1YWfDA3TlDVIfNjc/QFfbIPWBLQJ0hQ1SH9goQFfVIPWBB58OOzkev9/efjmvwP3Xz5dcwKPwqQ+b22gCevKZv+QopD6w9QR0/uRfchSSHnjSh7u7n4ddnRp07g1L9OR4pT7t73ctx+mflh0noD+NQm84EEkPvMbWAXoxQycv9uj5syTpgSdtvQW7wNAuPfAMAXqXEukOvIYAvVmMRAf+KwECMtu9iAhcDwECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAYfKLzN+R+ne3p0qAAAAAElFTkSuQmCC', 'sha256': '593d4053ac92e4c72fd7ed3bf0b7dd4683f03c59d3225416061ce9fbb52d6d4e'}, 'data/binding-v1/images/binding-evaluation-0075-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQElEQVR4nO3aMW5bRxRAUTqwl2NAhdtsKK3X4TYbSuuCgLfiQoCbFAIIQpYpJZZ4OW/OaSXhz4f4r96M/rv7++8HgMIfyVUBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIvO8uvYy7u08v+bbj8evbrwVGeXd//71ew8LR+RUxgpcQoNfszs+UCC4QoBel58fnj5d//MOXbxe+KkPwJAF6Oj3PFud/9EiG4JGtA/QW6TknQ3DZvgF6VJ9X7M6zJdIg2DdA10zPORla/V8K13ec/iHZ7kXEqj4/X2vA4wG/aa8XEc+f+Wum59FFT6PQ3d2n8X/i4IKNApTX5/zqGnRN//z947CUP//6cNjDLluw26nPz2uwF2NbWwTo1urzQINgfoBusz4PNIjNDQ/QLdfngQaxs8kBuv36PNAgtjU5QEvUZ5UVwlsYG6DTKLHKs31apyGIfcwM0OrP8Orrh60DdLLK+LPiauH3DQzQcpuvczZibGVggIBVTAvQ0uPPA0MQ+5gWIGAhowI0b2SYd0cwNkAn6+6/Zqwftg4QsIQ5ARpw/HzOUTQ7mBMgYDkCBGQECMgMCdDsg5LZd8fOhgToZMYJ9Lx7gS0CBCxEgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzLQAffjy7TDFpHuByQE6Hr8e5pp9d+xsSICAFQkQkBEgIPPHvIOSGWe3p7twAMRgcwIELGdmgFYfglZfP+wYoHm7lXl3BGMDBKxlWoAGHEU7fmYf0wIELGRggJYegow/bGVggM6t1aC1Vgu/b2aAVh8fVl8/bB2gFTdiNl9saGyAzt1+g25/hfAWJgfofJS45Sf8fG3GH7YyOUBLNEh92NnwAN14g9SHzc0P0M02SH1giwDdYIPUBzYK0E01SH3gwfvDTo7Hr3d3n84r8OPzx2su4FH41IfNbTQBPfnMX3MUUh/YegI6f/KvOQpJDzzp3f3998OuTg0694olenK8Up/297uW4/RPy44T0K9GoVcciKQHXmLrAD2boZNne3T5LEl64Elbb8GuMLRLD1wgQG9SIt2BlxCgV4uR6MB/JUBAZrsXEYHbIUBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEg4FD5F0xDR+k+G+6eAAAAAElFTkSuQmCC', 'sha256': '4baf3b9e5197618caf7a262b4f5fca945b3e28c8efb4e0717b0cba6151367f30'}, 'data/binding-v1/images/binding-evaluation-0076-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFXElEQVR4nO3aQU4bSxRAUecr7IdBpGwi4ywy47+JSBmwmAyQmGSAZFkYnOAQble/c0YoQNez5L4pV/Ph/v7nAaDwX7IqgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMx27p5d3efrr8A3d3P95rFljSh/v7n/UM+ynOZXoETwjQe6TnlAzBkQBd352bz98u//rD968XvqtEq/9fsh13y76XnAG97m362+i89MPnMXpcYt23Dvw9Afqj+ryqO886XuFJiW5vP2kQYwnQP0/Psxc8zZCtUOL/m5vD4r48PBwW5++AXqzPzedvb16fCxff5dkEXCZAz9z//zQ9p54spEFMI0DP1OedV9cgxhKguD7n62oQc0wP0PFuf7ePXS85HUCDGGJ0gE7rc9gGDWKUuQHaYH0eaRBzDA3QKvf2KnPCdSYGaAunzpc5k2aIiQHaeH22Pxu8lXEB2uzRzzmHQezerACteyevOzlcMCtAR9vf/qw1J1xnUIBW30SsPj+MDtCi24q1poVXmRggYCOmBGihh1/nPA5jr6YECNigEQHa08ZhT68FRgToaMXPX6tPDhfMChCwKQIEZAQIyOw/QEs/gD/lYTz7s/8AAZslQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAZv8Burv78fjFw/evh5Ud5z++Iljd/gMEbJYAARkBAjKzArTuMdC6k8P0AO3p1HZPrwVGBAjYpikBWvphvAfw7NWUAAEbNDFAa22C1poWXmVQgFb//LL6/DA6QCtuK1aZE64zK0DrbiLWnRwumBWgtR6HefjF7o0L0KktN2jLs8FbmRig0w3FNu/z06lsf9ixiQFa6K5eZU64ztAAbfkwyNEPc8wN0DYbpD6MMjpATxrUZuh0APVhiOkB2siZtFNnZhKgvkHqw1gC9HyD3idDTxZSH6YRoMNL9/8/zdD5xdWHgT7WA2zLYwVubz8d/+UxEzefv73VEudRkx7GEqBn3N39OG3QaTWuLtFLmyn1YTIB+tOt0HlHfhujy5/gpAcE6PAnjTgv0dXPy3QHjgTorzZEV1wEOBKg6wvy2x4pDlwmQNfTl6V9eXioR8DfAQEdAQIyH+7vf3arA6PZAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBBwqvwAewDihXMAsRAAAAABJRU5ErkJggg==', 'sha256': '379f8a4ca9f3ef23fe0c8cfa11faf64a427efd4a818a82a121fae042d2ad8f33'}, 'data/binding-v1/images/binding-evaluation-0076-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUklEQVR4nO3aMW4TWxSA4fBElvOkFLRsiPatg5YN0VJYYisUkWhegWRZ2HGIIfxz53xfFSmJ51jy/Llzb948Pn67Ayj8k1wVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmbXfp5T08vLv+A4fDl781CyzpzePjt3qG/RTnOj2CnwjQ30jPKRmCIwG6vTufP32//uvvP9xf+a4Srf63ZDsOy36W7AG97GP6bHSe+uHzGP24xLofHfh9AvRL9XlRdy46vsJPJXp4eKdBjCVAr56eiy94miFLocT3//69W9z9x693ixOgJ+vzx9PzbIY0iGkE6EJ9XjU9VzKkQUzjP6Gz+ly84i7PaOApAhTX5/y6GsQc0x/Bjnd7lZ6Lj2OexRhi9ApoO/U5Ok5iHcQEcwO0wfr8oEHMMTRAq9zbq8wJt5kYoC3sOl9nT5ohJgZo4/XZ/mzwp4wL0Ga3fs7ZDGL3ZgVo3Tt53cnhilkBOtr+8metOeE2gwK0+iJi9flhdIAWXVasNS28yMQAARsxJUALHX6dcxzGXk0JELBBIwK0p4XDnt4LjAjQ0YrPX6tPDlfMChCwKQIEZAQIyOw/QEsfwJ9yGM/+7D9AwGYJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ2X+ADocvP754/+H+bmXH+Y/vCFa3/wABmyVAQEaAgMysAK27DbTu5DA9QHvatd3Te4ERAQK2aUqAlj6MdwDPXk0JELBBEwO01iJorWnhRQYFaPXnl9Xnh9EBWnFZscqccJtZAVp3EbHu5HDFrACtdRzm8IvdGxegU1tu0JZngz9lYoBOFxTbvM9Pp7L8YccmBmihu3qVOeE2QwO05c0gWz/MMTdA22yQ+jDK27vZDocvDw/vjnf+50/fq0ns+zDQ6BXQdvak1YeZBKhvkPow1vRHsPNnsb/5OPZT7NSHaQTocoNeO0Pn6yz1YSABulCBV82Q9MCRAD2/FDqtxs0lemprSX2YTIB+dSl03pFnY3R9P1t6QIDufqUR5yW6+bxMd+BIgH5rQXTDiwBHAnR7QZ7tkeLAdQJ0O31Z2v3Hr/UI+E9ooCNAQObN4+O37urAaFZAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABd5X/AdV8NmIEy08sAAAAAElFTkSuQmCC', 'sha256': 'efb1961c184b1df8f765aa982a8c6ba7c225be53d11cd189e7120a8fc73e5541'}, 'data/binding-v1/images/binding-evaluation-0076-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVElEQVR4nO3bQW4TWRRA0dCC/WTAlB2wEhbESnoHPWXgxTCIlEkPkCwLOw4xSW79eueMIpG4niXX5df/yYeHh593AIV/kqsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh87C69vPv7z9e/4XD48V6zwJI+PDz8rGfYT3Gu0yP4jQC9R3pOyRAcCdDt3fnv++P1H//y7dOVf1Wi1f8v2Y7Dsp8le0Av+5g+G52nvvk8Rr8use5HB/6eAP1RfV7UnYuOr/Bbie7vP2sQYwnQm6fn4gueZshSKPHvp2sPyEv4+vjKH873J0BP1ufV0/NshjSIaQToQn3eND1XMqRBTOM3obP6XLziLs9o4CkCFNfn/LoaxBzTH8GOd3uVnouPY57FGGL0Cmg79Tk6TmIdxARzA7TB+vyiQcwxNECr3NurzAm3mRigLew6X2dPmiEmBmjj9dn+bPBaxgVos1s/52wGsXuzArTunbzu5HDFrAAdbX/5s9accJtBAVp9EbH6/DA6QIsuK9aaFl5kYoCAjZgSoIUOv845DmOvpgQI2KARAdrTwmFP7wVGBOhoxeev1SeHK2YFCNgUAQIyAgRk9h+gpQ/gTzmMZ3/2HyBgswQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMjsP0CHw49fX3z59uluZcf5j+8IVrf/AAGbJUBARoCAzKwArbsNtO7kMD1Ae9q13dN7gREBArZpSoCWPox3AM9eTQkQsEETA7TWImitaeFFBgVo9eeX1eeH0QFacVmxypxwm1kBWncRse7kcMWsAK11HObwi90bF6BTW27QlmeD1zIxQKcLim3e56dTWf6wYxMDtNBdvcqccJuhAdryZpCtH+aYG6BtNkh9GOXj3WyHw4/7+8/HO/+/74/VJPZ9GGj0Cmg7e9Lqw0wC1DdIfRhr+iPY+bPYez6O/RY79WEaAbrcoLfO0Pk6S30YSIAuVOBNMyQ9cCRAzy+FTqtxc4me2lpSHyYToD9dCp135NkYXd/Plh4QoLs/acR5iW4+L9MdOBKgv1oQ3fAiwJEA3V6QZ3ukOHCdAN1OX5b29TH7sxuO/CY0kBEgIPPh4eFnd3VgNCsgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAu8r/Drw3P2rHeeQAAAAASUVORK5CYII=', 'sha256': '79683f5a805001fc272d9c7d9db29262cde4140c0551efb6cc4b986667996dec'}, 'data/binding-v1/images/binding-evaluation-0077-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPUlEQVR4nO3dS2obWRiAUSXEy2nwIJCdZCO9Dm+kdxLIwOCtZGDIpAcGYSxb8TNf3frPGWkgl67A9XEfhf3p9vbXAaDwOflUAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzpfto+BCXl18Py7q+/nmYxAwIyAgQkLEEY7cuvv13WMTvH98PI5kBARkBAjICBGQECMjYhH7/x0ymPcoBryZA7/9g24P36xE8RYA+/Jnau4vIEJwSoBd35/e//5z/8YurmzPXVCI4EqBnpeeP0XnqzacxMiGCIwE6V58XdedRxys8KNHl5VcNAgH6qPQ8esH7GTIVAgF6WJ93T88fM6RBjDU9QPfr86HpOZMhDWKs0U9CJ/V59BOX/hta8GpzA9TW5/RzNYiBhi7Bjnd7lZ5Hl2PWYkwzcQa0nfocHUdiHsQo4wK0wfrc0SAGmhWgVe7tVcYJbzQoQFvYdT7PnjTTDArQxuuz/bHBu5sSoM1u/ZyyGcQcIwK07p287sjhOUYE6Gj705+1xglvtP8ArT6JWH38MDpAi04r1hotvM6gAAFbs/MALXT4dcpxGLu38wABW7bnAO1p4rCn7wIjAnS04vpr9ZHDc4wIELBNAgRkBAjI7DZASx/A3+cwnh3bbYCA7RMgICNAQEaAgIwAARkBAjICBGQECMgIEJARICCz2wBdX/+8e3FxdXNY2XH8x28Eu7HbAAHbJ0BARoCAzIgArbsNtO7IYXqA9rRru6fvAiMCBGzczgO09GG8A3h2b+cBArZsUIDWmgStNVp4nf0HaPX1y+rjhzO+HCa5uLpZ4p9kmP68i98/vtdD4DB9BrT0JGLdkcNzjAjQWsdhDr+YY9YSbPsLse33cfuEeyGfZ/5ebvM+vz8qdxETDArQQnf1KuOEN5oVoC1vBtn6YaBxAdpmg9SHmT7d3v46jHR5+fX4OtyTtu/DZBNnQNvZk1YfhpsboLxB6gNzl2CPrsX+znLsQezUh7EE6JEGfVyGTudZ6sNkAvSXMiQ9cEqA/tCgN5boqa0l9QEBelmGnhmj8/vZ0gNHAvT6Er2I7sApAfrwDEkPPEWA3r9HigPPJEBAZvST0EBLgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwKHyP4bYLn/IWXReAAAAAElFTkSuQmCC', 'sha256': '6e19ab2fff13c5e95ff423f84c4da127a026b6742c4116a6bcb39b59c57eaa25'}, 'data/binding-v1/images/binding-evaluation-0077-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNElEQVR4nO3dPU4jSxhAUc9o2A8BKRtiQW9DL52AxUyA9JIXILUQBob/29XfOZEDZJcRfVVVXTY/7u7+nAAKP5NXBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyv7qXhi9xeXl1Wtbt7e/TJGZAQEaAgIwlGIf17z//nRZxfXNxGskMCMgIEJARICAjQEDGJvTnHzOZdpQD3k2APv9g26Of1yN4jgB9+Zna+yeRITgnQG/uzl9Plzx5pmN7TiWCjQC9Kj1vOtL28IfPY2RCBBsBeqk+Hz9Kuz3DoxJdXl5pEAjQV6XnySd8mCFTIRCgx/X50g8QPZkhDWKs6QF6WJ9v++ziowxpEGONPgmd1OfJV1z6O7Tg3eYGqK3P+etqEAMNXYJtV3v+lTEPl2PWYkwzcQa0n/pstpGYBzHKuADtsD73NIiBZgVolWt7lXHCBw0K0B52nV9mT5ppBgVo5/XZ/9jg000J0G63fs7ZDGKOEQFa90ped+TwGiMCtNn/9GetccIHHT9Aq08iVh8/jA7QotOKtUYL7zMoQMDeHDxAC938Oud2GId38AABe3bkAB1p4nCk9wIjArRZcf21+sjhNUYECNgnAQIyAgRkDhugpW/AP+RmPAd22AAB+ydAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmsAG6vf19/+D65uK0sm382zuCwzhsgID9EyAgI0BAZkSA1t0GWnfkMD1AR9q1PdJ7gREBAnbu4AFa+ma8G/Ac3sEDBOzZoACtNQlaa7TwPscP0Orrl9XHDy/4dZrk+uZiiX+SYfrzKfwa9+/4M6ClJxHrjhxeY0SA1rod5uYXc8xagu1/Ibb/Pu6fcC/k58y/y31e5w9H5SpigkEBWuiqXmWc8EGzArTnzSBbPww0LkD7bJD6MNOPu7s/p5EuL6+2x+GetH0fJps4A9rPnrT6MNzcAOUNUh+YuwR7ci32PcuxR7FTH8YSoCca9HUZOp9nqQ+TCdA3ZUh64JwA/aVBHyzRc1tL6gMC9LYMvTJGL+9nSw9sBOj9JXoT3YFzAvTlGZIeeI4AfX6PFAdeSYCAzOiT0EBLgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwKnyPwiWMtk9+A1PAAAAAElFTkSuQmCC', 'sha256': 'e5c8a24b6550f1fb395d750c66a075d7fe9f9e76c0bfb709fb9c938c12fca21b'}, 'data/binding-v1/images/binding-evaluation-0077-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPklEQVR4nO3dO04cWRiA0bZl9kOA5IhteBEsaBYx25jIkgMW4wDJyQSWWoiGNk9/des/J+qgqb4tUZ/uowSf7u5+HgAKn5NPBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyX7qPhg9xeXl1WNbt7Y/DJGZAQEaAgIwlGLt18fXfwyJ+ff92GMkMCMgIEJARICAjQEDGJvT7P2Yy7VEOeDUBev8H2x68X4/gKQL04c/U/r6IDMEpAXpxd/7759f5H7++uThzTSWCIwF6Vnr+GJ2n3nwaIxMiOBKgc/V5UXcedbzCgxJdXl5pEAjQR6Xn0Qvez5CpEAjQw/q8e3r+mCENYqzpAbpfnw9Nz5kMaRBjjX4SOqnPo5+49N/QglebG6C2Pqefq0EMNHQJdrzbq/Q8uhyzFmOaiTOg7dTn6DgS8yBGGRegDdbnNw1ioFkBWuXeXmWc8EaDArSFXefz7EkzzaAAbbw+2x8bvLspAdrs1s8pm0HMMSJA697J644cnmNEgI62P/1Za5zwRvsP0OqTiNXHD6MDtOi0Yq3RwusMChCwNTsP0EKHX6cch7F7Ow8QsGV7DtCeJg57+i4wIkBHK66/Vh85PMeIAAHbJEBARoCAzG4DtPQB/H0O49mx3QYI2D4BAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyuw3Q7e2P3y+uby4OKzuO//iNYDd2GyBg+wQIyAgQkBkRoHW3gdYdOUwP0J52bff0XWBEgICN23mAlj6MdwDP7u08QMCWDQrQWpOgtUYLr7P/AK2+fll9/HDGl8Mk1zcXS/yTDNOfd/Hr+7d6CBymz4CWnkSsO3J4jhEBWus4zOEXc8xagm1/Ibb9Pm6fcC/k88zfy23e5/dH5S5igkEBWuiuXmWc8EazArTlzSBbPww0LkDbbJD6MNOnu7ufh5EuL6+Or8M9afs+TDZxBrSdPWn1Ybi5AcobpD4wdwn26Frs7yzHHsROfRhLgB5p0Mdl6HSepT5MJkB/KUPSA6cE6A8NemOJntpaUh8QoJdl6JkxOr+fLT1wJECvL9GL6A6cEqAPz5D0wFME6P17pDjwTAIEZEY/CQ20BAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBBwq/wMC0jRHJzsIZQAAAABJRU5ErkJggg==', 'sha256': 'c44188c0e8582093f6edc03eed3e0f38aa9c4be38b818e0abdf59203db0181a6'}, 'data/binding-v1/images/binding-evaluation-0078-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKElEQVR4nO3aQU5bSRRAUdKKl9MSA5bBbnod2Q3LyAApW2FgyZMeWLJo29hAG279X+eMiABXIfg3r0r+sd2+3AEU/kpWBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMzPbulFur9/uPwFz8+/v2svsHg/ttuXeg+Lj85bxAguE6Dbd+eUEsFZAvSx9DxtNpe/93G3e+tTMgRHBOh6eq5G50MxkiE4EKA30/Pp7rynRDIEAnS+PjdMz4UMaVB1JVfxGz/lfUD/+ct+2my+qD6nL76CJwr+p6nfB/Q9g8+Rp83mMArtN+A/RqY1b4CS+hzWen0cu79/0KD32/3z992ibH79qbcwrkmPYN927HqL4xhMGqCj+oQ70SAmN2OABqnPOHuAynQBOgwa4zz5h50YgpjNXAEa/wkff4dwQxMFaJyrn1Mug5jTRAEatj4j7wq+1CwBGvDq55TLIGYzS4CAAU0RoEWMP3uGIKYyRYCAMa0/QAsaf/YMQcxj/QEChiVAQGblAVrc+WvPKYxJrDxAwMgECMgIEJBZc4AWegG05xqIGaw5QMDgBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyKw5QM/Pv/cfPO52d0tz2PPhp4D1WXOAgMEJEJARICCz8gAt9BrIBRCTWHmAgJEJEJBZf4AWdwpz/mIe6w8QMKwpArSgIcj4w1SmCBAwplkCtIghyPjDbGYJ0GtjNmjMXcGXmihAr8eK0Z721/sx/jCPiQK0iGd7/B3CDc0VoDEvg1z9MK3pAvTaCA0aYQ9QmTFA41wGufphcjMG6LRB35+ho0XVhzlNGqDTZ/47G3S0lvowrZ93E9s/+ff3D/t/Pu52T5vNVy9q8IGDeSeg7z+OOXbBEQE6fxy7bYZOX1B9YPYj2IXj2Ouz0qfPZWcrJj1wIEBXMnTUkasxujA6SQ8cEaD3Zmjvc0cz6YGzBOh6Nc6W6EOvAJwlQB/ryNUYiQ68nwB9jL7ADQkQC7P59afeAjfjfUBARoCAzI/t9qVbHZiaCQjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEg4K7yL3NIMn+alQ5qAAAAAElFTkSuQmCC', 'sha256': 'bf95340c2c2e280a79b8dd0319c9e3109e87eb3206196d787f9ace5d3db63681'}, 'data/binding-v1/images/binding-evaluation-0078-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJUlEQVR4nO3bPU5jSRhAURg1+3FA2jvolfSCWMnsgJTAiyGwRDKBJctjzI+7wbfeq3MiJMCvLPyuviqb293u+Qag8E9yVQABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMj86C69SJvN/fs/sN0+XWstsHi3u91zvYbFR+ctYgTvE6Cv785rSgRnCdBl6Xl8eHn/d3/+vnvrWzIEJwTo4/R8GJ2LYiRDcCBAb6bnj7vzmRLJEAjQ+fp8YXreyZAGVUdyFX/x17wN/79X9jel5/jBDxnabO69Ipnc1B9E3Gzur1afs1c5WQDMZt4J6DrbrrMeH16Ot2NGoYv8e/fm+4xj+vVyvZfW4kwaoOsPPidsx2DSLVhen7NXtxdjQjMGaJD6jLMGqEwXoMOgMc6df1iJIYjZzBWg8e/w8VcIX2iiAI1z9POawyDmNFGAhq3PyKuCbzVLgAY8+nnNYRCzmSVAwICmCNAixp89QxBTmSJAwJjWH6AFjT97hiDmsf4AAcMSICCz8gAtbv+1ZxfGJFYeIGBkAgRkBAjIrDlACz0A2nMMxAzWHCBgcAIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGTWHKDt9mn/xc/fdzdLc1jz4VnA+qw5QMDgBAjICBCQWXmAFnoM5ACISaw8QMDIBAjIrD9Ai9uF2X8xj/UHCBjWFAFa0BBk/GEqUwQIGNMsAVrEEGT8YTazBOjYmA0ac1XwrSYK0PFYMdrdfrwe4w/zmChAi7i3x18hfKG5AjTmYZCjH6Y1XYCOjdCgEdYAlRkDNM5hkKMfJvfjZkrb7dNmc39cgceHl2su4CR86sOcZpyAzt7z1xyF1AdmD9D+zr/+duxk26U+zGzSLViyHTP4wAkBOm3Qd2To9WylPiBAp0V4naG/KdHZPZ30wIEAfZChk458GKN3DpKkB04I0Gcz9DcH1dIDZwnQx9U4W6KLHgE4S4Au68iHMRId+DwBuoy+wBcSIBbm18tV/2mGbzX1J6GBlgABmdvd7rm7OjA1ExCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwE3lPxzBKEJ9NIOhAAAAAElFTkSuQmCC', 'sha256': '8b0fae8604071a2500fe6a6429d8fb135f0f361273399a7bd27b6fa78c885f2b'}, 'data/binding-v1/images/binding-evaluation-0078-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJUlEQVR4nO3aPU5bWRiAYRiF5YzkgjYbSjvrSMuGaFMgZSsUlmimsGR5bPM3MX7Pved5qkgJ3BPBffWdD2632+cbgMJfyVMBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg86179CJtNvdv/4Onp1/XOgss3u12+1yfYfHReY0YwdsE6PLdOaVEcJYAfS49jw8vb3/s9x93r/2VDMERAXo/Pe9G51MxkiHYE6BX0/O/u/OREskQCND5+lwwPW9kSIOqlVzFV/yUH8P/5zv7i9Jz+Mn3Gdps7n1HMrmpfxFxs7m/Wn3OPuXoADCbeSeg61y7znp8eDm8jhmFPuXln79vFuXu5+/6COOaNEDXH3yOuI7BpFewvD5nn+4uxoRmDNAg9RnnDFCZLkD7QWOcN39/EkMQs5krQOO/4eOfEC5oogCNs/o5ZRnEnCYK0LD1GflU8KVmCdCAq59TlkHMZpYAAQOaIkCLGH92DEFMZYoAAWNaf4AWNP7sGIKYx/oDBAxLgIDMygO0uPvXjlsYk1h5gICRCRCQESAgs+YALXQBtGMNxAzWHCBgcAIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGTWHKCnp1+7P3z/cXezNPsz7/8XsD5rDhAwOAECMgIEZFYeoIWugSyAmMTKAwSMTICAzPoDtLhbmPsX81h/gIBhTRGgBQ1Bxh+mMkWAgDHNEqBFDEHGH2YzS4AOjdmgMU8FX2qiAB2OFaO97YfnMf4wj4kCtIh3e/wTwgXNFaAxl0FWP0xrugAdGqFBI5wBKjMGaJxlkNUPk/t2M6Wnp1+bzf1hBR4fXq55gKPwqQ9zmnECOvvOX3MUUh+YPUC7N//617Gja5f6MLNJr2DJdczgA0cE6LhBX5Gh09lKfUCAjotwmqE/KdHZO530wJ4AvZOho468G6M3FknSA0cE6KMZ+pNFtfTAWQL0fjXOluhTnwE4S4A+15F3YyQ68HEC9Dn6AhckQCzM3c/f9RG4mKl/ExpoCRCQud1un7unA1MzAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE3FT+BbKIJ2XFDlDbAAAAAElFTkSuQmCC', 'sha256': 'dc8f0ef367810b9886364a64b4fdda941eca4601a3131f3d4ce1fb6c8a7be7e6'}, 'data/binding-v1/images/binding-evaluation-0079-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVUlEQVR4nO3bPU4jSRiAYVgN93GANNFcYw4xB9pDzDU2GonAhyGwRLKBJa+FF2PMz9vV9TwR4q8/S/Sr6ipzu9s93gAU/kquCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfOsuParN5v78N2y3D181C4ztdrd7rGcYvjjn6RG8RIA+MT3HZAhOCdCbu3P3/ff5H3/68/PMV5UIDgToovS8Gp23xkiGQIBeqc/V3bmkRBoEAvTp6TkmQ3BMgJ7X55PScyZDGsS0Zg/QcX2+ID0vZUiDFn6I2dqu989j6ndCh/V5dsXV3CrwJvMGqK3P6XU1iAlN+q8Yh7u9Ss/BfoD949hmc7/ixXbon7+fbgb049fdzdrNuAJaTn0ODpNYBzGV6QK0wPrsaRATmitAo9zbo8wJ7zRRgJaw63yePWlmM1GAFl6f5c8GH26WAC126+eUzSDmMUWAxr2Tx50cLjFFgA6Wv/wZa054p/UHaPRFxOjzw9QBGnRZMda0cJ2JAgQszcoDNNDh1ynHYazeygMELNmaA7SmhcOaXgtMEaCDEZ+/Rp8cLjFFgIBlEiAgI0BAZrUBGvoA/pjDeFZstQEClk+AgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMagO03T7sP3j68/NmZIf5D68IVmO1AQKWT4CAjAABmSkCNO420LiTw+wBWtOu7ZpeC0wRIGDhVh6goQ/jHcCzeisPELBkEwVorEXQWNPCddYfoNGfX0afH6YO0IjLilHmhHeaIkDjLiLGnRwuMUWAxjoOc/jFPGYJ0LElN2jJs8GHmyhAxwuKZd7nx1NZ/jCDiQI00F09ypzwTnMFaMmbQbZ+mNB0AVpmg9SHOc0YoGcNajN0PID6MJtJA7SQPWm7zkxu3gDlDVIfmDpApw36mgw9u5D6MK3ZA3R6/39qhk5/ufows2/1AIuwr8Bmc3/4zD4Td99/f9QlTqMmPSBA/9luH44bdFyNq0v00mJKfUCAXl8KnXbk1Ridf4KTHjgQoHONOC3R1edlugOnBOiaBdEVvwQ4JUBvLsirPVIcuJAAvZm+wEcRINbvx6+7egT+nzciAhkBAjK3u91jd3VgalZAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECbir/AjruRkQHlhcWAAAAAElFTkSuQmCC', 'sha256': '6acdd1b91e19627f0079bc247eb6564cf8226db453d5a865b8e8b7022fab0ab0'}, 'data/binding-v1/images/binding-evaluation-0079-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFTklEQVR4nO3bTU4bWRSAUdMK+/EAqUfZRhaRBWURvY2MIjHwYhhYYtIDJLeFG2PMz1ev3jkjpITUtZT6dOsV3Oz3DxuAwl/JVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmW/dpUe13d6d/wu73f1XzQJju9nvH+oZhi/OeXoELxGgT0zPMRmCUwL05u78/vV4/tu//7w986dKBAcCdFF6Xo3OW2MkQyBAr9Tn6u5cUiINAgH69PQckyE4JkDP6/NJ6TmTIQ1iWrMH6Lg+X5CelzKkQQt/idnarfe/x9Q/CR3W59kVV3OrwJvMG6C2PqfX1SAmNOmvYhzu9io9B08DPD2Obbd3K162Q7d//7MZ0OOfH5u1m3EDWk59Dg6T2IOYynQBWmB9nmgQE5orQKPc26PMCe80UYCWcOp8njNpZjNRgBZen+XPBh9ulgAt9ujnlMMg5jFFgMa9k8edHC4xRYAOlr/+jDUnvNP6AzT6EjH6/DB1gAZdK8aaFq4zUYCApVl5gAZ6+XXK6zBWb+UBApZszQFa0+Kwps8CUwToYMTnr9Enh0tMESBgmQQIyAgQkFltgIZ+AX/My3hWbLUBApZPgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzGoDtNvdP33x/eftZmSH+Q+fCFZjtQEClk+AgIwAAZkpAjTuMdC4k8PsAVrTqe2aPgtMESBg4VYeoKFfxnsBz+qtPEDAkk0UoLGWoLGmheusP0CjP7+MPj9MHaAR14pR5oR3miJA4y4R404Ol5giQGO9DvPyi3nMEqBjS27QkmeDDzdRgI4XimXe58dTWX+YwUQBGuiuHmVOeKe5ArTkwyBHP0xougAts0Hqw5y+baa0291vt3eHO//3r8dqEuc+zGzGDWg5Z9Lqw+TmDVDeIPWBSR/BTp/FvvJx7Fns1IdpzR6gZw367Ayd7lnqw8wE6L8KfGqGpAdOCdCLq9BxNa4u0UtHS+oDAvT6KnTakVdjdP48W3rgQIDONeK0RFe/L9MdOCVA1yxEV/wjwCkBenNBXu2R4sCFBOjN9AU+igCxfo9/ftQj8P+m/lUMoCVAQOZmv3/org5MzQYEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESBgU/kXikI29s1Q9JAAAAAASUVORK5CYII=', 'sha256': 'ca4d41ae4615a7cebe11de9674187544e07323b394d44d9ad25e1a90c9cbc48f'}, 'data/binding-v1/images/binding-evaluation-0079-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQ0lEQVR4nO3bPU4cSxSAUfxk74fAqTfkBXlDTglmMQRIThwgzRsxYoDB8HX1PSdC4qfvCPpTdRXz5eHh/gag8F9yVQABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5mt36VXd3n6//AWHw91nzQJr+/LwcF/PsHxxLtMjeI4AfWB6TskQnBOgN3fn968/l7/9x89vFz6rRHAkQK9Kz4vReWuMZAgE6IX6XN2d15RIg0CAPjw9p2QITgnQ0/p8UHouZEiDGGt6gE7r8wnpeS5DGrTxQ8zWYb9/HqP/Ezqsz5Mr7uZWgTeZG6C2PufX1SAGGvpWjOPdXqXn6HGAx8ex29vvO15sh/Lf8nV+XPyHsn2YuALaTn2OjpNYBzHKuABtsD6PNIiBZgVolXt7lTnhnQYFaAu7zpfZk2aaQQHaeH22Pxv8c1MCtNmtn3M2g5hjRIDWvZPXnRxeY0SAjra//FlrTnin/Qdo9UXE6vPD6AAtuqxYa1q4zqAAAVuz8wAtdPh1znEYu7fzAAFbtucA7WnhsKfXAiMCdLTi89fqk8NrjAgQsE0CBGQECMjsNkBLH8CfchjPju02QMD2CRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkNltgA6Hu8cPfvz8drOy4/zHVwS7sdsAAdsnQEBGgIDMiACtuw207uQwPUB72rXd02uBEQECNm7nAVr6MN4BPLu38wABWzYoQGstgtaaFq6z/wCt/vyy+vwwOkArLitWmRPeaUSA1l1ErDs5vMaIAK11HObwizmmBOjUlhu05dngnxsUoNMFxTbv89OpLH+YYFCAFrqrV5kT3mlWgLa8GWTrh4HGBWibDVIfZvp6M9LhcHd7+/145//+9aeaxL4Pk01cAW1nT1p9GG5ugPIGqQ8MfQQ7fxb7zMexJ7FTH8aaHqAnDfroDJ2vs9SHyQTo/wp8aIakB84J0LNLodNqXF2i57aW1AcE6OWl0HlHXozR5f1s6YEjAbrUiPMSXX1epjtwToCuWRBd8UOAcwL05oK82CPFgVcSoDfTF/hXBIj92867jnli9FsxgJYAAZkvDw/33dWB0ayAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE3FT+AmiONYgNZEXhAAAAAElFTkSuQmCC', 'sha256': '37fb3bad3cc295c0b2aa8e07f5d9506cd8250f21ffcab877b77c7e026156d0e3'}, 'data/binding-v1/images/binding-evaluation-0080-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMUlEQVR4nO3dQXIaRxiAUTll3YdFttzAJ/GBcpLcgK0XHMYLVWmTStkmBBCSLODrod/buUpmeqH5/E97hvn09PT9AaDwR3JUAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD5/DC31erPhzuy3X6rlwDvYAICMgIEZGa/BNv5+/HxYbG+PD/XS4DfYQICMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjI+E5ouNhLnLwW6b0ECC725rj9HxajtxAguMobK3efoERnCBC8NT2bv155/dH66+NLHyhDJwkQnEvPq9F56YcPYiRDJwkQnK7Pu9Jz7MdfP86QBu0TIDiszwfTcz5DGrRPgJja9dJzJkMux3bciMi8blOflz5/9eH/aLsDJiAmtX/+Xzs9Z0ah7dxzkAmIGSX1OXnE1dxzkAAxnbY+x8ddTdwgAWJeVX1GOPogBIi57MaNEc7/za81TDsECRATGfk8Xw28tusRIGYxwtbPsc3cm0ECxHTGqc+Y67klAWIKSxkuVgtZ56UIEHMZc9zYDLmqGxAg7t+yxorVolb7QQLEREYeNDYDr+16BAjICBB3bqg7D8/bzHdfogABGQECMgIEZASIe7agDaA5t4EECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDu2e7N67vXsQ9u/Wudk7wzXoCAjAABGQECMgLEnVvQNtB6sg0gAQJKn9OjD+TL8zK+sIqPWH99HPabydbDD2jXYALi/i3rima7qNV+kAAxlzEHjfWQq7qB2S/BpvrXZmbb7bdFfMvydrJfSBMQ0xlt3FgPtp5bEiBmsT9cjHPOr/dWMtv4I0DMZeQzfDvw2q5HgJjLUPclrue78/CAADGvtkHrAQqYEyCmM8Jm0ORbPzsCxIzaBqnPzqenp+///QlmcnBn0A2e0jiI3Xbu+piAmNrB+X/tUUh9jpmA4OqjkPS8RIDgX8cPalwkQ8dTlfrsEyD46eTDYr+doZMXdOpzQIDgf848s/pqjM7sIknPSQIEJ1zw0XnpOUOA4Col0p23ECC4WIxE570ECMi4ERHICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEg4KHyD5j0DuJwjQd9AAAAAElFTkSuQmCC', 'sha256': 'c358263d1752c8b56107bf664ea39e9ea050a99dcf7491bb0dfa5731b5db6b84'}, 'data/binding-v1/images/binding-evaluation-0080-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJklEQVR4nO3dPW4iWRhAUXrU3g/BpL0DVtILmpWwg0k7YDEdWCKZwGqEMMZm+LkP3jmRA7vqJXX11XOp6tvr6+8FQOGv5KwAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPN9Mbfl8u/FE9lsftVLgDOYgICMAAGZ2W/Bdv79Z7t4WD9+vtRLgP/DBARkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJDxTmi42kecfBbpXAIEV/ty3P4vi9FXCBDc5IuVuyMo0QkCBF9Nz/rlk88frbbbjw4oQ0cJEJxKz6fR+eiXD2IkQ0f5Lxgcr8/65eWs+nzlzy+/s3syAgSHXbgwPacPpUH73IIxtffpucVZ3g67uylzO7ZjAmJe96nPR8dfGoUEiGntX/9XvOc67eBEy+kbJEDM6KA+dz67Bu0IENNp6/P+vMuJGyRAzKuqzwhnH4QAMZfduDHC9b/+s4ZphyABYiIjX+fLgdd2OwLELEbY+nlvPfdmkAAxnXHqM+Z67kmAmMKjDBfLB1nntQgQcxlz3FgPuao7ECCe32ONFcuHWu2FBIiJjDxorAde2+0IEJARIJ7cUE8enrae77lEAQIyAgRkBAjICBDP7IE2gObcBhIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgHhmm82vtx9W2+3iEaz+rHO38ucmQEBGgICMAAEZAeLJPdA20GqyDSABAkrf07MP5MfPx3hhFZdYbbfDvplsNfyAdgsmIJ7fY93RbB5qtRcSIOYy5qCxGnJVd/Dt9fV3vQaY/f3Qq/m2n9+YgJjOaOPGarD13JMAMYv94WKca361t5LZxh8BYi4jX+Gbgdd2OwLEXIZ6LnE169bPjgAxr7ZBqwEKmBMgpjPCZtDkWz87AsSM2gapz47ngJjXwffX7/B80EHsNnPXxwTE1A6u/1uPQurzngkIbj4KSc9HBAiONOhaGXo/VanPPgGCDxt0SYaO3tCpzwEBgs8z9MUYndhFkp6jBAjOy9C5pOcEAYKblEh3vkKA4GoxEp1zCRCQ8SAikBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMCi8h8PR/smeCEW1QAAAABJRU5ErkJggg==', 'sha256': 'ae57f0d8a84c0f4af8c26252ffaa9c39617c78d0f077d02c9e7cacc0ee82f7a2'}, 'data/binding-v1/images/binding-evaluation-0080-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFG0lEQVR4nO3dPW4iWRhAUXrU3g/BLKOWyzI6YDEdWCKZwGqEMKbN8HMfvHMiB3bVC1zXXz2XqB/v779XAIV/krMCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMzP1dzW639XL2S7/VUvAS5gAgIyAgRkZr8F29u8va2e1rLb1UuA/8MEBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPGZ0HCzlzh5LdKlBAhu9ua4w28Wo+8QILjLGyv3R1CiMwQIvpuev7676eT7kT4OKEMnCRCcS89FL4w7/OajGMnQSf4LBqfrs3l7u+Z1lSd//Po7uxcjQHDchSvTc/5QGnTILRhT+5yee5zl47D7mzK3Y3smIOb1mPp8dfy1UUiAmNbh9X/De67zjk60nr5BAsSMjurz4LNr0J4AMZ22Pp/Pu564QQLEvKr6jHD2QQgQc9mPGyNc/5s/a5h2CBIgJjLydb4eeG33I0DMYoStn882c28GCRDTGac+Y67nkQSIKTzLcLF+knXeigAxlzHHjc2Qq3oAAeL1PddYsX6q1V5JgJjIyIPGZuC13Y8AARkB4sUN9eTheZv5nksUICAjQEBGgICMAPHKnmgDaM5tIAECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQEiFe23f76+GLZ7VbPYPmzzv3KX5sAARkBAjICBGQEiBf3RNtAy2QbQAIElH6mZx/I+H8eud6y2w37yWTLlL+BJiBe33Pd0WyfarVXEiDmMuagsQy5qgf48f7+u14DzP750Mt8288fTEBMZ7RxYxlsPY8kQMzicLgY55pfDlYy2/gjQMxl5Ct8O/Da7keAmMtQzyUus2797AkQ82obtAxQwJwAMZ0RNoMm3/rZEyBm1DZIffY8B8S8jt6//oDng45it527PiYgpnZ0/d97FFKfz0xAcPdRSHq+IkBwokG3ytDnqUp9DgkQfNmgazJ08oZOfY4IEPw9Q9+M0ZldJOk5SYDgsgxdSnrOECC4S4l05zsECG4WI9G5lAABGQ8iAhkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgSsKv8BRRL8vB7VOA0AAAAASUVORK5CYII=', 'sha256': '76528395909d6740648374e1a733f6d87c56eaff76f3b4f148ad18dd7840e053'}, 'data/binding-v1/images/binding-evaluation-0081-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFSElEQVR4nO3cQW4aSRiAUTIa7sMikm+SQ/omkbLgMFlY6s0srCAExGQc7K+76r21TReS+9NfReMvLy8/dwCFf5KrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+Xc3rsPh624Cx+OPegnwTiYgICNAQGbkLdjJ/ul5N5zl+7d6CfC3TEBARoCAjAABGQECMgIEZKb4FAw++nFWj4O+jwDBA56hP/8VMfpzAgQP/u7O6XWU6C4Bgjvpufsg6+8eCn19WRl6gwAxtZvp+b+Pzl/8/EWPZOgNAsSkrtPzqK/snF7nvEQydJMAMZ2PS8/Nl5WhN3gOiKnrs396/ujvKl9fYpL/VPUnBIiJnN/5n5CecxeX06BXtmBMOvgky9g/PZ92ZAfbMRMQM1hJfW5e/TD3KCRADC7cdv2O7diJADFRfXZrstcgAWJga67Pq/30DRIgxrfO+qx/bZ9AgBjTaaBY/x2+/7XCCYcgAWJA272TD5td+fsIEKNZ/9HPtf2sh0ECxLC2Up8trvZRBIihbOjo59qEh0ECBGQEiHFsevyZcwgSICAjQAxigPFnwiFIgICMADGCYcaf2YYgAQIyAgRkBIjNG2z/NdUuTICAjAABGQECMgLEto19RDL8exQgBjHSCfSo7+iaAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgLEIJbv33ZjWYZ7R9cEiG07Hn/sRncc9z0KEJARICAjQEBGgBjniGSkU9vl13sZ+ABIgICSAAEZAWIEg+3Cljn2XwIElASIQQwzBC3TjD8CBJQEiHEMMAQtM40/AgSUBIihbHoIWiYbfwSIkW2rQcumVvsoAsRozseHrdzVy9k65xl/BIgxbfcePm525e8jQIxpQ4dBy3xHPycCxPjW3KBlxWv7BALEsNZ/GLTMevRz8uXl5eduUIfD190E5vzDfd+fwf7pebcay/T1MQEx3Ry0hlHoYhnHWesjQEzh4g5vG3Rx9ePE9Rl8CwZr244ZfC4IEHO5OBn8tAwZfG4SIKZz/enEh2boesenPicCxKQ+IUPSc5cAMbWbz2r8ZYluHnJLz00CBHceGbvbo7c/VpOeNwgQfNSTq9JzlwDBg0ukO39OgOABMRKd9xEgIOOrGEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjYVf4D9+Q34JqDv/MAAAAASUVORK5CYII=', 'sha256': '97e2ade6b1135e5e3cfc48974848b668cb30dd9a8a9fac396e26285c71b05784'}, 'data/binding-v1/images/binding-evaluation-0081-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFTklEQVR4nO3cQW4TSxRA0eYL78cDltHL9TIYeDEMImXyBxFWZJskBMfXXXXOiAG4qyX68qrS5tvT068FoPBfclUAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjLfl3Ht9z+WCRyPP+slwCeZgICMAAGZkbdgJ4fdbhnO+vxcLwH+lQkIyAgQkBEgICNAQEaAgMwUPwWDr36d1eugnyNAcIN36F//ETH6OAGCG3935/Q5SvQuAYJ30vPui6x/ein05WNl6A0CxNSupudvX50/+/1nPZKhNwgQk7pMz62+snP6nNclkqGrBIjpfF16rn6sDL3Be0BMXZ/DbvfV31W+vMQk/1PVRwgQE3n95N8hPa+dXU6DXtiCMengkyzjsNuddmR72zETEDN4kPpcvfp+7lFIgBhcuO36E9uxEwFiovosj+SgQQLEwB65Pi8O0zdIgBjfY9bn8dd2BwLEmE4DxeM/4YffK5xwCBIgBrTdJ3m/2ZV/jgAxmsc/+rl0mPUwSIAY1lbqs8XV3ooAMZQNHf1cmvAwSICAjAAxjk2PP3MOQQIEZASIQQww/kw4BAkQkBEgRjDM+DPbECRAQEaAgIwAsXmD7b+m2oUJEJARICAjQEBGgNi2sY9Ihr9HAWIQI51Aj3pHlwQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIAaxPj8vY1mHu6NLAsS2HY8/l9Edx71HAQIyAgRkBAjICBDjHJGMdGq7/r6XgQ+ABAgoCRCQESBGMNgubJ1j/yVAQEmAGMQwQ9A6zfgjQEBJgBjHAEPQOtP4I0BASYAYyqaHoHWy8UeAGNm2GrRuarW3IkCM5vX4sJWnen21znnGHwFiTNt9ho+bXfnnCBBj2tBh0Drf0c+JADG+R27Q+sBruwMBYliPfxi0znr0c/J9mcBj/uXjDo7Hn/v9j5dfr8/Ph91ueRjr9PUxATHdHPQI/xqdLeM4a30EiCmcPeFtg86ufpy4PsuyfHt6+lWvAe7ktB1bliXZjhl8zggQ8zbonhky+FwlQMzeoK/O0OWOT31OBIhJ3SFD0vMuAWJqlxn69xJdPeSWnqsECK5n6OM9evvHatLzBgGCD2XoE6TnXQIENy6R7nycAMENYiQ6nyNAQMZXMYCMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCwVP4HFFs2v2xRmg4AAAAASUVORK5CYII=', 'sha256': 'ac144c9db808f8df19a1d4f117b0a999c37743a394422f3026f9266e43001f4c'}, 'data/binding-v1/images/binding-evaluation-0081-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFS0lEQVR4nO3cMW4bRxiAUToI78PCQC6xdQ6ZmpcI4IKHcSFATQrDhEDSpCJL+nZn3qst7iyg/fDPcOUvT0/fdwCFP5KrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+XM3rsPh624Cp9O3egnwRiYgICNAQGbkLdjZ/q9/dsN5/vfvegnwu0xAQEaAgIwAARkBAjICBGSm+BYMPvp1Vq+Dvo0AwTu8Q//yR8To9QQI3vlvd86fo0QPCRA8SM9xv7//48vz852PlaE7BIip3UzPw+Lc//cXPZKhOwSISV2n5/9251fOn/OyRDJ0kwAxnY9Lz82PlaE7vAfE1PU57vcfVJ87l5jkf6p6DQFiIi+f/E9Iz0sXl9OgH2zBmHTwSZZx3O/PO7KD7ZgJiBmspD43r36YexQSIAYXbrt+xXbsTICYqD67NTlqkAAxsDXX54fj9A0SIMa3zvqsf22fQIAY03mgWP8Tfvy5wgmHIAFiQNt9kg+bXfnbCBCjWf/Rz7XjrIdBAsSwtlKfLa72vQgQQ9nQ0c+1CQ+DBAjICBDj2PT4M+cQJEBARoAYxADjz4RDkAABGQFiBMOMP7MNQQIEZAQIyAgQmzfY/muqXZgAARkBAjICBGQEiG0b+4hk+HsUIAYx0gn0qHd0TYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBYhDL8/NuLMtwd3RNgNi20+nbbnSnce9RgICMAAEZAQIyAsQ4RyQjndouP+9l4AMgAQJKAgRkBIgRDLYLW+bYfwkQUBIgBjHMELRMM/4IEFASIMYxwBC0zDT+CBBQEiCGsukhaJls/BEgRratBi2bWu17ESBG83J82MpTvbxY5zzjjwAxpu0+w6fNrvxtBIgxbegwaJnv6OdMgBjfmhu0rHhtn0CAGNb6D4OWWY9+zr48PX3fDepw+LqbwJy/uG/7NTju97vVWKavjwmI6eagNYxCF8s4zVofAWIKF09426CLq58mrs/gWzBY23bM4HNBgJjLxcngp2XI4HOTADGd628nPjRD1zs+9TkTICb1CRmSnocEiKndfFfjN0t085Bbem4SIHjwytjDHt3/Wk167hAg+Kg3V6XnIQGCdy6R7ryeAME7xEh03kaAgIw/xQAyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DArvIfDnk1sJx2oRIAAAAASUVORK5CYII=', 'sha256': 'b6085b9a434ac2031f9cd7d6d89bfd2bd5c272e0a230d818d7f8fe84199a39de'}, 'data/binding-v1/images/binding-evaluation-0082-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFOElEQVR4nO3dQU4bSRiA0c4ofZyRvOBmcw5uloWlXCULJDZZWGMZjyFhAv66q95bsQC7WnJ//qssiy9PTz8WgMJfybMCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg83UZy+HwsOzW8fitXgLc1WgBgq2923lfeYMAweeO2Jd/IkazBOj5n7+XnVgfv9dL4E5b+/PjKNHgAYI7pOeX73OvvbucHvY4fYYECN6RnvdO1le/f9Wjw/QZEiD4dXo+akd/fpzLEh0mzpAAwT3Sc/Nh1+kzJEBwuz53+Bzj+VaGpmqQAMF1fe78EerzywxN1SBfxWB2h8NDWJ+bz3u1pIEJEFO7/7brDc8vn32GBtmCMa8tDD6Tb8dMQExqg/V5bTu2jEuAmNGW6zNVgwSIqW2zPttf20cRIKZzHii2f4c//7vCUYcgAWIu+72TD7td+RsEiIls/+hntsMgAWJGe6nPHlf7LgLELHZ09DPPYZAAARkBYgq7Hn8GHoIECMgIEOMbYPwZdQgSICAjQAxumPFnyCFIgICMAAEZAWJkg+2/xtuFCRCQESAgI0BARoAY1gBHJMNfowAxvpFOoAe7IgECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQEiPGtj9+XsayjXJEAMazj8dsyuuPOr1GAgIwAARkBAjICxBRHJMOc2i4X17L3AyABAkoCBGQEiMENtgtbB9p/CRBQEiDGN8wQtI41/ggQUBIgpjDAELQON/4IEFASIGax6yFoHXH8WZbl6zKoPb7IuJv18fuO/rnoOu6L2QTERC7Hh73c1evFOgcbfwSI6ez3Hj7uduVv+PL09KNeA9zb4fBw+mHjG7F10KOfMxMQU9vyRmzd8No+igAxo+0fBq1DH/2cCRCT2nKD1jnq4wyI2Z0PgzZyJLS+TOHY9TEBMburO7wdhdbJ6mMCghujUDIHrdNsuy4JEMTbsXW+wedMgODVBn12htb/7Pimqo8AQZMh6TkRIPjdDP15iW4ech+nTM+JAMG7M/T7PXr7Y7XjxOk5ESD40wz9D9JzIkBwvxLpzhUBgs+Nkei8QYCAjK9iABkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELBUfgKnJz9pqQoe+wAAAABJRU5ErkJggg==', 'sha256': '98b60167d1892cf17d65a55a917f894ca164983cbed4be6a29e65be9ffff7f42'}, 'data/binding-v1/images/binding-evaluation-0082-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRklEQVR4nO3dMW7bSBiAUWUR3UdFgFyCdQ65tS4RIIUOk8KAmi2MFRTHVuwk1kfOvFe5sKhhwc//jGX4w8PD9x1A4Z/kXQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD5uBvL4fBpt1mn07d6CXBXowUI1vbTzs+VGwQI3nfEvn6JGM0SoP3nf3cbcf76pV4Cd9raX66jRIMHCO6QnuN+f/vly/l847Kn6TMkQPCG9PyyOLe//0mPDtNnSIDg1+l5a3decrnOdYkOE2dIgOAe6Xn2ssv0GfJBRHi+Psf9/p3qc+MtDlv+INtvECB4+uTfIT3Xjj++3VQNsgVjdj8PPskyjvv9ZUc2z3bMBMTUVlKfRxNuxwSIeYXbrpccJ9uOCRCTelKf3Zocp2mQADGjNddnqgYJEFNbZ33Wv7a/RYCYzmWgWP8Tfvx/haMOQQLEXLb7JB82u/IbBIiJrP/oZ7bDIAFiRlupzxZX+yYCxCw2dPQzz2GQAAEZAWIKmx5/Bh6CBAjICBDjG2D8GXUIEiAgI0AMbpjxZ8ghSICAjAABGQFiZIPtv8bbhQkQkBEgICNAQEaAGNYARyTD36MAMb6RTqAHuyMBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBIjxLefzbizLKHckQAzrdPq2G91p4/coQEBGgICMAAEZAWKKI5JhTm13V/ey9QMgAQJKAgRkBIjBDbYLWwbafwkQUBIgxjfMELSMNf4IEFASIKYwwBC0DDf+CBBQEiBmsekhaBlx/Nntdh93gzp//VIvgfVazucN/XPRZYPFfCUTEBO5Hh+28lQvV+scbPwRIKaz3Wf4tNmV3/Dh4eF7vQa4t8Ph0+MXK9+ILYMe/VyYgJjamjdiy4rX9rcIEDNa/2HQMvTRz4UAMak1N2iZoz7OgJjd5TBoJUdCy48pHLs+JiBm9+QJb0ehZbL6mIDgmVEomYOWabZd1wQI4u3YMt/gcyFA8GKD3jtDy087vqnqI0DQZEh6HgkQvDZDf16iZw+5T1Om55EAwZsz9Poe3f612mni9DwSIPjTDP0G6XkkQHC/EunOEwIE7xsj0blBgICMP8UAMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYFf5D4EyMIvv2CNCAAAAAElFTkSuQmCC', 'sha256': 'c7c6ddbf04a160d10c0fdccc45c19a468f0fbb7321e8a083e8d8286c2619c6f2'}, 'data/binding-v1/images/binding-evaluation-0082-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFR0lEQVR4nO3dsW0bSRiAUfpglnMAA5ex3Vwd7oZlOCDgVhwIYOJAOIKWJVqyLX67M+9FCsTlbLCf/hlR0IeHh287gMI/ybsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg83E3lsPh026zTqcv9RLgrkYLEKztp52fKzcIELzviH39EjGaJUDn//7dbcT+89d6Cdxpa3+5jhINHiC4Q3qO+/3tly/n843LnqbPkADBG9Lzy+Lc/v4nPTpMnyEBgl+n563decnlOtclOkycIQGCe6Tn2csu02fIBxHh+foc9/t3qs+Ntzhs+YNsv0GA4OmTf4f0XDv++HZTNcgWjNn9PPgkyzju95cd2TzbMRMQU1tJfR5NuB0TIOYVbrtecpxsOyZATOpJfXZrcpymQQLEjNZcn6kaJEBMbZ31Wf/a/hYBYjqXgWL9T/jx/xWOOgQJEHPZ7pN82OzKbxAgJrL+o5/ZDoMEiBltpT5bXO2bCBCz2NDRzzyHQQIEZASIKWx6/Bl4CBIgICNAjG+A8WfUIUiAgIwAMbhhxp8hhyABAjICBGQEiJENtv8abxcmQEBGgICMAAEZAWJYAxyRDH+PAsT4RjqBHuyOBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgxrecz7uxLKPckQAxrNPpy250p43fowABGQECMgIEZASIKY5Ihjm13V3dy9YPgAQIKAkQkBEgBjfYLmwZaP8lQEBJgBjfMEPQMtb4I0BASYCYwgBD0DLc+CNAQEmAmMWmh6BlxPFnt9t93A1q//lrvQTWazmfN/TPRZcNFvOVTEBM5Hp82MpTvVytc7DxR4CYznaf4dNmV37Dh4eHb/Ua4N4Oh0+PX6x8I7YMevRzYQJiamveiC0rXtvfIkDMaP2HQcvQRz8XAsSk1tygZY76OANidpfDoJUcCS0/pnDs+piAmN2TJ7wdhZbJ6mMCgmdGoWQOWqbZdl0TIIi3Y8t8g8+FAMGLDXrvDC0/7fimqo8AQZMh6XkkQPDaDP15iZ495D5NmZ5HAgRvztDre3T712qnidPzSIDgTzP0G6TnkQDB/UqkO08IELxvjETnBgECMv4UA8gIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBd5TvNCS/W6PVtPAAAAABJRU5ErkJggg==', 'sha256': 'c2c76dc185fd08854ba99046288df83849f1d55a46488800565712188af5a6a6'}, 'data/binding-v1/images/binding-evaluation-0083-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFklEQVR4nO3bPW5aWxSAURIl86Fwy4QyoEzIrQsGk8JSmldY4aELwvkxfPees1blwoITyffTPhvy6fX1xw6g8Dl5VwABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZkvu1Hs90+7IRyPL/UR4EFMQEBGgIDMOFewk+fvP3cbdPj2tT4CPNqAAYLHbxht7v6OAMEHfKxx/sti9PsECD7449TTKyjRuwQI3k/Pu4vFqyu8txeUoRsECK6n548+zTj/5UWMZOgGAWJql+n5909RT69wXiIZukqAmNeiPh/+BY63F1xkSIPOCRCTOq/PXb87tsiQBp3zTWims98/Paw+V99lcYCZCRBzufe164bFe+01yBWMqTx+8FlwHVswATGLvD5X330/9xwkQExnDf9deQ1nWAMBYgqnQWM9T/7zr5PMPAQJEONb/xO+X/0J70SAGNx6Vj+XnqdfBgkQs1hbfdZ8qocRIEa2wtXPpeeJl0ECBGQEiGFtYvyZfAgSICAjQIxpQ+PPzEOQAAEZAQIyAsSANnf/mvYWJkBARoCAjAABGQFiNBtdAM25BhIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgBjN8fjy9sPh29fd1hx+nfn0rxibAAEZAQIyAgRkBIgBbXQNdJhsASRAQEmAgIwAMabN3cIO892/BAgoCRDD2tAQdJhy/BEgoCRAjGwTQ9Bh1vFHgJjIOht0WOWpHkaAGNz5WLG2p/1wdp4Jxx8BYgrrf7aPqz/hnQgQU1jhMugw8ernRICYzhoatIYzrIEAMYv1LIOsfk6+/P8jjO54fNnvn84r8Pz95yMPsAjfce76mICYzuKZf+QopD6XBIjpHI8vj7+OLa5d6vPm0+vrj90QTqP11vnTrP5m7nQdM/jcYAfEvM5XQvfYCl3OVuqzIEBM7a0Ilxn6lxJdvdNJz+BXMLjfLf7dGN1YJEnPDQIE91omSs+7BAg+uES68/sECD4gRqLzdwQIyPgiIpARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECdpX/ANwlHO0iatMJAAAAAElFTkSuQmCC', 'sha256': 'f85de9fad07686f571add25f7987e9438821bb998c4640dabcb4a691e351590c'}, 'data/binding-v1/images/binding-evaluation-0083-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFH0lEQVR4nO3aQY4aRxiAURyZ40Ri4WP0bXKO3IZjeIGUq3gxEpssRkYIMONMgK+7673VjMR010j0p78Kvry9/dgAFP5I7gogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzdbMWu923zSocDt/rJcCLmICAjAABmfVswU6Of/25WaDt3//US4BXW2GA4PUnjE7uPkeA4AEfa5y/WIx+nwDBgz9OPV1BiT4kQPBxevbb7f2/nY7HX11Qhu4QILidng+j86sXX8RIhu4QIIZ2nZ7/1J2bTlc4L5EM3eR7QIzroj777fb/1+f+BVfzff1HESAGdd6Ch6fn3MXFNeicLRjDuR58XnDT/XZ72pHZjp2YgBhLUp+b99oZhQSIobxs2/UrtmMXBIhRXNQnXIkGnQgQw2nrM581zIEAMYTToDGfJ3//cyUjD0ECxPrN/wnfzX6FTyJArNx8jn6u7Yc/DBIgRjG3+sx5VS8jQKzZDI9+ru0HPgwSICAjQKzWIsafwYcgAQIyAsQ6LWj8GXkIEiAgI0BARoBYocXtv4bdhQkQkBEgICNAQEaAWJuFHgCNeQwkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjACxNofD9/cfpuNxszTTzzWf/ot1EyAgI0BARoCAjACxQgs9BpoGOwASIKAkQEBGgFinxe3CpvH2XwIElASI1VrQEDQNOf4IEFASINZsEUPQNOr4I0AMZJ4Nmma5qpcRIFbufKyY29M+na1nwPFHgBjC/J/tw+xX+CQCxBBmeBg0DXz0cyJADGcODZrDGuZAgBjFfA6DHP2cCBDjNuj1Gbq46WHs+ggQw7l45l/ZoIt7HYavz2az+VovAF7t/cnf7b69/zodj/vt9tk3Nfjc9OXt7cdmFU7vp6Xz7qzeM0/KkMHnDlswxnW9HXvsjuz6gupzwRaMoV1sx84Hlk8PRDcrJj0r34LB83bxH8bozugkPXcIEDzrMFF6PiRA8OAS6c7vEyB4QIxE53MECMj4GB7ICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZvKv10FFom00bdPAAAAAElFTkSuQmCC', 'sha256': '3c3f2e3a6ca68684abb7b5311f92ce1e0b9c3fc463116156134b247433831a43'}, 'data/binding-v1/images/binding-evaluation-0083-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHklEQVR4nO3aMY4aSRiAUbwy9yFwujfok+yBfBJu4NQBh3EwEskGIyMEDHhnga/pei+akVB3IdGf/ir48vb2awVQ+Cu5K4AAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMzX1VJsNt9Wi7Db/ayXAE9iAgIyAgRklrMFO/jxfb96QX//s66XAM+2wADB808Yndx9jgDBHb7WOH6xGP05AYI7f516uIIS3SRAcDs92/WNE7ppv//ogjJ0hQDB5fTcjM5HLz6JkQxdIUAM7Tw9/6k7Fx2ucFwiGbrI74AY10l9tuv1/6/P9Qsu5vf69yJADOq4BXdPz7GTi2vQMVswhnM++Dzhptv1+rAjsx07MAExlqQ+F++1MQoJEEN52rbrI7ZjJwSIUZzUJ1yJBh0IEMNp6zOfNcyBADGEw6Axnyd/+3slIw9BAsTyzf8J38x+hQ8iQCzcfI5+zm2HPwwSIEYxt/rMeVVPI0As2QyPfs5tBz4MEiAgI0As1kuMP4MPQQIEZASIZXqh8WfkIUiAgIwAARkBYoFebv817C5MgICMAAEZAQIyAsTSvOgB0JjHQAIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEEuz2/18/2Pa71evZvq95sO7WDYBAjICBGQECMgIEAv0osdA02AHQAIElAQIyAgQy/Ryu7BpvP2XAAElAWKxXmgImoYcfwQIKAkQS/YSQ9A06vgjQAxkng2aZrmqpxEgFu54rJjb0z4drWfA8UeAGML8n+3d7Ff4IALEEGZ4GDQNfPRzIEAMZw4NmsMa5kCAGMV8DoMc/RwIEOM26PkZOrnpbuz6CBDDOXnmn9mgk3vthq/ParX6Wi8Anu39yd9svr3/O+332/X60Tc1+Fz05e3t12oRDp+nV+fTWX1mHpQhg88VtmCM63w7dt8d2fkF1eeELRhDO9mOHQ8snx6ILlZMeha+BYPH7eJvxujK6CQ9VwgQPOowUXpuEiC4c4l0588JENwhRqLzOQIEZHwND2QECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAVeVf4UQWiUBfUXAAAAAASUVORK5CYII=', 'sha256': 'a923d3fd7d5debd7ebd093e97fbeffb35ace4ebcaf4861703ba12e2e92a2183f'}, 'data/binding-v1/images/binding-evaluation-0084-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFI0lEQVR4nO3cTU4iURhAUe00+6mBy6jluowesBgHJE56QLchQONPI7eKd85EE4X3EuD6vYL4uNu9PAAUfiSrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyP7ulYWWm6enyL2y3v261lzvxuNu91HuAFUfnX8ToIwQIrtmdU0p0gQDBR9PzvNlcvu38+vqvH8nQWQIEl9LzbnQ+FSMZOiJAcL4+X07P5Qxp0KFxA3TFQ/4SeFpf65lwle68WyKP157PATG029Tn9J7v7O/flwkQ4zqswPNm8331ObvEpEE+iPitf/Ru4MLbLnyqPjdb93mzeXvUpulp8LOYCYgRVfU5XXEaew4SIIbT1ud03WngBgkQY1lCfU5Xn0ZtkAAxkOXU53QP05ANEiBGtIT6LG0nCQFiFG8jxtJe889/9zPgECRADGEtr+1pJfu8FgFiLEsbf5a8qxsQIO7fYg9fh8Y8iAkQkBEg7twaB4pphXv+GgFiFEs+f61lh1cnQEBGgLhnq7j8PPKlaAECMgIEZAQIyAgQd2t1F4AGvAwkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjABxt7bbX/tv5tfXh/WY/+72bf93TICAjAABGQECMgLEPVvdZaB5pAtAAgSUBIhRLH8Imhe/w6sTIO7cGs8y2xXu+WsECMgIEPdvFZei58EuP+8JEGNZZoPmRe7qBgSIIaxlrNiuZJ/XIkCMYrEHsXnIw9eeADGi5TRoXsxOEgLEQA5HjCW88ueDPQw4/ggQw1lOg+bh6yNAjGgJDVKfPQFiRG2D1OeNADGoqkHqc0iAGNdRg747Q0dLbIevjwAxuqMKfF+Dju5ZffZ+/vkKo9q3YJqejkrxvNn8/52fFk16DpmA4HwX/vNQdvbm6nPEBATnR6G9w4i8OxNdCJb0nCVA8H6G9r42EEnPBQIEl6pxtkSfugcuECD4aEfejZHofJYAwUfpy9V5FwzICBCQESAgI0BARoCAjAABGW/DL+J/A8OYTEBARoCAzONu99KtDgzNBARkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIOCh8hvF8CdYKC7twwAAAABJRU5ErkJggg==', 'sha256': 'b7a3ea8ebed4e6cba463ebb6eadcc085daf08ebfac220df3d9d5683051b941f1'}, 'data/binding-v1/images/binding-evaluation-0084-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJ0lEQVR4nO3cTU5aURiAYW3qfu7AxJ10ke6kSQcsxgHJnXRAawhQ/Cny3st5nokmCuckwOt3LsT77fblDqDwLVkVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD53i0NKzNNj+d/YbP5da293Ij77fal3gOsODr/IkbvIUBwye4cU6IzBAjem56Hp+fzt51//vjXj2ToJAGCc+l5MzofipEMHRAgOF2fT6fnfIY0aN+4AbrgIX8JPK0v9Uy4SHfeLJHHa8fngBjadepzfM839vfv0wSIce1X4OHp+evqc3KJSYN8EPFL/+hdwZm3XfhQfa627sPT8+ujNk2Pg5/FTECMqKrP8YrT2HOQADGctj7H604DN0iAGMsS6nO8+jRqgwSIgSynPsd7mIZskAAxoiXUZ2k7SQgQo3gdMZb2mn/4u58BhyABYghreW1PK9nnpQgQY1na+LPkXV2BAHH7Fnv42jfmQUyAgIwAcePWOFBMK9zz5wgQo1jy+WstO7w4AQIyAsQtW8Xl55EvRQsQkBEgICNAQEaAuFmruwA04GUgAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZASIm7XZ/Np9M//8cbce89/dvu7/hgkQkBEgICNAQEaAuGWruww0j3QBSICAkgAxiuUPQfPid3hxAsSNW+NZZrPCPX+OAAEZAeL2reJS9DzY5ecdAWIsy2zQvMhdXYEAMYS1jBWblezzUgSIUSz2IDYPefjaESBGtJwGzYvZSUKAGMj+iLGEV/68t4cBxx8BYjjLadA8fH0EiBEtoUHqsyNAjKhtkPq8EiAGVTVIffYJEOM6aNBXZ+hgic3w9REgRndQga9r0ME9q8/O9z9fYVS7FkzT40EpHp6e///Oj4smPftMQHC6C/95KDt5c/U5YAKC06PQzn5E3pyJzgRLek4SIHg7QzufG4ik5wwBgnPVOFmiD90DZwgQvLcjb8ZIdD5KgOC99OXivAsGZAQIyAgQkBEgICNAQEaAgIy34Rfxv4FhTCYgICNAQOZ+u33pVgeGZgICMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBBwV/kN/Ykl8IsIebQAAAAASUVORK5CYII=', 'sha256': 'bfa480802ffec4e84761a53fd1e02539e7fc2123a59426f90896126595829fe3'}, 'data/binding-v1/images/binding-evaluation-0084-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMElEQVR4nO3aTU4bWRSAUdOK91MDJDaRcRbJuDcRKQMvhgFSTXrgDrJsx/zE8FX5nTMhUsDvSsFf7iu4e35+2gAU/klOBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkvnVHw8pM0/3lT9jtfn3VLDfi7vn5qZ4BVhydPxGjtxAguGZ3TinRBQIEb03P9uHx8tfOP3/86a9k6CwBgkvpeTU674qRDB0RIDhfnw+n53KGNOjQuAG64iV/CXxbX+s74SrdebVE/r32/B4QQ/ua+py+8o39//dhAsS4DiuwfXj8vPqcPWLSIL+IuNls/t1uN6v1fZ7rEW6kPl927vbh8eU6Nk33g9/FbECMqKrP6YnT2HuQADGctj6n504DN0iAGMsS6nN6+jRqgwSIgSynPqczTEM2SIAY0RLqs7RJEgLEKF5WjKW957e/5xlwCRIghrCW9/a0kjmvRYAYy9LWnyVP9QUEiNu32MvXoTEvYgIEZASIG7fGhWJa4cwfI0CMYsn3r7VMeHUCBGQEiFu2isfPIz+KFiAgI0BARoCAjABxs1b3AGjAx0ACBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBA3a7f7tf/D/PPHZj3m39O+zH/DBAjICBCQESAgI0DcstU9BppHegAkQEBJgBjF8pegefETXp0AcePWeJfZrXDmjxEgICNA3L5VPIqeB3v8vCdAjGWZDZoXOdUXECCGsJa1YreSOa9FgBjFYi9i85CXrz0BYkTLadC8mEkSAsRADleMJbzz54MZBlx/BIjhLKdB8/D1ESBGtIQGqc+eADGitkHq80KAGFTVIPU5JECM66hBn52hoyN2w9dHgBjdUQU+r0FHr6w+e9/+/wij2rdgmu6PSrF9ePz7Fz8tmvQcsgHB+S785aXs7JerzxEbEJxfhfYOI/LqTnQhWNJzlgDB6xna+9hCJD0XCBBcqsbZEr3rFbhAgOCtHXk1RqLzXgIEb6UvV+enYEBGgICMAAEZAQIyAgRkBAjI+DH85vs81yPAoGxAQEaAgMzd8/NTdzowNBsQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCATeU/SQkmGAY6+1oAAAAASUVORK5CYII=', 'sha256': 'd9da29788b84280942bebf1a4e95396513454092bf9c0aeda904c99c60f622d0'}, 'data/binding-v1/images/binding-evaluation-0085-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFI0lEQVR4nO3cQUqjSRiAYR26j9OQhTA36Yt4jrnI3KShF8JcxUXAzSyEEEybxOTXt+rP82zVVIHk5asi+e+32+c7gMJfyaoAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGS+dUtDZrN5OP4LT0+/v2ovN+1+u32u9wBDROc9YvR5BIiVu7g7h5RocQLELabn5fHH8b/9/s9/7/1IhhYkQNxKek5G50MxkqFFCBDrr8/F6TmeIQ26ngCx2vQs0p2TJZKha/gcECvxNfU5fOUFL7lvkAmINdivwOel58goZA66jAmI6SX1ebOWOegyAsTcqvocrqhBFxAgJtbW53BdDfooAWJWI9TncHUN+hABYkrj1OdwDxp0PgFibiPUZ7SdTESAmM9uxBjtPb/bjyHoTALEZGZ5b8+yz9Y6H0i2gv+9D7adNNr48+rl8ceRb9LzhgmImQx7+NrnIHY+AQIy6zyC7Xz/+9+7qbz8+llvYVwzDhSbzYPT9BEmIOYz8vlrlh0OQoCAjAAxhymun/e5ij6HAAEZAQIyAgRkBIgJTHcB9Mo10EkCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBAT2D1UcK7nve9266GI7xEgICNAQEaAgIwAMYfproFcAJ1DgICMADGf8Yeg8Xc4CAFiGjOeZWbc81cSICAjQMxkiqto18/nEyBmNWaDxtzVsASIycwyVsyyz5YAMZ9hD2IOXx8lQMxtnAaNs5OJCBBT2h8xRnjn7+/B+HM+AWJW4zRIfS4mQExshAapzzUEiLm1DVKfKwkQ06sapD7X+7bAa0Dt6en3ZvOw34WXxx+ft9ybzKnPxUxArMSbCnzeKKQ+CzIBsR6vLXgzCi01DR0WTXquJ0Cs+Ti2yKHsj8OU+ixCgFj/KHTYkZMxOnKCk54FCRC3laFrboikZ3ECxMrtqvHHEn3oFVicAHEr9jtyMkai8zUEiFukL4PwOSAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjIr/zb8y6+f9RaAd5mAgIwAAZn77fa5Wx24aSYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMBd5X98XCBnX79RRAAAAABJRU5ErkJggg==', 'sha256': 'edddb413b273ed965718fee7a2d016f8b596244db9841e65e1c87326c5f7e3a3'}, 'data/binding-v1/images/binding-evaluation-0085-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFI0lEQVR4nO3cTWpbSRhAUbuJl9OggcE76Y30OrIR76ShB4FsxQNBJhkYhJBi/du36umcqa33CowuX5Wl97hevz0AFP5K7gogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZbd2vIrFbPh3/hx4//v2otd+1xvX6r1wBDROcjYvR5BIiFu7g7+5To5gSIe0zP08vr4df++u+fj34kQzckQNxLeo5G56wYydBNCBDLr8/F6TmcIQ26ngCx2PTcpDtHSyRD1/A5IBbia+qzf+UbHnLfIRMQS7Bdgc9Lz4FRyBx0GRMQ00vqs3Mvc9BlBIi5VfXZv6MGXUCAmFhbn/37atC5BIhZjVCf/btr0FkEiCmNU5/9NWjQ6QSIuY1Qn9FWMhEBYj6bEWO09/xmPYagEwkQk5nlvT3LOlvLfCDZAv72Pth21Gjjz7unl9cD36RnhwmImQy7+dpmI3Y6AQIyy9yCbfz69++HqTx9/1kvYVwzDhSr1bPd9AEmIOYz8v5rlhUOQoCAjAAxhymOn7c5ij6FAAEZAQIyAgRkBIgJTHcA9M4x0FECBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBAT2DxUcK7nvW9W66GIHxEgICNAQEaAgIwAMYfpjoEcAJ1CgICMADGf8Yeg8Vc4CAFiGjPuZWZc81cSICAjQMxkiqNox8+nEyBmNWaDxlzVsASIycwyVsyyzpYAMZ9hN2I2X+cSIOY2ToPGWclEBIgpbY8YI7zzt9dg/DmdADGrcRqkPhcTICY2QoPU5xoCxNzaBqnPlQSI6VUNUp/rCRALbNBnZ2jnFupzMQFiIXYq8HkN2rmy+lzj21WvhpG8t2C1et4pxdPL6/UX3y+a9FzPBMTS7Hfhyk3ZH1+uPjdhAmL5o9C77YgcnYkOBEt6bkiAuK8MvbtsIJKemxMgFm5TjT+W6KwrcHMCxL3Y7sjRGInO1xAg7pG+DMJ/wYCMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyCz82/BP33/WSwA+ZAICMgIEZB7X67fu7sBdMwEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAh4qvwHtkSlgAK9dqQAAAABJRU5ErkJggg==', 'sha256': '286573b7a825d3e2fd84ce1765bc6c4e944baeae2c60cc9323ce928c6ddac7a7'}, 'data/binding-v1/images/binding-evaluation-0085-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3dQU5qSRiA0WfnuR8GJu6kF+lOOunBW4wDkp70gIQQUEAu+lVdzpkq91Zi+PyrBHzabt9/ART+Su4KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmd3dryGw2L+e/4c+ff39qLQ/tabt9r9cAQ0TnM2L0fQSIlbu5O6eU6O4EiEdMz/Pr2/nH/vfP3599SYbuSIB4lPRcjM6XYiRDdyFArL8+N6fnfIY0aDkBYrXpuUt3LpZIhpbwOiBW4mfqc3rlOx5yPyATEGtwWIHvS8+ZUcgcdBsTENNL6nN0L3PQbQSIuVX1Ob2jBt1AgJhYW5/T+2rQVwkQsxqhPqd316AvESCmNE59TtegQdcTIOY2Qn1GW8lEBIj57EeM0Z7z+/UYgq4kQExmluf2LOtsrfMDyVbws/fCtotGG392nl/fzryTniMmIGYy7ObrkI3Y9QQIyKxzC7Y38u/JD5nez5hxoNhsXuymzzABMZ/xf6+Mv8JBCBCQESDmMMXx8yFH0dcQICAjQEBGgICMADGB6Q6AdhwDXSRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMADGB/YcKzvWJkfvV+lDEzwgQkBEgICNAQEaAmMN0x0AOgK4hQEBGgJjP+EPQ+CschAAxjRn3MjOu+ScJEJARIGYyxVG04+frCRCzGrNBY65qWALEZGYZK2ZZZ0uAmM+wGzGbr68SIOY2ToPGWclEBIgpHY4YIzzzD9dg/LmeADGrcRqkPjcTICY2QoPUZwkBYm5tg9RnIQFielWD1Gc5AWKFDfruDB3dQn1uJkCsxFEFvq9BR1dWnyV+L3o0jGTXgs3m5agUz69vyy9+WjTpWc4ExNqcdmHhpuzDh6vPXZiAWP8otHMYkYsz0ZlgSc8dCRCPlaGd2wYi6bk7AWLl9tX4sERfugJ3J0A8isOOXIyR6PwMAeIR6csg/BUMyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMyt8NP8I/7QU+YwICMgIEZJ622/fu7sBDMwEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAn5V/ge/TCoVNzGKaQAAAABJRU5ErkJggg==', 'sha256': '181c91d74bc435b507178ca1e067cdba20bee585e184ffb8879243b5243bdcf4'}, 'data/binding-v1/images/binding-evaluation-0086-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIUlEQVR4nO3aMU4bTRiAYRKF40RyQesLpf3PkTYXoqWwlKukQKJJYf2WtTY2yLHf9c7zVAjBzlDsq28Gf3l9/fMAUPiarAogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZbt/S8rFZPD4uz2bzUW4BTTEBARoCAjCPY1Nt/3x/u3OPP3/UWBjqbO+deQoDgogvB/R8Wo88SIPhn/4jYPUGJPkiA4FR6nn+9nf7d9Y/H9x4oQ2cJEEM7mp6z0XnvhycxkqGzBIhBHabnU905aveE/RLJ0AkCxIgm9bk8PRPbB04ypEGHBIih6/PP03MiQxp0yAcRGchq9XSz+hxdZbIBBIhRXPvYdcJkLQ3acQRjCLcffCYcx44yAbF8eX2Orr4yBwkQQ2nrM589zIcAsXC7QWM+b/5uJ6vhhyABYsnm/4avZr/DqxIgFms+Vz+HXAZtCRDLN7f6zHlXNyZALNMMr34OPQ9/GSRAQEaAWKC7GH+2Bh+CBAjICBBLc0fjz9bIQ5AAARkBAjICxKLc3flr8FOYAAEZAQIyAgRkBIjluNMLoJGvgQQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIJZjs3nZfrH+8fhwb9b/73n3V4xAgICMAAEZAQIyAsSi3Ok10HrICyABAkoCBGQEiKW5u1PYetTzlwABJQFige5oCFoPPP4IEFASIJbpLoag9djjjwAxhHk2aD3LXd2YALFY+2PF3N72/f1sRh1/BIiFm/+7vZn9Dq9KgFi4GV4GufrZESAGMocGzWEP8yFALN98LoNc/Ux8m34DlmizeVmtnvYr8Pzr7ZYbmIRPfbZMQIxi8s7fchRSn/cIEAPZbF5ufxybHLvUZ58jGMO52XHM4HOWADF6g66RocPZSn2OEiAGtS3CYYYuKdHRM530nCBADO0wQ5OOnI3RiYsk6TlLgOB4hi65qJaeDxKgqcefv+st0NhV42iJPvUEPkiA4FRHzsZIdC4hQHCKvlzVl9fXP9ddAeAdPgkNZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DAQ+UvKxUdefq9kn4AAAAASUVORK5CYII=', 'sha256': 'd11a5f73d824d6e155a7155a92a92d388b1ec6807d7f3b42960ed322c4ef6b6d'}, 'data/binding-v1/images/binding-evaluation-0086-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKklEQVR4nO3aMU4bWxSAYfIU78dFJKpsg0VkQVkE23gVUgovJgWSm1dYz7HGxoYA/sdzv68iEpm5FP51zoUvz8+/7wAK/yRvBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg87V79bys19/uFmez+VUfAc4xAQEZAQIyVrCpf39u727c9x+r+ggD7eb23PcQIHjXheDhN4vRWwkQfNgvIvZPUKJXEiA4l57V/eP5/7t9enjpgTJ0kQAxtJPpuRidl755EiMZukiAGNRxet7UnZP2TzgskQyd4dfwjGhSn9X94/vrc/6Bi/xL1/cTIIZz2IIPT8+hycM16JgVjKEHnyu8dHX/uN/IrGMTJiBGkdTn5LuMQnsCxBCutna9xDp2kgAxXH3Ck2jQhAAxkLY+8znDfAgQC7cfNObzyd+fZD38ECRALNn8P+Hr2Z/wUwkQizWfq59jLoN2BIjlm1t95nyqKxMglmmGVz/HVsNfBgkQkBEgFugmxp+dwYcgAQIyAsTS3ND4szPyECRAQEaAgIwAsSg3t38NvoUJEJARICAjQEBGgFiOG70AGvkaSICAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBYjk2m1+7L7ZPD3e3Zvv/mfc/xQgECMgIEJARICAjQCzKjV4DbYe8ABIgoCRAQEaAWJqb28K2o+5fAgSUBIgFuqEhaDvw+CNAQEmAWKabGIK2Y48/AsQQ5tmg7SxPdWUCxGIdjhVz+7Qfnmcz6vgjQCzc/D/bm9mf8FMJEAs3w8sgVz97AsRA5tCgOZxhPgSI5ZvPZZCrnwkBYsQGXT9Dk5eqz44AMYrJZ/6aDZq8S332vv75EpZu98lfr7/t/rl9eljdP372Sw0+Z5iAGM7V1jFr10UCxIiO17GPzdDxA9XnJCsYg5qsY4e70l/vZScrJj1nCBBDO87QpCMXY3RmdJKeiwQITmdo5+9WM+l5JQGa+v5jVR+Bxr4aJ0v0pifwSgIE5zpyMUai8x4CBOfoy6f68vz8+3PfAPACfwcEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DAXeU/SpgeJo/MAwEAAAAASUVORK5CYII=', 'sha256': '8fc0f7048ee67096dc56499b4f83b261e91dea7e126d814cfd31b24fb11483f6'}, 'data/binding-v1/images/binding-evaluation-0086-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKklEQVR4nO3aQU4bWRRAUdKKl9OSB5HYCRvJOrIRdoKUgaVsJQMkT3pgtWOVjQ0BfMv1zxkRiVR9Br5678OX5+ffdwCFf5K3AggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dq9el7W6293i7PZ/KyPAOeYgICMAAEZK9jU9vu/dzdu9eNXfYSBdnN77nsIELzrQvDwm8XorQQIPuwXEfsnKNErCRCcS8/q/vH8/90+Pbz0QBm6SIAY2sn0XIzOS988iZEMXSRADOo4PW/qzkn7JxyWSIbO8Gt4RjSpz+r+8f31Of/ARf6l6/sJEMM5bMGHp+fQ5OEadMwKxtCDzxVeurp/3G9k1rEJExCjSOpz8l1GoT0BYghXW7teYh07SYAYrj7hSTRoQoAYSFuf+ZxhPgSIhdsPGvP55O9Psh5+CBIglmz+n/D17E/4qQSIxZrP1c8xl0E7AsTyza0+cz7VlQkQyzTDq59jq+EvgwQIyAgQC3QT48/O4EOQAAEZAWJpbmj82Rl5CBIgICNAQEaAWJSb278G38IECMgIEJARICAjQCzHjV4AjXwNJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAsRybzc/dF9unh7tbs/3/zPufYgQCBGQECMgIEJARIBblRq+BtkNeAAkQUBIgICNALM3NbWHbUfcvAQJKAsQC3dAQtB14/BEgoCRALNNNDEHbsccfAWII82zQdpanujIBYrEOx4q5fdoPz7MZdfwRIBZu/p/tzexP+KkEiIWb4WWQq589AWIgc2jQHM4wHwLE8s3nMsjVz4QAMWKDrp+hyUvVZ0eAGMXkM3/NBk3epT57X/98CUu3++Sv1992/9w+PazuHz/7pQafM0xADOdq65i16yIBYkTH69jHZuj4gepzkhWMQU3WscNd6a/3spMVk54zBIihHWdo0pGLMTozOknPRQIEpzO083ermfS8kgBNrX78qo9AY1+NkyV60xN4JQGCcx25GCPReQ8BgnP05VN9eX7+/blvAHiBvwMCMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgrvIfvuse3ypYrfgAAAAASUVORK5CYII=', 'sha256': 'cca1ee78d5f61cb788e467ff80324c403d14794aa1e63638282c67fe870b6729'}, 'data/binding-v1/images/binding-evaluation-0087-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFWklEQVR4nO3dO24bSRRAUWpg7UeBU++gV+IFeSXagdMJuBgFBJRMQAyhj0XJ1Od2V52TGf50NeC+flVqi1eHw90OoPBPclUAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg8627NIzj5ub7W37Zfv/v569lS64Oh7t6DTBydF6yFyMBgi/uznP7iUskQPAx6bm9vj7/25f7+zM/u58yQwIEF6bn1eJc0KP9ZBkSIPi79LyzO28p0X6aDAkQZOl5aM4MCRC8Xp9PTc+ZDA3fIAGCc/X5svS8lKGxGyRA0A8+045C/isGrK4+u2dX//CXj1bCBASr2HbNuR0zAcF667N7vJ7x5iABYnZrrs/wDRIgWHV91r+29xAgpnYaKNb/hN/+v8KRhiABYl7bfZJvNrvyJwSISa3/6GeGwyABYnZbqc8WVzvRe0Bj/IMw8Bsfq7Kho5/zLwdt/e+JCQjICBDT2fr4sxvoK2IDfirG71/nvvHlyv34udVHAi5gAmIuA4w/Iw1BAgRkBIiJDDP+DDMECRCQESAgI0DMYrD91xi7MAECMgIEZAQIyAgQU9joEcnw9yhAzGWkE+gB7kiAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAWIup4/0G8ay5TsSIKaw9U8QHfUeBQjICBCQESAgI0DM4nREsulT2ydO97LFAyABAkoCBGQEiIkMtgtbNr7/EiCgJEDMZZghaNn++CNAQEmAmM4AQ9AyxPgjQEBJgJjRpoegZZTxR4BgYw1aNrXaVwkQk3o4PmzlqV4erHOA8UeAmNp2n+H9Zlf+hAAxtQ0dBi0DHf2cCBDs1t+gZcVrew8BYnbrPwxahjv6OREgWHWDlnHrI0Dw5watIUPL42WMVx8Bgt1LT3jboOXx1Yesz263+1YvAFbk+Jzf3Hw//nC5v7+9vv76ZSyjDz4nJiBY0XZsmWDb9ZAAwZu2Y5+doeXZJYavjy0YvHU7dtoZffimbHmWthnScyRAcEmG3l+i5U8j1TzpORIguCRDzwvyao/Ob+L2k6XnSIDgXRk6ufiQaD9leo4ECC7sxUsl+ts/Z2YCBB9QkDfGSHSeECD4AMpyGe8BARkBAjICBGQECMgIEJARICAjQEBmwPeAfvwMvoMUcAETEJARICBzdTjcdVcHpmYCAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABu8p/rzBp/8cQxJYAAAAASUVORK5CYII=', 'sha256': 'a24cab461ab81b2d01e78227b0fbf4e128d11cc79981878ed56901235965db3b'}, 'data/binding-v1/images/binding-evaluation-0087-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVklEQVR4nO3dS24TSxiAUeeK7CcDJDbBmEUyvptAugMvJgNLPWFgXSsP4hjn8XVXnTNDELpaSn/8VWnkm8PhfgdQ+Ce5KoAAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD50l0axnF39/WSP7bf//fxa9mSm8Phvl4DjBydl+zFSIDgk7vz3H7iEgkQvE96br/9PP/ly68fZ353P2WGBAiuTM+rxbmiR/vJMiRA8HfpeWN3LinRfpoMCRBk6XlozgwJELxenw9Nz5kMDd8gAYJz9fm09LyUobEbJEDQDz7TjkL+Kwasrj67Z1d/95ePVsIEBKvYds25HTMBwXrrs3u8nvHmIAFidmuuz/ANEiBYdX3Wv7a3ECCmdhoo1v+E3/6/wpGGIAFiXtt9ku82u/InBIhJrf/oZ4bDIAFidlupzxZXO9F7QGP8gzDwGx+rsqGjn/MvB239+8QEBGQEiOlsffzZDfQTsQE/FePf29vdZn1flnoJ8HlMQMxlgPFnpCFIgICMADGRYcafYYYgAQIyAgRkBIhZDLb/GmMXJkBARoCAjAABGQFiChs9Ihn+HgWIuYx0Aj3AHQkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMzl9JF+w1i2fEcCxBS2/gmio96jAAEZAQIyAgRkBIhZnI5INn1q+8TpXrZ4ACRAQEmAgIwAMZHBdmHLxvdfAgSUBIi5DDMELdsffwQIKAkQ0xlgCFqGGH8ECCgJEDPa9BC0jDL+CBBsrEHLplb7KgFiUg/Hh6081cuDdQ4w/ggQU9vuM7zf7MqfECCmtqHDoGWgo58TAYLd+hu0rHhtbyFAzG79h0HLcEc/JwIEq27QMm59BAj+3KA1ZGh5vIzx6iNAsHvpCW8btDy++pD12e12X+oFwIocn/O7u6/HXy6/ftx++/n5y1hGH3xOTECwou3YMsG26yEBgou2Yx+doeXZJYavjy0YXLodO+2M3n1TtjxL2wzpORIguCZDby/R8qeRap70HAkQXJOh5wV5tUfnN3H7ydJzJEDwpgydXH1ItJ8yPUcCBFf24qUS/e3fMzMBgncoyIUxEp0nBAjegbJcx3tAQEaAgIwAARkBAjICBGQECMgIEJAZ8D2g78tSLwG4iAkIyAgQkLk5HO67qwNTMwEBGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBd5TfgJmido3bBLwAAAABJRU5ErkJggg==', 'sha256': '37d556ae5a1a61eeab2afb990db19e8bc9a696a3b36521346c36386105999e9f'}, 'data/binding-v1/images/binding-evaluation-0087-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFXElEQVR4nO3dTW4TSRiAYWdE7pMFEiuuwSE40ByCa8wKaRY+TBaWejMLa6yQEMc4P2931fPsEISultIvX1Ua+eZwuN8BFP5KrgogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkPnWXhnHc3X2+5I/t9/++/1q25OZwuK/XACNH5zl7MRIg+ODuPLWfuEQCBG+TntsvP85/+fLz25nf3U+ZIQGCK9PzYnGu6NF+sgwJEPxZel7ZnUtKtJ8mQwIEWXoemjNDAgQv1+dd03MmQ8M3SIDgXH0+LD3PZWjsBgkQ9IPPtKOQ/4oBq6vP7snV3/zlo5UwAcEqtl1zbsdMQLDe+ux+Xc94c5AAMbs112f4BgkQrLo+61/bawgQUzsNFOt/wm//X+FIQ5AAMa/tPsl3m135IwLEpNZ/9DPDYZAAMbut1GeLq53oPaAx/kEY+I2PVdnQ0c/5l4O2/n1iAgIyAsR0tj7+7Ab6idiAn4rxz9/LbrO+fr+tlwAfxwTEXAYYf0YaggQIyAgQExlm/BlmCBIgICNAQEaAmMVg+68xdmECBGQECMgIEJARIKaw0SOS4e9RgJjLSCfQA9yRAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgLEXE4f6TeMZct3JEBMYeufIDrqPQoQkBEgICNAQEaAmMXpiGTTp7aPnO5liwdAAgSUBAjICBATGWwXtmx8/yVAQEmAmMswQ9Cy/fFHgICSADGdAYagZYjxR4CAkgAxo00PQcso448AwcYatGxqtS8SICb1cHzYylO9PFjnAOOPADG17T7D+82u/BEBYmobOgxaBjr6OREg2K2/QcuK1/YaAsTs1n8YtAx39HMiQLDqBi3j1keA4PcNWkOGll+XMV59BAh2zz3hbYOWX68+ZH12u92negGwIsfn/O7u8/GXy89vt19+fPwyltEHnxMTEKxoO7ZMsO16SIDgou3Ye2doeXKJ4etjCwaXbsdOO6M335QtT9I2Q3qOBAiuydDrS7T8bqSaJz1HAgTXZOhpQV7s0flN3H6y9BwJELwqQydXHxLtp0zPkQDBlb14rkR/+vfMTIDgDQpyYYxE5xEBgjegLNfxHhCQESAgI0BARoCAjAABGQECMgIEZAZ8D+jr99t6CcBFTEBARoCAzM3hcN9dHZiaCQjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBOwq/wEQyWcHuXwf3QAAAABJRU5ErkJggg==', 'sha256': '12f5a5ce6a78f030a659c9d8499caf7cc7a8c673a5a4980241280c5a12917113'}, 'data/binding-v1/images/binding-evaluation-0088-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFO0lEQVR4nO3aQUpjSxiAUX20+8lAcBM97kU6fptocJDFOBCc9CAQgtHERM1XlXvOSFBTv5j7UVV6+/LyfANQ+C9ZFUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dUtDbHV6v7wF6zXT5eaZaFuX16e6xlgoOh8RIx+ggCxCGd3Z58SLT1A3/hmGpk3+k+/Ve4eHg9/++vfPwc+6xf0de6AWFZ9jkbnoy/ej9Fqda9BXyRALCI9J3XnXdtX2C3RZiEZWmiA/r+7u7k6v19f6xGuqj5fT88bmxd8kyENWmKA4JLpOZAhW6Hz+EdErsQl6/PRKgv588g3EiCuwe6Tf/fweJn6vLucBp1EgLi2+iQzaNB5BIi5jVCf/dU16JMEiImNU5/9GTToMwSIWY1Wnw0NOokAMb1x6jPmPCMTIKa03VyM+bRvp7IJOkyAmM9cT/Vc016YADGxMbc/4882DgFiMoMfvnY5iB0lQMxk3id53sl/lAAxpfG3P3PNWREgICNATGOi259dboIOECAgI0BARoCYw6Tnrw2nsI8IEJARICAjQEBGgJjA1BdAG66B3iVAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMADGB9fpp88Hr3z83c9pOvv1ZECCgJEBARoCAjAAxh6mvgVwAfUSAgIwAARkBYhqTnsKcvw4QICAjQExplk3QLHNWBIiZzHuKmXfyHyVATGaimyC3P0cJEBMbuUEjzzYOAWI+c20o5pr2wgSIKQ1+EHP4+iQBYnqjNWi0eUYmQMxqd3MxzjO/O4ntz1ECxMRGa5D6nEqAmNs4DVKfMwgQ0xuhQepzHgHiCht0yQy9WU59TiJAXIk3T/5lGvRmFfU51a+TvwNGtXn+V6v73TrcPTz+xFrS8y3sgLj+rdD37ob2X1B9zmYHxPVvhXY3LGdviN6tmPR8kQBxtdbrp90G7XfkaIwOb53U5+sEiGu2bcR+ic6+qNadbyRALMLhEp30CnwjAWJZdjtyNEai89MEiOXSl5w/wwMZAQIyAgRkBAjICBCQESAgI0BAZu7/A/r9+lqPAJzPDgjICBCQuX15ee5WBxbNDgjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAm4q/wBgsy9fpNEgFgAAAABJRU5ErkJggg==', 'sha256': '54f0c9fec076e861d703d0d65e91720cbed53908f41bb45cafd906449092b21b'}, 'data/binding-v1/images/binding-evaluation-0088-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMElEQVR4nO3cTUpjSxiAYb10L6chA3fW6+idOQj0VhwEenIHgRCMJpq/tyrneUaCmlPiOS9flcHnzebtCaDwX3JVAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmR3dpiK1WL8e/YL1+vddaFup5s3mr1wADReczYnQLAsQinN2dQ0q09ABd8WYamRv91rfKv9+/jn/7zz9/j3zWL+hyzoBYVn1ORuezLz6M0Wr1okEXEiAWkZ5vdedDu1fYL9H2QjK00ABdflcN6PjYz3frc/WbZPuC7zKkQUsMENwzPUcyZBQ6jzci8iDuWZ/PrrKQP49ckQmIR7D/5N95Y344CpmDvs4ExPTC+nx4XXPQ1wkQcxuhPodX16AvEiAmNk59DtegQV8hQMxqtPpsadC3CBDTG6c+Y65nZALElHbDxZhP+25VhqDjBIj5zPVUz7XaOxMgJjbm+DP+2sYhQExm8M3XPhuxkwSImcz7JM+78psSIKY0/vgz1zorAgRkBIhpTHT6s89J0BECBGQECMgIEHOYdP+1ZRf2GQECMgIEZAQIyAgQE5j6AGjLMdCHBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgJrBev24/+Pnn79Ocdivf/SwIEFASICAjQEBGgJjD1MdADoA+I0BARoCAjAAxjUl3YfZfRwgQkBEgpjTLEDTLOisCxEzm3cXMu/KbEiAmM9FJkNOfkwSIiY3coJHXNg4BYj5zDRRzrfbOBIgpDb4Rs/n6IgFieqM1aLT1jEyAmNX+cDHOM7+/EuPPSQLExEZrkPp8lwAxt3EapD5nECCmN0KD1Oc8P878PhjJev26Wr3st+Df71/3ufS75KnPt5iAeBDvnvz7jELqcyETEI9j+/zfZxSSnqsQIB55O3aLDB3OVupzNgHi8Ueh/WqcXaIP93TScyEBYimj0GFHTsbo+EGS+lxOgHhku0Yclujsg2rduSIBYhGOl+hbr8AVCRDLst+RkzESnVsTIJZLX3LeiAhkBAjICBCQESAgI0BARoCAjAABmbnfBzTCvwEGzmYCAjICBGSeN5u37urAopmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgqfI/5RYrGdFrOwwAAAAASUVORK5CYII=', 'sha256': 'b4ce98b11df845ef6c9775025b95586d9d7b9b061542b2817332fd8eda2ff457'}, 'data/binding-v1/images/binding-evaluation-0088-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFOUlEQVR4nO3awU1bSxiAUXgK5URiQRnpJnWkm1cGC6S0wsKSN1lYsiwMNtjY34zvOSskwDNRfD/9M/h+tXq9Ayj8l6wKIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMj86JaG2OPj0+EfeHl5vtZeFup+tXqt9wADRecjYnQJAsQinNydfUq09AB945tpZN7ol36rrH//PPzrD3/+Hviu/6DzuQNiWfU5Gp2Pfng/Ro+PTxp0JgFiEen5UnfetX2F3RJtFpKhhQbo/4eHu5vza72ut3BT9Tk/PW9sXvBNhjRoiQGCa6bnQIaMQqfxQURuxDXr89EqC/nzyDcyAXELdp/866TnwChkDvo8ExDTC+vz7rrmoM8TIOY2Qn32V9egTxIgJjZOffb3oEGfIUDMarT6bGjQlwgQ0xunPmPuZ2QCxJS2w8WYT/t2V4agwwSI+cz1VM+12ysTICY25vgz/t7GIUBMZvDD1y4HsaMEiJnM+yTPu/OLEiCmNP74M9c+KwIEZASIaUx0+7PLTdABAgRkBAjICBBzmPT8teEU9hEBAjICBGQECMgIEBOY+gJowzXQuwQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICbw8vK8+eLhz9+7OW13vv23IEBASYCAjAABGQFiDlNfA7kA+ogAARkBAjICxDQmPYU5fx0gQEBGgJjSLEPQLPusCBAzmfcUM+/OL0qAmMxEN0Fuf44SICY2coNG3ts4BIj5zDVQzLXbKxMgpjT4Qczh65MEiOmN1qDR9jMyAWJWu8PFOM/87k6MP0cJEBMbrUHq81UCxNzGaZD6nECAmN4IDVKf0/w48fdgJC8vz4+PT7stWP/+eZ2l3yRPfb7EBMSNePPkX2cUUp8zmYC4HZvn/zqjkPR8CwHilo9jl8jQ/mylPicTIG5/FNqtxsklevdMJz1nEiCWMgrtd+RojA5fJKnP+QSIW7ZtxH6JTr6o1p1vJEAswuESfekV+EYCxLLsduRojETn0gSI5dKXnA8iAhkBAjICBGQECMgIEJARICAjQEBm7s8B/Vqv6y0ApzMBARkBAjL3q9VrtzqwaCYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAi4q/wDby8r9ghQe78AAAAASUVORK5CYII=', 'sha256': '6618856e13685b215f4d415283b945819264e1faaf8c9995aed5f7e1a880a238'}, 'data/binding-v1/images/binding-evaluation-0089-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFI0lEQVR4nO3cQW4aSRiAUTKK7+NFpKxyjTlEDpRDzDWyGikLHyYLS97MwhqEgGCDga+r+r2VF0CXZPj0V5n2p+fn3xuAwl/JVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQObzZnyPj18203l6+lUvAW7OBARkBAjIzLAF23r4+s9mcC///l0vAe7HBARkBAjICBCQESAgI0BARoCAjAABmam+BwQ3vafH/TFXJ0Cs2lk3Eu4+WIyuQoBYo4/fwLx9BSX6CAFiXU6k5+ePl9PP/fb94U8vKEOXESBWnZ43o/OnB+/FSIYuI0CstD5npefQ69MPM6RBZxEg1pWeD3Znz/bVtiUyCp3F94CY2U3rc+KVp/wvnbdgAmJauxW4XXr2LrE7CpmD3mQCYk53rs/Ra5mD3iRATKiqz+EVNeg0AWI2bX0Or6tBJwgQU1lCfQ6vrkF/IkDMYzn1OVyDBh0lQExoCfVZ2kqWSYCYxHbEWNpnfrseQ9AhAWIGo3y2R1nn3QgQU1na+LPkVS2BADG8xW6+dtmIHSVAQEaAGNuIA8WIa74RAWISS95/jbLC+xMgICNADGyI4+ddjqL3CBCQESAgI0BARoAY1XAHQK8cA+0SICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoAY1dPTr9cfvn1/2Ixju9qn/9e/ZgIEZAQIyAgQkBEgBjbcMZADoD0CBGQEiEksfwha/grvT4AY24h7mRHXfCMCBGQEiOENcRTt+PkoAWIqy2zQMle1BALEDEYZK0ZZ590IEJNY7EbM5usEAWJCy2nQclayTALEPHZHjCV88nfXYPw5SoCYynIapD7vIUDMZgkNUp93EiAm1DZIfd5PgJhT1SD1Ocun5+ffm8E9Pn7ZTMd79xbvjZ8/Xm53rb3M+Q2+hwmIme1V4HajkPpc5vOFz4NBvLZgOwptS3GVaeiwaNKzui0YXLxVvzhDR4cp9TmXALEuJ04M34zRiR2c9FxGgFijK/7hQno+QoBYtYtLpDtXIUDw3hiJztUJEJDxPSAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjaV/wDGFhCyvsWuNgAAAABJRU5ErkJggg==', 'sha256': '914a32733ba39458caa565c7cf6aabd566543ce0ca33874b4eb5a036afce7d28'}, 'data/binding-v1/images/binding-evaluation-0089-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIUlEQVR4nO3csW5aSRiAUbKKHyeSi7R5oW3zHGn3hbZNYSmvksKSmy2sRQgINhj47sw9p3IB3JEMn/4Zc/3p+fn3BqDwV3JVAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+bwZ3+Pj1810np5+1kuAmzMBARkBAjIzbMG2/v3nZTO4b38/1EuA+zEBARkBAjICBGQECMgIEJARICAjQEBmqu8BwU3v6XF/zNUJEKt21o2Euw8Wo6sQINbo4zcwb19BiT5CgFiXE+l5+f7l9HMffvz60wvK0GUEiFWn583o/OnBezGSocsIECutz1npOfT69MMMadBZBIh1peeD3dmzfbVtiYxCZ/E9IGZ20/qceOUp/0vnLZiAmNZuBW6Xnr1L7I5C5qA3mYCY053rc/Ra5qA3CRATqupzeEUNOk2AmE1bn8PratAJAsRUllCfw6tr0J8IEPNYTn0O16BBRwkQE1pCfZa2kmUSICaxHTGW9pnfrscQdEiAmMEon+1R1nk3AsRUljb+LHlVSyBADG+xm69dNmJHCRCQESDGNuJAMeKab0SAmMSS91+jrPD+BAjICBADG+L4eZej6D0CBGQECMgIEJARIEY13AHQK8dAuwQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIEb19PTz9YeHH78249iu9un/9a+ZAAEZAQIyAgRkBIiBDXcM5ABojwABGQFiEssfgpa/wvsTIMY24l5mxDXfiAABGQFieEMcRTt+PkqAmMoyG7TMVS2BADGDUcaKUdZ5NwLEJBa7EbP5OkGAmNByGrSclSyTADGP3RFjCZ/83TUYf44SIKaynAapz3sIELNZQoPU550EiAm1DVKf9xMg5lQ1SH3O8un5+fdmcI+PXzfT8d69xXvj5fuX211rL3N+g+9hAmJmexW43SikPpf5fOHzYBCvLdiOQttSXGUaOiya9KxuCwYXb9UvztDRYUp9ziVArMuJE8M3Y3RiByc9lxEg1uiKf7iQno8QIFbt4hLpzlUIELw3RqJzdQIEZHwPCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCATeU//B8Oh3jGC48AAAAASUVORK5CYII=', 'sha256': 'bbdefe0959185a652c42f245dc2abfad939314c67ebec9be2a8269dcb0a89e7d'}, 'data/binding-v1/images/binding-evaluation-0089-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIklEQVR4nO3cQW4aSRiAUTKKjxPJi0i5yVwk58hF5iYjZWEpV8nCkjezsAYhINhg4Ouqfm/lBdAlGT79Vab96fn59wag8FdyVQABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPm8Gd/j49fNdJ6eftZLgJszAQEZAQIyM2zBth6+/bMZ3Mu/f9dLgPsxAQEZAQIyAgRkBAjICBCQESAgI0BAZqrvAcFN7+lxf8zVCRCrdtaNhLsPFqOrECDW6OM3MG9fQYk+QoBYlxPpefn+5fRzH378+tMLytBlBIhVp+fN6PzpwXsxkqHLCBArrc9Z6Tn0+vTDDGnQWQSIdaXng93Zs321bYmMQmfxPSBmdtP6nHjlKf9L5y2YgJjWbgVul569S+yOQuagN5mAmNOd63P0WuagNwkQE6rqc3hFDTpNgJhNW5/D62rQCQLEVJZQn8Ora9CfCBDzWE59DtegQUcJEBNaQn2WtpJlEiAmsR0xlvaZ367HEHRIgJjBKJ/tUdZ5NwLEVJY2/ix5VUsgQAxvsZuvXTZiRwkQkBEgxjbiQDHimm9EgJjEkvdfo6zw/gQIyAgQAxvi+HmXo+g9AgRkBAjICBCQESBGNdwB0CvHQLsECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESBG9fT08/WHhx+/NuPYrvbp//WvmQABGQECMgIEZASIgQ13DOQAaI8AARkBYhLLH4KWv8L7EyDGNuJeZsQ134gAARkBYnhDHEU7fj5KgJjKMhu0zFUtgQAxg1HGilHWeTcCxCQWuxGz+TpBgJjQchq0nJUskwAxj90RYwmf/N01GH+OEiCmspwGqc97CBCzWUKD1OedBIgJtQ1Sn/cTIOZUNUh9zvLp+fn3ZnCPj1830/HevcV74+X7l9tday9zfoPvYQJiZnsVuN0opD6X+Xzh82AQry3YjkLbUlxlGjosmvSsbgsGF2/VL87Q0WFKfc4lQKzLiRPDN2N0YgcnPZcRINboin+4kJ6PECBW7eIS6c5VCBC8N0aic3UCBGR8DwjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgE3lP+VGDoe5Bj6yAAAAAElFTkSuQmCC', 'sha256': 'a8769bbb676d39ccb44e3b6088973f340c19fdc5f332706292a3a2b91b42a371'}, 'data/binding-v1/images/binding-evaluation-0090-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3bvW0bWRhAUXphlrOAApehbraO7UZlOBCwrTgQwGQDAgQhUvwX77yZcyIH9MwT5Ln43hv6x8fHnxVA4a/krgACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg83O1AC8vv1az8P7+u14CPJIJCMgIEJBZxBZsZ/PP36sBrf/9r14CfAsTEJARICCzrC0Y3Pae1PvHbyJALNRVX87Y/7AYPZAAsSz3fylsdwUlup8AsRQn0vO2Xp/+u6+bzVcXlKF7CBALTc/Z6Hz14U8xkqF7CBDLSs9V3Tlqd4X9EsnQbbyGZyn1eVuv76/P6QvO5n8dPo0AMU/7LXh4evZ9urgGXcUWjPkPPk+46dt6vduR2Y5dzgTErCT1OXovo9AlBIj5eNq26yu2Y9cSIOZZn3AlGnQ5AWJu2vpMZw1DECDmYDdoTOfJ363EEHSCADG86T/h019hRYAY23SOfg45DDpLgJiJqdVnyquaDgFiYBM8+jnkMOgEAQIyAsSohhh/tgxBXxEgICNADGmg8WfLEHSUAAEZAQIyAsR4htt/bdmFHRIgICNAQEaAgIwAMZhBD4C2HAN9IkBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAMZj399/bP7xuNqvR7Na8+ykWToCAjAABGQECMgLEeAY9BnIAdEiAgIwAARkBYkjD7cLsv44SICAjQIxqoCHI+PMVAQIyAsTAhhiCjD8nCBAzMc0GTXNV0yFAjG1/rJja076/HuPPUQLE8Kb/bE9/hRUBYg4meBjk6OcSAsTcTKFBU1jDEASImZjOYZCjn8sJELNt0PMz9Omm6nPWj4+PP6u5e3n5tZoF/6Bv+HW/rdfPue+n3vllXeLnRZ+CcWyf/F2GXjebJzTI4HMbWzDm6WnbMduueyxiC8ZiHe6+HzgNHUZNfa4lQKyWeQh4c4mODlPScxsBYilOvIs4G6MTOzjpuYcAsSwPfCUqPfcTIBbq5hLpzgMJEJyPkeh8EwECMr4HBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAlaV/wHlIBRDfy7lLgAAAABJRU5ErkJggg==', 'sha256': '88299c0ab89ecbffcffee4ee9aef707a641ecc4e0b78a223b3885c31d55d7e02'}, 'data/binding-v1/images/binding-evaluation-0090-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFklEQVR4nO3aTW4aWRSAUdKKlxOJQZbRu8k6sptehgdI2UoGljzpARKyAJfNj/nqVZ0z8gDDi2I+3fvg28vL3w1A4Z/kVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5vtmBbbbn5tF2O2e6yPAPZmAgIwAAZlVrGAH/z09bQb07+trfQT4EiYgICNAQGZdKxhc9zmpzx+/iACxUhd9OePtg8XojgSIdbn9S2GHZ1Ci2wkQazGRntdfP6Z/9+n3n/eeUIZuIUCsND0fRue9Bx/FSIZuIUCsKz0XdeeswzO8LZEMXUeAWEt9bk/Pkf0THmVIgy4iQCy/PndPz0SGNOgivojI0my3Px9Wn7OvcnQAJggQi/LVa9eEo9fSoM+wgrEcjx98jljHLmUCYiHy+px9dXPQNAFiadr6zOcMQxAgluAwaMznnX84iSFoggAxvPm/w+d/wooAMbb5XP2cchn0IQFiIeZWnzmfaj4EiIHN8OrnlMugCQIEZASIUQ0x/uwZgt4jQEBGgBjSQOPPniHoLAECMgIEZASI8Qy3f+3Zwk4JEJARICAjQEBGgBjMoBdAe66BjggQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQAxmt3ve//D0+89mNIczH/4VKydAQEaAgIwAARkBYjyDXgO5ADolQEBGgICMADGk4bYw+9dZAgRkBIhRDTQEGX/eI0BARoAY2BBDkPFnggCxEPNs0DxPNR8CxNjejhVze7e/PY/x5ywBYnjzf2/P/4QVAWIJZngZ5OrnMwSIpZlDg+ZwhiEIEAsxn8sgVz+f9/2Cx8K87XbP2+3PtxV4/fXjkQc4Cp/6fOjby8vfzdId/ihH5w/6iv/uhzVIfa5gBWNpdrvnx69jR2uX+nySFYxletg6ZvC5xSpWMFbrdPu+Y4ZOZyv1uZQAsVnnJeDVJTq700nPdQSItZj4LOLDGE1cJEnPLQSIdbnjR6LSczsBYqWuLpHu3JEAwccxEp0vIkBAxhcRgYwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwKbyPyWUG8wPqJByAAAAAElFTkSuQmCC', 'sha256': '0558f677281bf72a1a1e859a0e0e84aefb5bf2b964408a5cadcc758003854d21'}, 'data/binding-v1/images/binding-evaluation-0090-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEUlEQVR4nO3cPW4aURhAURIly4lE4Z1lHdmZC6RsJYWlNCmQEAIC5vfOmzmncoGGZxmuvvdm5C8fH39WAIWvybsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMy31QKs12+rWdhs3uslwCOZgICMAAGZRWzBdv7+/LEa0Pdfv+slwFOYgICMAAGZZW3B4Lb7pO4/PokAsVBXPZyx/2IxeiABYlnufyhsdwUlup8AsRRn0nPx9ujJG5HbC8rQPQSIhabnqmcy9l98ECMZuocAsaz03P8s2O4K+yWSodsIEEupz8MfQ91e8CBDGnQVAWL+9XnqE/AHGdKgq3gQkblZr99eVp+T73KwAM4QIGbl2duuMw7eS4M+wxaM+Xj94HPAduxaJiBmIq/PyXc3B50nQMzNFP7pyhTWMAQBYg52g8Z0vvm7lRiCzhAghjf9b/j0V1gRIMY2naOfYw6DLhIgZmJq9ZnyqqZDgBjYBI9+jjkMOkOAgIwAMaohxp8tQ9D/CBCQESCGNND4s2UIOkmAgIwAARkBYjzD7b+27MKOCRCQESAgI0BARoAYzKAHQFuOgQ4IEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0AMZrN53/7w/dfv1Wh2a979FgsnQEBGgICMAAEZAWI8gx4DOQA6JkBARoCAjAAxpOF2YfZfJwkQkBEgRjXQEGT8+R8BAjICxMCGGIKMP2cIEDMxzQZNc1XTIUCMbX+smNq3fX89xp+TBIjhTf+7Pf0VVgSIOZjgYZCjn88QIOZmCg2awhqGIEDMxHQOgxz9fN63K14L07bZvK/Xb/sV+PvzxysXcBA+9bnoy8fHn9Xc7T6Uo/OBvuHP/bIGqc8NbMGYm83m/fXbsYNtl/p8ki0Y8/Sy7ZjB5x6L2IKxWMe77wdm6Hi2Up9rCRCrZR4C3lyik3s66bmNALEUZ+5FXIzRmYMk6bmHALEsD7wlKj33EyAW6uYS6c4DCRBcjpHoPIkAARkPIgIZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBV5R/DjxvMsvrkEwAAAABJRU5ErkJggg==', 'sha256': '4de8a034305ecd8ffac6dc1ad80fc7986bb6c430ee49bc215191c5345425eb6d'}, 'data/binding-v1/images/binding-evaluation-0091-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEUlEQVR4nO3cPU4jWRhAUXrU7McBElFvoxfRC5pFsI2JWiLwYjpAIpnAkrtkhI1/4L6qOidCyFXvJXX16sPi28vLnzuAwj/JqgACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyHzvlh7FZvNwtwLb7XO9BTjkBARkBAjIeAX7679/X+8W58ev+3oL8C4nICAjQEBGgICMAAEZQ2jW7uQXwXyF6vMIEGt01rdPpx8Wo9sSINblyi++7y6XoVsRINbenfvHp+PXvv7++d7dlOhKAsQa63MyOu99+CBGm82DBl1DgFiyK9Pz1u7yaYa8lF3Dn+FZS33uH5+urM+RW63kfyrcnACxTNMi3DA9Uwe31aALCBDLr8+nrqVB1xAgluYr6/N2FQ06iwCxKPvn/5Neu94zXU6DPk6AWGZ9kg1o0LkEiIUY7ZkfbT9jEiCW4OvnPu8xDzqLALEobX3G2cNcCBCzl49+3jIM+iABYt7Gf8LH32FIgFiIcY4/Y+5nTAIEZASIGRtw+jNlEnSSAAEZAWKuBj/+7DgEHSdAQEaAgIwAARkBYpZmMQDaMQY6QoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBYpa22+fdD6+/f96Nbb/D/Z7ZEyAgI0BARoCAjAAxV7MYAxkAHSdAQEaAmLHBD0GOPycJEJARIBZitEPQaPsZkwAxb+O/3Yy/w5AAMXsDToJMfz5IgFiUERo0wh7mQoBYgulBo33+p6s7/pwkQCzEaE/7aPsZkwCxHPkwyOjnXALEYhv0lRmaLqc+HydALM3Xz4PMfS4mQCzQVzZIfa4hQKyiQZ+RoYPbqs8FBIjFOijCDTP09lbqc5nvF14Hc7DrwmbzsP/NLhz3j0+X3fBtwqTnGgLE8m23z9MGHXTkZIyOnJvU50oCxCrsS3FQosum1LpzKwLE2l/KLricWxEg1mjakZMxEp3PI0Csnb6E/BkeyAgQkBEgICNAQMYQ+q8fv+7rLcC6OAEBGQECMt9eXv50qwOr5gQEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIC7yv/SDwYmef+2FwAAAABJRU5ErkJggg==', 'sha256': 'dae3d4e0f5e64df6960108a572974e0f1ab5e16c22f1d2c99fc1faac610cd0b2'}, 'data/binding-v1/images/binding-evaluation-0091-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFBElEQVR4nO3dTWobSxhAUeeRLOeBB4G3k2wk68hGspNABoa3lQwMnmQgUIQUS9aPdaurzxm71TXpS9VH4/7w/PzrAaDwT3JXAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkPna3HsXj4+eHFXh6+lkvAfbZAQEZAQIyjmB/fPrv+8N0Xn58qZcAr7IDAjICBGQECMgIEJAxhGbtTr4I5hWq9yNArNFZb5/u/rEY3ZYAsS5Xvvi+uVyGbkWAWHt3Xr7+e/zaT9/+f+3XlOhKAsQa63MyOq/98V6MHh8/a9A1BIiZXZmeQ5vLdzPkUHYNAWIt9bkyPSczpEEXECDmr88N03MkQxp0AS8iMqE71Oevv7+S/y11QwLEbO5Zn8O7aNBZHMGYyvb5v096/noccxZ7Ozsg5lHVZ2t7X/ugNxIgJjHaMz/aesYkQMzg/nOf15gHnUWAmEpbn3HWsBQCxOLlo59DhkFvJEAs2/hP+PgrDAkQkxhn+zPmesYkQEBGgFiwAac/u0yCThIgICNALNXg258Nm6DjBAjICBCQESAgI0As0iIGQBvGQEcIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0As0vbTo5uPkY5su0KfSz0kQEBGgICMAAEZAWKpFjEGMgA6ToCAjACxYINvgmx/ThIgICNATGK0TdBo6xmTALFs459uxl9hSIBYvAEnQaY/byRATGWEBo2whqUQIGawu9Fon//du9v+nCRATGK0p3209YxJgJhHPgwy+jnXx7OvgIE9Pf3cfIB004K7fTfVyesydkDM5v7zIPW5mAAxoXs2SH2u4QjG5Gex9zuO7aVNfS4gQKyiQbfN0OGuSn0uI0DMbNOFG2ZIem5LgFjdVmivIydjdGSKpD5XEiBWYVuKvRJdNqXWnVsRINZ+KLvgcm5FgFij3Y6cjJHovB8BYu30JeRFRCAjQEBGgICMAAEZQ+g/Xn58qZcA62IHBGQECMh8eH7+1d0dWDU7ICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEPFR+Awh8ButWlbKAAAAAAElFTkSuQmCC', 'sha256': '8eff61e98118163ec1339cf36e05ba0bb13e5dd7ad1be27d4ddf0c7c94f4c401'}, 'data/binding-v1/images/binding-evaluation-0091-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFA0lEQVR4nO3dPW7bShhAUSeIl/MAF2mzobRZR9q3obQpDLytpDCQ5hUCFEGKJevHusPhObUpTsOLmQ+E+eHl5dcDQOFjclcAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQ+dbcexdPT54cVeH7+WS8B9tkBARkBAjKOYH/8+Pf3w3S+fH2slwCvsgMCMgIEZAQIyAgQkDGEZu1OvgjmFar3I0Cs0Vlvn+7+sRjdlgCxLle++L65XIZuRYBYe3d+f/vn+LWP3/977deU6EoCxBrrczI6r/3xXoyenj5r0DUEiJldmZ5Dm8t3M+RQdg0BYi31uTI9JzOkQRcQIOavzw3TcyRDGnQBLyIyoTvU56+/v5L/LXVDAsRs7lmfw7to0FkcwZjK9vm/T3r+ehxzFns7OyDmUdVna3tf+6A3EiAmMdozP9p6xiRAzOD+c5/XmAedRYCYSlufcdawFALE4uWjn0OGQW8kQCzb+E/4+CsMCRCTGGf7M+Z6xiRAQEaAWLABpz+7TIJOEiAgI0As1eDbnw2boOMECMgIEJARICAjQCzSIgZAG8ZARwgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQCzS9tOjm4+Rjmy7Qp9LPSRAQEaAgIwAARkBYqkWMQYyADpOgICMALFgg2+CbH9OEiAgI0BMYrRN0GjrGZMAsWzjn27GX2FIgFi8ASdBpj9vJEBMZYQGjbCGpRAgZrC70Wif/9272/6cJEBMYrSnfbT1jEmAmEc+DDL6Odens6+AgT0//9x8gHTTgrt9N9XJ6zJ2QMzm/vMg9bmYADGhezZIfa7hCMbkZ7H3O47tpU19LiBArKJBt83Q4a5KfS4jQMxs04UbZkh6bkuAWN1WaK8jJ2N0ZIqkPlcSIFZhW4q9El02pdadWxEg1n4ou+BybkWAWKPdjpyMkei8HwFi7fQl5EVEICNAQEaAgIwAARlD6D++fH2slwDrYgcEZAQIyHx4efnV3R1YNTsgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgQ8VP4HELAFfUdPsP0AAAAASUVORK5CYII=', 'sha256': 'e34e5ffdb7228d5e8f8281f524dd3b2626bdabfa61378e9306eceacab8e75894'}, 'data/binding-v1/images/binding-evaluation-0092-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJElEQVR4nO3aQU4bSRiAURjFx4nkBcfIbXKO3GaOwcJSrsLCkjezcMZCbWODCP66u95bIWS6C+T++Kvgcb9/eQAo/JPcFUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZb92tYQG226frL9jtnu+1lhV63O9f6jXM+u21dB6Pe74r/LQ/ygQEf+230ekKSvROAgTX0nP4+f36125+/X7rgjJ0kwD98e9m87AiPw6HegkLTs/N6Lz14kmMZOgmAWJQ5+n5UHcuOl3hdYlk6AoBYkST+nw+PRPHC04ypEHnBIih6/PX03MlQxp0zj8iMpDt9ulu9bl4l8kCECBG8dXbrism99KgE1swhnD/wWfCduwiExDrl9fn4t235iABYihtfeazhvkQIFbuNGjM58k/rWQ7/BAkQKzZ/J/w7exX+KUEiNWaz9HPOYdBRwLE+s2tPnNe1Z0JEOs0w6Ofc4fhD4MECMgIECu0iPHnaPAhSICAjACxNgsaf45GHoIECMgIEJARIFZlcfuvwXdhAgRkBAjICBCQESDWY6EHQCMfAwkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQKzHbvd8/GDz6/fD0mz+X/PpuxiBAAEZAQIyAgRkBIhVWegx0GbIAyABAkoCBGQEiLVZ3C5sM+r+S4CAkgCxQgsagjYDjz8CBJQEiHVaxBC0GXv8ESCGMM8GbWa5qjsTIFbr9Vgxt6f99Xp2o44/AsTKzf/Z3s1+hV9KgFi5GR4GOfo5ESAGMocGzWEN8yFArN98DoMc/Ux8m34C1mi3e95un15X4PDz+z0XMAmf+hyZgBjF5Jm/5yikPm8RIAay2z3ffzs22Xapz2u2YAznbtsxg89NAsToDfqKDJ3PVupzkQAxqGMRzjP0mRJd3NNJzxUCxNDOMzTpyM0YXTlIkp6bBAguZ+gzB9XS804CBNNqXCzRh67AOwkQXOvIzRiJzmcIEFyjL1/KPyICGQECMgIEZAQIyAgQkPFXsD9+HA71EmA4JiAgI0BA5nG/f+nuDgzNBARkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEPlf8AevUhXJyDKyUAAAAASUVORK5CYII=', 'sha256': 'ba2d5d0139a9ba3ea0696f645a3b2779891a5283bf7aefc80c4bc8d0fe976814'}, 'data/binding-v1/images/binding-evaluation-0092-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHElEQVR4nO3cMW4aWxiAUTtK9kPhlg1lQdmQWxcsxgVSmlfwgtBAwJYD38zcc6ooIjPXFvP5v9coz/v9+xNA4VtyVwABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkvne3hgXYbF6uv2C3e3vUWlboeb9/r9cw67fX0nk8Hvmu8N3+LBMQ/LOfRscrKNEHCRBcS8/rr9/X/+3254+/XVCGbhKgj77PluXiU8EH0/OpN8Ppiyffdhm6SYAY1Hl6vv5D6HiF0xLJ0BUCxIgm9fnn8+/hgpMMadA5AWLo+tx16z3JkAad80FEBrLZvDysPhfvMlkAAsQo7r3tumJyLw06sgVjCI8ffCZsxy4yAbF+eX0u3n1jDhIghjKHT3vNYQ3zIUCs3HHQmM+Tf1zJZvghSIBYs/k/4ZvZr/CuBIjVms/RzzmHQQcCxPrNrT5zXtWDCRDrNMOjn3Ovwx8GCRCQESBWaBHjz8HgQ5AAARkBYm0WNP4cjDwECRCQESAgI0CsyuL2X4PvwgQIyAgQkBEgICNArMdCD4BGPgYSICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoBYj93u7fCH7c8fT0uz/bPm41cxAgECMgIEZAQIyAgQq7LQY6DtkAdAAgSUBAjICBBrs7hd2HbU/ZcAASUBYoUWNARtBx5/BAgoCRDrtIghaDv2+CNADGGeDdrOclUPJkCs1ulYMben/XQ9u1HHHwFi5eb/bO9mv8K7EiBWboaHQY5+jgSIgcyhQXNYw3wIEOs3n8MgRz8T36d/AWu0271tNi+nFXj99fuRC5iET30OTECMYvLMP3IUUp+/ESAGstu9PX47Ntl2qc8pWzCG87DtmMHnJgFi9AbdI0Pns5X6XCRADOpQhPMMfaVEF/d00nOFADG08wxNOnIzRlcOkqTnJgGCyxn6ykG19HyQAMG0GhdL9Kkr8EECBNc6cjNGovMVAgTX6Mtd+SAikBEgICNAQEaAgIwAARm/Bfuf/6cOHs8EBGQECMg87/fv3d2BoZmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEg4KnyH96UHw1yUQWtAAAAAElFTkSuQmCC', 'sha256': '140a41f06059ccd663bc35adbf3b5ae8c45ed205b2a5ce2e80532b832442e341'}, 'data/binding-v1/images/binding-evaluation-0092-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJUlEQVR4nO3cMU4jSRiAUVgN93FA6hvMSeZAc5K9ASmBD0NgiWQDz1qobWwQg7/urvcihKC7QO6PvwrE/X7/cgdQ+Ce5K4AAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyP7pbwwJsNo+XP2C3e77VWlbofr9/qdcw65fX0nk8bvmq8N3+LBMQ/LWfRscrKNEHCRBcSs/T79fLn7v99fDeBWXoKgH649+HMy+j5fr5euWx4UJ6rkbnvQ+exEiGrhIgBnWank9156zjFd6WSIYuECBGNKnP19MzcbjgJEMadEqAGLo+fz09FzKkQaf8ISID2Wweb1afs3eZLAABYhTfve26YHIvDTqyBWMItx98JmzHzjIBsX55fc7efWMOEiCG0tZnPmuYDwFi5Y6Dxnye/ONKNsMPQQLEms3/Cd/MfoXfSoBYrfkc/ZxyGHQgQKzf3Ooz51XdmACxTjM8+jn1NPxhkAABGQFihRYx/hwMPgQJEJARINZmQePPwchDkAABGQECMgLEqixu/zX4LkyAgIwAARkBAjICxHos9ABo5GMgAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZASI9djtng9vbH893C3N9v81H7+KEQgQkBEgICNAQEaAWJWFHgNthzwAEiCgJEBARoBYm8Xtwraj7r8ECCgJECu0oCFoO/D4I0BASYBYp0UMQduxxx8BYgjzbNB2lqu6MQFitd6OFXN72t+uZzfq+CNArNz8n+3d7Ff4rQSIlZvhYZCjnyMBYiBzaNAc1jAfAsT6zecwyNHPxI/pO2CNdrvnzebxbQWefr/ecgGT8KnPgQmIUUye+VuOQurzHgFiILvd8+23Y5Ntl/q8ZQvGcG62HTP4XCVAjN6g78jQ6WylPmcJEIM6FOE0Q18p0dk9nfRcIEAM7TRDk45cjdGFgyTpuUqA4HyGvnJQLT0fJEAwrcbZEn3qCnyQAMGljlyNkeh8hQDBJfryrfwhIpARICAjQEBGgICMAAEZvwX74+frTf87DGACAkoCBGTu9/uX7u7A0ExAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQcFf5Dz7YIKNn7847AAAAAElFTkSuQmCC', 'sha256': 'dfbf3f15c7743ef11e5661ed5b21a8d009ae5d5aa988529ecbb4c26fc237f398'}, 'data/binding-v1/images/binding-evaluation-0093-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFL0lEQVR4nO3dQU4jVxRA0SZq9uNBS9kE415kxmyipQxYDANLTKIIBTk2FG7AvlX/nzME7HoS+Pq5XMg3+/3jN4DCH8lRAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABme/doWHtdrsfyz/w8PD3tWYZ081+/1jPAFuKzlvE6AMECD7VnVNKdD4BYnYL6bm/vV2+7d3T01vfkqFzCBDzejU970bnt2IkQ8sEiEmd1ufD6VnOkAYtECBmdFSfL0nPQoY06C0CxFwunZ5DMvQuFyIykWvW5/T+v/CNtmFMtAEN9uv3dPqZP4BLp2dhFfKLO2QDYgphfY6OONgT4ScJEONr63N6XA2a+n/Bbv/869tmPf36WY+wYVV9Xo6+cOHinGxADO5l3WjrczSDJeiZADGyNT/Odyue7WoEiGGt4dTPKSeDDgkQ41tPfdY5T0iAGNNWlovdRua8EAFicOtcN9Y51fUJEAPa1lqx29S0X0uAGNmaF437Fc92NQIEZASI0azqysNl99NflyhAQEaAgIwAARkBYigbOgH0bPLTQAIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEEN5+eT1rXwE4N1/c875mfECBGQECMgIEJARIEazodNAd3OfABIgoCRAjGzNS9Ddime7GgFiQNt6RfOwqWm/lgAxuHUuGuuc6voEiDFtZa142MicFyJAjG9t68ba5gkJEMM6XC7W85g/nORh7vVHgBjcmh/hDyue7WoEiMGt6rpEVx4eESAm0jZoDQVcGwFifGs4GeTUz6sEiCm0DVKftwgQkzboOhk6OpD6HBEgJnL0+L90g47uX31OfX/lazCu5wrsdj8OG3F/e/u1R5GeM9mAmNHpKvRV29DpXanPAgFiUqdd+GSGXr25+izzEox5Hb0ce3YYkXdfmi0ES3rOIUDM7tUMPfvYQiQ95xMg+F81Xi3Rb90D5xMgeLMj78ZIdD5JgOBN+nJp3gUDMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMjM+L9gT79+1iMA/7IBARkBAjI3+/1jd3RgajYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIOBb5R/OjChKMnifqwAAAABJRU5ErkJggg==', 'sha256': '4bcd637858874718b978fee5ff39570409059ec0a1cfd36f08f180cc99fe96eb'}, 'data/binding-v1/images/binding-evaluation-0093-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFLUlEQVR4nO3dMVIjRxiA0cW1ex8Cp9xgT+ID+SS+waYb6DAEqiJxQJnCEhq0C9LX6n4vFWi6SszHr2bE3O33j18ACn8kRwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGS+doeG0d3f/7n9Bbvdz2utZU53+/1jvQa4peicIka/QYDgQ905pkTnEyBWt5GeH38/bX/vw1/fTj0kQ+cQINb1Znrejc4vxUiGtgkQizquz2+nZztDGrRBgFjRQX0+JT0bGdKgUwSItVw6Pa/J0LtciMhCrlmf4+f/xD+0TWOhCWiyl9+v04/8AFw6PRujkBfuNRMQSwjrc3DEyX4RfpAAMb+2PsfH1aClPwv2z7eTF4+N7/tTc/7MoarPy9E3LlxckwmIyb2MG219DtZgCHomQMxs5PP8fuC1XY0AMa0Rtn6O2Qx6TYCY3zj1GXM9IQFiTrcyXNzfyDovRICY3Jjjxpiruj4BYkK3NVbc39RqP5cAMbORB40fA6/tagQIyAgQsxnqysNtP5a/LlGAgIwAARkBAjICxFRuaAPo2eLbQAIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEFN5ufP6rdwC8OG/da55z3gBAjICBGQECMgIELO5oW2gh7U3gAQIKAkQMxt5CHoYeG1XI0BM6Lbe0exuarWfS4CY3JiDxpiruj4BYk63MlbsbmSdFyJAzG+0cWO09YQEiGm9Hi7GOedfr2S39vgjQExu5DN8N/DarkaAmNxQ1yW68vCAALGQtkEjFHA0AsT8RtgMsvXzJgFiCW2D1OeUrycfgbnsdj9f7r/+XIQr3D/+IHbqc8AExEIOzv9Lj0Lq8y4TEGt5rsClRyHpOZMAsfrbsc/N0PFUpT4bBIhFHTTo4xl68w2d+mwTINZ18HbsuCPvxmhjF0l6ziFArO7NDH1kl1p6zidA8L9qvFmiX3oGzidAcLIj78ZIdD5IgOAkfbk0FyICGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGRW/CzY96eL/yty4BwmICAjQEDmbr9/7I4OLM0EBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgR8qfwLOpwTyEo0c6kAAAAASUVORK5CYII=', 'sha256': 'ba553775fd0def36cb91abf6ae7b60f31379a979082bc3d5e56b3cc97c2f6ad1'}, 'data/binding-v1/images/binding-evaluation-0093-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFL0lEQVR4nO3dTU4jVxSAUYi698MgUkZsoxeRBWUR2QajlnrgxTCwxCSKUBCxceFusL+q986Z8lNXov1x/bpQ3e73jzcAhd+SqwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjJfukvD2t3d/b78Cbvdj2vNMqbb/f6xngG2FJ1TxOgXCBB8qDvHlOh8AsTsFtLz8NfT8tfe//n11Idk6BwCxLzeTM+70fmpGMnQMgFiUsf1+eX0LGdIgxYIEDM6qM+npGchQxp0igAxl0un5zUZepcbEZnINetz/P0/8T/ahjHRBjTYj9+v04/8A7h0ehZWIT+412xATCGsz8EVB/tF+EECxPja+hxfV4Om/luwr3/8fbNZT9+/1SNsWFWfl6sv3Lg4JxsQg3tZN9r6HMxgCXomQIxsza/zuxXPdjUCxLDWcPRzzGHQawLE+NZTn3XOExIgxrSV5eJuI3NeiAAxuHWuG+uc6voEiAFta62429S0n0uAGNmaF42HFc92NQIEZASI0azqzsNlD9PflyhAQEaAgIwAARkBYigbOgB6NvkxkAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICxFBenry+lUcA3v8355zPjBcgICNAQEaAgIwAMZoNHQPdz30AJEBASYAY2ZqXoPsVz3Y1AsSAtvWOZrepaT+XADG4dS4a65zq+gSIMW1lrdhtZM4LESDGt7Z1Y23zhASIYb1eLtbzmn89yW7u9UeAGNyaX+G7Fc92NQLE4FZ1X6I7Dw8IEBNpG7SGAq6NADG+NRwGOfp5kwAxhbZB6nPKl5MfgbHsdj9enr/+XIQrPD/+IHbqc8AGxEQOXv+XXoXU5102IObyXIFLr0LScyYBYva3Y5+boeOtSn0WCBCTOmjQxzP05hs69VkmQMzr4O3YcUfejdHCKZL0nEOAmN2bGfrIKbX0nE+A4H/VeLNEP/UdOJ8AwcmOvBsj0fkgAYKT9OXS3IgIZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJCZ8W/Bnr5/q0cA/mUDAjICBGRu9/vH7urA1GxAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMBN5R+q9BOgAF9xhwAAAABJRU5ErkJggg==', 'sha256': 'bbd38be4f77ce689ae3f95f1c1d723cfb7d3a065ed80f0e4494f2d91664787b3'}, 'data/binding-v1/images/binding-evaluation-0094-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFc0lEQVR4nO3aQWobSRiAUWWIjjOgRSA3yUXmHHOR3CQwC8FcJQuDNrMwI4xsy4oi+1NVvbcKwVH/bbu/VHX3p4eHnxuAwh/JUQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZD5vFrPbfdlMYb//px5h6d8T3/+bWC5AcJP/mU6+Xo+uI0As7VYr4uPnKNEvWTdAh7/+3Axo+/e/9QhLpGf79fv5f3748e3Mx8rQhdYNEMt6LT1vRue1L34eIxm6kACxenp+qTsvOn7CSYlk6E0CxKL1+f3uXFii3e6LBr3Ge0As4QPqc+bzp3n54+asgJjcB6fn5EDHpZDt2IusgJhZVZ/XjmgpdEKAmNbTq3379fvH1+fFQ2vQUwLEEvVJZzmdQYOOBIgJ3Vt9HmnQcwLEbO6zPo806IQAMa17q889T1URIKZyXFbc83V+nG23/CJIgJjHiNfzbsCZb0iAmMQ93/p5zs2gRwLEbO6/PmPN+a4EiBmMvojYDT7/1QSIqYy1rNgONe17ECCGN8STr9ds134iJkBARoAY20wLh91E53IhAWISI+6/Rp/89wkQkBEgBjbfnmU33RmdJ0DMYPRdzHbw+a8mQEBGgICMAAEZAWJUQ78A/dx2yVeiBQjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgRrXf//P4h8OPb5vxHf4/i+N5rUCAgIwAARkBAjICxAxGvw10GHz+qwkQA5vvfu1+ujM6T4CAjAAxiXF3MYdhJ/99AsTYZtqz7Cc6lwsJEJARIIY39CvRhyVfgD4SIKYyVoMOQ037HgSIGYy+fNgPPv/VBIjZjLKsOAwy57sSICbxdBFx/9f20wn3qy5/BIipjHgl7wec+YYEiKkM8URs8SdfTwkQ07rPBt3nVBUBYjb3fDPIrZ8TAsSE7rNB6vOcADGne2uQ+rxIgFilQVWGTg6tPk99enj4uVnJbvdlMwW/x9f9xLdfv3/k0U+q56d24vPpX8BcHq/5Y4Yei/ABGZKeS9iCsYST6/+9t2Pqc6HltmCs7MUN+A1XQy92TX3OECCW89p9wKtL9Np6SnreJEAs6vzjiDdjdH4TJz0XEiCWdvOnotLzSwQIblAi3bmOAME1PVKcmxAgIOM9ICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZvKf8XQKjsgjpuMAAAAAElFTkSuQmCC', 'sha256': '4605673d624d001e03bcac946b7307f9741f9aa14449211d77f51be05723e858'}, 'data/binding-v1/images/binding-evaluation-0094-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFc0lEQVR4nO3aQW4TWRRAUdOC/XiAxMjb6EWwIBbR28ioJQZeDINITHoQtRU5tuMYJ7f+r3NGCAX7Fbgur8r16fHx1wag8FfyrgACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMp83K7Pdft1MYb//WY+w6s+Jv/+7WF2A4C7/Mx39vB7dRoBYtXttxIfXUaI3WW+Avnz7ZzOg3//+XY+wivQ8/Ph9+Y/vvn+58LIydKX1BojVOpeeV6Nz7odfxkiGriRArD09b+rOSYdXOCqRDL1KgFhpff68O1eWaLv9qkHneA6IVfiA+lx4/Wke/rg7GxCT++D0HL3RYRVyOXaSDYiZVfU5945WoSM2IKb1/Gz/+PRcWIXsQQc2IOa0kPqcnMEedCBATGhp9XmiQS8JELNZZn2eaNARAWJaS6vPkqeqCBBTOawVSz7PD7NtV78ECRDzGPF83g448x0JEJNY8q2fl9wMeiJAzGb59RlrznclQMxg9CViO/j8NxMgpjLWWvEw1LTvQYAY3hDffJ3zsO5vxAQIyAgQY5tpcdhOdCxXEiAmMeL11+iT/zkBAjICxMDmu2bZTndElwkQMxj9KuZh8PlvJkBARoCAjAABGQFiVEM/AP3SwyofiRYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgBjVfv/z6Re7718249v9fxSH41oDAQIyAgRkBAjICBAzGP020G7w+W8mQAxsvvu1++mO6DIBAjICxCTGvYrZDTv5nxMgxjbTNct+omO5kgABGQFieEM/Er1b5QPQBwLEVMZq0G6oad+DADGD0deH/eDz30yAmM0oa8VukDnflQAxiedLxPLP7ecT7te6/ggQUxnxTN4POPMdCRBTGeIbsZV/8/WcADGtZTZomVNVBIjZLPlmkFs/RwSICS2zQerzkgAxp6U1SH1O+nz6t2F8+/3P7fbr8/P/4cfvjx/jKH/q89ynx8dfmzU5fCJH53N827/4BzdIfS6zATG5p3P+41ch6bmGe0CswtH5/953hdTnSqu7BGPNTl6A33EbOtk19blAgFidc/cBby7RuX1Kel4lQKzU5a8jXo3R5Ys46bmSALFqd/9WVHreRIDgDiXSndsIENzSI8W5CwECMp4DAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKn8B1AdMWX5dNWYAAAAAElFTkSuQmCC', 'sha256': 'b6bcfd5509294ba8bc02163c5763640dbc625646b84323ccf9cbae150c8a7c10'}, 'data/binding-v1/images/binding-evaluation-0094-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFcklEQVR4nO3aMW4TaRiAYbOC46zkgtYXot1zbMuF0lJY4ioUkWi2iNaKHNtxjJN3/n+ep0Io2N+A5+Wb8Xx6fPy1ASj8lbwrgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM583KbLdfN1PY73/UI6z6c+Lv/y5WFyC4y/9MRz+vR7cRIFbtXhvx4XWU6E3WG6Df//y9GdCXf3/WI6wiPQ/ff1/+47tvXy68rAxdab0BYrXOpefV6Jz74ZcxkqErCRBrT8+bunPS4RWOSiRDrxIgVlqfP+/OlSXabr9q0DmeA2IVPqA+F15/moc/7s4GxOQ+OD1Hb3RYhVyOnWQDYmZVfc69o1XoiA2IaT0/2z8+PRdWIXvQgQ2IOS2kPidnsAcdCBATWlp9nmjQSwLEbJZZnycadESAmNbS6rPkqSoCxFQOa8WSz/PDbNvVL0ECxDxGPJ+3A858RwLEJJZ86+clN4OeCBCzWX59xprzXQkQMxh9idgOPv/NBIipjLVWPAw17XsQIIY3xDdf5zys+xsxAQIyAsTYZlocthMdy5UEiEmMeP01+uR/ToCAjAAxsPmuWbbTHdFlAsQMRr+KeRh8/psJEJARICAjQEBGgBjV0A9Av/SwykeiBQjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgRrXf/3j6xe7bl834dv8fxeG41kCAgIwAARkBAjICxAxGvw20G3z+mwkQA5vvfu1+uiO6TICAjAAxiXGvYnbDTv7nBIixzXTNsp/oWK4kQEBGgBje0I9E71b5APSBADGVsRq0G2ra9yBAzGD09WE/+Pw3EyBmM8pasRtkznclQEzi+RKx/HP7+YT7ta4/AsRURjyT9wPOfEcCxFSG+EZs5d98PSdATGuZDVrmVBUBYjZLvhnk1s8RAWJCy2yQ+rwkQMxpaQ1Sn5M+n/5tGN9+/2O7/fr8/H/4/vvjxzjKn/o89+nx8ddmTQ6fyNH5HN/2L/7BDVKfy2xATO7pnP/4VUh6ruEeEKtwdP6/910h9bnS6i7BWLOTF+B33IZOdk19LhAgVufcfcCbS3Run5KeVwkQK3X564hXY3T5Ik56riRArNrdvxWVnjcRILhDiXTnNgIEt/RIce5CgICM54CAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRsKv8B0YAxZWG9p7UAAAAASUVORK5CYII=', 'sha256': '8808378edfb59edb74c2b1a3082993027502ced7ce60770e87418e56fcba1f0d'}, 'data/binding-v1/images/binding-evaluation-0095-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUUlEQVR4nO3dMU4kRxiAUdZajrPSBKS+kNM9h1NfyOkGI/kqBEgkDpBHLfA2wzA7X1f1eymgrtZMf/qrGMGXp6fHO4DCb8lVAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD52l0atuJweDjn247HH79+Lfvy5enpsV4DbDc6PyNGVyFA7Msnu/OWEn2GALEX6+l5/v5t/cfv//xn5asydBkBYqfpebc4F/RIhj5KgJjZr0jPkgx9kgCxl/pcsTvvlkiDziRATOiW6VmSoY/yQURmU9Xn7bWu/hu3+ZiAmMrymb9lelZGIXPQChMQ89hIfV5d3Ry0QoCYxHbq83YNGvQze9yCDf1uMM8PUZ8Te7F1JiCGt9n6mIPeJUCMbcv1eaFBK3b95zj+/uv5bhC//3FfL2GLtl+fF8/fv532YofDg73YiQmIGWy5PqOsMCFADD/+jPJsn9ZpI3YiQAxp9Gd49PVfiwAxtlHGnxFXewMCxHiG23wt2YgtCRCQESAGM/T488IQdCJAQEaAGMl8I8Nhujv6EAFiSOPuv+ZY/7UIEJARIIYxwfHz0rOjaAECQgIEZAQIyAgQY5j7oOQw9d2tECAGM8cJ9Hz3chkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBIjBnP7J+gTuJ7qXywgQYzgef9zN6zj13a0QICAjQEBGgICMADHeQckcZ7f3/93Fbg+ABAgoCRBDGn0IGn391yJAjGS+3cpxujv6EAECMgLEYCY4inb8fCJAQEaAGM/QQ5DxZ0mAGNtYDRprtTcgQAxp9PFh9PVfiwAxquE2YjZfbwkQM9h+g7a/woQAMbDlKLHlJ3y5NuPPkgAxtu03SH1WCBDD23KD1GedADGDbTZIfd4lQExiaw1Sn3MIEPPYToPU50xfz/1GGMHx+ONweFhW4Pn7t1su4FX41GedCYjZvHrmbzkKqc9HmYCY0MuTf8tRSHouI0DsYju2bMQVS/S/45X6nE+A2NEodMWBSHquQoDYdYZO3u3R+lmS9Fzmy9PT493OvHojjsUbfWuvvlfkM0xA7MupF58ske5chQCxU8uCnBkj0bm6PW7BgI3wQUQgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBBwV/kXNZ1NJjsqSZ0AAAAASUVORK5CYII=', 'sha256': '13beb98398b4615667d1fba3cafdbe7e7b0df9c1e0ed33e34911f48d898d4b5d'}, 'data/binding-v1/images/binding-evaluation-0095-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUUlEQVR4nO3dMU4jSRiAUVjNHGclB6RzoU33HJtyIdIJLM1VJkAi2QCt1YKdxoDHX1f1eymgrpbdn/4qLLh9fPx5A1D4I7kqgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjJfukvDVhwOd+d82/H4/fevZV9uHx9/1muA7UbnV8ToIgSIfflkd15Tos8QIPZiPT0P90/rP/7tr68rX5WhjxEgdpqeN4vzgR7J0HsJEDP7HelZkqFPEiD2Up8LdufNEmnQmQSICV0zPUsy9F4+iMhsqvq8vtbFf+M2HxMQU1k+89dMz8ooZA5aYQJiHhupz4urm4NWCBCT2E59Xq9Bg35lj1uwod8N5vkh6nNiL7bOBMTwNlsfc9CbBIixbbk+zzRoxa7/HMfT33/eDOLrPz/qJWzR9uvz7OH+6bQXOxzu7MVOTEDMYMv1GWWFCQFi+PFnlGf7tE4bsRMBYkijP8Ojr/9SBIixjTL+jLjaKxAgxjPc5mvJRmxJgICMADGYocefZ4agEwECMgLESOYbGQ7T3dG7CBBDGnf/Ncf6L0WAgIwAMYwJjp+XHhxFCxAQEiAgI0BARoAYw9wHJYep726FADGYOU6g57uXjxEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgBjM6Z+sT+DbRPfyMQLEGI7H7zfzOk59dysECMgIEJARICAjQIx3UDLH2e23/+5itwdAAgSUBIghjT4Ejb7+SxEgRjLfbuU43R29iwABGQFiMBMcRTt+PhEgICNAjGfoIcj4syRAjG2sBo212isQIIY0+vgw+vovRYAY1XAbMZuv1wSIGWy/QdtfYUKAGNhylNjyE75cm/FnSYAY2/YbpD4rBIjhbblB6rNOgJjBNhukPm8SICaxtQapzzkEiHlsp0Hqc6Yv534jjOB4/H443C0r8HD/dM0FvAif+qwzATGbF8/8NUch9XkvExATen7yrzkKSc/HCBC72I4tG3HBEv3veKU+5xMgdjQKXXAgkp6LECB2naGTN3u0fpYkPR9z+/j482ZnXrwRx+KNvrVX3yvyGSYg9uXUi0+WSHcuQoDYqWVBzoyR6FzcHrdgwEb4ICKQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAi4qfwLkUlHXgT4hEkAAAAASUVORK5CYII=', 'sha256': 'e25c89db2a84e929d90972fd7296b302419eeb34d8de80cc35945d82511a3af7'}, 'data/binding-v1/images/binding-evaluation-0095-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFR0lEQVR4nO3dMU4jSRiAUWY1ex8HpHOhPdBciHQCH4bAEskGaK0W7DTGePx1Vb+XYuRq2f3pr8LAt9Pp+QGg8FfyrAACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfO+eGrbicHi85GHH468/v5Z9+XY6PddrgO1G53fE6CYEiH35YnfeU6KvECD2Yj09Tz9f1r/9xz9/r3xVhq4jQOw0PR8W54oeydBnCRAz+xPpWZKhLxIg9lKfG3bnwxJp0IUEiAndMz1LMvRZPojIbKr6vH+um//EbT4mIKayvOfvmZ6VUcgctMIExDw2Up83z24OWiFATGI79Xm/Bg36nT1uwYZ+N5jnh6jPmb3YOhMQw9tsfcxBHxIgxrbl+rzSoBW7/nMc23y/XvGLSLu1/fq8evr5cn4FD4dHe7EzExAz2HJ9RllhQoAYfvwZ5d4+r9NG7EyAGNLo9/Do678VAWJso4w/I672DgSI8Qy3+VqyEVsSICAjQAxm6PHnlSHoTICAjAAxkvlGhsN0V/QpAsSQxt1/zbH+WxEgICNADGOC4+elJ0fRAgSEBAjICBCQESDGMPdByWHqq1shQAxmjhPo+a7lOgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEIM5/5P1CfyY6FquI0CM4Xj89TCv49RXt0KAgIwAARkBAjICxHgHJXOc3f747yp2ewAkQEBJgBjS6EPQ6Ou/FQFiJPPtVo7TXdGnCBCQESAGM8FRtOPnMwECMgLEeIYegow/SwLE2MZq0FirvQMBYkijjw+jr/9WBIhRDbcRs/l6T4CYwfYbtP0VJgSIgS1HiS3f4cu1GX+WBIixbb9B6rNCgBjelhukPusEiBlss0Hq8yEBYhJba5D6XEKAmMd2GqQ+F/p+6QNhBMfjr8PhcVmBp58v91zAm/CpzzoTELN5c8/fcxRSn88yATGh1zv/nqOQ9FxHgNjFdmzZiBuW6H/HK/W5nACxo1HohgOR9NyEALHrDJ192KP1syTpuc630+n5YWfevBHH4o2+tVffK/IVJiD25dyLL5ZId25CgNipZUEujJHo3Nwet2DARvggIpARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECHio/AsM60deLyhiEQAAAABJRU5ErkJggg==', 'sha256': 'f120d387c310e94f155ae68676a600a3f14f41e8ffc4a499c5e40f1b6ec33e82'}, 'data/binding-v1/images/binding-evaluation-0096-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFfklEQVR4nO3aPVLcWBhA0faU2Y8Cp96BVuIFzUrYwaQOejEOqCKZmupxVw9DAwbD1dM7JyIA6am6dfn08+nu7scBoPBHslcAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnc7RoGtixfnv6F4/H7R61lYJ/u7n4c9vslGJ0v8W6+bD7KR5mA4CP+yZ23o0SXBAhek57bm5un/3y9v39iszK0qwD99efjH/agvn575stNkp5no3Ptl/8fIxnaVYDg/dLzS9151HkLD0q0TJ8hAYLH6/P27rywRMvyZdoGeQ8IPqg+T2x/2fvz3GtMQMzug9PzYEfrz1FozssxExBTq+pzbY/LZKOQADGvy7P99ubm4+vz6K6napAAMakH9UnX8o85GyRAzGhr9Zm2QQLEdLZZnzkbJEDMa2v12fKq3okAMZfzWLHl8/z259p2PwQJEBMZ8XxeBlzzywkQs9jyrZ9pbwYJENPZfn3GWudbCBBTGH2IWAZf/zUCxFzGGituh1rtKwgQ+zfEk685n4gJEJARIHZuT4PDsqNjOREgZjHi9dfoK3+WAAEZAWLP9nfNsuzriASIKYx+FXM7+PqvESAgI0BARoCAjACxW0O/AD3JK9ECBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBC7dTx+P/2w3t8fxrf+PIrzce2AAAEZAQIyAgRkBIgpjH4baB18/dcIEHu2p/u1uzwiAQIyAsQsxr2KWYdd+bMEiJ3b0zXLcUfHciJAQEaA2L+hX4le9/gC9JkAMZexGrQOtdpXECCmMPr4cBx8/dcIENMZZaxYB1nnWwgQs7gcIrZ/bq8XK9zr+CNAzGXEM/k44JpfToCYyxBPxNZdP/m6JEDMa5sNWje5qnciQExnyzeD1jlu/ZwJEDPaZoPWyeojQMxraw1a56uPADG1Bw2qMrT+d9fz1EeAmN2Ds/3jG7T+d49T1edwOHyuFwCx0zm/LF8ui3B7c/Pe+13nTs+JCQiCUUh9TkxA8K/j8ft5DrpsxG+chh7t2nHW+ggQPHU59rtKdG2eOk6cnhMBghdl6EFHno3R0xdx0nMiQPBrGXrjTSLpuSRA8KJeXCvRr26HSwIErynIsz1SnIkC9PXbu7+1AZf05bfwHhCQESAg8+nu7ke3d2BqJiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIOlb8BahkynbTFxfAAAAAASUVORK5CYII=', 'sha256': '50ffc707244f00e32b7a41b429e2e8ed9847b96f45c9d0486ac5c29d739de49f'}, 'data/binding-v1/images/binding-evaluation-0096-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFdklEQVR4nO3asXHbWBRAUXrH6oeBU3TgSlyQK9kOlDpgMQ40o2QDrjRcLUnJks2Lj39O5EAiH0bE9QOITw8PP3cAhb+SdwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmc/dW8PA9vsv13/gcPhxq1kG9unh4eduux+C0fkQb+bD5k95lg0IbvGf3PPrKNEpAYL3pOf+++P1X1++3V15WRnaVID+vjv/xx7U18dXPtwk6Xk1Opd++P8xkqFNBQj+XHp+qTtnPb/CixLtp8+QAMH5+ny8O28s0X7/ZdoGeQ4IblSfK6+/3/r3uZfYgJjdjdPz4o2Wp1VozssxGxBTq+pz6R33k61CNiDmdXq23z49V1ahefYgGxCTWkl9zs4wzx4kQMxobfWZtkECxHTWWZ85GyRAzGtt9VnzVH+IADGX57Vizef5/dNsm1+CBIiJjHg+7wec+e0EiFms+dbPtDeDBIjprL8+Y835EQLEFEZfIvaDz3+JADGXsdaK+6GmfQcBYvuG+OZrzm/EBAjICBAbt6XFYb+hYzkSIGYx4vXX6JO/SoCAjACxZdu7Ztlv64gEiCmMfhVzP/j8lwgQkBEgICNAQEaA2KyhH4Ce5JFoAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZASIzTocfhz/sXy7241veTqK5+PaAAECMgIEZAQIyAgQUxj9NtAy+PyXCBBbtqX7tZs8IgECMgLELMa9ilmGnfxVAsTGbema5bChYzkSICAjQGzf0I9EL1t8APqZADGXsRq0DDXtOwgQUxh9fTgMPv8lAsR0RlkrlkHm/AgBYhanS8T6z+3lZMKtrj8CxFxGPJMPA878dgLEXIb4RmzZ9DdfpwSIea2zQcsqp/pDBIjprPlm0DLHrZ9nAsSM1tmgZbL6CBDzWluDlvnqs9vtPtcDQOZw+LHffzk9/++/P95+jOW/+ZunPjYgZvfibL/9KrRMXB8bEPx7zt9+FVrmTs+RDQiCVUh9jmxAcOaW0GkjfuM2dLZrh1nrI0Bw7XLsd5Xo0j51mDg9RwIEb8rQi468GqPrF3HScyRA8GsZ+uBNIuk5JUDwpl5cKtGvvg6nBAjeU5BXe6Q4EwXo62PwACsz05ffwnNAQEaAgMynh4ef3bsDU7MBARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKv8A0GqMqHtI0mRAAAAAElFTkSuQmCC', 'sha256': 'f37fac4cd0e84650d08d317f5cecef5bafa71b2a86f32d62c6a8ebc39ca7cba1'}, 'data/binding-v1/images/binding-evaluation-0096-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFcElEQVR4nO3aTU7cWBhA0Uor7MeDLMPLrWVkUIvJAIlJK6pOqUJTQCBw/fzOGWVA7GdhXz7/fLm//3EAKPyT7BVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZL52u4aBLcu353/gdPr+WWsZ2Jf7+x+H/Z4Eo3MS7+Zk86t8kgkIPuOP3GU7SnRNgOAt6Tne3T3/39eHh2c2K0O7CtCLZ8NYbp27tOn5o9Ps+of//wuVoV0FCD4uPe//83bZwqMSLdNnSIDg6fp8xFj9ZImW5du0DfIdEHxSfZ7Z/rL397m3mICY3Sen59GO1l+j0Jy3YyYgplbV59Yel8lGIQFiXtdX+/HurnqXevx911M1SICY1KP6pGv5ac4GCRAz2lp9pm2QADGdbdZnzgYJEPPaWn22vKoPIkDM5TJWbPk6P/5a2+6HIAFiIiNez8uAa349AWIWW370M+3DIAFiOtuvz1jrfA8BYgqjDxHL4Ou/RYCYy1hjxXGo1b6BALF/Q7z5mvONmAABGQFi5/Y0OCw7OpYzAWIWI95/jb7yFwkQkBEg9mx/9yzLvo5IgJjC6Hcxx8HXf4sAARkBAjICBGQEiN0a+gPoST6JFiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaA2K3T6fv5H+vDw2F866+juBzXDggQkBEgICNAQEaAmMLoj4HWwdd/iwCxZ3t6XrvLIxIgICNAzGLcu5h12JW/SIDYuT3ds5x2dCxnAgRkBIj9G/qT6HWPH0BfCBBzGatB61CrfQMBYgqjjw+nwdd/iwAxnVHGinWQdb6HADGL6yFi+9f2erXCvY4/AsRcRrySTwOu+fUEiLkM8UZs3fWbr2sCxLy22aB1k6v6IALEdLb8MGid49HPhQAxo202aJ2sPgLEvLbWoHW++ggQU3vUoCpD6++7nqc+AsTsHl3tn9+g9fc9TlWfw+HwtV4AxM7X/LJ8uy7C8e7uo/e7zp2eMxMQBKOQ+pyZgOA/p9P3yxx03Yi/OA092bXTrPURIHjuduxvlejWPHWaOD1nAgSvytCjjrwYo+dv4qTnTIDgzzL0zodE0nNNgOBVvbhVoj/dDtcECN5SkBd7pDgTBWgLn9IzFX35K3wHBGQECMh8ub//0e0dmJoJCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgEPlX+j8NDNVl08pAAAAAElFTkSuQmCC', 'sha256': 'f018f48eb320cab8b1d01ff39fc825226970c9cd9c537e23d9759c2944faec4e'}, 'data/binding-v1/images/binding-evaluation-0097-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFXklEQVR4nO3aO27bWhRAUechmY+LAJlJBpmZBEjhwaQwkOYVBgRDtmTZ+ux76LWqFIp4CZHbh58vj49/7wAK/yVbBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnabRpi9/ffj3/g4eHPrdbySX15fPxbrwFWKc5xenRxswN05vG0Jkf5+seJ3+hSXILxedPz7cev4//93++fR75Whs4nQHyu9LwZnUMffhkjGTrfRgL0rqNqTYf+2HKR9Jx/hOy+Ye+XkqFzbCRA8Gp9rvGX6dUS3d9/16AP8B4Q23GD+hz5/k0+Erk2ExBbcOP07G1oNwq5HHsvExDjVfU5tEWj0OkEiNmen+3ffvyqHkfsbVqDTiRAbKc+6Vr216BBpxAgplqtPk806F0EiJHWrM8TDTqdADHbavVZeVULEiDm2Y0VK5/nu7UZgo4QIIaZeD5PXPNtCBCTrHzr5yU3g94kQIy0fn1mrbMiQIwxfYiYvv5rECDmmTVWzFrtjQkQM4x48nWIJ2KHCBCQESAG2NLgsKV9OZ8AMcnE66/pK78qAQIyAsTqtnfNsr09+jABYozpVzHT138NAgRkBAjICBCQESCWNvoF6Je8Er1HgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQFiaQ8Pf57+8e/3z7v5dnux269PToCAjAABGQECMgLEGNNvA01f/zUIEKvb3v3a7e3RhwkQkBEgJpl7FTN35VclQAywpWuWLe3L+QQIyAgQM4x+JdoL0IcIEPPMatCs1d6YADHG9PFh+vqvQYAYacpYMWWdFQFi6hCx/rn9fIXGn1cJEMNMPJMnrvk2BIh5RjwR8+TrFALEbGs2aM1VLUiAGGnlm0Fu/ZxOgJhqzQapz7sIEIOt1iD1eS8BYlMNqjK0t2n1OZEAMd7e2X77Bu1tUX1O9/Udn4VVPZ3z9/ffnxfh249f196u9JzJBMR23HgUUp/zmYDYlIeHP7s56HkjLjgNvdo19fkYAWLjl2OXKtGheUp6ziFAfKIM7XXkzRgdv4iTnvN9eXz8ezfWy8NrAxzW6x8nfqNLMQGxfbtenFki3bk4AeIT2SvImz1SnGubfQkGjOY9ICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABd5X/Aav0MwNH4Y0MAAAAAElFTkSuQmCC', 'sha256': '8397a39e084f193771de18bae38353b38391ae01b27b6267dec458edbbc441dd'}, 'data/binding-v1/images/binding-evaluation-0097-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFW0lEQVR4nO3aS04bWxRA0eQpzIdGhpHhZhhpMJg0kOi8BpKFHGxs/Nn3FGu1aBjXKVG1ufX5/vz89xtA4b9kqwACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjI/uk1D7PHx5/EPPD39udcsX9T35+e/9QywSnGO06Ormx2gC4+nNTnK1z9O/I2uxSUYXzc9vx8ejv/6r5eXI18rQ5cTIL5Wej6MzqEP/xsjGbrcRgJ01lG1pkP/bLlKei4/QnbfsPeXkqFLbCRA8G59bvGf6d0SPT7+1KBP8B4Q23GH+hz5/k0+Erk1KyC24M7p2dvQbinkcuxcVkCMV9Xn0BYthU4nQMz29mz//fBQPY7Y27QGnUiA2E590ln2Z9CgUwgQU61Wn1cadBYBYqQ16/NKg04nQMy2Wn1WnmpBAsQ8u2XFyuf5bjaLoCMEiGEmns8TZ74PAWKSlW/9/MvNoA8JECOtX59Zc1YEiDGmLyKmz38LAsQ8s5YVs6a9MwFihhFPvg7xROwQAQIyAsQAW1o4bGlfLidATDLx+mv65DclQEBGgFjd9q5ZtrdHnyZAjDH9Kmb6/LcgQEBGgICMAAEZAWJpo1+A/pdXovcIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0As7enpz+sPv15evs2324vdfn1xAgRkBAjICBCQESDGmH4baPr8tyBArG5792u3t0efJkBARoCYZO5VzNzJb0qAGGBL1yxb2pfLCRCQESBmGP1KtBegDxEg5pnVoFnT3pkAMcb05cP0+W9BgBhpyrJiypwVAWLqImL9c/vthJY/7xIghpl4Jk+c+T4EiHlGPBHz5OsUAsRsazZozakWJECMtPLNILd+TidATLVmg9TnLALEYKs1SH3OJUBsqkFVhvY2rT4nEiDG2zvb79+gvS2qz+l+nPFZWNXrOf/4+PNtEX4/PNx6u9JzISsgtuPOSyH1uZwVEJvy9PRntw5624grrobe7Zr6fI4AsfHLsWuV6NB6SnouIUB8oQztdeTDGB2/iJOey20kQCu8A8KgDF142EjPtWwkQHBKLw6V6Nzv4VoEiC9kryAf9khxbu378/Pfm28E4D3eAwIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQ8K3yP45ANBI1uz9qAAAAAElFTkSuQmCC', 'sha256': '0e7bd498942602fe300fbc9e66cfcb4f22661d938bec7e3b7147cfc9fa00a5c0'}, 'data/binding-v1/images/binding-evaluation-0097-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFYklEQVR4nO3aMU4jWRRAURg1+3HQEpuYuBdJPJtoiYDFECCRTIBkIQPGYOxbr3xO1IHb9UpUXf1fruunp8crgMI/yVEBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkfnWHhthm83v/Bx4e7s81y4W6fnp6rGeApRRnPz36cbMDdOT1tEyu8uVfJ/5GP8UWjMtNz83t3f7//vz3z56vlaHjCRCXlZ5Po/PRh9/GSIaOt5IA/XdzczXcv8/P9QhrTs+XuvOu7TfslEiGjrGSAMG79Tm+OweWaLP5rUHf4D0g1uMM9dnz/av8SeTUrIBYgzOnZ+dA26WQ7dhXWQExXlWfj45oKXQ4AWK213f7ze3d+evz7qE16EACxHrqk86yO4MGHUKAmGpp9XmhQV8iQIy0zPq80KDDCRCzLa0+S55qgQSIebbLiiXf59vZLIL2ECCGmXg/T5z5PASISZb86OctD4M+JUCMtPz6zJqzIkCMMX0RMX3+UxAg5pm1rJg17ZkJEDOM+OXrI34R+4gAARkBYoA1LRzWdC7HEyAmmbj/mj75SQkQkBEglm59e5b1ndG3CRBjTN/FTJ//FAQIyAgQkBEgICNALNroF6Df8kr0DgECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQEiEV7eLh/+cfz3z9X823PYnteF06AgIwAARkBAjICxBjTHwNNn/8UBIilW9/z2vWd0bcJEJARICaZu4uZO/lJCRADrGnPsqZzOZ4AARkBYobRr0R7AfojAsQ8sxo0a9ozEyDGmL58mD7/KQgQI01ZVkyZsyJATF1ELP/efj2h5c+7BIhhJt7JE2c+DwFinhG/iPnl6xACxGzLbNAyp1ogAWKkJT8M8ujncALEVMtskPp8iQAx2NIapD5fJUCsqkFVhnYOrT4HEiDG27nbz9+gnSOqz+F+feGzsFQv9/xm8/t1EW5u7059XOk5khUQ63HmpZD6HM8KiFV5eLjfroNeN+IHV0Pvdk19vkeAWPl27KdK9NF6SnqOIUBcUIZ2OvJpjPZv4qTneNdPT49XY729vFbAZb3868Tf6KdYAbF+214cWSLd+XECxAXZKcinPVKcU5u9BQNG8x4QkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICryv+NfDMDq1yCEwAAAABJRU5ErkJggg==', 'sha256': '31ea40bfc8de7253c3fcf07b4b0dbf955797c02b196b4e8dc32d042efff4796f'}, 'data/binding-v1/images/binding-evaluation-0098-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE+ElEQVR4nO3dMW7bSBiAUWWRHGcBF7nZniM3S2Fgr5LCQJotBAiClLUskdY3HL5XW+I0/DDzmza/vL39OgAU/kquCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyXw8zenn5ftiB19ef9RJgETsgICNAQGbOI9jJ73/+Pkzn249/6yXs68DuqPt5Jg8QLJ8Snv+wGK1LgNiXhb+gOH5chtYiQOy9OzfP6ddn3tO3KdFCAsQe63PXcPD8hy9i9PLyXYOWECBmtjA9144fP8+QQ9kSAsRe6rPir0T/mCENeoAAMX99PulpjIsMadADPIjIhJ5Qnz9+/07+BmhFAsRsnlmf66to0F0cwZjK6f5/8kPw58cxZ7GPswNiHlV9Tk7XtQ/6IAFiEqPd86OtZ0wCxAyeP/f5P+ZBdxEgpjLC/z8YYQ1bIUBsXj76uWYY9EECxLaNf4ePv8KQADGJcbY/Y65nTAIEZASIDRtw+nPOJOgmAQIyAsRWDb79ObIJep8AARkBAjICBGQEiE3axADoyBjoHQIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJt0evXo8WWkIzut0OtSrwkQkBEgICNAQEaA2KpNjIEMgN4nQEBGgNiwwTdBtj83CRCQESAmMdomaLT1jEmA2LbxTzfjrzAkQGzegJMg058PEiCmMkKDRljDVggQMzjfaLT3//nVbX9uEiAmMdrdPtp6xiRAzCMfBhn93Ovr3Z+Agb2+/jy+gPTYgqe9N9XJ6zF2QMzm+fMg9XmYADGhZzZIfZZwBGPys9jnHccu0qY+DxAgdtGgdTN0vatSn8cIEDM7dmHFDEnPugSI3W2FLjpyM0bvTJHUZyEBYhdOpbgo0WNTat1ZiwCx90PZAx9nLQLEHp135GaMROfzCBB7py8hDyICGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZnJn4T2hiYYmR0QkBEgIPPl7e1Xd3Vg1+yAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEHCr/AW/dBjTKesufAAAAAElFTkSuQmCC', 'sha256': 'f3f1f7deb2a2c605b1532da92521b98b8e095b0ca4440bca827b201f305e6750'}, 'data/binding-v1/images/binding-evaluation-0098-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFB0lEQVR4nO3aMW7iWhiAUeZpsh+KSNnE1LPI1G8TI6VgMVNEopkCCSGiQIJJPvv6nCqKsO9t/On6hx+vr383AIX/klUBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQObnZkTb7eNmBXa7l3oLMIkTEJARICAz5ivY0f8PD5vh/Nrv6y2s64Xdq+7XGTxAMH1KePphMbovAWJdJn5Bcbhchu5FgFh7dx6eni9fu//z+727KdFEAsQa63M1Ou99+CxG2+2jBk0hQIxsYnreOlx+miEvZVP4Gp611Ofh6XlifS7caiW/fb07AWJMp0W4Y3pOnd1Wg24gQIxfny9dS4OmECBG8531ebuKBn2KADGU4/P/Ra9d7zldToM+ToAYsz7JBjToswSIQcztmZ/bfuZJgBjB98993mMe9CkCxFDa+sxnD0shQCxePvp5yzDogwSIZZv/Ez7/HYYEiEHM5/gzz/3MkwABGQFiwWY4/TllEnSVAAEZAWKpZn78OXAIukyAgIwAARkBAjICxCItYgB0YAx0gQABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICxCLtdi+HP/Z/fm/m7bjD4545EiAgI0BARoCAjACxVIsYAxkAXSZAQEaAWLCZH4Icf64SICAjQAxiboegue1nngSIZZv/2838dxgSIBZvhpMg058PEiCGMocGzWEPSyFAjOD0oNE+/6erO/5cJUAMYm5P+9z2M08CxDjyYZDRz2cJEMM26DszdLqc+nycADGa758HmfvcTIAY0Hc2SH2mECBW0aCvyNDZbdXnBgLEsM6KcMcMvb2V+tzm543XwRIcurDdPh7/cwjHw9PzbTd8mzDpmUKAGN9u93LaoLOOXI3RhXOT+kwkQKzCsRRnJbptSq079yJArP2l7IbLuRcBYo1OO3I1RqLzdQSItdOXkK/hgYwAARkBAjICBGQECMgIEJARICAjQEBGgIDM4L+E/rXf11sA3uUEBGQECMj8eH39260OrJoTEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgE3lH8d4B7oGu20HAAAAAElFTkSuQmCC', 'sha256': '95962a1ab7421e870654c94dc8265ea6d6cdd204e95b17add27df5c11aafb08c'}, 'data/binding-v1/images/binding-evaluation-0098-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE/UlEQVR4nO3dMW7bSBiAUWcRHyeAixwjt9lz7G1yjBQG9iopDKhJIUAQrFiyRVnfcPheLYrT8MPMbxn88vLy+wGg8E9yVwABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+fowo6en7w8b8Pz8q14CLGIHBGQECMjMeQQ7+Pn4+DCdH7tdvYRtHdgddT/P5AGC5VPC4w+L0W0JENuy8A8U+8tl6FYEiK13Z/fvt/PXPv73/1vfpkQLCRBbrM/F6Lz14Vcxenr6rkFLCBAzW5ieU/vLjzPkULaEALGV+ixMz8UMadAVBIj563PD9JzJkAZdwQ8RmdAd6vPX79/I/wDdkAAxm3vW5/QuGvQhjmBM5fD83yc9fz2OOYu9nx0Q86jqc3C4r33QOwkQkxjtmR9tPWMSIGZw/7nPW8yDPkSAmEpbn3HWsBYCxOrlo59ThkHvJECs2/hP+PgrDAkQkxhn+zPmesYkQEBGgFixAac/x0yCLhIgICNArNXg2589m6DzBAjICBCQESAgI0Cs0ioGQHvGQGcIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0Cs0uHVo/uXkY7ssEKvSz0lQEBGgICMAAEZAWKtVjEGMgA6T4CAjACxYoNvgmx/LhIgICNATGK0TdBo6xmTALFu459uxl9hSIBYvQEnQaY/7yRATGWEBo2whrUQIGZwvNFon//ju9v+XCRATGK0p3209YxJgJhHPgwy+vmorx++Agb2/Pxr/wLSfQvu9t5UJ6/r2AExm/vPg9TnagLEhO7ZIPVZwhGMyc9in3cce5U29bmCALGJBt02Q6e7KvW5jgAxs30Xbpgh6bktAWJzW6FXHbkYozNTJPVZSIDYhEMpXpXouim17tyKALH1Q9kVl3MrAsQWHXfkYoxE5/MIEFunLyE/RAQyAgRkBAjICBCQESAgI0BARoCAjAABGQECMpP/EvrHblcvAXiTHRCQESAg8+Xl5Xd3d2DT7ICAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgQ8VP4AT2cHEU0RuN4AAAAASUVORK5CYII=', 'sha256': 'b4916b22bb6558bb6a760b679509816ed99d4cf3cbf7a4dec9cdad8d64572af5'}, 'data/binding-v1/images/binding-evaluation-0099-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE8ElEQVR4nO3dPW4aQQCAUSdK7kOR1hfKgXIhty44TApLaVKgIAQYm/DzDTvvlRawU3g/zQy77Je3t99PAIWvyVEBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZbd+hBrVY/nh7Wev1aDwHOYAYEZAQIyFiCvevl15+nB/H883s9BPgfZkBARoCAjAABGQECMgIEZAQIyPgaHi66Mt7V55cQILjodpzdF4vRuQQIrnYn4ObtMvR5AgSnuvPhBfGHl6FvP02JPiRAzO6wPmfdhbP74r0YrVY/NOg0AWJeF6bn0ObtuxmyKDtNgJjUXn2ueO/x0Qxp0FECxOz1udHPHuxlSIOOciEi07lDfY5+/kP/2OaNCBBzuWd9Do+iQXsswZjI9vy/86/N7S7HrMV2mQExi6o+W9vjmgdtCRBTGO2cH208FQFi+e6/7/Me+0F7BIiJjPCggRHGMA4BYuHyrZ9DNoO2BIglG/8MXw0/wpsSIKYwzvRnzPFUBAjICBCLNeDuz64XO0ECBIQEiGUafPqz8TL9JEiAgIwAARkBAjICxAI9xAbQxuTbQAIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEAu0ffTo5mGkI3v+N8I5H5cqQEBGgICMAAEZAWKZHmIb6HnuDSABAkoCxGINPgl6nn76I0BASYCYwmiToNHGUxEglmz81c16+BHelACxcAPuBNn92RIgJjJCg0YYwzgEiOXbnWi05//u0dfTT38EiFmMdraPNp7Kl7e339nBh/TQz4fzbz3yAwtt/Rz6duRvsFDr9eumQZsW3C1DVl7vsQRjLvffD1KfEyzBmH2hfdN5kPqcJkBMam+z7+oZ2pteqc9RAsS8Dr9wuEqGDld26vMeAWJ2V8yQ9JxLgODUtRcfxujETrb6fEiA4MpXgenO5wkQXC1D0nMuAYKLYiQ6lxAgIONKaCAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMBT5S+dsAnlnWhomwAAAABJRU5ErkJggg==', 'sha256': 'df7e8f8514b58d6f3b3e2124df945d5d27b57ffbc31beceffe908dfdcf4f9095'}, 'data/binding-v1/images/binding-evaluation-0099-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE8klEQVR4nO3dPW4bRxiAYTkwjxOAhY+R2/gcvk2O4YJAruJCAJsURAiCv1ZI8R3uPE9piNopvC9mPq20X97ff70BFP5IrgogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMl+7Sw9qvf729rI2m5/1EuAD7ICAjAABGUewi/5erd5exF/bbb0E+D/sgICMAAEZAQIyAgRkBAjICBCQ8WN4uOvJeE+f30OA4K5fxzn8YjH6KAGCh/0m4O7jMvT7BAiudWf7/c/rn139+OfSd1OimwSI2Z3W52Z0Ln3xUYzW628adJ0AMa8703Nq9/HDDDmUXSdATOqoPnem52aGNOgsAWL2+jwwPVcypEFneRCR6TyhPme//0v/sc1PIkDM5Zn1Ob2KBh1xBGMi+/v/Oek5exxzFjtkB8Qsqvrs7a9rH7QnQExhtHt+tPVUBIjle/7c5xLzoCMCxETa+oyzhnEIEAuXj35OGQbtCRBLNv4dvh5+hZ9KgJjCONufMddTESAgI0As1oDTn0NbkyABAkICxDINvv3Z2U6/CRIgICNAQEaAgIwAsUAvMQDamXwMJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAsUD7V4/uXkY6stV/K5zzdakCBGQECMgIEJARIJbpJcZAq7kHQAIElASIxRp8E7SafvsjQEBJgJjCaJug0dZTESCWbPzTzWb4FX4qAWLhBpwEmf7sCRATGaFBI6xhHALE8h1uNNr7//Dqm+m3PwLELEa720dbT+XL+/uv7OJDeun3w/lvPfILC41+Tn0982+wUJvNz12Ddi14WoacvC5xBGMuz58Hqc8VjmDMftD+1H2Q+lwnQEzqaNj38Awdba/U5ywBYl6nP3B4SIZOT3bqc4kAMbsHZkh6PkqA4NqzFzdjdGWSrT43CRA8+Ckw3fl9AgQPy5D0fJQAwV0xEp17CBCQ8SQ0kBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEg4K3yLzvMBB1H3CioAAAAAElFTkSuQmCC', 'sha256': '1553ff8606a3190f3f4c4bb2858153b582fcc0f8149be3c6dd531d7d5bdf3096'}, 'data/binding-v1/images/binding-evaluation-0099-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE8klEQVR4nO3dPVIbSxhAUfzK7EeBU3bglbwFvZW8HZA60GIIqHLiQGWVSr/GErqt6XNCCpgOmFvdHwPz5f397Qmg8E9yVQABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dpdelCr1benh7Ve/6iXAB9gBwRkBAjIOIKd9P/z89OD+P7zZ70E+Bt2QEBGgICMAAEZAQIyAgRkBAjI+DU8XPVkvKfPryFAcNWf4+x+shh9lADBzf4ScPPlMvTnBAjOdef1vwtPmb/8+3zquynRRQLE7A7rczE6pz55L0ar1TcNOk+AmNeV6Tm0+fLdDDmUnSdATGqvPlem52KGNOgoAWL2+twwPWcypEFHeRCR6dyhPke//0P/s81PIkDM5Z71ObyKBu1xBGMi2/v/Puk5ehxzFttlB8Qsqvpsba9rH7QlQExhtHt+tPVUBIjlu//c5xTzoD0CxETa+oyzhnEIEAuXj34OGQZtCRBLNv4dvhp+hZ9KgJjCONufMddTESAgI0As1oDTn12vJkECBIQEiGUafPuz8Tr9JkiAgIwAARkBAjICxAI9xABoY/IxkAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICxAJtXz26eRnpyF5+r3DO16UKEJARICAjQEBGgFimhxgDvcw9ABIgoCRALNbgm6CX6bc/AgSUBIgpjLYJGm09FQFiycY/3ayHX+GnEiAWbsBJkOnPlgAxkREaNMIaxiFALN/uRqO9/3evvp5++yNAzGK0u3209VS+vL+/ZRcf0kO/H86P9cgvLDT6OfT1yMdgodbrH5sGbVpwtww5eZ3iCMZc7j8PUp8zHMGY/aD9qfsg9TlPgJjU3rDv5hna216pz1ECxLwOf+FwkwwdnuzU5xQBYnY3zJD0fJQAwblnLy7G6MwkW30uEiC48VNguvPnBAhuliHp+SgBgqtiJDrXECAg40loICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwFPlF45vCeVryXPVAAAAAElFTkSuQmCC', 'sha256': '35e3af97e791efe275d539e411b88159c1f1c3a9b95630745cb7f4fb65f9d617'}, 'data/binding-v1/images/binding-evaluation-0100-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFPUlEQVR4nO3dMXITSxRAUfEL7UfBX8YsV8sg0GIIXOWEwIVKJVm2MTJ3pvucGDStYC6vG9x8e3r6uQMo/Jc8FUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIy37tHr93h8P9us06nH/US4H0mICAjQEDGFux9x/1+txHL83O9BPgDJiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQMaPYsBj7kVwA8EnCBA85jKWy98iRh8kQPD4S6DOn6NEbxMgeD89796IcO8egpePlaF7BIjZvZqeP72D5erXX/VIhu4RIOZ1m55H3f10/pzLEsnQLQFiRl+Xnlc/Vobu8e+AmL0+x/3+qy+9vH3Epv/LgwcSIOZy+eb/g/RcunrcQYNswZh58EmWcdzvzzuyw/TbMRMQU1hJfV59+mHiUUiAGF+47brHduyFADFXfXZrcpy+QQLEyNZcnxeTN0iAmMI667P+tX01AWJY54Fi/W/48fcKZxuCBIgxbfdNPmx25Z8gQAxo/Uc/t45THgYJECPbSn22uNqHECBGs6Gjn1uzHQYJEJARIIay6fFnwiFIgICMADGOAcaf2YYgAQIyAsQghhl/phqCBAjICBCQESBGMNj+a55dmAABGQECMgIEZASIzRv4iGT47yhAjGOkE+hRv9EVAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZASIcSzPz7uxLMN9oysCxOadTj92ozsN+h2/1wvYgOH/FIKKCQjICBCQ+fb09LN7OjzMePfSL7/3/qMeAJmAgJIAARkBYhDnfcoYf2u5TLD/EiCgJECMY5ghaJlj/BEgoCRADGWAIWiZZvwRIKAkQIxm00PQMtP4I0AMblsNWja12ocQIAZ0OT5s5a1eLtY5yfgjQAxru+/wabMr/wQBYlgbOgxaJjv6ORMgprDmBi0rXttXEyBGtv7DoGXKo58zAWJwa27QMnd9XEjGdNeVreTSsqsUnqasjwmIWVy94e0opD5nJiDmHYWSOci265IAMZ1qO2bwuSVAzOiqQV+dodsdn/q8ECDm9Q8yJD1vEyBmd5uhvy/Rq4fc0nNLgOBuhj7eo7f/Wk167hEg+GiGPkF63iZA8PgS6c4HCRA8Jkai8wkCBGT8KAaQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE7Cq/ACrLL/fPCq33AAAAAElFTkSuQmCC', 'sha256': '2fb6d0f1b8dd09f8890a6de45e7813ffd65f4a04bf16c0fa6f7c8ba8e5b43a55'}, 'data/binding-v1/images/binding-evaluation-0100-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQElEQVR4nO3dQW4TSxRAUYPwfjxAyk5YZHaCxMCLYRCpJ38QYVl2nIR8h9tddc4Y3OVBX14VpPjy9PR7B1D4mjwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjLfukev3eHwfbdZx+OvegnwNhMQkBEgIGML9rb9w+NuI5afP+olwF8wAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMn4UA+5zL4IbCD5AgOA+l7Gc/xYxeicBgvtfAnX6HCV6nQDB2+l580aEW/cQPH+sDN0iQMzuxfT87R0sF7/+okcydIsAMa/r9Nzr7qfT55yXSIauCRAz+rz0vPixMnSLfwfE7PXZPzx+9qWX14/Y9H95cEcCxFzO3/x/kJ5zF487aJAtGDMPPsky9g+Ppx3ZYfrtmAmIKaykPi8+/TDxKCRAjC/cdt1iO/ZMgJirPrs12U/fIAFiZGuuz7PJGyRATGGd9Vn/2j6bADGs00Cx/jd8/2eFsw1BAsSYtvsmHza78g8QIAa0/qOfa/spD4MEiJFtpT5bXO1dCBCj2dDRz7XZDoMECMgIEEPZ9Pgz4RAkQEBGgBjHAOPPbEOQAAEZAWIQw4w/Uw1BAgRkBAjICBAjGGz/Nc8uTICAjAABGQECMgLE5g18RDL8dxQgxjHSCfSo3+iCAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgLEOJafP3ZjWYb7RhcEiM07Hn/tRncc9Dt+qxewAcP/KQQVExCQESAg8+Xp6Xf3dLib8e6lX/7s/Uc9ADIBASUBAjICxCBO+5Qx/tZymWD/JUBASYAYxzBD0DLH+CNAQEmAGMoAQ9AyzfgjQEBJgBjNpoegZabxR4AY3LYatGxqtXchQAzofHzYylu9nK1zkvFHgBjWdt/h42ZX/gECxLA2dBi0THb0cyJATGHNDVpWvLbPJkCMbP2HQcuURz8nAsTg1tygZe76uJCM6a4rW8mlZRcpPE5ZHxMQs7h4w9tRSH1OTEDMOwolc5Bt1zkBYjrVdszgc02AmNFFgz47Q9c7PvV5JkDM6x9kSHpeJ0DM7jpD/79ELx5yS881AYKbGXp/j17/azXpuUWA4L0Z+gDpeZ0Awf1LpDvvJEBwnxiJzgcIEJDxoxhARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKv8B+OpMp+6c5W5AAAAAElFTkSuQmCC', 'sha256': '44adc8434ac3ea3fda6dcb9d5818a85326803e56908bc86a546955567a469515'}, 'data/binding-v1/images/binding-evaluation-0100-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFR0lEQVR4nO3dMW4bRxiAUTow76MiQC6xtQ+ZWpcw4IKHcSFATQrBBEGKlKxQ/nZn3qtF7rDYD/+MqNWXp6efO4DCX8lVAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDztbv02j08/L3brMPhR70EeJsJCMgIEJCxBXvb/p9/dxvx/P1bvQT4DSYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDGn2LAfZ6L4AkEHyBAcJ+HsZy+RIzeSYDg/g+BOr6PEt0mQPB2eh73+9svX56fb7ytDF0jQMzu1fS8WZzbP3/WIxm6RoCY12V6frc71xzf57REMnRJgJjR56Xn1beVoWt8D4jZ6/O4339SfW5cYtP/8uCOBIi5nN75fyA9p84u96BBtmDMPPgky3jc7487sofpt2MmIKawkvq8evWHiUchAWJ84bbrGtuxFwLEXPXZrcnj9A0SIEa25vq8mLxBAsQU1lmf9a/tswkQwzoOFOu/wx9/rXC2IUiAGNN27+SHza78AwSIAa3/6OfS45SHQQLEyLZSny2u9i4EiNFs6Ojn0myHQQIEZASIoWx6/JlwCBIgICNAjGOA8We2IUiAgIwAMYhhxp+phiABAjICBGQEiBEMtv+aZxcmQEBGgICMAAEZAWLzBj4iGf4zChDjGOkEetRPdEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAWIcy/PzbizLcJ/ojACxeYfDj93oDoN+xq/1Ajbg+fu3egkwJhMQkBEgIPPl6elnd3W4m/GeS7/8OoEe9QDIBASUBAjICBCDOO5TxvjuzDLB/kuAgJIAMY5hhqBljvFHgICSADGUAYagZZrxR4CAkgAxmk0PQctM448AMbhtNWjZ1GrvQoAY0On4sJW7ejlZ5yTjjwAxrO3ew4fNrvwDBIhhbegwaJns6OdIgJjCmhu0rHhtn02AGNn6D4OWKY9+jgSIwa25Qcvc9fFAMqZ7XNlKHlp2lsLDlPUxATGLszu8HYXU58gExLyjUDIH2XadEiCmU23HDD6XBIgZnTXoszN0ueNTnxcCxLz+QIak5zYBYnaXGfr/JXr1kFt6LgkQXM3Q+3t0+9dq0nONAMF7M/QB0nObAMH9S6Q77yRAcJ8Yic4HCBCQ8acYQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELCr/AdStS/PbTc2jQAAAABJRU5ErkJggg==', 'sha256': '6c500955c1b54eeb5ab1b6bf99cd9801bc9be7d5d69b2509fb1088dd0122d265'}, 'data/binding-v1/images/binding-evaluation-0101-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFT0lEQVR4nO3avW0bSxhAUfrBLOcBDASoEzfy6nAj6sSAAwJuxYGATRwQ1iNI/VGidHd3zokUUDujYK++GfDL/f3vDUDhn2RVAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZr93S87Lb3WxWZ7//WW8BniNAcM1/VKJ/EQGCa07Hxx8WoxcJ0Knpv383C7f9/qvewhq8/1T+8AQleooAwQXd2d7ePf/r049vzzxTiU4IELxQnxej89SHz2O0291o0DEBgsfTc1F3HvXwhOMSHRaSoQPfA4LT+mxv795fn+cfuMqvfbyBCYihnafn49Y6PPxhGtoZhUxAjOwz6/PUKruxRyEBYlDHb/7Vz1zPO1luN3CDBIgRndQn2cNWgwSIAc2hPuer74ZskAAxlvnU52DwBgkQA5lbfQ5GbpAAMaL51Gee+/k0AsQoHoaLeb7t27+7GmoIEiCGsKy3ereo3b6HADGWeY4/89/bBxEg1m/mh6+RD2ICxMot903eLXbnrydAjGL+48+y9nkVAgRkBIg1W9Dtz5g3QQIEZAQIyAgQq7XQ89dQpzABAjICBGQECMgIEOu06Augca6BBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEg1mm//3n4YfrxbbNM09+dP/wt6yNAQEaAgIwAARkBYrUWfQ00DXABJEBASYCAjACxZgs9hU1jnL8ECCgJEKNYyhA0LWSfVyFArNxyTzH7xe789QSI9VvQTdA0zO3PgQAxljk3aJrx3j6IADGEZQ0U+0Xt9j0EiFHM/CA2DXb4OhAgRjS3Bk0z28+nESAGcjxczOedn452MtT4I0AMZ24NmgaujwAxovk0aBq7PgLEoObQoGn4+ggQ4zpp0Gdm6GS5/aj1ESCGdvLmf06DTlbZD1yfzWbztd4AlA7v/253c1yH7e3dR6wlPedMQPDIKHTdaej8gepzYAKCR0ah44HlzQPRoxWTnmMCBP/b738eN+i8Iy/G6PnRSX1OCBA83ojzEr35olp3niJAp7bff9VbYAEluugJPEWA4IKOvBgj0bmIAMEF9OW6vtzf/77yIwFex/eAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKn8AXe/JN9i7cZyAAAAAElFTkSuQmCC', 'sha256': '963b2e87bacacebde5c08a148389ca5f32b22a2c7156fe260a347c4a58ae0fca'}, 'data/binding-v1/images/binding-evaluation-0101-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRUlEQVR4nO3aTW4TWRSAUdOiloPkQSR2wkZYBxvpnSBlYImtMIiUSQ8sIrft/Jg4/qrqnTNiEKoug/q47ymfHh5+bwAK/yRvBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dy9el6227vN6ux29/UI8BIBgmv+RyX6FxEguOZ2fPjDYvQqATo2ff13s3CPP7/VI6zB+0/lT09QoucIEFzQncfvX17+69OPXy88U4mOCBC8Up9Xo/PcD5/GaLu906BDAgTn03NRd856esJhifYvkqE9AYLj+rw/PUf2DzzK0E6DBIjBfXR6XsjQ1irkN6EZ2S3r89xbtmv8Ddi3swExqMMv/zbpeWEV2o26B9mAGFFYn7Pv3Y66BwkQw5lDfU7fvh2yQQLEWOZTn73BGyRADGRu9dkbuUECxIjmU595znMzAsQonpaLeX7tj3+mGmoJEiCGsKyveruoad9DgBjLPNef+c/2QQSI9Zv54Wvkg5gAsXLL/ZK3i5387QSIUcx//VnWnFchQEBGgFizBd3+jHkTJEBARoCAjACxWgs9fw11ChMgICNAQEaAgIwAsU6LvgAa5xpIgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQFinXa7+/0fph+/Nss0/Zn86d+yPgIEZAQIyAgQkBEgVmvR10DTABdAAgSUBAjICBBrttBT2DTG+UuAgJIAMYqlLEHTQua8CgFi5ZZ7itktdvK3EyDWb0E3QdMwtz97AsRY5tygacazfRABYgjLWih2i5r2PQSIUcz8IDYNdvjaEyBGNLcGTTOb52YEiIEcLhfz+eang0mGWn8EiOHMrUHTwPURIEY0nwZNY9dHgBjUHBo0DV+fzWbzuR4AGrvd/XZ7d9iCx+9fbvPqo+TtRq2PDYihHX35t1mF1OeQDYih7b//26xC0nNKgOB/x7GPyNDpbqU+ewIEZ1ahw2r8dYnOnumk55AAwbOr0GlHXo3RyxdJ6nNEgOB8I05L9NcX1brzHAE69vjzWz0CCyjRRU/gOQIEF3Tk1RiJzkUECC6gL9f16eHh95UfCfA2fhMayAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABm8p/8e0k5zp9GrAAAAAASUVORK5CYII=', 'sha256': 'feaff83e033ffc3687152541aa9cf772c46cdd7783d671b97695254a353581b6'}, 'data/binding-v1/images/binding-evaluation-0101-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQElEQVR4nO3aPW7bShhAUeUh3g8LA95JFumdBHChxaQwwCaFED9B8p9s2ZfknFO5EDjjglffDPTj8fHPDqDwX7IqgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMz27pZZmm293m7PcP9RbgNQIE1/yiEv2LCBBcczo+/rAYvUmATt3c3e9Wbv79q97CFnz+VP70BCV6iQDBBd158/vp2for0UsECN6oz0VD8fGHz2M0TbcadEyA4Pn0fP4w/vSE4xIdFpKhA78DgtP63NzdX/cq8PyBm/zZxweYgBjaeXq+bq3Dw5+mockoZAJiZN9Zn5dWmcYehQSIQR2/+Vc/c73uZLlp4AYJECM6qU+yhxsNEiAGtIT6nK8+DdkgAWIsy6nPweANEiAGsrT6HIzcIAFiRMupzzL3820EiFE8DRfLfNtv/u1qqCFIgBjCut7qaVW7/QwBYizLHH+Wv7cvIkBs38IPXyMfxASIjVvvmzytdufvJ0CMYvnjz7r2eRUCBGQEiC1b0e3PmDdBAgRkBAjICBCbtdLz11CnMAECMgIEZAQIyAgQ27TqC6BxroEECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESC2ab9/OPwx//61W6f5386f/pftESAgI0BARoCAjACxWau+BpoHuAASIKAkQEBGgNiylZ7C5jHOXwIElASIUaxlCJpXss+rECA2br2nmP1qd/5+AsT2regmaB7m9udAgBjLkhs0L3hvX0SAGMK6Bor9qnb7GQLEKBZ+EJsHO3wdCBAjWlqD5oXt59sIEAM5Hi6W887PRzsZavwRIIaztAbNA9dHgBjRcho0j10fAWJQS2jQPHx9BIhxnTToOzN0stx+1PoIEEM7efO/p0Enq+wHrs9ut/tZbwBKh/d/mm6P63Bzd/8Va0nPORMQPDMKXXcaOn+g+hyYgOCZUeh4YPnwQPRsxaTnmADB//b7h+MGnXfkzRi9PjqpzwkBgucbcV6iD19U685LBOjUEn4dy/JLdNETeIkAwQUdeTNGonMRAYIL6Mt1/Xh8/HPlRwK8j98BARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYFf5C1FnJZQeZIoJAAAAAElFTkSuQmCC', 'sha256': '7a42023e3b3e853af573fd4c19971c87c6b0dcf50c3ab3d56a4ecce193c0143f'}, 'data/binding-v1/images/binding-evaluation-0102-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFN0lEQVR4nO3aPW4iWRhAUTzqWk5LBE57Q5P2OjqdDU3aQUm9FQdITiZAzSA3xr9wq3jnRASYehbU1fce3O12DxuAwl/JVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD50l0aVmm7vT//hHn+ea21rN7dbvdQrwFWH53niNF5AvQ5n7OWT/nyPw/eo5NsweBt3Xn8/vX8n08/fp15TSU6JkDwcn1ejM5zT/4zRtvtvQYdCNBp//7zuFm8b39P9RJuPD1v6s5Jh1c4LtH+QrMMCRCcrM/H0/PE/gWfZGgevkECxOgunZ4zGdoOPwr5ISJDu2Z9nrvKdp3fvX4KExDjOr7zr5OeM6PQPOQcZAJiUGF9Tl53O+QcJECMaAn12Ru8QQLEcJZTn72RGyRAjGVp9Rm8QQLEoJZTn2Wu5zoEiIEchotl3u2Pv1c1zhAkQIxiXXf1dlWrfTcBYjjLHH+Wv7ZLECCGsPDN17AbMQHi9q33Tt6uduWvJEAMZPnjz7rW+XECBGQEiBu3otOfAU+CBAjICBCQESBu2Ur3X+PswgQIyAgQkBEgICNA3KxVHwANcgwkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjABxs+b55/7B9OPXZp2m3ys//C83RoCAjAABGQECMgLELVv1MdB06wdAAgSUBAjICBA3bqW7sGmA/ZcAASUBYiBrGYKmlazz4wSI27feXcy82pW/kgAxhBWdBE1jnP7sCRDDWXKDpgWv7RIEiFGsa6CYV7XadxMgBrLwjdg00uZrT4AY1NIaNC1sPdchQIzleLhYzj0/Ha1knPFHgBjR0ho0jVofAWJQy2nQNHB9BIhxLaFB09j12Ww2d7vdQ72GBdlu7zcrNOZn9xLv+OP3r9e57pPkzaO+gyYghvbkzr/OKKQ+B1/+fwhD2t//h1FoX4cLjULS84QtGDy7Af/EDP05W83D10eA4FXngO8u0ck9nfQcCBC87buIF2N0/iBJfY4JEFzjW1HdOUmA4IIl0p3zBAg+OUai83oCBGT8EBHICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZvKf9cSJDHKyGr0AAAAAElFTkSuQmCC', 'sha256': '331bebf282c7193aa2814913d8060ffdd8fbe993f1df4f9d2dcf6757b3b33950'}, 'data/binding-v1/images/binding-evaluation-0102-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFOUlEQVR4nO3aPW4iWRhAUTxq74fAUke1jVlEL6gXMdtwNFIHLMYBkpMJUDOIxvgXbhXvnIgAU8+Cuvreg7vt9mkFUPgruSqAAAElAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIy37pLwyKt1w/nn7DZ/LrWWhbvbrt9qtcAi4/OS8ToPAH6ms9Zy6d8/p8H79FJtmDwvu48/nw+/+fTj/szr6lEhwQIXq/Pq9F56cl/xmi9ftCgPQE67f77P6vZe/7373oJN56ed3XnpP0rHJZod6GNDAkQnKzP59NzZPeCRxnaDN8gAWJ0l07PmQythx+F/BCRoV2zPi9dZb3M716/hAmIcR3e+ddJz5lRaDPkHGQCYlBhfU5edz3kHCRAjGgO9dkZvEECxHDmU5+dkRskQIxlbvUZvEECxKDmU595ruc6BIiB7IeLed7tj79XNc4QJECMYll39XpRq/0wAWI48xx/5r+2SxAghjDzzdewGzEB4vYt905eL3blbyRADGT+48+y1vl5AgRkBIgbt6DTnwFPggQIyAgQkBEgbtlC91/j7MIECMgIEJARICAjQNysRR8ADXIMJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAcbM2m1+7B9OP+9UyTb9Xvv9fbowAARkBAjICBGQEiFu26GOg6dYPgAQIKAkQkBEgbtxCd2HTAPsvAQJKAsRAljIETQtZ5+cJELdvubuYzWJX/kYCxBAWdBI0jXH6syNADGfODZpmvLZLECBGsayBYrOo1X6YADGQmW/EppE2XzsCxKDm1qBpZuu5DgFiLIfDxXzu+elgJeOMPwLEiObWoGnU+ggQg5pPg6aB6yNAjGsODZrGrs9qtbrbbp/qNczIev2wWqAxP7uXeMcffz5f57pHyduM+g6agBja0Z1/nVFIffa+/f8QhrS7//ej0K4OFxqFpOeILRi8uAH/wgz9OVtthq+PAMGbzgE/XKKTezrp2RMgeN93Ea/G6PxBkvocEiC4xreiunOSAMEFS6Q75wkQfHGMROftBAjI+CEikBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQJWlf8ALLInFQaRTE4AAAAASUVORK5CYII=', 'sha256': '8b749386e863c6ffc7461bfa3f879f5414348eb5fdddaa43eeec3c4976b30df3'}, 'data/binding-v1/images/binding-evaluation-0102-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFOElEQVR4nO3aTW4aWRhAUdxKLScSA0vZSW8k68hGeictZVBStuIBkic9QKGRg/Ev3CreOSMGmHoW1NX3Htztdg8bgMJfyVUBBAgoCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dJdGlZpu70//4R5/nmttaze3W73UK8BVh+d54jReQL0OZ+zlk/58j8P3qOTbMHgbd15/P71/J9PP36deU0lOiZA8HJ9XozOc0/+M0bb7b0GHQjQadO3fzaL9/jv3/USbjw9b+rOSYdXOC7R/kKzDAkQnKzPx9PzxP4Fn2RoHr5BAsToLp2eMxnaDj8K+SEiQ7tmfZ67ynad371+ChMQ4zq+86+TnjOj0DzkHGQCYlBhfU5edzvkHCRAjGgJ9dkbvEECxHCWU5+9kRskQIxlafUZvEECxKCWU59lruc6BIiBHIaLZd7tj79XNc4QJECMYl139XZVq303AWI4yxx/lr+2SxAghrDwzdewGzEB4vat907ernblryRADGT548+61vlxAgRkBIgbt6LTnwFPggQIyAgQkBEgbtlK91/j7MIECMgIEJARICAjQNysVR8ADXIMJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAcbPm+ef+wfTj12adpt8rP/wvN0aAgIwAARkBAjICxC1b9THQdOsHQAIElAQIyAgQN26lu7BpgP2XAAElAWIgaxmCppWs8+MEiNu33l3MvNqVv5IAMYQVnQRNY5z+7AkQw1lyg6YFr+0SBIhRrGugmFe12ncTIAay8I3YNNLma0+AGNTSGjQtbD3XIUCM5Xi4WM49Px2tZJzxR4AY0dIaNI1aHwFiUMtp0DRwfQSIcS2hQdPY9dlsNne73UO9hgXZbu83KzTmZ/cS7/jj96/Xue6T5M2jvoMmIIb25M6/ziikPgdf/n8IQ9rf/4dRaF+HC41C0vOELRg8uwH/xAz9OVvNw9dHgOBV54DvLtHJPZ30HAgQvO27iBdjdP4gSX2OCRBc41tR3TlJgOCCJdKd8wQIPjlGovN6AgRk/BARyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGbyn/rHSQxr6uYlgAAAABJRU5ErkJggg==', 'sha256': '989c5f9570677ac26b7a2fcf04efbdb6c07acf81a529148650f4fd0db2318ecb'}, 'data/binding-v1/images/binding-evaluation-0103-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIUlEQVR4nO3dvXEbRxiAYcoj9qPAqTtQJSrIlbgDpwpYjALOMHHAGQgGwCN+CL57e88TQ7hN7p1vF0fdl+fnXw8AhT+SqwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPO1u/Rwvn3782FqT08/6yXA/5iAgIwAARlbsBP+eXx8mMj3l5d6CXCaCQjICBCQsQWDC34J9UvixxIgtu6ixy/2PyxGtxMgNur2x75236BEVxMgNmchPf/+/c4vhn/9eHzrC2XoCgLE1tPzbnTe+vBBjGToCgLEdutzUXqOvf7z4wxp0PkEiC3W58b0LGdIg84nQEzufulZyJDt2Jk8iMjMPqc+b33/9P+/wu1MQExr//6/d3oWRiFz0AITEHNK6nPyiuagBQLEhNr6HF9Xg94iQMysqs8IV18FAWI2u3FjhPt/twZD0EkCxFRGvs9HXltFgJjHCEc/xxwGLRAgJjROfcZczzgEiEmsZbhYyzo/hwAxmzHHjTFXlRMgZrCusWJdq70rAWIqIw8aI6+tIkBARoBYvaGePFzmucQDAgRkBAjICBCQESDWbUUHQK8cA+0TICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoBYt92b13evYx/cbp1P3hkvQEBIgICMAAEZAWL1VnQM5ADogAABGQFiKiMPQSOvrSJAzGBdO5p1rfauBIjZjDlojLmqnAAxibWMFWtZ5+cQICY02rgx2nrGIUDMY3+4GOee31+J8eeAADGVke/wkddWESBmM9RziZ48XCZAzKxt0AgFHJwAMaERDoMc/ZxDgJhT2yD1OdPXcz8Ia/P09HP3/vXXInzC++MPYqc+y0xAzOzg/r/3KKQ+lzIBMbnXCtx7FJKe6wgQm9uOfWyGjqcq9TmfALHRBt2eoZMbOvW5iACx3e3YcUfejdHCKZL0XEGA2JyTGbrllFp6riZAbNSuGidLdNE3cDUBYuv2O/JujETnYwkQ/KYvn8yDiEBGgICMLdgJ31/u/hdDgAkIKAkQkPny/PyruzqwaSYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBDwUPkPL7MMWiGZUD8AAAAASUVORK5CYII=', 'sha256': '049239d5f47f228cbfd78cfcd4c45deba7a2165a14a3fa4c396498a1f4fd4ce7'}, 'data/binding-v1/images/binding-evaluation-0103-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFLklEQVR4nO3bMW4bRxiAUTkw76PCgC/h2odMrUsYcKHDuBCgJgURhqEokqIkfrOz71UqyJ1p9sM/y9WXp6c/dwCFv5JVAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dotPZz7+293U3t8/F1vAf7HBARkBAjIOIIdsfn+991Enn/9rLcAx5mAgIwAARlHMHjDL6F+SfxYAsTaven1i/0Pi9H7CRAr9f7XvnZXUKKrCRCrcyI9D5vN6e/+eH5+7YIydAUBYu3pORud1z58ECMZuoJfwVhvfR42mzfV55KvT/8PPR9LgFiFgy68Mz2nL6VBl3MEY3Xp+YxVtpfdHcocxy5kAmJmt6nPa9c3Cp0lQExr//7/wDPXaQcLadBpAsQq6nPj1TXoQgLEhNr6vFxXg14jQMysqs8Iqy+CADGb3bgxwv2/24Mh6CgBYioj3+cj760iQMxjhEc/L3kYdIIAMaFx6jPmfsYhQExiKcPFUvZ5GwLEbMYcN8bcVU6AmMGyxopl7fZTCRBTGXnQGHlvFQECMgLE4g315uFp3ks8IEBARoCAjAABGQFi2Rb0AGjLY6B9AgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQy/b4+Hv7x4/n57sl2O3z8d+dr5kAARkBAjICBGQEiMVb0GMgD4AOCBCQESCmMvIQNPLeKgLEDJZ1olnWbj+VADGbMQeNMXeVEyAmsZSxYin7vA0BYkKjjRuj7WccAsQ89oeLce75/Z0Yfw4IEFMZ+Q4feW8VAWI2Q72X6M3D0wSImbUNGqGAgxMgJjTCwyCPfi4hQMypbZD6XEiAWEuDbpOhg4XU5zQBYmYH9/9nN+jg+upz1tfzH4El21bg/v7bfiMeNpuPXUV6rmMCYqWj0EdNQy8vpT6XEyDW4mUX3pmho19XnzdxBGO9x7Gt/YicPZqdCJb0XEGAWJ2jGdq6biCSnqsJECu1q8bREr3pClxNgFi7/Y6cjZHofCwBgv/oy435FQzICBCQcQQ74vnXz3oLsAomICAjQEDmy9PTn251YNVMQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEg4K7yDwsuDbyfXO+wAAAAAElFTkSuQmCC', 'sha256': '1de42a3400c4076c59d83e9fceaf80243590dbc992b271e767f4289f229a0923'}, 'data/binding-v1/images/binding-evaluation-0103-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFJUlEQVR4nO3dPW4bRxiAYTmw7+MigKtcw4fIgXKIXCOVARc6jAsBalIIYBiSWvFH1Ds7+zw1xZ1mX3wzXGg/PT39egAo/JZcFUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmc/dpYfz9evvD1N7fPxZLwH+xwQEZAQIyNiCnfDl298PE3n+8b1eApxmAgIyAgRkbMHggl9C/ZL4vgSIrbvo8Yv9D4vR7QSIjbr9sa/dNyjR1QSIzVlIzz9/PS//7R9/fnntC2XoCgLE1tPzZnRe+/BBjGToCgLEdutzUXqOvfz5cYY06HwCxBbrc2N6ljOkQecTICZ3v/QsZMh27EweRGRmH1Of175/+v+vcDsTENPav//vnZ6FUcgctMAExJyS+py8ojlogQAxobY+x9fVoNcIEDOr6jPC1VdBgJjNbtwY4f7frcEQdJIAMZWR7/OR11YRIOYxwtHPMYdBCwSICY1TnzHXMw4BYhJrGS7Wss6PIUDMZsxxY8xV5QSIGaxrrFjXau9KgJjKyIPGyGurCBCQESBWb6gnD5d5LvGAAAEZAQIyAgRkBIh1W9EB0AvHQPsECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDWbffm9d3r2Ae3W+ejd8YLEBASICAjQEBGgFi9FR0DOQA6IEBARoCYyshD0MhrqwgQM1jXjmZdq70rAWI2Yw4aY64qJ0BMYi1jxVrW+TEEiAmNNm6Mtp5xCBDz2B8uxrnn91di/DkgQExl5Dt85LVVBIjZDPVcoicPlwkQM2sbNEIBBydATGiEwyBHP+cQIObUNkh9zvT53A/C2jw+/ty9f/2lCB/w/viD2KnPMhMQMzu4/+89CqnPpUxATO6lAvcehaTnOgLE5rZj75uh46lKfc4nQGy0Qbdn6OSGTn0uIkBsdzt23JE3Y7RwiiQ9VxAgNudkhm45pZaeqwkQG7WrxskSXfQNXE2A2Lr9jrwZI9F5XwIE/9GXD+ZBRCAjQEDGFuyE5x/f6yXAJpiAgIwAAZlPT0+/uqsDm2YCAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABD5V/AbqgDDI4IxgHAAAAAElFTkSuQmCC', 'sha256': '153984c62221478dd07868e4a26271b5624fbefb851715bb4971922a86cdc779'}, 'data/binding-v1/images/binding-evaluation-0104-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHElEQVR4nO3bO24bSRRAUXpg7keBAW/C8SxyYm1iAAdcjAMBSiYQhiBImR9T4u2qPidSQHYXIPLiVan15eXl1wag8FdyVwABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg83UzuKenb5vp7HY/6yXAI5iAgIwAAZnht2B72+//bAb3+u/f9RLgoUxAQEaAgIwAARkBAjLzHELDY54s85TWBxIguO1x1sMXi9GdBIj1uv8x+v0VlOjPCBBrdCY9z9vt+ff+eH393QVl6FYCxLq8m56L0fndi49iJEO38lcwVl2f5+32pvpc8/Yp/0H6k5iAWIWjKNwTnVP7q+0HIqPQlUxAzO9T63PmykahiwSIyR1W4M4N1zWObqFB5wkQK6rPw+6rQVcSIKZV1ef0jhr0OwLEnNr6nN5Xg94lQExoCfU5vbsGnRIgZrOc+pyuQYOOCBDTWkJ9lraSpREgprIfMZb2nd+vxxB0SICYxyjf7VHW+QACxISWNv4seVUtAWISi918HbIROyJAQEaAmMGIA8XTgGv+cALEVJa8/xplhY8kQEBGgBjeEMfPhxxF7wkQkBEgICNAQEaAGNtwB0BvHAO9ESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAGNtu9/Pthx+vr5tx7Fe7+3/96yRAQEaAgIwAARkBYnjDHQM5ANoTICAjQExl+UPQ8lf4SALEDEbcy+wGXPOHEyAgI0BMYoijaMfPRwSICS2zQctcVUuAmMcoY8Uo63wAAWIqi92I2Xy9S4CY1nIatJyVLI0AMZvDEWMJ3/zDNRh/jggQE1pOg9TnPAFiTktokPpcJEBMq22Q+lxDgJhZ1SD1uZIAsa4GfXaGjm6hPucJEPM7qsDnNejoyupz0dfLL4HxvbXg6enbUSmet9v7L35aNOm50peXl1+bke0/UjPx8X3wB+aPM/TuMOXXdz0TEKsehU47cjFGZ3Zw0nMrAWKN3s3QPSdE0rPSLRiEG3nduZMAwW0xEp0PJEBAxnNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKn8B6ljDh8YobsnAAAAAElFTkSuQmCC', 'sha256': 'f31515219dfa8a42ba91b5178a43c4b220088e8b164443414c8137b434860068'}, 'data/binding-v1/images/binding-evaluation-0104-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFGklEQVR4nO3cMW6aSxhAUfKULCeSi+ws68jOXCBlKykspXkFeggBj0CMuTPDOZULzD9S4Oqb0Tif3t5+bQAK/yRPBRAgoCRAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIynzeTe3n5tlnOdvtaLwEewQQEZAQIyEy/Bdv7/f3rZnJffvyslwAPZQICMgIEZAQIyAgQkFnnEBoec7PMLa07EiC47Trr4YvF6J0EiOf1/mv0+3dQor8jQDyjC+n544Wys9e1dm8oQ7cSIJ7L2fTcdIv18MVHMZKhWwkQT12fd16g3/36aYY06EoCxDOm575/uLN/t32JjEJXcg+I9X1ofS6885L/V8x9mYBY3GEFHvAXy0ebMtuxy0xArOzB9Tn7LHPQBQLEsqr6nD5Rg/6PALGmtj6nz9WgswSIBY1Qn9Ona9ApAWI149TndA0adESAWNYI9RltJaMRIJayHzFG+87v12MIOiRArGOW7/Ys63wAAWJBo40/I6+qJUAsYtjN1yEbsSMCBGQEiBXMOFC8TLjmuxMgljLy/muWFT6SAAEZAWJ6Uxw/H3IUvSdAQEaAgIwAARkBYm7THQDtOAbaESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAmNt2+7r74cuPn5t57Fe7/W/9z0mAgIwAARkBAjICxPSmOwZyALQnQEBGgFjK+EPQ+Ct8JAFiBTPuZbYTrvnuBAjICBCLmOIo2vHzEQFiQWM2aMxVtQSIdcwyVsyyzgcQIJYy7EbM5ussAWJZ4zRonJWMRoBYzeGIMcI3/3ANxp8jAsSCxmmQ+lwmQKxphAapzx8JEMtqG6Q+1xAgVlY1SH2u9PnaF8KcttvXl5dvh134/f3rxz3uKHPqc5kJiPUdVeDjRiH1uZUJiKewa8HRKHSvaei0aNJzpU9vb782M9t/pFbi4/vgD8xfZ+jsMOWf73omIJ56FDrtyB9jdGEHJz23EiCe0dkMveeESHqedAsG4UZed95JgOC2GInOHQkQkHEPCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjaVfwEPURFX9Mtw7QAAAABJRU5ErkJggg==', 'sha256': '009263553c57b656d075fd7faaa1f3938f252c4ea3c665f9544029b2e45eff72'}, 'data/binding-v1/images/binding-evaluation-0104-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHUlEQVR4nO3bvW0bSRiAYfpglnOAApfhbq6O60ZlOCBwrTgQoMSBcARByvwxJb47s88TKSB3BxD54pvR6svLy88NQOGv5K4AAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQObrZnBPT98209ntftRLgEcwAQEZAQIyw2/B9l7/+XszuO2//9VLgIcyAQEZAQIyAgRkBAjIzHMIDY95ssxTWh9IgOC2x1kPXyxGdxIg1uv+x+j3V1CiPyNArNGZ9Dxvt+ff+/319XcXlKFbCRDr8m56Lkbndy8+ipEM3cpfwVh1fZ6325vqc83bp/wH6U9iAmIVjqJwT3RO7a+2H4iMQlcyATG/T63PmSsbhS4SICZ3WIE7N1zXOLqFBp0nQKyoPg+7rwZdSYCYVlWf0ztq0O8IEHNq63N6Xw16lwAxoSXU5/TuGnRKgJjNcupzugYNOiJATGsJ9VnaSpZGgJjKfsRY2nd+vx5D0CEBYh6jfLdHWecDCBATWtr4s+RVtQSISSx283XIRuyIAAEZAWIGIw4UTwOu+cMJEFNZ8v5rlBU+kgABGQFieEMcPx9yFL0nQEBGgICMAAEZAWJswx0AvXEM9EaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAWJsu92Ptx++v75uxrFf7e7/9a+TAAEZAQIyAgRkBIjhDXcM5ABoT4CAjAAxleUPQctf4SMJEDMYcS+zG3DNH06AgIwAMYkhjqIdPx8RICa0zAYtc1UtAWIeo4wVo6zzAQSIqSx2I2bz9S4BYlrLadByVrI0AsRsDkeMJXzzD9dg/DkiQExoOQ1Sn/MEiDktoUHqc5EAMa22QepzDQFiZlWD1OdKAsS6GvTZGTq6hfqcJ0DM76gCn9egoyurz0VfL78ExvfWgqenb0eleN5u77/4adGk50pfXl5+bka2/0jNxMf3wR+YP87Qu8OUX9/1TECsehQ67cjFGJ3ZwUnPrQSINXo3Q/ecEEnPSrdgEG7kdedOAgS3xUh0PpAAARnPAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMCm8gszUQ4fLHQerQAAAABJRU5ErkJggg==', 'sha256': '0dfe384a61ae8a3e6fd14d2a524e3d149bcf0a88ce16305c06375f8ff7e1b499'}, 'data/binding-v1/images/binding-evaluation-0105-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFM0lEQVR4nO3cQWobWRRAUaWJ96OBwaNsI4vIgnoR3kaPAhl4MRkYPOmBQAg7kWWprPu/6pxRAk7VC6Qu71dsfXl+/r0BKPyT3BVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjJfN+uz3d5vJvT09KseARZmAwIyAgRk1ngE2/vv35fN8L79uKtHgM9iAwIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMqv+TmhW690fSPajv9chQKzFhz4F4fCLxejzCBA37vJPX9lfQYkWJ0CssTt3D4/H//jLz+9HrqlESxEg1lKfd6Pzty9+G6Pt9l6DFiFA3Hh6PtSdP9pf4bBEuxvJ0IX8Nzw3W5+7h8fL63P8gpN+vO84bEDcgrfp+bx77S6+34asQpewATG9a9bnb3exCp1HgJjb4ZO/+JnruFe306AzCBC3U59kBg26hAAxqxHq8/buGvQhAsSUxqnP2xk06HQCxHxGq8+OBp1BgJjYOPUZc57xCRCT2S8XYz7t+6ksQacQIGYy11M917QJAWJKY64/4882GgFiGoMfvg45iJ1IgJjDvE/yvJNfgQAxmfHXn7nmbAkQkBEgJjDR259D3gS9S4CAjAABGQFidJOev3acwo4TICAjQEBGgICMADG0qV8A7XgNdIQAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAsTQnp5+7X7x8vP7Zk77yfd/F/YECMgIEJARICAjQIxu6tdAXgAd93WzYt9+3NUjwKrZgICMADGBSU9hzl/vWuMRzL8GGIQNiMnMsgTNMmdLgJjDvHvrvJNfgQAxjYneBHn7cyIBYkojN2jk2UYjQMxkroVirmkTAsRkBj+IOXx9iAAxsdEaNNo84xMg5nO4XIzzzB9OYv05kQAxpdEapD7nESBmNU6D1OdsAsTERmiQ+lziy/Pz74suALXt9v7wt3cPj9e576vkqc8ZbEBM79WTf51VSH0WYQPidlxnFZKeBQkQt9ygZTP0drdSnwsJEKvI0CUl+uOZTnoWIUCsqEGnx+j4iyT1WYoAseoSfYjuLE6AWIuzS6Q7n0eAWKN3YyQ61yFAQMY3IgIZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCATeV/bVAVxo7pzbIAAAAASUVORK5CYII=', 'sha256': '4f77ce0730c5c1e7ab8070e11c29e3e06443af01b7c48187c03e29b1be0dbbb7'}, 'data/binding-v1/images/binding-evaluation-0105-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFLUlEQVR4nO3dMU5bWxRAUecrDOdLFGkzobR/HGkzobQpkDKVFEhpfoFkWZAYjI33Pby1KpDgvYvE2zr3yoYP9/e/dgCFf5K7AggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5uNue25vP+0Gurv7US8BLswEBGQECMhscQu29/u/f3fLu/n6s14CvBUTEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ2fQrodmsZ9+Q7K2/1yFAbMVJfwXh8IvF6O0IEO/c+X99ZX8FJbo4AWKL3fn+7ffxb//85ebINZXoUgSIrdTn2ej87Yufxuj29pMGXYQA8c7Tc1J3/mh/hcMSPdxIhs4kQLzb+pyfnkceLvgoQxp0DgHiPXjr9BzJkFHoHF6IyHjXrM/f7jL0Px3kTEDMdvjkXyc9R0Yhc9CpTEAMFtbnj/c1B51KgJhqhfo8vbsGnUSAGGmd+jxdgwa9nAAxz2r1eaBBryBADLZOfdZcz/oEiGH2w8WaT/t+VYaglxAgJpn1VM9abUKAGGnN8Wf9ta1GgBhj8c3XIRuxFxIgZpj7JM9d+RUIEMOsP/7MWmdLgICMADHAoNOfQ06CniVAQEaAgIwAsbqh+68HdmHHCRCQESAgI0BARoBY2ugDoAeOgY4QICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoBY2t3dj4cPPn+52c20X/n+Z2FPgICMAAEZAQIyAsTqRh8DOQA67uNuw26+/qyXAJtmAgIyAsQAQ3dh9l/P2uIWzG8DLMIExDBThqAp62wJEDPMnVvnrvwKBIgxBp0EOf15IQFipJUbtPLaViNATDJroJi12oQAMcziGzGbr5MIEIOt1qDV1rM+AWKew+FinWf+cCXGnxcSIEZarUHq8zoCxFTrNEh9Xk2AGGyFBqnPOT7c3/866wJQu739dPjp92+/r3PfR8lTn1cwATHeoyf/OqOQ+lyECYj34zqjkPRckADxnht02Qw9na3U50wCxCYydE6J/rink56LECA21KCXx+j4QZL6XIoAsekSnUR3Lk6A2IpXl0h33o4AsUXPxkh0rkOAgIwXIgIZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAXeV/NdYjHZ9fkVQAAAAASUVORK5CYII=', 'sha256': '5062fd5cdd5ebd7f484168c30924a1aa2975bde4a0dbb4f8be7b6e0ee4a369aa'}, 'data/binding-v1/images/binding-evaluation-0105-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFM0lEQVR4nO3cQWobWRRAUaWJl9OggcE7yUZ6HdmIdxLIwJCtZGDwpAcCIexElqWy7v+qc0YJOFUvkLq8X7H15fn59wag8E9yVwABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyHzdrM92e7+Z0NPTz3oEWJgNCMgIEJBZ4xFs7+W/fzfDu/v+qx4BPosNCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIrPo7oVmtd38g2Y/+XocAsRYf+hSEwy8Wo88jQNy4yz99ZX8FJVqcALHG7tw9PB7/4y8/vh25phItRYBYS33ejc7fvvhtjLbbew1ahABx4+n5UHf+aH+FwxLtbiRDF/Lf8Nxsfe4eHi+vz/ELTvrxvuOwAXEL3qbn8+61u/h+G7IKXcIGxPSuWZ+/3cUqdB4BYm6HT/7iZ67jXt1Og84gQNxOfZIZNOgSAsSsRqjP27tr0IcIEFMapz5vZ9Cg0wkQ8xmtPjsadAYBYmLj1GfMecYnQExmv1yM+bTvp7IEnUKAmMlcT/Vc0yYEiCmNuf6MP9toBIhpDH74OuQgdiIBYg7zPsnzTn4FAsRkxl9/5pqzJUBARoCYwERvfw55E/QuAQIyAgRkBIjRTXr+2nEKO06AgIwAARkBAjICxNCmfgG04zXQEQIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEEN7evq5+8XLj2+bOe0n3/9d2BMgICNAQEaAgIwAMbqpXwN5AXTc182K3X3/VY8Aq2YDAjICxAQmPYU5f71rjUcw/xpgEDYgJjPLEjTLnC0BYg7z7q3zTn4FAsQ0JnoT5O3PiQSIKY3coJFnG40AMZO5Foq5pk0IEJMZ/CDm8PUhAsTERmvQaPOMT4CYz+FyMc4zfziJ9edEAsSURmuQ+pxHgJjVOA1Sn7MJEBMboUHqc4kvz8+/L7oA1Lbb+8Pf3j08Xue+r5KnPmewATG9V0/+dVYh9VmEDYjbcZ1VSHoWJEDccoOWzdDb3Up9LiRArCJDl5Toj2c66VmEALGiBp0eo+MvktRnKQLEqkv0IbqzOAFiLc4uke58HgFijd6NkehchwABGd+ICGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQI2lf8BEdsWf/FJvpUAAAAASUVORK5CYII=', 'sha256': 'fd6a2a88d74a5b61dd9d1e8545f73f2f47d651e4775524a8fd6b7c65d531948a'}, 'data/binding-v1/images/binding-evaluation-0106-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIklEQVR4nO3dMW7jRhiAUW+wPk6ALXKMvU3OkdvkGFsYyFW2MKAmhQFBsGxaFiV/M+R7tUVOww8/BwPz2/Pz7weAwh/JXQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg87279R39+PHXw0Y9Pf2qlwA3YwICMgIEZLb5Cnb07+Pjwyb8PBzqJcDtmYCAjAABGQECMhvfA4KVJzace7grAWJ3PnVM7PSPxejmBIi9WH889XgFJboVAWLX6Tn8/efybx//+e+9C8rQegLE7tLzYXTe++NXMZKh9QSIHdXnU+k59/Lz8wxp0NUEiF3UZ2V6ljOkQVcTILbmfulZyJDXses4iMimfE193rv+hv8PzJ2YgNiO0+f/3ulZGIXMQZczAbERSX3evKM56HICxBa09Tm/rwZdSIDYlKo+I9x9RgLE9I7jxgjP/3ENhqBLCBBzG/k5H3ltgxAgJjbC1s85m0GXEyC2YJz6jLmeYQkQs5pluJhlnQkBYnpjjhtjrmo0AsSU5hor5lrtVxIg5jbyoDHy2gYhQEBGgJjPUCcPlzmXuEyAgIwAARkBAjICxGQm2gB6YRtogQABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICxGSOX14/fo59cMd1+mb8OQECMgIEZAQIyAgQ85loG8gG0DIBAjICxNxGHoJGXtsgBIgpzfVGM9dqv5IAMb0xB40xVzUaAWJWs4wVs6wzIUBswWjjxmjrGZYAMbHT4WKcZ/50JcafZQLE3EZ+wkde2yAEiOkNdS7RycNPESA2pW3QCAWciwCxBSNsBtn6uYIAsRFtg9TnOt+v/B2M5+np1/H76y9F+ILvx7+Knfp8igmITXn1/N97FFKflUxAbM1LBe49CknPTQgQ238du22Gzqcq9bmaALGXBq3P0JsvdOqzhgCxo9ex8458GKOFXSTpWU+A2GmG1uxSS8+tCBB7cazGmyX61BW4FQFid0478mGMROeuNh6gn4dDvQSGpi8tBxGBjAABmW/Pz7+7uwO7ZgICMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECHir/A+DyDRHgi3L+AAAAAElFTkSuQmCC', 'sha256': 'adf8f2e5a6f20e32c4192d6fbf9eca60d57159fe1f265d4021bd80a5de7b1f7d'}, 'data/binding-v1/images/binding-evaluation-0106-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFLUlEQVR4nO3bvW0bSxhAUdkwyzGgwGW4m1eHu1EZDgS4FQcCmLyAAEFQFiX+iHdm95xIAbkzyV58s1x9eXn5+wBQ+JqsCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZb93Sn+jx8cfDQj0//663ADdjAgIyAgRklnkE29v+9/1hETa//tRbgNszAQEZAQIyAgRkFv4MCK58Y8N7D59KgFids14TO/ywGN2cALEW17+eur+CEt2KALHq9DxtNqe/+3O7feuCMnQ9AWJ16Xk3Om99+ChGMnQ9v4Kxovo8bTZn1ecjX1/wPx7egQCxTEdduDI9py+lQRdzBGP56fmMVXaX3R/KHMcuYwJiUe5Tn7eubxQ6lwCxHIf3/w3PXKcdLaRBZxEgllmfO6+uQZcRIJagrc/rdTXogwSIRanqM8LqMxIgprcfN0a4//d7MAR9hAAxt5Hv85H3NggBYmIjPPp5zcOgjxMglmCc+oy5n2EJELOaZbiYZZ8JAWJ6Y44bY+5qNALElOYaK+ba7T0JEHMbedAYeW+DECAgI0DMZ6g3D0/zXuJpAgRkBAjICBCQESAmM9EDoB2PgU4QICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCYzPPz790fP7fbhxns97nfOXsCBGQECMgIEJARIOYz0WMgD4BOEyAgI0DMbeQhaOS9DUKAmNJcJ5q5dntPAsT0xhw0xtzVaASIWc0yVsyyz4QAsQSjjRuj7WdYAsTEDoeLce75w50Yf04TIOY28h0+8t4GIUBMb6j3Er15eBYBYlHaBo1QwLkIEEswwsMgj34uIEAsRNsg9bmMALHYBt0nQ0cLqc9ZBIhFObr/P7tBR9dXn3N9O/sbMLZdBR4ffxw24mmzue0q0nMTJiDWMgrdahp6fSn1uZgAsVivu3Blhv75dfW5hiMYKzqO7RxG5N2j2YlgSc/1BIiVZmjnsoFIem5FgFiLfTX+WaKzrsCtCBCrc9iRd2MkOp9q4QHa/PpTb4Gh6UvLr2BARoCAzJeXl7/d6sCqmYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAh8r/Tm4NBcgRk+IAAAAASUVORK5CYII=', 'sha256': '5a684aad2970a95c5aa2269f6500edc67fb0f6bf84a358a76455f97d2ab4d8c0'}, 'data/binding-v1/images/binding-evaluation-0106-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHUlEQVR4nO3dMW4jNxiAUW+wOU6ALXKznCM328JArrKFgTQpBAiC5ciyRtJHct6rZQ2b+fAPQWu+vb39egEo/JZcFUCAgJIAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIy37tLP9CPH3++LOr19We9BLgbExCQESAgs+Yj2NG/f/3xsoTf//6nXgLcnwkIyAgQkBEgILP4HhBsPLHh3MNDCRC786VjYqcfFqO7EyD2Yvvx1OM3KNG9CBC7Ts+nBzU+PABx+EIZ2k6A2F16vnQ67PTD72IkQ9sJEDuqz8aDqYc/P8+QBt1MgNhFfe54Jv48Qxp0MwFiNY9Lz4UMeRy7jYOILOU59fm/71/4d2AexATEOk7v/6f9H/L5KGQOup4JiEUk9fnwiuag6wkQK2jrc35dDbqSALGU9heglvn9qacRIKZ3HDdGuP+PazAEXUOAmNvI9/nIaxuEADGxEbZ+ztkMup4AsYJx6jPmeoYlQMxqluFilnUmBIjpjTlujLmq0QgQU5prrJhrtc8kQMxt5EFj5LUNQoCAjAAxn6FOHl7mXOJlAgRkBAjICBCQESAmM9EG0IFtoAsECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAmc3zz+vF17IM7rtM7488JEJARICAjQEBGgJjPRNtANoAuEyAgI0DMbeQhaOS1DUKAmNJcTzRzrfaZBIjpjTlojLmq0QgQs5plrJhlnQkBYgWjjRujrWdYAsTEToeLce7505UYfy4TIOY28h0+8toGIUBMb6hziU4efokAsZS2QSMUcC4CxApG2Ayy9XMDAWIRbYPU5zbfb/w7GM/r68/j+9cPRXjC++PfxU59vsQExFLe3f+PHoXUZyMTEKs5VODRo5D03IUAsf7j2H0zdD5Vqc/NBIi9NGh7hj58oFOfLQSIHT2OnXfk0xhd2EWSnu0EiJ1maMsutfTciwCxF8dqfFiiL30D9yJA7M5pRz6Nkeg81OIB8r85XKYvLQcRgYwAAZlvb2+/uqsDu2YCAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAl4q/wGFHgw0skJqpQAAAABJRU5ErkJggg==', 'sha256': '06962a307d997526b9b829eddbee9fb3fa23dd0bffc4677ed0cfe6a9daa41d99'}, 'data/binding-v1/images/binding-evaluation-0107-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE9ElEQVR4nO3dMW4bRxiAUTqw77OFAVd7jRwiB/Ihcg1XAVzwMC4EuEmxCKFIpAhL4n5Dz3s1qR1A2g8zPwnth4eHHweAwh/JVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg87G79IiW5fPhbh2P3+slwK+xAwIyAgRkHMHO+/Tl78Od+PnPn/US4JXsgICMAAEZRzC41WemPpe8SoDgVl/UePxiMTpLgGCP74htb5ehJwQIXtmdb19/vvze9a9Pl36aEm0ECH6hPlejc+nFT2K0LJ81SIDgJul5bnv74wwtDmUCBFfr88b0XM3QceIGCRBcrM87pueFDC0TN8gXEWHX+pz9+cs9/xuGtxAgCOrz/CrLlA1yBIP/3f/7pOfscWyZ7yxmBwRZfU5O151tHyRAzG60e34ZbD03JUBMbf+5zyXfppwHCRD09RlnDTsTIOaVj36em20YJEBMavw7fBl+hW8nQMxunO3PmOu5KQECMgLEjAac/sw5CRIgICNATGfw7c9UmyABAjICBGQECMgIEHO5iwHQPGMgAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZASIuZwePbo9jHRk638r/I0flypAQEaAgIwAARkBYjp3MQZaJxgACRBQEiBmNPgmaJ1j+yNAQEmAmN1om6B1sPXclAAxqfFPN8fhV/h2AsS8BpwErdNMfzYCBKM0aB1gDTsTIKb2eKPR3v/ro6tPsv0RIBjubj8Otp6bEiDoh0HrZKOfk4/1AmAIx+P37QGkWwt2e27qOuXJ68QOCLJ50Dp3fQQIsgat09fncDh8eHj4Ua9hINsm/E5N+0d86z+Ddz+OPUnbceJfnBkQXJwHbd5xKvR8V3WcuD4CBOdtXXjHDEnPWY5g8MpT+dUYvTBFUp+NAMF+80HdeUKAYI8MSc9ZAgS3ipHoXCVAQMYXEYGMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgUPkXFtkE7UgF5r4AAAAASUVORK5CYII=', 'sha256': '5171791ea9b154bfe2cb4eaa702809c15e9ec0155bfc87b5ad3791210d63d34d'}, 'data/binding-v1/images/binding-evaluation-0107-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFAElEQVR4nO3bQWobSRiA0c4QHWegFwbfJBeZc8xFfJNAFoK5ShaG3mQhEhTZsolt6StNvbfyQuousPqj+m/p0+Pj9wWg8FdyVgABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfO5OPaJ1vVtu1n7/rV4C/Bk7ICAjQEDGLdjztn/+Xm7E7t//6iXAG9kBARkBAjJuweBSz0w9l3yVAMGlvqhx/GIxepYAwTW+I3Z4uwydECB4Y3d29w8vv3f7+uXc0ZToQIDgD+rzanTOvfgkRut6p0ECBBdJz1OHtx9naHVT5jE8vFqf3f3DO+vzwqHWW/754fsJECznivCB6Tl2cth14gYJEJytz0XPtdMgAYKkPk/Psk7ZIAGC367/C912nbM7Ot2EDRIg+K0+yQJ2szZIgJjdaNf8Oth6LkqAmNr15z7n7KacBwkQ9PUZZw1XJkDMKx/9PDXbMEiAmNT4V/g6/ArfT4CY3TjbnzHXc1ECBGQEiBkNOP2ZcxIkQEBGgJjO4NufqTZBAgRkBAjICBCQESDmchMDoHnGQAIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEHPZ778d/ti+flnGtv1c4a81//8IEJARICAjQEBGgJjOTYyBtgkGQAIElASIGQ2+Cdrm2P4IEFASIGY32iZoG2w9FyVATGr8u5v98Ct8PwFiXgNOgrZppj8HAgSjNGgbYA1XJkBM7Xij0V7/29HZJ9n+CBAMd7XvB1vPRQkQ9MOgbbLRzy8CBKcNumaGtqPTzVYfAYJyHrRNOfc5JkDQNGibvj7Lsnx6fPxer2Eg63q33KxpP8SX/hjs7h8+9vgnadtP/I+zA4JTJ0X4wKnQ00PtJ67Psiyf6wXAiA5dON4KHcLx5t3Q04Tt507PgVsweONd+asxemHfpD4HAgTXmw/qzgkBgmtkSHqeJUBwqRiJzqsECMh4DA9kBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABS+UHlDT/PpE7zAMAAAAASUVORK5CYII=', 'sha256': 'b575b84a5b512f16e15e62ae7e4fcf72106dbde005a7efba768f06881ba80440'}, 'data/binding-v1/images/binding-evaluation-0107-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE80lEQVR4nO3dMW4bRxiAUTqwjxNgC7d7obQ5h9tcyK0LAr6KCwFuUixCKBIpwpK439DzXk1qB5D2w8xPQvvh4eHHAaDwR3JVAAECSgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh87C49omX5fLhbx+O3egnwa+yAgIwAARlHsPN+/v3n4U58+vK9XgK8kh0QkBEgIOMIBrf6zNTnklcJENzqixqPXyxGZwkQ7PEdse3tMvSEAMEru/P1n58vv3f969Oln6ZEGwGCX6jP1ehcevGTGC3LZw0SILhJep7b3v44Q4tDmQDB1fq8MT1XM3ScuEECBBfr847peSFDy8QN8kVE2LU+Z3/+cs//huEtBAiC+jy/yjJlgxzB4H/3/z7pOXscW+Y7i9kBQVafk9N1Z9sHCRCzG+2eXwZbz00JEFPbf+5zydcp50ECBH19xlnDzgSIeeWjn+dmGwYJEJMa/w5fhl/h2wkQsxtn+zPmem5KgICMADGjAac/c06CBAjICBDTGXz7M9UmSICAjAABGQECMgLEXO5iADTPGEiAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAWIup0ePbg8jHdn63wp/48elChCQESAgI0BARoCYzl2MgdYJBkACBJQEiBkNvgla59j+CBBQEiBmN9omaB1sPTclQExq/NPNcfgVvp0AMa8BJ0HrNNOfjQDBKA1aB1jDzgSIqT3eaLT3//ro6pNsfwQIhrvbj4Ot56YECPph0DrZ6OfkY70AGMLx+G17AOnWgt2em7pOefI6sQOCbB60zl0fAYKsQev09TkcDh8eHn7UaxjItgm/U9P+Ed/6z+Ddj2NP0nac+BdnBgQX50Gbd5wKPd9VHSeujwDBeVsX3jFD0nOWIxi88lR+NUYvTJHUZyNAsN98UHeeECDYI0PSc5YAwa1iJDpXCRCQ8UVEICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECDhU/gWPwATtc9R7dgAAAABJRU5ErkJggg==', 'sha256': 'ce7d7c9aab93fff00162d94d2f4adbcbd67402ed056682922ef5e007f4706b42'}, 'data/binding-v1/images/binding-evaluation-0108-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUUlEQVR4nO3ay20bSRRAUWpghjOAFg6js5k4JhuG4QWBSUULAdzMQjBB6EPrf7urzlnL7GqJff2qyJv7+7sdQOGv5KoAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzo7s0rN3t7c/X/Njx+Ovr1zKmm/v7u3oNW3qrbYVHonon+M2/iQkIPvN/oPPrKNFrCBCzu56ew35//Z8vp9OVl5Wh6wTosdM/f+82a//vf/USNp+ePxbn+s8/6pEMXSdAzOhpet7anZecX+eyRDL0EgFiLl+XnmdfVoau8z0g5q3PYb//ovpcucRgn7d+kAAxi8sn/xvSc+nR5TTozBaMGQefZBmH/f68I7Mde2ACYnArqc+zV7+dfhQSIEYWbrteYjt2SYCYpT67NdGgBwLEmNZcnwcHDRIghrfO+qx/bd9DgBjQeaBY/xN++L3COYcgAWI0232Sbze78ncTIIay/qOfpw4THwYJEGPaSn22uNpPJECMY0NHP08dpjwMEiAgI0AMYtPjz7RDkAABGQFiBAOMP3MOQQIEZASIzRtm/JlwCBIgICNAQEaA2LbB9l+z7cIECMgIEJARICAjQGzY8Ecku9HvUYAYwUgn0KPe0bMECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESBGsJxOu7Esw93RswSIDTsef+1Gdxz6HgUIyAgQkBEgICNADHJEMtKp7fL7XsY+ABIgoCRAQEaA2LzBdmHLNPsvAQJKAsQIhhmClpnGHwECSgLEIAYYgpbJxh8BAkoCxDg2PQQt840/AsSwttWgZVOr/UQCxFAux4etPNXLxTqnGn8EiAFt9xk+bnbl7yZADGhDh0HLlEc/ZwLE4NbcoGXFa/seAsSY1n8YtEx89HMmQAxrzQ1SnwcCxEQNWkOGHi3jOHF9drvdzf39Xb2GVbi9/bkbyORv6+t/3MN+X63kUQGP0/+ZftQLgC/38JyfM7ScTkmDDD5P2YIxi3A7Ztv1ElswZt9rf+k09DRz6nNJgJjRN2RIel5DgJjXs588fLBEz+7spOclAsTsrn8A+sceXT9Lkp7rBAi+5HsY0vMaAgSfWSLdeRMBgo/GSHTeTYCAjC8iAhkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjYVf4H2RQ+w/MYdZAAAAAASUVORK5CYII=', 'sha256': '35bc5885661be316852f359fe533fc2b2dc1a78a9cdf41f2a412040347a0283a'}, 'data/binding-v1/images/binding-evaluation-0108-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFSElEQVR4nO3aMVIbSRiAUbFl30cB6VzIB/KFlBLoMASqcrKB1ypKCC1g4Jvpfi9xgqUemPn4u9Hd6fS4Ayj8k7wrgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfOveGtZuv79/zZcdjw+fv5Yx3Z1Oj/UatnSrbYVHoroTfOffxAQEH/kb6Pw6SvQaAsTsbqfn8PPX7f++/Ph+42Vl6DYBevMNt2YvPQy8Pj1vvQEuvv7iRyBDtwkQM3qeno/6xXN+naclkqGXCBBz+bz0XH1ZGbpNgJi3Pl+w3b6aIQ06EyBmrM8Xn/RdZEiDznwSmvHt9/dhfa6+78WSpiVADO7rt103XLz7fvoG2YIxsjUMPhdsx54yATGsFdbnpe3YblYCxJjWXJ/fDhokQAxvnfVZ/9q+hgAxoPNAsf4n/PBnhXMOQQLEaLb7JO83u/J3EyCGsv6jn+cOEx8GCRBj2kp9trjaDyRAjGNDRz/PHaY8DBIgICNADGLT48+0Q5AAARkBYgQDjD9zDkECBGQEiM0bZvyZcAgSICAjQEBGgNi2wfZfs+3CBAjICBCQESAgI0Bs2PBHJLvRr1GAGMFIJ9CjXtFVAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQI1h+fN+NZRnuiq4SIDbseHzYje449DUKEJARICAjQEBGgBjkiGSkU9vlz7WMfQAkQEBJgICMALF5g+3Clmn2XwIElASIEQwzBC0zjT8CBJQEiEEMMAQtk40/AgSUBIhxbHoIWuYbfwSIYW2rQcumVvuBBIihPB0ftvJUL0/WOdX4I0AMaLvP8HGzK383AWJAGzoMWqY8+jkTIAa35gYtK17b1xAgxrT+w6Bl4qOfMwFiWGtukPr89u2/f2FEx+PDfn//9Jk//PzVLukihceJ67Pb7e5Op8d6Datwvk3HMPltffuHGzZIfS7YgjG+4/FhDduxi23Xcfr62IIxkXA7ZvB5iS0Ys++1PzVDz6ct9XlKgJjRF2RIel5DgJjX1b88/GWJrh4wSc9LBIjZ3f4D6P/26PaRtvTcJkDwKZ/DkJ7XECD4yBLpzpsIEPxtjETn3QQIyPgkNJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAXeVfa1ZLfyGJMA8AAAAASUVORK5CYII=', 'sha256': '09abe7c27d41f9273533d37666b7a04ee63256ac9a7c85be17957008d5097806'}, 'data/binding-v1/images/binding-evaluation-0108-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFU0lEQVR4nO3aPVIbSRiAYbFl3YfA6d5gTrIH2pPoBps60GEIqCLZgLJKBUJg/t6Z7ueJsaYHNK+/bunm/v5uB1D4K7kqgAABJQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjI/OguDWt3e/vzLT92PP76+rWM6eb+/q5ew5bealvhkajeCX7zf8QEBJ/5P9DpdZToLQSI2V1Pz2G/v/7Pl4eHKy8rQ9cJ0FP//Xv5/bQJf//zytPCq+l5tTjXf/5Jj2ToOgFiRs/T86fdecnpdc5LJEMvESDm8nXpufiyMnSd7wExb30O+/0X1efKJQb7vPWDBIhZnD/535Cec08up0EntmDMOPgkyzjs96cdme3YIxMQg1tJfS5e/Xb6UUiAGFm47XqJ7dg5AWKW+uzWRIMeCRBjWnN9Hh00SIAY3jrrs/61fQ8BYkCngWL9T/jh9wrnHIIEiNFs90m+3ezK302AGMr6j36eO0x8GCRAjGkr9dniaj+RADGODR39PHeY8jBIgICMADGITY8/0w5BAgRkBIgRDDD+zDkECRCQESA2b5jxZ8IhSICAjAABGQFi2wbbf822CxMgICNAQEaAgIwAsWHDH5HsRr9HAWIEI51Aj3pHFwkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQIxgeXjYjWUZ7o4uEiA27Hj8tRvdceh7FCAgI0BARoCAjAAxyBHJSKe2y+97GfsASICAkgABGQFi8wbbhS3T7L8ECCgJECMYZghaZhp/BAgoCRCDGGAIWiYbfwQIKAkQ49j0ELTMN/4IEMPaVoOWTa32EwkQQzkfH7byVC9n65xq/BEgBrTdZ/i42ZW/mwAxoA0dBi1THv2cCBCDW3ODlhWv7XsIEGNa/2HQMvHRz4kAMaw1N0h9HgkQEzVoDRl6sozjxPXZ7XY39/d39RpW4fb2524gk7+tr/9xD/t9tZInBTxO/2f6US8Avtzjc37K0PLwkDTI4POcLRizCLdjtl0vsQVj9r32l05DzzOnPucEiBl9Q4ak5y0EiHld/OThgyW6uLOTnpcIELO7/gHoqz26fpYkPdcJEHzJ9zCk5y0ECD6zRLrzRwQIPhoj0Xk3AQIyvogIZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIGBX+R9huD7DTWirsgAAAABJRU5ErkJggg==', 'sha256': 'e6e8dd4c786978b57eacf4698d35cc515a6f67a17eda79d6b8dbfefd82bcc1da'}, 'data/binding-v1/images/binding-evaluation-0109-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVUlEQVR4nO3bMW4bVxRAUTow98PCgDeR2ot0nU0YcMHFuBAwTQrCDMEojGRTupw351QqJM4fSHPx/if14enpxw6g8EdyVQABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPNxt2aHw6fdaMfj93oJ8IZMQEBGgIDMurdgZ3/t97tB/lyWegnwHkxAQGbIBAThuxzeK/hlAgS/+9bq5Y+I0asIENzzIx3n11GilxAgtu52evafv97+8eXblxsvK0O3CRDb9Wx6/rc4t7//qkcydJsAsUV3Sc+zTi8iQy8kQGy9PnfpzpXza16W6HD4pEFXBIgNeYf03B6IjEJXfBCRrXj/+vzXtcb/E/XLCRCbcPnM7z9/fc/6PHtRDToRIDZXn3AlGnRFgBjucerz7zUcNt8gAWKyR6vPiQadCRBjPWZ9TjToRICY6ZHrc7LXIAFipMevz8l+8w0SICZ75PqsZYVvSoCY5jxKrOXZ3v9c5waHIAFilLU/w4eVr/+1BIiZ1jL+rHG1dyRAzLG6zdelbW7EBAjICBBDrHr82ewQJEBARoCYYN7IcBh3R88SIEZZ7/5rxvpfS4CAjACxegOOnzd7FC1AQEaAgIwAARkBYt1mH5QcRt+dADHHjBPoefdymwABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICxBDLty+7KZZB93KbALFux+P33VzH0XcnQEBJgICMAAEZAWLOQcmMs9vl512MPwASIKAkQIyy9iFoWfn6X0uAmGDebuU47o6eJUBARoAYYsBR9LKl4+cTAQIyAsQcqx6Clu2NPwLEWOtq0LKq1d6RADHK2seH48rX/1oCxDSr24gtm9x8nQgQkz1+g5aHX+GbEiAGuhwlHvkJXy7WtsHxR4AY6/EbtGy+PgLEZI/cIPU5ESAme8wGqc+ZADHcozVIfS4JEPM9ToPU54oAscUGvX+Gri6qPicfnp5+7FbrcPi0G82f6Zv+wew/f32f6171zq/17OM/X8J0pyf/nKFTF940Q9JzmwCxOcfj98tR6NyIO5bo2S2e+kzbgsHdt/C/mSHpeRUBYutunyT+b49un2dLz20CBG/yhob0vIQAwT1LpDuvIkDwuzESnV8mQEDGJ6GBjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwK7yNwW2N+RXvftXAAAAAElFTkSuQmCC', 'sha256': '6cbbbf598f1e9e8ea0e28b36880ea6fb175f45ca5de136f071cfc76f1ff5bff5'}, 'data/binding-v1/images/binding-evaluation-0109-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFV0lEQVR4nO3bsXEbVxRAUcgj9oNAqTpAJS7IlbADpwpQjALOMHGAEYyBaYiUQN7dt+dECiTs3yH3zvt/oU9PT993AIU/kqsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZz7s12++/7EY7Hr/VS4B3ZAICMgIEZNa9BTv7+6/n3SBf/3yolwAfwQQEZIZMQBC+5fCu4JcJEPzuq9XLfyJGbyJAcM+vdJw/R4leQ4DYutvpeXz4yQuBw/PzjY+VodsEiO16MT0/Lc7tv3/VIxm6TYDYoruk50WnD5GhVxIgtl6fu3TnyvkzL0u033/RoCsCxIZ8QHpuD0RGoSu+iMhWfHx9/u9a4/8T9esJEJtw+cw/Pjx8ZH1evKgGnQgQm6tPuBINuiJADLec+vx3DfvNN0iAmGxp9TnRoDMBYqxl1udEg04EiJmWXJ+TRw0SIEZafn1OHjffIAFisiXXZy0rfFcCxDTnUWItz/bjj3VucAgSIEZZ+zO8X/n630qAmGkt488aV3tHAsQcq9t8XdrmRkyAgIwAMcSqx5/NDkECBGQEiAnmjQz7cXf0IgFilPXuv2as/60ECMgIEKs34Ph5s0fRAgRkBAjICBCQESDWbfZByX703QkQc8w4gZ53L7cJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0AMcXh+3k1xGHQvtwkQ63Y8ftvNdRx9dwIElAQIyAgQkBEg5hyUzDi7Pfy4i/EHQAIElASIUdY+BB1Wvv63EiAmmLdbOY67oxcJEJARIIYYcBR92NLx84kAARkBYo5VD0GH7Y0/AsRY62rQYVWrvSMBYpS1jw/Hla//rQSIaVa3ETtscvN1IkBMtvwGHRa/wnclQAx0OUos+Qk/XKxtg+OPADHW8ht02Hx9BIjJltwg9TkRICZbZoPU50yAGG5pDVKfSwLEfMtpkPpcESC22KCPz9DVRdXn5NPT0/fdau33X3aj+TV911+Yx4eHj7nuVe/8WM8+//tHmO705J8zdOrCu2ZIem4TIDbnePx2OQqdG3HHEr24xVOfaVswuPsW/jczJD1vIkBs3e2TxJ/26PZ5tvTcJkDwLi80pOc1BAjuWSLdeRMBgt+Nkej8MgECMr4JDWQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQJ2lX8AxUE3RLW0llUAAAAASUVORK5CYII=', 'sha256': '74768c15d106809ac933914f64764f84b3f8d0df7b73b4fb153ddd78f41252f3'}, 'data/binding-v1/images/binding-evaluation-0109-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFWklEQVR4nO3bMW4bVxRAUTow98PCgCtvw4vIgrIIbyOVARdcjAsBbFIQZghGoUWb0uW8OadSIXH+QJqL9z+pd09P3zcAhT+SqwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZn3myXb7T5sRtvvv9VLgFdkAgIyAgRklr0FO/n7r8NmkE9/buslwFswAQGZIRMQhO9yeK/glwkQ/O5bq+c/IkY3ESC450c6Tq+jRC8hQKzd9fRsP365/uOHr5+vvKwMXSdArNez6flpca5//0WPZOg6AWKN7pKeZx1fRIZeSIBYe33u0p0Lp9c8L9Fu90GDLggQK/IG6bk+EBmFLvggImvx9vX5v2uN/yfqlxMgVuH8md9+/PKW9Xn2ohp0JECsrj7hSjToggAx3OPU579r2K2+QQLEZI9WnyMNOhEgxnrM+hxp0JEAMdMj1+doq0ECxEiPX5+j7eobJEBM9sj1WcoKX5UAMc1plFjKs739sc4VDkECxChLf4Z3C1//rQSImZYy/ixxtXckQMyxuM3XuXVuxAQIyAgQQyx6/FntECRAQEaAmGDeyLAbd0fPEiBGWe7+a8b6byVAQEaAWLwBx8+rPYoWICAjQEBGgICMALFssw9KdqPvToCYY8YJ9Lx7uU6AgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAWKIw9fPmykOg+7lOgFi2fb7b5u59qPvToCAkgABGQECMgLEnIOSGWe3hx93Mf4ASICAkgAxytKHoMPC138rAWKCebuV/bg7epYAARkBYogBR9GHNR0/HwkQkBEg5lj0EHRY3/gjQIy1rAYdFrXaOxIgRln6+LBf+PpvJUBMs7iN2GGVm68jAWKyx2/Q4eFX+KoEiIHOR4lHfsIPZ2tb4fgjQIz1+A06rL4+AsRkj9wg9TkSICZ7zAapz4kAMdyjNUh9zgkQ8z1Og9TnggCxxga9fYYuLqo+R++enr5vFmu3+7AZzZ/pq/7BbD9+eZvrXvTOr/Xk/b9fwnTHJ/+UoWMXXjVD0nOdALE6+/2381Ho1Ig7lujZLZ76TNuCwd238L+ZIem5iQCxdtdPEn/ao+vn2dJznQDBq7yhIT0vIUBwzxLpzk0ECH43RqLzywQIyPgkNJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjYVP4BTg835GL2zYMAAAAASUVORK5CYII=', 'sha256': 'f9b12ce2d625f521ed0f1bb041b13aa70905c4732db5c718952e947ecd773e67'}, 'data/binding-v1/images/binding-evaluation-0110-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFbklEQVR4nO3bTWobSRiAYWdIHyegRWBukovkHLnI3CSQRUOukoVBm4ERI4TlyD+R/aq6nmdlsC19De7X1VXow/39rzuAwl/JuwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5mP31jCY3e7z5R9Y1x/vNctGfLi//1XPAKMW5zI9epIAwfXTc0qGLhAgeG539l8/Xf715dvPC99VonMCBJfS82R0XhojGTolQPBIfV7dneeUSIOOBIipvWl6TsnQowSIeT2ozxul50KG1ukbJEBM6rQ+75Ce32VonbtBswToiqeq72/yv9GN1edAgw58FIPp5PV58L67kf87/iEfxWAux7u9Ss/R/r8BDkuh3e7znOug6QK0/P3P3SD237/UI2zN7dTnaP/108wN8gjGLG6wPg/mmfBZTICYwij39m6QOa9FgNi+W9h1vmw/6560ADGR26zP7c/2dgSIjbvZrZ9zE24GCRBbNu6dvBt28hcRIKZw+8ufsea8FgFis0ZfROwGn/85BIjtG2tZsR9q2j8kQEBGgNimgQ6/Zj4OEyAgI0Bs0JYWDrsNXcs5AWLLRnz+Gn3yFxEgICNAQEaAgIwAsTVDH8DPdhgvQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjACxNev64/DF8u3n3ciW/+c/XtH2CBCQESAgI0BARoDYsnG3gZZhJ38RAWKDtrRru27oWs4JEJARILZp6MP4ZYID+AMBAjICxPaNtQhahpr2DwkQmzX688s6+PzPIUBMYZRlxTLInNciQGzZuIuIddjJX0SA2LiBjsOWaQ6/jgSIidxyg5Ybnu3tCBDbd7qguM37fDmZap7ljwAxi1Hu6nWQOa9FgJjFzW4GLfNt/RwJEBO5wQYtE9fn7u7uYz0AvKt1/bHbfT7e+fuvn6pJlln3fU5ZATGdW9iTVp8DAWJGbYPU58gjGLM/i73n49iD2K1z10eAmNppg946Q+frrHX6+ggQsztU4E0zJD0XCBA8XAqdVuPVJfrd1pL6nBIgeHwpdN6RJ2N0eT9bes4JEDzSiPMSvfq8THcumC5A++9f6hEYeEH0ihfhgukCBK8uyJM9UpyXEiB4Ln25ug/397+u/6oAz+CjGEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIC7yr9RfkOin56R1QAAAABJRU5ErkJggg==', 'sha256': '1a1729841567a4cf98547947e7fd8a4c76fe7b2bf70dd16f2573147ea7a9cb22'}, 'data/binding-v1/images/binding-evaluation-0110-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFdUlEQVR4nO3bTWobSRiAYWWI7qNFwKtcI4fIgeYQvsasAlnoMFkYejMMYoSwHPknsl9V1/OsguOoP4H7dXVV9Onh4dcGoPBXclUAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyHzuLg2D2e2+XP6G/f7nR82yEp8eHn7VM8CoxblMj54lQHD99JySoQsECF7ane3d/eV/vvz4duFvleicAMGl9DwbndfGSIZOCRA8UZ83d+clJdKgIwFiau+anlMy9CQBYl6P6vNO6bmQof30DRIgJnVanw9Iz+8ytJ+7QbME6Iqnqh9v8p/RldXnQIMOfBSD6eT1eXTd3ci/Hf+QADGX492+vbuv6nM+wG7WBk33WbB//l42g/j6fVuPsOb6bG7D9u7+8Di2232Z8FnMCohZ3GB9DmZeBwkQUxjl3t4NMue1CBDrdwu7zpdtZ92TFiAmcpv1uf3Z3o8AsXI3u/VzbsLNIAFizca9k3fDTv4qAsQUbn/5M9ac1yJArNboi4jd4PO/hACxfmMtK7ZDTfuHBAjICBDrNNDh18zHYQIEZASIFVrTwmG3ovdyToBYsxGfv0af/FUECMgIEJARICAjQKzN0Afwsx3GCxCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNArM1+//Pwh+XHt83Ilv/nP76j9REgICNAQEaAgIwAsWbjbgMtw07+KgLECq1p13a/ovdyToCAjACxTkMfxi8THMAfCBCQESDWb6xF0DLUtH9IgFit0Z9f9oPP/xICxBRGWVYsg8x5LQLEmo27iNgPO/mrCBArN9Bx2DLN4deRADGRW27QcsOzvR8BYv1OFxS3eZ8vJ1PNs/wRIGYxyl29H2TOaxEgZnGzm0HLfFs/RwLERG6wQcvE9REgpm5Qm6HlZIA56yNAzOgW9qSn3XV+RICYUdsg9TkSICb1qEEfk6FHF9rPXR8BYmqP7v93zdD5i++nr89ms/lcDwClQwV2uy/Hrxwysb27v9YlzqMmPUcCBP8V4bRBp9V4c4l+t5hSn1MCBE8vhc478myMLj/BSc85AYInGnFeojefl+nOBdMF6Ov3bT0CAy+I3vAiXDBdgODNBXm2R4rzWgIEL6UvV/fp4eHX9V8V4AX8R0QgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DApvIvXCxN/Uk094IAAAAASUVORK5CYII=', 'sha256': '7a32bbc63db469e63ad8e0954fbc1137101d2daf78fffb779ecf92705ebd9d94'}, 'data/binding-v1/images/binding-evaluation-0110-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFbklEQVR4nO3bTWobSRiAYWdIHyegRba5ULY5R7Zzodlm0ZCrZGHQZmDECGE58k9kv6qu51kZbEtfg/t1dRX6cH//6w6g8FfyrgACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+di9NQxmt/t8+QfW9cd7zbIRH+7vf9UzwKjFuUyPniRAcP30nJKhCwQIntud/bdPl399+f7zwneV6JwAwaX0PBmdl8ZIhk4JEDxSn1d35zkl0qAjAWJqb5qeUzL0KAFiXg/q80bpuZChdfoGCRCTOq3PO6Tndxla527QLAG64qnq+5v8b3Rj9TnQoAMfxWA6eX0evO9u5P+Of8hHMZjL8W6v0nO0/2+Aw1Jot/s85zpougD98/f+bhBfvi71CFtzO/U52n/7NHODPIIxixusz4N5JnwWEyCmMMq9vRtkzmsRILbvFnadL9vPuictQEzkNutz+7O9HQFi42526+fchJtBAsSWjXsn74ad/EUEiCnc/vJnrDmvRYDYrNEXEbvB538OAWL7xlpW7Iea9g8JEJARILZpoMOvmY/DBAjICBAbtKWFw25D13JOgNiyEZ+/Rp/8RQQIyAgQkBEgICNAbM3QB/CzHcYLEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0Bszbr+OHyxfP95N7Ll//mPV7Q9AgRkBAjICBCQESC2bNxtoGXYyV9EgNigLe3arhu6lnMCBGQEiG0a+jB+meAA/kCAgIwAsX1jLYKWoab9QwLEZo3+/LIOPv9zCBBTGGVZsQwy57UIEFs27iJiHXbyFxEgNm6g47BlmsOvIwFiIrfcoOWGZ3s7AsT2nS4obvM+X06mmmf5I0DMYpS7eh1kzmsRIGZxs5tBy3xbP0cCxERusEHLxPW5u7v7WA8A72pdf+x2n493/v7bp2qSZdZ9n1NWQEznFvak1edAgJhR2yD1OfIIxuzPYu/5OPYgduvc9REgpnbaoLfO0Pk6a52+PgLE7A4VeNMMSc8FAgQPl0Kn1Xh1iX63taQ+pwQIHl8KnXfkyRhd3s+WnnMCBI804rxErz4v050LpgvQl69LPQIDL4he8SJcMF2A4NUFebJHivNSAgTPpS9X9+H+/tf1XxXgGXwUA8gIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEHBX+RetUkI0m9wASAAAAABJRU5ErkJggg==', 'sha256': 'ebaf34dcdba8a0a5a3d52ade76f6ae0cf81387b64dc2085f15c17e883d9aad13'}, 'data/binding-v1/images/binding-evaluation-0111-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVElEQVR4nO3dP26aSRyAYWeVHGclirRcaNs9R9pcKG0KpFzFhaU0q8haC4HBdoh5mZnnKf2P+WR9r34zYPPh4eH+DqDwV/KoAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEDmY/fQcKM2m8+v+bLd7vv7r2VyHx4e7us1wDDROUWMfo8AsbQLu3NMid5EgFjU+fR8+/rz/Ldv//l05rMy9EoCxHKeTc+LxfmNHsnQiwSIpdNzYXdeUyIZOkOAWMIV0rNPhl5JgFiuPu+anjMZ0qBjAsRC9blaek5lSIMOCBDTqgafY0ahU/wpBnO6nfocP/off/HRuExATCjfdp1iO3bABMRsbrY+B+vZmIMEiMnccn0eadA+AWJOt1mf21/blQkQ83gaKG7/Dn9a4WbtIUiAmMS4d/Jm2JVfbulnwab5xXs+5faPfp61Xf5JMRMQUxmoPnejrfY9CBDDG+jo59i3tQ+D/E/oX37++/fdgD59+VEvAS5iAmJsQ48/j1YeggQIyAgQA5tg/Fl8CBIgICNAjGqa8WflIUiAgIwAARkBYkiT7b+W3YUJEJARICAjQEBGgBjPCkckmwWuUYAY20wn0LNe0XkCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBAD239r4zlsp7ui8wSI8azwNuq7Ba5RgICSAAEZAQIyAsTYRyQzndpu/7+WRQ6ABAgoCRCQESBGNdkubLve/kuAgJIAMbBphqDtkuOPAAElAWJsEwxB21XHHwECSgLE8IYegrYLjz8CxGzGatB2qNW+BwFiBvvjwyh39XZvnWuOPwLEPMa9h3fDrvxyAsQ8BjoMWvzo54kAMadbbtAtr+3KBIip3P5hkKOffQLEbG65Qepz4OPhB2B8u933zebz/j3/7evPdkkHKVSfRyYg5nRwh7ejkPqcIkBMa7f7fgvbsYNtl/rsswVjcuF2zODzIgFirQZdJ0PH05b6PEuAWMLj/X+FDEnPmwgQCzmVoctL9OwBk/S8SIBYznGGjgvyYo/OH2lLzysJEIt6NkOXP2UmPW8iQCztqRenSvTWn8ObCBAcFuSVMRKdywkQHFKWq/FKaCAjQEBGgICMM6BfPn35US8BVmQCAjICBGQ+PDzcd48OLM0EBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgrvIf7GdTOvHJxpwAAAAASUVORK5CYII=', 'sha256': 'db35bb53234a64bc05844261d682b2cb0aa0d46d3ba1c35e07d3bb17b8d2c8ce'}, 'data/binding-v1/images/binding-evaluation-0111-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVElEQVR4nO3dPW5bRxhAUTkwl2OAhVtvyK3XkTYbSuuCQLaiQoCawBAiEKRISWbEy5k5p9Qf50F4F98MKfHTw8P9HUDhj+RRAQQIKAkQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM5+6h4UZtt1/f8mW73c+PX8vkPj083NdrgGGic4oY/R4BYmkXdueYEr2LALGo8+l5/PHl/Ldv/vznzGdl6I0EiOW8mJ5Xi/MbPZKhVwkQS6fnwu68pUQydIYAsYQrpGefDL2RALFcfT40PWcypEHHBIiF6nO19JzKkAYdECCmVQ0+x4xCp/hTDOZ0O/U5fvT//cVH4zIBMaF823WK7dgBExCzudn6HKxnaw4SICZzy/V5okH7BIg53WZ9bn9tVyZAzON5oLj9O/x5hdu1hyABYhLj3snbYVd+uaWfBZvmF+/5lNs/+nnRZvknxUxATGWg+tyNttqPIEAMb6Cjn2OPax8G+Z/Qv/z91+PdgL5939RLgIuYgBjb0OPPk5WHIAECMgLEwCYYfxYfggQIyAgQo5pm/Fl5CBIgICNAQEaAGNJk+69ld2ECBGQECMgIEJARIMazwhHJdoFrFCDGNtMJ9KxXdJ4AARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAsTA9t/aeA6b6a7oPAFiPCu8jfpugWsUIKAkQEBGgICMADH2EclMp7ab/65lkQMgAQJKAgRkBIhRTbYL26y3/xIgoCRADGyaIWiz5PgjQEBJgBjbBEPQZtXxR4CAkgAxvKGHoM3C448AMZuxGrQZarUfQYCYwf74MMpdvdlb55rjjwAxj3Hv4d2wK7+cADGPgQ6DFj/6eSZAzOmWG3TLa7syAWIqt38Y5OhnnwAxm1tukPoc+Hz4ARjfbvdzu/26f88//vjSLukgherzxATEnA7u8HYUUp9TBIhp7XY/b2E7drDtUp99tmBMLtyOGXxeJUCs1aDrZOh42lKfFwkQS3i6/6+QIel5FwFiIacydHmJXjxgkp5XCRDLOc7QcUFe7dH5I23peSMBYlEvZujyp8yk510EiKU99+JUid77c3gXAYLDgrwxRqJzOQGCQ8pyNV4JDWQECMgIEJBxBvTLt++begmwIhMQkBEgIPPp4eG+e3RgaSYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAF3lX8BXrNadG7leMMAAAAASUVORK5CYII=', 'sha256': '4cb4295879d6b04f447ddf80714cf3bf24005db7725a067e8a2eb2c736433123'}, 'data/binding-v1/images/binding-evaluation-0111-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRElEQVR4nO3dPU5jSRhA0WbUsx8HnXpDs6DeECmBF9MBUicToEGWjQ20B99XVeeE/Lme4F19VQbz8Pz86xtA4a/kUQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzPfuoWGjdrsfH/mww+Hp69cyuYfn51/1GmCY6FwiRn9GgFjajd05p0SfIkAs6np6Hn/+vv7p+3/+vvJeGfogAWI5b6bn3eL8QY9k6F0CxNLpubE7HymRDF0hQCzhDuk5JkMfJEAsV58vTc+VDGnQOQFiofrcLT2XMqRBJwSIaVWDzzmj0CX+FIM5bac+54/+v//y0bhMQEwo33ZdYjt2wgTEbDZbn5P17MxBAsRktlyfFxp0TICY0zbrs/213ZkAMY/XgWL7d/jrCndrD0ECxCTGvZN3w678dks/CzbNN97zKds/+nnTfvknxUxATGWg+nwbbbVfQYAY3kBHP+ce1z4M8prQo/7gvvuaWLB9JiDGNvT482LlIUiAgIwAMbAJxp/FhyABAjICxKimGX9WHoIECMgIEJARIIY02f5r2V2YAAEZAQIyAgRkBIjxrHBEslvgGgWIsc10Aj3rFV0nQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAAxsPn+M9p+uiu6ToAYzwr/Rv2wwDUKEFASICAjQEBGgBj7iGSmU9v9f9eyyAGQAAElAQIyAsSoJtuF7dfbfwkQUBIgBjbNELRfcvwRIKAkQIxtgiFov+r4I0BASYAY3tBD0H7h8UeAmM1YDdoPtdqvIEDM4Hh8GOWu3h+tc83xR4CYx7j38GHYld9OgJjHQIdBix/9vBIg5rTlBm15bXcmQExl+4dBjn6OCRCz2XKD1OfE99M3wPgOh6fd7sfxPf/483e7pJMUqs8LExBzOrnD21FIfS4RIKZ1ODxtYTt2su1Sn2O2YEwu3I4ZfN4lQKzVoPtk6HzaUp83CRBLeLn/75Ah6fkUAWIhlzJ0e4nePGCSnncJEMs5z9B5Qd7t0fUjben5IAFiUW9m6PanzKTnUwSIpb324lKJPvt1+BQBgtOCfDBGonM7AYJTynI3fhMayAgQkBEgIOMMaIuvGgOLMAEBGQECMg/Pz7+6RweWZgICMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEPCt8i+zcFKBn6tYtgAAAABJRU5ErkJggg==', 'sha256': '6e25e7d4009f25e701cb114497ad3c465bea6332ff2ea10bd613c1f5de018a65'}, 'data/binding-v1/images/binding-evaluation-0112-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHUlEQVR4nO3cO24jRxRAUY5h7oeBAG9iYi/SsTZhYAItZgIBShwQ4NC0Ph5Bo9v1+pxIgURWQ+iLV6Wmvjw+fj8AFH5L3hVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+b17a3i/0+nuMNTDw7fDbpiAgIwAARlbMNZ2/OOvwwhPf/952B8TEJARICAjQEBGgICMAAEZfwXbxPNyu3r2DC4EaBMP6V7/iBixHwK0uQ8HXF5HiRhPgH55eu6Px9d//OvT0ysvK0MMJkAfn543i/P699/0SIYYTIDK9Dzr/CIyxB4I0AfU50O6c+PymtclOp3uNIhJBGhz6Xl9IDIKMYkHETddn5fea/C/42JXBOj/ur7n74/Hz6zPs2+qQQwgQO+pT7gSDWISAVqpPv9dgwaxNAFarD5nGsQMArRefc40iAEEaMn6nGkQqxOgVetzpkEsTYDesOX6rLJCeIkAPeMySqxyb1/WaQhiLQJ0a/V7ePX1sysC9KJVxp8VVwtnArT25uuajRjLESAgI0BDxp8zQxBrESAgI0CHqSPDvCtiHgG6te7+a8b62RUBAjICNOT4+ZqjaFYhQEBGgICMAAEZARp+UDL76lidAP0w4wR63rUwmAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjIC9MPXp6fDFJOuhcEE6PDw8O0w1+yrY3UCBGQECMgIEJARoH8dlMw4u71chQMgNk6AgIwA3Vp9CFp9/eyKAB2m7lbmXRHzCBCQEaBRR9GOn1mLAAEZAZozBBl/WI4AvWitBq21WjgToFurjw+rr59dEaAJGzGbLxYlQG/YfoO2v0J4iQA973qU2PIdfr024w/L+fL4+L1ew3adTneXr++Px8PG7Lk+17+aYR729Ks0Aa06B+25PowhQEs2SH2YwRZsvb2Y+jCGCWixOUh9mMQE9P6Dz08ehW7Cpz4MYAL6CTf3/GeOQurDSCagrY9C0sNgAvSRz6F8YImeHa/Uh2EEaHMZkh72Q4B++VO5b/bo9bMk6WEwAdruhwOkh/EEaHMl0h32Q4A2ESPRYZ8ECMh4EBHICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIOBQ+QdLPy40ok7LhgAAAABJRU5ErkJggg==', 'sha256': '3ae2dd8a5b06e18de9aa45b2bbc76e2c22ed76417fbff6190c749d03d9f1d043'}, 'data/binding-v1/images/binding-evaluation-0112-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHklEQVR4nO3cPU4jWRSAUfdo2I8DJDYxcS+SeDbRUgcspgMkkgksuT0efqYRzVfv1jkRAdivhOrTfY8yXx4ffxwACn8k7wogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjI/Nm9Nbzf8Xh7GOrh4fthN0xAQEaAgIwtGGv7++bmMMJfT0+H/TEBARkBAjICBGQECMgIEJDxV7BNPC+3q2fP4EyANvGQ7uWPiBH7IUCb+3DA+XWUiPEE6Len5+bu/vUff/r29ZWXlSEGE6CPT8+bxXn9+696JEMMJkBlep51ehEZYg8E6APq8yHduXJ+zcsSHY+3GsQkArS59Lw+EBmFmMSDiJuuz0vvNfjfcbErAvR/Xd7zN3f3n1mfZ99UgxhAgN5Tn3AlGsQkArRSff67Bg1iaQK0WH1ONIgZBGi9+pxoEAMI0JL1OdEgVidAq9bnRINYmgC9Ycv1WWWF8BIBesZ5lFjl3j6v0xDEWgTo2ur38OrrZ1cE6EWrjD8rrhZOBGjtzdclGzGWI0BARoCGjD8nhiDWIkBARoAOU0eGeVfEPAJ0bd3914z1sysCBGQEaMjx8yVH0axCgICMAAEZAQIyAjT8oGT21bE6Afppxgn0vGthMAECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQE6Kenb18PU0y6FgYToMPDw/fDXLOvjtUJEJARICAjQEBGgP51UDLj7PZ8FQ6A2DgBAjICdG31IWj19bMrAnSYuluZd0XMI0BARoBGHUU7fmYtAgRkBGjOEGT8YTkC9KK1GrTWauFEgK6tPj6svn52RYAmbMRsvliUAL1h+w3a/grhJQL0vMtRYst3+OXajD8s58vj4496Ddt1PN6ev765uz9szJ7rc/mrGeZhT79KE9Cqc9Ce68MYArRkg9SHGWzB1tuLqQ9jmIAWm4PUh0lMQO8/+PzkUegqfOrDACagX3B1z3/mKKQ+jGQC2vooJD0MJkAf+RzKB5bo2fFKfRhGgDaXIelhPwTotz+V+2aPXj9Lkh4GE6DtfjhAehhPgDZXIt1hPwRoEzESHfZJgICMBxGBjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIOlX8Aow8wFMM1b5YAAAAASUVORK5CYII=', 'sha256': '14abe47d4aba0b983bf9962a424a6f9b5951716618c01d20fd118f623e0f0d51'}, 'data/binding-v1/images/binding-evaluation-0112-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFG0lEQVR4nO3cMVIbSxRA0bHL2o8ClsFyWQaBFuOAKiUOpkrW10fGpoA73XNORABSTzCX182Iby8vPxeAwvfkXQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZH91bw/sdjw/LpE6n52U3TEBARoCAjC0YY3s6HJYpPJ7Py/6YgICMAAEZAQIyAgRkBAjI+CvYJp6X29WzZ3AhQJt4SPf6R8SI/RCgzX044PI6SsT0BOjT0/Pmk3L3nkBbX1aGmJgAfXx6/vXZ3Jvvv+mRDDExASrT86r1RWSIPRCgD6jPZ3wc6fKa1yU6Hh80iJkI0ObS8+eByCjETDyIuOn63Huvif8dF7siQH/r+p5/Ohy+/r9A3LypBjEBAXpPfcKVaBAzEaCR6vP/NWgQQxOgweqz0iDmIEDj1WelQUxAgIasz0qDGJ0AjVqflQYxNAF6w5brM8oK4R4BesVllBjl3r6s0xDEWATo1uj38OjrZ1cE6K5Rxp8RVwsrARp783XNRozhCBCQEaBJxp+VIYixCBCQEaBl1pFhvitiPgJ0a9z91xzrZ1cECMgI0CTHz9ccRTMKAQIyAgRkBAjICNDkByVzXx2jE6Df5jiBnu9amJgAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAvTb4/m8zGKma2FiArScTs/LvOa+OkYnQEBGgICMAAEZAfrPQckcZ7eXq3AAxMYJEJARoFujD0Gjr59dEaBl1t3KfFfEfAQIyAjQVEfRjp8ZiwABGQGaZwgy/jAcAbprrAaNtVpYCdCt0ceH0dfPrgjQDBsxmy8GJUBv2H6Dtr9CuEeAXnc9Smz5Dr9em/GH4fyoF7Bdp9Pz8fiwfv14Pj8dDsvGqM/Gfz3wJhPQqHOQ+jABARqyQerDHL69vPys1zCAy15sWZZ8L6Y+TMMENNgcpD7MxAT0zjno60ehm/CpDxMwAf2Dm3v+K0ch9WFKJqCtj0LSw8QE6GMa9OElenW8Uh8mI0Cby5D0sB8C9FkZ+vse/fksSXqYmAB9UYbeQXqYngBtrkS6w34I0CZiJDrskwABGQ8iAhkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIELJVfaP0vQ5xtD4UAAAAASUVORK5CYII=', 'sha256': '9d605bc36d975d2af9d7c89fc220217b1675dc14d7fb2488d2d9d9f964d01721'}, 'data/binding-v1/images/binding-evaluation-0113-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFSklEQVR4nO3cQU4bSRiA0Q4KxxmJRSRukovMOXIRbhIpC6RcJQukbGaBZFlgDGEcvq6q93ZI2F296E9/FTafHh5+bQCFq+SqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD53F0aGjc3X7Zh3d//2CZiAgIyAgRkbMFY1+9//9kGcf3t5zYjExCQESAgI0BARoCAjAABGQECMv4Mf4GPzE724VT4MAJ0gc/pH79EjODtBOjC3w86vI8SwasE6JX0XN/enX/57+9fz7ytDMEZSwfoZHpeLc7533/SIxmCMxYN0PP0/Gl3XnJ4n+MSyRCctFyA/l56Tr6tDMEZVyvX5/r27i/V58wlhv5vWHBZCwXo+Mn/gPQce3I5DYKFtmDPB59kGde3d4cdme0YLDEB7aQ+J69uFGJxkwco3Ha9xHYMlgjQk/pse6JBMHOA9lyfRxoE0wZo5/XZ/9rgA8wZoMNAsf8n/LBCQxALmjBA4z7J464c3me2AO3/6Oc5h0Esa7YADVefEVcLlzJVgAY6+nnOYRALmipAwFjmCdDQ488jQxCrmSdAwHAmCdAE488jQxBLmSRAwIhmCNA0488jQxDrmCFAwKAECMgMH6DJ9l+P7MJYxPABAsYlQEBGgIDM2AFa4YhkhXtkWWMH6GCmE+hZ7wimDRAwIgECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjKTBOj396/bXOa7I5gtQPf3P7bZrXCPLGvsAAFDEyAgI0BA5mqaI5KZTm0P9+IAiLl9rhcAmetvP+slrG74CQgY1wwBmmwXZv/FOmzBWI6y78cME9BMQ5Dxh6VMEiBgRPMEaIIhyPjDauYJEDCcqQI09BBk/GFBUwXo2FgNGmu1cCmzBeh4fBjlqT5ep/GHpcwWoKGf4XFXDu8zYYDGOgxy9MPK5gzQsT03aM9rgw8wbYD2fxjk6AemDdDOG6Q+sG3bp4eHX9vUbm6+HP94fXu3pZ6kUH1Y2cwT0MknvB2F1AfWmoBOjkLJHGTbBesGKNyOGXzgpLUC9LxBfztDz3d86gPrBujDMiQ98KpFA/RShv5/iU4ecksPnLR0gM5k6O09Ov9nNemBMwToTRl6B+mBVwnQhUukO/B2AnSBGIkOvI8AAZn5v4oB7JYAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQK2yn/mPzSB7WVeugAAAABJRU5ErkJggg==', 'sha256': '3d79653a0158d0634ef5457c3dcbde4995ee38b29cf44deaef7dd323521ef0bb'}, 'data/binding-v1/images/binding-evaluation-0113-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQklEQVR4nO3bQU5bSRRAUScKy4nEgGVkN1lHdtPLyAApW8kAiUkPkJAFjqFpwnW9OmeGhL/rD/7Vq7L96e7u9wGg8Dl5VwABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfOneGhrX1zeHZd3e/jwMYgICMgIEZGzB2Nc/V1eHRXy7vz9MZAICMgIEZAQIyAgQkBEgICNAQMbH8O/wldlhX06FDyNA7/A9/eOXiBG8ngC98++DHq+jRPAiAXohPfffv55/+dWPX2cuK0NwxtYBOpmeF4tz/v+f9EiG4IxNA/Q8Pf+1O3/yeJ3jEskQnLRdgP5eek5eVobgjC871+cvpefFDGkQbBeg4/p8QHrOZEiDYKNvQl9f34T1Ofm+T5YEe5ofoI/fdp3x5N01iM0N34JdwuDzhO0YbDEBXWB9/rQdS9cCmbEBuuT6PNAgGBugC6/P5a8NPsDMAD0OFJf/hD+u0BDEhgYGaN0ned2Vw9tMC9DlH/085zCIbU0L0HL1WXG18F5GBWiho5/nHAaxoVEBAtYyJ0BLjz8PDEHsZk6AgOUMCdCA8eeBIYitDAkQsKIJARoz/jwwBLGPCQECFiVAQGb5AA3bfz2wC2MTywcIWJcAARkBAjJrB2iHI5Id7pFtrR2gR5NOoKfeEYwNELAiAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMkMCdPXj12GWeXcE0wJ0e/vzMN0O98i21g4QsDQBAjICBGQ+jzkimXRq+3gvDoCY7Uu9AMh8u7+vl7C75ScgYF0TAjRsF2b/xT5swdiOsl+OCRPQpCHI+MNWhgQIWNGcAA0Ygow/7GZOgIDljArQ0kOQ8YcNjQrQsbUatNZq4b1MC9Dx+LDKU328TuMPW5kWoKWf4XVXDm8zMEBrHQY5+mFnMwN07JIbdMlrgw8wNkCXfxjk6AfGBujCG6Q+cDgcPt3d/T6Mdn19c/zn/fevh9STFKoPO5s8AZ18wttRSH1grwno5CiUzEG2XbBvgMLtmMEHTtorQM8b9Lcz9HzHpz6wb4A+LEPSAy/aNEB/ytD/L9HJQ27pgZO2DtCZDL2+R+c/VpMeOEOAXpWhN5AeeJEAvXOJdAdeT4DeIUaiA28jQEBm/k8xgIslQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAQ+VfQ4g/2vq/nQYAAAAASUVORK5CYII=', 'sha256': '48926f4645019be7af187a48043f599bd0c5e52f50579632bc26aedde45bd739'}, 'data/binding-v1/images/binding-evaluation-0113-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRklEQVR4nO3cMU4jSRiA0R403IdgJC4x8RySeC8x0gQcZgIkkg2QLAuMYVnD11X1XoaE3dVBf/qrsPn28PB3AyhcJVcFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMz37tLQuLn5sQ3r/v7PNhETEJARICBjC8a6/rm+3gbx8/Fxm5EJCMgIEJARICAjQEBGgICMAAEZf4a/wEdmJ/twKnwZAbrA5/SPXyJG8H4CdOHvBx3eR4ngTQL0Rnqub+/Ov/zx968zbytDcMbSATqZnjeLc/73n/VIhuCMRQP0Mj3/tTuvObzPcYlkCE5aLkCfl56TbytDcMbVyvW5vr37pPqcucTQ/w0LLmuhAB0/+V+QnmPPLqdBsNAW7OXgkyzj+vbusCOzHYMlJqCd1Ofk1Y1CLG7yAIXbrtfYjsESAXpWn21PNAhmDtCe6/NEg2DaAO28PvtfG3yBOQN0GCj2/4QfVmgIYkETBmjcJ3nclcPHzBag/R/9vOQwiGXNFqDh6jPiauFSpgrQQEc/LzkMYkFTBQgYyzwBGnr8eWIIYjXzBAgYziQBmmD8eWIIYimTBAgY0QwBmmb8eWIIYh0zBAgYlAABmeEDNNn+64ldGIsYPkDAuAQIyAgQkBk7QCsckaxwjyxr7AAdzHQCPesdwbQBAkYkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAZpIAPf7+tc1lvjuC2QJ0f/9nm90K98iyxg4QMDQBAjICBGSupjkimenU9nAvDoCY2/d6AZD5+fhYL2F1w09AwLhmCNBkuzD7L9ZhC8ZylH0/ZpiAZhqCjD8sZZIAASOaJ0ATDEHGH1YzT4CA4UwVoKGHIOMPC5oqQMfGatBYq4VLmS1Ax+PDKE/18TqNPyxltgAN/QyPu3L4mAkDNNZhkKMfVjZngI7tuUF7Xht8gWkDtP/DIEc/MG2Adt4g9YFt2749PPzdpnZz8+P4x+vbuy31LIXqw8pmnoBOPuHtKKQ+sNYEdHIUSuYg2y5YN0DhdszgAyetFaCXDfrsDL3c8akPrBugL8uQ9MCbFg3Qaxn6/yU6ecgtPXDS0gE6k6H39+j8n9WkB84QoHdl6AOkB94kQBcuke7A+wnQBWIkOvAxAgRk5v8qBrBbAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQI2Cr/AmsSNV7wAS44AAAAAElFTkSuQmCC', 'sha256': '8dab87e536e869ca49a05b9a78eacf4e9837ee03966cdd735e1d10f5e5ba7ca9'}, 'data/binding-v1/images/binding-evaluation-0114-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFGklEQVR4nO3cMU5bWRSAYWcUlhOJIu1saNqsI+1saFoKS9lKCiSaKSxZliEOSQz/9TvfV1HAexcZ/zr3YvvD4+P3HUDhr+SuAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfOxuDT93f/95t2n7/cNuMBMQkBEgIGMLxm3479+n3Yb8/c9dvYQlmICAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARlvxbj++7OHv78ZXk+Arv+hEKffLEZwgQC97YfRHK+gRPCcAP1yep6+fLr8s3dfv/3ogjIEpwToVen5aXR+9M1nMZIhOCVAl9LzS9150fEKpyWSITgQoJfr8+fpOXO44FmGNIjhBOi8PldPz4UMaRDDTX8h4v3953erz4t3OVsAjDI6QG+97brg7F4axExzt2DvP/icsR2DoRNQXp8X724OYpqhAVqkPuusARITA3QcNNZ55h9XYghilHEBWv8Zvv4K4VpmBWido5/nHAYx0KwALVuflVcFb2dQgBY8+nnOYRCjDAoQsJopAbqJ8efAEMQcUwIELGhEgG5o/DkwBDHEiAABaxIgILP9AN3c/uvALowJth8gYFkCBGQECMhsPEA3egB04BiIzdt4gICVCRCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkNl4gPb7h8MXd1+/7W7Ncc3H3wI2ZuMBAlYmQEBGgIDM9gN0o8dADoCYYPsBApYlQEBmRIBubhdm/8UQIwIErGlKgG5oCDL+MMeUAAELGhSgmxiCjD+MMihAp9Zs0JqrgrczK0CnY8Vqz/bT9Rh/GGJWgG7iub3+CuFaxgVozcMgRz/MNDFAp1Zo0AprgMTQAK1zGOToh8k+7qba7x/u7z+fVuDpy6f3XMBZ+NSHgT48Pn7fDXZs0MG7NUh9fu8B2p797Id+6BbsaL9/eP/t2Nm2a/ifIJPN3YIl2zGDD5yavgW7PO1fMUPPZyv1AQF61aHDb5foxT2d9MCBAP3y2edPY3ThIEl64JQAvdO/YKQHnhOgty2R7sAFAnT9GIkOvJIAAZnpL0QEQgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBOwq/wNJwhvMVT/2bQAAAABJRU5ErkJggg==', 'sha256': 'bdcd5dce85671e5ea66f3b7591270d026961fd6d9dc3e73b477f726fddb22cac'}, 'data/binding-v1/images/binding-evaluation-0114-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3csW1bSRRAUXph9aPAqTvYSlyQK9kOnCpgMQ4EKNmAAEFQMiXblO7wv3MiBQT/CBQv3oz4+enx8ecOoPBPclUAAQJKAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQ+d5eG193ff9lt2n7/sBvMBARkBAjI2IJxG/67u9ttyL9PT/USlmACAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRk3Ipx/fuzh9/fDG8nQNf/UojTB4sRXCBA7/tlNMdnUCJ4ToB+Oz0/vr9yH/PXb3e/ekIZglMC9Kb0vBqdXz34LEYyBKcE6FJ6fqs7Lzo+w2mJZAgOBOjl+vx9es4cnvAsQxrEcAJ0Xp+rp+dChjSI4aZ/EPH+/suH1efFq5wtAEYZHaD33nZdcHYtDWKmuVuwjx98ztiOwdAJKK/Pi1c3BzHN0AAtUp911gCJiQE6DhrrvPOPKzEEMcq4AK3/Dl9/hXAtswK0ztHPcw6DGGhWgJatz8qrgvczKEALHv085zCIUQYFCFjNlADdxPhzYAhijikBAhY0IkA3NP4cGIIYYkSAgDUJEJDZfoBubv91YBfGBNsPELAsAQIyAgRkNh6gGz0AOnAMxOZtPEDAygQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMhsPED7/cPhh6/f7na35rjm428BG7PxAAErEyAgI0BAZvsButFjIAdATLD9AAHLEiAgMyJAN7cLs/9iiBEBAtY0JUA3NAQZf5hjSoCABQ0K0E0MQcYfRhkUoFNrNmjNVcH7mRWg07FitXf76XqMPwwxK0A38d5ef4VwLeMCtOZhkKMfZpoYoFMrNGiFNUBiaIDWOQxy9MNkn3dT7fcP9/dfTivw4/vTRy7gLHzqw0CfHh9/7gY7NujgwxqkPn/2Am3PfvZLP3QLdrTfP3z8duxs2zX8T5DJ5m7Bku2YwQdOTd+CXZ72r5ih57OV+oAAvenQ4Y9L9OKeTnrgQIB+++zz1RhdOEiSHjglQB/0LxjpgecE6H1LpDtwgQBdP0aiA28kQEBm+gcRgZAAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAG7yv+U+R9p8K8MeAAAAABJRU5ErkJggg==', 'sha256': 'bfbc6d4c3edab78f13471cf0d4b90eca9860dbbebc4dae9e995bd46f54528a94'}, 'data/binding-v1/images/binding-evaluation-0114-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3cQU5bSRRAUacVlhOJAcvo3WQd2U0vg4GlbCUDJCY9sGRZhjgkMdzyf+eMGMD/hYyvXhW2Pz09/dgBFP5J7gogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMzn7tbwa/f3D7tN2+8fd4OZgICMAAEZWzBuw393d7sN+ff5uV7CEkxAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMt2Jc//3Zw9/fDG8nQNf/UIjTbxYjuECA3vfDaI5XUCJ4SYB+Oz3PX79c/tm7b99/dkEZglMC9Kb0/DI6P/vmsxjJEJwSoEvp+a3uvOp4hdMSyRAcCNDr9fn79Jw5XPAsQxrEcAJ0Xp+rp+dChjSI4aa/EPH+/uHD6vPqXc4WAKOMDtB7b7suOLuXBjHT3C3Yxw8+Z2zHYOgElNfn1bubg5hmaIAWqc86a4DExAAdB411nvnHlRiCGGVcgNZ/hq+/QriWWQFa5+jnJYdBDDQrQMvWZ+VVwfsZFKAFj35echjEKIMCBKxmSoBuYvw5MAQxx5QAAQsaEaAbGn8ODEEMMSJAwJoECMhsP0A3t/86sAtjgu0HCFiWAAEZAQIyGw/QjR4AHTgGYvM2HiBgZQIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQ2HqD9/vHwxd2377tbc1zz8beAjdl4gICVCRCQESAgs/0A3egxkAMgJth+gIBlCRCQGRGgm9uF2X8xxIgAAWuaEqAbGoKMP8wxJUDAggYF6CaGIOMPowwK0Kk1G7TmquD9zArQ6Vix2rP9dD3GH4aYFaCbeG6vv0K4lnEBWvMwyNEPM00M0KkVGrTCGiAxNEDrHAY5+mGyz7up9vvH+/uH0wo8f/3ykQs4C5/6MNCnp6cfu8GODTr4sAapz589QNuzn/3QD92CHe33jx+/HTvbdg3/E2SyuVuwZDtm8IFT07dgl6f9K2bo5WylPiBAbzp0+OMSvbqnkx44EKDfPvv8ZYwuHCRJD5wSoA/6F4z0wEsC9L4l0h24QICuHyPRgTcSICAz/YWIQEiAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBd5X8BaRvMgQIdagAAAABJRU5ErkJggg==', 'sha256': 'c7922d100d576c4ccbec8c07c4298cd8ca38693e242354c7b32d6253341d3fb2'}, 'data/binding-v1/images/binding-evaluation-0115-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUUlEQVR4nO3dS04bWxRAUfIU5kMj0mtlGhlEBpRBvGmkFSkNBkPDEp3XQEIWBhvz8a5771pdPj4lUVunygV82e3urgAK/ySvCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh87V56SDc3345/wu3t30vNAsP7stvd1TOMXZzj9AiOEKBP6c4hJYJDAnReen7/uj/+5d9/Xh/5qAzBPgE6nZ6T0Tk3RjIEDwToxfS8uTuvLJEMgQA9rc8HdudkiTSIxa3+HNAl63P4/T/8VjeMZd0N6MLpecIqBOtuQG19rEKw7ga0f7ZfPj1HViF7EKtZbgPaVH2ezGAPYjVrBWhr9XmgQSxroQBtsz4PNIg1LRSgzdZny1PBp1olQI9rxZbP88fZLEEsYokAjXg+jzgznGv+AG351s8hN4NYyvwBGqg+Y80J7zd5gEZfIkafH5YO0KBrxVjTwpvNHKAh3vl6iXfEWIH/isE8Joj17WK/DzjtBjTBz+KUxwJLBOjRiNdfo08Or+QSjAld//vf1VDu//y4WtKcG9B81yzzHRFMG6BprmJGnx+WDhCwZQIEZAQIyEwYoKEfgD7kkWgmNmGAgFEIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQmTBAj39V9/vP66vxPR7Fan8tmBVMGCBgFAIEZAQIyEweoNFvA40+P6wYoPnu1853RDBtgIAhzB+gca9ixp0cVg/QTNcsMx0LLBEgYPtmDtDQj0R7AJoVzBygfWM1aKxp4c0mD9Do68Po88PSARpxrRhlTni/+QO0v0Rs/9zen9D6w/TmD9CgZ/KIM8O5lgjQKO+IeeeL1awSoH3bbNA2p4JPtVCAtnwzyK0f1rRQgDbbIPVhWWsFaIMNUh9W9vVqPbe3f29uvu2f/79/3V9+jCf5Ux8WtNwG9OzZfvlVSH1g0Q1o/5y//CokPbD6BlStQuoD+9bdgJ69JbTfiA/chp7tmvqAAD1zOfZRJXppn5IeeCBAJzL0pCMnY3T8Ik56YJ8AvTZD77xJJD1wSIBO9OKlEp37fYBDAnReQU72SHHg9QToPPoCH2j154CAkAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARm/isGE7v/8qEfgVWxAQEaAgMyX3e6ue3VgaTYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBBwVfkfEBE29n65XR0AAAAASUVORK5CYII=', 'sha256': '9448504405536a7841248ca4d60f201847b7e57316a27349092b51df2193e39b'}, 'data/binding-v1/images/binding-evaluation-0115-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFRUlEQVR4nO3dS04bWxRAUXgK86GRYWS4GUYaDIaGpXReA8my7OAPGHbde9fqJqGOJWrnVLkwj7vd6wNA4b/kqAACBJQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMj+7QQ3p+/nn+L7y8/PmuWWB4j7vdaz3D2MU5T4/gDAH6ku6cUiI4JUC3pef309P5f/7r798zfypDcEiALqfnYnRujZEMwRsBejc9H+7OlSWSIRCg4/rcsTsXS6RBLG7154C+sz6nX//ut7phLOtuQN+cniNWIVh3A2rrYxWCdTegw7P9+9NzZhWyB7Ga5TagTdXnaAZ7EKtZK0Bbq88bDWJZCwVom/V5o0GsaaEAbbY+W54KvtQqAdqvFVs+z/ezWYJYxBIBGvF8HnFmuNX8AdryrZ9TbgaxlPkDNFB9xpoTPm/yAI2+RIw+PywdoEHXirGmhQ+bOUBDvPP1Hu+IsQK/FYN5TBDrl8V+HnDaDWiC78UpXwssEaC9Ea+/Rp8cruQSjAkN1+5fZ3+ZysTm3IDmu2aZ7xXBtAEa93/CyeaHpQMEbJkAARkBAjITBmjoB6BPeSSaiU0YIGAUAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZCYM0P5Tdef4kKf9q1jt04JZwYQBAkYhQEBGgIDM5AEa/TbQ6PPDigGa737tfK8Ipg0QMIT5AzTuVcy4k8PqAZrpmmWm1wJLBAjYvpkDNPQj0R6AZgUzB+jQWA0aa1r4sMkDNPr6MPr8sHSARlwrRpkTPm/+AB0uEds/tw8ntP4wvfkDNOiZPOLMcKslAjTKO2Le+WI1qwTo0DYbtM2p4EstFKAt3wxy64c1LRSgzTZIfVjWWgHaYIPUh5UtF6DTBlUZOjq0+rCgFQN0erZ/f4OOjqg+rOnHw6rezvnn55+HRfj99PTVx5UeWH0DqlYh9YFD625Aey8vf/Z70GEj7rgN/bNr6gMC9I/LsXuV6L19SnrgjQBdyNBRRy7G6PxFnPTAIQG6NkOfvEkkPXBKgC704r0S3fp1gFMCdFtBLvZIceB6AnQbfYE7Wv05ICAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BAxo9iMKEt/L4TrmEDAjICBGQed7vX7ujA0mxAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESDgofI/6nND0gAlzqQAAAAASUVORK5CYII=', 'sha256': 'c8a2dc3148c9583a69c537f1f6754612776b5ea04d8a65e3c9f9a57b0de44649'}, 'data/binding-v1/images/binding-evaluation-0115-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFUUlEQVR4nO3dPU4bURSAUYiS/VCkzQ6ykiwoK8kO0qZgMRSWaFIgIQuDjfnxN++9c9oQfC0xH3fGg329291dARS+JI8KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyHztHnpINzffj3/B7e2/S80Cw7ve7e7qGcYuznF6BEcI0Kd055ASwSEBOi89f3/fH//vP359O/KvMgT7BOh0ek5G59wYyRA8EKAX0/Pm7ryyRDIEAvS0Ph/YnZMl0iAWt/p9QJesz+H3//BL3TCWdTegC6fnCasQrLsBtfWxCsG6G9D+0X759BxZhexBrGa5DWhT9Xkygz2I1awVoK3V54EGsayFArTN+jzQINa0UIA2W58tTwWfapUAPa4VWz7OH2ezBLGIJQI04vE84sxwrvkDtOVLP4dcDGIp8wdooPqMNSe83+QBGn2JGH1+WDpAg64VY00LbzZzgIZ45eslXhFjBT4Vg3lMEOvbxf4ecNoNaIKfxSmfCywRoEcjnn+NPjm8klMwJvTn27HPJtmgn/eL/rKZcwOa75xlvmcE0wZomrOY0eeHpQMEbJkAARkBAjITBmjoG6APuSWaiU0YIGAUAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZCYM0OO76v74Ndi7Uj3r8Vms9m7BrGDCAAGjECAgI0BAZvIAjX4ZaPT5YcUAzXe9dr5nBNMGCBjC/AEa9yxm3Mlh9QDNdM4y03OBJQIEbN/MARr6lmg3QLOCmQO0b6wGjTUtvNnkARp9fRh9flg6QCOuFaPMCe83f4D2l4jtH9v7E1p/mN78ARr0SB5xZjjXEgEa5RUxr3yxmlUCtG+bDdrmVPCpFgrQli8GufTDmhYK0GYbpD4sa60AbbBB6sPKvl6t5/b2383N9/3j/+/v+8uP8SR/6sOCltuAnj3aL78KqQ8sugHtH/OXX4WkB1bfgKpVSH1g37ob0LOXhPYb8YHb0LNdUx8QoGdOxz6qRC/tU9IDDwToRIaedORkjI6fxEkP7BOg12bonReJpAcOCdCJXrxUonO/D3BIgM4ryMkeKQ68ngCdR1/gA61+HxAQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIONPMZjQz/vgUwZ4AxsQkBEgIHO92911jw4szQYEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQKuKv8BQ3U3HmboaKIAAAAASUVORK5CYII=', 'sha256': '63ee79f7feaa88aade5f2f626c40fd522cb24e737c045f958422b5296de80fcb'}, 'data/binding-v1/images/binding-evaluation-0116-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVUlEQVR4nO3aQWobWRRAUSVEy2nwIMvIbnodvRsvIwNDbyUDgyY9EAhhOXbaiX2r3j9nZpBVv5Dq8v5Hnx4ffxwACp+TqwIIEFASICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD50l16r+7uvv7Kyx4evr//WmDfPj0+/qjXMCc6PyNG8CwBesfu3FKivX+CW/Aw6FtkC/aWL+798fjyv387nV5420lfIPgdAvRL6Xm1OC+//kmPZAjOBOil9Pzf7vzM5X2uSyRDH+n091+HnTv+8+9hHAF63/Q8+7YyBGd+B/S0PvfH4zvV54VLjDwrhVetHqDrJ/8D0nPtyeU0iAWtuwW7HXySZdwfj5cdme0Yq1l0AtpIfZ69ulGIdawYoHDb9TO2Y6zp8+L1OWyJBrGatQK05fqcaRBLWStAG6/P9tcGf9ZCAboMFNt/wi8rNAQx2yoB2u+TvN+Vw6uWCND2j35uOQxiBUsEaHf12eNq4Q3mB2hHRz+3HAYx2/wAAZs1PEC7Hn/ODEEMNjxAwJZNDtCA8efMEMRUkwMEbNzYAI0Zf84MQYw0NkDA9gkQkJkZoGH7rzO7MOaZGSBgFwQIyAgQkBkYoBWOSFa4R1YwMEAXk06gp94Ri5scIGDjBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyEwO0LfT6TDLvDticQMD9PDw/TDdCvfICgYGCNgLAQIyAgRkPs8+Ipl0anu5FwdAjDEzQMAuCBCQGRugYbsw+y9GGhsgYPsmB2jMEGT8YarJAQI2bniABgxBxh8GGx4gYMvmB2jXQ5Dxh9nmB+javhq0r9XCGywRoOvxYS9P9fU6jT9MtUSAdv0M73fl8KpVArSvwyBHPyxioQBd23KDtrw2+LPWCtD2D4Mc/bCUtQK08QapD6tZLkC3DdpChp4sQ31YxIoBun3C2wY9ubr6sI4vh1Wdn/O7u6/nP7+dTvfH48cvw+DDyhadgLawHbPtgk+Pjz/qNfQuc9DFu05Dt5lTnw/7ZAd4GPRtWXcL9sJ27NKIP54h6YFrAvR6hn6/RM/u7KQHbMHeMrq/2qOXz5KkB84E6ENPEKQHrgnQR5RId+BZAvReMRIdeJUAAZnVf4gIhAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAYfKf0GVRSODhz/5AAAAAElFTkSuQmCC', 'sha256': '033b8874e2694137114eaa27eff282e32d6e22ec79fe734506b4025253b8353b'}, 'data/binding-v1/images/binding-evaluation-0116-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFU0lEQVR4nO3aQWobWRRAUSUkywl4YOid9Eayjmykd9KQgSBbycCQSQ8EQsiOnXZi3ar3zxnLql+o6vL+x+8eHr4fAArvk6sCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dBdeq/u7u5/5WPH49e3Xwvs27uHh+/1GuZE52fECJ4kQG/YnceUaO+/4BYcBz1FtmCveXB/fP70/J9//PLtma+d9ADB7xCgX0rPi8V5/vNXPZIhOBGg59Lzf7vzM+fvuSyRDN3Sx7/+Oezcj3//PowjQG+bnie/VobgRICu6/NG6XkxQxrEglYP0GV9bpCeZzKkQSxo3f+Evru7D+vz5HWvlgTjLRqg22+7nnF1dQ1iHStuwbYw+FyxHWNNy01AG6zPz7Zj6VrgFtYK0Jbrc6JBLGWtAG28PttfG/xZCwXoPFBs/w0/r9AQxGyrBGi/b/J+Vw4vWiJA2z/6ecxhECtYIkC7q88eVwuvMD9AOzr6ecxhELPNDxCwWcMDtOvx58QQxGDDAwRs2eQADRh/TgxBTDU5QMDGjQ3QmPHnxBDESGMDBGyfAAGZmQEatv86sQtjnpkBAnZBgICMAAGZgQFa4YhkhXtkBQMDdDbpBHrqHbG4yQECNk6AgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDM5AB9/PLtMMu8O2JxAwN0PH49TLfCPbKCgQEC9kKAgIwAAZn3s49IJp3anu/FARBjzAwQsAsCBGTGBmjYLsz+i5HGBgjYvskBGjMEGX+YanKAgI0bHqABQ5Dxh8GGBwjYsvkB2vUQZPxhtvkBurSvBu1rtfAKSwTocnzYy1t9uU7jD1MtEaBdv8P7XTm8aJUA7eswyNEPi1goQJe23KAtrw3+rLUCtP3DIEc/LGWtAG28QerDaj4c1nM8fr27u7985398/tQu6SqF6sMilpuAnnzD21FIfVjWogE6vedb2I5dbbvUh6WsuAXbyHbM4APvHh6+12vonRt09qYZejxtqc/NftkBjoOeltUnoMtf9PJhfaNpSHrgkgC9nKHfL9GTB0zSA7ZgrxndX+zR80fa0gMnAnTTEwTpgUsCdIsS6Q48SYDeKkaiAy8SICCz7n9CAzkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMCh8h9gBUoVQ5K/wAAAAABJRU5ErkJggg==', 'sha256': '0c68b5e2c67eaec4f16fbd8c10144cc6a319130265341ae27d0129592931c8b3'}, 'data/binding-v1/images/binding-evaluation-0116-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFVElEQVR4nO3aQWobWRRAUSVE+/Eg0JvIuBfZY2+iIQMvJgODJz0QCGE7UtqJfaveP2dmkFW/kOry/kefHh9/HAAKn5OrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPnSXXqv7u6+/srLHh6+v/9aYN8+PT7+qNcwJzo/I0bwKgF6x+68pER7/wS34GHQt8gW7C1f3Pvj8fq/f3t6uvK2k75A8DsE6JfSc7M411//rEcyBCcCdC09/7c7P3N+n8sSydBHOv71z2Hnnv79+zCOAL1vel59WxmCE78Del6f++Pxnepz5RIjz0rhptUDdPnkf0B6Lj27nAaxoHW3YC8Hn2QZ98fjeUdmO8ZqFp2ANlKfV69uFGIdKwYo3Hb9jO0Ya/q8eH0OW6JBrGatAG25PicaxFLWCtDG67P9tcGftVCAzgPF9p/w8woNQcy2SoD2+yTvd+Vw0xIB2v7Rz0sOg1jBEgHaXX32uFp4g/kB2tHRz0sOg5htfoCAzRoeoF2PPyeGIAYbHiBgyyYHaMD4c2IIYqrJAQI2bmyAxow/J4YgRhobIGD7BAjIzAzQsP3XiV0Y88wMELALAgRkBAjIDAzQCkckK9wjKxgYoLNJJ9BT74jFTQ4QsHECBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkJgfo29PTYZZ5d8TiBgbo4eH7YboV7pEVDAwQsBcCBGQECMh8nn1EMunU9nwvDoAYY2aAgF0QICAzNkDDdmH2X4w0NkDA9k0O0JghyPjDVJMDBGzc8AANGIKMPww2PEDAls0P0K6HIOMPs80P0KV9NWhfq4U3WCJAl+PDXp7qy3Uaf5hqiQDt+hne78rhplUCtK/DIEc/LGKhAF3acoO2vDb4s9YK0PYPgxz9sJS1ArTxBqkPq1kuQC8btIUMPVuG+rCIFQP08glvG/Ts6urDOr4cVnV6zu/uvp7+/Pb0dH88fvwyDD6sbNEJaAvbMdsu+PT4+KNeQ+88B5296zT0MnPq82Gf7AAPg74t627BrmzHzo344xmSHrgkQLcz9PslenVnJz1gC/aW0f1mj66fJUkPnAjQh54gSA9cEqCPKJHuwKsE6L1iJDpwkwABmdV/iAiEBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABh8p/t6dFIwYZ85kAAAAASUVORK5CYII=', 'sha256': '4e9d7be2ae7e46752ff9122014a13c3e8ea34c1ddc8019221e5d21f4bb18529a'}, 'data/binding-v1/images/binding-evaluation-0117-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMUlEQVR4nO3cMU4cSRiA0fZquQ+BJaK9hg/hA+0huMZGlhxwGAdIJA5Gmh3DeDysYb+uqvciApipFupPf9U0fHh8/LYBFP5I3hVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABmT+7t4Zr3d5+3Kb28PB1W5IJCMgIEJCxBWMk//z9tE3kr88329pMQEBGgICMAAEZAQIyDqHjR1eWfQAEBKh/Xu70R8SI1QjQjp7TPb6OErEIAXrH9Nzc3V/+8acvny68rAwxPQF6y/T8sjiXv/9Zj2SI6QnQG6Tntd35mePrnJZIhpiYAO0iPWdfVoaYnueA/mN9bu7u36k+F95i+v9KwWoE6Cqnd/7/kJ5Tz95Og5iJLdirB59kGTd398cdme0Y0zABDVCfs+9uFGICArTHbdfP2I4xGQG6qj7bnmgQ0xCgwepzoEHMQYAu2Wd99r82uJIAPXccKPZ/hx9XaAhiUAL0g3Hv5HFXzsoEaKSjn5ccBjE0ATpjlPqMuFo4JUDjHf285DCIQQkQkBGg4cefA0MQIxIgICNAM4w/B4YghiNAQGb1AE0z/hwYghjL6gECQgIEZJYO0GT7rwO7MAaydICAlgABGQECMusGaIUjkhWukaGtG6CjmU6gZ70iZiVAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAG1PXz5tc5nvipjVugF6ePi6zW6Fa2Ro6wYIyAkQkBEgILN0gI5HJDOd2h6vxQEQ+7d0gICWAAGZ1QM02S7M/ouxrB4gICRA8wxBxh+GI0BARoAmGYKMP4xIgICMAM0wBBl/GJQAnTFWg8ZaLZwSoH+djg+j3NWn6zT+MBwB+sG49/C4K2dlAjTwYZCjH0YnQJfsuUF7Xhtc6cPj47drv3clt7cfj1/f3N1vO7Pa0c/pr2NKDwv8Es8yAY13IL1afZiYAF3boD1k6Nky1IfR2YK9bvgPt2PPCqg+TECAxjgSMvgwJQHa+yhk8GFiAvRbn8W8a4ZenjqpD5MRoD1mSHpYhAC95ZMpv1misx+0SQ8TE6B3fEDulz26/NG+9DA9Adrjc7rSwyIEaEcl0h1WI0BxjESHlQkQkPG3YEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQI2CrfAc0cN7fo9L7IAAAAAElFTkSuQmCC', 'sha256': 'effa14c87f3761b73aa489ce816da395818736c983353d6de9f9e8064afd2878'}, 'data/binding-v1/images/binding-evaluation-0117-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFMUlEQVR4nO3bQU4bSRiAUWcU7uNFpFnlGnOIHCiHmGtkNRILHyYLJDazsORxDBgIMF9X1XsrFmBXC/XHX+Xm093dzx1A4Y/kXQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+dy9NbzUfv9lN7XD4Xa3JBMQkBEgIGMLxkhu/vx7N5H7f/7arc0EBGQECMgIEJARICDjEDp+dGXZB0BAgPrn5c5/RIxYjQBt6Dnd0+soEYsQoA9Mz4/v99d//Ou3mysvK0NMT4DeMz3PFuf691/0SIaYngC9Q3pe252nnF7nvEQyxMQEaBPpefRlZYjpCdBv1ueD0vNshjSImQjQq+vzP6TnSoY0iJl4EvoZ+/2XsD6Pvu/FkmBcArStbdcVF++uQUzAFuxJWxh8LtiOMRkT0DD1eWo7lq4F3kSABqvPkQYxBwG6Zpv12f7a4IUE6NJpoNj+HX5aoSGIQQnQL8a9k8ddOSsToJGOfh5yGMTQBOgRo9RnxNXCOQEa7+jnIYdBDEqAgIwADT/+HBmCGJEAARkBmmH8OTIEMRwBAjKrB2ia8efIEMRYVg8QEBIgILN0gCbbfx3ZhTGQpQMEtAQIyAgQkFk3QCsckaxwjQxt3QCdzHQCPesVMSsBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBGj39dvNbi7zXRGzWjdAh8PtbnYrXCNDWzdAQE6AgIwAAZmlA3Q6Ipnp1PZ0LQ6A2L6lAwS0BAjIrB6gyXZh9l+MZfUAASEBmmcIMv4wHAECMgI0yRBk/GFEAgRkBGiGIcj4w6AE6BFjNWis1cI5AfrP+fgwyl19vk7jD8MRoF+Mew+Pu3JWJkADHwY5+mF0AnTNlhu05bXBC326u/v50u9dyX7/5fT1j+/3u41Z7ejn/NcxpcMCv8RHmYDGO5BerT5M7HO9gO06HG5Pf3iP93w+Cl2kUH0YnS3Y64b/sEHqw3wEaIwjIdsupiRAWx+FDD5MTIDe9FnMh2bo4eG3+jAZAdpihqSHRQjQez6Z8sYSPfp5v/QwMQH6wAfknu3R9SeMpIfpCdAWn9OVHhYhQBsqke6wGgGKYyQ6rEyAgIx/RgUyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwK7yL/b5SY2Zne1fAAAAAElFTkSuQmCC', 'sha256': '6b313957a02af59a8a5a8e7fb0fb3ac6362824b62845fbe71e81dc0be26c2da5'}, 'data/binding-v1/images/binding-evaluation-0117-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFKElEQVR4nO3cQU4bSRiA0c5ouA+LSNwkh+QmkbLwYbJAyiYLSx7HOMQMkK+r6r0VC3C3hfrTX+WCT09P3zeAwj/JVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+be7NNzq/v7zNrXD4du2JBMQkBEgIGMJxkjuHh63ifz4+mVbmwkIyAgQkBEgICNAQMYmdHx0ZdkDICBA/Xm58x8RI1YjQDs6p3t6HSViEQL0gen546GV3x0DOb6sDDE9AXrP9Lz2mNzF91/0SIaYngC9Q3re63ju6XXOSyRDTEyAdpGeqy8rQ0zPOaD/WZ+7h8eP/ruk55eY/r9SsBoBusn5k/8X0nPu4nIaxEwswV49+CS3cffweFqRWY4xDRPQAPW5enWjEBMQoD0uu37HcozJCNBN9dn2RIOYhgANVp8jDWIOAvSSfdZn//cGNxKgS6eBYv9P+OkODUEMSoB+Me6TPO6dszIBGmnr5zmbQQxNgK4YpT4j3i2cE6Dxtn6esxnEoAQIyAjQ8OPPkSGIEQkQkBGgGcafI0MQwxEgILN6gKYZf44MQYxl9QABIQECMksHaLL115FVGANZOkBAS4CAjAABmXUDtMIWyQrvkaGtG6CTmXagZ31HzEqAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAdp+fP2yzWW+d8Ss1g3Q4fBtm90K75GhrRsgICdAQEaAgMzSATptkcy0a3t6LzaA2L+lAwS0BAjIrB6gyVZh1l+MZfUAASEBmmcIMv4wHAECMgI0yRBk/GFEAgRkBGiGIcj4w6AE6IqxGjTW3cI5AfrP+fgwylN9fp/GH4YjQL8Y9xke985ZmQANvBlk64fRCdBL9tygPd8b3OjT09P3W793Jff3n09f3z08bjuz2tbP+a9jSocFfolXmYDG25BerT5MTIBubdAeMnRxG+rD6CzBXjf8h8uxiwKqDxMQoDG2hAw+TEmA9j4KGXyYmAC96bOYD83Q810n9WEyArTHDEkPixCg9zyZ8sYSXf2gTXqYmAB94AG5P/bo5Y/2pYfpCdAez+lKD4sQoB2VSHdYjQDFMRIdViZAQMbfggEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYKv8BKUbN7ePJSrmAAAAAElFTkSuQmCC', 'sha256': '4a96cd859e74d5650a5318e2e5a1423c1b3997c950ba7a57ac5ed0df913d818d'}, 'data/binding-v1/images/binding-evaluation-0118-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFCUlEQVR4nO3dPW5aQRhAURLFy4nkIsvIbrKO7CbLcIGUraSw5CYFEiEQ/EPAd96bcyoXwBvJcPXNCD9/eHz8tQEofEyuCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMh86i4Nb3N//2Uzge32YTMNExCQESAgYwvG8vy4u9usztenp818TEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjL+GPWdblUz1U1e4JUE6J3uj3X4YDGCHQEK7su3fwUlYnICdJ30PH37/Pxz777/PPeCMsS0BOjy9LwYnXMPPoqRDDEtAbqkPm9Kz6nd008zpEHMRoDekJ7/7M6R/avtS2QUYja+B9TU55lXnuSfz4AJ6KzDCtwuPUeXOByFzEHMwATU1+ef1zIHMQMBGqU+p1fUIFZPgAaqz+l1NYh1E6Cx6nN6dQ1ixQRouPqcrkGDWCsBOjZCfUZbCdyIAP01Yoz2md+vxxDEKgnQYj7bS1knvJ4A/THa+DPyquAqZg/QsJuvQzZirNXsAQJCUwdoiQPFEtcM50wdoL2R919LWSFcQICAzLwBWsTx8yFH0azPvAECcgIEZAQIyEwaoMUdAO04BmJlJg0QMAIBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIykwZou33Y/XD3/edmOfar3a8fFm3SAAEjECAgI0BAZt4ALe4YyAEQ6zNvgICcAC1jCBp/hXCBqQO0xL3MEtcM50wdIKA1e4AWcRTt+Jm1mj1Ah8Zs0JirgqsQoMWMFUtZJ7yeAA29EbP5Yt0E6Ng4DRpnJXAjArQ5HTFG+OQfrsH4w1oJ0IgNUh8mIUDDNUh9mIcAjdUg9WEqAjRQg9SH2XyqFzCo7fbh/v7LYReevn2+3eWOMqc+TOLD4+Oveg3j2jdo50YNUp/Lfh1rtZ3pDWACevmtcDQKXatEp5u7qd55IEBv3o5dZVP2z3Ml9WFCtmDX2QK8GKNnDrOlh2kJUHkSIT1MToCCEukO7AjQO8VIdOCUAAEZ34QGMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0DApvIblhkRqMmvS7cAAAAASUVORK5CYII=', 'sha256': '14c9499a34deeb526f43035c7b7f8909ca0429bea0d95a5161df4ea22df82557'}, 'data/binding-v1/images/binding-evaluation-0118-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFCklEQVR4nO3cQW4TSRiAUYPwfbJA4hKs55CzziWQWPgwLCJ5wyJSj8cmiUmcfNVd761Ail2/hP2pqiL608PDrx1A4XOyKoAAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDzpVsa/s7d3dfdBA6Hn7tp2AEBGQECMo5grM/+27+7zTn++Gc3HzsgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZ/xn1gx5VM9VDXuBKAvRBz8c6/WExgkcCFDyXb3kHJWJyAnSb9Nzv98+/9vvx+NQbyhDTEqDXp+fF6Dz1w2cxkiGm5bdgr6nP/X7/V/W55uWTPHEdTtkBPecsCm+JzqXl3ZYNka0Qs7EDaurzzDvbCjEPAfqz0wq88cB1jbMlNIhJCNDL9fmwdTWI2QjQKPW5XFGD2DwBGqg+l+tqENsmQGPV53J1DWLDBGi4+lzOoEFslQCdG6E+o00C70SA/rfFGO07v8xjE8QmCdBqvttrmROuJ0D/GW37M/JUcBOzB2jYw9cpBzG2avYAAaGpA7TGDcUaZ4anTB2gxcjnr7VMCK8gQEBm3gCt4vr5lKtotmfeAAE5AQIyAgRkJg3Q6i6AHrkGYmMmDRAwAgECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjKTBuhw+Pn4h+/H4249lmmX+WHVJg0QMAIBAjICBGTmDdDqroFcALE98wYIyAnQOjZB408IrzB1gNZ4llnjzPCUqQMEtGYP0Cquol0/s1WzB+jUmA0acyq4CQFazbZiLXPC9QRo6IOYwxfbJkDnxmnQOJPAOxGg3eUWY4Rv/ukMtj9slQCN2CD1YRICNFyD1Id5CNBYDVIfpiJAAzVIfZiNAF3VoPfO0NkS6sMkPj08/KpnGNfd3dfTv97v9++xylnd1OfKf46tOsz0AfhSD7CCj8LyuV9KcZMSXe6qpvrkgSPYVS678MZD2R9frj5MyBHsNkeAF/dEzwRLepiWAJU3EdLD5AQoKJHuwCMB+qAYiQ5cEiAg47dgQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQI2FV+A547DwllEsYvAAAAAElFTkSuQmCC', 'sha256': '06e6eed3847a03a768692998bd017c035c3727ab67ea971b9a82771b70a4fc52'}, 'data/binding-v1/images/binding-evaluation-0118-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFDUlEQVR4nO3dTW5aSRhAURLFy4nkQaTeSW8k68hGspNIHiBlKxlY8qQHSDSB4B8CvvVenTPyAHglGa6+KuHnD4+PvzYAhY/JVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZT92l4W3u779sJrDdPmymYQICMgIEZGzBWJ67f75vVufpx7+b+ZiAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRk/DHqO92qZqqbvMArCdA73R/r8MFiBDsCFNyXb/8KSsTkBOg66Xn6+vn55959+3nuBWWIaQnQ5el5MTrnHnwUIxliWgJ0SX3elJ5Tu6efZkiDmI0AvSE9f9mdI/tX25fIKMRsfA+oqc8zrzzJP58BE9BZhxW4XXqOLnE4CpmDmIEJqK/PH69lDmIGAjRKfU6vqEGsngANVJ/T62oQ6yZAY9Xn9OoaxIoJ0HD1OV2DBrFWAnRshPqMthK4EQH6bcQY7TO/X48hiFUSoMV8tpeyTng9AfrfaOPPyKuCq5g9QMNuvg7ZiLFWswcICE0doCUOFEtcM5wzdYD2Rt5/LWWFcAEBAjLzBmgRx8+HHEWzPvMGCMgJEJARICAzaYAWdwC04xiIlZk0QMAIBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyEwaoO32YffD3befm+XYr3a/fli0SQMEjECAgIwAAZl5A7S4YyAHQKzPvAECcgK0jCFo/BXCBaYO0BL3MktcM5wzdYCA1uwBWsRRtONn1mr2AB0as0FjrgquQoAWM1YsZZ3wegI09EbM5ot1E6Bj4zRonJXAjQjQ5nTEGOGTf7gG4w9rJUAjNkh9mIQADdcg9WEeAjRWg9SHqQjQQA1SH2bzqV7AoLbbh/v7L4ddePr6+XaXO8qc+jCJD4+Pv+o1jGvfoJ0bNUh9Lvt1rNV2pjeACejlt8LRKHStEp1u7qZ654EAvXk7dpVN2R/PldSHCdmCXWcL8GKMnjnMlh6mJUDlSYT0MDkBCkqkO7AjQO8UI9GBUwIEZHwTGsgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZvKf7dhEag/kq81AAAAAElFTkSuQmCC', 'sha256': 'ceafeb6fac379a77923ba1275efc8ca6612837fdeeae7b84bb552e6c5c73269b'}, 'data/binding-v1/images/binding-evaluation-0119-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNklEQVR4nO3bMW5TSxiAUfNElvMkF2nZEO1bBy0bok1hia1QREpDEekqz3ZsEux8nrnnVAiF63FxP/0zQz49Pv7aABT+ST4VQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfO4+Gt5gu73frMBu97BZExMQkBEgIGMLxmCe/vt3M527bz83q2QCAjImoFs5Q13b6SMI0A3d2uz9vB6xBgJ0oxfGzw+RIeYmQB/UnR/fn07/8y9f7048U4mYkgBdMT1no/PaDx/GyEDElATo8vV5U3eOWp6wV6Lt9l6DmIkA3VZ6jj7wZYaMQsxEgC5Tn4un52yGNIgJCNDf1ueq6TmRIQ1iAv4n9Bj1OfqJK/kFcSYmQCPV5/BzNYih2YK9wfK2V+k5uh2zF2NcJqDx6rNYVmIOYlACNGp9nmkQQxOg80Z5t0dZJywEaIBT59OcSTMuAfpTt1mf218bnCBAQx79HHIYxIgE6FXjvsnjrpy1EaDzbn/8GWudsBCgOYeI0dfPSgjQVGPFWKsFAQIyAjT25dch12EMRICAjADtm2lwmOm7MCUBetWI+6/RV87aCBCQESAgI0BARoDmuYB/yWU8QxAgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgP5nt3t4/sOXr3ebkS3rX74R3CABAjICBGQECMgI0KvGPQYad+WsjQDtm+nUdqbvwpQECMgI0GyX8S7gGYgAARkBOmOsIWis1YIAHTf6/mX09bMSAjTPWDHKOmEhQBMOEeOunLURoEmuw1x+MSIB+lO33KBbXhucIEBnvBwobvM9f7kq4w9jEaDzRnmrR1knLARo7MMgRz8MTYAGbpD6MLrP9QJGsts9bLf3y5v/4/tTtRLnPszBBDTembT6MA0BGqxB6sNMbMH+ai/2kduxvdipDxMQoAs06NoZOpyz1Ic5CND7PVfgqhmSHuYmQBcehV5W490leu1oSX2YjABdZRQ67MjZGJ0+z5YepiRAF7M04rBE774v0x3mJkAfNxC94yEcuvv2s14CFyNA17JXkLM9UhxWSIA+iL7AoU+Pj7+O/DXA9flVDCAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCATeU3evEvf7mc9VEAAAAASUVORK5CYII=', 'sha256': '5cc31a67a7735ec88e78f67c1b1c33cdaafbf0f989cbe55ae4247cb75488a10f'}, 'data/binding-v1/images/binding-evaluation-0119-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFN0lEQVR4nO3bMU4bQRiAUScK93GBxE1ySG4SicKHSYGUJgWS5diAg8H+dmbfqxBCu7+L/TQ7Y749P//eABS+J3cFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyP7pbwwdst/ebFdjtnjZrYgUEZAQIyHgFYzB3D4+b6fz59XOzSlZAQMYKaCl7qGvbfQQBWtCpzdHf6xFrIEALPTB+uYgMMTcBulF3zm6dvroNub+mEjElAbpiej50XnP4x6cxsiBiSgL09fX5/Dnx/gpHJdpu7zWImQjQstLz6gUPM2QpxEx8D+hr6nP38Hi9L8idXnwl/xjF9AToEofP/1XTc+joRhrEBATos/W58d01iJkI0Ej1Ob2vBjE0AfqA/dN+s9eutxwOoEGMS4Auqc9mGTSI0QnQqPV5oUEMTYDOG+XZHmVO2BOgAXad32dPmnEJ0P9aZn2WPxu8Q4CG3Po5ZTOIEQnQm8Z9ksednLURoPOWv/wZa07YE6A5FxGjz89KCNBUy4qxpgUBAjICNPbh1ynHYQxEgICMAB2baeEw02dhSgL0phHfv0afnLURICAjQEBGgICMAM1zAH/IYTxDECAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaA/rHbPb388OfXz83I9vPvPxEskAABGQECMgIEZAToTeNuA407OWsjQMdm2rWd6bMwJQECMgI022G8A3gGIkBARoDOGGsRNNa0IECvG/39ZfT5WQkBmmdZMcqcsCdAEy4ixp2ctRGgSY7DHH4xIgH6X0tu0JJng3cI0BmHC4plPueHU1n+MBYBOm+Up3qUOWFPgMbeDLL1w9AEaOAGqQ+jE6ALG9Rm6HAA9WFcAjTenrRdZ6YhQIM1SH2YiQB9QYNuk6GjG6kPExCgCx09/1fN0OnF1Yc5/KgHGNhLBbbb+/1vXjJx9/D4Vbc4jZr0MBMB+qzd7umwQYfVuLhEby2m1IfJCNBVlkKnHTkbo/ff4KSHKQnQl9k34rREF5+X6Q5zE6DbLYguuAinlvNNdD5PgK7lqCBne6Q4rJAA3Yi+wKlvz8+/X/k1wPX5IiKQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwKbyF5vQMZbanUfLAAAAAElFTkSuQmCC', 'sha256': 'd1130f73b06f9d91d0c504f556e6875f3f6b3bd3680972d4972889a78fc90951'}, 'data/binding-v1/images/binding-evaluation-0119-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFN0lEQVR4nO3bMW5TSxiAUfNE9uMiEhXbYBEsiEW8bVAhpfBiKCKleUWkqzzbsUmw83nmnlMhFK7Hxf30zwz59Pj4ewNQ+Cf5VAABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPO5+2h4g+32frMCu93DZk1MQEBGgICMLRiDufvy72Y6T7++bVbJBARkTEC3coa6ttNHEKAburXZ+3k9Yg0E6EYvjJ8fIkPMTYA+qDs/fzyd/udfv9+deKYSMSUBumJ6zkbntR8+jJGBiCkJ0OXr86buHLU8Ya9E2+29BjETAbqt9Bx94MsMGYWYiQBdpj4XT8/ZDGkQExCgv63PVdNzIkMaxAT8T+gx6nP0E1fyC+JMTIBGqs/h52oQQ7MFe4Plba/Sc3Q7Zi/GuExA49VnsazEHMSgBGjU+jzTIIYmQOeN8m6Psk5YCNAAp86nOZNmXAL0p26zPre/NjhBgIY8+jnkMIgRCdCrxn2Tx105ayNA593++DPWOmEhQHMOEaOvn5UQoKnGirFWCwIEZARo7MuvQ67DGIgAARkB2jfT4DDTd2FKAvSqEfdfo6+ctREgICNAQEaAgIwAzXMB/5LLeIYgQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAD9z2738PyHr9/vNiNb1r98I7hBAgRkBAjICBCQEaBXjXsMNO7KWRsB2jfTqe1M34UpCRCQEaDZLuNdwDMQAQIyAnTGWEPQWKsFATpu9P3L6OtnJQRonrFilHXCQoAmHCLGXTlrI0CTXIe5/GJEAvSnbrlBt7w2OEGAzng5UNzme/5yVcYfxiJA543yVo+yTlgI0NiHQY5+GJoADdwg9WF0n+sFjGS3e9hu75c3/+ePp2olzn2YgwlovDNp9WEaAjRYg9SHmdiC/dVe7CO3Y3uxUx8mIEAXaNC1M3Q4Z6kPcxCg93uuwFUzJD3MTYAuPAq9rMa7S/Ta0ZL6MBkBusoodNiRszE6fZ4tPUxJgC5macRhid59X6Y7zE2APm4gesdDOPT061u9BC5GgK5lryBne6Q4rJAAfRB9gUOfHh9/H/lrgOvzqxhARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZvKfxenMDRbDLT7AAAAAElFTkSuQmCC', 'sha256': '5d58c91cf49cc07fd692f7156aa142dbe0179fe645b935da1089ceeb58024a99'}, 'data/binding-v1/images/binding-evaluation-0120-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNklEQVR4nO3cvW0bWRhAUXqx7IeBUnfgSlzQVqIONnXAYhwIULIwCHCFESlRsuQ77805kQKC8wbEXHzzpy8PDz93AIW/kq0CCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZm/u03D+x0Od7upHY8/dhtgAgIyAgRknIIxtn//edxN5Ov3/W5LTEBARoCAjAABGQECMgIEZAQIyAgQkBEgIONBxP6tpY289QPPCVD/nuTi83rEdgjQ6t7PPn+PEjE9Afr09NzvX3m759vj4wtfK0NMTIA+JT2vRufah5/HSIaYmAB9ZHre1J2Lzt+wKJEMMSUB+oD6/H53bizR4XCnQczEc0BrrM8L3z/9vyJlU0xA603PYkPnUcjpGNMwAa29Pte2aBRiAgJ0k6dH+/1+/+frc3HTGsToBOjN9UnXslyDBjE0ARqsPicaxBwEaLz6nGgQExCgm6ytPmteFdxOgK46jxVrPs7PazMEMSIBumzE43nENbNxAjTYpZ/nXAxiXAL0kvXXZ6x1woIALY0+RIy+fjZFgCYZK8ZaLZwI0Hh3vq5xR4zhCBCQEaD/zTQ4zLQvTEyALhjx/Gv0lbNNAgRkBGjac5b59oj5CNBsZzGjr59NESAgI0BARoCAjAAN/wD0cx6JZhQCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICNAvx+OP0x/fHh934zvvxXm/YJ0ECMgIEJARICAjQEujXwYaff1sigDtZr1eO98eMR8BAjICNNVZzLgrZ5sEaM5zlpn2hYkJEJARoHkeifYANMMRoKvGatBYq4UTAVoafXwYff1sigDNMFaMsk5YEKBXhoj1H9tPV2j8YSwCdNmIR/KIa2bjBGjsO2LufDE0AbrJOhu0zlXB7b48PPx8w8e353C4O/99v9/vVmPjl36e/i5TOm7jNzUBDXlBeuP1YRoCNF6D1IdpOAV758yfnI4t8qc+jM4EdKvF0f7nRyH1YT4moAFGIelhViagtY9C6sPETEAfeRv4A6ehi11THyYjQJ/yNMq7S3RtnpIepiRAn/5Q3KsxevkkTnqYmACt99lc6WF6AvTxfrNEusN2CFDfI8VhswQIyHgOCMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESBgV/kPWAUirRxojp4AAAAASUVORK5CYII=', 'sha256': '5f41a149d20d64c22bbac6f56f6a9c7ac57f658367e34ab3d207b69d4315e68c'}, 'data/binding-v1/images/binding-evaluation-0120-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFNklEQVR4nO3cMW7bSBiAUWWxPs4CKtzmQmlzjm1zobQpBOQqKQy4WQQCtAYtyYpj5+MM36tcyNIQBj/8JEf+8PDwYwdQ+Cv5VAABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg83f30fB6+/39bmqHw7fdBpiAgIwAARmXYIzt8fM/u4nc/ft9tyUmICAjQEBGgICMAAEZAQIyAgRkBAjICBCQsRGx/9bSRr71A88JUP89ycXr9YjtEKDVfT/79D5KxPQE6N3T8/XL4/Vf//jp7srbyhATE6B3Sc+L0bn04ucxkiEmJkBvmZ5f6s5Zp3dYlEiGmJIAvUF9fr87N5Zov7/XIGZiH9Aa63Pl/af/V6RsiglovelZfNBpFHI5xjRMQGuvz6VPNAoxARPQTZ6e7X8+PVdGIXMQQzMBDVOfs2swBzE0ARqsPkcaxBwEaLz6HGkQExCgm6ytPmteFdxOgC46jRVrPs9PazMEMSIBOm/E83nENbNxAjTYrZ/n3AxiXAJ0zfrrM9Y6YUGAlkYfIkZfP5siQJOMFWOtFo4EaLwnX5d4IsZwBAjICND/ZhocZjoWJiZAZ4x4/TX6ytkmAQIyAjTtNct8R8R8BGi2q5jR18+mCBCQESAgI0BARoCG3wD9nC3RjEKAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAfrpcPh2/OHjp7vd+E5HcTouWCcBAjICBGQECMgI0NLot4FGXz+bIkC7We/XzndEzEeAgIwATXUVM+7K2SYBmvOaZaZjYWICBGQEaJ4t0TZAMxwBumisBo21WjgSoKXRx4fR18+mCNAMY8Uo64QFAXphiFj/uf10hcYfxiJA5414Jo+4ZjZOgMZ+IubJF0MToJuss0HrXBXc7sPDw49fePn27Pf3p5+/fnncrcbGb/08/btM6bCNv6kJaMgb0huvD9MQoPEapD5MwyXYK2f+5HJskT/1YXQmoFstzvY/PwqpD/MxAQ0wCkkPszIBrX0UUh8mZgJ6y8fAbzgNne2a+jAZAXqX3SivLtGleUp6mJIAvfumuBdjdP0iTnqYmACtd2+u9DA9AXp7v1ki3WE7BKjvkeKwWQIEZOwDAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjYVf4D/T8w+z4fOpIAAAAASUVORK5CYII=', 'sha256': '0545ffbc0a8360af86ab144a604c6a572cafd02590a0f44a200b55617d160e9a'}, 'data/binding-v1/images/binding-evaluation-0120-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFN0lEQVR4nO3cQU4bSRiAUWc0Ps5IXnCM3CbnmNtwjCyQ5ipZILGJRpYc1NjGEMjXVf3eioVxVwv1p7/abb48Pv7YART+So4KIEBASYCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGT+7g4N73c43O2m9vDwfbcBJiAgI0BAxhaMsT19+2c3kf2//+22xAQEZAQIyAgQkBEgICNAQEaAgIwAARkBAjIeROy/tbSRb/3ASwLUf09y8Xo9YjsEaHXfzz69jxIxPQH69PTc7/fXf/3r09OVt5UhJiZAn5KeV6Nz6cUvYyRDTEyAPjI9b+rOWad3WJRIhpiSAH1AfX6/OzeW6HC40yBm4jmgNdbnyvtP/69I2RQT0HrTszjQaRSyHWMaJqC11+fSEY1CTECAbvL8ar/f7/98fc4eWoMYnQC9uT7pWpZr0CCGJkCD1edIg5iDAI1XnyMNYgICdJO11WfNq4LbCdBFp7Fizdf5aW2GIEYkQOeNeD2PuGY2ToAGu/XzkptBjEuArll/fcZaJywI0NLoQ8To62dTBGiSsWKs1cKRAI33ydclPhFjOAIEZATol5kGh5nOhYkJ0Bkj7r9GXznbJEBARoCm3bPMd0bMR4Bm28WMvn42RYCAjAABGQECMgI0/APQL3kkmlEIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0D/e3j4fvzh69PTbnynszidF6yTAAEZAQIyAgRkBGhp9NtAo6+fTRGg3az3a+c7I+YjQEBGgKbaxYy7crZJgObcs8x0LkxMgICMAM3zSLQHoBmOAF00VoPGWi0cCdDS6OPD6OtnUwRohrFilHXCggC9MkSs/9p+vkLjD2MRoPNGvJJHXDMbJ0BjfyLmky+GJkA3WWeD1rkquN2Xx8cfb3j59hwOd6ef7/f73Wps/NbP87/LlB628Tc1AQ15Q3rj9WEaAjReg9SHadiCvXPmT7Zji/ypD6MzAd1qcbX/+VFIfZiPCWiAUUh6mJUJaO2jkPowMRPQR34M/IHT0NmuqQ+TEaBPeRrl3SW6NE9JD1MSoE9/KO7VGF3fxEkPExOg9T6bKz1MT4A+3m+WSHfYDgHqe6Q4bJYAARnPAQEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIE7Co/AcJfIq0Eftj6AAAAAElFTkSuQmCC', 'sha256': 'ed13a495bfd1472faa40e8d308adadf81ffcf7204789d11b7e14b3c56717e17f'}, 'data/binding-v1/images/binding-evaluation-0121-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFElEQVR4nO3aMW6aSRiAYbJa38dFJF9i6xzS9V4iUgofJoUlN1tEQggCYW3wO//M81Qu8D9TwKtvBr68vv7cART+SlYFECCgJEBARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQ+btbGmKPj193m/Xy8mO3fSYgICNAQMYRDHb/PjzsNuKft7fdRAToNpcFcxzI4ZMJ0G1uKA9fLEZwJQG6/Tcj+ycoEVwmQFel5+Hp+fL/vn3/du6BMgTnCNDZ9PwxOudefBQjGYJzfA3/m/o8PD3/r/pc8++b/s0b3MnSE9BRFD4SnVP7p+0HIqMQHFl3ArprfS482SgEqwfosAIfPHBd42gJDYJ1A3RUn09bV4Ng9QBV9TldUYNgrQC19TldV4NY3EIBGqE+p6trECtbJUDj1Od0DxrEslYJ0FD1GW0nUFkiQPsRY7TP/H4/hiDWNH+AtvLZ3so+4YbmD9DeaOPPyLuCzzF5gIY9fB1yEGNZkwcIGNnMAdriQLHFPcO7zRygvZHPX1vZIdzDEgECxjRtgDZx/XzIVTQLmjZAwPgECMgIEJCZM0CbuwD6xTUQq5kzQMAmCBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkJkzQC8vP3798fb922479rvd7x/mNmeAgE0QICAjQEBm2gBt7hrIBRALmjZAwPiWCND4Q9D4O4R7mDlAWzzLbHHP8G4zBwgY3OQB2sRVtOtnljV5gA6N2aAxdwWfY/4AbWWs2Mo+4YbmD9DIBzGHLxa3RIAOjdOgcXYClVUCdDhijPDJP9yD8YdlrRKgoRqkPrBcgAZpkPrAogHKG6Q+cOjL6+vP3XoeH7/u/354ev6cRdVn5LfB5rxM8RZabgL67Rx071HoaIk53jrwcYsG6LQC92vQ0ZPVB1Y/gl2ew29yKDstmvTAEQE6exfw7gz9dphSHzglQFddSf4xRhdOcNID5wjQHb8ZkR64TIBuXyLdgSsJ0G1iJDrwDgIEZNb9HRCQEyAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECNhV/gN/7Ql/8XpsrQAAAABJRU5ErkJggg==', 'sha256': '9a6caa7de864c3782c98a87701f96ca2f5a9ddf59c0a78fa104fa74642e104f3'}, 'data/binding-v1/images/binding-evaluation-0121-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFUlEQVR4nO3aMW6aSRiAYbJajrOSixwjt9lz7G1yjBSW9iopLLnZIhJCYAhrg9/5Z56ncoH/mQJefTPw5eXl5w6g8EeyKoAAASUBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgMyf3dIQe3r6utus5+cfu+0zAQEZAQIyjmCwe/37r91G7P/5dzcRAbrPZcEcB3L4ZAJ0nxvK4xeLEdxIgO7/zcjhCUoE1wnQTen5vt9f/99vr6+XHihDcIkAXUzPb6Nz6cUnMZIhuMTX8G/U5/t+/7/qc8u/b/o3b/AgS09AJ1H4SHTOHZ52GIiMQnBi3QnoofW58mSjEKweoOMKfPDAdYuTJTQI1g3QSX0+bV0NgtUDVNXnfEUNgrUC1NbnfF0NYnELBWiE+pyvrkGsbJUAjVOf8z1oEMtaJUBD1We0nUBliQAdRozRPvOH/RiCWNP8AdrKZ3sr+4Q7mj9AB6ONPyPvCj7H5AEa9vB1zEGMZU0eIGBkMwdoiwPFFvcM7zZzgA5GPn9tZYfwCEsECBjTtAHaxPXzMVfRLGjaAAHjEyAgI0BAZs4Abe4C6BfXQKxmzgABmyBAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBmzgA9P//49ce319fddhx2e9g/zG3OAAGbIEBARoCAzLQB2tw1kAsgFjRtgIDxLRGg8Yeg8XcIjzBzgLZ4ltninuHdZg4QMLjJA7SJq2jXzyxr8gAdG7NBY+4KPsf8AdrKWLGVfcIdzR+gkQ9iDl8sbokAHRunQePsBCqrBOh4xBjhk3+8B+MPy1olQEM1SH1guQAN0iD1gUUDlDdIfeDYl5eXn7v1PD19Pfz9fb//nEXVZ+S3weY8T/EWWm4CenMOevQodLLEHG8d+LhFA3Regcc16OTJ6gOrH8Guz+F3OZSdF0164IQAXbwLeHeG3hym1AfOCdBNV5K/jdGVE5z0wCUC9MBvRqQHrhOg+5dId+BGAnSfGIkOvIMAAZl1fwcE5AQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQJ2lf8ARnYIP+GoXhIAAAAASUVORK5CYII=', 'sha256': 'c72643c2977da7454bc2a88c72566d91b32507ae441d9cbb74f082ec82dac3c2'}, 'data/binding-v1/images/binding-evaluation-0121-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFGElEQVR4nO3aMWqbSRiAYWVZH2fBRcA3yUX2HHsR3ySQwrBXSWFws0VACMlytLbkd/6Z56lcyP9MIb18M9KX5+efO4DCH8mqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMn92S0Ps/v7rbrOenn7sts8EBGQECMg4gsHu5e+/dhtx98+/u4kI0HUuC+Y4kMMnE6Dr3FAevliM4EICdP1vRvZPUCJ4mwBdlJ67h8e3//fl+7dzD5QhOEeAzqbnt9E59+KjGMkQnONr+Ffqc/fw+L/qc8m/b/o3b3AjS09AR1H4SHRO7Z+2H4iMQnBk3QnopvV548lGIVg9QIcV+OCB6xJHS2gQrBugo/p82roaBKsHqKrP6YoaBGsFqK3P6boaxOIWCtAI9TldXYNY2SoBGqc+p3vQIJa1SoCGqs9oO4HKEgHajxijfeb3+zEEsab5A7SVz/ZW9glXNH+A9kYbf0beFXyOyQM07OHrkIMYy5o8QMDIZg7QFgeKLe4Z3m3mAO2NfP7ayg7hFpYIEDCmaQO0ievnQ66iWdC0AQLGJ0BARoCAzJwB2twF0C+ugVjNnAECNkGAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMnAF6evrx64+X799227Hf7X7/MLc5AwRsggABGQECMtMGaHPXQC6AWNC0AQLGt0SAxh+Cxt8h3MLMAdriWWaLe4Z3mzlAwOAmD9AmrqJdP7OsyQN0aMwGjbkr+BzzB2grY8VW9glXNH+ARj6IOXyxuCUCdGicBo2zE6isEqDDEWOET/7hHow/LGuVAA3VIPWB5QI0SIPUBxYNUN4g9YFDX56ff+7Wc3//df/33cPj5yyqPiO/DTbnaYq30HIT0Ktz0K1HoaMl5njrwMctGqDTCtyuQUdPVh9Y/Qj29hx+lUPZadGkB44I0Nm7gHdn6NVhSn3glABddCX52xi9cYKTHjhHgG74zYj0wNsE6Pol0h24kABdJ0aiA+8gQEBm3d8BATkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAXeU/E5sJf1Nsb5sAAAAASUVORK5CYII=', 'sha256': '252c66aea32b22af2af9da1bc4e28c134ee0748b3447a47262af7b10d14a7577'}, 'data/binding-v1/images/binding-evaluation-0122-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFQklEQVR4nO3bQWobSRiAUWWIjxPwwjA3mYvkHLnI3GTAC0GukoXBm1kIhEaWldjR+Ouqem+VhSNVE/fHX9WdT09PP3YAhT+SbwUQIKAkQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAzOfuq+GW7u8fdpPa7x93kzIBARkBAjK2YMzm7s+/d1N4/uev3ewEKDiMmHhLD28iQMHx59nP6xHLEqD+scvxc5SI1QjQ7dPz/PXL9b9+9+37lY+VIdYhQLdJz0+j89oPv4yRDLEOAfqt9LypOxcdP+GsRDLECgToPfX5/e78Yonu7x80iIl5EXET9bny+RP/DwMwAW0oPWdfdByFbMeYlQloc/V57RuNQszHBHTZ6d3+8em5MgqZg5iJCWi79bm4BnMQMxGgrdfnQIOYkgANUJ8DDWI+AnTZ1uqz5VXBuwnQ7uVYseX7/Lg2QxATEKDduPfziGuGUwK09aOflxwGMQ0B+o/t12esdcJ1AjT8EDH6+lmZAI06Voy1Wrho9QAN8eTrNZ6IMbrVAwSElg7QTIPDTNfCOpYO0NGI+6/RVw4CBJTWDdB8e5b5rojprRugaXYxo6+flQkQkBEgICNAQGbRAA39AvRLXolmUIsGCNgCAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMosGaL9/PPzh7tv33fiOV3G8LhjCogECtkCAgIwAARkBGv4YaPT1s7J1AzTfee18V8T01g0QkBOgsXcx464cVg/QTHuWma6FdSwdIKC1eoCGfiXaC9CMbvUAnRqrQWOtFi4SoOHHh9HXz8oEaMixYpR1wnUCdD5EbP/ePl2h8YehCdDAd/KIa4ZTAjTYEzFPvpiJAF22zQZtc1XwbgI0zGGQox/mI0BjNEh9mJIADdAg9WFWn+sFbNR+/3h//3B6/z9//fLxyzjLn/owGRPQq87u9o8fhdSH6ZmArjnc8x8/CkkPi/j09PSjXsMAjg06+F8bpD43+TeayX7e3wET0JuPhE4bccMSXdziTfybBwL0/u3YrUr02tGS9LACW7DbT/s/jdH182zpYR0CtKFDB+lhNQLUl0h3WJYABT1SHDgQICDjTWggI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQJ2lX8B3VkvrPcmTcMAAAAASUVORK5CYII=', 'sha256': '640d06573b9bad93d403363535a606f928c275d7865ff1f98fc962f98830a8db'}, 'data/binding-v1/images/binding-evaluation-0122-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFR0lEQVR4nO3bTWobSRiAYWWIjzPghcE3yUXmHHMR3ySQhSBXycLgzSwEQiPLP3GUvF1Vz7PKwlF/TdwvVaXOp8fHHzuAwl/JVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAg87m7NFzT7e3dblL7/bfdpKyAgIwAARlbMGbz9M/fuync/Pt9NzsBCg4jJt7Sw08RoOD48+zn9YhlCVD/tcvxc5SI1QjQ9dNzc//w+l9/+vrllY+VIdYhQNdJz5vReemHn8dIhliHAP1Sen6qOxcdP+GsRDLECgToI/X59e68s0S3t3caxMS8iLiJ+rzy+RP/DwOwAtpQes4udFwK2Y4xKyugzdXnpStaCjEfAbrs9Gm/uX/48/W5eGkNYjIC9HZ90lnOZ9AgZiJAW6/PgQYxJQEaoD4HGsR8BOiyrdVny1PBhwnQ7vmyYsvP+XE2iyAmIEC7cZ/nEWeGUwK09aOf5xwGMQ0B+p/t12esOeF1AjT8ImL0+VmZAI26rBhrWrho9QAN8c3XS3wjxuhWDxAQWjpAMy0cZroX1rF0gI5G3H+NPjkIEFBaN0Dz7VnmuyOmt26AptnFjD4/KxMgICNAQEaAgMyiARr6BejnvBLNoBYNELAFAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZBYN0H7/7fCHp69fduM73sXxvmAIiwYI2AIBAjICBGQEaPhjoNHnZ2XrBmi+89r57ojprRsgICdAY+9ixp0cVg/QTHuWme6FdSwdIKC1eoCGfiXaC9CMbvUAnRqrQWNNCxcJ0PDLh9HnZ2UCNOSyYpQ54XUCdL6I2P6zfTqh5Q9DE6CBn+QRZ4ZTAjTYN2K++WImAnTZNhu0zangwwRomMMgRz/MR4DGaJD6MCUBGqBB6sOsBOhdDaoydHZp9WEyAvSis6f9zzfo7Irqw3w+1wNs2uGZv729Oy3Czf3D776u9LCIT4+PP+oZBnBs0MFvbZD6XOXfaCb7eX8HrIDeZb//dvr7fWzEFUt0cYs38W8eCNDHt2PXKtFLR0vSwwpswa6/2n8zRq+fZ0sP6xCgDR06SA+rEaC+RLrDsgQo6JHiwIEAARlvQgMZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIELCr/Ackni2JFxN7wwAAAABJRU5ErkJggg==', 'sha256': '05f1a79d93241d2e0ddaea386cce22b0e2bcb190af0995f1429afdf92e160a07'}, 'data/binding-v1/images/binding-evaluation-0122-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFN0lEQVR4nO3bTW4TSxhAUfMEy0HyIDtjHewsA0tshUEkJgwsWX7+CUowuV1V54wYRHa1SF99Vd359PLycwdQ+C/5VgABAkoCBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjIfO6+Gh5pv3/aTepweN5NygQEZAQIyNiCMZtf377upvDl+4/d7AQoOIyYeEsPbyJAwfHnxc/rEcsSoP6xy+lzlIjVCNDj0/PHM4h7e/vjx8oQ6xCgx6TnTQef5z98HSMZYh0C9Ffp+fsHLqdPuCiRDLECAXpPff7Fg96bJdrvnzSIiXkRcRP1eeXzJ/4LAzABbSg9F190GoVsx5iVCWhz9bn3jUYh5mMCuu38bg9f7b8ehcxBzMQEtN363FyDOYiZCNDW63OkQUxJgAaoz5EGMR8Bum1r9dnyquDdBGh3PVZs+T4/rc0QxAQEaDfu/TzimuGcAG396OeawyCmIUD/s/36jLVOeJ0ADT9EjL5+ViZAo44VY60Wblo9QEM8+brHEzFGt3qAgNDSAZppcJjpWljH0gE6GXH/NfrKQYCA0roBmm/PMt8VMb11AzTNLmb09bMyAQIyAgRkBAjILBqgoV+AvuaVaAa1aICALRAgICNAQEaAgIwAARkBAjICBGQECMgIEJARICCzaIAOh+fjP758/7Eb3+kqTtcFQ1g0QMAWCBCQESAgI0DDHwONvn5Wtm6A5juvne+KmN66AQJyAjT2LmbclcPqAZppzzLTtbCOpQMEtFYP0NCvRHsBmtGtHqBzYzVorNXCTQI0/Pgw+vpZmQANOVaMsk54nQBdDhHbv7fPV2j8YWgCNPCdPOKa4ZwADfZEzJMvZiJAt22zQdtcFbybAA1zGOToh/kI0BgNUh+mJEADNEh9mNXnegEbdTg87/dP5/f/r29fP34ZF/lTHyZjArrr4m7/+FFIfZieCeg1x3v+40ch6WERn15eftZrGMCpQUf/tEHq85D/o5kc5v0dMAG9+UjovBEPLNHNLd7Ev3kgQO/fjj2qRPeOlqSHFdiCPX7a/2OMXj/Plh7WIUAbOnSQHlYjQH2JdIdlCVDQI8WBIwECMt6EBjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgYFf5DWuFL6xzjuZAAAAAAElFTkSuQmCC', 'sha256': '3dfaac89620575a8bf40a681aafa970b46c97aa5a951d4f9757c1cbe1494f561'}, 'data/binding-v1/images/binding-evaluation-0123-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE/klEQVR4nO3bPW4aQRiAYRIl96FwmwvlQLlQWhccJoUlNymQEFqjAOHn3Rk/T2XJaHdc7KtvhvWXt7c/G4DC1+SuAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMt+7W8Azb7ctmUrvd62ZwJiAgI0BAxhaMz+L3r/fNFH78/L6ZhQkIyJiAVnEOOsFpIvwHAVrFly/HHxYjPg8BWt2XvocrKBHTE6BnpOfs8efJY8X9BWWIiQnQo9Jz1Xcuxx9exEiGmJgA3Tk9t3/Xe7jCcYlkiCkJ0N3qc/fXTPYXXGRIg5iJAN2hPg99w22RIQ1iJl5EvNp2+/K0+py8y2IBMC4BWte26x8W99IgJmALdoXnDz4LtmNMxgQ0TH1O3t0cxNAEaMh/ql7DGuB2AnSRw6Cxnif/sBJDEOMSoPPW/4Svf4VwkgANc/TzkcMgRidAl1pbfda8KriQAA129PORwyDGJUBARoDGHn/2DEEMSoCAjAANP/7sGYIYkQABGQECMgI0w/5rzy6M4QgQkBEgICNAQEaAJjkA2nMMxFgECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQEaCl3e51/8OPn983ozms+fBXwJoJEJARICAjQEBGgOY5BnIAxHAECMgIEJARoEl2YfZfjEiAgIwAzTAEGX8YlAABGQEafggy/jAuAbrUOhu0zlXBhQTojOOxYm1P+/F6jD+MSIDOW/+zvf4VwkkCNOphkKMfJiBAV1tDg9awBridAI13GOToh2l8qxcwkt3udbt9Oa7A71/vz1zAInzqw+hMQNdZPPPPHIXUh/kI0NV2u9fnb8cW2y71YQ62YGvfjhl8mJgA3adBj8jQx9lKfZiMAN1kX4SPGbqlRCf3dNLDlAToIRladORsjP5xkCQ9TEyAHpuhWw6qpYfpCdCdHapxskRXXYH78vr4CgnQoxx35GyMRIfPSYCeQV/gpC9vb39O/wbgwbwJDWQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgE3lL0IFHMJs2qtBAAAAAElFTkSuQmCC', 'sha256': 'd9210e975a3d63979550b35ff996fa8d1683e166eb07a1543ce9133da3af98c6'}, 'data/binding-v1/images/binding-evaluation-0123-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAE/klEQVR4nO3bPWobQRiAYSUkxwmo8M1yjtzMhSBXSWFwk0IgxFpEUvTz7oyfpzJY7I6LfflmtP7y9vZnA1D4mtwVQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD51t0anmG7fdlMard73QzOBARkBAjI2ILxWbz//LGZwvdfvzezMAEBGRPQKs5BJzhNhP8gQKv48uX4w2LE5yFAq/vS93AFJWJ6AvSM9Jw9/jx5rLi/oAwxMQF6VHqu+s7l+MOLGMkQExOgO6fn9u96D1c4LpEMMSUBult97v6ayf6CiwxpEDMRoDvU56FvuC0ypEHMxIuIV9tuX55Wn5N3WSwAxiVA69p2/cPiXhrEBGzBrvD8wWfBdozJmICGqc/Ju5uDGJoADflP1WtYA9xOgC5yGDTW8+QfVmIIYlwCdN76n/D1rxBOEqBhjn4+chjE6AToUmurz5pXBRcSoMGOfj5yGMS4BAjICNDY48+eIYhBCRCQEaDhx589QxAjEiAgI0BARoBm2H/t2YUxHAECMgIEZAQIyAjQJAdAe46BGIsAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyArS0273uf/j+6/dmNIc1H/4KWDMBAjICBGQECMgI0DzHQA6AGI4AARkBAjICNMkuzP6LEQkQkBGgGYYg4w+DEiAgI0DDD0HGH8YlQJdaZ4PWuSq4kACdcTxWrO1pP16P8YcRCdB563+2179COEmARj0McvTDBAToamto0BrWALcToPEOgxz9MI1v9QJGstu9brcvxxV4//njmQtYhE99GJ0J6DqLZ/6Zo5D6MB8Butpu9/r87dhi26U+zMEWbO3bMYMPExOg+zToERn6OFupD5MRoJvsi/AxQ7eU6OSeTnqYkgA9JEOLjpyN0T8OkqSHiQnQYzN0y0G19DA9AbqzQzVOluiqK3BfXh9fIQF6lOOOnI2R6PA5CdAz6Auc9OXt7c/p3wA8mDehgYwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQsKn8Bc0jHjSrmArTAAAAAElFTkSuQmCC', 'sha256': 'f8daaa9e249bb7446d931c3f1084ad0cf391d59d96bb5abcbe9eb81542c45e82'}, 'data/binding-v1/images/binding-evaluation-0123-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFCklEQVR4nO3aO24bSRRAUXpgLWcABkq9IaezDqfekFMFBLwVBwKcTECAIFq0JJqf21U6JxIguVkO+uLVkz49P//aABT+ST4VQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD53H003MN2+7iZ1G73tBmcCQjICBCQcQXjo/j937+bKTx8+7mZhQkIyJiAVrEHnWCbCH9BgFbxy5fjHxYjPg4BWt0vfQ9PUCKmJ0D3SM+P779f/7dfvj786YEyxMQE6FbpeTM6f/rhRYxkiIkJ0JXTc1Z3Tjo84bhEMsSUBOhq9bk8PQv7By4ypEHMRICuUJ+rp+eVDGkQM/GHiGfbbh/vVp+Tn7I4AIxLgNZ17XrF4rM0iAm4gp3h/oPPgusYkzEBDVOfk59uDmJoAnS2tj7rOQNcToDe5TBorOfNP5zEEMS4BOht63/D139COEmAhln9vGQZxOgE6L3WVp81nwreSYAGW/28ZBnEuAQIyAjQ2OPPniGIQQkQkBGg4cefPUMQIxIgICNAQEaAZrh/7bmFMRwBAjICBGQECMgI0CQLoD1rIMYiQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAAt7XZP+y++fH3YjOZw5sP/AtZMgICMAAEZAQIyAjTPGsgCiOEIEJARICAjQJPcwty/GJEAARkBmmEIMv4wKAECMgI0/BBk/GFcAvRe62zQOk8F7yRAbzgeK9b2th+fx/jDiATobet/t9d/QjhJgEZdBln9MAEBOtsaGrSGM8DlBGi8ZZDVD9P4XB9gJLvd03b7eFyBH99/3/MAi/CpD6MzAZ1n8c7fcxRSH+YjQGfb7Z7ufx1bXLvUhzm4gq39OmbwYWICdJ0G3SJDL2cr9WEyAnSRfRFeZuiSEp2800kPUxKgm2Ro0ZE3Y/TKIkl6mJgA3TZDlyyqpYfpCdCVHapxskRnPYHrevj2sz4CSwJ0K8cdeTNGosPHJED3oC9w0qfn51+nvwNwY/4SGsgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZvK/5zPHXtziDTHAAAAAElFTkSuQmCC', 'sha256': '8bd080fa26c8a8de735b3b2cd023d746d80f0484677d0bf8fb2739f457bfd313'}, 'data/binding-v1/images/binding-evaluation-0124-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFklEQVR4nO3cQU4iTRiAYf0z3MfFHMPjeoxZeBgXJG7+BQlhQHEA8a3uep6VC+j6Epo31aXxcbt9ewAo/JesCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZlf3dIr8fT0+/wLXl///NQssDCP2+1bPcMKo/MZMYJDAvQT3TmlRAv6sFqvq75VPILdeje/bDbn3/v8/v7ZBdd9b8GXBOia9HwZnc9efBQjGWJyAnRZfS5Kz6nd208zpEH3c+NHVnn+aOO8PgL0T+n53pt4f7X9TWYrxJz8HdBP1+fMlVdzbgr/SICOHVbgZbO59wb+aAkNYioCdK4+P7auBjEnAerrc7qiBjEJARqiPqfrahAzEKBR6nO6ugaxegI0UH1OZ9Ag1k2AHoaqz2iTwF3NHqD9FmO07/x+HpsgVmzqAC3lu72UOeFSUwdob7Ttz8hTwTeaN0DDPnwd8iDGus0bICA3aYCWuKFY4sxw3qQB2hv5+WspE8LVZg8QEJoxQIs4fj7kKJq1mjFAwCAECMgIEJCZLkCLOwDacQzEKk0XIGAcAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZKYL0Ovrn90Pz+/vD8uxn3Y/P6zAdAECxiFAQEaAgMyMAVrcMZADINZqxgABg5g9QONvgsafEK42aYCW+CyzxJnhvEkDBIxg3gAt4ija8TPrNm+ADo3ZoDGngm80dYCWsq1YypxwqakDNPKDmIcvZjB7gA6N06BxJoG7EqC/thgjfPMPZ7D9Yd0EaKwGqQ9TEaCBGqQ+zEaARmmQ+jAhARqiQerDnAToiwbdO0NHS6gPUxGgDxxV4H4NOrqy+jCbX/UAg9q14Onp91EpXjab2y9+WjTpYU52QOecduHGh7IP364+TMsO6LKt0M5hRL7cE50JlvQwOQG6PkM7122IpAcE6MpqfFiii64ACNDD7R35MkaiA58RoFvpC1xNgFi5Ef7DAZ/xa3ggI0BA5nG7fetWB6ZmBwRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAwEPlfxDxG64j1nY2AAAAAElFTkSuQmCC', 'sha256': 'b4c57387f2728493133d6cb7909507f1838e3061e956ea916f708a0787609197'}, 'data/binding-v1/images/binding-evaluation-0124-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFI0lEQVR4nO3aMU5bSxiAUfMU9kMRiSrbyCKyoLcItvGqSClYDIUlmhSWLD8bTGwC39w751QU9p1fwv40M3Cz3T5tAAr/JKsCCBBQEiAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BA5ku39Erc3X09/4LHx1+fNQsszM12+1TPsMLovEaM4JAAfUZ3TinRgn5ZrcdVf1Qcwd77ab69fzj/3uef31974Lo/W/AmAbomPW9G57UXH8VIhpicAF1Wn4vSc2r39tMMadDH+e/f580Cfftxu5mAAP1Ret7ZnSP7p+1LZCvEnPwf0GfX58yTV3NvCn9IgI4dVuD2/uHj6vPiEhrEVAToXH0+bV0NYk4C1NfndEUNYhICNER9TtfVIGYgQKPU53R1DWL1BGig+pzOoEGsmwBthqrPaJPAh5o9QPstxmjf+f08NkGs2NQBWsp3eylzwqWmDtDeaNufkaeCv2jeAA17+DrkIMa6zRsgIDdpgJa4oVjizHDepAHaG/n8tZQJ4WqzBwgIzRigRVw/H3IVzVrNGCBgEAIEZAQIyEwXoMVdAO24BmKVpgsQMA4BAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIy0wXo8fHX7ofnn983y7Gfdj8/rMB0AQLGIUBARoCAzIwBWtw1kAsg1mrGAAGDmD1A42+Cxp8QrjZpgJZ4llnizHDepAECRjBvgBZxFe36mXWbN0CHxmzQmFPBXzR1gJayrVjKnHCpqQM08kHM4YsZzB6gQ+M0aJxJ4EMJ0P+2GCN88w9nsP1h3QRorAapD1MRoIEapD7MRoBGaZD6MCEBGqJB6sOcBOiNBn10ho6WUB+mIkAvOKrAxzXo6Mnqw2y+1AMMateCu7uvR6W4vX94/8NPiyY9zMkO6JzTLrzzUPbi29WHadkBXbYV2jmMyJt7ojPBkh4mJ0DXZ2jnug2R9IAAXVmNF0t00RMAAdq8vyNvxkh04DUC9F76AlcTIFbu24/begRe5c/wQEaAgMzNdvvUrQ5MzQ4IyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIBN5TfUExl4kexzXAAAAABJRU5ErkJggg==', 'sha256': '675c86051de249b55e52e5d68bef0a2bd335553b4930009ca9a778c8f80edcde'}, 'data/binding-v1/images/binding-evaluation-0124-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFI0lEQVR4nO3aMU5bSxiAUXiK90ORNjvISrKgtxJ28NoULIbCEk0KS5afDSbYmG/unXMqCnznl7A/zQy+326f7wAK/ySrAggQUBIgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQOZbt/RKPDx8P/8LT0+/v2oWWJj77fa5nmGF0XmLGMEhAfqK7pxSogX9sVpPq36rOIJd+25+3GzOv/bny8tbD1z3ewveJUCXpOfd6Lz1y0cxkiEmJ0Afq8+H0nNq9/LTDGnQ7fz37ys70PH9+HXVO20pBOiv0nNld47sn7Yvka0Qc/I9oK+uz5knr+beFP6SAB07rMDjZnO7+ry6hAYxFQE6V58vW1eDmJMA9fU5XVGDmIQADVGf03U1iBkI0Cj1OV1dg1g9ARqoPqczaBDrJkB3Q9VntEngpmYP0H6LMdpnfj+PTRArNnWAlvLZXsqc8FFTB2hvtO3PyFPBJ5o3QMMevg45iLFu8wYIyE0aoCVuKJY4M5w3aYD2Rj5/LWVCuNjsAQJCMwZoEdfPh1xFs1YzBggYhAABGQECMtMFaHEXQDuugVil6QIEjEOAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMdAF6evq9++Hny8vdcuyn3c8PKzBdgIBxCBCQESAgM2OAFncN5AKItZoxQMAgZg/Q+Jug8SeEi00aoCWeZZY4M5w3aYCAEcwboEVcRbt+Zt3mDdChMRs05lTwiaYO0FK2FUuZEz5q6gCNfBBz+GIGswfo0DgNGmcSuCkB+t8WY4RP/uEMtj+smwCN1SD1YSoCNFCD1IfZCNAoDVIfJiRAQzRIfZiTAL3ToFtn6GgJ9WEqAvSKowrcrkFHT1YfZvOtHmBQuxY8PHw/KsXjZnP9w0+LJj3MyQ7onNMuXHkoe/Xl6sO07IA+thXaOYzIu3uiM8GSHiYnQJdnaOeyDZH0gABdWI1XS/ShJwACdHd9R96NkejAWwToWvoCFxMgVu7Hr0/45gQ34t/wQEaAgMz9dvvcrQ5MzQ4IyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIC7yh/sixoYUrIG9gAAAABJRU5ErkJggg==', 'sha256': 'c2982dbe2ffed9e6d14eae0be8f01c4ec9a326d25b8dd0a8631532accc434563'}, 'data/binding-v1/images/binding-evaluation-0125-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFB0lEQVR4nO3dQU5bSRRAUafV2U8GLWUnWWR2EikDFpMBUiYZIFlfNm0C2Nz6fueMGIB/ScDVq6KQPz0+/joAFP5JngogQEBJgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJD5t3s0XMeXL/8dduvh4edhMBMQkBEgIGMLxv34/PX7YSd+//hWL2EJJiAgI0BARoCAjAABGYfQN7+EMvyiB1wgQDe/+bb9ZDGCLQH60Bu3x1dQIhCgK6Tnxbsnz974eHpBGWI4AXpLel514W37yScxkiGG81ew19Xn89fv77lu++yX7/p/KeE9TEDPO4nCde/4H1/tOBAZhZjJBPTR9bnwykYhphGgU9sKvHPD9TdOHqFBjCJAl+rzYc/VIGYSoL4+50/UIIYQoCXqc/5cDWICAVqlPudP1yDungAtVJ/zNWgQ902ADkvVZ7WVwE1ND9BxxFjtd/64HkMQd2x0gPbyu72XdcJrjQ7Q0Wrjz8qrgiuaG6BlN19bNmLct7kBAnJDA7THgWKPa4bLhgboaOX9115WCG82PUBAaGKAdnH8vOUomns1MUDAIgQIyAgQkBkXoN0dAD1xDMRdGhcgYB0CBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkxgXo4eHn0we/f3w77Mdxtcf1wx0YFyBgHQIEZAQIyEwM0O6OgRwAca8mBghYxPQArT8Erb9CeLOhAdrjXmaPa4bLhgYIWMHcAO3iKNrxM/dtboC21mzQmquCKxodoL2MFXtZJ7zW6ACtvBGz+WKC6QHaWqdB66wEburT4+Ov2z5hD7bv9pe/YeG2Psafv7HrN2t8mP0tNgGd/hC004f6MIoALdQg9WEaW7BV9mLqw0AmoCXmIPVhJhPQy4eaNx2FTjKnPoxiAnrGSQVuNwqpD8OZgF73992rTEPnRZMeZhKgt9wxeXOGnh2m1IexBOi9V91ejNGFHZz0MJwANTdupQcE6KNLpDuwJUA3j5HowP8RICDjHhCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAYfKH842BPmMSjoVAAAAAElFTkSuQmCC', 'sha256': '13d8623883a5740b63895c447881e5359cfa5e3361322b60d4aaf89e42a4d700'}, 'data/binding-v1/images/binding-evaluation-0125-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFDklEQVR4nO3dMW5bRxRAUTqIlhPARdpsKG3W4TYbSuuCgLfiQoCbFAIIgpQoixZ15/OdU6mQ+AeQdPFmNBA/PT5+3wEUfkueCiBAQEmAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkPm9ezS8j8+f/9xt1n7/dTeYCQjICBCQsQXjfvz374/dRvz190O9hCWYgICMAAEZAQIyAgRkHELf/BLK8IsecIEA3fzm2/EnixEcE6APvXF7eAUlAgF6h/T8+OePy1/78OXbSy8oQwwnQNek59XovPTJJzGSIYYToLfV503pOff05ecZ0iBmEqCfSs8vdufE4dUOJTIKMZN7QB9dnwuvvOl/KwFXMAGdOq7A7dJz8ojjUcgcxBwmoLI+zz7LHMQcAtTX5/yJGsQQArREfc6fq0FMIECr1Of86RrE3ROghepzvgYN4r4J0G6p+qy2Erip6QE6jBir/c4f1mMI4o6NDtBWfre3sk54q9EBOlht/Fl5VfCO5gZo2c3XMRsx7tvcAAG5oQHa4kCxxTXDZUMDdLDy/msrK4SrTQ8QEJoYoE0cPx9zFM29mhggYBECBGQECMiMC9DmDoCeOAbiLo0LELAOAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMuMCtN9/ffrg4cu33XYcVntYP9yBcQEC1iFAQEaAgMzEAG3uGMgBEPdqYoCARUwP0PpD0PorhKsNDdAW9zJbXDNcNjRAwArmBmgTR9GOn7lvcwN0bM0GrbkqeEejA7SVsWIr64S3Gh2glTdiNl9MMD1Ax9Zp0DorgZv69Pj4/bZP2ILjd/vL37DwuD7Gn5+x6Tdr3M/+FpuATn8I2ulDfRhFgBZqkPowjS3YKnsx9WEgE9ASc5D6MJMJ6PVDzZuOQieZUx9GMQE946QCtxuF1IfhTEBv+/vuu0xD50WTHmYSoGvumFydoWeHKfVhLAH61atur8bowg5OehhOgJobt9IDAvTRJdIdOCZAN4+R6MBLBAjIuAcEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMCu8j/34hAZib3TdwAAAABJRU5ErkJggg==', 'sha256': 'c6b703d9a4a048906ebae684dd67c5a088931d2f419070b3a0dde9e0c8fa390c'}, 'data/binding-v1/images/binding-evaluation-0125-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFEElEQVR4nO3dvW1bSRSAUXqx6sfBAo62DRfhgrYIt7GRAQcqxoEAJQ4MEAQpUb/UN4/3nEgByTeAxA93hk/gp7u7XzuAwl/JVQEECCgJEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjJ/d5eG9/H58z+7zbq9/bkbzAQEZAQIyNiCcT3+/+9+txH/frupl7AEExCQESAgI0BARoCAjEPoi9+EMvxGDzhDgC5+59vhg8UIDgnQh95xu38FJQIBeof03Hz5fv659z++PvaCMsRwAvSa9DwZnccefBQjGWI4n4K9rD43X76/qD7Pefqm/5cS3sIE9LCjKLwlOqf2r7YfiIxCzGQC+uj6nHlloxDTCNCxwwq8ccP1HEeX0CBGEaBz9fmw62oQMwlQX5/TK2oQQwjQEvU5va4GMYEArVKf06trEFdPgBaqz+kaNIjrJkC7peqz2krgoqYHaD9irPae36/HEMQVGx2grby3t7JOeKnRAdpbbfxZeVXwjuYGaNnN1yEbMa7b3AABuaEB2uJAscU1w3lDA7S38v5rKyuEV5seICA0MUCbOH4+5CiaazUxQMAiBAjICBCQGRegzR0A/eEYiKs0LkDAOgQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMiMC9Dt7c8/P9z/+Lrbjv1q9+uHKzAuQMA6BAjICBCQmRigzR0DOQDiWk0MELCI6QFafwhaf4XwakMDtMW9zBbXDOcNDRCwgrkB2sRRtONnrtvcAB1as0Frrgre0egAbWWs2Mo64aVGB2jljZjNFxNMD9ChdRq0zkrgoj7d3f267BW24PDb/vIvLDysj/HnOTb9ZY23s3/FJqDjP4J2+lAfRhGghRqkPkxjC7bKXkx9GMgEtMQcpD7MZAJ6+lDzoqPQUebUh1FMQA84qsDlRiH1YTgT0Ms+332Xaei0aNLDTAL0mntMXp2hB4cp9WEsAXrrrW5PxujMDk56GE6AmjtupQcE6KNLpDtwSIAuHiPRgccIEJBxHxCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAbvKb8K/BPmE7qWvAAAAAElFTkSuQmCC', 'sha256': 'abcf2ad9dad1fbd99ffc4b9db50abdd60f7ff6f37d89e81577065361858d0d6c'}, 'data/binding-v1/images/binding-evaluation-0126-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFDUlEQVR4nO3dS04jSRRAUWhVLQfJA3bGOnpnDCz1VhhYYlIDS4i2wQbj8o3MOGfKJ0Oy8upFYJP3u93LHUDhn+SqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZX92l12OzeTz9Ddvt863WAktyv9u91GtYZ3Q+I0bwRoBu1J1jSgQCdIX0vD49nP7Z3//+99mXZIiZCdCF6TkbnW/FSIaYkwB9Lz0Xd+crJZIhZiNAX63PFdNzIkMaxFQE6Hx9/lJ6PsuQBjEPAWoGn2NGISbkndBD1Of4Wlf8ez8MywSUbbs+Yzv2Q4tu93ayV9wENFZ9Dq6+6HsJzhKgQ219xlkD3IAPo/5v0Bjnzn99etjvxTabx9nG8msZ59U86/fn75VfNxPQArY5468QLjN7gMY5+jnmMIjVmz1Aw9Zn5FXBtUwdoAGPfo69rc0QxPpMHSCgNW+AFjH+7BmCWKt5AwTkJg3QgsafPUMQqzRpgIARCBCQmTFAi9t/7dmFsT4zBggYhAABGQECMtMFaKEHQHuOgViZ6QIEjEOAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMdAF6e8roEp9F+bZmz0plHaYLEDAOAQIyAgRkZgzQQo+BHACxPjMGCBiEAAGZSQO0uF2Y/RerNGmAgBHMG6AFDUHGH9Zq3gABuakDtIghyPjDik0doPfGbNCYq4JrmT1A78eK0e729+sx/rBKswdoEff2+CuEywjQoIdBjn6YgQAdGqFBI6wBbkCAhjsMcvTDPH7VCxjIdvu82Ty+r8Dr08MtF3AQPvVh9UxAdyfu+VuOQurDhATo0Hb7fPvt2MG2S32YhC1YvB0z+DAzAfpSg/5Gho5nK/VhNgJ0yr4Ixxn6SYk+3NNJD3MSoEsydNCRszE6cZAkPcxMgH6UoZ8cVEsPCNCF1fiwRN/6DYAA3f28I2djJDrwIQG6An2By3gjIpARICBjC8Zq+a8m4zMBARkBAjL3u91Ld3VgaiYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQEC7ip/AJmAL8PZPpoCAAAAAElFTkSuQmCC', 'sha256': 'bf8f60d90a6579556d2283d32c177c860afff1a4095c80a9bc9c963eadf05ab4'}, 'data/binding-v1/images/binding-evaluation-0126-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFDUlEQVR4nO3dPU4jSxhAUXiatx8HpLOhtyA2RErgxRBYIpnAEuLZYIPx+FZ3nZPy0yVZffVVYdP3u93LHUDhn+SqAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZX92l12OzeTj9Ddvt863WAktyv9u91GtYZ3Q+I0bwRoBu1J1jSgQCdIX0PD2+nv7Z3//9+9mXZIiZCdCF6TkbnW/FSIaYkwB9Lz0Xd+crJZIhZiNAX63PFdNzIkMaxFQE6Hx9/lJ6PsuQBjEPAWoGn2NGISbkndBD1Of4Wlf8ez8MywSUbbs+Yzv2Q4tu93ayV9wENFZ9Dq6+6HsJzhKgQ219xlkD3IAPo/5v0Bjnzn96fN3vxTabh9nG8msZ59U86/fn75VfNxPQArY5468QLjN7gMY5+jnmMIjVmz1Aw9Zn5FXBtUwdoAGPfo69rc0QxPpMHSCgNW+AFjH+7BmCWKt5AwTkJg3QgsafPUMQqzRpgIARCBCQmTFAi9t/7dmFsT4zBggYhAABGQECMtMFaKEHQHuOgViZ6QIEjEOAgIwAARkBAjICBGQECMgIEJARICAjQEBGgIDMdAF6e8roEp9F+bZmz0plHaYLEDAOAQIyAgRkZgzQQo+BHACxPjMGCBiEAAGZSQO0uF2Y/RerNGmAgBHMG6AFDUHGH9Zq3gABuakDtIghyPjDik0doPfGbNCYq4JrmT1A78eK0e729+sx/rBKswdoEff2+CuEywjQoIdBjn6YgQAdGqFBI6wBbkCAhjsMcvTDPH7VCxjIdvu82Ty8r8DT4+stF3AQPvVh9UxAdyfu+VuOQurDhATo0Hb7fPvt2MG2S32YhC1YvB0z+DAzAfpSg/5Gho5nK/VhNgJ0yr4Ixxn6SYk+3NNJD3MSoEsydNCRszE6cZAkPcxMgH6UoZ8cVEsPCNCF1fiwRN/6DYAA3f28I2djJDrwIQG6An2By3gjIpARICBjC8Zq+a8m4zMBARkBAjL3u91Ld3VgaiYgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQEC7ip/AAxEKIkSaovyAAAAAElFTkSuQmCC', 'sha256': 'ae018ed15b3555577f48071cc6d7b992e226c1947a03bdd7ebfdca0e3470530f'}, 'data/binding-v1/images/binding-evaluation-0126-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFF0lEQVR4nO3dMU5bWRSAYRiF5SC5oM2G0rKOaWdD01JYylYokGimsIQYGzAYx/99735fS5J3JfR+nXMD8vXT0+MVQOGv5KkAAgSUBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAAZkf3aPXY7O5+/gPbLcPlzoLLMn109NjfYZ1Ruc9YgQvBOhC3TmkRCBAZ0jP8/3tx3/35u/f731JhpiZAJ2YnqPR+VKMZIg5CdDX0nNydz5TIhliNgL02fqcMT0fZEiDmIoAHa/PH0rPexnSIOYhQM3gc8goxIT8JPQQ9Tl81hn/vx+GZQLK1q73WMe+adHt3k72HTcBjVWfvacv+l2CowRoX1ufcc4AF+CXUf83aIzz5j/f3+52sc3mbrax/Fz+/ef5aiF+/rq5mpIJaAFrzvgnhNPMHqBxrn4OuQxi9WYP0LD1GflUcC5TB2jAq59DL2czBLE+UwcIaM0boEWMPzuGINZq3gABuUkDtKDxZ8cQxCpNGiBgBAIEZGYM0OL2rx1bGOszY4CAQQgQkBEgIDNdgBZ6AbTjGoiVmS5AwDgECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjITBegl08Z3X3u6LK8nNlnpbIO0wUIGIcAARkBAjIzBmih10AugFifGQMEDEKAgMykAVrcFmb/YpUmDRAwgnkDtKAhyPjDWs0bICA3dYAWMQQZf1ixqQP02pgNGvNUcC6zB+j1WDHa2/76PMYfVmn2AC3i3R7/hHAaARr0MsjVDzMQoH0jNGiEM8AFCNBwl0GufpjHj/oAA9luHzabu9cVeL6/veQB9sKnPqyeCejqg3f+kqOQ+jAhAdq33T5cfh3bW7vUh0lYweJ1zODDzAToUw36Exk6nK3Uh9kI0Ed2RTjM0HdK9OZOJz3MSYBOydBeR47G6IOLJOlhZgL0rQx956JaekCATqzGmyX60r8ACNDV9ztyNEaiA28SoDPQFziNH0QEMgIEZKxgrNbPXzf1ETjCBARkBAjIXD89PXZPB6ZmAgIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIOCq8h9fly8KOvzN1AAAAABJRU5ErkJggg==', 'sha256': '26c53697635b58f1e51003088b5f56be8df49d19f46a942e19116ba537f131b8'}, 'data/binding-v1/images/binding-evaluation-0127-causal.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFHElEQVR4nO3dTU4bWRSAUacV74dBpuwgK2FBvZLsoKcZsJgMkJhELSSLuGzMn/PVe3XOKIqQfS1Un269AvHl4eHXDqDwT/KuAAIElAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAGZr91bwzvd3HzbDej+/mc9wurYgICMAAEZt2DxzYK1/CN+7Pe71fv++FiPsF4CFB9PHH29HrEpArSuk9GnF5EhNkKArtud//69sH7f3u1feE0lYm4C9PnpuRidc1+8jJGFiLkJ0KfV503dOenwCkclurn5pkFMSYBWkZ6TL/g8Q1YhpiRAH6rPp6fnYoY0iJkI0Dvrc9X0vJAhDWImfhJ61fU5+Y6D/iYULAnQAPVZvq8GMQe3YJcdrvYqPSdvx9yLMQEb0DD1OThMYg9idAI0WH2eaBBzEKCzRrm2R5kTlgRovafOL3MmzQQE6IJ11mf9s8FrCNBIRz9LDoMYmgAdG/dKHndyNkuAzlr/+jPWnLAkQFMtEaPPz9YI0AxrxVjTwoEAARkBGvLh15LHYYxIgICMAO3mWxxm+izMTYCOjXj/NfrkbJYAARkBAjICBGQEaPgH8M95GM9YBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBGg/93f/3z6x+3dfjeyw/yHTwRrJkBARoCAjAABGQE6Nu4x0LiTs1kCtJvv1Hamz8LcBAjICNAkD+M9gGdEAgRkBOi0sZagsaaFAwH6w+j3L6PPz9YI0PBrxShzwpIAzbNEjDs5myVAYz8O8/CLoX2tB1i727v9av9a4fr7eG3fH1f6reGVbECnPV8o1nmdP5/K+sOgBOisUa7qUeaEpS8PD79O/Dfr/rPxjn6Ygw1ovANp9WEaNqC37UHtKuTch8nYgIY5k1Yf5iNAYzRIfZiSW7D334v9nduxo9ipDzMRoI826HoZWu5Z6sNkBGiNGZIeNkKAPrNBHyzRuaMl9WFWAnStDL0yRi+fZ0sPcxOgv1SiN9EdNkKA1pUh6WFTBCjukeKwZQIEZPwkNJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgF3lN0kTMl4Wc/frAAAAAElFTkSuQmCC', 'sha256': 'c8009da064b3fc5ea2a1e1a6eb0eed6a2e48bb112d48731c749fed7629cc5c50'}, 'data/binding-v1/images/binding-evaluation-0127-nuisance.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFIElEQVR4nO3dPWobURSAUSXE+3GRNjvwSrKgrCQ7SJvCi0lhUBOCQSj6sS3b8jfvzTlVCEa6wszHnTcCf3p4+LMBKHxO3hVAgICSAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgIPOle2t4pdvbr5sB3d//rkdYHBsQkBEgIOMWLL5ZsJa/xa8f283ifft+U4+wXAIUH08c/LwesSoCtKyT0ccXkSFWQoCu252fN8+s33fb7ROvqUTMTYDePz3PRufcDx/HyELE3ATo3epzUXdO2r3CQYlub79qEFMSoEWk5+QL7mfIKsSUfA/oTfX5eXPz7vV54sUH/QYwnCNAF9i//q+ann0Hb6RBzESAXlmfD353DWJKAjRAfY7fV4OYgwA9b3e1f9ht1zn7A2gQExCgC+qzWQYNYhoCNFh9HmkQcxCgs0a5tkeZE44J0HJPnZ/mTJoJCNAzllmf5c8GLyFAIx39HHMYxNAE6NC4V/K4k7NaAnTW8tefseaEYwI01RIx+vysjQDNsFaMNS3sCBCQEaAhH34d8ziMEQkQkBGgzXyLw0yfhbkJ0KER779Gn5zVEiAgI0BARoCAjAAN/wB+n4fxjEWAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAfrn/v734z/uttvNyHbz7z4RLJkAARkBAjICBGQE6NC4x0DjTs5qCdBmvlPbmT4LcxMgICNAkzyM9wCeEQkQkBGg08ZagsaaFnYE6D+j37+MPj9rI0DDrxWjzAnHBGieJWLcyVktARr7cZiHXwztSz3A0t1tt4v9a4XL7+O1ffu+0F8NL2QDOm1/oVjmdb4/lfWHQQnQWaNc1aPMCcc+PTz8OfHfLPvPxjv6YQ42oPEOpNWHadiALtuD2lXIuQ+TsQENcyatPsxHgMZokPowJbdgr78X+5jbsYPYqQ8zEaC3Nuh6GTres9SHyQjQEjMkPayEAL1ng95YonNHS+rDrAToWhl6YYyePs+WHuYmQB9UoovoDishQMvKkPSwKgIU90hxWDMBAjK+CQ1kBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARIGBT+QsIzxjmgI8lPQAAAABJRU5ErkJggg==', 'sha256': '78d866f4d94c2d60926f9733599cbb4d295b35ad7d55ca24af8f4d39297f8e9c'}, 'data/binding-v1/images/binding-evaluation-0127-original.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAAYAAAAEACAIAAAA1FTYwAAAFFElEQVR4nO3dTU5bSRSAUacV9sMgUzbEgnpDPc2AxWSAlEmrhWS5bRNC+PnerXfOCCFkl6P4060qI748Pv44ABT+Sp4VQICAkgABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICDztXtq+EO3t98OAz08fK+XsDkmICAjQEDGFizeLBjL3+Kfv38eNu/u/qZewnYJUHw8cfbzesSuCNC2TkafHkSG2AkB+tjuvLhHuDqfHx9TiVibAL1/el51MHH6w5cxMhCxNgF6t/q8/UD0+AhnJbq9/aZBLEmANpGeqw94miGjEEsSoDfV50Ovga9mSINYiQD9YX0+7RMoZxnSIFbik9Cbrs/VZxz6m1BwSYAG1OfyeTWINdiCvez4bs8/+H+6HbMXYwEmoDH1OTquxBzEdAI0rD5PNIg1CNCzpry3p6wTLgnQdk+df82ZNAsQoBdssz7bXxv8DgGadPRzyWEQownQubnv5LkrZ7cE6FnbH39mrRMuCdBSQ8T09bM3ArTCWDFrtXAkQEBGgEZefl1yHcZEAgRkBOiw3uCw0mthbQJ0buL+a/rK2S0BAjICBGQECMgI0PgL+FMu45lFgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQH6z8PD96cv7u5vDpMd1398RbBlAgRkBAjICBCQEaBzc4+B5q6c3RKgw3qntiu9FtYmQEBGgBa5jHcBz0QCBGQE6LpZQ9Cs1cKRAP3P9P3L9PWzNwI0fqyYsk64JEDrDBFzV85uCdDs6zCXX4z2tV7A1t3d32z2rxVuv48fzb/AdCag604Him3+Lz9dlfGHoQToWVPe1VPWCZe+PD7+uPJttv1n4x39sAYT0LwDafVhGSag181B7Sjk3IfFmIDGnEmrD+sRoBkNUh+WZAv253uxz9mOncVOfViJAL21QR+Xocs5S31YjABtMUPSw04I0Hs26I0leu5oSX1YlQB9VIZ+M0a/Ps+WHtYmQJ9UolfRHXZCgLaVIelhVwQo7pHisGcCBGR8EhrICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQEBGgICMAAEZAQIyAgRkBAjICBCQESAgI0BARoCAjAABGQECMgIEZAQIyAgQkBEgICNAQEaAgIwAARkBAjICBGQECMgIEJARICAjQMCh8i/q8DDIAJD/0gAAAABJRU5ErkJggg==', 'sha256': '6bd7061db3360fc6dcffbfe67f0871484dba8880476e3e8843ca881ff52ebe04'}, 'causalsight/__init__.py': {'base64': 'IiIiQ2F1c2FsU2lnaHQ6IGNvbnRyb2xsZWQgdmlzdWFsIGV2aWRlbmNlIGludGVydmVudGlvbiByZXNlYXJjaC4iIiIK', 'sha256': '397ea8b7fcd26504118bfb5630cd5eddd8d3620a5f4bf6c06c578fe13b3e6380'}, 'causalsight/generate_pilot.py': {'base64': 'IiIiR2VuZXJhdGUgc21hbGwsIHBhaXJlZCBzeW50aGV0aWMgc2NlbmVzIGZvciBpbnRlcnZlbnRpb24tcGlwZWxpbmUgcGlsb3RzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCByYW5kb20KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gUElMIGltcG9ydCBJbWFnZSwgSW1hZ2VEcmF3CgpDT0xPUlMgPSAoInJlZCIsICJibHVlIiwgImdyZWVuIiwgInllbGxvdyIpClJHQiA9IHsKICAgICJyZWQiOiAoMjIwLCA0NSwgNDUpLAogICAgImJsdWUiOiAoNDUsIDk1LCAyMTApLAogICAgImdyZWVuIjogKDM1LCAxNTUsIDg1KSwKICAgICJ5ZWxsb3ciOiAoMjM1LCAxOTAsIDM1KSwKfQpCQUNLR1JPVU5EID0gKDI0OCwgMjQ4LCAyNDQpCklOSyA9ICg0MCwgNDAsIDQwKQpXSURUSCwgSEVJR0hUID0gMzg0LCAyNTYKCgpkZWYgX3JlbmRlcihwYXRoOiBQYXRoLCB0YXJnZXRfY29sb3I6IHN0ciwgZGlzdHJhY3Rvcl9jb2xvcjogc3RyLCB0YXJnZXRfeDogaW50LCBkaXN0cmFjdG9yX3g6IGludCkgLT4gTm9uZToKICAgIGltYWdlID0gSW1hZ2UubmV3KCJSR0IiLCAoV0lEVEgsIEhFSUdIVCksIEJBQ0tHUk9VTkQpCiAgICBkcmF3ID0gSW1hZ2VEcmF3LkRyYXcoaW1hZ2UpCiAgICB5ID0gSEVJR0hUIC8vIDIKICAgIHJhZGl1cyA9IDM4CiAgICAjIFRoZSBjaXJjbGUgaXMgdGhlIHF1ZXJpZWQgb2JqZWN0LiBUaGUgc3F1YXJlIGlzIGFuIGlycmVsZXZhbnQgZGlzdHJhY3Rvci4KICAgIGRyYXcuZWxsaXBzZSgodGFyZ2V0X3gtcmFkaXVzLCB5LXJhZGl1cywgdGFyZ2V0X3grcmFkaXVzLCB5K3JhZGl1cyksIGZpbGw9UkdCW3RhcmdldF9jb2xvcl0sIG91dGxpbmU9SU5LLCB3aWR0aD0zKQogICAgc2lkZSA9IDYyCiAgICBkcmF3LnJlY3RhbmdsZSgoZGlzdHJhY3Rvcl94LXNpZGUvLzIsIHktc2lkZS8vMiwgZGlzdHJhY3Rvcl94K3NpZGUvLzIsIHkrc2lkZS8vMiksIGZpbGw9UkdCW2Rpc3RyYWN0b3JfY29sb3JdLCBvdXRsaW5lPUlOSywgd2lkdGg9MykKICAgIGltYWdlLnNhdmUocGF0aCwgZm9ybWF0PSJQTkciKQoKCmRlZiBnZW5lcmF0ZShvdXRwdXQ6IFBhdGgsIGNvdW50OiBpbnQgPSAxMDAsIHNlZWQ6IGludCA9IDE3KSAtPiBkaWN0W3N0ciwgQW55XToKICAgICIiIldyaXRlIGRldGVybWluaXN0aWMgb3JpZ2luYWwvY2F1c2FsL251aXNhbmNlIFBOR3MgYW5kIHBhaXJlZCBKU09OTCByb3dzLiIiIgogICAgaWYgY291bnQgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNvdW50IG11c3QgYmUgYXQgbGVhc3QgMSIpCiAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgaW1hZ2VzID0gb3V0cHV0IC8gImltYWdlcyIKICAgIGltYWdlcy5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgcm5nID0gcmFuZG9tLlJhbmRvbShzZWVkKQogICAgcm93czogbGlzdFtkaWN0W3N0ciwgQW55XV0gPSBbXQogICAgZm9yIGluZGV4IGluIHJhbmdlKGNvdW50KToKICAgICAgICBzY2VuZV9pZCA9IGYicGlsb3Qte2luZGV4OjA1ZH0iCiAgICAgICAgY29sb3JzID0gcm5nLnNhbXBsZShDT0xPUlMsIDIpCiAgICAgICAgdGFyZ2V0X2NvbG9yLCBkaXN0cmFjdG9yX2NvbG9yID0gY29sb3JzCiAgICAgICAgIyBNaXJyb3IgcG9zaXRpb25zIHRvIHByZXZlbnQgcG9zaXRpb24gZnJvbSBpZGVudGlmeWluZyB0aGUgcXVlcmllZCBhbnN3ZXIuCiAgICAgICAgaWYgcm5nLnJhbmRvbSgpIDwgMC41OgogICAgICAgICAgICB0YXJnZXRfeCwgZGlzdHJhY3Rvcl94ID0gMTEyLCAyNzIKICAgICAgICBlbHNlOgogICAgICAgICAgICB0YXJnZXRfeCwgZGlzdHJhY3Rvcl94ID0gMjcyLCAxMTIKICAgICAgICBxdWVzdGlvbiA9ICJXaGF0IGNvbG9yIGlzIHRoZSBjaXJjbGU/IEFuc3dlciB3aXRoIG9uZSBjb2xvci4iCiAgICAgICAgc3RhdGVzID0gewogICAgICAgICAgICAib3JpZ2luYWwiOiAodGFyZ2V0X2NvbG9yLCBkaXN0cmFjdG9yX2NvbG9yKSwKICAgICAgICAgICAgImNhdXNhbCI6IChybmcuY2hvaWNlKFtjIGZvciBjIGluIENPTE9SUyBpZiBjICE9IHRhcmdldF9jb2xvcl0pLCBkaXN0cmFjdG9yX2NvbG9yKSwKICAgICAgICAgICAgIm51aXNhbmNlIjogKHRhcmdldF9jb2xvciwgbmV4dChjIGZvciBjIGluIENPTE9SUyBpZiBjIG5vdCBpbiAodGFyZ2V0X2NvbG9yLCBkaXN0cmFjdG9yX2NvbG9yKSkpLAogICAgICAgIH0KICAgICAgICBmb3Iga2luZCwgKHRhcmdldF9hZnRlciwgZGlzdHJhY3Rvcl9hZnRlcikgaW4gc3RhdGVzLml0ZW1zKCk6CiAgICAgICAgICAgIGltYWdlX3JlbCA9IGYiaW1hZ2VzL3tzY2VuZV9pZH0te2tpbmR9LnBuZyIKICAgICAgICAgICAgX3JlbmRlcihvdXRwdXQgLyBpbWFnZV9yZWwsIHRhcmdldF9hZnRlciwgZGlzdHJhY3Rvcl9hZnRlciwgdGFyZ2V0X3gsIGRpc3RyYWN0b3JfeCkKICAgICAgICAgICAgaWYga2luZCA9PSAib3JpZ2luYWwiOgogICAgICAgICAgICAgICAgaW50ZXJ2ZW50aW9uID0gTm9uZQogICAgICAgICAgICAgICAgYW5zd2VyID0gdGFyZ2V0X2NvbG9yCiAgICAgICAgICAgIGVsaWYga2luZCA9PSAiY2F1c2FsIjoKICAgICAgICAgICAgICAgIGludGVydmVudGlvbiA9IHsidHlwZSI6ICJjYXVzYWwiLCAiY2hhbmdlZF9vYmplY3QiOiAiY2lyY2xlIiwgImNoYW5nZWRfYXR0cmlidXRlIjogImNvbG9yIiwgImJlZm9yZSI6IHRhcmdldF9jb2xvciwgImFmdGVyIjogdGFyZ2V0X2FmdGVyfQogICAgICAgICAgICAgICAgYW5zd2VyID0gdGFyZ2V0X2FmdGVyCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBpbnRlcnZlbnRpb24gPSB7InR5cGUiOiAibnVpc2FuY2UiLCAiY2hhbmdlZF9vYmplY3QiOiAic3F1YXJlIiwgImNoYW5nZWRfYXR0cmlidXRlIjogImNvbG9yIiwgImJlZm9yZSI6IGRpc3RyYWN0b3JfY29sb3IsICJhZnRlciI6IGRpc3RyYWN0b3JfYWZ0ZXJ9CiAgICAgICAgICAgICAgICBhbnN3ZXIgPSB0YXJnZXRfY29sb3IKICAgICAgICAgICAgcm93cy5hcHBlbmQoewogICAgICAgICAgICAgICAgImV4YW1wbGVfaWQiOiBmIntzY2VuZV9pZH0te2tpbmR9IiwKICAgICAgICAgICAgICAgICJwYWlyX2lkIjogc2NlbmVfaWQsCiAgICAgICAgICAgICAgICAic2NlbmVfaWQiOiBzY2VuZV9pZCwKICAgICAgICAgICAgICAgICJzcGxpdCI6ICJkZXZlbG9wbWVudCIgaWYgaW5kZXggJSA1ID09IDAgZWxzZSAidHJhaW4iLAogICAgICAgICAgICAgICAgInZhcmlhbnQiOiBraW5kLAogICAgICAgICAgICAgICAgImltYWdlIjogaW1hZ2VfcmVsLAogICAgICAgICAgICAgICAgInF1ZXN0aW9uIjogcXVlc3Rpb24sCiAgICAgICAgICAgICAgICAiZXhwZWN0ZWRfYW5zd2VyIjogYW5zd2VyLAogICAgICAgICAgICAgICAgIm9yaWdpbmFsX2Fuc3dlciI6IHRhcmdldF9jb2xvciwKICAgICAgICAgICAgICAgICJpbnRlcnZlbnRpb24iOiBpbnRlcnZlbnRpb24sCiAgICAgICAgICAgICAgICAic2NlbmUiOiB7CiAgICAgICAgICAgICAgICAgICAgIm9iamVjdHMiOiBbCiAgICAgICAgICAgICAgICAgICAgICAgIHsibmFtZSI6ICJjaXJjbGUiLCAic2hhcGUiOiAiY2lyY2xlIiwgImNvbG9yIjogdGFyZ2V0X2FmdGVyLCAiY2VudGVyX3giOiB0YXJnZXRfeCwgImNlbnRlcl95IjogSEVJR0hULy8yfSwKICAgICAgICAgICAgICAgICAgICAgICAgeyJuYW1lIjogInNxdWFyZSIsICJzaGFwZSI6ICJzcXVhcmUiLCAiY29sb3IiOiBkaXN0cmFjdG9yX2FmdGVyLCAiY2VudGVyX3giOiBkaXN0cmFjdG9yX3gsICJjZW50ZXJfeSI6IEhFSUdIVC8vMn0sCiAgICAgICAgICAgICAgICAgICAgXQogICAgICAgICAgICAgICAgfSwKICAgICAgICAgICAgICAgICJzZWVkIjogc2VlZCwKICAgICAgICAgICAgICAgICJnZW5lcmF0b3JfdmVyc2lvbiI6ICIwLjEuMCIsCiAgICAgICAgICAgIH0pCiAgICBtYW5pZmVzdCA9IG91dHB1dCAvICJleGFtcGxlcy5qc29ubCIKICAgIHdpdGggbWFuaWZlc3Qub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IlxuIikgYXMgaGFuZGxlOgogICAgICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICAgICAgaGFuZGxlLndyaXRlKGpzb24uZHVtcHMocm93LCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iKQogICAgcmV0dXJuIHsiZXhhbXBsZXMiOiBsZW4ocm93cyksICJzY2VuZXMiOiBjb3VudCwgIm1hbmlmZXN0Ijogc3RyKG1hbmlmZXN0KSwgImltYWdlcyI6IHN0cihpbWFnZXMpfQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dCIsIHR5cGU9UGF0aCwgZGVmYXVsdD1QYXRoKCJkYXRhL3BpbG90LXYwIikpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNvdW50IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTAwKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTcpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcHJpbnQoanNvbi5kdW1wcyhnZW5lcmF0ZShhcmdzLm91dHB1dCwgYXJncy5jb3VudCwgYXJncy5zZWVkKSwgaW5kZW50PTIpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkKCg==', 'sha256': 'f2bd2fa25b7562e88220919a2f80fbc68b5c5e4525367c6797800b81b515b493'}, 'causalsight/generate_binding.py': {'base64': 'IiIiR2VuZXJhdGUgYW5kIGF1ZGl0IHRoZSBiYWxhbmNlZCBvYmplY3QtY29sb3IgYmluZGluZyBleHBlcmltZW50LiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNvcHkKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGl0ZXJ0b29scwppbXBvcnQganNvbgppbXBvcnQgcmFuZG9tCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIsIGRlZmF1bHRkaWN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKZnJvbSBQSUwgaW1wb3J0IEltYWdlLCBJbWFnZUNob3BzLCBJbWFnZURyYXcKZnJvbSAuZ2VuZXJhdGVfcGlsb3QgaW1wb3J0IEJBQ0tHUk9VTkQsIENPTE9SUywgSU5LLCBSR0IsIFdJRFRILCBIRUlHSFQKClZBUklBTlRTID0gKCJvcmlnaW5hbCIsICJjYXVzYWwiLCAibnVpc2FuY2UiLCAicXVlc3Rpb25fc3dpdGNoIikKCgpkZWYgcmVuZGVyKHNjZW5lOiBkaWN0KSAtPiBJbWFnZS5JbWFnZToKICAgIGltYWdlID0gSW1hZ2UubmV3KCJSR0IiLCAoV0lEVEgsIEhFSUdIVCksIEJBQ0tHUk9VTkQpCiAgICBkcmF3ID0gSW1hZ2VEcmF3LkRyYXcoaW1hZ2UpCiAgICBmb3Igb2JqIGluIHNjZW5lWyJvYmplY3RzIl06CiAgICAgICAgeCwgeSwgciA9IG9ialsiY2VudGVyX3giXSwgb2JqWyJjZW50ZXJfeSJdLCBvYmpbInJhZGl1cyJdCiAgICAgICAgYm91bmRzID0gKHgtciwgeS1yLCB4K3IsIHkrcikKICAgICAgICBkcmF3X2ZuID0gZHJhdy5lbGxpcHNlIGlmIG9ialsic2hhcGUiXSA9PSAiY2lyY2xlIiBlbHNlIGRyYXcucmVjdGFuZ2xlCiAgICAgICAgZHJhd19mbihib3VuZHMsIGZpbGw9UkdCW29ialsiY29sb3IiXV0sIG91dGxpbmU9SU5LLCB3aWR0aD0zKQogICAgcmV0dXJuIGltYWdlCgoKZGVmIGdlbmVyYXRlKG91dHB1dDogUGF0aCwgc2VlZDogaW50ID0gMTEwMTcpIC0+IGRpY3Q6CiAgICBpZiBvdXRwdXQuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgRmlsZUV4aXN0c0Vycm9yKGYiRGF0YXNldCB2ZXJzaW9uIGFscmVhZHkgZXhpc3RzOiB7b3V0cHV0fSIpCiAgICAob3V0cHV0IC8gImltYWdlcyIpLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgIHJvd3MgPSBbXQogICAgY29tYmluYXRpb25zID0gbGlzdChpdGVydG9vbHMucHJvZHVjdChyYW5nZSgyKSwgcmFuZ2UoMiksIHJhbmdlKDQpLCByYW5nZSg0KSkpCiAgICAjIERldmVsb3BtZW50IGNvdmVycyBzaGFwZSwgc2lkZSBhbmQgdGFyZ2V0IGNvbG9yOyBldmFsdWF0aW9uIGNvdmVycyB0aGUgZnVsbCBmYWN0b3JpYWwgdHdpY2UuCiAgICBkZXZlbG9wbWVudCA9IFsocywgcCwgdCwgKHQrMipzK3ApICUgNCkgZm9yIHMsIHAsIHQgaW4gaXRlcnRvb2xzLnByb2R1Y3QocmFuZ2UoMiksIHJhbmdlKDIpLCByYW5nZSg0KSldCiAgICBmb3Igc3BsaXQsIHNwZWNzIGluIFsoImRldmVsb3BtZW50IiwgWygwLCBjKSBmb3IgYyBpbiBkZXZlbG9wbWVudF0pLAogICAgICAgICAgICAgICAgICAgICAgICAgKCJldmFsdWF0aW9uIiwgWyhyZXAsIGMpIGZvciByZXAgaW4gcmFuZ2UoMikgZm9yIGMgaW4gY29tYmluYXRpb25zXSldOgogICAgICAgIGZvciBpbmRleCwgKHJlcCwgKHMsIHAsIHQsIGQpKSBpbiBlbnVtZXJhdGUoc3BlY3MpOgogICAgICAgICAgICBwYWlyX2lkID0gZiJiaW5kaW5nLXtzcGxpdH0te2luZGV4OjA0ZH0iCiAgICAgICAgICAgIGZhbWlseV9zZWVkID0gaW50LmZyb21fYnl0ZXMoaGFzaGxpYi5zaGEyNTYoZiJ7c2VlZH06e3BhaXJfaWR9Ii5lbmNvZGUoKSkuZGlnZXN0KClbOjhdLCAiYmlnIikKICAgICAgICAgICAgcm5nID0gcmFuZG9tLlJhbmRvbShmYW1pbHlfc2VlZCkKICAgICAgICAgICAgdGFyZ2V0ID0gKCJjaXJjbGUiLCAic3F1YXJlIilbc10KICAgICAgICAgICAgZGlzdHJhY3RvciA9ICgic3F1YXJlIiwgImNpcmNsZSIpW3NdCiAgICAgICAgICAgIHhzID0gWzExMitybmcucmFuZGludCgtMTQsIDE0KSwgMjcyK3JuZy5yYW5kaW50KC0xNCwgMTQpXQogICAgICAgICAgICBzY2VuZSA9IHsib2JqZWN0cyI6IFt7Im5hbWUiOiBuYW1lLCAic2hhcGUiOiBuYW1lLCAiY29sb3IiOiBDT0xPUlNbY29sb3JdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJjZW50ZXJfeCI6IHhzW3NpZGVdLCAiY2VudGVyX3kiOiAxMjgrcm5nLnJhbmRpbnQoLTEyLCAxMiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInJhZGl1cyI6IHJuZy5yYW5kaW50KDMwLCAzOCl9CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgbmFtZSwgY29sb3IsIHNpZGUgaW4gWyh0YXJnZXQsIHQsIHApLCAoZGlzdHJhY3RvciwgZCwgMS1wKV1dfQogICAgICAgICAgICBmb3IgdmFyaWFudCBpbiBWQVJJQU5UUzoKICAgICAgICAgICAgICAgIHN0YXRlID0gY29weS5kZWVwY29weShzY2VuZSkKICAgICAgICAgICAgICAgIHF1ZXJpZWQgPSB0YXJnZXQKICAgICAgICAgICAgICAgIGludGVydmVudGlvbiA9IE5vbmUKICAgICAgICAgICAgICAgIGlmIHZhcmlhbnQgaW4gKCJjYXVzYWwiLCAibnVpc2FuY2UiKToKICAgICAgICAgICAgICAgICAgICBjaGFuZ2VkX2luZGV4ID0gMCBpZiB2YXJpYW50ID09ICJjYXVzYWwiIGVsc2UgMQogICAgICAgICAgICAgICAgICAgIG9sZF9pbmRleCA9IHQgaWYgY2hhbmdlZF9pbmRleCA9PSAwIGVsc2UgZAogICAgICAgICAgICAgICAgICAgIG9mZnNldCA9IDEgKyAoKHJlcCtzK3ArKGQgaWYgY2hhbmdlZF9pbmRleCA9PSAwIGVsc2UgdCkpICUgMykKICAgICAgICAgICAgICAgICAgICBhZnRlciA9IENPTE9SU1sob2xkX2luZGV4K29mZnNldCkgJSA0XQogICAgICAgICAgICAgICAgICAgIHN0YXRlWyJvYmplY3RzIl1bY2hhbmdlZF9pbmRleF1bImNvbG9yIl0gPSBhZnRlcgogICAgICAgICAgICAgICAgICAgIGludGVydmVudGlvbiA9IHsidHlwZSI6IHZhcmlhbnQsICJjaGFuZ2VkX29iamVjdCI6IHN0YXRlWyJvYmplY3RzIl1bY2hhbmdlZF9pbmRleF1bIm5hbWUiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImNoYW5nZWRfYXR0cmlidXRlIjogImNvbG9yIiwgImJlZm9yZSI6IENPTE9SU1tvbGRfaW5kZXhdLCAiYWZ0ZXIiOiBhZnRlcn0KICAgICAgICAgICAgICAgIGlmIHZhcmlhbnQgPT0gInF1ZXN0aW9uX3N3aXRjaCI6CiAgICAgICAgICAgICAgICAgICAgcXVlcmllZCA9IGRpc3RyYWN0b3IKICAgICAgICAgICAgICAgICAgICBpbnRlcnZlbnRpb24gPSB7InR5cGUiOiB2YXJpYW50LCAiY2hhbmdlZF9hdHRyaWJ1dGUiOiAicXVlcmllZF9zaGFwZSIsICJiZWZvcmUiOiB0YXJnZXQsICJhZnRlciI6IGRpc3RyYWN0b3J9CiAgICAgICAgICAgICAgICBpbWFnZV9yZWwgPSBmImltYWdlcy97cGFpcl9pZH0teydvcmlnaW5hbCcgaWYgdmFyaWFudCA9PSAncXVlc3Rpb25fc3dpdGNoJyBlbHNlIHZhcmlhbnR9LnBuZyIKICAgICAgICAgICAgICAgIGlmIHZhcmlhbnQgIT0gInF1ZXN0aW9uX3N3aXRjaCI6CiAgICAgICAgICAgICAgICAgICAgcmVuZGVyKHN0YXRlKS5zYXZlKG91dHB1dCAvIGltYWdlX3JlbCkKICAgICAgICAgICAgICAgIGFuc3dlciA9IG5leHQob1siY29sb3IiXSBmb3IgbyBpbiBzdGF0ZVsib2JqZWN0cyJdIGlmIG9bIm5hbWUiXSA9PSBxdWVyaWVkKQogICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoeyJleGFtcGxlX2lkIjogZiJ7cGFpcl9pZH0te3ZhcmlhbnR9IiwgInBhaXJfaWQiOiBwYWlyX2lkLCAic2NlbmVfaWQiOiBwYWlyX2lkLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICJzcGxpdCI6IHNwbGl0LCAidmFyaWFudCI6IHZhcmlhbnQsICJpbWFnZSI6IGltYWdlX3JlbCwgInNjZW5lIjogc3RhdGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInF1ZXN0aW9uIjogZiJXaGF0IGNvbG9yIGlzIHRoZSB7cXVlcmllZH0/IEFuc3dlciB3aXRoIG9uZSBjb2xvci4iLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICJxdWVyaWVkX3NoYXBlIjogcXVlcmllZCwgInRhcmdldF9zaGFwZSI6IHRhcmdldCwgInRhcmdldF9zaWRlIjogKCJsZWZ0IiwgInJpZ2h0IilbcF0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImV4cGVjdGVkX2Fuc3dlciI6IGFuc3dlciwgIm9yaWdpbmFsX2Fuc3dlciI6IENPTE9SU1t0XSwgImludGVydmVudGlvbiI6IGludGVydmVudGlvbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAic2VlZCI6IHNlZWQsICJmYW1pbHlfc2VlZCI6IGZhbWlseV9zZWVkLCAiZ2VuZXJhdG9yX3ZlcnNpb24iOiAiYmluZGluZy0xLjAuMCJ9KQogICAgd2l0aCAob3V0cHV0IC8gImV4YW1wbGVzLmpzb25sIikub3BlbigieCIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IlxuIikgYXMgaGFuZGxlOgogICAgICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICAgICAgaGFuZGxlLndyaXRlKGpzb24uZHVtcHMocm93LCBzb3J0X2tleXM9VHJ1ZSkrIlxuIikKICAgIGF1ZGl0ID0gdmFsaWRhdGUob3V0cHV0IC8gImV4YW1wbGVzLmpzb25sIikKICAgIChvdXRwdXQgLyAiYXVkaXQuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhhdWRpdCwgaW5kZW50PTIpKyJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gYXVkaXQKCgpkZWYgdmFsaWRhdGUobWFuaWZlc3Q6IFBhdGgpIC0+IGRpY3Q6CiAgICBncm91cHMgPSBkZWZhdWx0ZGljdChkaWN0KQogICAgcm93cyA9IFtqc29uLmxvYWRzKGxpbmUpIGZvciBsaW5lIGluIG1hbmlmZXN0LnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKS5zcGxpdGxpbmVzKCkgaWYgbGluZS5zdHJpcCgpXQogICAgaWRzID0gc2V0KCkKICAgIG9yaWdpbmFsX2lucHV0cyA9IHNldCgpCiAgICBvcmlnaW5hbF9pbWFnZXNfYnlfc3BsaXQgPSBkZWZhdWx0ZGljdChzZXQpCiAgICBmb3Igcm93IGluIHJvd3M6CiAgICAgICAgaWYgcm93WyJleGFtcGxlX2lkIl0gaW4gaWRzIG9yIHJvd1sidmFyaWFudCJdIGluIGdyb3Vwc1tyb3dbInBhaXJfaWQiXV06CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkR1cGxpY2F0ZSBleGFtcGxlIG9yIGZhbWlseSB2YXJpYW50IikKICAgICAgICBpZHMuYWRkKHJvd1siZXhhbXBsZV9pZCJdKQogICAgICAgIGdyb3Vwc1tyb3dbInBhaXJfaWQiXV1bcm93WyJ2YXJpYW50Il1dID0gcm93CiAgICAgICAgaW1hZ2VfcGF0aCA9IChtYW5pZmVzdC5wYXJlbnQgLyByb3dbImltYWdlIl0pLnJlc29sdmUoKQogICAgICAgIGlmIG5vdCBpbWFnZV9wYXRoLmlzX3JlbGF0aXZlX3RvKG1hbmlmZXN0LnBhcmVudC5yZXNvbHZlKCkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJJbWFnZSBwYXRoIGVzY2FwZXMgZGF0YXNldCIpCiAgICAgICAgd2l0aCBJbWFnZS5vcGVuKGltYWdlX3BhdGgpIGFzIGltYWdlOgogICAgICAgICAgICBpbWFnZSA9IGltYWdlLmNvbnZlcnQoIlJHQiIpCiAgICAgICAgICAgIGlmIEltYWdlQ2hvcHMuZGlmZmVyZW5jZShpbWFnZSwgcmVuZGVyKHJvd1sic2NlbmUiXSkpLmdldGJib3goKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJSZW5kZXJlZCBwaXhlbHMgZGlzYWdyZWUgd2l0aCBtZXRhZGF0YToge3Jvd1snZXhhbXBsZV9pZCddfSIpCiAgICAgICAgICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KGltYWdlLnRvYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKICAgICAgICBxdWVyaWVkID0gbmV4dChvIGZvciBvIGluIHJvd1sic2NlbmUiXVsib2JqZWN0cyJdIGlmIG9bIm5hbWUiXSA9PSByb3dbInF1ZXJpZWRfc2hhcGUiXSkKICAgICAgICBpZiBxdWVyaWVkWyJjb2xvciJdICE9IHJvd1siZXhwZWN0ZWRfYW5zd2VyIl0gb3Igcm93WyJxdWVzdGlvbiJdICE9IGYiV2hhdCBjb2xvciBpcyB0aGUge3Jvd1sncXVlcmllZF9zaGFwZSddfT8gQW5zd2VyIHdpdGggb25lIGNvbG9yLiI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlF1ZXN0aW9uL2xhYmVsIG1pc21hdGNoIikKICAgICAgICBpZiByb3dbInZhcmlhbnQiXSA9PSAib3JpZ2luYWwiOgogICAgICAgICAgICBpZGVudGl0eSA9IChkaWdlc3QsIHJvd1sicXVlc3Rpb24iXSkKICAgICAgICAgICAgaWYgaWRlbnRpdHkgaW4gb3JpZ2luYWxfaW5wdXRzOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUmVwZWF0ZWQgb3JpZ2luYWwgaW5wdXQiKQogICAgICAgICAgICBvcmlnaW5hbF9pbnB1dHMuYWRkKGlkZW50aXR5KQogICAgICAgICAgICBvcmlnaW5hbF9pbWFnZXNfYnlfc3BsaXRbcm93WyJzcGxpdCJdXS5hZGQoZGlnZXN0KQogICAgY291bnRzLCB0cmFuc2l0aW9ucyA9IGRlZmF1bHRkaWN0KENvdW50ZXIpLCBkZWZhdWx0ZGljdChDb3VudGVyKQogICAgZm9yIGZhbWlseSwgdmFyaWFudHMgaW4gZ3JvdXBzLml0ZW1zKCk6CiAgICAgICAgaWYgc2V0KHZhcmlhbnRzKSAhPSBzZXQoVkFSSUFOVFMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW5jb21wbGV0ZSBmYW1pbHk6IHtmYW1pbHl9IikKICAgICAgICBvcmlnaW5hbCA9IHZhcmlhbnRzWyJvcmlnaW5hbCJdCiAgICAgICAgdGFyZ2V0LCBkaXN0cmFjdG9yID0gb3JpZ2luYWxbInNjZW5lIl1bIm9iamVjdHMiXQogICAgICAgIGxlZnQgPSB0YXJnZXRbImNlbnRlcl94Il0gPCBkaXN0cmFjdG9yWyJjZW50ZXJfeCJdCiAgICAgICAgaWYgb3JpZ2luYWxbInRhcmdldF9zaWRlIl0gIT0gKCJsZWZ0IiBpZiBsZWZ0IGVsc2UgInJpZ2h0Iik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRhcmdldC1zaWRlIG1ldGFkYXRhIG1pc21hdGNoIikKICAgICAgICBjb3VudHNbb3JpZ2luYWxbInNwbGl0Il1dWyh0YXJnZXRbIm5hbWUiXSwgb3JpZ2luYWxbInRhcmdldF9zaWRlIl0sIHRhcmdldFsiY29sb3IiXSwgZGlzdHJhY3RvclsiY29sb3IiXSldICs9IDEKICAgICAgICBmb3IgdmFyaWFudCwgY2hhbmdlZF9pbmRleCBpbiBbKCJjYXVzYWwiLCAwKSwgKCJudWlzYW5jZSIsIDEpXToKICAgICAgICAgICAgcm93ID0gdmFyaWFudHNbdmFyaWFudF0KICAgICAgICAgICAgZGlmZmVyZW5jZXMgPSBbKGksIGtleSkgZm9yIGksIChhLCBiKSBpbiBlbnVtZXJhdGUoemlwKG9yaWdpbmFsWyJzY2VuZSJdWyJvYmplY3RzIl0sIHJvd1sic2NlbmUiXVsib2JqZWN0cyJdKSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGtleSBpbiBhIGlmIGFba2V5XSAhPSBiW2tleV1dCiAgICAgICAgICAgIGNoYW5nZWQgPSByb3dbInNjZW5lIl1bIm9iamVjdHMiXVtjaGFuZ2VkX2luZGV4XQogICAgICAgICAgICBiZWZvcmUgPSBvcmlnaW5hbFsic2NlbmUiXVsib2JqZWN0cyJdW2NoYW5nZWRfaW5kZXhdCiAgICAgICAgICAgIGV4cGVjdGVkX2VkaXQgPSB7InR5cGUiOiB2YXJpYW50LCAiY2hhbmdlZF9vYmplY3QiOiBjaGFuZ2VkWyJuYW1lIl0sICJjaGFuZ2VkX2F0dHJpYnV0ZSI6ICJjb2xvciIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImJlZm9yZSI6IGJlZm9yZVsiY29sb3IiXSwgImFmdGVyIjogY2hhbmdlZFsiY29sb3IiXX0KICAgICAgICAgICAgaWYgZGlmZmVyZW5jZXMgIT0gWyhjaGFuZ2VkX2luZGV4LCAiY29sb3IiKV0gb3Igcm93WyJpbnRlcnZlbnRpb24iXSAhPSBleHBlY3RlZF9lZGl0OgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW50ZXJ2ZW50aW9uIGNoYW5nZXMgdGhlIHdyb25nIGF0dHJpYnV0ZSBvciBoYXMgaW52YWxpZCBtZXRhZGF0YSIpCiAgICAgICAgICAgIGlmIHJvd1sicXVlc3Rpb24iXSAhPSBvcmlnaW5hbFsicXVlc3Rpb24iXToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlZpc3VhbCBpbnRlcnZlbnRpb24gY2hhbmdlcyBxdWVzdGlvbiIpCiAgICAgICAgICAgIHRyYW5zaXRpb25zW29yaWdpbmFsWyJzcGxpdCJdKyJfIit2YXJpYW50XVtiZWZvcmVbImNvbG9yIl0rIi0+IitjaGFuZ2VkWyJjb2xvciJdXSArPSAxCiAgICAgICAgc3dpdGNoZWQgPSB2YXJpYW50c1sicXVlc3Rpb25fc3dpdGNoIl0KICAgICAgICBpZiBzd2l0Y2hlZFsic2NlbmUiXSAhPSBvcmlnaW5hbFsic2NlbmUiXSBvciBzd2l0Y2hlZFsiaW1hZ2UiXSAhPSBvcmlnaW5hbFsiaW1hZ2UiXSBvciBzd2l0Y2hlZFsicXVlcmllZF9zaGFwZSJdICE9IGRpc3RyYWN0b3JbIm5hbWUiXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUXVlc3Rpb24gc3dpdGNoIG11c3QgcHJlc2VydmUgdGhlIGltYWdlIGFuZCBxdWVyeSB0aGUgb3RoZXIgb2JqZWN0IikKICAgICAgICBpZiBsZW4oe3JbInNwbGl0Il0gZm9yIHIgaW4gdmFyaWFudHMudmFsdWVzKCl9KSAhPSAxOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJGYW1pbHkgc3BhbnMgc3BsaXRzIikKICAgIGlmIG9yaWdpbmFsX2ltYWdlc19ieV9zcGxpdFsiZGV2ZWxvcG1lbnQiXSAmIG9yaWdpbmFsX2ltYWdlc19ieV9zcGxpdFsiZXZhbHVhdGlvbiJdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkltYWdlIG92ZXJsYXAgYmV0d2VlbiBkZXZlbG9wbWVudCBhbmQgZXZhbHVhdGlvbiIpCiAgICBpZiBsZW4oY291bnRzWyJldmFsdWF0aW9uIl0pICE9IDY0IG9yIHNldChjb3VudHNbImV2YWx1YXRpb24iXS52YWx1ZXMoKSkgIT0gezJ9OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV2YWx1YXRpb24gbXVzdCBoYXZlIHR3byBmYW1pbGllcyBwZXIgZmFjdG9yaWFsIGNlbGwiKQogICAgaWYgc3VtKGNvdW50c1siZGV2ZWxvcG1lbnQiXS52YWx1ZXMoKSkgIT0gMTY6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRGV2ZWxvcG1lbnQgbXVzdCBjb250YWluIDE2IGZhbWlsaWVzIikKICAgIHJldHVybiB7ImZhbWlsaWVzIjogbGVuKGdyb3VwcyksICJyb3dzIjogbGVuKHJvd3MpLCAiaW1hZ2VzIjogbGVuKGxpc3QoKG1hbmlmZXN0LnBhcmVudCAvICJpbWFnZXMiKS5nbG9iKCIqLnBuZyIpKSksCiAgICAgICAgICAgICJtYW5pZmVzdF9zaGEyNTYiOiBoYXNobGliLnNoYTI1NihtYW5pZmVzdC5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICAgICAiZmFtaWxpZXNfYnlfc3BsaXQiOiB7czogc3VtKGMudmFsdWVzKCkpIGZvciBzLCBjIGluIGNvdW50cy5pdGVtcygpfSwKICAgICAgICAgICAgImV2YWx1YXRpb25fY2VsbHMiOiBsZW4oY291bnRzWyJldmFsdWF0aW9uIl0pLCAiZmFtaWxpZXNfcGVyX2V2YWx1YXRpb25fY2VsbCI6IDIsCiAgICAgICAgICAgICJ0cmFuc2l0aW9uX2NvdW50cyI6IHtrZXk6IGRpY3Qoc29ydGVkKHZhbHVlLml0ZW1zKCkpKSBmb3Iga2V5LCB2YWx1ZSBpbiB0cmFuc2l0aW9ucy5pdGVtcygpfSwKICAgICAgICAgICAgImNoZWNrcyI6ICJFeGFjdCByZW5kZXJlZCBwaXhlbHMsIHNpbmdsZSBjb2xvciBjaGFuZ2VzLCBxdWVzdGlvbiBzd2l0Y2hlcywgbGFiZWxzLCB1bmlxdWUgb3JpZ2luYWwgaW5wdXRzLCBzcGxpdCBzZXBhcmF0aW9uIGFuZCBmYWN0b3JpYWwgYmFsYW5jZSBwYXNzZWQuIn0KCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9UGF0aCgiZGF0YS9iaW5kaW5nLXYxIikpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD0xMTAxNykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tdmFsaWRhdGUiLCB0eXBlPVBhdGgpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcHJpbnQoanNvbi5kdW1wcyh2YWxpZGF0ZShhcmdzLnZhbGlkYXRlKSBpZiBhcmdzLnZhbGlkYXRlIGVsc2UgZ2VuZXJhdGUoYXJncy5vdXRwdXQsIGFyZ3Muc2VlZCksIGluZGVudD0yKSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==', 'sha256': '0bbc74cc0f1fab427d163e8095b5ceb1d126a9e05fac2e0903f1f63a5b2f7d14'}, 'causalsight/evaluate_binding.py': {'base64': 'IiIiRXZhbHVhdGUgYmluZGluZy12MSBhZ2FpbnN0IGl0cyBtYW5pZmVzdCwgcmV0YWluaW5nIG1pc3NpbmcgYW5kIGZhaWxlZCByZXF1ZXN0cy4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCByZQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyLCBkZWZhdWx0ZGljdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClZBUklBTlRTID0gKCJvcmlnaW5hbCIsICJjYXVzYWwiLCAibnVpc2FuY2UiLCAicXVlc3Rpb25fc3dpdGNoIikKCgpkZWYgcGFyc2VfYW5zd2VyKHRleHQ6IHN0ciB8IE5vbmUpIC0+IHN0ciB8IE5vbmU6CiAgICAjIERvIG5vdCBleHRyYWN0IGNvbG9ycyBmcm9tIHJlZnVzYWxzLCBuZWdhdGlvbnMsIG9yIGV4cGxhbmF0aW9ucy4KICAgIGlmIG5vdCBpc2luc3RhbmNlKHRleHQsIHN0cik6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIG1hdGNoID0gcmUuZnVsbG1hdGNoKHIiXHMqKHJlZHxibHVlfGdyZWVufHllbGxvdylbLiFdP1xzKiIsIHRleHQsIHJlLklHTk9SRUNBU0UpCiAgICByZXR1cm4gbWF0Y2guZ3JvdXAoMSkubG93ZXIoKSBpZiBtYXRjaCBlbHNlIE5vbmUKCgpkZWYgcmF0ZShzdWNjZXNzOiBpbnQsIHRvdGFsOiBpbnQpIC0+IGRpY3Q6CiAgICByZXR1cm4geyJudW1lcmF0b3IiOiBzdWNjZXNzLCAiZGVub21pbmF0b3IiOiB0b3RhbCwgInJhdGUiOiBzdWNjZXNzL3RvdGFsIGlmIHRvdGFsIGVsc2UgTm9uZX0KCgpkZWYgc3VtbWFyaXplKHBhaXJzOiBsaXN0W2RpY3RdKSAtPiBkaWN0OgogICAgbiA9IGxlbihwYWlycykKICAgIGNvcnJlY3Rfb3JpZ2luYWxzID0gW3AgZm9yIHAgaW4gcGFpcnMgaWYgcFsib3JpZ2luYWxfY29ycmVjdCJdXQogICAgZGlmZmVyZW50ID0gW3AgZm9yIHAgaW4gcGFpcnMgaWYgbm90IHBbInNhbWVfY29sb3IiXV0KICAgIHNhbWUgPSBbcCBmb3IgcCBpbiBwYWlycyBpZiBwWyJzYW1lX2NvbG9yIl1dCiAgICBkZWYgbWV0cmljKGtleTogc3RyLCBwb3B1bGF0aW9uOiBsaXN0W2RpY3RdID0gcGFpcnMpIC0+IGRpY3Q6CiAgICAgICAgcmV0dXJuIHJhdGUoc3VtKHBba2V5XSBmb3IgcCBpbiBwb3B1bGF0aW9uKSwgbGVuKHBvcHVsYXRpb24pKQogICAgcmV0dXJuIHsKICAgICAgICAiZmFtaWxpZXMiOiBuLCAib3JpZ2luYWxfYWNjdXJhY3kiOiBtZXRyaWMoIm9yaWdpbmFsX2NvcnJlY3QiKSwKICAgICAgICAiY2F1c2FsX3RhcmdldF9hY2N1cmFjeSI6IG1ldHJpYygiY2F1c2FsX2NvcnJlY3QiKSwKICAgICAgICAiY29ycmVjdF9jYXVzYWxfdXBkYXRlIjogbWV0cmljKCJjb3JyZWN0X2NhdXNhbF91cGRhdGUiKSwKICAgICAgICAibnVpc2FuY2VfYW5zd2VyX3ByZXNlcnZhdGlvbiI6IG1ldHJpYygibnVpc2FuY2VfcHJlc2VydmVkIiksCiAgICAgICAgImNvcnJlY3RfbnVpc2FuY2VfcHJlc2VydmF0aW9uIjogbWV0cmljKCJjb3JyZWN0X251aXNhbmNlX3ByZXNlcnZhdGlvbiIpLAogICAgICAgICJjb25zaXN0ZW50bHlfd3JvbmdfbnVpc2FuY2VfcHJlc2VydmF0aW9uIjogbWV0cmljKCJ3cm9uZ19udWlzYW5jZV9wcmVzZXJ2YXRpb24iKSwKICAgICAgICAiY29tcGxldGVfdHJpcGxldF9jb3JyZWN0bmVzcyI6IG1ldHJpYygidHJpcGxldF9jb3JyZWN0IiksCiAgICAgICAgInF1ZXN0aW9uX3N3aXRjaF9jb3JyZWN0X2RpZmZlcmVudF9jb2xvcnMiOiBtZXRyaWMoInF1ZXN0aW9uX3N3aXRjaF9jb3JyZWN0IiwgZGlmZmVyZW50KSwKICAgICAgICAicXVlc3Rpb25fc3dpdGNoX2NvcnJlY3Rfc2FtZV9jb2xvcnMiOiBtZXRyaWMoInF1ZXN0aW9uX3N3aXRjaF9jb3JyZWN0Iiwgc2FtZSksCiAgICAgICAgImNvbmRpdGlvbmVkX29uX29yaWdpbmFsX2NvcnJlY3QiOiB7CiAgICAgICAgICAgICJjb3JyZWN0X2NhdXNhbF91cGRhdGUiOiBtZXRyaWMoImNvcnJlY3RfY2F1c2FsX3VwZGF0ZSIsIGNvcnJlY3Rfb3JpZ2luYWxzKSwKICAgICAgICAgICAgIm51aXNhbmNlX2Fuc3dlcl9wcmVzZXJ2YXRpb24iOiBtZXRyaWMoIm51aXNhbmNlX3ByZXNlcnZlZCIsIGNvcnJlY3Rfb3JpZ2luYWxzKSwKICAgICAgICAgICAgImNvbXBsZXRlX3RyaXBsZXRfY29ycmVjdG5lc3MiOiBtZXRyaWMoInRyaXBsZXRfY29ycmVjdCIsIGNvcnJlY3Rfb3JpZ2luYWxzKSwKICAgICAgICB9LAogICAgfQoKCmRlZiBldmFsdWF0ZShtYW5pZmVzdDogUGF0aCwgcmF3X3BhdGg6IFBhdGgsIG91dHB1dF9kaXI6IFBhdGgsIHNwbGl0OiBzdHIgPSAiZXZhbHVhdGlvbiIpIC0+IGRpY3Q6CiAgICBleHBlY3RlZCA9IHtyWyJleGFtcGxlX2lkIl06IHIgZm9yIHIgaW4gKGpzb24ubG9hZHMobGluZSkgZm9yIGxpbmUgaW4gbWFuaWZlc3QucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpLnNwbGl0bGluZXMoKSBpZiBsaW5lLnN0cmlwKCkpCiAgICAgICAgICAgICAgICBpZiByWyJzcGxpdCJdID09IHNwbGl0fQogICAgaWYgbm90IGV4cGVjdGVkOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJObyBtYW5pZmVzdCBleGFtcGxlcyBmb3Igc3BsaXQge3NwbGl0fSIpCiAgICBzYXZlZCA9IHt9CiAgICBpZGVudGl0aWVzID0gc2V0KCkKICAgIGRpYWdub3N0aWNzID0gQ291bnRlcigpCiAgICBmb3IgbGluZSBpbiByYXdfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpOgogICAgICAgIGlmIG5vdCBsaW5lLnN0cmlwKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcm93ID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgIGtleSA9IHJvd1siZXhhbXBsZV9pZCJdCiAgICAgICAgaWYga2V5IG5vdCBpbiBleHBlY3RlZDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuZXhwZWN0ZWQgZXhhbXBsZSBmb3Igc2VsZWN0ZWQgc3BsaXQ6IHtrZXl9IikKICAgICAgICBpZiBrZXkgaW4gc2F2ZWQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJEdXBsaWNhdGUgcHJlZGljdGlvbjoge2tleX0iKQogICAgICAgIGZvciBmaWVsZCBpbiAoInBhaXJfaWQiLCAidmFyaWFudCIsICJzcGxpdCIsICJleHBlY3RlZF9hbnN3ZXIiLCAicXVlc3Rpb24iLCAiaW1hZ2UiKToKICAgICAgICAgICAgaWYgcm93LmdldChmaWVsZCkgIT0gZXhwZWN0ZWRba2V5XVtmaWVsZF06CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiUHJlZGljdGlvbiBtZXRhZGF0YSBkaXNhZ3JlZXMgd2l0aCBtYW5pZmVzdDoge2tleX0sIHtmaWVsZH0iKQogICAgICAgIGlkZW50aXRpZXMuYWRkKChyb3cuZ2V0KCJydW5faWQiKSwgcm93LmdldCgicHJvdmlkZXIiKSwgcm93LmdldCgibW9kZWwiKSkpCiAgICAgICAgc2F2ZWRba2V5XSA9IHJvdwogICAgaWYgbGVuKGlkZW50aXRpZXMpID4gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDYW5ub3QgY29tYmluZSBkaWZmZXJlbnQgcnVucy9wcm92aWRlcnMvbW9kZWxzIikKICAgIGdyb3VwcyA9IGRlZmF1bHRkaWN0KGRpY3QpCiAgICBmb3Iga2V5LCBleGFtcGxlIGluIGV4cGVjdGVkLml0ZW1zKCk6CiAgICAgICAgcm93ID0gc2F2ZWQuZ2V0KGtleSwgeyJzdGF0dXMiOiAibWlzc2luZyJ9KQogICAgICAgIHN0YXR1cyA9IHJvdy5nZXQoInN0YXR1cyIsICJlcnJvciIpCiAgICAgICAgcGFyc2VkID0gcGFyc2VfYW5zd2VyKHJvdy5nZXQoInJhd19yZXNwb25zZSIpKSBpZiBzdGF0dXMgPT0gIm9rIiBlbHNlIE5vbmUKICAgICAgICBkaWFnbm9zdGljc1tzdGF0dXNdICs9IDEKICAgICAgICBpZiBzdGF0dXMgPT0gIm9rIjoKICAgICAgICAgICAgZGlhZ25vc3RpY3NbInBhcnNlX2ZhaWx1cmUiXSArPSBwYXJzZWQgaXMgTm9uZQogICAgICAgICAgICBkaWFnbm9zdGljc1siZW1wdHlfZmluYWxfYW5zd2VyIl0gKz0gbm90IHJvdy5nZXQoInJhd19yZXNwb25zZSIsICIiKS5zdHJpcCgpCiAgICAgICAgICAgIHByb3ZpZGVyID0gcm93LmdldCgicHJvdmlkZXJfcmVzcG9uc2UiLCB7fSkKICAgICAgICAgICAgZGlhZ25vc3RpY3NbImJ1ZGdldF9yZWFjaGVkIl0gKz0gYm9vbChyb3cuZ2V0KCJidWRnZXRfcmVhY2hlZCIpIG9yIHByb3ZpZGVyLmdldCgiZG9uZV9yZWFzb24iKSA9PSAibGVuZ3RoIikKICAgICAgICBncm91cHNbZXhhbXBsZVsicGFpcl9pZCJdXVtleGFtcGxlWyJ2YXJpYW50Il1dID0geyJleGFtcGxlIjogZXhhbXBsZSwgImFuc3dlciI6IHBhcnNlZCwgInN0YXR1cyI6IHN0YXR1c30KICAgIHBhaXJzID0gW10KICAgIGZvciBwYWlyX2lkLCBtZW1iZXJzIGluIHNvcnRlZChncm91cHMuaXRlbXMoKSk6CiAgICAgICAgaWYgc2V0KG1lbWJlcnMpICE9IHNldChWQVJJQU5UUyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJNYW5pZmVzdCBmYW1pbHkgaW5jb21wbGV0ZToge3BhaXJfaWR9IikKICAgICAgICBwcmVkaWN0aW9uID0ge3Y6IG1lbWJlcnNbdl1bImFuc3dlciJdIGZvciB2IGluIFZBUklBTlRTfQogICAgICAgIGNvcnJlY3QgPSB7djogcHJlZGljdGlvblt2XSBpcyBub3QgTm9uZSBhbmQgcHJlZGljdGlvblt2XSA9PSBtZW1iZXJzW3ZdWyJleGFtcGxlIl1bImV4cGVjdGVkX2Fuc3dlciJdIGZvciB2IGluIFZBUklBTlRTfQogICAgICAgIG9yaWdpbmFsID0gbWVtYmVyc1sib3JpZ2luYWwiXVsiZXhhbXBsZSJdCiAgICAgICAgb2JqZWN0cyA9IG9yaWdpbmFsWyJzY2VuZSJdWyJvYmplY3RzIl0KICAgICAgICBpbnZhcmlhbnQgPSBwcmVkaWN0aW9uWyJvcmlnaW5hbCJdIGlzIG5vdCBOb25lIGFuZCBwcmVkaWN0aW9uWyJvcmlnaW5hbCJdID09IHByZWRpY3Rpb25bIm51aXNhbmNlIl0KICAgICAgICBjaGFuZ2VkID0gcHJlZGljdGlvblsib3JpZ2luYWwiXSBpcyBub3QgTm9uZSBhbmQgcHJlZGljdGlvblsiY2F1c2FsIl0gaXMgbm90IE5vbmUgYW5kIHByZWRpY3Rpb25bIm9yaWdpbmFsIl0gIT0gcHJlZGljdGlvblsiY2F1c2FsIl0KICAgICAgICBwYWlycy5hcHBlbmQoeyJwYWlyX2lkIjogcGFpcl9pZCwgInByZWRpY3Rpb25zIjogcHJlZGljdGlvbiwKICAgICAgICAgICAgICAgICAgICAgICJleHBlY3RlZCI6IHt2OiBtZW1iZXJzW3ZdWyJleGFtcGxlIl1bImV4cGVjdGVkX2Fuc3dlciJdIGZvciB2IGluIFZBUklBTlRTfSwKICAgICAgICAgICAgICAgICAgICAgICJzdGF0dXNlcyI6IHt2OiBtZW1iZXJzW3ZdWyJzdGF0dXMiXSBmb3IgdiBpbiBWQVJJQU5UU30sCiAgICAgICAgICAgICAgICAgICAgICAidGFyZ2V0X3NoYXBlIjogb3JpZ2luYWxbInRhcmdldF9zaGFwZSJdLCAidGFyZ2V0X3NpZGUiOiBvcmlnaW5hbFsidGFyZ2V0X3NpZGUiXSwKICAgICAgICAgICAgICAgICAgICAgICJ0YXJnZXRfY29sb3IiOiBvcmlnaW5hbFsiZXhwZWN0ZWRfYW5zd2VyIl0sICJzYW1lX2NvbG9yIjogb2JqZWN0c1swXVsiY29sb3IiXSA9PSBvYmplY3RzWzFdWyJjb2xvciJdLAogICAgICAgICAgICAgICAgICAgICAgIm9yaWdpbmFsX2NvcnJlY3QiOiBjb3JyZWN0WyJvcmlnaW5hbCJdLCAiY2F1c2FsX2NvcnJlY3QiOiBjb3JyZWN0WyJjYXVzYWwiXSwKICAgICAgICAgICAgICAgICAgICAgICJjb3JyZWN0X2NhdXNhbF91cGRhdGUiOiBjb3JyZWN0WyJvcmlnaW5hbCJdIGFuZCBjb3JyZWN0WyJjYXVzYWwiXSBhbmQgY2hhbmdlZCwKICAgICAgICAgICAgICAgICAgICAgICJudWlzYW5jZV9wcmVzZXJ2ZWQiOiBpbnZhcmlhbnQsCiAgICAgICAgICAgICAgICAgICAgICAiY29ycmVjdF9udWlzYW5jZV9wcmVzZXJ2YXRpb24iOiBjb3JyZWN0WyJvcmlnaW5hbCJdIGFuZCBjb3JyZWN0WyJudWlzYW5jZSJdIGFuZCBpbnZhcmlhbnQsCiAgICAgICAgICAgICAgICAgICAgICAid3JvbmdfbnVpc2FuY2VfcHJlc2VydmF0aW9uIjogaW52YXJpYW50IGFuZCBub3QgY29ycmVjdFsib3JpZ2luYWwiXSwKICAgICAgICAgICAgICAgICAgICAgICJ0cmlwbGV0X2NvcnJlY3QiOiBhbGwoY29ycmVjdFt2XSBmb3IgdiBpbiAoIm9yaWdpbmFsIiwgImNhdXNhbCIsICJudWlzYW5jZSIpKSwKICAgICAgICAgICAgICAgICAgICAgICJxdWVzdGlvbl9zd2l0Y2hfY29ycmVjdCI6IGNvcnJlY3RbIm9yaWdpbmFsIl0gYW5kIGNvcnJlY3RbInF1ZXN0aW9uX3N3aXRjaCJdfSkKICAgIG1ldHJpY3MgPSB7Im1ldHJpY192ZXJzaW9uIjogImJpbmRpbmctMS4wLjAiLCAic3BsaXQiOiBzcGxpdCwgKipzdW1tYXJpemUocGFpcnMpLAogICAgICAgICAgICAgICAicmVxdWVzdHMiOiB7ImV4cGVjdGVkIjogbGVuKGV4cGVjdGVkKSwgInNhdmVkIjogbGVuKHNhdmVkKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICoqe2tleTogZGlhZ25vc3RpY3Nba2V5XSBmb3Iga2V5IGluICgib2siLCAiZXJyb3IiLCAibWlzc2luZyIsICJwYXJzZV9mYWlsdXJlIiwgImVtcHR5X2ZpbmFsX2Fuc3dlciIsICJidWRnZXRfcmVhY2hlZCIpfX0sCiAgICAgICAgICAgICAgICJwYXJzZXIiOiAiRXhhY3RseSBvbmUgY29sb3Igd29yZCwgb3B0aW9uYWxseSBmb2xsb3dlZCBieSAuIG9yICE7IG5vIHByb3NlIG9yIHRoaW5raW5nLWZpZWxkIGV4dHJhY3Rpb24uIiwKICAgICAgICAgICAgICAgImludGVycHJldGF0aW9uIjogIkRlc2NyaXB0aXZlIHJlc3VsdHMgZm9yIGEgZml4ZWQgYmFsYW5jZWQgc3ludGhldGljIGdyaWQ7IG5vIHBvcHVsYXRpb24gY29uZmlkZW5jZSBpbnRlcnZhbHMgb3IgZ2VuZXJhbCBncm91bmRpbmcgY2xhaW0uIiwKICAgICAgICAgICAgICAgInN0cmF0YSI6IHtmaWVsZDoge3N0cih2YWx1ZSk6IHN1bW1hcml6ZShbcCBmb3IgcCBpbiBwYWlycyBpZiBwW2ZpZWxkXSA9PSB2YWx1ZV0pCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgdmFsdWUgaW4gc29ydGVkKHtwW2ZpZWxkXSBmb3IgcCBpbiBwYWlyc30pfQogICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBmaWVsZCBpbiAoInRhcmdldF9zaGFwZSIsICJ0YXJnZXRfc2lkZSIsICJ0YXJnZXRfY29sb3IiLCAic2FtZV9jb2xvciIpfX0KICAgIGlmIGFueShyLmdldCgiZXhwYW5kZWRfdGV4dF9vbmx5IikgZm9yIHIgaW4gc2F2ZWQudmFsdWVzKCkpOgogICAgICAgIG1ldHJpY3NbInRleHRfb25seV9wcm9qZWN0aW9uIl0gPSB7CiAgICAgICAgICAgICJhY3R1YWxfcmVxdWVzdHMiOiBsZW4oe3JbInNvdXJjZV9yZXF1ZXN0X2lkIl0gZm9yIHIgaW4gc2F2ZWQudmFsdWVzKCl9KSwKICAgICAgICAgICAgInByb2plY3RlZF9wcmVkaWN0aW9uX3Jvd3MiOiBsZW4oc2F2ZWQpLAogICAgICAgICAgICAibm90ZSI6ICJSZXBlYXRlZCBpZGVudGljYWwgcXVlc3Rpb25zIHNoYXJlIGEgdGV4dC1vbmx5IHJlc3BvbnNlOyBwcm9qZWN0ZWQgcm93cyBhcmUgbm90IGluZGVwZW5kZW50IHRyaWFscy4iLAogICAgICAgIH0KICAgIGlmIG91dHB1dF9kaXIuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgRmlsZUV4aXN0c0Vycm9yKGYiUmVzdWx0cyBhbHJlYWR5IGV4aXN0OiB7b3V0cHV0X2Rpcn0iKQogICAgb3V0cHV0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUpCiAgICAob3V0cHV0X2RpciAvICJtZXRyaWNzLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWV0cmljcywgaW5kZW50PTIpKyJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICB3aXRoIChvdXRwdXRfZGlyIC8gInBhaXJzLmpzb25sIikub3BlbigieCIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IlxuIikgYXMgaGFuZGxlOgogICAgICAgIGZvciBwYWlyIGluIHBhaXJzOgogICAgICAgICAgICBoYW5kbGUud3JpdGUoanNvbi5kdW1wcyhwYWlyLCBzb3J0X2tleXM9VHJ1ZSkrIlxuIikKICAgIGxpbmVzID0gW2YiIyBCaW5kaW5nIGV4cGVyaW1lbnQ6IHtzcGxpdH0iLCAiIiwgZiJGYW1pbGllczoge2xlbihwYWlycyl9LiBTYXZlZCByZXF1ZXN0czoge2xlbihzYXZlZCl9L3tsZW4oZXhwZWN0ZWQpfS4iLCAiIiwKICAgICAgICAgICAgICJ8IE1lYXN1cmVtZW50IHwgU3VjY2VzcyAvIHRvdGFsIHwgUmF0ZSB8IiwgInwtLS18LS0tOnwtLS06fCJdCiAgICBmb3Iga2V5LCB2YWx1ZSBpbiBtZXRyaWNzLml0ZW1zKCk6CiAgICAgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgZGljdCkgYW5kICJudW1lcmF0b3IiIGluIHZhbHVlOgogICAgICAgICAgICBwZXJjZW50ID0gZiJ7dmFsdWVbJ3JhdGUnXTouMSV9IiBpZiB2YWx1ZVsicmF0ZSJdIGlzIG5vdCBOb25lIGVsc2UgInVuZGVmaW5lZCIKICAgICAgICAgICAgbGluZXMuYXBwZW5kKGYifCB7a2V5fSB8IHt2YWx1ZVsnbnVtZXJhdG9yJ119L3t2YWx1ZVsnZGVub21pbmF0b3InXX0gfCB7cGVyY2VudH0gfCIpCiAgICBsaW5lcyArPSBbIiIsIG1ldHJpY3NbImludGVycHJldGF0aW9uIl0sICIiLCAiTWlzc2luZywgZmFpbGVkIGFuZCB1bnBhcnNlYWJsZSByZXF1ZXN0cyByZW1haW4gaW4gZGVub21pbmF0b3JzLiJdCiAgICAob3V0cHV0X2RpciAvICJyZXBvcnQubWQiKS53cml0ZV90ZXh0KCJcbiIuam9pbihsaW5lcykrIlxuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHJldHVybiBtZXRyaWNzCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YXNldCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcnVuIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQtZGlyIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zcGxpdCIsIGNob2ljZXM9KCJkZXZlbG9wbWVudCIsICJldmFsdWF0aW9uIiksIGRlZmF1bHQ9ImV2YWx1YXRpb24iKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHJlc3VsdCA9IGV2YWx1YXRlKGFyZ3MuZGF0YXNldCwgYXJncy5ydW4sIGFyZ3Mub3V0cHV0X2RpciwgYXJncy5zcGxpdCkKICAgIHByaW50KGpzb24uZHVtcHMoe2s6IHYgZm9yIGssIHYgaW4gcmVzdWx0Lml0ZW1zKCkgaWYgayAhPSAic3RyYXRhIn0sIGluZGVudD0yKSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==', 'sha256': '243c6500e51423d66a48c107ce0f3b2dc9f818e3ae3cadb0feae34b79166b196'}, 'causalsight/run_binding.py': {'base64': 'IiIiUnVuIHRoZSBmcmVlLWNvbXB1dGUgYmluZGluZyBleHBlcmltZW50IHdpdGggaW1tdXRhYmxlLCByZXN1bWFibGUgYXJ0aWZhY3RzLiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHBsYXRmb3JtCmltcG9ydCB0aW1lCmltcG9ydCB1dWlkCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHVybGxpYi5yZXF1ZXN0IGltcG9ydCBSZXF1ZXN0LCB1cmxvcGVuCgpmcm9tIC5ldmFsdWF0ZV9iaW5kaW5nIGltcG9ydCBldmFsdWF0ZQoKCmRlZiBzaGEyNTYocGF0aDogUGF0aCkgLT4gc3RyOgogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHBhdGgucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKQoKCmNsYXNzIFRyYW5zZm9ybWVyc0FkYXB0ZXI6CiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWxfaWQ6IHN0ciwgbWF4X3Rva2VuczogaW50LCByZXZpc2lvbjogc3RyIHwgTm9uZSA9IE5vbmUpOgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vIGZyZWUgR1BVIHJ1bnRpbWUgYXZhaWxhYmxlLiBVc2UgdGhlIGZyZWUgQ29sYWIgbm90ZWJvb2s7IGRvIG5vdCBwdXJjaGFzZSBjb21wdXRlLiIpCiAgICAgICAgaW1wb3J0IHRyYW5zZm9ybWVycwogICAgICAgIGltcG9ydCBhY2NlbGVyYXRlCiAgICAgICAgZnJvbSBodWdnaW5nZmFjZV9odWIgaW1wb3J0IEhmQXBpCiAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IFF3ZW4zVkxGb3JDb25kaXRpb25hbEdlbmVyYXRpb24sIEF1dG9Qcm9jZXNzb3IKICAgICAgICBzZWxmLnRvcmNoID0gdG9yY2gKICAgICAgICBzZWxmLm1heF90b2tlbnMgPSBtYXhfdG9rZW5zCiAgICAgICAgc2VsZi5yZXZpc2lvbiA9IHJldmlzaW9uIG9yIEhmQXBpKCkubW9kZWxfaW5mbyhtb2RlbF9pZCkuc2hhCiAgICAgICAgcHJvcGVydGllcyA9IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9wcm9wZXJ0aWVzKDApCiAgICAgICAgaWYgcHJvcGVydGllcy50b3RhbF9tZW1vcnkgPCA2KjIqKjMwOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5lZWQgYXQgbGVhc3QgNiBHaUIgR1BVIG1lbW9yeSBmb3IgdGhpcyBGUDE2IGNvbmZpZ3VyYXRpb24iKQogICAgICAgIHNlbGYubWV0YWRhdGEgPSB7Im1vZGVsIjogbW9kZWxfaWQsICJyZXZpc2lvbiI6IHNlbGYucmV2aXNpb24sICJ0b3JjaCI6IHRvcmNoLl9fdmVyc2lvbl9fLAogICAgICAgICAgICAgICAgICAgICAgICAgInRyYW5zZm9ybWVycyI6IHRyYW5zZm9ybWVycy5fX3ZlcnNpb25fXywgImFjY2VsZXJhdGUiOiBhY2NlbGVyYXRlLl9fdmVyc2lvbl9fLAogICAgICAgICAgICAgICAgICAgICAgICAgImdwdSI6IHByb3BlcnRpZXMubmFtZSwgInZyYW1fYnl0ZXMiOiBwcm9wZXJ0aWVzLnRvdGFsX21lbW9yeSwKICAgICAgICAgICAgICAgICAgICAgICAgICJjdWRhIjogdG9yY2gudmVyc2lvbi5jdWRhLCAiZHR5cGUiOiAiZmxvYXQxNiIsICJhdHRlbnRpb24iOiAic2RwYSJ9CiAgICAgICAgc2VsZi5wcm9jZXNzb3IgPSBBdXRvUHJvY2Vzc29yLmZyb21fcHJldHJhaW5lZChtb2RlbF9pZCwgcmV2aXNpb249c2VsZi5yZXZpc2lvbikKICAgICAgICBzZWxmLm1vZGVsID0gUXdlbjNWTEZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbi5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgICAgIG1vZGVsX2lkLCByZXZpc2lvbj1zZWxmLnJldmlzaW9uLCBkdHlwZT10b3JjaC5mbG9hdDE2LCBkZXZpY2VfbWFwPXsiIjogMH0sIGF0dG5faW1wbGVtZW50YXRpb249InNkcGEiKS5ldmFsKCkKCiAgICBkZWYgaW5mZXIoc2VsZiwgaW1hZ2VfcGF0aDogUGF0aCB8IE5vbmUsIHByb21wdDogc3RyKSAtPiBkaWN0OgogICAgICAgIGZyb20gUElMIGltcG9ydCBJbWFnZQogICAgICAgIHRvcmNoID0gc2VsZi50b3JjaAogICAgICAgIGNvbnRlbnQgPSBbXQogICAgICAgIGlmIGltYWdlX3BhdGggaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHdpdGggSW1hZ2Uub3BlbihpbWFnZV9wYXRoKSBhcyBzb3VyY2U6CiAgICAgICAgICAgICAgICBpbWFnZSA9IHNvdXJjZS5jb252ZXJ0KCJSR0IiKQogICAgICAgICAgICBjb250ZW50LmFwcGVuZCh7InR5cGUiOiAiaW1hZ2UiLCAiaW1hZ2UiOiBpbWFnZX0pCiAgICAgICAgY29udGVudC5hcHBlbmQoeyJ0eXBlIjogInRleHQiLCAidGV4dCI6IHByb21wdH0pCiAgICAgICAgaW5wdXRzID0gZ2VuZXJhdGVkID0gdG9rZW5zID0gTm9uZQogICAgICAgIHRyeToKICAgICAgICAgICAgdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cygpCiAgICAgICAgICAgIGlucHV0cyA9IHNlbGYucHJvY2Vzc29yLmFwcGx5X2NoYXRfdGVtcGxhdGUoCiAgICAgICAgICAgICAgICBbeyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6IGNvbnRlbnR9XSwgdG9rZW5pemU9VHJ1ZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUsCiAgICAgICAgICAgICAgICByZXR1cm5fZGljdD1UcnVlLCByZXR1cm5fdGVuc29ycz0icHQiKS50byhzZWxmLm1vZGVsLmRldmljZSkKICAgICAgICAgICAgd2l0aCB0b3JjaC5pbmZlcmVuY2VfbW9kZSgpOgogICAgICAgICAgICAgICAgZ2VuZXJhdGVkID0gc2VsZi5tb2RlbC5nZW5lcmF0ZSgqKmlucHV0cywgZG9fc2FtcGxlPUZhbHNlLCBtYXhfbmV3X3Rva2Vucz1zZWxmLm1heF90b2tlbnMpCiAgICAgICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQogICAgICAgICAgICB0b2tlbnMgPSBnZW5lcmF0ZWRbMCwgaW5wdXRzWyJpbnB1dF9pZHMiXS5zaGFwZVsxXTpdCiAgICAgICAgICAgIHJldHVybiB7InRleHQiOiBzZWxmLnByb2Nlc3Nvci5kZWNvZGUodG9rZW5zLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUsIGNsZWFuX3VwX3Rva2VuaXphdGlvbl9zcGFjZXM9RmFsc2UpLAogICAgICAgICAgICAgICAgICAgICJyYXciOiB7ImdlbmVyYXRlZF90b2tlbl9pZHMiOiB0b2tlbnMudG9saXN0KCksICJwZWFrX2dwdV9hbGxvY2F0ZWRfYnl0ZXMiOiB0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKCl9LAogICAgICAgICAgICAgICAgICAgICJ1c2FnZSI6IHsicHJvbXB0X3Rva2VucyI6IGlucHV0c1siaW5wdXRfaWRzIl0uc2hhcGVbMV0sICJjb21wbGV0aW9uX3Rva2VucyI6IGxlbih0b2tlbnMpfSwKICAgICAgICAgICAgICAgICAgICAiYnVkZ2V0X3JlYWNoZWQiOiBsZW4odG9rZW5zKSA+PSBzZWxmLm1heF90b2tlbnN9CiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgZGVsIGlucHV0cywgZ2VuZXJhdGVkLCB0b2tlbnMKCgpjbGFzcyBMb2NhbEFkYXB0ZXI6CiAgICAiIiJPcHRpb25hbCBPbGxhbWEgZGlhZ25vc3RpYzogbmV2ZXIgc2lsZW50bHkgc3Vic3RpdHV0ZSBmb3IgdGhlIEluc3RydWN0IEZQMTYgc3R1ZHkuIiIiCiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWw6IHN0ciwgbWF4X3Rva2VuczogaW50KToKICAgICAgICBzZWxmLm1vZGVsID0gbW9kZWwKICAgICAgICBzZWxmLm1heF90b2tlbnMgPSBtYXhfdG9rZW5zCiAgICAgICAgd2l0aCB1cmxvcGVuKCJodHRwOi8vbG9jYWxob3N0OjExNDM0L2FwaS90YWdzIikgYXMgcmVzcG9uc2U6CiAgICAgICAgICAgIHRhZ3MgPSBqc29uLmxvYWQocmVzcG9uc2UpCiAgICAgICAgbWF0Y2hlcyA9IFttIGZvciBtIGluIHRhZ3NbIm1vZGVscyJdIGlmIG1bIm5hbWUiXSA9PSBtb2RlbF0KICAgICAgICBpZiBub3QgbWF0Y2hlczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1vZGVsIG5vdCBpbnN0YWxsZWQ6IHttb2RlbH07IG5vIGF1dG9tYXRpYyBkb3dubG9hZCIpCiAgICAgICAgd2l0aCB1cmxvcGVuKCJodHRwOi8vbG9jYWxob3N0OjExNDM0L2FwaS92ZXJzaW9uIikgYXMgcmVzcG9uc2U6CiAgICAgICAgICAgIHZlcnNpb24gPSBqc29uLmxvYWQocmVzcG9uc2UpCiAgICAgICAgc2VsZi5tZXRhZGF0YSA9IHsibW9kZWwiOiBtYXRjaGVzWzBdLCAicnVudGltZSI6IHZlcnNpb24sICJzY29wZSI6ICJzZXBhcmF0ZSBsb2NhbCBkaWFnbm9zdGljIn0KCiAgICBkZWYgaW5mZXIoc2VsZiwgaW1hZ2VfcGF0aDogUGF0aCB8IE5vbmUsIHByb21wdDogc3RyKSAtPiBkaWN0OgogICAgICAgIGltcG9ydCBiYXNlNjQKICAgICAgICBtZXNzYWdlID0geyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6IHByb21wdH0KICAgICAgICBpZiBpbWFnZV9wYXRoIGlzIG5vdCBOb25lOgogICAgICAgICAgICBtZXNzYWdlWyJpbWFnZXMiXSA9IFtiYXNlNjQuYjY0ZW5jb2RlKGltYWdlX3BhdGgucmVhZF9ieXRlcygpKS5kZWNvZGUoImFzY2lpIildCiAgICAgICAgcGF5bG9hZCA9IHsibW9kZWwiOiBzZWxmLm1vZGVsLCAibWVzc2FnZXMiOiBbbWVzc2FnZV0sICJzdHJlYW0iOiBGYWxzZSwKICAgICAgICAgICAgICAgICAgICJvcHRpb25zIjogeyJ0ZW1wZXJhdHVyZSI6IDAsICJudW1fcHJlZGljdCI6IHNlbGYubWF4X3Rva2VucywgIm51bV9jdHgiOiA0MDk2fSwgImtlZXBfYWxpdmUiOiAiMm0ifQogICAgICAgIHJlcXVlc3QgPSBSZXF1ZXN0KCJodHRwOi8vbG9jYWxob3N0OjExNDM0L2FwaS9jaGF0IiwgZGF0YT1qc29uLmR1bXBzKHBheWxvYWQpLmVuY29kZSgpLCBoZWFkZXJzPXsiQ29udGVudC1UeXBlIjogImFwcGxpY2F0aW9uL2pzb24ifSkKICAgICAgICB3aXRoIHVybG9wZW4ocmVxdWVzdCwgdGltZW91dD0zMDApIGFzIHJlc3BvbnNlOgogICAgICAgICAgICByYXcgPSBqc29uLmxvYWQocmVzcG9uc2UpCiAgICAgICAgcmV0dXJuIHsidGV4dCI6IHJhdy5nZXQoIm1lc3NhZ2UiLCB7fSkuZ2V0KCJjb250ZW50IiwgIiIpLCAicmF3IjogcmF3LAogICAgICAgICAgICAgICAgInVzYWdlIjogeyJwcm9tcHRfdG9rZW5zIjogcmF3LmdldCgicHJvbXB0X2V2YWxfY291bnQiKSwgImNvbXBsZXRpb25fdG9rZW5zIjogcmF3LmdldCgiZXZhbF9jb3VudCIpfSwKICAgICAgICAgICAgICAgICJidWRnZXRfcmVhY2hlZCI6IHJhdy5nZXQoImRvbmVfcmVhc29uIikgPT0gImxlbmd0aCJ9CgoKZGVmIGludm9rZShhZGFwdGVyLCBpbWFnZTogUGF0aCB8IE5vbmUsIHByb21wdDogc3RyKSAtPiBkaWN0OgogICAgc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIHRyeToKICAgICAgICBpbmZlcmVuY2UgPSBhZGFwdGVyLmluZmVyKGltYWdlLCBwcm9tcHQpCiAgICAgICAgcmV0dXJuIHsic3RhdHVzIjogIm9rIiwgInJhd19yZXNwb25zZSI6IGluZmVyZW5jZVsidGV4dCJdLCAicHJvdmlkZXJfcmVzcG9uc2UiOiBpbmZlcmVuY2VbInJhdyJdLAogICAgICAgICAgICAgICAgInVzYWdlIjogaW5mZXJlbmNlWyJ1c2FnZSJdLCAiYnVkZ2V0X3JlYWNoZWQiOiBpbmZlcmVuY2UuZ2V0KCJidWRnZXRfcmVhY2hlZCIsIEZhbHNlKSwKICAgICAgICAgICAgICAgICJsYXRlbmN5X3NlY29uZHMiOiByb3VuZCh0aW1lLnBlcmZfY291bnRlcigpLXN0YXJ0ZWQsIDQpfQogICAgZXhjZXB0IChFeGNlcHRpb24sIEtleWJvYXJkSW50ZXJydXB0KSBhcyBlcnJvcjoKICAgICAgICByZXR1cm4geyJzdGF0dXMiOiAiZXJyb3IiLCAiZXJyb3IiOiBmInt0eXBlKGVycm9yKS5fX25hbWVfX306IHtlcnJvcn0iLAogICAgICAgICAgICAgICAgImxhdGVuY3lfc2Vjb25kcyI6IHJvdW5kKHRpbWUucGVyZl9jb3VudGVyKCktc3RhcnRlZCwgNCl9CgoKZGVmIHJ1bihjb25maWdfcGF0aDogUGF0aCwgb3V0cHV0OiBQYXRoLCBzcGxpdDogc3RyLCBhZGFwdGVyPU5vbmUsIHJlc3VtZTogYm9vbCA9IEZhbHNlKSAtPiBkaWN0OgogICAgY29uZmlnID0ganNvbi5sb2Fkcyhjb25maWdfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICByb290ID0gY29uZmlnX3BhdGgucmVzb2x2ZSgpLnBhcmVudHNbMl0KICAgIG1hbmlmZXN0ID0gcm9vdCAvIGNvbmZpZ1siZGF0YXNldCJdCiAgICBpZiBzaGEyNTYobWFuaWZlc3QpICE9IGNvbmZpZ1siZGF0YXNldF9zaGEyNTYiXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJEYXRhc2V0IGhhc2ggZGlmZmVycyBmcm9tIGV4cGVyaW1lbnQgY29uZmlndXJhdGlvbiIpCiAgICBmcm9tIC5nZW5lcmF0ZV9iaW5kaW5nIGltcG9ydCB2YWxpZGF0ZQogICAgdmFsaWRhdGUobWFuaWZlc3QpCiAgICBleGFtcGxlcyA9IFtyIGZvciByIGluIChqc29uLmxvYWRzKGxpbmUpIGZvciBsaW5lIGluIG1hbmlmZXN0LnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKS5zcGxpdGxpbmVzKCkgaWYgbGluZS5zdHJpcCgpKSBpZiByWyJzcGxpdCJdID09IHNwbGl0XQogICAgaWYgbGVuKGV4YW1wbGVzKSAhPSBjb25maWdbImV4cGVjdGVkX3Jvd3MiXVtzcGxpdF06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVW5leHBlY3RlZCBudW1iZXIgb2Ygc2VsZWN0ZWQgZXhhbXBsZXMiKQogICAgZnJlZXplID0gb3V0cHV0LnBhcmVudCAvICJkZXZlbG9wbWVudCIgLyAiZnJlZXplLmpzb24iCiAgICBpZiBzcGxpdCA9PSAiZXZhbHVhdGlvbiIgYW5kIG5vdCBmcmVlemUuZXhpc3RzKCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ29tcGxldGUgZGV2ZWxvcG1lbnQgYW5kIGl0cyBvcGVyYXRpb25hbCBnYXRlIGJlZm9yZSBldmFsdWF0aW9uIikKICAgIGZyb3plbiA9IGpzb24ubG9hZHMoZnJlZXplLnJlYWRfdGV4dCgpKSBpZiBzcGxpdCA9PSAiZXZhbHVhdGlvbiIgZWxzZSBOb25lCiAgICBpZiBmcm96ZW4gYW5kIGZyb3plblsiY29uZmlnX3NoYTI1NiJdICE9IHNoYTI1Nihjb25maWdfcGF0aCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ29uZmlndXJhdGlvbiBjaGFuZ2VkIGFmdGVyIGRldmVsb3BtZW50IGZyZWV6ZSIpCiAgICBlbnZpcm9ubWVudF9wYXRoID0gb3V0cHV0IC8gImVudmlyb25tZW50Lmpzb24iCiAgICBwcmV2aW91cyA9IE5vbmUKICAgIGlmIG91dHB1dC5leGlzdHMoKToKICAgICAgICBpZiBub3QgcmVzdW1lOgogICAgICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3Iob3V0cHV0KQogICAgICAgIHByZXZpb3VzID0ganNvbi5sb2FkcyhlbnZpcm9ubWVudF9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICBpZiBwcmV2aW91c1siY29uZmlnX3NoYTI1NiJdICE9IHNoYTI1Nihjb25maWdfcGF0aCkgb3IgcHJldmlvdXNbInNwbGl0Il0gIT0gc3BsaXQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlJlc3VtZSBjb25maWd1cmF0aW9uIG9yIHNwbGl0IG1pc21hdGNoIikKICAgIGVsaWYgcmVzdW1lOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNhbm5vdCByZXN1bWUgYSBtaXNzaW5nIHJ1biIpCiAgICBpZiBhZGFwdGVyIGlzIE5vbmU6CiAgICAgICAgaWYgY29uZmlnWyJwcm92aWRlciJdID09ICJ0cmFuc2Zvcm1lcnMiOgogICAgICAgICAgICByZXZpc2lvbiA9IChwcmV2aW91cyBvciBmcm96ZW4gb3Ige30pLmdldCgibW9kZWxfcmV2aXNpb24iKQogICAgICAgICAgICBhZGFwdGVyID0gVHJhbnNmb3JtZXJzQWRhcHRlcihjb25maWdbIm1vZGVsIl0sIGNvbmZpZ1sibWF4X3Rva2VucyJdLCByZXZpc2lvbikKICAgICAgICBlbGlmIGNvbmZpZ1sicHJvdmlkZXIiXSA9PSAib2xsYW1hIjoKICAgICAgICAgICAgYWRhcHRlciA9IExvY2FsQWRhcHRlcihjb25maWdbIm1vZGVsIl0sIGNvbmZpZ1sibWF4X3Rva2VucyJdKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk9ubHkgZnJlZSBsb2NhbCBleGVjdXRpb24gcHJvdmlkZXJzIGFyZSBwZXJtaXR0ZWQiKQogICAgbWV0YWRhdGEgPSBhZGFwdGVyLm1ldGFkYXRhCiAgICBpZiBwcmV2aW91cyBhbmQgbWV0YWRhdGEgIT0gcHJldmlvdXNbImFkYXB0ZXIiXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJSdW50aW1lL21vZGVsIGNoYW5nZWQ7IGNhbm5vdCBhcHBlbmQgdG8gZXhpc3RpbmcgcnVuIikKICAgIGlmIGZyb3plbiBhbmQgbWV0YWRhdGEgIT0gZnJvemVuWyJhZGFwdGVyIl06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUnVudGltZS9tb2RlbCBkaWZmZXJzIGZyb20gZGV2ZWxvcG1lbnQgZnJlZXplIikKICAgIHJ1bl9pZCA9IHByZXZpb3VzWyJydW5faWQiXSBpZiBwcmV2aW91cyBlbHNlIHN0cih1dWlkLnV1aWQ0KCkpCiAgICBlbnZpcm9ubWVudCA9IHsicnVuX2lkIjogcnVuX2lkLCAic3BsaXQiOiBzcGxpdCwgImNvbmZpZyI6IGNvbmZpZywgImNvbmZpZ19zaGEyNTYiOiBzaGEyNTYoY29uZmlnX3BhdGgpLAogICAgICAgICAgICAgICAgICAgIm1vZGVsX3JldmlzaW9uIjogbWV0YWRhdGEuZ2V0KCJyZXZpc2lvbiIpLCAiYWRhcHRlciI6IG1ldGFkYXRhLCAicHl0aG9uIjogcGxhdGZvcm0ucHl0aG9uX3ZlcnNpb24oKSwKICAgICAgICAgICAgICAgICAgICJpbWFnZV9oYXNoZXMiOiB7clsiaW1hZ2UiXTogc2hhMjU2KG1hbmlmZXN0LnBhcmVudCAvIHJbImltYWdlIl0pIGZvciByIGluIGV4YW1wbGVzfX0KICAgIGlmIHByZXZpb3VzIGFuZCBlbnZpcm9ubWVudFsiaW1hZ2VfaGFzaGVzIl0gIT0gcHJldmlvdXNbImltYWdlX2hhc2hlcyJdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkltYWdlIGJ5dGVzIGNoYW5nZWQgc2luY2Ugb3JpZ2luYWwgcnVuIikKICAgIGlmIG5vdCBwcmV2aW91czoKICAgICAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlKQogICAgICAgIGVudmlyb25tZW50X3BhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKGVudmlyb25tZW50LCBpbmRlbnQ9MikrIlxuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHJhdyA9IG91dHB1dCAvICJ2aXN1YWwuanNvbmwiCiAgICBjb21wbGV0ZWQgPSB7fQogICAgaWYgcmF3LmV4aXN0cygpOgogICAgICAgIGZvciBsaW5lIGluIHJhdy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpOgogICAgICAgICAgICByID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgICAgICBpZiByWyJleGFtcGxlX2lkIl0gaW4gY29tcGxldGVkOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRHVwbGljYXRlIHNhdmVkIHJvdyIpCiAgICAgICAgICAgIGNvbXBsZXRlZFtyWyJleGFtcGxlX2lkIl1dID0gcgogICAgICAgIGlmIG5vdCBzZXQoY29tcGxldGVkKSA8PSB7ZVsiZXhhbXBsZV9pZCJdIGZvciBlIGluIGV4YW1wbGVzfToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVW5leHBlY3RlZCBzYXZlZCBleGFtcGxlIikKICAgIHN0b3AgPSBGYWxzZQogICAgd2l0aCByYXcub3BlbigiYSIgaWYgcHJldmlvdXMgZWxzZSAieCIsIGVuY29kaW5nPSJ1dGYtOCIsIG5ld2xpbmU9IlxuIikgYXMgaGFuZGxlOgogICAgICAgIGZvciBleGFtcGxlIGluIGV4YW1wbGVzOgogICAgICAgICAgICBpZiBleGFtcGxlWyJleGFtcGxlX2lkIl0gaW4gY29tcGxldGVkOgogICAgICAgICAgICAgICAgY29udGludWUgICMgRXZlbiBmYWlsZWQgcmVxdWVzdHMgYXJlIGltbXV0YWJsZTsgbm8gc2VsZWN0aXZlIHJldHJpZXMuCiAgICAgICAgICAgICMgTG9jYWwgZGlhZ25vc3RpYyBtdXN0IHN0b3Agd2hlbiBtZW1vcnkgaGVhZHJvb20gYmVjb21lcyB1bnNhZmUuCiAgICAgICAgICAgIGlmIGNvbmZpZ1sicHJvdmlkZXIiXSA9PSAib2xsYW1hIiBhbmQgcGxhdGZvcm0uc3lzdGVtKCkgPT0gIldpbmRvd3MiOgogICAgICAgICAgICAgICAgaW1wb3J0IGN0eXBlcwogICAgICAgICAgICAgICAgY2xhc3MgTWVtb3J5KGN0eXBlcy5TdHJ1Y3R1cmUpOgogICAgICAgICAgICAgICAgICAgIF9maWVsZHNfID0gWygibGVuZ3RoIiwgY3R5cGVzLmNfdWxvbmcpLCAoImxvYWQiLCBjdHlwZXMuY191bG9uZyldICsgWyhmInZhbHVle2l9IiwgY3R5cGVzLmNfdWxvbmdsb25nKSBmb3IgaSBpbiByYW5nZSg3KV0KICAgICAgICAgICAgICAgIG1lbW9yeSA9IE1lbW9yeSgpOyBtZW1vcnkubGVuZ3RoID0gY3R5cGVzLnNpemVvZihtZW1vcnkpCiAgICAgICAgICAgICAgICBjdHlwZXMud2luZGxsLmtlcm5lbDMyLkdsb2JhbE1lbW9yeVN0YXR1c0V4KGN0eXBlcy5ieXJlZihtZW1vcnkpKQogICAgICAgICAgICAgICAgaWYgbWVtb3J5LnZhbHVlMSA8IDEqMioqMzA6CiAgICAgICAgICAgICAgICAgICAgc3RvcCA9IFRydWUKICAgICAgICAgICAgICAgICAgICBwcmludCgiU3RvcHBlZDogbGVzcyB0aGFuIDEgR2lCIGF2YWlsYWJsZSBSQU0iLCBmbHVzaD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIHJlc3VsdCA9IGludm9rZShhZGFwdGVyLCBtYW5pZmVzdC5wYXJlbnQgLyBleGFtcGxlWyJpbWFnZSJdLCBleGFtcGxlWyJxdWVzdGlvbiJdKQogICAgICAgICAgICByb3cgPSB7KipleGFtcGxlLCAqKnJlc3VsdCwgInJ1bl9pZCI6IHJ1bl9pZCwgInByb3ZpZGVyIjogY29uZmlnWyJwcm92aWRlciJdLCAibW9kZWwiOiBjb25maWdbIm1vZGVsIl0sCiAgICAgICAgICAgICAgICAgICAiaW5mZXJlbmNlX2NvbmZpZyI6IHsiZG9fc2FtcGxlIjogRmFsc2UsICJtYXhfdG9rZW5zIjogY29uZmlnWyJtYXhfdG9rZW5zIl19fQogICAgICAgICAgICBoYW5kbGUud3JpdGUoanNvbi5kdW1wcyhyb3csIHNvcnRfa2V5cz1UcnVlKSsiXG4iKTsgaGFuZGxlLmZsdXNoKCkKICAgICAgICAgICAgY29tcGxldGVkW2V4YW1wbGVbImV4YW1wbGVfaWQiXV0gPSByb3cKICAgICAgICAgICAgcHJpbnQoanNvbi5kdW1wcyh7ImV4YW1wbGVfaWQiOiBleGFtcGxlWyJleGFtcGxlX2lkIl0sICJzdGF0dXMiOiByb3dbInN0YXR1cyJdLCAiYW5zd2VyIjogcm93LmdldCgicmF3X3Jlc3BvbnNlIiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJzZWNvbmRzIjogcm93WyJsYXRlbmN5X3NlY29uZHMiXSwgImNvbXBsZXRlZCI6IGxlbihjb21wbGV0ZWQpLCAidG90YWwiOiBsZW4oZXhhbXBsZXMpfSksIGZsdXNoPVRydWUpCiAgICAgICAgICAgIGlmIHJvd1sic3RhdHVzIl0gIT0gIm9rIjoKICAgICAgICAgICAgICAgIHN0b3AgPSBUcnVlCiAgICAgICAgICAgICAgICBicmVhawogICAgZXZhbHVhdGlvbl9wYXRoID0gb3V0cHV0IC8gIm1ldHJpY3MiCiAgICBpZiBub3QgZXZhbHVhdGlvbl9wYXRoLmV4aXN0cygpOgogICAgICAgIG1ldHJpY3MgPSBldmFsdWF0ZShtYW5pZmVzdCwgcmF3LCBldmFsdWF0aW9uX3BhdGgsIHNwbGl0KQogICAgZWxzZToKICAgICAgICAjIEEgcmVzdW1lZCBydW4gZ2V0cyBhIG5ldyBhbmFseXNpcyBkaXJlY3Rvcnk7IG5ldmVyIG92ZXJ3cml0ZSBhIHBhcnRpYWwgcmVwb3J0LgogICAgICAgIGV2YWx1YXRpb25fcGF0aCA9IG91dHB1dCAvICgibWV0cmljcy0iK3V1aWQudXVpZDQoKS5oZXhbOjhdKQogICAgICAgIG1ldHJpY3MgPSBldmFsdWF0ZShtYW5pZmVzdCwgcmF3LCBldmFsdWF0aW9uX3BhdGgsIHNwbGl0KQogICAgaWYgc3RvcCBvciBsZW4oY29tcGxldGVkKSAhPSBsZW4oZXhhbXBsZXMpOgogICAgICAgIHJldHVybiB7InN0YXRlIjogImluY29tcGxldGUiLCAic2F2ZWQiOiBsZW4oY29tcGxldGVkKSwgImV4cGVjdGVkIjogbGVuKGV4YW1wbGVzKX0KICAgICMgVHdvIGFjdHVhbCB0ZXh0LW9ubHkgcmVxdWVzdHM7IHByb2plY3RlZCByb3dzIGFyZSBleHBsaWNpdGx5IGxpbmtlZCB0byB0aGVpciBzb3VyY2UuCiAgICB0ZXh0X3BhdGggPSBvdXRwdXQgLyAidGV4dC1vbmx5LXJlcXVlc3RzLmpzb25sIgogICAgdGV4dF9yZXF1ZXN0cyA9IHt9CiAgICBpZiB0ZXh0X3BhdGguZXhpc3RzKCk6CiAgICAgICAgdGV4dF9yZXF1ZXN0cyA9IHtyWyJxdWVzdGlvbiJdOiByIGZvciByIGluIChqc29uLmxvYWRzKGxpbmUpIGZvciBsaW5lIGluIHRleHRfcGF0aC5yZWFkX3RleHQoKS5zcGxpdGxpbmVzKCkpfQogICAgd2l0aCB0ZXh0X3BhdGgub3BlbigiYSIgaWYgdGV4dF9wYXRoLmV4aXN0cygpIGVsc2UgIngiLCBlbmNvZGluZz0idXRmLTgiLCBuZXdsaW5lPSJcbiIpIGFzIGhhbmRsZToKICAgICAgICBmb3IgcHJvbXB0IGluIHNvcnRlZCh7ZVsicXVlc3Rpb24iXSBmb3IgZSBpbiBleGFtcGxlc30pOgogICAgICAgICAgICBpZiBwcm9tcHQgbm90IGluIHRleHRfcmVxdWVzdHM6CiAgICAgICAgICAgICAgICByZXN1bHQgPSB7KippbnZva2UoYWRhcHRlciwgTm9uZSwgcHJvbXB0KSwgInF1ZXN0aW9uIjogcHJvbXB0LCAic291cmNlX3JlcXVlc3RfaWQiOiBzdHIodXVpZC51dWlkNCgpKX0KICAgICAgICAgICAgICAgIGhhbmRsZS53cml0ZShqc29uLmR1bXBzKHJlc3VsdCkrIlxuIik7IGhhbmRsZS5mbHVzaCgpCiAgICAgICAgICAgICAgICB0ZXh0X3JlcXVlc3RzW3Byb21wdF0gPSByZXN1bHQKICAgICAgICAgICAgICAgIGlmIHJlc3VsdFsic3RhdHVzIl0gIT0gIm9rIjoKICAgICAgICAgICAgICAgICAgICByZXR1cm4geyJzdGF0ZSI6ICJ0ZXh0LW9ubHktZXJyb3IiLCAic2F2ZWQiOiBsZW4oY29tcGxldGVkKX0KICAgIHByb2plY3RlZCA9IG91dHB1dCAvICJ0ZXh0LW9ubHktcHJvamVjdGVkLmpzb25sIgogICAgaWYgbm90IHByb2plY3RlZC5leGlzdHMoKToKICAgICAgICB3aXRoIHByb2plY3RlZC5vcGVuKCJ4IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iXG4iKSBhcyBoYW5kbGU6CiAgICAgICAgICAgIGZvciBleGFtcGxlIGluIGV4YW1wbGVzOgogICAgICAgICAgICAgICAgcm93ID0geyoqZXhhbXBsZSwgKip0ZXh0X3JlcXVlc3RzW2V4YW1wbGVbInF1ZXN0aW9uIl1dLCAicnVuX2lkIjogcnVuX2lkKyItdGV4dC1vbmx5IiwKICAgICAgICAgICAgICAgICAgICAgICAicHJvdmlkZXIiOiBjb25maWdbInByb3ZpZGVyIl0sICJtb2RlbCI6IGNvbmZpZ1sibW9kZWwiXSwgImV4cGFuZGVkX3RleHRfb25seSI6IFRydWV9CiAgICAgICAgICAgICAgICBoYW5kbGUud3JpdGUoanNvbi5kdW1wcyhyb3csIHNvcnRfa2V5cz1UcnVlKSsiXG4iKQogICAgaWYgbm90IChvdXRwdXQgLyAidGV4dC1vbmx5LW1ldHJpY3MiKS5leGlzdHMoKToKICAgICAgICBldmFsdWF0ZShtYW5pZmVzdCwgcHJvamVjdGVkLCBvdXRwdXQgLyAidGV4dC1vbmx5LW1ldHJpY3MiLCBzcGxpdCkKICAgICMgRnJlZXplIGFuIG9wZXJhdGlvbmFsbHkgdmFsaWQgZGV2ZWxvcG1lbnQgcnVuLCByZWdhcmRsZXNzIG9mIGl0cyBhY2N1cmFjeS4KICAgIHJlcXVlc3RzID0gbWV0cmljc1sicmVxdWVzdHMiXQogICAgZ2F0ZSA9IGFsbChyZXF1ZXN0c1trZXldID09IDAgZm9yIGtleSBpbiAoImVycm9yIiwgIm1pc3NpbmciLCAiZW1wdHlfZmluYWxfYW5zd2VyIiwgImJ1ZGdldF9yZWFjaGVkIiwgInBhcnNlX2ZhaWx1cmUiKSkKICAgIGlmIHNwbGl0ID09ICJkZXZlbG9wbWVudCIgYW5kIGdhdGUgYW5kIG5vdCAob3V0cHV0IC8gImZyZWV6ZS5qc29uIikuZXhpc3RzKCk6CiAgICAgICAgKG91dHB1dCAvICJmcmVlemUuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyh7ImNvbmZpZ19zaGEyNTYiOiBzaGEyNTYoY29uZmlnX3BhdGgpLCAiYWRhcHRlciI6IG1ldGFkYXRhLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIm1vZGVsX3JldmlzaW9uIjogbWV0YWRhdGEuZ2V0KCJyZXZpc2lvbiIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInJ1bGUiOiAiTm8gcmVxdWVzdCBlcnJvcnMsIG1pc3Npbmcgcm93cywgZW1wdHkgYW5zd2VycywgdHJ1bmNhdGlvbiBvciBwYXJzZSBmYWlsdXJlczsgbm8gYWNjdXJhY3kgdGhyZXNob2xkLiJ9LCBpbmRlbnQ9MikrIlxuIikKICAgIHJldHVybiB7InN0YXRlIjogImNvbXBsZXRlIiBpZiBzcGxpdCA9PSAiZXZhbHVhdGlvbiIgb3IgZ2F0ZSBlbHNlICJkZXZlbG9wbWVudC1nYXRlLWZhaWxlZCIsICJzYXZlZCI6IGxlbihjb21wbGV0ZWQpLCAibWV0cmljcyI6IHN0cihldmFsdWF0aW9uX3BhdGgpfQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jb25maWciLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1kaXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNwbGl0IiwgY2hvaWNlcz0oImRldmVsb3BtZW50IiwgImV2YWx1YXRpb24iKSwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmVzdW1lIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBwcmludChqc29uLmR1bXBzKHJ1bihhcmdzLmNvbmZpZywgYXJncy5vdXRwdXRfZGlyLCBhcmdzLnNwbGl0LCByZXN1bWU9YXJncy5yZXN1bWUpLCBpbmRlbnQ9MikpCg==', 'sha256': 'e2ca24eadf3fb52fbd32519d513ce76c362a157e24684bf0d552fb52311b57fc'}, 'causalsight/binding_baselines.py': {'base64': 'IiIiUnVuIGRldGVybWluaXN0aWMgc2hvcnRjdXQgYmFzZWxpbmVzIHdpdGggdGhlIHNhbWUgbWFuaWZlc3QgYW5kIGV2YWx1YXRvci4iIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gLmV2YWx1YXRlX2JpbmRpbmcgaW1wb3J0IGV2YWx1YXRlCgoKZGVmIHJ1bihtYW5pZmVzdDogUGF0aCwgb3V0cHV0OiBQYXRoLCBzcGxpdDogc3RyID0gImV2YWx1YXRpb24iKSAtPiBkaWN0OgogICAgaWYgb3V0cHV0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcihvdXRwdXQpCiAgICBvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlKQogICAgZXhhbXBsZXMgPSBbanNvbi5sb2FkcyhsaW5lKSBmb3IgbGluZSBpbiBtYW5pZmVzdC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KICAgIHJlc3VsdHMgPSB7fQogICAgZm9yIGJhc2VsaW5lIGluICgiY29uc3RhbnRfcmVkIiwgImxlZnRtb3N0X29iamVjdCIsICJkaXN0cmFjdG9yX29iamVjdCIpOgogICAgICAgIHJhdyA9IG91dHB1dCAvIChiYXNlbGluZSsiLmpzb25sIikKICAgICAgICB3aXRoIHJhdy5vcGVuKCJ4IiwgZW5jb2Rpbmc9InV0Zi04IiwgbmV3bGluZT0iXG4iKSBhcyBoYW5kbGU6CiAgICAgICAgICAgIGZvciBleGFtcGxlIGluIGV4YW1wbGVzOgogICAgICAgICAgICAgICAgaWYgZXhhbXBsZVsic3BsaXQiXSAhPSBzcGxpdDoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgb2JqZWN0cyA9IGV4YW1wbGVbInNjZW5lIl1bIm9iamVjdHMiXQogICAgICAgICAgICAgICAgaWYgYmFzZWxpbmUgPT0gImNvbnN0YW50X3JlZCI6CiAgICAgICAgICAgICAgICAgICAgYW5zd2VyID0gInJlZCIKICAgICAgICAgICAgICAgIGVsaWYgYmFzZWxpbmUgPT0gImxlZnRtb3N0X29iamVjdCI6CiAgICAgICAgICAgICAgICAgICAgYW5zd2VyID0gbWluKG9iamVjdHMsIGtleT1sYW1iZGEgbzogb1siY2VudGVyX3giXSlbImNvbG9yIl0KICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgYW5zd2VyID0gbmV4dChvWyJjb2xvciJdIGZvciBvIGluIG9iamVjdHMgaWYgb1sibmFtZSJdICE9IGV4YW1wbGVbInF1ZXJpZWRfc2hhcGUiXSkKICAgICAgICAgICAgICAgIHJvdyA9IHsqKmV4YW1wbGUsICJzdGF0dXMiOiAib2siLCAicmF3X3Jlc3BvbnNlIjogYW5zd2VyLCAicnVuX2lkIjogZiJiYXNlbGluZS17YmFzZWxpbmV9LXtzcGxpdH0iLAogICAgICAgICAgICAgICAgICAgICAgICJwcm92aWRlciI6ICJtZXRhZGF0YS1oZXVyaXN0aWMiLCAibW9kZWwiOiBiYXNlbGluZX0KICAgICAgICAgICAgICAgIGhhbmRsZS53cml0ZShqc29uLmR1bXBzKHJvdywgc29ydF9rZXlzPVRydWUpKyJcbiIpCiAgICAgICAgcmVzdWx0c1tiYXNlbGluZV0gPSBldmFsdWF0ZShtYW5pZmVzdCwgcmF3LCBvdXRwdXQgLyBiYXNlbGluZSwgc3BsaXQpCiAgICAob3V0cHV0IC8gInN1bW1hcnkuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXN1bHRzLCBpbmRlbnQ9MikrIlxuIiwgZW5jb2Rpbmc9InV0Zi04IikKICAgIGxpbmVzID0gWyIjIFNob3J0Y3V0IGJhc2VsaW5lIHJlc3VsdHMiLCAiIiwgIk1ldGFkYXRhIGhldXJpc3RpY3MsIG5vdCBtb2RlbCBpbmZlcmVuY2UuIiwgIiIsCiAgICAgICAgICAgICAifCBCYXNlbGluZSB8IE9yaWdpbmFsIGFjY3VyYWN5IHwgQ29ycmVjdCBjYXVzYWwgdXBkYXRlIHwgTnVpc2FuY2UgcHJlc2VydmF0aW9uIHwgQ29ycmVjdCB0cmlwbGV0IHwgUXVlc3Rpb24gc3dpdGNoIChkaWZmZXJlbnQgY29sb3JzKSB8IiwKICAgICAgICAgICAgICJ8LS0tfC0tLTp8LS0tOnwtLS06fC0tLTp8LS0tOnwiXQogICAgZm9yIG5hbWUsIG1ldHJpY3MgaW4gcmVzdWx0cy5pdGVtcygpOgogICAgICAgIGNlbGxzID0gW2Yie21ldHJpY3Nba2V5XVsnbnVtZXJhdG9yJ119L3ttZXRyaWNzW2tleV1bJ2Rlbm9taW5hdG9yJ119ICh7bWV0cmljc1trZXldWydyYXRlJ106LjElfSkiCiAgICAgICAgICAgICAgICAgZm9yIGtleSBpbiAoIm9yaWdpbmFsX2FjY3VyYWN5IiwgImNvcnJlY3RfY2F1c2FsX3VwZGF0ZSIsICJudWlzYW5jZV9hbnN3ZXJfcHJlc2VydmF0aW9uIiwgImNvbXBsZXRlX3RyaXBsZXRfY29ycmVjdG5lc3MiLCAicXVlc3Rpb25fc3dpdGNoX2NvcnJlY3RfZGlmZmVyZW50X2NvbG9ycyIpXQogICAgICAgIGxpbmVzLmFwcGVuZCgifCAiK25hbWUrIiB8ICIrIiB8ICIuam9pbihjZWxscykrIiB8IikKICAgIGxpbmVzICs9IFsiIiwgIk51aXNhbmNlIHByZXNlcnZhdGlvbiBjYW4gYmUgaGlnaCBmb3IgYSBjb25zaXN0ZW50bHkgd3JvbmcgcHJlZGljdG9yLiBJbnNwZWN0IGNvcnJlY3RuZXNzIGpvaW50bHkuIl0KICAgIChvdXRwdXQgLyAicmVwb3J0Lm1kIikud3JpdGVfdGV4dCgiXG4iLmpvaW4obGluZXMpKyJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICByZXR1cm4gcmVzdWx0cwoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQtZGlyIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zcGxpdCIsIGNob2ljZXM9KCJkZXZlbG9wbWVudCIsICJldmFsdWF0aW9uIiksIGRlZmF1bHQ9ImV2YWx1YXRpb24iKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHJlc3VsdHMgPSBydW4oYXJncy5kYXRhc2V0LCBhcmdzLm91dHB1dF9kaXIsIGFyZ3Muc3BsaXQpCiAgICBwcmludChqc29uLmR1bXBzKHtuYW1lOiBkYXRhWyJvcmlnaW5hbF9hY2N1cmFjeSJdIGZvciBuYW1lLCBkYXRhIGluIHJlc3VsdHMuaXRlbXMoKX0sIGluZGVudD0yKSkK', 'sha256': '222df5afddaa704f48bad510ff9b7cdcae2ff6caa684a0cee4ac4aabf1e56bb9'}}

for relative, asset in ASSETS.items():
    data = base64.b64decode(asset["base64"])
    assert hashlib.sha256(data).hexdigest() == asset["sha256"]
    target = WORK / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.read_bytes() != data:
        raise FileExistsError(f"Changed existing bundled file: {target}")
    target.write_bytes(data)
sys.path.insert(0, str(WORK))
CONFIG = WORK / "configs/experiments/EXP-005-binding-v1.json"
RUN_DIR = WORK / "results" / ("EXP-005-colab-" + uuid.uuid4().hex[:8])
from causalsight.generate_binding import validate
from causalsight.run_binding import run, TransformersAdapter
from causalsight.binding_baselines import run as run_baselines
audit = validate(WORK / "data/binding-v1/examples.jsonl")
print(audit["families_by_split"], "Dataset audit passed")
print("Artifacts:", RUN_DIR)

In [ ]:
config = json.loads(CONFIG.read_text())
adapter = TransformersAdapter(config["model"], config["max_tokens"])
print("Pinned model revision:", adapter.revision)
development = run(CONFIG, RUN_DIR / "development", "development", adapter=adapter)
print(json.dumps(development, indent=2))
print((RUN_DIR / "development/metrics/report.md").read_text())

Review the development results above. If transport, empty-answer, truncation or parser
failures occurred, do not change settings and silently continue on evaluation. Record a
new development/configuration version instead. The next cell requires freeze.json.

In [ ]:
if not (RUN_DIR / "development/freeze.json").exists():
    raise RuntimeError("Development operational gate failed. Download artifacts and investigate.")
evaluation = run(CONFIG, RUN_DIR / "evaluation", "evaluation", adapter=adapter)
print(json.dumps(evaluation, indent=2))
print((RUN_DIR / "evaluation/metrics/report.md").read_text())
baselines = run_baselines(WORK / "data/binding-v1/examples.jsonl", RUN_DIR / "baselines")
print((RUN_DIR / "baselines/report.md").read_text())

In [ ]:
import zipfile
from google.colab import files
if not RUN_DIR.exists():
    raise RuntimeError("No run artifacts yet")
archive = WORK / (RUN_DIR.name+"-"+uuid.uuid4().hex[:8]+".zip")
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(RUN_DIR.parent))
    bundle.write(CONFIG, "experiment-config.json")
    bundle.writestr("bundle-hashes.json", json.dumps({p:a["sha256"] for p,a in ASSETS.items()}, indent=2))
files.download(str(archive))

In [ ]:
import gc
del adapter
gc.collect()
torch.cuda.empty_cache()
print("Use Runtime → Disconnect and delete runtime after downloading results.")